# Slice transformer — fold 2, Colab
Additional matched comparison alongside Kaggle folds 1 and 3. Uses the original
ConvNeXt-Tiny V1 backbone and protocol selection in both arms. One short smoke
test, then 12 control epochs and 12 slice-transformer epochs. No parent training
or smoke timeout; Colab session limits still apply.

Select one GPU (A100 if available), set `DRIVE_CACHE` to your existing full-shard
folder, then run in order and authorize Drive. The same 224-pixel, 24-slice cache
is copied to VM-local storage and verified; no image shards are downloaded.
Only missing small labels, side predictions or competition metadata may require
Kaggle access (`KAGGLE_API_TOKEN` Colab secret or a hidden prompt).

Best checkpoints and logs are saved to Drive throughout training. A rerun creates
a new attempt from epoch 0; it does not resume an interrupted optimizer/EMA state.
Single-GPU results must be compared with this notebook's matched control.
Gold is diagnostic; the final `comparison.json` contains the mined OOF gate.


In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/kagglehub"
DRIVE_OUT = "/content/drive/MyDrive/knee_runs/slice_transformer_fold2_v1"
LOCAL_INPUTS = "/content/knee_slice_inputs"


In [ ]:
%pip install -q timm==1.0.26 lightning==2.6.5 kagglehub pyarrow scikit-learn opencv-python-headless "pydicom>=3.0" numpy pandas tqdm
import importlib
# Exercise imports before copying 30 GB or creating a training attempt.
for package in ("pydicom", "cv2", "numpy", "pandas", "pyarrow", "sklearn",
                "tqdm", "timm", "lightning.pytorch", "kagglehub"):
    importlib.import_module(package)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError("Choose a runtime with exactly one GPU")
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__)
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
"""Drive staging and execution for the standalone slice-transformer pair."""

import ast
import hashlib
import json
import os
from pathlib import Path
import shutil
import sys
import uuid

import subprocess
class ColabPaths(ast.NodeTransformer):
    def __init__(self, output):
        self.output = str(output)

    def visit_Constant(self, node):
        if isinstance(node.value, str) and node.value.startswith("/kaggle/working"):
            return ast.copy_location(ast.Constant(
                self.output + node.value[len("/kaggle/working"):]), node)
        return node

def relocate(source, output):
    tree = ColabPaths(output).visit(ast.parse(source))
    return ast.unparse(ast.fix_missing_locations(tree)) + "\n"

def run_process(script, directory):
    with (directory / "train.log").open("w", buffering=1) as log:
        process = subprocess.Popen([sys.executable, "-u", str(script)], cwd=directory,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            if process.wait() != 0:
                raise RuntimeError(f"Training failed; inspect {directory / 'train.log'}")
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            process.stdout.close()



def copy_verified(source, target):
    """Stream-copy from Drive; verify bytes locally before exposing the input."""
    source, target = Path(source), Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        # Adopt copies left by the original staging cell, which wrote no
        # receipt until every metadata file had been resolved. Size alone is
        # insufficient: verify both contents before reusing the local copy.
        print(f"Verifying existing local copy: {target.name}", flush=True)
        with source.open("rb") as handle:
            expected = hashlib.file_digest(handle, "sha256").hexdigest()
        with target.open("rb") as handle:
            actual = hashlib.file_digest(handle, "sha256").hexdigest()
        if actual == expected:
            print(f"Reusing {target.name}; no recopy", flush=True)
            return actual
    if shutil.disk_usage(target.parent).free < source.stat().st_size:
        raise RuntimeError(f"Insufficient local disk to copy {source.name}")
    temporary = target.with_name(target.name + ".partial")
    digest = hashlib.sha256()
    with source.open("rb") as incoming, temporary.open("wb") as outgoing:
        for chunk in iter(lambda: incoming.read(8 << 20), b""):
            digest.update(chunk)
            outgoing.write(chunk)
    with temporary.open("rb") as copied:
        actual = hashlib.file_digest(copied, "sha256").hexdigest()
    if temporary.stat().st_size != source.stat().st_size or actual != digest.hexdigest():
        raise RuntimeError(f"Copy verification failed: {source}")
    temporary.replace(target)
    return actual


def choose_metadata(hits, name, expected_hash=None):
    """Collapse byte-identical copies; otherwise fetch the authoritative file.

    Never pick an arbitrary Drive version. The labels have a pinned hash, so
    select the matching copy even when older label matrices coexist.
    """
    groups = {}
    for path in sorted(map(Path, hits)):
        with path.open("rb") as handle:
            digest = hashlib.file_digest(handle, "sha256").hexdigest()
        groups.setdefault(digest, []).append(path)
    if expected_hash is not None:
        matches = groups.get(expected_hash, [])
        if matches:
            return matches[0]
    elif len(groups) == 1:
        copies = next(iter(groups.values()))
        if len(copies) > 1:
            print(f"{name}: {len(copies)} identical Drive copies; using {copies[0]}")
        return copies[0]
    if groups:
        print(f"{name}: Drive versions differ or do not match the required hash; "
              "fetching official metadata from Kaggle", flush=True)
    return None


def stage_drive_inputs(drive_cache, local_root, labels_hash):
    import numpy as np
    import pandas as pd

    root, local = Path(drive_cache), Path(local_root)
    if not root.is_dir():
        raise RuntimeError(f"Set DRIVE_CACHE to the folder containing your shards: {root}")
    local.mkdir(parents=True, exist_ok=True)
    cache_local = local / "cache"
    cache_local.mkdir(exist_ok=True)

    def unique(pattern, expected):
        matches = sorted(root.rglob(pattern))
        if len(matches) != expected:
            raise RuntimeError(f"Expected {expected} {pattern} in DRIVE_CACHE; found {matches}. "
                               "Choose one complete cache folder, without duplicate versions.")
        return matches

    shards = unique("cache_train_series_*.npy", 3)
    manifests = unique("manifest_train_series_*.parquet", 3)
    by_name = {path.name: path for path in manifests}
    pairs = []
    for shard in shards:
        name = shard.name.replace("cache_", "manifest_", 1).replace(".npy", ".parquet")
        if name not in by_name:
            raise RuntimeError(f"Missing matching manifest: {name}")
        pairs.append((shard, by_name[name]))
    needed = sum(path.stat().st_size for pair in pairs for path in pair
                 if not (cache_local / path.name).is_file()
                 or (cache_local / path.name).stat().st_size != path.stat().st_size)
    # Existing completed copies do not need another 30 GB of free space.
    if shutil.disk_usage(local).free < needed + 5 * 10**9:
        raise RuntimeError("Not enough local disk for three full shards and checkpoints")
    receipt, frames = {}, []
    for shard, manifest in pairs:
        for source in (shard, manifest):
            print(f"Staging {source.name}: {source.stat().st_size / 1e9:.2f} GB", flush=True)
            receipt[source.name] = copy_verified(source, cache_local / source.name)
        array = np.load(cache_local / shard.name, mmap_mode="r")
        frame = pd.read_parquet(cache_local / manifest.name)
        if array.dtype != np.uint8 or array.shape != (len(frame), 24, 224, 224):
            raise RuntimeError(f"Wrong cache geometry or manifest length: {shard.name} {array.shape}")
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    if len(combined) != 24371 or not combined.SeriesInstanceUID.is_unique:
        raise RuntimeError("Cache must cover exactly 24,371 unique series")

    # Only fetch small metadata if Drive does not already contain it. Images
    # always come from the user's existing Drive shards.
    for name in ("train_labels.parquet", "side_predictions.parquet", "train.csv", "train_series.csv"):
        source = choose_metadata(root.rglob(name), name,
                                 labels_hash if name == "train_labels.parquet" else None)
        if source is None:
            import kagglehub
            from kagglehub.config import get_kaggle_credentials, set_kaggle_api_token
            if get_kaggle_credentials() is None:
                from getpass import getpass
                token = getpass("Kaggle API token for missing metadata (hidden): ").strip()
                if not token:
                    raise RuntimeError("Enable KAGGLE_API_TOKEN in Colab secrets or provide the token")
                set_kaggle_api_token(token)
                del token
            if name == "train_labels.parquet":
                directory = kagglehub.dataset_download("pssv27/knee-labels")
                candidates = list(Path(directory).rglob(name))
                if len(candidates) != 1:
                    raise RuntimeError("Ambiguous downloaded labels")
                source = candidates[0]
            elif name == "side_predictions.parquet":
                directory = kagglehub.notebook_output_download("pssv27/knee-side-classifier")
                candidates = list(Path(directory).rglob(name))
                if len(candidates) != 1:
                    raise RuntimeError("Ambiguous downloaded side predictions")
                source = candidates[0]
            else:
                source = Path(kagglehub.competition_download(
                    "rsna-knee-abnormality-detection", path=name))
        destination = local / "metadata" if name.endswith(".csv") else cache_local
        receipt[name] = copy_verified(source, destination / name)
    if receipt["train_labels.parquet"] != labels_hash:
        raise RuntimeError("Drive label file differs from the Kaggle experiment")
    (local / "input_receipt.json").write_text(json.dumps(receipt, indent=2))
    mount_inputs(local)
    print("All three unchanged full shards and metadata are staged locally", flush=True)


def mount_inputs(local_root, inputs_root="/content/knee_slice_mounts"):
    # The corrected training finder skips roots containing train.csv to avoid
    # walking raw DICOM. Keep competition CSVs in a separate mount.
    import glob
    local, inputs = Path(local_root), Path(inputs_root)
    inputs.mkdir(parents=True, exist_ok=True)
    for name in ("cache", "metadata"):
        target, source = inputs / f"slice-colab-{name}", local / name
        if not source.is_dir():
            raise RuntimeError(f"Missing staged inputs: {source}")
        if target.exists() or target.is_symlink():
            if target.resolve() != source.resolve():
                raise RuntimeError("Conflicting input mount; use a fresh runtime")
        else:
            target.symlink_to(source.resolve(), target_is_directory=True)
    for pattern, count in (("cache_train_series_*.npy", 3),
                           ("manifest_train_series_*.parquet", 3),
                           ("train_labels.parquet", 1), ("side_predictions.parquet", 1),
                           ("train.csv", 1), ("train_series.csv", 1)):
        if len(glob.glob(f"{inputs}/**/{pattern}", recursive=True)) != count:
            raise RuntimeError(f"Duplicate or incomplete mounted {pattern}; use a fresh runtime")


class ColabInputPaths(ast.NodeTransformer):
    def __init__(self, inputs_root):
        self.inputs_root = str(inputs_root)

    def visit_Constant(self, node):
        if isinstance(node.value, str) and (node.value == "/kaggle/input"
                                            or node.value.startswith("/kaggle/input/")):
            return ast.copy_location(ast.Constant(
                self.inputs_root + node.value[len("/kaggle/input"):]), node)
        return node


def relocate_inputs(source, inputs_root):
    tree = ColabInputPaths(inputs_root).visit(ast.parse(source))
    return ast.unparse(ast.fix_missing_locations(tree)) + "\n"


def relocated_pair(source, attempt, inputs_root="/content/knee_slice_mounts"):
    """Relocate both outer runner and embedded training sources to one attempt."""
    tree = ast.parse(source)
    for node in tree.body:
        if not isinstance(node, ast.Assign) or not isinstance(node.targets[0], ast.Name):
            continue
        name = node.targets[0].id
        if name == "TRAINING_SCRIPTS":
            scripts = ast.literal_eval(node.value)
            node.value = ast.parse(repr({k: relocate_inputs(relocate(v, attempt), inputs_root)
                                        for k, v in scripts.items()}), mode="eval").body
        elif name == "SMOKE_SOURCE":
            node.value = ast.Constant(relocate_inputs(
                relocate(ast.literal_eval(node.value), attempt), inputs_root))
    return relocate_inputs(relocate(ast.unparse(ast.fix_missing_locations(tree)), attempt), inputs_root)


def run_pair_on_drive(pair_source, drive_root, local_root):
    import importlib
    for package in ("pydicom", "cv2", "numpy", "pandas", "pyarrow", "sklearn",
                    "tqdm", "timm", "lightning.pytorch"):
        try:
            importlib.import_module(package)
        except ImportError as exc:
            raise RuntimeError(f"Training dependency {package} failed to import; "
                               "rerun the notebook's dependency setup cell") from exc
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Select exactly one Colab GPU before running")
    mount_inputs(local_root)
    root = Path(drive_root)
    mount = Path("/content/drive/MyDrive")
    if not mount.is_dir() or not root.resolve().is_relative_to(mount.resolve()):
        raise RuntimeError("Outputs must be inside mounted MyDrive")
    attempt = root / f"attempt_{uuid.uuid4().hex[:12]}"
    attempt.mkdir(parents=True)
    source = relocated_pair(pair_source, attempt)
    (attempt / "train_pair.py").write_text(source)
    shutil.copy2(Path(local_root) / "input_receipt.json", attempt / "input_receipt.json")
    (attempt / "colab_execution.json").write_text(json.dumps({
        "platform": "colab", "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "pair_sha256": hashlib.sha256(pair_source.encode()).hexdigest(),
        "runtime_script_sha256": hashlib.sha256(source.encode()).hexdigest(),
        "note": "Single GPU, effective batch 8; not bitwise equivalent to Kaggle DDP",
        "parent_timeout": None,
        "inputs_root": "/content/knee_slice_mounts",
    }, indent=2))
    print("Checkpoints, metrics, progress and logs:", attempt, flush=True)
    # Streaming wait has no training timeout. Interrupting this cell terminates
    # the child; an interrupted attempt stays on Drive and is not overwritten.
    run_process(attempt / "train_pair.py", attempt)
    progress = json.loads((attempt / "pair_progress.json").read_text())
    if progress["state"] != "complete":
        raise RuntimeError(f"Pair incomplete: {progress}")
    print("FOLD 2 PAIR COMPLETE:", attempt, flush=True)
    print((attempt / "comparison.json").read_text())
    return str(attempt)


In [ ]:
PAIR_SOURCE = '\ndef attached_files(pattern):\n    import pathlib\n    found = []\n    for root in sorted(pathlib.Path("/kaggle/input").iterdir()):\n        if root.name not in ("competitions", "rsna-knee-abnormality-detection"):\n            found.extend(root.rglob(pattern))\n    return found\n\nTRAINING_SCRIPTS = {\'baseline\': \'\\ndef attached_files(pattern):\\n    import pathlib\\n    found = []\\n    for root in sorted(pathlib.Path("/kaggle/input").iterdir()):\\n        if root.name not in ("competitions", "rsna-knee-abnormality-detection"):\\n            found.extend(root.rglob(pattern))\\n    return found\\n\\nimport hashlib, os, pathlib\\nEXPECTED_FOLD = 2\\nEXPECTED_VAL = 870\\nMATCH_CLASSWISE = False\\nMATCH_BACKBONE = False\\nMATCH_SLICE = True\\nBASELINE_CONTRACT_PATH = "/kaggle/working/baseline_fold2/run_contract.json"\\nos.environ["KNEE_EXPERIMENT"] = \\\'baseline\\\'\\nos.environ["PYTHONUNBUFFERED"] = "1"\\npaths = attached_files("train_labels.parquet")\\nassert len(paths) == 1, f"Expected one label file, found {paths}"\\nassert hashlib.sha256(paths[0].read_bytes()).hexdigest() == \\\'e54e46531aec6cd16131d70aa283c990542dfa17daa890ac54da663cf74569b0\\\', "Label file changed"\\nimport faulthandler\\nfaulthandler.enable()\\n# Keep fatal-signal diagnostics, but do not arm a background stack-dump timer.\\n# Fold 2 and fold 4 SIGSEGVs occurred inside scheduled dumps while training\\n# was progressing. Parent time limits are configured by the staging command.\\n\\n\\nimport os\\nimport subprocess\\nos.environ[\\\'PYTORCH_NVML_BASED_CUDA_CHECK\\\'] = \\\'1\\\'\\nimport torch\\nDEVICES = torch.cuda.device_count()\\nif DEVICES == 0:\\n    raise SystemExit(\\\'Settings -> Accelerator -> GPU T4 x2\\\')\\nprint(subprocess.run([\\\'nvidia-smi\\\', \\\'--query-gpu=name,memory.total\\\', \\\'--format=csv,noheader\\\'], capture_output=True, text=True).stdout.strip())\\nprint(\\\'devices:\\\', DEVICES, \\\'| cuda initialised:\\\', torch.cuda.is_initialized())\\nif DEVICES < 2:\\n    print(\\\'NOTE: one GPU visible. Settings -> Accelerator -> GPU T4 x2 roughly halves wall-clock. The run is correct either way -- ACCUM_STEPS below adapts to hold the effective batch constant.\\\')\\ntry:\\n    import lightning.pytorch as pl\\nexcept ImportError:\\n    try:\\n        import pytorch_lightning as pl\\n    except ImportError:\\n        raise SystemExit(\\\'pip install lightning, then re-run this cell\\\')\\nprint(\\\'lightning:\\\', pl.__version__)\\n\\nimport importlib, pathlib, sys\\nFILES = {\\\'src/__init__.py\\\': \\\'\\\', \\\'src/config.py\\\': \\\'"""Central configuration: label names, cache geometry, and environment paths.\\\\n\\\\nEverything here is imported by both the training pipeline and the submission\\\\nnotebook, so it must stay dependency-free apart from the standard library.\\\\n"""\\\\n\\\\nimport os\\\\nfrom pathlib import Path\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Labels\\\\n# ---------------------------------------------------------------------------\\\\n# Exact submission spelling. The apostrophe in "Baker\\\\\\\'s" and the spaces are\\\\n# part of the column names -- do not normalise them anywhere that touches a\\\\n# submission file.\\\\nLABELS = [\\\\n    "ACL",\\\\n    "MCL",\\\\n    "Medial Meniscus",\\\\n    "Lateral Meniscus",\\\\n    "Medial OA",\\\\n    "Lateral OA",\\\\n    "PF OA",\\\\n    "Effusion",\\\\n    "Synovitis",\\\\n    "Baker\\\\\\\'s",\\\\n    "Contusion",\\\\n    "Fracture",\\\\n]\\\\n\\\\nID_COL = "StudyInstanceUID"\\\\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\\\\n\\\\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\\\\n# any augmentation that mirrors the image must also swap these pairs.\\\\nLATERAL_LABEL_PAIRS = [\\\\n    ("Medial Meniscus", "Lateral Meniscus"),\\\\n    ("Medial OA", "Lateral OA"),\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cache geometry\\\\n# ---------------------------------------------------------------------------\\\\n# Kept as constants so a higher-resolution variant is a config change rather\\\\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\\\\n# x 24,371 series ~= 30 GB, which fits Kaggle\\\\\\\'s 200 GB private dataset limit.\\\\nCACHE_SIZE = 224          # in-plane resolution after resampling\\\\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\\\\nCACHE_DTYPE = "uint8"\\\\n\\\\n# Robust intensity normalisation percentiles. MRI has no standardised units,\\\\n# so every volume is scaled against its own distribution.\\\\nCLIP_PCT_LOW = 1.0\\\\nCLIP_PCT_HIGH = 99.0\\\\n\\\\n# Canonical laterality: left knees are mirrored to match right knees so the\\\\n# medial compartment always falls on the same side of the image.\\\\nCANONICAL_SIDE = "R"\\\\n\\\\n# Mirroring is applied at load time, not baked into the cache.\\\\n#\\\\n# The side tag is blank on most series in this dataset and the series\\\\n# descriptions are anonymised, so roughly 42% of series have no recoverable\\\\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\\\\n# would mean rebuilding all of it every time side detection improves -- and it\\\\n# is expected to improve, via study-level propagation and then a classifier\\\\n# trained on the series whose side is known. The cache therefore stores volumes\\\\n# in their acquired orientation with the side recorded alongside, and the data\\\\n# loader flips on read, which costs nothing.\\\\nMIRROR_IN_CACHE = False\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Paths\\\\n# ---------------------------------------------------------------------------\\\\nCOMPETITION = "rsna-knee-abnormality-detection"\\\\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\\\\n\\\\n\\\\n# What a competition data root must contain to be recognised.\\\\n_MARKERS = ("train_series", "test_series")\\\\n\\\\n\\\\ndef _detect_data_root() -> Path:\\\\n    """Locate the competition data across Kaggle, Colab, and local runs.\\\\n\\\\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\\\\\\\'s input directory is\\\\n    searched by content rather than by name: the mount is not reliably at\\\\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\\\\n    through to the local fallback, which then fails much later with a confusing\\\\n    path in the message.\\\\n    """\\\\n    env = os.environ.get("KNEE_DATA_ROOT")\\\\n    if env:\\\\n        return Path(env)\\\\n\\\\n    kaggle_input = Path("/kaggle/input")\\\\n    if kaggle_input.exists():\\\\n        exact = kaggle_input / COMPETITION\\\\n        if exact.exists():\\\\n            return exact\\\\n        # Search two levels: Kaggle mounts competition data under\\\\n        # /kaggle/input/competitions/<slug>/ as well as directly under\\\\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\\\\n        try:\\\\n            for child in sorted(kaggle_input.iterdir()):\\\\n                if not child.is_dir():\\\\n                    continue\\\\n                if any((child / m).exists() for m in _MARKERS):\\\\n                    return child\\\\n                for grandchild in sorted(child.iterdir()):\\\\n                    if grandchild.is_dir() and any(\\\\n                            (grandchild / m).exists() for m in _MARKERS):\\\\n                        return grandchild\\\\n        except OSError:\\\\n            pass\\\\n\\\\n    return PROJECT_ROOT / "data"\\\\n\\\\n\\\\nDATA_ROOT = _detect_data_root()\\\\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\\\\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\\\\n\\\\nTRAIN_CSV = META_DIR / "train.csv"\\\\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\\\\nTEST_CSV = META_DIR / "test.csv"\\\\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\\\\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\\\\n\\\\nINDEX_PARQUET = META_DIR / "index.parquet"\\\\n\\\\nON_KAGGLE = Path("/kaggle").exists()\\\\n\\\', \\\'src/data/__init__.py\\\': \\\'\\\', \\\'src/data/dicom_io.py\\\': \\\'"""DICOM reading, geometric slice ordering, and laterality resolution.\\\\n\\\\nTwo operations here are load-bearing and fail silently if done naively:\\\\n\\\\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\\\\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\\\\n   slice normal derived from ImageOrientationPatient, which is defined by the\\\\n   scanner geometry rather than by convention.\\\\n\\\\n2. **Laterality.** Four of the twelve labels are compartment-specific\\\\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\\\\n   on the opposite side of the frame from a right knee, so left knees must be\\\\n   mirrored into a canonical orientation before a model can learn the\\\\n   distinction.\\\\n"""\\\\n\\\\nimport re\\\\nfrom pathlib import Path\\\\nfrom typing import Iterable, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pydicom\\\\n\\\\n# Header fields pulled during the index scan. Kept explicit so the parquet\\\\n# schema is stable and the scan stays header-only.\\\\nHEADER_FIELDS = [\\\\n    "StudyInstanceUID",\\\\n    "SeriesInstanceUID",\\\\n    "SOPInstanceUID",\\\\n    "Rows",\\\\n    "Columns",\\\\n    "PixelSpacing",\\\\n    "ImagePositionPatient",\\\\n    "ImageOrientationPatient",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "ScanningSequence",\\\\n    "SequenceVariant",\\\\n    # (0018,0010). The one header that states contrast administration\\\\n    # outright, and the index built before 2026-08-31 does not carry it --\\\\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\\\\n    # which are anonymised away. Absence of a string is not absence of\\\\n    # contrast; absence of this tag is at least a direct question asked.\\\\n    "ContrastBolusAgent",\\\\n    "SeriesDescription",\\\\n    "ProtocolName",\\\\n    "BodyPartExamined",\\\\n    "Manufacturer",\\\\n    "ManufacturerModelName",\\\\n    "MagneticFieldStrength",\\\\n    "ImageLaterality",\\\\n    "Laterality",\\\\n    "InstanceNumber",\\\\n]\\\\n\\\\n# DICOM DS/IS values carry their original string representation, which would\\\\n# land in the index as object dtype. Cast these explicitly.\\\\nNUMERIC_FIELDS = {\\\\n    "Rows",\\\\n    "Columns",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "MagneticFieldStrength",\\\\n    "InstanceNumber",\\\\n}\\\\n\\\\n_LEFT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n_RIGHT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n\\\\n\\\\ndef read_header(path) -> dict:\\\\n    """Read one DICOM\\\\\\\'s headers without decoding pixel data."""\\\\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\\\\n    row = {"path": str(path)}\\\\n    for field in HEADER_FIELDS:\\\\n        value = getattr(ds, field, None)\\\\n        if value is None:\\\\n            row[field] = None\\\\n        elif isinstance(value, pydicom.multival.MultiValue):\\\\n            # Geometry fields are numeric; ScanningSequence and friends are\\\\n            # multi-valued strings, so fall back to a joined token.\\\\n            try:\\\\n                row[field] = [float(v) for v in value]\\\\n            except (TypeError, ValueError):\\\\n                row[field] = "\\\\\\\\\\\\\\\\".join(str(v) for v in value)\\\\n        elif field in NUMERIC_FIELDS:\\\\n            try:\\\\n                row[field] = float(value)\\\\n            except (TypeError, ValueError):\\\\n                row[field] = None\\\\n        elif isinstance(value, (int, float, str)):\\\\n            row[field] = str(value) if isinstance(value, str) else value\\\\n        else:\\\\n            row[field] = str(value)\\\\n    return row\\\\n\\\\n\\\\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\\\\n    """Unit normal of the image plane, from the two direction cosines.\\\\n\\\\n    Falls back to the z axis when orientation is missing, which keeps a\\\\n    malformed series ordered by patient-superior position rather than crashing.\\\\n    """\\\\n    if orientation is None or len(orientation) != 6:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n\\\\n    row_cosine = np.asarray(orientation[:3], dtype=float)\\\\n    col_cosine = np.asarray(orientation[3:], dtype=float)\\\\n    normal = np.cross(row_cosine, col_cosine)\\\\n\\\\n    magnitude = np.linalg.norm(normal)\\\\n    if magnitude == 0:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n    return normal / magnitude\\\\n\\\\n\\\\ndef slice_position(header: dict) -> float:\\\\n    """Scalar position of a slice along its own stack axis."""\\\\n    position = header.get("ImagePositionPatient")\\\\n    if position is None or len(position) != 3:\\\\n        # No geometry available; fall back to InstanceNumber so ordering is at\\\\n        # least deterministic. The index scan flags these series separately.\\\\n        number = header.get("InstanceNumber")\\\\n        return float(number) if number is not None else 0.0\\\\n\\\\n    normal = slice_normal(header.get("ImageOrientationPatient"))\\\\n    return float(np.dot(np.asarray(position, dtype=float), normal))\\\\n\\\\n\\\\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\\\\n    """Order a series\\\\\\\' slices along the acquisition axis."""\\\\n    return sorted(headers, key=slice_position)\\\\n\\\\n\\\\ndef has_geometry(headers: Sequence[dict]) -> bool:\\\\n    """True when every slice carries the position and orientation tags."""\\\\n    for header in headers:\\\\n        position = header.get("ImagePositionPatient")\\\\n        orientation = header.get("ImageOrientationPatient")\\\\n        if position is None or len(position) != 3:\\\\n            return False\\\\n        if orientation is None or len(orientation) != 6:\\\\n            return False\\\\n    return True\\\\n\\\\n\\\\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\\\\n    """Determine whether a series images a left or right knee.\\\\n\\\\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\\\\n    description fields, which carry side markers in several languages. Returns\\\\n    "L", "R", or None when the side cannot be established -- callers must treat\\\\n    None as unusable rather than guessing, since a wrong side silently swaps the\\\\n    medial and lateral labels.\\\\n    """\\\\n    for header in headers:\\\\n        for field in ("ImageLaterality", "Laterality"):\\\\n            value = header.get(field)\\\\n            if isinstance(value, str):\\\\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\\\\n                # often present-but-empty than absent. An empty string is not a\\\\n                # side and must not be read as one.\\\\n                token = value.strip().upper()\\\\n                if token in ("L", "LEFT"):\\\\n                    return "L"\\\\n                if token in ("R", "RIGHT"):\\\\n                    return "R"\\\\n\\\\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\\\\n        for header in headers:\\\\n            text = header.get(field)\\\\n            if not isinstance(text, str):\\\\n                continue\\\\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\\\\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\\\\n            left = bool(_LEFT_PATTERN.search(spaced))\\\\n            right = bool(_RIGHT_PATTERN.search(spaced))\\\\n            # Ambiguous strings mentioning both sides tell us nothing.\\\\n            if left and not right:\\\\n                return "L"\\\\n            if right and not left:\\\\n                return "R"\\\\n\\\\n    return None\\\\n\\\\n\\\\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\\\\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\\\\n\\\\n    Applies the rescale slope and intercept when present. Slices whose in-plane\\\\n    shape differs from the first are skipped rather than silently broadcast.\\\\n    """\\\\n    slices = []\\\\n    reference_shape = None\\\\n    skipped = 0\\\\n\\\\n    for path in paths:\\\\n        try:\\\\n            ds = pydicom.dcmread(str(path), force=True)\\\\n            pixels = ds.pixel_array.astype(np.float32)\\\\n        except Exception:  # noqa: BLE001\\\\n            # A truncated or malformed file loses one slice, not the series.\\\\n            # The dataset contains files whose declared pixel length exceeds\\\\n            # what is actually stored, and dropping 23 good slices over one\\\\n            # bad one costs a whole study its imaging.\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\\\\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\\\\n        if slope != 1.0 or intercept != 0.0:\\\\n            pixels = pixels * slope + intercept\\\\n\\\\n        if reference_shape is None:\\\\n            reference_shape = pixels.shape\\\\n        elif pixels.shape != reference_shape:\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slices.append(pixels)\\\\n\\\\n    if not slices:\\\\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\\\\n\\\\n    return np.stack(slices, axis=0)\\\\n\\\\n\\\\ndef series_paths(series_dir) -> List[str]:\\\\n    """All DICOM files under one series directory."""\\\\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\\\\n\\\\n\\\\ndef resolve_study_laterality(series_sides: dict) -> dict:\\\\n    """Fill in unknown series sides from other series in the same study.\\\\n\\\\n    A knee MRI study images one knee, so a side established on any series\\\\n    applies to all of them. This matters because the DICOM side tags are\\\\n    populated inconsistently across the contributing institutions -- a study\\\\n    can carry the tag on its coronal series and omit it on the sagittals.\\\\n\\\\n    Report text is deliberately not used as a source here. Reports exist only\\\\n    for training studies, so recovering a side from them would mirror the\\\\n    training volumes and leave the test volumes untouched -- a train/test skew\\\\n    that is worse than the missing side it fixes.\\\\n\\\\n    Args:\\\\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\\\\n\\\\n    Returns:\\\\n        The same structure with unknown entries filled where the study allows.\\\\n        Studies whose series disagree are left untouched, since a conflict\\\\n        means the assumption of one knee per study does not hold there.\\\\n    """\\\\n    resolved = {}\\\\n    for study_uid, series in series_sides.items():\\\\n        known = {side for side in series.values() if side in ("L", "R")}\\\\n\\\\n        if len(known) == 1:\\\\n            side = known.pop()\\\\n            resolved[study_uid] = {uid: side for uid in series}\\\\n        else:\\\\n            # No evidence, or contradictory evidence. Guessing would silently\\\\n            # swap the medial and lateral labels on four of twelve findings.\\\\n            resolved[study_uid] = dict(series)\\\\n\\\\n    return resolved\\\\n\\\\n\\\\ndef laterality_coverage(series_sides: dict) -> dict:\\\\n    """Counts before and after study-level propagation, for auditing."""\\\\n    before = sum(1 for s in series_sides.values() for v in s.values()\\\\n                 if v in ("L", "R"))\\\\n    total = sum(len(s) for s in series_sides.values())\\\\n\\\\n    filled = resolve_study_laterality(series_sides)\\\\n    after = sum(1 for s in filled.values() for v in s.values()\\\\n                if v in ("L", "R"))\\\\n\\\\n    conflicts = sum(\\\\n        1 for s in series_sides.values()\\\\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\\\\n    )\\\\n\\\\n    return {\\\\n        "total_series": total,\\\\n        "known_before": before,\\\\n        "known_after": after,\\\\n        "recovered": after - before,\\\\n        "conflicting_studies": conflicts,\\\\n    }\\\\n\\\', \\\'src/data/cache.py\\\': \\\'"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\\\\n\\\\nThe same code path runs during cache construction and at submission time. Any\\\\ndivergence between the two produces a train/test preprocessing skew, which is\\\\ninvisible in local validation and only shows up as a collapsed leaderboard\\\\nscore -- so callers must import from here rather than reimplementing.\\\\n\\\\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\\\\norientation it was acquired in. Mirroring to a canonical side happens at load\\\\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport cv2\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\\\\n    """Scale a volume to uint8 using its own robust percentile range.\\\\n\\\\n    MRI carries no standardised units -- intensity depends on scanner, field\\\\n    strength, coil, and whether fat suppression was applied -- so each volume is\\\\n    scaled against its own distribution rather than a global constant.\\\\n    """\\\\n    finite = volume[np.isfinite(volume)]\\\\n    if finite.size == 0:\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\\\\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\\\\n        # Flat or degenerate volume; nothing to stretch.\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\\\\n    return (scaled * 255.0).round().astype(np.uint8)\\\\n\\\\n\\\\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\\\\n    """Resample every slice to size x size with area/linear interpolation."""\\\\n    size = size or config.CACHE_SIZE\\\\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\\\\n\\\\n    for i, plane in enumerate(volume):\\\\n        # Area interpolation avoids the aliasing that bilinear introduces when\\\\n        # downsampling 800px acquisitions; linear is correct when upsampling.\\\\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\\\\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\\\\n\\\\n    return output\\\\n\\\\n\\\\ndef select_slices(count: int, target: int = None) -> List[int]:\\\\n    """Indices of `target` slices spread evenly across a stack of `count`.\\\\n\\\\n    Short series are handled by repeating edge slices rather than zero-padding,\\\\n    so the volume stays anatomically continuous.\\\\n    """\\\\n    target = target or config.CACHE_SLICES\\\\n    if count <= 0:\\\\n        raise ValueError("cannot select slices from an empty series")\\\\n    if count == target:\\\\n        return list(range(count))\\\\n\\\\n    positions = np.linspace(0, count - 1, target)\\\\n    return [int(round(p)) for p in positions]\\\\n\\\\n\\\\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\\\\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\\\\n\\\\n    Mirroring a knee means reflecting through the patient\\\\\\\'s left-right axis,\\\\n    and which array axis that corresponds to depends on the acquisition plane:\\\\n\\\\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\\\\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\\\\n\\\\n    A blanket column flip is wrong for sagittal series. It leaves medial and\\\\n    lateral exactly where they were -- they are separated across slices, not\\\\n    across the image -- and instead reflects anterior-posterior, moving the\\\\n    patella to the back of the knee.\\\\n\\\\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\\\\n    """\\\\n    left_right = np.array([1.0, 0.0, 0.0])\\\\n\\\\n    if orientation is None or len(orientation) != 6:\\\\n        # No geometry to reason from. Columns is the common case (coronal and\\\\n        # axial together outnumber sagittal), but the caller should prefer not\\\\n        # to mirror at all when geometry is missing.\\\\n        return 2\\\\n\\\\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\\\\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\\\\n    normal = dicom_io.slice_normal(orientation)\\\\n\\\\n    alignments = [\\\\n        (abs(float(normal @ left_right)), 0),\\\\n        (abs(float(height @ left_right)), 1),\\\\n        (abs(float(width @ left_right)), 2),\\\\n    ]\\\\n    return max(alignments)[1]\\\\n\\\\n\\\\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\\\\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\\\\n    """Mirror left knees so the medial compartment is always on one side.\\\\n\\\\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\\\\n    Lateral OA). Without this the model would have to infer the side before it\\\\n    could place a finding, from a cue that is not reliably in the image.\\\\n\\\\n    The axis reflected is chosen from the acquisition geometry -- see\\\\n    mirror_axis -- because it differs between sagittal and coronal series.\\\\n    """\\\\n    if side is None:\\\\n        # Unknown side: leave untouched and let the caller decide. Guessing\\\\n        # would silently swap four of the twelve labels.\\\\n        return volume\\\\n    if side.upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = mirror_axis(orientation)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef submission_indices(count: int, slices: int) -> List[int]:\\\\n    """Raw slice indices a `slices`-deep submission actually reads.\\\\n\\\\n    **Not** `select_slices(count, slices)`, and the difference is the whole\\\\n    reason this exists. The submission decodes `CACHE_SLICES` and then\\\\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\\\\n    `slices` evenly across the raw stack in one step lands on *different\\\\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\\\\n    then be scored on anatomy it was not trained to read, and nothing would\\\\n    raise -- the shapes agree either way. That is the same failure the\\\\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\\\\n\\\\n    So the two selections are composed rather than collapsed: the coarse pick\\\\n    the cache would have made, then the fine pick the model applies to it.\\\\n    """\\\\n    coarse = select_slices(count)\\\\n    if slices == config.CACHE_SLICES:\\\\n        return coarse\\\\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\\\\n\\\\n\\\\ndef build_series_volume(headers: Sequence[dict],\\\\n                        slices: Optional[int] = None) -> Dict:\\\\n    """Full pipeline for one series: sort, decode, normalise, resize.\\\\n\\\\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\\\\n    not by default. The recorded `laterality` is what the loader mirrors on.\\\\n    Returns the volume plus the provenance needed to audit it later.\\\\n\\\\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\\\\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\\\\n    decoding done for images the model never sees, and inference time is a\\\\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\\\\n    construction is unchanged and every existing shard stays valid.\\\\n\\\\n    **It is not bit-identical to decoding the full depth.**\\\\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\\\\n    computed over whatever was decoded, so a shallower stack shifts the\\\\n    intensity mapping slightly. The slices themselves are the same files --\\\\n    see `submission_indices` -- but the uint8 values differ a little, and the\\\\n    submission notebook measures that difference rather than assuming it away.\\\\n    """\\\\n    if not headers:\\\\n        raise ValueError("no headers supplied")\\\\n\\\\n    depth = config.CACHE_SLICES if slices is None else slices\\\\n    ordered = dicom_io.sort_slices(headers)\\\\n    first_header = ordered[0]\\\\n    side = dicom_io.resolve_laterality(ordered)\\\\n    geometry_ok = dicom_io.has_geometry(ordered)\\\\n\\\\n    indices = submission_indices(len(ordered), depth)\\\\n    selected = [ordered[i] for i in indices]\\\\n\\\\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\\\\n\\\\n    # Slices skipped as unreadable leave the volume short of the requested\\\\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\\\\n    # decoded so the depth is exact regardless of how many files were dropped.\\\\n    if raw.shape[0] != depth:\\\\n        raw = raw[select_slices(raw.shape[0], depth)]\\\\n\\\\n    resized = resize_volume(raw.astype(np.float32))\\\\n    normalized = normalize_intensity(resized)\\\\n\\\\n    # Left as acquired by default; the loader mirrors using the recorded side.\\\\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\\\\n    volume = (canonicalize_side(normalized, side,\\\\n                                first_header.get("ImageOrientationPatient"))\\\\n              if config.MIRROR_IN_CACHE else normalized)\\\\n\\\\n    first = ordered[0]\\\\n    return {\\\\n        "volume": volume,\\\\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\\\\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\\\\n        "n_slices_raw": len(ordered),\\\\n        "laterality": side,\\\\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\\\\n                         and side.upper() != config.CANONICAL_SIDE),\\\\n        "geometry_ok": geometry_ok,\\\\n        "rows": int(first.get("Rows") or 0),\\\\n        "cols": int(first.get("Columns") or 0),\\\\n    }\\\\n\\\\n\\\\ndef build_series_record(item):\\\\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\\\\n\\\\n    Defined at module level deliberately. A worker defined inside a notebook\\\\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\\\\n    spawn start method and survives under fork only by accident. Keeping it\\\\n    importable makes the cache build work the same way everywhere.\\\\n\\\\n    Failures are returned rather than raised: one unreadable series must not\\\\n    end a ninety-minute run, and the manifest records what to investigate.\\\\n    """\\\\n    series_uid, headers = item\\\\n    try:\\\\n        result = build_series_volume(headers)\\\\n        return series_uid, result["volume"], {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": result["StudyInstanceUID"],\\\\n            "n_slices_raw": result["n_slices_raw"],\\\\n            "laterality": result["laterality"],\\\\n            "mirrored": result["mirrored"],\\\\n            "geometry_ok": result["geometry_ok"],\\\\n            "rows": result["rows"],\\\\n            "cols": result["cols"],\\\\n            "error": None,\\\\n        }\\\\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\\\\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\\\\n        return series_uid, None, {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": study_uid,\\\\n            "n_slices_raw": len(headers) if headers else 0,\\\\n            "laterality": None,\\\\n            "mirrored": False,\\\\n            "geometry_ok": False,\\\\n            "rows": 0,\\\\n            "cols": 0,\\\\n            "error": str(exc)[:200],\\\\n        }\\\\n\\\', \\\'src/data/index.py\\\': \\\'"""Header-only scan of the DICOM tree into a single parquet index.\\\\n\\\\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\\\\nthe cost is dominated by file-system traversal rather than decoding. The\\\\nresulting index drives cache construction and lets series be filtered and\\\\naudited without touching the 570 GB of imaging again.\\\\n\\\\nUsage (in a Kaggle notebook):\\\\n\\\\n    from src.data.index import build_index\\\\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\\\\n"""\\\\n\\\\nimport argparse\\\\nimport os\\\\nfrom concurrent.futures import ThreadPoolExecutor\\\\nfrom pathlib import Path\\\\nfrom typing import Iterator, List, Optional\\\\n\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef iter_dicom_paths(root) -> Iterator[Path]:\\\\n    """Yield every .dcm under root/study/series/ without materialising a list."""\\\\n    root = Path(root)\\\\n    for study_dir in sorted(root.iterdir()):\\\\n        if not study_dir.is_dir():\\\\n            continue\\\\n        for series_dir in sorted(study_dir.iterdir()):\\\\n            if not series_dir.is_dir():\\\\n                continue\\\\n            for path in sorted(series_dir.glob("*.dcm")):\\\\n                yield path\\\\n\\\\n\\\\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\\\\n    """Read headers for a batch of files in parallel.\\\\n\\\\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\\\\n    released during file reads and a process pool would only add overhead.\\\\n    """\\\\n    def safe_read(path):\\\\n        try:\\\\n            return dicom_io.read_header(path)\\\\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\\\\n            return {"path": str(path), "error": str(exc)}\\\\n\\\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\\\n        rows = list(pool.map(safe_read, paths))\\\\n\\\\n    return pd.DataFrame(rows)\\\\n\\\\n\\\\ndef build_index(\\\\n    split: str = "train_series",\\\\n    out_path: Optional[str] = None,\\\\n    data_root=None,\\\\n    workers: int = 8,\\\\n    batch_size: int = 5000,\\\\n    limit: Optional[int] = None,\\\\n) -> pd.DataFrame:\\\\n    """Scan one split\\\\\\\'s DICOM tree and write a parquet index.\\\\n\\\\n    Processes in batches so a long scan reports progress and so peak memory\\\\n    stays flat regardless of how many files the split holds.\\\\n    """\\\\n    root = Path(data_root or config.DATA_ROOT) / split\\\\n    if not root.exists():\\\\n        raise FileNotFoundError(f"{root} does not exist")\\\\n\\\\n    try:\\\\n        from tqdm.auto import tqdm\\\\n    except ImportError:  # progress is a convenience, never a requirement\\\\n        tqdm = None\\\\n\\\\n    frames = []\\\\n    batch: List[Path] = []\\\\n    scanned = 0\\\\n\\\\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\\\\n    # counts files without a total rather than forcing a full listing first,\\\\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\\\\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\\\\n\\\\n    for path in iter_dicom_paths(root):\\\\n        batch.append(path)\\\\n        if limit is not None and len(batch) + scanned >= limit:\\\\n            break\\\\n        if len(batch) >= batch_size:\\\\n            frames.append(scan_paths(batch, workers=workers))\\\\n            scanned += len(batch)\\\\n            if progress is not None:\\\\n                progress.update(len(batch))\\\\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\\\\n            else:\\\\n                print(f"  scanned {scanned} files", flush=True)\\\\n            batch = []\\\\n\\\\n    if batch:\\\\n        frames.append(scan_paths(batch, workers=workers))\\\\n        scanned += len(batch)\\\\n        if progress is not None:\\\\n            progress.update(len(batch))\\\\n\\\\n    if progress is not None:\\\\n        progress.close()\\\\n\\\\n    if not frames:\\\\n        raise RuntimeError(f"no DICOM files found under {root}")\\\\n\\\\n    index = pd.concat(frames, ignore_index=True)\\\\n    print(f"scanned {scanned} files across {index[\\\\\\\'SeriesInstanceUID\\\\\\\'].nunique()} series")\\\\n\\\\n    if "error" in index.columns:\\\\n        failures = int(index["error"].notna().sum())\\\\n        if failures:\\\\n            print(f"WARNING: {failures} files failed to parse")\\\\n\\\\n    if out_path:\\\\n        os.makedirs(Path(out_path).parent, exist_ok=True)\\\\n        index.to_parquet(out_path, index=False)\\\\n        print(f"wrote {out_path}")\\\\n\\\\n    return index\\\\n\\\\n\\\\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\\\\n    """Per-series roll-up: slice count, geometry completeness, laterality.\\\\n\\\\n    Surfaces the series that need attention -- missing geometry, unresolvable\\\\n    side -- before an hour of cache construction is spent on them.\\\\n    """\\\\n    records = []\\\\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\\\\n        headers = group.to_dict("records")\\\\n        records.append({\\\\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "n_slices": len(headers),\\\\n            "geometry_ok": dicom_io.has_geometry(headers),\\\\n            "laterality": dicom_io.resolve_laterality(headers),\\\\n            "rows": headers[0].get("Rows"),\\\\n            "cols": headers[0].get("Columns"),\\\\n        })\\\\n\\\\n    return pd.DataFrame(records)\\\\n\\\\n\\\\ndef main():\\\\n    parser = argparse.ArgumentParser(description=__doc__)\\\\n    parser.add_argument("--split", default="train_series")\\\\n    parser.add_argument("--out", default=None)\\\\n    parser.add_argument("--workers", type=int, default=8)\\\\n    parser.add_argument("--limit", type=int, default=None)\\\\n    args = parser.parse_args()\\\\n\\\\n    build_index(split=args.split, out_path=args.out,\\\\n                workers=args.workers, limit=args.limit)\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/__init__.py\\\': \\\'\\\', \\\'src/model/side_classifier.py\\\': \\\'"""Predict whether a cached series images a left or right knee.\\\\n\\\\nThe DICOM side tag is blank on ~41% of series and the descriptions are\\\\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\\\\ncompartment-specific, and placing them requires knowing the side, so the\\\\nremainder is predicted from the imaging itself.\\\\n\\\\nThis trains on the 14,403 series whose side is known and applies to the rest.\\\\nBoth halves come from the same cache, so it behaves identically at submission\\\\ntime -- unlike report text, which exists only for training studies.\\\\n\\\\n**Why the whole ordered stack rather than one representative slice.** For\\\\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\\\\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\\\\nthe left-right axis IS the slice axis, so individual slices look much the same\\\\neither way -- what differs is the direction of travel through the volume, which\\\\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\\\\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\\\\nchannels, which preserves it.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:  # the module is importable without torch for testing\\\\n    torch = None\\\\n    nn = object\\\\n\\\\nfrom src import config\\\\n\\\\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\\\\n# so half resolution is ample and quadruples throughput.\\\\nINPUT_SIZE = 112\\\\n\\\\nSIDES = ["L", "R"]\\\\n\\\\n\\\\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\\\\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\\\\n\\\\n    Slices become channels, keeping their order, so the network can use the\\\\n    medial-to-lateral direction that distinguishes sides on sagittal series.\\\\n    """\\\\n    import cv2\\\\n\\\\n    if volumes.ndim == 3:\\\\n        volumes = volumes[None]\\\\n\\\\n    count, depth, _, _ = volumes.shape\\\\n    output = np.empty((count, depth, size, size), dtype=np.float32)\\\\n\\\\n    for i in range(count):\\\\n        for j in range(depth):\\\\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\\\\n                                      interpolation=cv2.INTER_AREA)\\\\n\\\\n    return output / 255.0\\\\n\\\\n\\\\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\\\\n    """Mirror a prepared batch along each item\\\\\\\'s medial-lateral axis.\\\\n\\\\n    Used for augmentation: mirroring an input flips its label, which doubles\\\\n    the training set and forces the model to key on anatomy rather than on any\\\\n    incidental left/right imbalance in the source institutions.\\\\n\\\\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\\\\n    in the prepared batch slices are dimension 1 and columns dimension 3.\\\\n    """\\\\n    output = np.empty_like(batch)\\\\n    for i, axis in enumerate(plane_axis):\\\\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\\\\n    return output\\\\n\\\\n\\\\nif torch is None:\\\\n    # Importable without torch so the array handling above can be tested on a\\\\n    # machine that never trains. The network itself needs the real thing.\\\\n    SideNet = None\\\\nelse:\\\\n  class SideNet(nn.Module):\\\\n      """A small convolutional classifier over slice-channel input.\\\\n\\\\n      Trained from scratch deliberately. It is a coarse two-class problem with\\\\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\\\\n      competition\\\\\\\'s stance on pretrained weights is still unconfirmed, which this\\\\n      piece of the pipeline should not depend on.\\\\n      """\\\\n\\\\n      def __init__(self, in_channels: int = None, width: int = 32):\\\\n          super().__init__()\\\\n          in_channels = in_channels or config.CACHE_SLICES\\\\n\\\\n          def block(inp, out, stride=2):\\\\n              return nn.Sequential(\\\\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n              )\\\\n\\\\n          self.features = nn.Sequential(\\\\n              block(in_channels, width),        # 112 -> 56\\\\n              block(width, width * 2),          # 56 -> 28\\\\n              block(width * 2, width * 4),      # 28 -> 14\\\\n              block(width * 4, width * 4),      # 14 -> 7\\\\n          )\\\\n          self.head = nn.Sequential(\\\\n              nn.AdaptiveAvgPool2d(1),\\\\n              nn.Flatten(),\\\\n              nn.Dropout(0.2),\\\\n              nn.Linear(width * 4, 1),\\\\n          )\\\\n\\\\n      def forward(self, x):\\\\n          return self.head(self.features(x)).squeeze(-1)\\\\n\\\\n\\\\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\\\\n    """Collapse per-series predictions to one side per study.\\\\n\\\\n    A study is one knee, so its series must agree. Voting across them is\\\\n    strictly more reliable than any single series -- and the observed\\\\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\\\\n\\\\n    Returns {study_uid: "L" | "R"}.\\\\n    """\\\\n    if len(predictions) != len(study_uids):\\\\n        raise ValueError(\\\\n            f"predictions and study_uids differ in length: "\\\\n            f"{len(predictions)} vs {len(study_uids)}")\\\\n    if confidences is None:\\\\n        confidences = np.ones(len(predictions))\\\\n    elif len(confidences) != len(predictions):\\\\n        # zip would truncate silently, shifting weights onto the wrong series.\\\\n        raise ValueError(\\\\n            f"confidences length {len(confidences)} does not match "\\\\n            f"{len(predictions)} predictions")\\\\n\\\\n    scores = {}\\\\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\\\\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\\\\n\\\\n    result = {}\\\\n    for study, entries in scores.items():\\\\n        total = sum(weight for _, weight in entries)\\\\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\\\\n        result[study] = "R" if mean >= 0.5 else "L"\\\\n    return result\\\\n\\\\n\\\\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\\\\n    """Fraction correct, with predictions as probabilities of the right knee."""\\\\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\\\\n\\\', \\\'src/model/embed.py\\\': \\\'"""Frozen DINOv2 slice embeddings.\\\\n\\\\nRuns the backbone once over every cached slice and stores the result, so the\\\\naggregation stack trains on cached features in minutes rather than hours. The\\\\nsame code runs at submission time, which is the point -- a divergence between\\\\nhow features are computed for training and for inference is invisible locally\\\\nand fatal on the leaderboard.\\\\n\\\\n**Why frozen, and why first.** The competition scores a second prize track on\\\\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\\\\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\\\\nthe aggregation design -- which is where the modelling actually happens -- be\\\\niterated dozens of times a day instead of three.\\\\n\\\\n**What is stored per slice.** The CLS token concatenated with the patch tokens\\\\nreduced by `POOL_GRID`:\\\\n\\\\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\\\\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\\\\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\\\\n\\\\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\\\\n\\\\nThe first model used grid 1, and its weakness appeared exactly where the mean\\\\npredicts one: regional findings (effusion, OA, Baker\\\\\\\'s) averaged 0.865 on gold\\\\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\\\\ntear occupies two or three patches out of 256, so a mean over all of them\\\\nburies it in 250-odd patches of normal tissue.\\\\n\\\\nWide embeddings must be projected before the aggregation stack -- see\\\\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\\\\nin it against 8.2M with a projection.\\\\n"""\\\\n\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:\\\\n    torch = None\\\\n    nn = None\\\\n\\\\nfrom src import config\\\\n\\\\n# The registers variant. Registers exist to absorb the high-norm artefact\\\\n# tokens that otherwise contaminate patch features -- precisely what gets\\\\n# pooled here.\\\\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\\\\n\\\\n# timm\\\\\\\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\\\\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\\\\n# position embeddings are interpolated rather than silently mismatched.\\\\nINPUT_SIZE = config.CACHE_SIZE\\\\n\\\\n# DINOv2-S token width.\\\\nTOKEN_DIM = 384\\\\n\\\\n# How the 16x16 patch grid is reduced before storage.\\\\n#\\\\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\\\\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\\\\n#\\\\n# The mean is what the first model used, and its weakness showed up exactly\\\\n# where predicted: regional findings averaged 0.865 on gold while focal ones\\\\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\\\\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\\\\n# attention pooling to find it.\\\\nPOOL_GRID = 1\\\\n\\\\n\\\\ndef embed_dim(grid: int = None) -> int:\\\\n    """Stored width per slice: CLS plus the pooled patch tokens."""\\\\n    grid = POOL_GRID if grid is None else grid\\\\n    tokens = 1 + (grid * grid if grid > 1 else 1)\\\\n    return TOKEN_DIM * tokens\\\\n\\\\n\\\\nEMBED_DIM = embed_dim()\\\\n\\\\n# ImageNet statistics, which DINOv2 was trained under.\\\\nMEAN = (0.485, 0.456, 0.406)\\\\nSTD = (0.229, 0.224, 0.225)\\\\n\\\\n\\\\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\\\\n                  checkpoint_path: Optional[str] = None):\\\\n    """A frozen DINOv2 feature extractor in eval mode.\\\\n\\\\n    `checkpoint_path` loads weights from disk, which the submission notebook\\\\n    needs: internet is disabled there, so timm cannot fetch anything.\\\\n    """\\\\n    import timm\\\\n\\\\n    model = timm.create_model(\\\\n        model_name,\\\\n        pretrained=pretrained and checkpoint_path is None,\\\\n        img_size=INPUT_SIZE,\\\\n        num_classes=0,\\\\n    )\\\\n\\\\n    if checkpoint_path:\\\\n        state = torch.load(checkpoint_path, map_location="cpu")\\\\n        state = state.get("state_dict", state)\\\\n        model.load_state_dict(state, strict=False)\\\\n\\\\n    model.eval()\\\\n    for parameter in model.parameters():\\\\n        parameter.requires_grad = False\\\\n    return model\\\\n\\\\n\\\\ndef normalize(slices):\\\\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\\\\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\\\\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\\\\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\\\\n    return (x - mean) / std\\\\n\\\\n\\\\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\\\\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\\\\n\\\\n    The patch sequence is reshaped back to its square grid before pooling, so\\\\n    neighbouring patches are averaged with their neighbours rather than with\\\\n    whatever happened to sit beside them in a flattened sequence.\\\\n    """\\\\n    import torch.nn.functional as F\\\\n\\\\n    count, n_patches, channels = patches.shape\\\\n    side = int(round(n_patches ** 0.5))\\\\n    if side * side != n_patches:\\\\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\\\\n        return patches.mean(dim=1)\\\\n\\\\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\\\\n    pooled = F.adaptive_avg_pool2d(square, grid)\\\\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\\\\n\\\\n\\\\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\\\\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\\\\n\\\\n    Uses forward_features so the patch tokens are reachable at all; the model\\\\\\\'s\\\\n    default pooled output discards them.\\\\n    """\\\\n    grid = POOL_GRID if grid is None else grid\\\\n\\\\n    with torch.no_grad():\\\\n        tokens = encoder.forward_features(normalize(slices))\\\\n\\\\n        if tokens.ndim == 2:\\\\n            # A backbone that already pools; repeat so the width is predictable.\\\\n            repeats = embed_dim(grid) // tokens.shape[-1]\\\\n            return tokens.repeat(1, repeats)\\\\n\\\\n        cls = tokens[:, 0]\\\\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\\\\n        patches = tokens[:, prefix:]\\\\n\\\\n        if grid > 1:\\\\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\\\\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\\\\n\\\\n\\\\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\\\\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\\\\n\\\\n    KneeNet expects a module mapping slices to features. When features are\\\\n    already computed, the "backbone" is an identity that reshapes -- which\\\\n    keeps one model definition for both the cached and the end-to-end path.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim: int = EMBED_DIM):\\\\n        super().__init__()\\\\n        self.feature_dim = feature_dim\\\\n\\\\n    def forward(self, x):\\\\n        return x\\\\n\\\', \\\'src/model/dataset.py\\\': \\\'"""Study-level dataset over the cached volumes.\\\\n\\\\nJoins four sources into one training sample per study:\\\\n\\\\n    cache shards      the uint8 volumes, memory-mapped\\\\n    manifest          which shard row holds which series\\\\n    fused labels      12 soft targets and 12 sample weights per study\\\\n    side predictions  L/R per series, from the tag or the classifier\\\\n\\\\nA sample is a whole study, not a series. The twelve findings are study-level\\\\nlabels, and a single series often cannot support them -- effusion reads off a\\\\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\\\\nhas to see the study\\\\\\\'s series together.\\\\n\\\\n**Laterality is handled two ways, depending on the path.**\\\\n\\\\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\\\\nside at load time, along the axis the acquisition plane dictates: slice order\\\\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\\\\nand axial, where it lies in-plane. Doing it at load time rather than in the\\\\ncache means side detection can improve without rebuilding 29 GB.\\\\n\\\\nFor the frozen path this is not available -- the embeddings are computed from\\\\nthe volume, so the image can no longer be flipped. There the side travels to\\\\nthe model as a covariate (`is_right`, `side_known`) instead. That still\\\\naddresses the original concern, which was that the model should not have to\\\\n*infer* the side from cues that are not reliably present; it is handed the\\\\nanswer either way. Which of the two performs better is an experiment, not a\\\\nsettled question.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\\\\nPLANE_MIRROR_AXIS = {\\\\n    "Sagittal": 0,   # medial-lateral is the slice axis\\\\n    "Coronal": 2,    # medial-lateral lies in-plane\\\\n    "Axial": 2,\\\\n}\\\\nDEFAULT_MIRROR_AXIS = 2\\\\n\\\\n# Series metadata fed to the model as conditioning. Findings are\\\\n# sequence-specific, and these three are given in both train and test.\\\\n# `is_right` and `side_known` carry laterality as a covariate rather than\\\\n# normalising it away by mirroring. With precomputed frozen embeddings the\\\\n# image can no longer be flipped -- the features are already computed -- so the\\\\n# side is handed to the model instead, which addresses the original concern\\\\n# (the model should not have to *infer* the side) by a different route.\\\\n# Mirroring remains available for the end-to-end path and is an A/B to run,\\\\n# not a settled question.\\\\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\\\\n                "is_coronal", "is_axial", "is_right", "side_known"]\\\\n\\\\n\\\\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\\\\n                  plane: Optional[str]) -> np.ndarray:\\\\n    """Reflect a left knee onto the canonical side, along the correct axis."""\\\\n    if not isinstance(side, str) or side.strip().upper() not in ("L", "R"):\\\\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\\\\n        # on four of the twelve findings.\\\\n        return volume\\\\n    if side.strip().upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef series_metadata(row: pd.Series) -> np.ndarray:\\\\n    """Conditioning vector for one series.\\\\n\\\\n    Fluid sensitivity and fat suppression are kept separate: the data\\\\n    description notes they are often correlated but not equivalent, so\\\\n    collapsing them would discard the cases where they disagree.\\\\n    """\\\\n    plane = str(row.get("Anatomical_Plane", ""))\\\\n    side = row.get("side_predicted")\\\\n    side = side.strip().upper() if isinstance(side, str) else None\\\\n    known = side in ("L", "R")\\\\n    return np.array([\\\\n        sequence_flag(row.get("Fluid_Sensitive")),\\\\n        sequence_flag(row.get("Fat_Suppression")),\\\\n        float(plane == "Sagittal"),\\\\n        float(plane == "Coronal"),\\\\n        float(plane == "Axial"),\\\\n        float(known and side == "R"),\\\\n        float(known),\\\\n    ], dtype=np.float32)\\\\n\\\\n\\\\ndef sequence_flag(value) -> float:\\\\n    """Parse CSV flags without treating NaN or the string \\\\\\\'False\\\\\\\' as true."""\\\\n    if value is None or pd.isna(value):\\\\n        return 0.0\\\\n    if isinstance(value, str):\\\\n        value = value.strip().lower()\\\\n        if value in ("true", "yes", "1", "1.0"):\\\\n            return 1.0\\\\n        if value in ("false", "no", "0", "0.0", ""):\\\\n            return 0.0\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    if value not in (0, 1):\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    return float(value)\\\\n\\\\n\\\\ndef select_series(rows: pd.DataFrame, maximum=None, strategy="uniform", rng=None):\\\\n    """Positions to retain, preserving order and optionally protocol coverage.\\\\n\\\\n    Protocol selection first covers acquisition planes, then distinct\\\\n    (plane, fluid sensitivity, fat suppression) tuples, then repeats. It\\\\n    uses only metadata available at inference, never labels or reports.\\\\n    """\\\\n    if strategy not in ("uniform", "protocol"):\\\\n        raise ValueError(f"unknown series selection: {strategy}")\\\\n    if maximum is not None and maximum < 1:\\\\n        raise ValueError("max_series must be positive or None")\\\\n    count = len(rows)\\\\n    if maximum is None or count <= maximum:\\\\n        return np.arange(count)\\\\n    if strategy == "uniform":\\\\n        chosen = (np.linspace(0, count - 1, maximum).round().astype(int)\\\\n                  if rng is None else rng.choice(count, maximum, replace=False))\\\\n        return np.sort(chosen)\\\\n    protocols = [(str(row.get("Anatomical_Plane", "")),\\\\n                  sequence_flag(row.get("Fluid_Sensitive")),\\\\n                  sequence_flag(row.get("Fat_Suppression")))\\\\n                 for _, row in rows.iterrows()]\\\\n    planes, covered, chosen = set(), set(), []\\\\n    priority = (np.arange(count) if rng is None else rng.permutation(count))\\\\n    while len(chosen) < maximum:\\\\n        remaining = [int(i) for i in priority if i not in chosen]\\\\n        best = max(remaining, key=lambda i: (\\\\n            protocols[i][0] in PLANE_MIRROR_AXIS and protocols[i][0] not in planes,\\\\n            protocols[i] not in covered,\\\\n            min((abs(i - j) for j in chosen), default=0)))\\\\n        chosen.append(best)\\\\n        planes.add(protocols[best][0])\\\\n        covered.add(protocols[best])\\\\n    return np.array(sorted(chosen))\\\\n\\\\n\\\\nclass StudyIndex:\\\\n    """Everything needed to assemble a study, resolved once up front.\\\\n\\\\n    Built from frames rather than files so it can be constructed in a test\\\\n    without touching disk.\\\\n    """\\\\n\\\\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\\\\n                 labels: Optional[pd.DataFrame] = None,\\\\n                 sides: Optional[pd.DataFrame] = None,\\\\n                 max_series: Optional[int] = None,\\\\n                 reports: Optional[pd.DataFrame] = None,\\\\n                 series_selection: str = "uniform"):\\\\n        frame = manifest.merge(\\\\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\\\\n                         "Fat_Suppression", "Anatomical_Plane"]],\\\\n            on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n\\\\n        if sides is not None:\\\\n            frame = frame.merge(\\\\n                sides[["SeriesInstanceUID", "side_predicted"]],\\\\n                on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n        else:\\\\n            frame["side_predicted"] = frame.get("laterality_resolved")\\\\n\\\\n        # Drop series whose volume failed to build; their row in the cache is\\\\n        # zeros and would train the model on blank images.\\\\n        if "error" in frame.columns:\\\\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\\\\n\\\\n        if series_selection == "protocol":\\\\n            frame = frame.sort_values("SeriesInstanceUID", kind="stable").reset_index(drop=True)\\\\n        self.series = frame\\\\n        self.max_series = max_series\\\\n        self.series_selection = series_selection\\\\n        select_series(frame.iloc[:0], max_series, series_selection)\\\\n        self.labels = labels\\\\n\\\\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\\\\n        self._by_study = {\\\\n            study: group.reset_index(drop=True)\\\\n            for study, group in frame.groupby(config.ID_COL)\\\\n        }\\\\n\\\\n        if labels is not None:\\\\n            if labels[config.ID_COL].duplicated().any():\\\\n                raise ValueError("duplicate study IDs in labels")\\\\n            keyed = labels.set_index(config.ID_COL)\\\\n            self.studies = [s for s in self.studies if s in keyed.index]\\\\n            self._labels = keyed\\\\n        else:\\\\n            self._labels = None\\\\n\\\\n        # Report sentence embeddings, a training-time-only signal: reports\\\\n        # exist for train studies and never for test, so these feed an\\\\n        # auxiliary loss rather than the model input. Stored as one float32\\\\n        # matrix rather than a DataFrame so __getitem__ is an array row.\\\\n        if reports is not None:\\\\n            keyed_reports = reports.set_index(config.ID_COL)\\\\n            self._report_rows = {study: i for i, study\\\\n                                 in enumerate(keyed_reports.index)}\\\\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\\\\n            self.report_dim = self._report_matrix.shape[1]\\\\n        else:\\\\n            self._report_rows = None\\\\n            self._report_matrix = None\\\\n            self.report_dim = None\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def series_for(self, study: str, rng=None) -> pd.DataFrame:\\\\n        """Series of a study, capped deterministically when a cap is set.\\\\n\\\\n        The cap exists for the efficiency track: inference cost scales with\\\\n        series count, and a study with a dozen series contributes far more\\\\n        runtime than one with three. Selection is by position rather than at\\\\n        random so a prediction is reproducible.\\\\n\\\\n        `rng` is the training-only exception. The deterministic cap discards\\\\n        the same 3,013 of 24,371 series (12.4%) on every epoch of every fold --\\\\n        1,432 studies carry more series than `max_series` and always show the\\\\n        model the same subset. Passing a generator resamples the cap per epoch,\\\\n        which reaches those series without adding a byte to the cache. Order is\\\\n        preserved, so the series-axis metadata still lines up. Never pass one\\\\n        for validation or inference: a prediction that changes between two\\\\n        identical runs cannot be compared against anything.\\\\n        """\\\\n        rows = self._by_study[study]\\\\n        keep = select_series(rows, self.max_series, self.series_selection, rng)\\\\n        return rows.iloc[keep].reset_index(drop=True)\\\\n\\\\n    def targets_for(self, study: str):\\\\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\\\\n        if self._labels is None or study not in self._labels.index:\\\\n            return None, None\\\\n        row = self._labels.loc[study]\\\\n        targets = np.array([float(row[name]) for name in config.LABELS],\\\\n                           dtype=np.float32)\\\\n        weights = np.array(\\\\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\\\\n            dtype=np.float32)\\\\n        return targets, weights\\\\n\\\\n    def report_for(self, study: str) -> Optional[np.ndarray]:\\\\n        """The study\\\\\\\'s report embedding, or None when unavailable."""\\\\n        if self._report_rows is None or study not in self._report_rows:\\\\n            return None\\\\n        return self._report_matrix[self._report_rows[study]]\\\\n\\\\n\\\\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray],\\\\n               series_rng=None) -> Dict:\\\\n    """Assemble one study: mirrored volumes, conditioning, targets, weights.\\\\n\\\\n    `series_rng` is forwarded to `series_for` and is training-only; see there.\\\\n    """\\\\n    rows = index.series_for(study, rng=series_rng)\\\\n\\\\n    volumes, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        volume = np.asarray(shard[int(row["position"])])\\\\n        volume = mirror_volume(volume, row.get("side_predicted"),\\\\n                               row.get("Anatomical_Plane"))\\\\n        volumes.append(volume)\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(volumes) if volumes else np.zeros(\\\\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\\\\n            dtype=np.uint8),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef load_study_features(index: StudyIndex, study: str,\\\\n                        shards: Dict[str, np.ndarray]) -> Dict:\\\\n    """Assemble one study from precomputed slice embeddings.\\\\n\\\\n    Mirroring is unavailable here -- the features were computed from the image\\\\n    and cannot be flipped after the fact -- so laterality reaches the model\\\\n    through `series_metadata` instead.\\\\n    """\\\\n    rows = index.series_for(study)\\\\n\\\\n    features, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\\\\n                                                                dtype=np.float32),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\\\\n    """Pad a batch of studies to the largest series count, with a mask.\\\\n\\\\n    Series counts vary from one to a dozen, so the batch is padded and masked\\\\n    rather than truncated. The mask must reach the attention pooling: without\\\\n    it, padding rows are averaged in as if they were real series.\\\\n    """\\\\n    import torch\\\\n\\\\n    longest = max(len(item["volumes"]) for item in batch)\\\\n\\\\n    # Per-series shape is taken from the data rather than assumed, so the same\\\\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\\\\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\\\\n                     None)\\\\n    if reference is None:\\\\n        raise ValueError("every study in the batch is empty")\\\\n    per_series = reference.shape[1:]\\\\n    dtype = reference.dtype\\\\n\\\\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\\\\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\\\\n    mask = np.zeros((len(batch), longest), dtype=bool)\\\\n\\\\n    for i, item in enumerate(batch):\\\\n        count = len(item["volumes"])\\\\n        if count:\\\\n            volumes[i, :count] = item["volumes"]\\\\n            meta[i, :count] = item["meta"]\\\\n            mask[i, :count] = True\\\\n\\\\n    out = {\\\\n        config.ID_COL: [item[config.ID_COL] for item in batch],\\\\n        "volumes": torch.from_numpy(volumes),\\\\n        "meta": torch.from_numpy(meta),\\\\n        "mask": torch.from_numpy(mask),\\\\n    }\\\\n\\\\n    if batch[0]["targets"] is not None:\\\\n        out["targets"] = torch.from_numpy(\\\\n            np.stack([item["targets"] for item in batch]))\\\\n        out["weights"] = torch.from_numpy(\\\\n            np.stack([item["weights"] for item in batch]))\\\\n\\\\n    # Report embeddings, present only where the index carries them. Studies\\\\n    # without one get a zero row and a False mask so the auxiliary loss can\\\\n    # exclude them without a ragged batch.\\\\n    reports = [item.get("report") for item in batch]\\\\n    dims = [r.shape[0] for r in reports if r is not None]\\\\n    if dims:\\\\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\\\\n        report_mask = np.zeros(len(batch), dtype=bool)\\\\n        for i, row in enumerate(reports):\\\\n            if row is not None:\\\\n                report[i] = row\\\\n                report_mask[i] = True\\\\n        out["report"] = torch.from_numpy(report)\\\\n        out["report_mask"] = torch.from_numpy(report_mask)\\\\n    return out\\\\n\\\', \\\'src/model/net.py\\\': \\\'"""Study-level model: slices to series to study to twelve logits.\\\\n\\\\nThe backbone is deliberately the least interesting part. It is swappable, and\\\\nwhich one wins is settled by measurement rather than by argument -- at 4,349\\\\nstudies with mined labels, the aggregation above it matters more.\\\\n\\\\n    (B, S, D, H, W) uint8 volumes\\\\n      -> backbone over slices          (B*S*D, F)\\\\n      -> attention pool over slices    (B*S, F)\\\\n      -> FiLM on sequence metadata     (B*S, F)\\\\n      -> transformer over series       (B, F)      masked\\\\n      -> head                          (B, 12)\\\\n\\\\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\\\\nand Baker\\\\\\\'s cysts read off fluid-sensitive series, cartilage loss off\\\\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\\\\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\\\\nevery series through one undifferentiated path throws away free signal.\\\\n\\\\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\\\\nrows must be excluded from both the attention pool and the transformer, or a\\\\nshort study is diluted by rows of zeros that the model reads as real series.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport torch\\\\nimport torch.nn as nn\\\\nimport torch.nn.functional as F\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import META_COLUMNS\\\\n\\\\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\\\\n# backbone expects. Gives each slice a little through-plane context for free.\\\\nSLICE_CONTEXT = 3\\\\n\\\\n\\\\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\\\\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\\\\n\\\\n    Edge slices repeat, which is preferable to zero padding: a black plane at\\\\n    the edge of a volume is a feature the model would learn to key on.\\\\n    """\\\\n    n, depth, height, width = volumes.shape\\\\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\\\\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\\\\n    stacked = torch.stack([previous, volumes, following], dim=2)\\\\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\\\\n\\\\n\\\\n# The statistics DINOv2 was pretrained under. `embed.py` has always applied\\\\n# these on the frozen path; the end-to-end path fed raw [0,1] and ConvNeXt\\\\n# absorbed the shift into its first layer over twelve epochs. A pre-norm ViT\\\\n# is far less forgiving, so the same trunk was being fed a distribution it\\\\n# never saw during pretraining.\\\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\\\n\\\\n# NORMALISATION_BUFFERS -- why input_mean/input_std are persistent.\\\\n#\\\\n# They were registered with persistent=False on the argument that they are\\\\n# constants of the backbone family rather than learned state. That was true\\\\n# and it was still wrong: absent from the state dict, they have to be supplied\\\\n# again at every construction site, a site that forgets raises nothing, and a\\\\n# strict load restores every weight while leaving normalisation off. Two\\\\n# scoring sites forgot. The 24-slice DINOv2 run logged gold 0.864 from the\\\\n# in-memory model and 0.823 from the same weights reloaded -- an\\\\n# ImageNet-normalised trunk scored on raw [0,1].\\\\n#\\\\n# So they travel now, and `KneeNet._load_from_state_dict` handles both\\\\n# directions: a checkpoint that carries them imposes them on whatever the\\\\n# caller built, and one written before this change loads strictly without\\\\n# them. ConvNeXt is unaffected either way -- its defaults are None, so its\\\\n# state dict gains no keys and every existing reference number stands.\\\\n\\\\n# How far below the head\\\\\\\'s learning rate a pretrained trunk should sit. The\\\\n# optimiser puts every parameter in one group at OneCycle\\\\\\\'s peak of\\\\n# `lr * max_lr_multiplier` = 3e-4, which a supervised ConvNeXt survives and a\\\\n# self-supervised ViT does not: it loses its pretrained features during warmup\\\\n# and then relearns them from 4,349 studies, which is the one regime ViTs are\\\\n# worst in. Measured symptom: gold 0.6 at epoch 4 where ConvNeXt reached\\\\n# 0.729, and *below* the same trunk frozen (0.770).\\\\nVIT_BACKBONE_LR_SCALE = 0.1\\\\n\\\\n\\\\ndef backbone_defaults(name: str) -> dict:\\\\n    """Input normalisation and trunk learning-rate scale for a backbone family.\\\\n\\\\n    Returns ConvNeXt\\\\\\\'s existing behaviour unchanged -- no normalisation, one\\\\n    learning rate -- so the 0.861 fold-1 reference stays comparable. That\\\\n    includes ConvNeXt-V2, whose FCMAE weights are a drop-in for V1 at the same\\\\n    `timm` interface, and DINOv3\\\\\\\'s ConvNeXt variants: keeping the convnet path\\\\n    identical is what makes a backbone swap a one-variable experiment.\\\\n\\\\n    ViT-family trunks are the exception on both axes, and both were measured\\\\n    rather than argued -- see VIT_BACKBONE_LR_SCALE.\\\\n    """\\\\n    lowered = name.lower()\\\\n    is_vit = "vit" in lowered or ("dinov" in lowered and "convnext" not in lowered)\\\\n    if is_vit:\\\\n        return {"input_mean": IMAGENET_MEAN, "input_std": IMAGENET_STD,\\\\n                "backbone_lr_scale": VIT_BACKBONE_LR_SCALE}\\\\n    return {"input_mean": None, "input_std": None, "backbone_lr_scale": 1.0}\\\\n\\\\n\\\\ndef build_backbone(name: str, pretrained: bool = True,\\\\n                   image_size: Optional[int] = None):\\\\n    """Create a timm trunk mapping (N, 3, H, W) -> (N, num_features).\\\\n\\\\n    Every backbone this project runs is constructed through here so the\\\\n    training and submission notebooks cannot build different trunks from the\\\\n    same name -- notebook 10 hard-coded ConvNeXt for a while, which would have\\\\n    met a DINOv2 checkpoint with a shape error at submission time.\\\\n\\\\n    ViT-family weights (DINOv2 among them) are published at a fixed input size\\\\n    and timm\\\\\\\'s patch embedding refuses anything else, so the cache\\\\\\\'s size is\\\\n    passed explicitly and the position embeddings are interpolated -- 224 is\\\\n    16x14, which divides cleanly. ConvNets accept any size and reject the\\\\n    argument, hence the fallback rather than an unconditional pass.\\\\n    """\\\\n    import timm\\\\n\\\\n    image_size = config.CACHE_SIZE if image_size is None else image_size\\\\n    kwargs = dict(pretrained=pretrained, num_classes=0,\\\\n                  in_chans=SLICE_CONTEXT)\\\\n    try:\\\\n        return timm.create_model(name, img_size=image_size, **kwargs)\\\\n    except TypeError:\\\\n        return timm.create_model(name, **kwargs)\\\\n\\\\n\\\\nclass AttentionPool(nn.Module):\\\\n    """Masked attention pooling over a sequence of embeddings.\\\\n\\\\n    Preferred to mean pooling because the informative slices are a minority --\\\\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\\\\n    dilutes it by the rest. This is the attention-based MIL pooling of Ilse et\\\\n    al., "Attention-based Deep Multiple Instance Learning" (ICML 2018,\\\\n    arXiv:1802.04712), equation 8: a trainable weighted average, permutation\\\\n    invariant, with the weights produced by a two-layer scorer.\\\\n\\\\n    `gated=True` selects the same paper\\\\\\\'s equation 9, which replaces the single\\\\n    tanh branch with `tanh(Vh) * sigmoid(Uh)`. The stated motivation is that\\\\n    tanh is close to linear over most of its range, so it cannot express the\\\\n    non-linear "how relevant is this instance" function well; the sigmoid\\\\n    branch supplies a multiplicative gate that can suppress an instance\\\\n    outright. It is a strictly larger family -- a gate saturated at 1\\\\n    reproduces the ungated pool -- and it costs one extra `dim x hidden`\\\\n    matrix per pool.\\\\n\\\\n    The two variants produce different state dict keys on purpose. A gated\\\\n    checkpoint loaded into an ungated model raises on the missing keys rather\\\\n    than scoring quietly through the wrong pooling, which is the failure mode\\\\n    every other geometry mismatch in this project has had.\\\\n    """\\\\n\\\\n    def __init__(self, dim: int, hidden: int = 128, gated: bool = False):\\\\n        super().__init__()\\\\n        self.gated = gated\\\\n        if gated:\\\\n            self.attend = nn.Linear(dim, hidden)\\\\n            self.gate = nn.Linear(dim, hidden)\\\\n            self.weigh = nn.Linear(hidden, 1)\\\\n        else:\\\\n            self.score = nn.Sequential(\\\\n                nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\\\\n\\\\n    def scores(self, x: torch.Tensor) -> torch.Tensor:\\\\n        """(B, N, F) -> (B, N) unnormalised attention logits."""\\\\n        if not self.gated:\\\\n            return self.score(x).squeeze(-1)\\\\n        return self.weigh(torch.tanh(self.attend(x))\\\\n                          * torch.sigmoid(self.gate(x))).squeeze(-1)\\\\n\\\\n    def forward(self, x: torch.Tensor,\\\\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\\\\n        # x: (B, N, F), mask: (B, N) with True for real entries.\\\\n        weights = self.scores(x)\\\\n        if mask is not None:\\\\n            weights = weights.masked_fill(~mask, float("-inf"))\\\\n            weights = torch.where(mask.any(dim=1, keepdim=True), weights,\\\\n                                  torch.zeros_like(weights))\\\\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return (x * weights).sum(dim=1)\\\\n\\\\n\\\\nclass ClassAttentionPool(nn.Module):\\\\n    """Separate gated series attention for each finding, sharing the features."""\\\\n\\\\n    def __init__(self, dim: int, n_labels: int, hidden: int = 128):\\\\n        super().__init__()\\\\n        self.attend = nn.Linear(dim, hidden)\\\\n        self.gate = nn.Linear(dim, hidden)\\\\n        self.weigh = nn.Linear(hidden, n_labels)\\\\n\\\\n    def forward(self, x, mask=None):\\\\n        scores = self.weigh(torch.tanh(self.attend(x)) * torch.sigmoid(self.gate(x)))\\\\n        if mask is not None:\\\\n            scores = scores.masked_fill(~mask.unsqueeze(-1), float("-inf"))\\\\n            scores = torch.where(mask.any(dim=1)[:, None, None], scores,\\\\n                                 torch.zeros_like(scores))\\\\n        weights = scores.softmax(dim=1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return torch.einsum("bsl,bsf->blf", weights, x)\\\\n\\\\n\\\\ndef aggregation_defaults(state: dict) -> dict:\\\\n    """Recover the pooling variant from weights at every scoring site."""\\\\n    return {\\\\n        "slice_transformer": "slice_encoder.project.weight" in state,\\\\n        "gated_pool": "slice_pool.gate.weight" in state,\\\\n        "classwise_pool": ("study_pool.weigh.weight" in state and\\\\n                           state["study_pool.weigh.weight"].shape[0] > 1),\\\\n    }\\\\n\\\\n\\\\nclass FiLM(nn.Module):\\\\n    """Feature-wise modulation of series embeddings by sequence metadata."""\\\\n\\\\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\\\\n        super().__init__()\\\\n        self.to_scale_shift = nn.Sequential(\\\\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\\\\n            nn.Linear(dim, dim * 2))\\\\n\\\\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\\\\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\\\\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\\\\n        return x * (1.0 + scale) + shift\\\\n\\\\n\\\\nclass SliceSequenceEncoder(nn.Module):\\\\n    """Contextualize sampled slices before pooling; positions are relative, not mm.\\\\n\\\\n    A narrow residual branch preserves the existing feature interface. Its\\\\n    output starts at zero, so the new model initially reproduces the control.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim, width=256, layers=2, dropout=0.2):\\\\n        super().__init__()\\\\n        self.norm = nn.LayerNorm(feature_dim)\\\\n        self.project = nn.Linear(feature_dim, width)\\\\n        self.position = nn.Linear(1, width)\\\\n        # Construct layers independently (TransformerEncoder otherwise clones\\\\n        # one layer with identical initial values).\\\\n        self.layers = nn.ModuleList([\\\\n            nn.TransformerEncoderLayer(width, 8, width * 2, dropout,\\\\n                                       batch_first=True, norm_first=True)\\\\n            for _ in range(layers)])\\\\n        self.output = nn.Linear(width, feature_dim)\\\\n        nn.init.zeros_(self.output.weight)\\\\n        nn.init.zeros_(self.output.bias)\\\\n\\\\n    def forward(self, features):\\\\n        positions = torch.linspace(-1, 1, features.shape[1],\\\\n                                   device=features.device, dtype=features.dtype)\\\\n        x = self.project(self.norm(features)) + self.position(positions[:, None])[None]\\\\n        for layer in self.layers:\\\\n            x = layer(x)\\\\n        return features + self.output(x)\\\\n\\\\n\\\\nclass KneeNet(nn.Module):\\\\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\\\\n\\\\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\\\\n    a frozen encoder is supported and is the intended first configuration: with\\\\n    the trunk frozen, slice embeddings are computed once and the stack above\\\\n    trains in minutes rather than hours.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\\\\n                 n_labels: int = None, series_layers: int = 2,\\\\n                 n_heads: int = 8, dropout: float = 0.2,\\\\n                 freeze_backbone: bool = False,\\\\n                 input_dim: Optional[int] = None,\\\\n                 input_mean=None, input_std=None,\\\\n                 gated_pool: bool = False,\\\\n                 residual_pairs=None, residual_beta=None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        # None is valid on the frozen path, where the trunk ran offline.\\\\n        self.backbone = backbone if backbone is not None else nn.Identity()\\\\n        self.feature_dim = feature_dim\\\\n        self.classwise_pool = classwise_pool\\\\n        n_labels = n_labels or len(config.LABELS)\\\\n\\\\n        if freeze_backbone:\\\\n            for parameter in self.backbone.parameters():\\\\n                parameter.requires_grad = False\\\\n\\\\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\\\\n        # running the whole stack at that width would put ~85M parameters in\\\\n        # FiLM alone. A projection keeps the aggregation at a sane size.\\\\n        self.project = (nn.Linear(input_dim, feature_dim)\\\\n                        if input_dim and input_dim != feature_dim\\\\n                        else nn.Identity())\\\\n\\\\n        # These now travel in the state dict, and `_load_from_state_dict`\\\\n        # below reconciles the two directions with older files. See\\\\n        # NORMALISATION_BUFFERS for why the earlier persistent=False cost a\\\\n        # measured run.\\\\n        #\\\\n        # Registered as None rather than assigned as a plain attribute: a\\\\n        # plain attribute of the same name makes a later register_buffer\\\\n        # raise "attribute already exists", and adopting a checkpoint\\\\\\\'s\\\\n        # statistics is exactly that late registration.\\\\n        if input_mean is not None:\\\\n            self.register_buffer(\\\\n                "input_mean", torch.tensor(input_mean).view(1, -1, 1, 1))\\\\n            self.register_buffer(\\\\n                "input_std", torch.tensor(input_std).view(1, -1, 1, 1))\\\\n        else:\\\\n            self.register_buffer("input_mean", None)\\\\n            self.register_buffer("input_std", None)\\\\n\\\\n        self.slice_pool = AttentionPool(feature_dim, gated=gated_pool)\\\\n        self.film = FiLM(feature_dim)\\\\n\\\\n        layer = nn.TransformerEncoderLayer(\\\\n            d_model=feature_dim, nhead=n_heads,\\\\n            dim_feedforward=feature_dim * 2, dropout=dropout,\\\\n            batch_first=True, norm_first=True)\\\\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\\\\n        self.study_pool = (ClassAttentionPool(feature_dim, n_labels)\\\\n                           if classwise_pool else AttentionPool(feature_dim, gated=gated_pool))\\\\n\\\\n        self.head = nn.Sequential(\\\\n            nn.LayerNorm(feature_dim),\\\\n            nn.Dropout(dropout),\\\\n            nn.Linear(feature_dim, n_labels))\\\\n\\\\n        # Residual parameterisation: for each (child, parent) pair,\\\\n        #\\\\n        #     child_logit = beta * stopgrad(parent_logit) + g(features)\\\\n        #\\\\n        # with g the head\\\\\\\'s own child column. The plan\\\\\\\'s Stage 1: the\\\\n        # synovitis head reconstructs from the effusion head at R^2 0.55 with\\\\n        # a coefficient of +0.843, so effusion is most of what it computes.\\\\n        # Handing it that term for free, detached, leaves g nothing to learn\\\\n        # except the part effusion does not explain -- which is the part the\\\\n        # conditional metric measures.\\\\n        #\\\\n        # `beta` is calibrated on out-of-fold predictions, not derived from\\\\n        # r_true = 0.403: a rank correlation between two binary labels does\\\\n        # not map onto a logit coefficient. See\\\\n        # `src.model.conditional.calibrate_coupling`.\\\\n        #\\\\n        # Registered as None when unconfigured, exactly like the\\\\n        # normalisation buffers above and for the same reason -- a run that\\\\n        # does not use this writes an unchanged state dict, and a run that\\\\n        # does carries the coupling into the file so a bare\\\\n        # `KneeNet(backbone, dim)` at a scoring site reproduces it instead of\\\\n        # silently scoring an uncoupled model.\\\\n        if residual_pairs:\\\\n            pairs = torch.tensor(list(residual_pairs), dtype=torch.long)\\\\n            if pairs.ndim != 2 or pairs.shape[1] != 2:\\\\n                raise ValueError(\\\\n                    "residual_pairs must be (child, parent) index pairs, got "\\\\n                    f"shape {tuple(pairs.shape)}")\\\\n            if residual_beta is None:\\\\n                raise ValueError("residual_pairs given without residual_beta")\\\\n            values = (list(residual_beta) if hasattr(residual_beta, "__len__")\\\\n                      else [float(residual_beta)] * len(pairs))\\\\n            beta = torch.tensor(values, dtype=torch.float32)\\\\n            if len(beta) != len(pairs):\\\\n                raise ValueError(\\\\n                    f"residual_beta has {len(beta)} values for "\\\\n                    f"{len(pairs)} pairs")\\\\n            self.register_buffer("residual_pairs", pairs)\\\\n            self.register_buffer("residual_beta", beta)\\\\n        else:\\\\n            self.register_buffer("residual_pairs", None)\\\\n            self.register_buffer("residual_beta", None)\\\\n\\\\n        # Add after all existing modules and restore CPU RNG so unchanged\\\\n        # modules (including the Lightning wrapper) initialize identically.\\\\n        with torch.random.fork_rng(devices=[]):\\\\n            self.slice_encoder = (SliceSequenceEncoder(feature_dim, dropout=dropout)\\\\n                                  if slice_transformer else nn.Identity())\\\\n\\\\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\\\\n                              missing_keys, unexpected_keys, error_msgs):\\\\n        """Reconcile the normalisation buffers with checkpoints either side.\\\\n\\\\n        Three cases, and the middle one is the bug this exists for:\\\\n\\\\n        - **The file carries the statistics and this model has none.** Adopt\\\\n          them. A bare `KneeNet(backbone, dim)` built at a scoring site now\\\\n          normalises the way the run that wrote the file did, instead of\\\\n          feeding an ImageNet-normalised trunk raw [0,1] and reading as a\\\\n          worse checkpoint.\\\\n        - **The file predates this change and this model has statistics.**\\\\n          Keep what the caller passed. Every checkpoint written before\\\\n          2026-08-28 is missing the two keys, and a strict load must not\\\\n          start failing on files this project already shipped.\\\\n        - **The file carries statistics and this model has different ones.**\\\\n          The file wins, for the same reason as the first case: the weights\\\\n          and the statistics they were trained under belong together, and the\\\\n          construction site is the thing that has been wrong before.\\\\n\\\\n        Adopting is a `register_buffer` on a slot registered as None in\\\\n        `__init__`, which is why it is registered there rather than assigned.\\\\n        """\\\\n        for name in ("input_mean", "input_std",\\\\n                     "residual_pairs", "residual_beta"):\\\\n            key = prefix + name\\\\n            if key in state_dict:\\\\n                self.register_buffer(name, state_dict[key].clone())\\\\n            elif self._buffers.get(name) is not None:\\\\n                # Pre-fill so the strict load below sees no missing key.\\\\n                state_dict[key] = self._buffers[name]\\\\n        super()._load_from_state_dict(\\\\n            state_dict, prefix, local_metadata, strict, missing_keys,\\\\n            unexpected_keys, error_msgs)\\\\n\\\\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\\\\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\\\\n        count, depth = volumes.shape[0], volumes.shape[1]\\\\n        if volumes.dtype == torch.uint8:\\\\n            volumes = volumes.float().div_(255.0)\\\\n\\\\n        slices = stack_2p5d(volumes)\\\\n        if self.input_mean is not None:\\\\n            slices = (slices - self.input_mean) / self.input_std\\\\n        features = self.backbone(slices)\\\\n        features = features.reshape(count, depth, self.feature_dim)\\\\n        return self.slice_pool(self.slice_encoder(features))\\\\n\\\\n    def pool_study(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                   mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, F) one embedding per study.\\\\n\\\\n        The study representation the head reads. Exposed separately so\\\\n        auxiliary objectives (report-embedding regression) can supervise the\\\\n        representation without their parameters entering this module -- the\\\\n        exported state dict must stay loadable by a bare `KneeNet`.\\\\n        """\\\\n        series = self.film(series, meta)\\\\n\\\\n        # The transformer takes True for positions to IGNORE, the inverse of\\\\n        # our mask.\\\\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\\\\n        # A fully padded row emits NaN from the attention; zero it before\\\\n        # pooling so one short study cannot poison the batch.\\\\n        encoded = torch.nan_to_num(encoded)\\\\n\\\\n        return self.study_pool(encoded, mask=mask)\\\\n\\\\n    def couple(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Add the detached parent term to each configured child logit.\\\\n\\\\n        A no-op unless the model was built (or loaded) with residual pairs, so\\\\n        every checkpoint written before this existed behaves identically.\\\\n\\\\n        The parent term is detached: the point is to relieve the child of\\\\n        learning effusion, not to let the synovitis objective reach back and\\\\n        reshape the effusion head, which is the one head on this finding that\\\\n        is reliable.\\\\n        """\\\\n        if self.residual_pairs is None:\\\\n            return logits\\\\n        child = self.residual_pairs[:, 0]\\\\n        parent = self.residual_pairs[:, 1]\\\\n        addition = torch.zeros_like(logits)\\\\n        addition[:, child] = (self.residual_beta.to(logits.dtype)\\\\n                              * logits[:, parent].detach())\\\\n        return logits + addition\\\\n\\\\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                  mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, n_labels).\\\\n\\\\n        Everything above the backbone. Shared by the end-to-end path and the\\\\n        frozen path so the two cannot drift into different architectures.\\\\n        """\\\\n        return self.classify(self.pool_study(series, meta, mask))\\\\n\\\\n    def classify(self, pooled):\\\\n        logits = self.head(pooled)\\\\n        if self.classwise_pool:\\\\n            logits = logits.diagonal(dim1=1, dim2=2)\\\\n        return self.couple(logits)\\\\n\\\\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\\\\n                         mask: torch.Tensor) -> torch.Tensor:\\\\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\\\\n\\\\n        The backbone has already run, so this starts at the slice pooling.\\\\n        """\\\\n        batch, series, depth, _ = features.shape\\\\n        projected = self.project(features.reshape(batch * series, depth, -1))\\\\n        pooled = self.slice_pool(projected)\\\\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\\\\n                              meta, mask)\\\\n\\\\n    def forward_pooled(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                       mask: torch.Tensor):\\\\n        """End-to-end path returning (logits, pooled study embedding).\\\\n\\\\n        The training loop wants both: logits for the label losses, the pooled\\\\n        representation for auxiliary supervision. Computing them together\\\\n        avoids a second pass through the backbone.\\\\n        """\\\\n        batch, series = volumes.shape[0], volumes.shape[1]\\\\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\\\\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\\\\n        pooled = self.pool_study(embeddings, meta, mask)\\\\n        return self.classify(pooled), pooled\\\\n\\\\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                mask: torch.Tensor) -> torch.Tensor:\\\\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\\\\n        logits, _ = self.forward_pooled(volumes, meta, mask)\\\\n        return logits\\\\n\\\\n\\\\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\\\\n                 weights: torch.Tensor, label_smoothing: float = 0.0,\\\\n                 pos_weight: Optional[torch.Tensor] = None,\\\\n                 symmetric_beta: float = 0.0) -> torch.Tensor:\\\\n    """BCE against soft targets, weighted per study and finding.\\\\n\\\\n    Targets are probabilities rather than 0/1 -- they come from two extractors\\\\n    that disagree often enough that thresholding would teach the model to\\\\n    reproduce extraction errors. Weights carry how much each target is\\\\n    believed: full for agreement, less for conflict, least for silence.\\\\n\\\\n    Three optional terms, all inert at their defaults so the reference runs\\\\n    stay bit-comparable. Each addresses something the plan names:\\\\n\\\\n    `label_smoothing` pulls every target toward 0.5 by `eps`. The mined labels\\\\n    disagree with image-derived ground truth about 18% of the time, and a\\\\n    confident target that is wrong 18% of the time is what late training\\\\n    memorises. Note that soft targets already carry some of this effect --\\\\n    smoothing on top of them is a floor on the target\\\\\\\'s distance from 0.5, not\\\\n    the usual one-hot correction.\\\\n\\\\n    `pos_weight` scales the positive term per finding. The metric is macro\\\\n    AUC, so a rare finding costs exactly as much as a common one; without this\\\\n    the loss is dominated by whatever is prevalent. Passed straight through to\\\\n    torch, which applies it to soft targets correctly.\\\\n\\\\n    `symmetric_beta` adds the reverse cross-entropy term of Wang et al.,\\\\n    "Symmetric Cross Entropy for Robust Learning with Noisy Labels" (ICCV\\\\n    2019). RCE swaps the roles of prediction and target, which bounds the loss\\\\n    a wrong label can contribute: a confidently mislabelled study cannot drag\\\\n    the gradient the way plain CE lets it. Targets are clamped off 0 and 1\\\\n    because RCE takes their log; the clamp is what the paper\\\\\\\'s `A` constant\\\\n    does.\\\\n    """\\\\n    if label_smoothing > 0:\\\\n        targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing\\\\n\\\\n    loss = F.binary_cross_entropy_with_logits(\\\\n        logits, targets, pos_weight=pos_weight, reduction="none")\\\\n\\\\n    if symmetric_beta > 0:\\\\n        probs = torch.sigmoid(logits)\\\\n        clamped = targets.clamp(1e-4, 1.0 - 1e-4)\\\\n        reverse = -(probs * torch.log(clamped)\\\\n                    + (1.0 - probs) * torch.log1p(-clamped))\\\\n        loss = loss + symmetric_beta * reverse\\\\n\\\\n    total = weights.sum()\\\\n    if total <= 0:\\\\n        return loss.mean() * 0.0\\\\n    return (loss * weights).sum() / total\\\\n\\\\n\\\\ndef pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                       ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                       weights: Optional[torch.Tensor] = None,\\\\n                       ref_weights: Optional[torch.Tensor] = None,\\\\n                       min_gap: float = 0.25,\\\\n                       pair_mask: Optional[torch.Tensor] = None,\\\\n                       max_margin: Optional[float] = None) -> torch.Tensor:\\\\n    """Per-finding RankNet loss between a batch and a reference set.\\\\n\\\\n    The metric is macro AUC -- a ranking statistic -- while BCE optimises\\\\n    calibration; this term optimises the ordering directly. Pairs are formed\\\\n    per finding between the current batch (`scores`/`targets`, (B, L), with\\\\n    gradient) and a reference set ((Q, L), typically detached scores from\\\\n    recent steps), because at one study per rank there are no pairs *within*\\\\n    a batch to rank.\\\\n\\\\n    Only pairs whose soft targets differ by at least `min_gap` count: the\\\\n    targets are extractor probabilities, and ordering two studies the\\\\n    extractor itself barely distinguishes would train the model to reproduce\\\\n    extraction noise. Surviving pairs are weighted by the target gap and by\\\\n    both studies\\\\\\\' label weights, so a confident 0.9-vs-0.1 pair outweighs a\\\\n    marginal one. Returns a zero that still touches `scores` when no pair\\\\n    qualifies, so the graph is identical either way.\\\\n\\\\n    `pair_mask` drops pairs before weighting. Broadcastable to (B, Q, L), so\\\\n    it can restrict which findings are ranked, which pairs are comparable, or\\\\n    both. `stratified_rank_loss` below builds one; at None every pair counts\\\\n    and this is the loss the reference runs used, unchanged.\\\\n\\\\n    `max_margin` clamps the logit difference before the softplus, and exists\\\\n    because leaving it unbounded diverged a real run (2026-08-31). A satisfied\\\\n    pair\\\\\\\'s loss decays to zero as its margin grows, but a **violated** pair\\\\\\\'s\\\\n    grows linearly and forever, so when the pair constraints contradict each\\\\n    other -- which is what within-stratum synovitis pairs do, since 90% of the\\\\n    mined targets sit at zero and the surviving pairs are mostly "mentioned vs\\\\n    silent" -- inflating the logit scale reduces loss on the majority faster\\\\n    than it raises it on the minority. The head then saturates, every study\\\\n    ties, AUC lands on exactly 0.5, and the shared trunk goes with it: that run\\\\n    took BCE from 0.60 to 5.99 and macro from 0.86 to 0.71. Clamping caps what\\\\n    any one pair can demand, so a contradictory set cannot pay for scale.\\\\n    Default None leaves every earlier run bit-identical.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    if ref_scores.numel() == 0:\\\\n        return zero\\\\n\\\\n    gap = targets.unsqueeze(1) - ref_targets.unsqueeze(0)      # (B, Q, L)\\\\n    pair_weight = gap.abs() * (gap.abs() >= min_gap)\\\\n    if weights is not None and ref_weights is not None:\\\\n        pair_weight = (pair_weight * weights.unsqueeze(1)\\\\n                       * ref_weights.unsqueeze(0))\\\\n    if pair_mask is not None:\\\\n        pair_weight = pair_weight * pair_mask.to(pair_weight.dtype)\\\\n\\\\n    total = pair_weight.sum()\\\\n    if total <= 0:\\\\n        return zero\\\\n\\\\n    margin = scores.unsqueeze(1) - ref_scores.unsqueeze(0)     # (B, Q, L)\\\\n    if max_margin is not None:\\\\n        margin = margin.clamp(-max_margin, max_margin)\\\\n    loss = F.softplus(-torch.sign(gap) * margin)\\\\n    return (loss * pair_weight).sum() / total\\\\n\\\\n\\\\ndef finding_mask(findings: Sequence[str], n_labels: int = None,\\\\n                 device=None) -> torch.Tensor:\\\\n    """(1, 1, L) mask selecting the findings a loss is allowed to touch.\\\\n\\\\n    The stratified objective is for synovitis. Applying it to all twelve\\\\n    findings would trade away ranking on eleven that are already fine, on a\\\\n    stratification that means nothing for them.\\\\n    """\\\\n    n_labels = n_labels or len(config.LABELS)\\\\n    mask = torch.zeros(n_labels, dtype=torch.float32, device=device)\\\\n    for name in findings:\\\\n        mask[config.LABELS.index(name)] = 1.0\\\\n    return mask.view(1, 1, -1)\\\\n\\\\n\\\\ndef caliper_pairs(strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                  caliper: float) -> torch.Tensor:\\\\n    """(B, Q, 1) mask of pairs whose stratifier scores are within `caliper`.\\\\n\\\\n    Matching on a continuous score rather than splitting it at a threshold,\\\\n    because the mined effusion label carries its own extraction error and a\\\\n    hard split inherits it at the boundary. The predicted effusion score is\\\\n    continuous, scores AUC 0.888, and is available at every training step.\\\\n    """\\\\n    distance = (strata.unsqueeze(1) - ref_strata.unsqueeze(0)).abs()\\\\n    return (distance <= caliper).unsqueeze(-1)\\\\n\\\\n\\\\ndef stratified_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                         ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                         strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                         findings: Sequence[str], caliper: float = 0.15,\\\\n                         weights: Optional[torch.Tensor] = None,\\\\n                         ref_weights: Optional[torch.Tensor] = None,\\\\n                         min_gap: float = 0.25,\\\\n                         max_margin: Optional[float] = 8.0) -> torch.Tensor:\\\\n    """RankNet restricted to pairs from the same effusion stratum.\\\\n\\\\n    Conditional AUC *is* within-stratum pair concordance, so this is not an\\\\n    analogy to the metric -- it is the metric, differentiably. Pairs the\\\\n    conditional number ignores are the ones effusion already orders correctly,\\\\n    and training on them is what produced a synovitis head that reconstructs\\\\n    from the effusion head at R^2 0.55.\\\\n\\\\n    The known risk, stated in the plan and not solved here: stripping effusion\\\\n    removes the one reliable cue and leaves a harder task supervised by labels\\\\n    that are silent in 82.2% of reports against a true prevalence of 46.6%. If\\\\n    label quality binds, this loss makes the finding worse, and the conditional\\\\n    metric is what will show it.\\\\n\\\\n    `strata` is the detached predicted effusion probability for the current\\\\n    batch and `ref_strata` the same for the queued reference studies.\\\\n\\\\n    `max_margin` defaults to a finite 8.0 here rather than to None, because\\\\n    this is the loss whose pairs contradict each other -- see\\\\n    `pairwise_rank_loss`. A margin of 8 logits is a probability ratio of about\\\\n    3,000:1; a pair that wants more than that is not asking for an ordering.\\\\n    """\\\\n    mask = caliper_pairs(strata, ref_strata, caliper) * finding_mask(\\\\n        findings, scores.shape[-1], scores.device)\\\\n    return pairwise_rank_loss(scores, targets, ref_scores, ref_targets,\\\\n                              weights, ref_weights, min_gap=min_gap,\\\\n                              pair_mask=mask, max_margin=max_margin)\\\\n\\\\n\\\\ndef coupling_penalty(scores: torch.Tensor, ref_scores: torch.Tensor,\\\\n                     child: int, parent: int, target: float) -> torch.Tensor:\\\\n    """Push corr(child logit, parent logit) toward `target`, not toward zero.\\\\n\\\\n    The findings genuinely co-occur: on the gold set the odds ratio is 6.09\\\\n    (p = 0.003) and the label correlation is 0.403, against the model\\\\\\\'s 0.651.\\\\n    Driving the two heads to independence optimises toward a value the truth\\\\n    contradicts, so `target` is required and is meant to be the observed\\\\n    coupling rather than 0.\\\\n\\\\n    Batches here are one study per rank, so the correlation is computed over\\\\n    the current batch concatenated with the detached reference queue. Returns\\\\n    a zero that still touches `scores` when there are too few studies to\\\\n    estimate a correlation, so the graph does not change shape between steps.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    pool = (torch.cat([scores, ref_scores], dim=0)\\\\n            if ref_scores.numel() else scores)\\\\n    if pool.shape[0] < 8:\\\\n        return zero\\\\n\\\\n    a = pool[:, child] - pool[:, child].mean()\\\\n    b = pool[:, parent] - pool[:, parent].mean()\\\\n    denominator = a.norm() * b.norm()\\\\n    if denominator <= 0:\\\\n        return zero\\\\n    return ((a * b).sum() / denominator - target) ** 2\\\\n\\\\n\\\\ndef load_knee_state(path, map_location="cpu") -> dict:\\\\n    """Read a `KneeNet` state dict from either checkpoint format.\\\\n\\\\n    The end-to-end path writes two things over its life: a bare state dict\\\\n    from `torch.save(model.state_dict(), ...)`, and a Lightning checkpoint,\\\\n    which nests the same tensors under `state_dict` and prefixes every key\\\\n    with `model.` because `KneeNet` is an attribute of the wrapper. The\\\\n    submission must not care which it was handed -- requiring a manual\\\\n    conversion step is requiring a step that can be forgotten, and it was.\\\\n\\\\n    Lightning checkpoints also carry optimiser state and callback objects,\\\\n    which `weights_only=True` refuses, so that is attempted first and relaxed\\\\n    only on failure. These are files this project wrote.\\\\n    """\\\\n    try:\\\\n        obj = torch.load(path, map_location=map_location, weights_only=True)\\\\n    except TypeError:       # torch too old to know the argument\\\\n        obj = torch.load(path, map_location=map_location)\\\\n    except Exception:       # noqa: BLE001 - a Lightning checkpoint\\\\n        obj = torch.load(path, map_location=map_location, weights_only=False)\\\\n\\\\n    state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj\\\\n    if any(key.startswith("model.") for key in state):\\\\n        state = {key[len("model."):]: value for key, value in state.items()\\\\n                 if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(f"no weights found in {path}")\\\\n    return state\\\\n\\\', \\\'src/model/experiments.py\\\': \\\'"""Pre-registered, single-change experiments. No preset implies a score gain."""\\\\n\\\\nfrom dataclasses import asdict, dataclass, replace\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass Experiment:\\\\n    name: str = "baseline"\\\\n    backbone: str = "convnext_tiny.fb_in22k_ft_in1k"\\\\n    slices: int = 16\\\\n    max_series: int = 5\\\\n    series_selection: str = "uniform"\\\\n    resample: bool = False\\\\n    classwise_pool: bool = False\\\\n    slice_transformer: bool = False\\\\n\\\\n    def to_dict(self):\\\\n        result = asdict(self)\\\\n        # Preserve existing experiment contracts and checkpoint sidecars.\\\\n        if not self.slice_transformer:\\\\n            result.pop("slice_transformer")\\\\n        return result\\\\n\\\\n\\\\nBASELINE = Experiment()\\\\nPRESETS = {\\\\n    "baseline": BASELINE,\\\\n    "resample": replace(BASELINE, name="resample", resample=True),\\\\n    "protocol": replace(BASELINE, name="protocol", series_selection="protocol"),\\\\n    "classwise": replace(BASELINE, name="classwise", classwise_pool=True),\\\\n    "convnextv2": replace(BASELINE, name="convnextv2",\\\\n                          backbone="convnextv2_tiny.fcmae_ft_in22k_in1k"),\\\\n    "slice_transformer": replace(BASELINE, name="slice_transformer",\\\\n                                 series_selection="protocol", slice_transformer=True),\\\\n}\\\\n\\\\n\\\\ndef get_experiment(name):\\\\n    if name not in PRESETS:\\\\n        raise ValueError(f"unknown experiment {name!r}; choose {sorted(PRESETS)}")\\\\n    return PRESETS[name]\\\\n\\\', \\\'src/model/artifacts.py\\\': \\\'"""Explicit model provenance and inference for heterogeneous ensembles."""\\\\n\\\\nimport hashlib\\\\nimport json\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nimport torch\\\\n\\\\nfrom src import config\\\\nfrom src.model import dataset, net\\\\n\\\\n\\\\ndef sha256(path):\\\\n    digest = hashlib.sha256()\\\\n    with open(path, "rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(8 << 20), b""):\\\\n            digest.update(chunk)\\\\n    return digest.hexdigest()\\\\n\\\\n\\\\ndef write_metadata(checkpoint, experiment, fold, train_uids, val_uids):\\\\n    """Bind input geometry and split membership to the actual exported weights."""\\\\n    train_uids, val_uids = sorted(train_uids), sorted(val_uids)\\\\n    if set(train_uids) & set(val_uids):\\\\n        raise ValueError("training and validation studies overlap")\\\\n    payload = {\\\\n        "schema_version": 1, "labels": config.LABELS,\\\\n        "sha256": sha256(checkpoint), "experiment": experiment,\\\\n        "cache_size": config.CACHE_SIZE, "cache_slices": config.CACHE_SLICES,\\\\n        "fold": int(fold), "train_uids": train_uids, "val_uids": val_uids,\\\\n    }\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    path.write_text(json.dumps(payload, indent=2) + "\\\\\\\\n")\\\\n    return path\\\\n\\\\n\\\\ndef load_metadata(checkpoint):\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    data = json.loads(path.read_text())\\\\n    if data["schema_version"] != 1 or data["labels"] != config.LABELS:\\\\n        raise ValueError(f"unsupported schema or label order: {path}")\\\\n    if data["sha256"] != sha256(checkpoint):\\\\n        raise ValueError(f"checkpoint checksum mismatch: {checkpoint}")\\\\n    if data["cache_size"] != config.CACHE_SIZE or data["cache_slices"] != config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint requires a different source cache geometry")\\\\n    spec = data["experiment"]\\\\n    if not 1 <= spec["slices"] <= config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint slice count is outside the cache")\\\\n    dataset.select_series(pd.DataFrame(), spec["max_series"], spec["series_selection"])\\\\n    return data\\\\n\\\\n\\\\ndef load_ensemble(manifest_path, device="cpu"):\\\\n    """Load exactly the members named in JSON, never a directory-wide glob.\\\\n\\\\n    Format: {"members": [{"checkpoint": "run/fold0.pt", "weight": 1.0}]}.\\\\n    Paths are relative to this manifest. Each checkpoint requires its .pt.json\\\\n    sidecar, written by notebook 08. Weights are global, not per finding.\\\\n    """\\\\n    manifest_path = Path(manifest_path)\\\\n    entries = json.loads(manifest_path.read_text())["members"]\\\\n    if not entries:\\\\n        raise ValueError("ensemble has no members")\\\\n    members, digests, identities = [], set(), set()\\\\n    for entry in entries:\\\\n        path = (manifest_path.parent / entry["checkpoint"]).resolve()\\\\n        weight = float(entry.get("weight", 1.0))\\\\n        if not np.isfinite(weight) or weight <= 0:\\\\n            raise ValueError("ensemble weights must be finite and positive")\\\\n        metadata = load_metadata(path)\\\\n        identity = (metadata["experiment"]["name"], metadata["fold"])\\\\n        if metadata["sha256"] in digests or identity in identities:\\\\n            raise ValueError("duplicate checkpoint or experiment/fold in ensemble")\\\\n        digests.add(metadata["sha256"])\\\\n        identities.add(identity)\\\\n        spec = metadata["experiment"]\\\\n        state = net.load_knee_state(path)\\\\n        trunk = net.build_backbone(spec["backbone"], pretrained=False)\\\\n        defaults = net.backbone_defaults(spec["backbone"])\\\\n        model = net.KneeNet(trunk, trunk.num_features,\\\\n                            input_mean=defaults["input_mean"],\\\\n                            input_std=defaults["input_std"],\\\\n                            **net.aggregation_defaults(state))\\\\n        model.load_state_dict(state)\\\\n        members.append({"model": model.to(device).eval(), "spec": spec,\\\\n                        "weight": weight, "metadata": metadata})\\\\n    return members\\\\n\\\\n\\\\n@torch.inference_mode()\\\\ndef predict_members(members, volumes, rows, device="cpu"):\\\\n    """Predict from full acquired volumes and one aligned metadata row per series."""\\\\n    if len(rows) != len(volumes) or not len(rows):\\\\n        raise ValueError("volumes and metadata must be nonempty and aligned")\\\\n    order = np.argsort(rows["SeriesInstanceUID"].astype(str).to_numpy(), kind="stable")\\\\n    rows = rows.iloc[order].reset_index(drop=True)\\\\n    volumes = [volumes[i] for i in order]\\\\n    outputs = []\\\\n    for member in members:\\\\n        spec = member["spec"]\\\\n        chosen = dataset.select_series(rows, spec["max_series"], spec["series_selection"])\\\\n        picked = rows.iloc[chosen]\\\\n        stack = []\\\\n        for i, (_, row) in zip(chosen, picked.iterrows()):\\\\n            volume = dataset.mirror_volume(volumes[i], row.get("side_predicted"),\\\\n                                            row.get("Anatomical_Plane"))\\\\n            if volume.shape != (config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE):\\\\n                raise ValueError(f"unexpected volume geometry: {volume.shape}")\\\\n            keep = np.linspace(0, len(volume) - 1, spec["slices"]).round().astype(int)\\\\n            stack.append(volume[keep])\\\\n        tensor = torch.from_numpy(np.stack(stack)).unsqueeze(0).to(device)\\\\n        meta = torch.from_numpy(np.stack([\\\\n            dataset.series_metadata(row) for _, row in picked.iterrows()\\\\n        ])).unsqueeze(0).to(device)\\\\n        mask = torch.ones(1, len(stack), dtype=torch.bool, device=device)\\\\n        device_type = torch.device(device).type\\\\n        with torch.autocast(device_type, enabled=device_type == "cuda"):\\\\n            logits = member["model"](tensor, meta, mask)\\\\n        probs = logits.float().sigmoid().cpu().numpy()[0]\\\\n        if not np.isfinite(probs).all():\\\\n            raise ValueError("non-finite model prediction")\\\\n        outputs.append(probs)\\\\n    return np.stack(outputs)\\\\n\\\', \\\'src/model/conditional.py\\\': \\\'"""Effusion-stratified conditional AUC: the metric synovitis is judged on.\\\\n\\\\nPooled AUC cannot measure this finding. On the 58 gold studies the *gold\\\\neffusion label by itself* scores 0.698 as a synovitis classifier, against the\\\\nsynovitis head\\\\\\\'s 0.701 -- so a number in that region is obtainable with no\\\\nsynovitis-specific skill at all, and seven consecutive experiments read as\\\\nnull partly because they were scored on it.\\\\n\\\\nConditional AUC is the concordance of positive/negative pairs drawn from\\\\n**within the same effusion stratum**. Cross-stratum pairs, where effusion\\\\nalone orders the pair correctly, are dropped. What remains is the only\\\\nquantity on this gold set that moves when synovitis-specific skill moves:\\\\n\\\\n    pooled       0.701      837 pairs, ~55% of them cross-stratum\\\\n    conditional  0.630      376 within-stratum pairs\\\\n\\\\nThe two are linked by pair mass, which is why `pooled_from_conditional` can\\\\nstate the exact pooled number a conditional target implies on this set --\\\\nconditional 0.75 is pooled 0.755, and a *perfect* conditional score is pooled\\\\n0.867. Anything above that is unreachable without changing the gating.\\\\n\\\\nDeliberately free of torch and of pandas-at-import: this module is imported by\\\\nthe analysis tools, bootstrapped into the notebooks, and used by the training\\\\nloop\\\\\\\'s validation hook, and `src/model/lightning.py` cannot be imported by any\\\\nof the first two without dragging in the Lightning framework.\\\\n\\\\nTwo rules this module exists to enforce, from the plan\\\\\\\'s standing rules:\\\\n\\\\n1. Never judge synovitis on macro, and never on pooled alone.\\\\n2. Compare paired. Independent Hanley-McNeil standard errors are +/-0.07 here\\\\n   and carry between-study variance that a matched comparison removes; a real\\\\n   movement of 0.05 is invisible against them and visible against a paired\\\\n   bootstrap of the difference.\\\\n"""\\\\n\\\\nfrom typing import Dict, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Pooled AUC, and the noise band around it\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef auc(y_true, scores) -> float:\\\\n    """ROC AUC via the rank statistic, ties averaged. Matches the training loop."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n    return float((ranks[positives].sum() - n_pos * (n_pos + 1) / 2)\\\\n                 / (n_pos * n_neg))\\\\n\\\\n\\\\ndef auc_se(a: float, n_pos: int, n_neg: int) -> float:\\\\n    """Hanley & McNeil (1982) standard error of an AUC.\\\\n\\\\n    Printed beside pooled numbers so a difference inside the band is visibly\\\\n    not evidence. Never used to judge a *difference* between two models on the\\\\n    same studies -- that is what `paired_bootstrap` is for.\\\\n    """\\\\n    if not np.isfinite(a) or n_pos < 1 or n_neg < 1:\\\\n        return float("nan")\\\\n    q1 = a / (2.0 - a)\\\\n    q2 = 2.0 * a * a / (1.0 + a)\\\\n    numerator = (a * (1 - a) + (n_pos - 1) * (q1 - a * a)\\\\n                 + (n_neg - 1) * (q2 - a * a))\\\\n    return float(np.sqrt(max(numerator, 0.0) / (n_pos * n_neg)))\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Strata\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef strata_from_scores(effusion, threshold: Optional[float] = None):\\\\n    """Binary effusion strata from a continuous effusion score.\\\\n\\\\n    The plan stratifies on *predicted* effusion rather than on the mined\\\\n    effusion label, because the prediction scores AUC 0.888 while the mined\\\\n    label carries its own extraction error that would blur the strata. The\\\\n    default split is the median, which is what reproduces the plan\\\\\\\'s 0.617\\\\n    against the gold label\\\\\\\'s 0.630 -- close enough that the metric is\\\\n    computable during training, where no gold effusion label exists.\\\\n\\\\n    A median split rather than a fixed probability because the head is not\\\\n    calibrated across runs: a 0.70 cut that balances one model\\\\\\\'s studies\\\\n    puts 80% of another model\\\\\\\'s on one side, and the number would then move\\\\n    between runs for a reason that has nothing to do with synovitis.\\\\n    """\\\\n    effusion = np.asarray(effusion, dtype=float)\\\\n    if threshold is None:\\\\n        threshold = float(np.median(effusion))\\\\n    return (effusion >= threshold).astype(int)\\\\n\\\\n\\\\ndef comparable_pairs(y_true, strata, caliper: Optional[float] = None):\\\\n    """(n_pos, n_neg) boolean mask of pairs that share an effusion stratum.\\\\n\\\\n    With `caliper=None`, `strata` is treated as categorical and pairs must\\\\n    match exactly -- the gold-label and median-split cases. With a caliper,\\\\n    `strata` is a continuous score and a pair is comparable when the two\\\\n    studies\\\\\\\' effusion scores lie within `caliper` of each other, which is the\\\\n    matching rule the stratified training loss uses.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    left, right = strata[positives], strata[~positives]\\\\n    if caliper is None:\\\\n        return left[:, None] == right[None, :]\\\\n    return np.abs(left[:, None].astype(float)\\\\n                  - right[None, :].astype(float)) <= caliper\\\\n\\\\n\\\\ndef _concordance(y_true, scores):\\\\n    """(n_pos, n_neg) matrix of 1 / 0.5 / 0 per pair. Ties count as half."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    difference = scores[positives][:, None] - scores[~positives][None, :]\\\\n    return (difference > 0).astype(float) + 0.5 * (difference == 0)\\\\n\\\\n\\\\ndef conditional_auc(y_true, scores, strata,\\\\n                    caliper: Optional[float] = None) -> float:\\\\n    """Concordance over within-stratum pairs only. NaN if there are none.\\\\n\\\\n    Pair-weighted, not stratum-averaged: the strata are 286 and 90 pairs on\\\\n    the gold set, and averaging them would give the 5-positive stratum equal\\\\n    say with the 22-positive one. Pair weighting is also what makes the\\\\n    conversion to pooled AUC exact.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    total = mask.sum()\\\\n    if total == 0:\\\\n        return float("nan")\\\\n    return float((_concordance(y_true, scores) * mask).sum() / total)\\\\n\\\\n\\\\ndef pair_counts(y_true, strata, caliper: Optional[float] = None) -> Dict[str, int]:\\\\n    """How many pairs the conditional number rests on, and the total."""\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    return {"within": int(mask.sum()), "cross": int((~mask).sum()),\\\\n            "total": int(mask.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Where the error sits\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef pair_decomposition(y_true, scores, strata) -> list:\\\\n    """Every discordant pair, split by the stratum of each member.\\\\n\\\\n    The table in the plan\\\\\\\'s §1.4. Reading it is how the addressable headroom\\\\n    was located: within-stratum pairs are 44.9% of the total and are the only\\\\n    ones a synovitis-specific improvement can move. The cell where a\\\\n    stratum-0 positive meets a stratum-1 negative is ranked worse than chance\\\\n    -- that is the gating tax, and it is paid on five studies.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    concordance = _concordance(y_true, scores)\\\\n    error = 1.0 - concordance\\\\n    total_error = error.sum()\\\\n\\\\n    rows = []\\\\n    for pos_level in sorted(set(strata[positives].tolist()), reverse=True):\\\\n        for neg_level in sorted(set(strata[~positives].tolist()), reverse=True):\\\\n            mask = ((strata[positives] == pos_level)[:, None]\\\\n                    & (strata[~positives] == neg_level)[None, :])\\\\n            pairs = int(mask.sum())\\\\n            if pairs == 0:\\\\n                continue\\\\n            cell_error = float((error * mask).sum())\\\\n            rows.append({\\\\n                "pos_stratum": int(pos_level), "neg_stratum": int(neg_level),\\\\n                "within": pos_level == neg_level,\\\\n                "pairs": pairs,\\\\n                "misranked": cell_error / pairs,\\\\n                "error_share": cell_error / total_error if total_error else float("nan"),\\\\n            })\\\\n    return rows\\\\n\\\\n\\\\ndef pooled_from_conditional(y_true, scores, strata,\\\\n                            targets: Sequence[float]) -> list:\\\\n    """The pooled AUC each conditional target implies, on these studies.\\\\n\\\\n    Cross-stratum concordance is held at what this model already achieves and\\\\n    only the within-stratum mass is moved, so the answer is arithmetic rather\\\\n    than a fit. It exists to keep two conversations in the same units: the\\\\n    leaderboard and every historical table are pooled, the plan\\\\\\\'s targets are\\\\n    conditional, and quoting one as the other is how a 0.63 -> 0.75 move gets\\\\n    dismissed as "+0.05, inside the noise".\\\\n\\\\n    The value at `target=1.0` is the ceiling under an unchanged gating -- 0.867\\\\n    here. It is an arithmetic bound, not a reachable target: one model produces\\\\n    both halves, and a residualised model can buy within-stratum ordering by\\\\n    selling the gating.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata)\\\\n    concordance = _concordance(y_true, scores)\\\\n    within, cross = mask.sum(), (~mask).sum()\\\\n    cross_concordant = float((concordance * ~mask).sum())\\\\n    total = within + cross\\\\n    return [{"conditional": float(t),\\\\n             "pooled": float((cross_concordant + t * within) / total)}\\\\n            for t in targets]\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Uncertainty: paired, always\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef _resamples(n: int, draws: int, seed: int):\\\\n    rng = np.random.default_rng(seed)\\\\n    return rng.integers(0, n, size=(draws, n))\\\\n\\\\n\\\\ndef bootstrap_conditional(y_true, scores, strata, caliper=None,\\\\n                          draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """Percentile CI for one model\\\\\\\'s conditional AUC, plus P(AUC <= 0.5).\\\\n\\\\n    Studies are resampled, not pairs: a study appears in many pairs and\\\\n    resampling pairs would treat those as independent observations and report\\\\n    an interval far narrower than the evidence supports.\\\\n\\\\n    `p_at_or_below_chance` is the fraction of resamples at or under 0.5. It is\\\\n    the number the gate is read on, because "is there any synovitis-specific\\\\n    signal at all" is a question about which side of chance the estimate sits,\\\\n    not about whether it differs from some other model.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    strata = np.asarray(strata)\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = conditional_auc(y_true[index], scores[index], strata[index],\\\\n                                caliper)\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = conditional_auc(y_true, scores, strata, caliper)\\\\n    if values.size == 0:\\\\n        return {"auc": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "sd": float("nan"), "p_at_or_below_chance": float("nan"),\\\\n                "draws": 0}\\\\n    return {"auc": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "sd": float(values.std(ddof=1)),\\\\n            "p_at_or_below_chance": float((values <= 0.5).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\ndef paired_bootstrap(y_true, scores_a, scores_b, strata=None, caliper=None,\\\\n                     draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """CI for (A - B) on the same studies, conditional or pooled.\\\\n\\\\n    `strata=None` compares pooled AUCs; otherwise both models are scored on\\\\n    the same within-stratum pairs. Either way the *same* resample indexes both\\\\n    models, so between-study variance -- which is most of the +/-0.07 an\\\\n    unpaired standard error reports -- cancels.\\\\n\\\\n    `p_a_not_better` is the fraction of resamples where the difference is at\\\\n    or below zero. Read it as the evidence against A, and pre-register the\\\\n    threshold before running, not after seeing it.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores_a = np.asarray(scores_a, dtype=float)\\\\n    scores_b = np.asarray(scores_b, dtype=float)\\\\n    strata = None if strata is None else np.asarray(strata)\\\\n\\\\n    def statistic(y, a, b, s):\\\\n        if s is None:\\\\n            return auc(y, a) - auc(y, b)\\\\n        return (conditional_auc(y, a, s, caliper)\\\\n                - conditional_auc(y, b, s, caliper))\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = statistic(y_true[index], scores_a[index], scores_b[index],\\\\n                          None if strata is None else strata[index])\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = statistic(y_true, scores_a, scores_b, strata)\\\\n    if values.size == 0:\\\\n        return {"delta": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "p_a_not_better": float("nan"), "draws": 0}\\\\n    return {"delta": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "p_a_not_better": float((values <= 0).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Calibrating the residual coupling\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef logit(probability, floor: float = 1e-6):\\\\n    """Probabilities to logits, clipped off the asymptotes."""\\\\n    clipped = np.clip(np.asarray(probability, dtype=float), floor, 1 - floor)\\\\n    return np.log(clipped / (1 - clipped))\\\\n\\\\n\\\\ndef calibrate_coupling(labels, parent_logits, iterations: int = 50) -> dict:\\\\n    """Fit `P(child=1) = sigmoid(a + b * parent_logit)` and return b.\\\\n\\\\n    This is the beta for the residual parameterisation\\\\n    `child_logit = beta * stopgrad(parent_logit) + g(features)`, and where it\\\\n    comes from matters more than its value:\\\\n\\\\n    - **Not** from the model\\\\\\\'s own +0.843 reconstruction coefficient. That is\\\\n      the coupling the current model already has, and reproducing it by\\\\n      construction would hand the residual head exactly the term it is\\\\n      supposed to replace.\\\\n    - **Not** from r_true = 0.403. A rank correlation between two binary\\\\n      labels does not map onto a logit coefficient, and the plan says so\\\\n      explicitly.\\\\n\\\\n    It is fitted out of fold, on labels, by IRLS -- the coupling the *truth*\\\\n    supports between the two findings, measured on whatever label set is\\\\n    trustworthy enough to fit it. `separated` flags the degenerate case where\\\\n    the fit runs away because the parent perfectly orders the child; the\\\\n    coefficient is then meaningless and should not be used.\\\\n    """\\\\n    y = np.asarray(labels, dtype=float)\\\\n    x = np.asarray(parent_logits, dtype=float)\\\\n    ok = np.isfinite(y) & np.isfinite(x)\\\\n    y, x = y[ok], x[ok]\\\\n    if y.size < 10 or y.std() == 0:\\\\n        return {"beta": float("nan"), "intercept": float("nan"),\\\\n                "n": int(y.size), "separated": True}\\\\n\\\\n    design = np.column_stack([np.ones_like(x), x])\\\\n    coefficients = np.zeros(2)\\\\n    for _ in range(iterations):\\\\n        # Clipped before the exponential: under separation the coefficients\\\\n        # run away and the naive form overflows on the way to being flagged.\\\\n        eta = np.clip(design @ coefficients, -700, 700)\\\\n        mu = 1.0 / (1.0 + np.exp(-eta))\\\\n        weight = np.clip(mu * (1 - mu), 1e-9, None)\\\\n        gradient = design.T @ (y - mu)\\\\n        hessian = design.T @ (design * weight[:, None])\\\\n        try:\\\\n            step = np.linalg.solve(hessian, gradient)\\\\n        except np.linalg.LinAlgError:\\\\n            return {"beta": float("nan"), "intercept": float("nan"),\\\\n                    "n": int(y.size), "separated": True}\\\\n        coefficients = coefficients + step\\\\n        if np.max(np.abs(step)) < 1e-8:\\\\n            break\\\\n    separated = bool(np.max(np.abs(coefficients)) > 20)\\\\n    return {"beta": float(coefficients[1]), "intercept": float(coefficients[0]),\\\\n            "n": int(y.size), "separated": separated}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# The Stage 0 gate\\\\n# ---------------------------------------------------------------------------\\\\n#\\\\n# The plan says "clearly > 0.5", "= 0.5" and "ambiguous" without giving\\\\n# numbers. They are fixed here, once, in code, so they cannot be chosen after\\\\n# seeing a result -- and here rather than in the tool because the training\\\\n# notebooks read the same gate and two copies would drift.\\\\n\\\\nGATE_ACTIONS = {\\\\n    "clear": ("synovitis-specific signal survives conditioning on effusion",\\\\n              "Stage 1 -- train the conditional task directly"),\\\\n    "chance": ("the score is a better effusion proxy, nothing more",\\\\n               "Stage 3 (plan §4) or Stage 4 descope (plan §10) -- the plan "\\\\n               "gives both; decide before running either"),\\\\n    "ambiguous": ("expected at n=58; the CI straddles chance",\\\\n                  "Stage 1 at reduced confidence, and prioritise Stage 2 "\\\\n                  "(expand the gold set)"),\\\\n}\\\\n\\\\n\\\\ndef gate_reading(stats: dict) -> str:\\\\n    """clear | chance | ambiguous, from a `bootstrap_conditional` result.\\\\n\\\\n        clear      2.5th percentile above 0.5, and P(AUC <= 0.5) < 0.05\\\\n        chance     point estimate <= 0.55 and P(AUC <= 0.5) >= 0.25\\\\n        ambiguous  anything else -- the expected outcome at n=58\\\\n    """\\\\n    if not np.isfinite(stats.get("auc", float("nan"))):\\\\n        return "ambiguous"\\\\n    if stats["lo"] > 0.5 and stats["p_at_or_below_chance"] < 0.05:\\\\n        return "clear"\\\\n    if stats["auc"] <= 0.55 and stats["p_at_or_below_chance"] >= 0.25:\\\\n        return "chance"\\\\n    return "ambiguous"\\\\n\\\', \\\'src/model/augment.py\\\': \\\'"""Training augmentation for study volumes.\\\\n\\\\nThe pipeline shipped with one augmentation -- a single intensity scale in\\\\n[0.9, 1.1] applied to a whole study -- while the plan (§3.4) calls for what the\\\\nRSNA winners used: geometric transforms, photometric jitter, noise, region\\\\ndropout, and slice-order permutation, all kept consistent across the slices of\\\\na series.\\\\n\\\\n**Series-consistent by construction.** Every transform draws its parameters\\\\nonce per series and applies them to all of that series\\\\\\\' slices. The alternative\\\\n-- fresh parameters per slice -- turns a coherent volume into a stack of\\\\nunrelated images, which is exactly the through-plane structure `stack_2p5d`\\\\nexists to give the backbone.\\\\n\\\\n**What is deliberately absent, and why:**\\\\n\\\\n- **Rotation, perspective, affine.** They need `scipy.ndimage` or `cv2` and run\\\\n  per-slice interpolation over ~80 slices per study; at 4,349 studies an epoch\\\\n  that is a dataloader bottleneck, not a free regulariser. Worth adding behind\\\\n  a measurement of loader throughput, not before.\\\\n- **Arbitrary slice permutation.** The plan lists it, but it would break the\\\\n  2.5D stacking: `stack_2p5d` gives each slice its two neighbours as channels,\\\\n  and a shuffled stack makes those neighbours meaningless. `slice_reverse`\\\\n  is the order change that keeps neighbour structure intact -- it reverses the\\\\n  through-plane direction rather than destroying it.\\\\n- **MixUp.** Not implementable at `BATCH_STUDIES = 1`: a within-batch\\\\n  permutation is the identity. Recorded in `docs/plan.md`.\\\\n\\\\n**Laterality flip is a study-level transform, not a series-level one.** All of\\\\na study\\\\\\\'s series image the same knee, so they must flip together, and the flip\\\\nswaps medial with lateral -- which means `config.LATERAL_LABEL_PAIRS` has to be\\\\nswapped in the targets at the same time. `config.py` has carried that constant\\\\nand that warning since before there was anything to use it. Getting this half\\\\nright (flipping pixels, not labels) corrupts four of the twelve findings\\\\nsilently, which is why the dataset refused to flip at all until now.\\\\n"""\\\\n\\\\nfrom dataclasses import dataclass\\\\nfrom typing import Optional, Tuple\\\\n\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import (DEFAULT_MIRROR_AXIS, META_COLUMNS,\\\\n                               PLANE_MIRROR_AXIS)\\\\n\\\\n# Index of `is_sagittal` in the per-series metadata row. The flip axis depends\\\\n# on the acquisition plane, and this is where that is already encoded -- no\\\\n# need to reach back into the index for `Anatomical_Plane`.\\\\nSAGITTAL_COLUMN = META_COLUMNS.index("is_sagittal")\\\\n\\\\n\\\\ndef flip_volume(volume: np.ndarray, plane=None,\\\\n                sagittal: Optional[bool] = None) -> np.ndarray:\\\\n    """Mirror one series across the medial-lateral axis.\\\\n\\\\n    The axis comes from `dataset.PLANE_MIRROR_AXIS`, the same table\\\\n    `mirror_volume` uses to canonicalise laterality -- deliberately, because a\\\\n    training flip and a test-time flip that disagreed about which axis is\\\\n    medial-lateral would be a mismatch that runs clean and scores worse. On a\\\\n    sagittal series medial-lateral is the slice axis; on coronal and axial it\\\\n    lies in-plane.\\\\n\\\\n    Give it either `plane` (the string from the series metadata) or `sagittal`\\\\n    (the bit already encoded in a metadata row).\\\\n    """\\\\n    if sagittal is not None:\\\\n        axis = PLANE_MIRROR_AXIS["Sagittal"] if sagittal else DEFAULT_MIRROR_AXIS\\\\n    else:\\\\n        axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef lateral_swap_order():\\\\n    """Index permutation that exchanges the medial/lateral finding pairs.\\\\n\\\\n    A mirrored knee puts the medial compartment where the lateral one was, so\\\\n    any prediction or target read off a flipped image has to be permuted by\\\\n    this before it means anything. Used by the training augmentation and by\\\\n    the submission\\\\\\\'s flip TTA, from one definition, so the two cannot drift.\\\\n    """\\\\n    order = list(range(len(config.LABELS)))\\\\n    index_of = {name: i for i, name in enumerate(config.LABELS)}\\\\n    for medial, lateral in config.LATERAL_LABEL_PAIRS:\\\\n        a, b = index_of[medial], index_of[lateral]\\\\n        order[a], order[b] = order[b], order[a]\\\\n    return order\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass AugmentConfig:\\\\n    """Which augmentations run, and how hard.\\\\n\\\\n    The defaults reproduce the incumbent behaviour exactly -- one intensity\\\\n    scale, nothing else -- so `AugmentConfig()` is the reference configuration\\\\n    and every measured result stays comparable. Turn things on one at a time.\\\\n    """\\\\n\\\\n    intensity_scale: Optional[Tuple[float, float]] = (0.9, 1.1)\\\\n    intensity_shift: float = 0.0        # +/- this many 8-bit levels\\\\n    gamma: Optional[Tuple[float, float]] = None\\\\n    noise_std: float = 0.0              # gaussian sigma, in 8-bit levels\\\\n    cutout_boxes: int = 0\\\\n    cutout_fraction: float = 0.25       # box side, as a fraction of the image\\\\n    slice_reverse: float = 0.0          # probability, per series\\\\n    laterality_flip: float = 0.0        # probability, per study\\\\n\\\\n    def __post_init__(self):\\\\n        if not 0.0 <= self.slice_reverse <= 1.0:\\\\n            raise ValueError("slice_reverse must be a probability")\\\\n        if not 0.0 <= self.laterality_flip <= 1.0:\\\\n            raise ValueError("laterality_flip must be a probability")\\\\n        if not 0.0 < self.cutout_fraction <= 1.0:\\\\n            raise ValueError("cutout_fraction must be in (0, 1]")\\\\n\\\\n\\\\nREFERENCE = AugmentConfig()\\\\n\\\\n\\\\ndef augment_series(volume: np.ndarray, rng: np.random.Generator,\\\\n                   cfg: AugmentConfig = REFERENCE) -> np.ndarray:\\\\n    """(D, H, W) uint8 -> (D, H, W) uint8, one parameter draw for the series.\\\\n\\\\n    Photometric work happens in float32 and is clipped back to uint8 once at\\\\n    the end, so a scale followed by a shift does not round twice.\\\\n    """\\\\n    working = volume.astype(np.float32)\\\\n\\\\n    if cfg.intensity_scale is not None:\\\\n        low, high = cfg.intensity_scale\\\\n        working *= rng.uniform(low, high)\\\\n\\\\n    if cfg.intensity_shift:\\\\n        working += rng.uniform(-cfg.intensity_shift, cfg.intensity_shift)\\\\n\\\\n    if cfg.gamma is not None:\\\\n        low, high = cfg.gamma\\\\n        # Gamma is defined on [0, 1]; the volume is 8-bit, so scale in and out.\\\\n        working = np.clip(working, 0, 255)\\\\n        working = 255.0 * np.power(working / 255.0, rng.uniform(low, high))\\\\n\\\\n    if cfg.noise_std > 0:\\\\n        # One noise field for the whole series rather than per slice: the\\\\n        # sensor noise being imitated is spatially structured, and per-slice\\\\n        # noise is trivially averaged away by the slice pooling.\\\\n        working = working + rng.normal(0.0, cfg.noise_std,\\\\n                                       size=working.shape[1:])[None]\\\\n\\\\n    if cfg.cutout_boxes > 0:\\\\n        depth, height, width = working.shape\\\\n        box_h = max(1, int(round(height * cfg.cutout_fraction)))\\\\n        box_w = max(1, int(round(width * cfg.cutout_fraction)))\\\\n        for _ in range(cfg.cutout_boxes):\\\\n            top = int(rng.integers(0, max(1, height - box_h + 1)))\\\\n            left = int(rng.integers(0, max(1, width - box_w + 1)))\\\\n            # The SAME box on every slice: a box that moves through the stack\\\\n            # is a different occlusion per slice, which the slice pooling can\\\\n            # simply average out instead of learning around.\\\\n            working[:, top:top + box_h, left:left + box_w] = 0.0\\\\n\\\\n    if cfg.slice_reverse and rng.random() < cfg.slice_reverse:\\\\n        working = working[::-1]\\\\n\\\\n    return np.clip(working, 0, 255).astype(np.uint8)\\\\n\\\\n\\\\ndef flip_laterality(item: dict, rng: np.random.Generator) -> dict:\\\\n    """Mirror every series of a study and swap the medial/lateral labels.\\\\n\\\\n    Volumes reach here already canonicalised to the right knee, so this maps\\\\n    the study onto the left and back-labels it accordingly -- doubling the\\\\n    laterality distribution the model sees without inventing anatomy.\\\\n\\\\n    The flip axis depends on the acquisition plane, exactly as\\\\n    `dataset.mirror_volume` decides it: on a sagittal series left-right runs\\\\n    *through* the plane, so mirroring reverses the slice order; on coronal and\\\\n    axial it lies in-plane, so it flips the columns. `is_sagittal` in the\\\\n    metadata row is what distinguishes them.\\\\n\\\\n    Targets and weights are permuted by `config.LATERAL_LABEL_PAIRS`. Both, not\\\\n    just the targets: a weight says how much a finding\\\\\\\'s label is believed, and\\\\n    it belongs to the finding, so it has to travel with it.\\\\n    """\\\\n    volumes, meta = item["volumes"], item["meta"]\\\\n    flipped = np.empty_like(volumes)\\\\n    for i in range(volumes.shape[0]):\\\\n        flipped[i] = flip_volume(\\\\n            volumes[i], sagittal=bool(meta[i, SAGITTAL_COLUMN] > 0.5))\\\\n    item["volumes"] = flipped\\\\n\\\\n    order = lateral_swap_order()\\\\n    for key in ("targets", "weights"):\\\\n        if item.get(key) is not None:\\\\n            item[key] = np.asarray(item[key])[order]\\\\n\\\\n    # `is_right` in the metadata now describes the wrong side. Left as it is on\\\\n    # purpose: it is a *prediction* from the side classifier describing the\\\\n    # image as acquired, and the model reads it as conditioning rather than as\\\\n    # a label. Rewriting it here would tell the model the flip happened, which\\\\n    # is precisely the invariance the augmentation exists to teach.\\\\n    return item\\\\n\\\\n\\\\ndef augment_study(item: dict, rng: np.random.Generator,\\\\n                  cfg: AugmentConfig = REFERENCE) -> dict:\\\\n    """Apply the study-level flip, then per-series photometric work."""\\\\n    if cfg.laterality_flip and rng.random() < cfg.laterality_flip:\\\\n        item = flip_laterality(item, rng)\\\\n\\\\n    volumes = item["volumes"]\\\\n    if volumes.shape[0]:\\\\n        item["volumes"] = np.stack(\\\\n            [augment_series(volumes[i], rng, cfg)\\\\n             for i in range(volumes.shape[0])])\\\\n    return item\\\\n\\\', \\\'src/model/averaging.py\\\': \\\'"""Weight averaging: EMA, SWA, and model soups.\\\\n\\\\n**Why this module exists.** Checkpoint selection here is broken in a specific,\\\\ndiagnosable way. Training labels are mined from radiology reports and agree\\\\nwith image-derived ground truth about 82% of the time, and the validation fold\\\\nis mined from the *same* distribution. So the network first fits the structure\\\\nthe two sources share and then memorises the 18% they do not -- Arpit et al.,\\\\n"A Closer Look at Memorization in Deep Networks" (ICML 2017), and Liu et al.,\\\\n"Early-Learning Regularization Prevents Memorization of Noisy Labels" (NeurIPS\\\\n2020). Mined validation agreement keeps *rising* through that second phase,\\\\nbecause agreeing with the noise is what it measures. It is therefore a biased\\\\nselector, not merely a noisy one, and "take the best epoch on validation"\\\\nselects into the memorisation phase on purpose.\\\\n\\\\nMeasured here: mined val rose to 0.860 at epoch 9 and fell to 0.843 by epoch\\\\n20, while gold held flat at 0.864-0.869 across eleven consecutive epochs.\\\\n\\\\nThe fix is to stop selecting. Averaging weights is selection-free:\\\\n\\\\n- **EMA** (Tarvainen & Valpola, NeurIPS 2017) -- an exponential moving average\\\\n  maintained during training. Cheap, standard, and already the default in most\\\\n  detection and segmentation pipelines.\\\\n- **SWA** (Izmailov et al., UAI 2018, arXiv:1803.05407) -- an equal-weight\\\\n  average over the tail of the trajectory, which lands in a flatter and better\\\\n  generalising region than any single point on it.\\\\n- **Model soups** (Wortsman et al., ICML 2022, arXiv:2203.05482) -- average\\\\n  the weights of several finished fine-tunes instead of picking the best one\\\\n  on a held-out set. The paper\\\\\\\'s argument is directed at exactly the recipe\\\\n  failing here: "pick the individual model which performs best on a held-out\\\\n  validation set, discarding the remainder".\\\\n\\\\n**Averaging weights is only meaningful for models in the same basin.** Two\\\\nruns whose aggregation was randomly initialised from different seeds are not,\\\\nand averaging them produces a model that is worse than either -- silently, as\\\\na plausible-looking set of numbers. `assert_soupable` checks what it can\\\\n(shapes and key sets); it cannot check the basin, so the training notebook\\\\nseeds every fold identically, and the docstrings below say when a soup is\\\\nwell-posed.\\\\n\\\\n**Nothing here may select on the gold 58.** `greedy_soup` takes a caller\\\\\\\'s\\\\nscore function precisely so the choice of scoring set stays visible at the\\\\ncall site: it must be mined validation or an out-of-fold dump. Selecting a\\\\nsoup on gold would turn the project\\\\\\\'s one honest measuring stick into a\\\\nmodel-selection set, which has already cost this project an inflated figure\\\\nonce.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\\\\n\\\\nimport torch\\\\n\\\\nState = Dict[str, torch.Tensor]\\\\n\\\\n\\\\ndef _is_averageable(tensor: torch.Tensor) -> bool:\\\\n    """Floating-point tensors average; integer counters and flags do not.\\\\n\\\\n    `num_batches_tracked` on a BatchNorm is the canonical example: averaging\\\\n    it produces a fractional count, and torch stores it as int64 anyway. The\\\\n    aggregation stack is all LayerNorm so this does not arise today, but a\\\\n    convnet backbone with BatchNorm would hit it immediately.\\\\n    """\\\\n    return tensor.is_floating_point()\\\\n\\\\n\\\\ndef assert_soupable(states: Sequence[State]) -> None:\\\\n    """Refuse states that cannot be averaged elementwise.\\\\n\\\\n    Catches the two mechanical failures -- a different architecture, and a\\\\n    checkpoint saved at a different width -- before they become a tensor of\\\\n    the wrong shape several cells later.\\\\n    """\\\\n    if not states:\\\\n        raise ValueError("no states to average")\\\\n    reference = states[0]\\\\n    for i, state in enumerate(states[1:], start=1):\\\\n        if set(state) != set(reference):\\\\n            missing = sorted(set(reference) - set(state))[:3]\\\\n            extra = sorted(set(state) - set(reference))[:3]\\\\n            raise ValueError(\\\\n                f"state {i} has different keys from state 0 "\\\\n                f"(missing {missing}, extra {extra}) -- these are not the "\\\\n                "same architecture")\\\\n        for key, tensor in state.items():\\\\n            if tensor.shape != reference[key].shape:\\\\n                raise ValueError(\\\\n                    f"state {i} has {key} at {tuple(tensor.shape)} against "\\\\n                    f"{tuple(reference[key].shape)} in state 0")\\\\n\\\\n\\\\ndef average_states(states: Sequence[State],\\\\n                   weights: Optional[Sequence[float]] = None) -> State:\\\\n    """Elementwise weighted average of state dicts.\\\\n\\\\n    `weights` defaults to equal, which is SWA and the uniform soup. Averaging\\\\n    happens in float64 and is cast back to each tensor\\\\\\\'s own dtype, so a run\\\\n    of fp16 checkpoints does not accumulate rounding across a dozen members.\\\\n\\\\n    Non-floating tensors are taken from the first state rather than averaged.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if weights is None:\\\\n        weights = [1.0] * len(states)\\\\n    if len(weights) != len(states):\\\\n        raise ValueError(f"{len(weights)} weights for {len(states)} states")\\\\n    total = float(sum(weights))\\\\n    if total <= 0:\\\\n        raise ValueError("weights must sum to something positive")\\\\n\\\\n    averaged: State = {}\\\\n    for key, reference in states[0].items():\\\\n        if not _is_averageable(reference):\\\\n            averaged[key] = reference.clone()\\\\n            continue\\\\n        accumulator = torch.zeros_like(reference, dtype=torch.float64)\\\\n        for state, weight in zip(states, weights):\\\\n            accumulator += state[key].to(torch.float64) * float(weight)\\\\n        averaged[key] = (accumulator / total).to(reference.dtype)\\\\n    return averaged\\\\n\\\\n\\\\nclass WeightAverage:\\\\n    """EMA and SWA of one model\\\\\\\'s weights, maintained during training.\\\\n\\\\n    Both are accumulated from the same call so a run pays one copy of the\\\\n    weights per update and produces both artefacts. `decay` is the EMA\\\\\\\'s, and\\\\n    0.999 is the usual choice; `swa_from` is the step at which the equal-weight\\\\n    tail average starts, so a caller sets it to 75% of the planned steps to\\\\n    average over the last quarter.\\\\n\\\\n    **The averages live wherever the model does, in float32.** Accumulating on\\\\n    the host instead would copy the whole trunk across the PCIe bus on every\\\\n    optimiser step -- ~110 MB for ConvNeXt-T plus the aggregation, twice per\\\\n    step once the SWA window opens, about 11,000 times over a twenty-epoch\\\\n    run. That is a wall-clock tax on the run for no benefit; two shadows on\\\\n    the device cost ~220 MB of a 16 GB T4, which the batch has to spare.\\\\n    Float32 regardless of training precision: under `16-mixed` the parameters\\\\n    are fp32 anyway (autocast does not change them), but an fp16 accumulation\\\\n    over thousands of steps would round away exactly the small updates an EMA\\\\n    exists to keep. `export` moves the result to the host.\\\\n\\\\n    `device` overrides the placement -- pass "cpu" if a run is ever tight\\\\n    enough on device memory that 220 MB matters.\\\\n    """\\\\n\\\\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999,\\\\n                 swa_from: Optional[int] = None, device=None):\\\\n        if not 0.0 < decay < 1.0:\\\\n            raise ValueError(f"decay must be in (0, 1), got {decay}")\\\\n        self.decay = decay\\\\n        self.swa_from = swa_from\\\\n        self.steps = 0\\\\n        self.swa_count = 0\\\\n        state = model.state_dict()\\\\n        self.ema: State = {\\\\n            k: (v.detach().clone().to(torch.float32) if device is None\\\\n                else v.detach().to(device, torch.float32).clone())\\\\n            for k, v in state.items() if _is_averageable(v)}\\\\n        self.swa: State = {k: torch.zeros_like(v) for k, v in self.ema.items()}\\\\n        # Everything not averaged -- integer counters. The normalisation\\\\n        # buffers are float, so they average harmlessly to themselves.\\\\n        self._fixed: State = {k: v.detach().cpu().clone()\\\\n                              for k, v in state.items()\\\\n                              if not _is_averageable(v)}\\\\n\\\\n    @torch.no_grad()\\\\n    def update(self, model: torch.nn.Module) -> None:\\\\n        """Fold one optimiser step\\\\\\\'s weights into both averages."""\\\\n        self.steps += 1\\\\n        state = model.state_dict()\\\\n        in_window = (self.swa_from is not None and self.steps >= self.swa_from)\\\\n        if in_window:\\\\n            self.swa_count += 1\\\\n\\\\n        for key, shadow in self.ema.items():\\\\n            # Fetched once and reused by both averages: reading it per average\\\\n            # doubled the copy for every step of the SWA window.\\\\n            current = state[key].detach().to(shadow.device, torch.float32,\\\\n                                             non_blocking=True)\\\\n            shadow.mul_(self.decay).add_(current, alpha=1.0 - self.decay)\\\\n            if in_window:\\\\n                accumulator = self.swa[key]\\\\n                # Running mean, so the tail can be any length without a second\\\\n                # pass or a list of checkpoints held in memory.\\\\n                accumulator.add_((current - accumulator) / self.swa_count)\\\\n\\\\n    def ema_state(self) -> State:\\\\n        """The EMA weights, as a state dict a `KneeNet` loads.\\\\n\\\\n        Moved to the host on the way out: a checkpoint holding CUDA tensors\\\\n        reloads onto a device that may not exist in the process reading it.\\\\n        """\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.ema.items()}}\\\\n\\\\n    def swa_state(self) -> Optional[State]:\\\\n        """The tail average, or None if no step fell inside the window."""\\\\n        if self.swa_count == 0:\\\\n            return None\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.swa.items()}}\\\\n\\\\n\\\\ndef greedy_soup(states: Sequence[State], score: Callable[[State], float],\\\\n                names: Optional[Sequence[str]] = None,\\\\n                tolerance: float = 0.0) -> Tuple[State, List[str]]:\\\\n    """Wortsman et al.\\\\\\\'s greedy soup: add a member only if it helps.\\\\n\\\\n    Members are ranked by their individual score, then considered in order;\\\\n    each is kept if the average *including* it scores at least as well as the\\\\n    average without it. The result is never worse than the best single member\\\\n    on the scoring set, which is what makes greedy preferable to a uniform\\\\n    soup when the members are of uneven quality.\\\\n\\\\n    `score` maps a state dict to a number that is better when larger --\\\\n    macro AUC here. **It must not read the gold 58.** Score on the mined\\\\n    validation fold or on an out-of-fold dump; the whole reason this takes a\\\\n    callable rather than a dataset is to keep that choice at the call site\\\\n    where a reviewer can see it.\\\\n\\\\n    `tolerance` lets a member in when it is within that much of the running\\\\n    best, for the case where the scoring set is too small to resolve the\\\\n    difference and averaging more members is the safer default. At 0.0 this is\\\\n    the paper\\\\\\\'s algorithm exactly.\\\\n\\\\n    Returns the souped state and the names of the members that went into it,\\\\n    in the order they were added.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if names is None:\\\\n        names = [f"member{i}" for i in range(len(states))]\\\\n    if len(names) != len(states):\\\\n        raise ValueError(f"{len(names)} names for {len(states)} states")\\\\n\\\\n    ranked = sorted(zip(names, states), key=lambda pair: score(pair[1]),\\\\n                    reverse=True)\\\\n\\\\n    kept_names: List[str] = [ranked[0][0]]\\\\n    kept_states: List[State] = [ranked[0][1]]\\\\n    best = score(kept_states[0])\\\\n\\\\n    for name, state in ranked[1:]:\\\\n        candidate = average_states(kept_states + [state])\\\\n        value = score(candidate)\\\\n        if value >= best - tolerance:\\\\n            kept_names.append(name)\\\\n            kept_states.append(state)\\\\n            best = max(best, value)\\\\n\\\\n    return average_states(kept_states), kept_names\\\\n\\\\n\\\\ndef rank_average(predictions: Sequence["np.ndarray"]) -> "np.ndarray":\\\\n    """Average ensemble members by within-member rank, not by probability.\\\\n\\\\n    The metric is macro ROC-AUC, which reads only the ordering of studies\\\\n    within a finding. Averaging probabilities lets a member with a compressed\\\\n    or shifted output distribution contribute less than one with a confident\\\\n    one, even when the two rank studies identically -- the mean is sensitive to\\\\n    calibration and AUC is not. Converting each member to ranks first removes\\\\n    that: every member gets one vote per finding, whatever its scale.\\\\n\\\\n    Each member is (N studies, L findings); ranking is per finding, per member,\\\\n    with ties averaged, and the result is scaled to [0, 1] so it can be written\\\\n    into a submission unchanged. Because AUC is invariant to any increasing\\\\n    transform, that scaling costs nothing and keeps the output looking like the\\\\n    probabilities the rest of the pipeline passes around.\\\\n\\\\n    A single member is returned rank-transformed rather than untouched, so that\\\\n    a one-model and a five-model submission are produced by the same code path.\\\\n    """\\\\n    import numpy as np\\\\n\\\\n    if not predictions:\\\\n        raise ValueError("no predictions to average")\\\\n    stacked = np.stack([np.asarray(p, dtype=float) for p in predictions])\\\\n    if stacked.ndim != 3:\\\\n        raise ValueError(\\\\n            f"expected (members, studies, findings), got {stacked.shape}")\\\\n\\\\n    n_studies = stacked.shape[1]\\\\n    if n_studies < 2:\\\\n        # Ranks are meaningless for a single study, and the scaling below\\\\n        # would divide by zero.\\\\n        return stacked.mean(axis=0)\\\\n\\\\n    ranked = np.empty_like(stacked)\\\\n    for member in range(stacked.shape[0]):\\\\n        for finding in range(stacked.shape[2]):\\\\n            column = stacked[member, :, finding]\\\\n            order = np.argsort(column, kind="mergesort")\\\\n            positions = np.empty(n_studies, dtype=float)\\\\n            positions[order] = np.arange(n_studies, dtype=float)\\\\n            # Average the positions of tied values, so a member that predicts\\\\n            # a constant contributes no ordering rather than an arbitrary one.\\\\n            unique, inverse, counts = np.unique(column, return_inverse=True,\\\\n                                                return_counts=True)\\\\n            sums = np.zeros(len(unique))\\\\n            np.add.at(sums, inverse, positions)\\\\n            ranked[member, :, finding] = (sums / counts)[inverse]\\\\n\\\\n    return ranked.mean(axis=0) / (n_studies - 1)\\\\n\\\', \\\'src/model/splits.py\\\': \\\'"""Cross-validation splits and the gold holdout.\\\\n\\\\nTwo rules govern every split here, and both exist because the obvious\\\\nalternative produces a number that looks better and means less.\\\\n\\\\n**Split by study, never by series.** A study\\\\\\\'s ~5.5 series image the same knee\\\\nin the same session. A random series split puts near-duplicates on both sides\\\\nand reports an accuracy the model does not have.\\\\n\\\\n**The 58 gold studies never enter training.** They are the only radiologist\\\\nlabels in the dataset and the only honest measurement available. Everything\\\\nelse is trained on mined labels, and a model evaluated against mined labels can\\\\nscore well by reproducing the extractor\\\\\\\'s mistakes. `assert_no_gold_leakage`\\\\nis called by the training entrypoint rather than trusted to discipline.\\\\n"""\\\\n\\\\nfrom typing import Iterator, List, Optional, Sequence, Tuple\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\nN_FOLDS = 5\\\\nSEED = 0\\\\n\\\\n\\\\ndef load_gold_uids(train_csv=None) -> set:\\\\n    """Study UIDs of the 58 radiologist-labelled studies."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()]\\\\n    return set(labelled[config.ID_COL])\\\\n\\\\n\\\\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\\\\n    """Fail loudly if a gold study appears in training data.\\\\n\\\\n    Called from the training entrypoint. The failure this guards against is\\\\n    silent -- a leaked gold study inflates the one number being used to decide\\\\n    whether the model works.\\\\n    """\\\\n    leaked = set(train_uids) & gold_uids\\\\n    if leaked:\\\\n        raise ValueError(\\\\n            f"{len(leaked)} gold studies leaked into training: "\\\\n            f"{sorted(leaked)[:3]}..."\\\\n        )\\\\n\\\\n\\\\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\\\\n    """A coarse key for balancing folds across a 12-label problem.\\\\n\\\\n    Full multi-label stratification over 2^12 combinations would leave most\\\\n    cells empty at 4,349 studies. The count of positive findings captures the\\\\n    axis that matters -- studies with many findings are rare and should not\\\\n    cluster in one fold.\\\\n    """\\\\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\\\\n    return positives.clip(upper=4)\\\\n\\\\n\\\\ndef scanner_key(index: pd.DataFrame) -> pd.Series:\\\\n    """A per-study scanner identity, from the header scan\\\\\\\'s index.\\\\n\\\\n    The dataset spans 16-19 sites on five continents, and a CNN reads scanner\\\\n    signature as readily as it reads pathology. A fold split that puts the same\\\\n    scanner on both sides therefore reports an accuracy that does not survive\\\\n    a new site: the plan\\\\\\\'s figure is ~0.053 of inflation from random K-fold,\\\\n    with one measured model showing a 0.136 grouped-vs-random gap.\\\\n\\\\n    Site is not given, so manufacturer + model + field strength stands in for\\\\n    it. That is coarser than a site id -- two hospitals with the same scanner\\\\n    collapse into one group, which makes the split *more* conservative, not\\\\n    less -- and it is what the DICOM headers actually carry.\\\\n\\\\n    Takes the frame `src/data/index.py` writes (one row per instance or\\\\n    series, carrying `StudyInstanceUID`), and returns one key per study,\\\\n    indexed by study UID. A study whose headers name none of the three fields\\\\n    gets "unknown", and those all land in one group together.\\\\n    """\\\\n    fields = [c for c in ("Manufacturer", "ManufacturerModelName",\\\\n                          "MagneticFieldStrength") if c in index.columns]\\\\n    if not fields:\\\\n        raise ValueError(\\\\n            "the index carries none of Manufacturer / ManufacturerModelName / "\\\\n            f"MagneticFieldStrength; it has {list(index.columns)[:8]}")\\\\n\\\\n    # fillna after astype: pandas keeps a missing value missing through\\\\n    # astype(str) rather than rendering it "nan", and the join below would\\\\n    # then fail on a float instead of producing a key.\\\\n    joined = index[fields].astype(str).fillna("nan").agg("|".join, axis=1)\\\\n    joined = joined.where(~joined.str.fullmatch(r"(nan\\\\\\\\|?)+"), "unknown")\\\\n    # The first series scanned decides the study. A study imaged on two\\\\n    # scanners is rare and either group is defensible; taking the first keeps\\\\n    # the assignment deterministic.\\\\n    return joined.groupby(index[config.ID_COL]).first()\\\\n\\\\n\\\\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\\\\n               seed: int = SEED,\\\\n               groups: Optional[pd.Series] = None) -> pd.DataFrame:\\\\n    """Assign each study to a fold, stratified by positive-finding count.\\\\n\\\\n    Returns the frame with a `fold` column added.\\\\n\\\\n    `groups` maps study UID -> group id (see `scanner_key`). When given, a\\\\n    whole group lands in one fold: the split becomes leave-scanner-out rather\\\\n    than leave-study-out, so validation measures transfer to unseen equipment\\\\n    instead of rewarding memorised scanner signature.\\\\n\\\\n    Grouping costs balance. Groups are sized very unequally -- one scanner may\\\\n    hold a quarter of the corpus -- so groups are assigned largest-first to\\\\n    whichever fold is currently smallest, which is the standard greedy\\\\n    approximation and the best available without splitting a group. Check\\\\n    `summarise()` afterwards: badly imbalanced folds are the price of grouping,\\\\n    and worth knowing before a run rather than after.\\\\n\\\\n    **Changing this re-cuts every fold**, and every reference number on record\\\\n    (fold 1: val 0.872 / gold 0.861 / LB 0.889) belongs to the ungrouped\\\\n    split. Adopt it when the fold ensemble is rebuilt, not in the middle of an\\\\n    experiment that is being compared against those numbers.\\\\n    """\\\\n    frame = labels.copy().reset_index(drop=True)\\\\n    key = stratification_key(frame)\\\\n    rng = np.random.default_rng(seed)\\\\n\\\\n    frame["fold"] = -1\\\\n    if groups is None:\\\\n        for value in key.unique():\\\\n            rows = np.flatnonzero(key.to_numpy() == value)\\\\n            rng.shuffle(rows)\\\\n            frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\\\\n    else:\\\\n        mapped = frame[config.ID_COL].map(groups).fillna("unknown")\\\\n        sizes = mapped.value_counts()\\\\n        # Shuffle before sorting so equal-sized groups do not always fall the\\\\n        # same way, then place the largest first into the emptiest fold.\\\\n        order = sizes.sample(frac=1.0, random_state=seed).sort_values(\\\\n            ascending=False, kind="mergesort").index\\\\n        counts = np.zeros(n_folds, dtype=int)\\\\n        for group in order:\\\\n            fold = int(np.argmin(counts))\\\\n            rows = np.flatnonzero(mapped.to_numpy() == group)\\\\n            frame.loc[rows, "fold"] = fold\\\\n            counts[fold] += len(rows)\\\\n\\\\n    if (frame["fold"] < 0).any():\\\\n        raise RuntimeError("some studies were not assigned a fold")\\\\n    return frame\\\\n\\\\n\\\\ndef iter_folds(labels: pd.DataFrame,\\\\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\\\\n    """Yield (train, validation) frames for each fold."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    for fold in range(n_folds):\\\\n        validation = labels[labels["fold"] == fold]\\\\n        train = labels[labels["fold"] != fold]\\\\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\\\\n\\\\n\\\\ndef series_for_studies(manifest: pd.DataFrame,\\\\n                       study_uids: Sequence[str]) -> pd.DataFrame:\\\\n    """All cached series belonging to a set of studies."""\\\\n    wanted = set(study_uids)\\\\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\\\\n\\\\n\\\\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\\\\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    rows = []\\\\n    for fold in range(n_folds):\\\\n        subset = labels[labels["fold"] == fold]\\\\n        row = {"fold": fold, "studies": len(subset)}\\\\n        for name in config.LABELS:\\\\n            row[name] = float((subset[name] > 0.5).mean())\\\\n        rows.append(row)\\\\n    return pd.DataFrame(rows).set_index("fold")\\\\n\\\', \\\'src/labels/__init__.py\\\': \\\'\\\', \\\'src/labels/calibrate.py\\\': \\\'"""Score a report extractor against the 58 radiologist-labelled studies.\\\\n\\\\nThose 58 labels are the only trustworthy signal in the dataset, and this is\\\\ntheir highest-value use: measuring how much a mined label can be believed,\\\\nper finding. They are never used for training.\\\\n\\\\nThe per-finding AUC produced here drives the kill criterion -- a finding whose\\\\nmined labels score below KILL_THRESHOLD is noise, and the model should predict\\\\nthe training prior for it rather than learn from corrupted targets.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Below this AUC against gold, a finding\\\\\\\'s mined labels are not worth training\\\\n# on. Chosen to sit meaningfully above chance while tolerating the noise of a\\\\n# 58-sample estimate.\\\\nKILL_THRESHOLD = 0.70\\\\n\\\\n\\\\ndef load_gold(train_csv=None) -> pd.DataFrame:\\\\n    """The 58 studies carrying radiologist labels."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()].copy()\\\\n    for name in config.LABELS:\\\\n        labelled[name] = labelled[name].astype(int)\\\\n    return labelled.reset_index(drop=True)\\\\n\\\\n\\\\ndef load_unlabelled(train_csv=None) -> pd.DataFrame:\\\\n    """The 4,349 studies carrying only a report."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    return frame[frame[config.LABELS[0]].isna()].reset_index(drop=True)\\\\n\\\\n\\\\ndef _auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via rank statistic, so sklearn stays an optional dependency."""\\\\n    positives = y_true == 1\\\\n    n_pos = int(positives.sum())\\\\n    n_neg = int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n\\\\n    # Average ranks within ties, otherwise a constant score would not score 0.5.\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)\\\\n    tie_sums = np.zeros(len(unique))\\\\n    np.add.at(tie_sums, inverse, ranks)\\\\n    ranks = (tie_sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef score_extractor(\\\\n    extract_fn: Callable[[str, str], Dict],\\\\n    gold: Optional[pd.DataFrame] = None,\\\\n) -> pd.DataFrame:\\\\n    """Per-finding reliability of an extractor against the gold studies.\\\\n\\\\n    `extract_fn(report, finding)` must return a dict with `present` (1/0/None)\\\\n    and `severity` (float or None), matching the lexicon module\\\\\\\'s contract.\\\\n    """\\\\n    gold = gold if gold is not None else load_gold()\\\\n    reports = gold["Report"].tolist()\\\\n    records = []\\\\n\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy()\\\\n        results = [extract_fn(report, name) for report in reports]\\\\n\\\\n        present = np.array([\\\\n            r["present"] if r["present"] is not None else -1 for r in results\\\\n        ])\\\\n        severity = np.array([\\\\n            r["severity"] if r["severity"] is not None else 0.0 for r in results\\\\n        ])\\\\n\\\\n        predicted = (present == 1).astype(int)\\\\n        true_pos = int(((predicted == 1) & (truth == 1)).sum())\\\\n        false_pos = int(((predicted == 1) & (truth == 0)).sum())\\\\n        false_neg = int(((predicted == 0) & (truth == 1)).sum())\\\\n\\\\n        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else np.nan\\\\n        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else np.nan\\\\n        auc = _auc(truth, severity)\\\\n\\\\n        # How often a finding goes unmentioned, and how often it is truly\\\\n        # positive when unmentioned. The second number is what fusion should\\\\n        # use as the soft target for silence -- treating it as a hard zero is\\\\n        # what injects systematic noise.\\\\n        unmentioned = present == -1\\\\n        silence_rate = float(unmentioned.mean())\\\\n        prior_if_silent = (\\\\n            float(truth[unmentioned].mean()) if unmentioned.any() else np.nan\\\\n        )\\\\n\\\\n        records.append({\\\\n            "finding": name,\\\\n            "n_positive": int(truth.sum()),\\\\n            "mention_rate": 1.0 - silence_rate,\\\\n            "precision": precision,\\\\n            "recall": recall,\\\\n            "auc": auc,\\\\n            "prior_if_silent": prior_if_silent,\\\\n            "usable": bool(auc >= KILL_THRESHOLD) if np.isfinite(auc) else False,\\\\n        })\\\\n\\\\n    return pd.DataFrame(records).set_index("finding")\\\\n\\\\n\\\\ndef format_report(table: pd.DataFrame) -> str:\\\\n    """Human-readable reliability table."""\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'n+\\\\\\\':>3s} {\\\\\\\'ment\\\\\\\':>5s} {\\\\\\\'prec\\\\\\\':>5s} {\\\\\\\'rec\\\\\\\':>5s} "\\\\n        f"{\\\\\\\'AUC\\\\\\\':>5s} {\\\\\\\'P(1|silent)\\\\\\\':>11s} {\\\\\\\'use\\\\\\\':>4s}",\\\\n        "-" * 66,\\\\n    ]\\\\n    for name, row in table.iterrows():\\\\n        lines.append(\\\\n            f"{name:18s} {int(row[\\\\\\\'n_positive\\\\\\\']):3d} "\\\\n            f"{row[\\\\\\\'mention_rate\\\\\\\']:4.0%} {row[\\\\\\\'precision\\\\\\\']:5.2f} "\\\\n            f"{row[\\\\\\\'recall\\\\\\\']:5.2f} {row[\\\\\\\'auc\\\\\\\']:5.3f} "\\\\n            f"{row[\\\\\\\'prior_if_silent\\\\\\\']:11.2f} {\\\\\\\'yes\\\\\\\' if row[\\\\\\\'usable\\\\\\\'] else \\\\\\\'NO\\\\\\\':>4s}"\\\\n        )\\\\n    lines.append("-" * 66)\\\\n    lines.append(f"macro AUC: {table[\\\\\\\'auc\\\\\\\'].mean():.3f}   "\\\\n                 f"usable findings: {int(table[\\\\\\\'usable\\\\\\\'].sum())}/{len(table)}")\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    from src.labels import lexicon\\\\n\\\\n    table = score_extractor(lexicon.extract_finding)\\\\n    print(format_report(table))\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/labels/lexicon.py\\\': \\\'"""Rule-based extraction of the twelve findings from multilingual reports.\\\\n\\\\nThe corpus spans roughly eleven languages -- English (39%), Greek, Croatian/\\\\nSerbian, Spanish, Turkish, German, Bulgarian/Russian, Dutch, French, Italian,\\\\nPortuguese. Rather than a language x finding cross-product, this module keys on\\\\nthe Greek and Latin stems that clinical vocabulary shares across all of them\\\\n(menisc- / menisk- / μηνίσκ- / мениск-), with language-specific additions only\\\\nwhere the local term is genuinely unrelated (Turkish "kıkırdak" for cartilage).\\\\n\\\\nEach finding resolves to one of three states:\\\\n\\\\n    1     an unnegated mention above the severity floor\\\\n    0     mentioned but negated, or present only below the severity floor\\\\n    None  never mentioned -- which is *not* the same as absent, because\\\\n          reports narrate positives and omit much else\\\\n\\\\nCallers must preserve the None state. Collapsing it to 0 injects systematic\\\\nnoise into exactly the findings that are least often narrated.\\\\n"""\\\\n\\\\nimport re\\\\nimport unicodedata\\\\nfrom typing import Dict, List, Optional\\\\n\\\\nfrom src import config\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cues\\\\n# ---------------------------------------------------------------------------\\\\n# Negation, across every language present in the corpus. Matched as whole words\\\\n# within a bounded window before the finding term.\\\\nNEGATION_CUES = [\\\\n    # English / Latin-script general\\\\n    "no", "not", "without", "absent", "negative", "unremarkable", "intact",\\\\n    "normal", "preserved", "free of", "rule out", "ruled out",\\\\n    # Spanish / Portuguese\\\\n    "sin", "no", "ausencia", "ausente", "conservado", "conservada", "integro",\\\\n    "íntegro", "integra", "íntegra", "normales", "sem", "nao", "não",\\\\n    # Dutch\\\\n    "geen", "niet", "zonder", "intacte", "ongestoord",\\\\n    # German\\\\n    "kein", "keine", "keinen", "nicht", "ohne", "unauffällig", "regelrecht",\\\\n    "unauffaellig",\\\\n    # French\\\\n    "pas", "sans", "absence", "aucun", "aucune", "normale",\\\\n    # Italian\\\\n    "non", "senza", "assenza", "nella norma",\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "değil", "degil", "normaldir", "korunmuş", "korunmus",\\\\n    # Croatian / Serbian / Bosnian\\\\n    "bez", "nema", "nije", "uredan", "uredna", "uredno", "očuvan", "ocuvan",\\\\n    # Bulgarian / Russian\\\\n    "без", "нет", "няма", "не", "отсутств", "интактн", "сохранен",\\\\n    # Greek\\\\n    "χωρίς", "δεν", "ουδεμία", "ουδέν", "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Hedged or sub-threshold mentions. Ground truth was adjudicated against\\\\n# explicit severity thresholds, so "trace effusion" and "large effusion" are\\\\n# not the same label.\\\\nMINIMAL_CUES = [\\\\n    "minimal", "mínimo", "minimo", "minima", "mínima", "trace", "tiny",\\\\n    "slight", "small", "mild", "leve", "ligero", "ligera", "discreto",\\\\n    "discreta", "gering", "geringe", "klein", "kleine", "wenig", "licht",\\\\n    "lichte", "petit", "petite", "faible", "lieve", "piccolo", "az", "küçük",\\\\n    "kucuk", "hafif", "minimalan", "mali", "blag", "малък", "минимален",\\\\n    "незначител", "ελάχιστ", "μικρ", "ήπι",\\\\n]\\\\n\\\\nUNCERTAIN_CUES = [\\\\n    "possible", "possibly", "suspicious", "suspected", "probable", "probably",\\\\n    "questionable", "equivocal", "cannot exclude", "cannot be excluded",\\\\n    "may represent", "posible", "sospecha", "sospechoso", "dudoso",\\\\n    "verdacht", "möglich", "moglich", "fraglich", "mogelijk", "verdenking",\\\\n    "suspect", "possible", "suspicion", "şüpheli", "supheli", "olası", "olasi",\\\\n    "sumnja", "moguć", "mogu", "възможн", "подозр", "πιθαν", "ύποπτ",\\\\n]\\\\n\\\\n# Negation that *follows* the finding term. Turkish is verb-final, so\\\\n# "eklem ici sivi yoktur" negates with a trailing word and a backwards-only\\\\n# search misses it entirely -- that is a tenth of this corpus. English and\\\\n# several others do the same with trailing adjectives ("meniscus intact").\\\\n# Kept deliberately narrower than NEGATION_CUES: a forward search over the full\\\\n# cue list would let an unrelated "normal" in the same clause cancel a real\\\\n# finding.\\\\nPOSTFIX_NEGATION_CUES = [\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "saptanmamistir", "değil", "degil", "normaldir", "normaldi", "korunmuş",\\\\n    "korunmus", "izlenmedi",\\\\n    # English / Latin-script trailing adjectives\\\\n    "intact", "unremarkable", "preserved", "normal", "normale", "normales",\\\\n    "within normal limits", "is normal", "are normal",\\\\n    # Dutch / German\\\\n    "intacte", "unauffällig", "unauffaellig", "regelrecht", "ongestoord",\\\\n    # Croatian / Serbian\\\\n    "uredan", "uredna", "uredno", "očuvan", "ocuvan", "očuvana", "ocuvana",\\\\n    # Bulgarian / Russian\\\\n    "сохранен", "интактн", "не се визуализира", "непроменена",\\\\n    # Greek\\\\n    "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Terms that signal a torn or damaged structure, used by the ligament and\\\\n# meniscus findings where the anatomy word alone means nothing.\\\\nINJURY_STEMS = [\\\\n    "tear", "torn", "rupture", "ruptur", "disrupt", "discontinu", "avuls",\\\\n    "sprain", "lesion", "lesión", "lesao", "lesão", "injur", "insufficien",\\\\n    "rotura", "ruptura", "roto", "rota", "desgarro", "riss", "einriss",\\\\n    "scheur", "letsel", "déchirure", "dechirure", "rupture", "lacerazion",\\\\n    "yırtık", "yirtik", "rüptür", "ruptur", "kopma", "puknuć", "puknuc",\\\\n    "ruptura", "razdor", "разрыв", "руптура", "скъсване", "увреж",\\\\n    "ρήξη", "ρηξη", "ρήγμα",\\\\n]\\\\n\\\\n\\\\ndef _norm(text: str) -> str:\\\\n    """Lowercase and strip combining marks, keeping non-Latin scripts intact."""\\\\n    lowered = text.lower()\\\\n    decomposed = unicodedata.normalize("NFD", lowered)\\\\n    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))\\\\n    return unicodedata.normalize("NFC", stripped)\\\\n\\\\n\\\\ndef _stem_pattern(stems: List[str]) -> re.Pattern:\\\\n    """Compile stems into one alternation matching at a word boundary."""\\\\n    escaped = sorted((re.escape(_norm(s)) for s in stems), key=len, reverse=True)\\\\n    return re.compile(r"(?<![^\\\\\\\\W\\\\\\\\d_])(?:" + "|".join(escaped) + r")", re.UNICODE)\\\\n\\\\n\\\\n# Degenerative disease stems. The three OA findings differ only by which\\\\n# compartment the disease is localised to, so they share this anchor and are\\\\n# separated by the localiser lists below. Anchoring on the compartment word\\\\n# instead does not work: "medial" appears in almost every report describing\\\\n# normal anatomy.\\\\nOA_STEMS = [\\\\n    "osteoarthr", "osteoarthrit", "arthrosis", "arthrose", "artrosis",\\\\n    "artroz", "artroza", "gonarthros", "gonartros", "gonartroz",\\\\n    "degenerative change", "degenerative disease", "cambios degenerativos",\\\\n    "degenerativne", "degenerativ", "chondropath", "condropat", "chondromalac",\\\\n    "condromalac", "chondral loss", "chondral thinning", "cartilage loss",\\\\n    "cartilage thinning", "perdida de cartilago", "adelgazamiento del cartilago",\\\\n    "knorpelverlust", "knorpelschaden", "kraakbeenverlies", "kikirdak kayb",\\\\n    "kikirdak incelme", "hrskavice", "denudacij", "osteofit", "osteophyt",\\\\n    "espolon", "spur formation", "остеоартр", "артроз", "остеофит",\\\\n    "хондромалац", "изтънен", "οστεοαρθρ", "αρθριτ", "χονδροπαθ",\\\\n    "χονδρομαλακ", "οστεοφυτ",\\\\n]\\\\n\\\\nMEDIAL_LOCALIZERS = [\\\\n    "medial", "media", "mediale", "medialen", "medialer", "medialne",\\\\n    "medijaln", "interno", "interna", "innen", "innenseit", "binnen",\\\\n    "ic kompartman", "медиальн", "медиалн", "вътреш", "εσω", "εσωτερικ",\\\\n]\\\\n\\\\nLATERAL_LOCALIZERS = [\\\\n    "lateral", "laterale", "lateralen", "lateraler", "lateraln", "externo",\\\\n    "externa", "aussen", "buiten", "dis kompartman", "латеральн", "латералн",\\\\n    "външ", "εξω", "εξωτερικ",\\\\n]\\\\n\\\\nPF_LOCALIZERS = [\\\\n    "patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",\\\\n    "retropatellar", "retropatelar", "patellar", "patelar", "patella",\\\\n    "rotulian", "rotuliana", "rotuliano", "trochlea", "troclea",\\\\n    "femoropatelarn", "patellofemoraln", "пателлофеморальн", "пателофеморалн",\\\\n    "надколенник", "капачка", "επιγονατιδ", "επιγονατιδομηριαι",\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Finding definitions\\\\n# ---------------------------------------------------------------------------\\\\n# `anatomy` locates the structure; `requires_injury` marks findings where the\\\\n# structure being named is not itself abnormal (a meniscus is always present --\\\\n# only a torn one is a positive label).\\\\nFINDINGS = {\\\\n    "ACL": {\\\\n        "anatomy": ["acl", "lca", "anterior cruciate", "ligamento cruzado anterior",\\\\n                    "voorste kruisband", "vorderes kreuzband", "kreuzband",\\\\n                    "ligament croise anterieur", "legamento crociato anteriore",\\\\n                    "on capraz bag", "onc apraz", "capraz bag", "prednji ukrizeni",\\\\n                    "prednja ukrstena", "predna krstna", "передня крестообразн",\\\\n                    "предна кръстна", "крестообразн", "prosthios chiastos",\\\\n                    "προσθιο χιαστο", "χιαστο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "MCL": {\\\\n        "anatomy": ["mcl", "medial collateral", "ligamento colateral medial",\\\\n                    "ligamento lateral interno", "mediale collaterale",\\\\n                    "innenband", "mediales kollateralband", "mediale band",\\\\n                    "ligament collateral medial", "ic kollateral",\\\\n                    "medial kollateral", "medijalni kolateralni",\\\\n                    "медиальн коллатеральн", "вътрешна колатерал",\\\\n                    "εσω πλαγιο", "εσωτερικο πλαγιο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Medial Meniscus": {\\\\n        "anatomy": ["medial menisc", "menisco medial", "menisco interno",\\\\n                    "mediale meniscus", "innenmeniskus", "medialer meniskus",\\\\n                    "menisque interne", "menisque medial", "menisco mediale",\\\\n                    "medial menisk", "ic menisk", "medijalni meniskus",\\\\n                    "medijalnog meniskusa", "медиальн мениск",\\\\n                    "вътрешния менискус", "εσω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Lateral Meniscus": {\\\\n        "anatomy": ["lateral menisc", "menisco lateral", "menisco externo",\\\\n                    "laterale meniscus", "aussenmeniskus", "lateraler meniskus",\\\\n                    "menisque externe", "menisque lateral", "menisco laterale",\\\\n                    "lateral menisk", "dis menisk", "lateralni meniskus",\\\\n                    "латеральн мениск", "външния менискус", "εξω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    # The OA family anchors on the disease stem and requires a compartment\\\\n    # localiser nearby, within a tighter window than other findings use.\\\\n    "Medial OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": MEDIAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Lateral OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": LATERAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "PF OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": PF_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Effusion": {\\\\n        "anatomy": ["effusion", "efusion", "derrame", "hydrops", "hidrartros",\\\\n                    "gelenkerguss", "erguss", "gewrichtsvocht", "vocht",\\\\n                    "epanchement", "versamento", "efuzyon", "eklem ici sivi",\\\\n                    "eklem sivisi", "izliv", "zglobni izliv", "wypot",\\\\n                    "выпот", "излив", "ставен излив", "συλλογη υγρου",\\\\n                    "αρθριτικη συλλογη", "ενδαρθρικη συλλογη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Synovitis": {\\\\n        "anatomy": ["synovit", "sinovit", "synovial thickening",\\\\n                    "engrosamiento sinovial", "synoviale verdickung",\\\\n                    "synoviale proliferation", "synovitis", "sinovitis",\\\\n                    "sinovijalis", "sinovijalna", "синовит", "υμενιτιδα",\\\\n                    "αρθρικου υμενα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Baker\\\\\\\'s": {\\\\n        "anatomy": ["baker", "popliteal cyst", "quiste poplite", "quistes poplite",\\\\n                    "bakerse cyste", "poplitealzyste", "bakerzyste",\\\\n                    "kyste poplite", "cisti poplitea", "poplitealna cista",\\\\n                    "bakerova cista", "poplitealni", "baker kisti",\\\\n                    "киста бейкера", "подколенна киста", "κυστη baker",\\\\n                    "κυστη μπεικερ", "ιγνυακη κυστη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Contusion": {\\\\n        "anatomy": ["contusion", "contusion osea", "bone contusion",\\\\n                    "bone marrow edema", "bone marrow oedema", "marrow edema",\\\\n                    "edema oseo", "edema de medula osea", "edema medular",\\\\n                    "knochenmarksodem", "knochenodem", "knochenmarködem",\\\\n                    "beenmergoedeem", "botoedeem", "oedeme osseux",\\\\n                    "edema midollare", "kemik ilig odem", "kemik odem",\\\\n                    "kontuzyon", "kostana kontuzija", "edem kosti",\\\\n                    "костномозгов отек", "костномозъчен едем", "оστικο οιδημα",\\\\n                    "οιδημα μυελου", "μυελικο οιδημα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Fracture": {\\\\n        "anatomy": ["fracture", "fractura", "fraktur", "fractuur", "frattura",\\\\n                    "kirik", "kırık", "prelom", "fraktura", "perelom",\\\\n                    "перелом", "фрактура", "счупване", "καταγμα", "καταγματ"],\\\\n        "requires_injury": False,\\\\n    },\\\\n}\\\\n\\\\n# Window, in characters, searched backwards from a term for a negation cue.\\\\nNEGATION_WINDOW = 60\\\\n# Window searched around a term for severity and certainty cues.\\\\nSEVERITY_WINDOW = 45\\\\n# Window searched forward from a term for a trailing negator.\\\\nPOSTFIX_NEGATION_WINDOW = 40\\\\n# Window searched around an anatomy match for the injury or context stem.\\\\nCONTEXT_WINDOW = 90\\\\n\\\\n_NEGATION_RE = _stem_pattern(NEGATION_CUES)\\\\n_POSTFIX_NEGATION_RE = _stem_pattern(POSTFIX_NEGATION_CUES)\\\\n_MINIMAL_RE = _stem_pattern(MINIMAL_CUES)\\\\n_UNCERTAIN_RE = _stem_pattern(UNCERTAIN_CUES)\\\\n_INJURY_RE = _stem_pattern(INJURY_STEMS)\\\\n\\\\n_COMPILED = {}\\\\nfor _name, _spec in FINDINGS.items():\\\\n    _COMPILED[_name] = {\\\\n        "anatomy": _stem_pattern(_spec["anatomy"]),\\\\n        "requires_injury": _spec["requires_injury"],\\\\n        "context": _stem_pattern(_spec["context"]) if _spec.get("context") else None,\\\\n        "context_window": _spec.get("context_window", CONTEXT_WINDOW),\\\\n        "context_is_localizer": _spec.get("context_is_localizer", False),\\\\n    }\\\\n\\\\n# Clause boundaries. Negation does not carry across them, so "no tear. Effusion\\\\n# present" must not read as a negated effusion.\\\\n_CLAUSE_SPLIT = re.compile(r"[.;:\\\\\\\\n]|\\\\\\\\bbut\\\\\\\\b|\\\\\\\\bhowever\\\\\\\\b|\\\\\\\\bpero\\\\\\\\b|\\\\\\\\bmaar\\\\\\\\b|\\\\\\\\baber\\\\\\\\b")\\\\n\\\\n\\\\ndef _clause_bounds(text: str, position: int) -> tuple:\\\\n    """Start and end of the clause containing `position`."""\\\\n    start = 0\\\\n    end = len(text)\\\\n    for match in _CLAUSE_SPLIT.finditer(text):\\\\n        if match.end() <= position:\\\\n            start = match.end()\\\\n        elif match.start() > position:\\\\n            end = match.start()\\\\n            break\\\\n    return start, end\\\\n\\\\n\\\\ndef _is_negated(text: str, position: int, length: int = 0) -> bool:\\\\n    """True when a negation cue applies to the term at `position`.\\\\n\\\\n    Searches backwards over the full cue list, and forwards over the narrower\\\\n    postfix list -- verb-final languages put the negator after the finding.\\\\n    Both searches stop at the clause boundary so a negation cannot leak into a\\\\n    neighbouring statement.\\\\n    """\\\\n    clause_start, clause_end = _clause_bounds(text, position)\\\\n\\\\n    window_start = max(clause_start, position - NEGATION_WINDOW)\\\\n    if _NEGATION_RE.search(text, window_start, position):\\\\n        return True\\\\n\\\\n    term_end = position + length\\\\n    window_end = min(clause_end, term_end + POSTFIX_NEGATION_WINDOW)\\\\n    return bool(_POSTFIX_NEGATION_RE.search(text, term_end, window_end))\\\\n\\\\n\\\\ndef _severity(text: str, position: int, length: int) -> float:\\\\n    """Confidence that the mention clears the adjudication threshold.\\\\n\\\\n    1.0 for an unqualified positive, lower when the report hedges the finding\\\\n    or calls it minimal. These scores feed the fusion step as soft targets\\\\n    rather than being thresholded here.\\\\n    """\\\\n    start = max(0, position - SEVERITY_WINDOW)\\\\n    end = min(len(text), position + length + SEVERITY_WINDOW)\\\\n    window = text[start:end]\\\\n\\\\n    score = 1.0\\\\n    if _MINIMAL_RE.search(window):\\\\n        score *= 0.35\\\\n    if _UNCERTAIN_RE.search(window):\\\\n        score *= 0.5\\\\n    return score\\\\n\\\\n\\\\ndef _has_context(text: str, position: int, length: int, pattern,\\\\n                 window: int = CONTEXT_WINDOW, clause_scoped: bool = True) -> bool:\\\\n    """True when a required injury, disease, or localiser stem sits nearby.\\\\n\\\\n    Injury context is clause-scoped: a tear described in one sentence must not\\\\n    license a finding named in the next. Compartment localisers are not, because\\\\n    reports routinely name the disease in one clause and the compartment in the\\\\n    following one ("Gonarthrosis. Most marked laterally."). Enforcing clause\\\\n    scope on localisers cost Lateral OA two thirds of its recall.\\\\n    """\\\\n    if pattern is None:\\\\n        return True\\\\n\\\\n    if clause_scoped:\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n    else:\\\\n        clause_start, clause_end = 0, len(text)\\\\n\\\\n    start = max(clause_start, position - window)\\\\n    end = min(clause_end, position + length + window)\\\\n    return bool(pattern.search(text, start, end))\\\\n\\\\n\\\\ndef extract_finding(report: str, finding: str) -> Dict:\\\\n    """Resolve one finding from one report."""\\\\n    spec = _COMPILED[finding]\\\\n    text = _norm(report)\\\\n\\\\n    context_pattern = spec["context"]\\\\n    context_window = spec["context_window"]\\\\n    localizer_based = spec["context_is_localizer"]\\\\n    if spec["requires_injury"]:\\\\n        context_pattern = _INJURY_RE\\\\n        context_window = CONTEXT_WINDOW\\\\n        localizer_based = False\\\\n\\\\n    mentioned = False\\\\n    best = None\\\\n\\\\n    for match in spec["anatomy"].finditer(text):\\\\n        position, length = match.start(), match.end() - match.start()\\\\n        in_context = _has_context(text, position, length, context_pattern,\\\\n                                  context_window,\\\\n                                  clause_scoped=not localizer_based)\\\\n\\\\n        if localizer_based and not in_context:\\\\n            # Degenerative disease was described, but not in this compartment.\\\\n            # That is silence about this finding, not evidence against it --\\\\n            # counting it as a mention would report a confident zero for a\\\\n            # compartment the radiologist never discussed.\\\\n            continue\\\\n\\\\n        mentioned = True\\\\n\\\\n        if not in_context:\\\\n            # The structure is named but nothing abnormal is asserted about it,\\\\n            # which is a normal-anatomy sentence rather than a finding.\\\\n            continue\\\\n        if _is_negated(text, position, length):\\\\n            continue\\\\n\\\\n        score = _severity(text, position, length)\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n        candidate = {\\\\n            "present": 1,\\\\n            "severity": score,\\\\n            "span": report[clause_start:clause_end].strip()[:200],\\\\n        }\\\\n        # Keep the most confident positive mention.\\\\n        if best is None or score > best["severity"]:\\\\n            best = candidate\\\\n\\\\n    if best is not None:\\\\n        return best\\\\n    if mentioned:\\\\n        return {"present": 0, "severity": 0.0, "span": None}\\\\n    return {"present": None, "severity": None, "span": None}\\\\n\\\\n\\\\ndef extract_report(report: str) -> Dict[str, Dict]:\\\\n    """Resolve all twelve findings from one report."""\\\\n    if not isinstance(report, str) or not report.strip():\\\\n        return {name: {"present": None, "severity": None, "span": None}\\\\n                for name in config.LABELS}\\\\n    return {name: extract_finding(report, name) for name in config.LABELS}\\\\n\\\\n\\\\ndef extract_frame(reports, ids=None):\\\\n    """Vectorise extraction over a series of reports into a DataFrame.\\\\n\\\\n    Returns one row per report with `<finding>` (0/1/NaN) and\\\\n    `<finding>_severity` columns.\\\\n    """\\\\n    import pandas as pd\\\\n\\\\n    records = []\\\\n    for report in reports:\\\\n        result = extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = result[name]["present"]\\\\n            row[f"{name}_severity"] = result[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    if ids is not None:\\\\n        frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\', \\\'src/labels/fuse.py\\\': \\\'"""Combine the rule-based and LLM extractors into soft training targets.\\\\n\\\\nProduces one row per study with, for each finding, a target in [0, 1] and a\\\\nsample weight. Neither is a hard label: the extractors disagree often enough,\\\\nand are wrong often enough, that training on thresholded output would teach the\\\\nmodel to reproduce extraction errors.\\\\n\\\\n**Nothing here is fitted on the 58 gold studies.** An earlier version derived\\\\nper-finding blend weights from gold AUCs and per-finding silence priors from\\\\ngold positive rates -- 24 numbers estimated from 58 samples. It scored macro AUC\\\\n0.860 in-sample and 0.812 under leave-one-out, which is worse than the LLM\\\\nextractor alone at 0.838. The apparent gain was the fit, not the fusion.\\\\n\\\\nWhat replaced it is deliberately parameter-free:\\\\n\\\\n**A fixed blend.** Targets are 0.10 x lexicon + 0.90 x LLM for every finding.\\\\nThe ratio sits on a plateau, not an argmax: with the dual-mode LLM scores,\\\\n0.00 through 0.15 all land within 0.002 on gold (0.889-0.891) and the weight\\\\nfalls away steadily above that -- the 0.30 chosen for the categorical-only\\\\nLLM cost ~0.010 against the stronger extractor. A small nonzero weight keeps\\\\nthe lexicon\\\\\\\'s independent evidence without letting it drag the ranking.\\\\n\\\\n**Silence scores zero, and is handled by the weight instead.** Measuring\\\\nP(positive | both silent) on gold and using it as the target was worse than\\\\nusing zero, on every variant tested. The concern that motivated it -- that a\\\\nsilent finding is not a confident negative -- is real, and is addressed by\\\\ngiving those studies a low sample weight rather than a fabricated target.\\\\n\\\\n**Disagreement lowers the weight rather than picking a winner.** A study where\\\\nthe extractors conflict is genuinely ambiguous and should not pull as hard as\\\\none where they concur.\\\\n"""\\\\n\\\\nfrom pathlib import Path\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.labels import calibrate, lexicon\\\\n\\\\n# Blend ratio. Chosen from a plateau, not an argmax -- see module docstring.\\\\nLEXICON_WEIGHT = 0.10\\\\nLLM_WEIGHT = 1.0 - LEXICON_WEIGHT\\\\n\\\\n# Sample weights by how much the two extractors corroborate each other.\\\\nWEIGHT_AGREE = 1.0        # both informative, same call\\\\nWEIGHT_DISAGREE = 0.4     # both informative, opposite calls\\\\nWEIGHT_SINGLE = 0.7       # one informative, one silent\\\\nWEIGHT_SILENT = 0.25      # neither informative; target is zero but barely counts\\\\n\\\\n\\\\ndef silence_rates(labels: pd.DataFrame) -> pd.Series:\\\\n    """Per finding, the share of studies neither extractor said anything about.\\\\n\\\\n    Measured on the mined corpus -- 4,349 studies -- and never on gold. That\\\\n    distinction is what separates this from the per-finding silence priors this\\\\n    module removed: those were 24 numbers fitted on 58 samples and scored 0.860\\\\n    in-sample against 0.812 leave-one-out. A rate over 4,349 studies is an\\\\n    observation, not a fit.\\\\n    """\\\\n    return pd.Series(\\\\n        {name: float((labels[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n         for name in config.LABELS})\\\\n\\\\n\\\\ndef rescale_silent_weights(labels: pd.DataFrame) -> pd.DataFrame:\\\\n    """Down-weight silence in proportion to how uninformative it is.\\\\n\\\\n    **The problem, measured on Synovitis.** 82.2% of studies are silent about\\\\n    it, the mined target averages 0.087, and the true prevalence on gold is\\\\n    46.6%. So roughly 1,666 of the 3,576 silent studies are true positives\\\\n    being trained toward zero -- and because silence is so common here, those\\\\n    studies carry 57% of the finding\\\\\\\'s gradient. The model is being taught to\\\\n    rank real positives low. Synovitis duly scores 0.682 on gold, the only\\\\n    finding where the fine-tuned model is WORSE than the retired frozen one\\\\n    (0.746), which is what fitting label noise harder looks like.\\\\n\\\\n    **The fix, and why it is not the one that failed.** Targets are untouched:\\\\n    fabricating P(positive | silent) as a target measured worse on every\\\\n    variant tested, and that stands. What changes is the *weight*, which this\\\\n    module\\\\\\\'s own docstring already names as the mechanism for silence being an\\\\n    unconfident negative. The weight becomes\\\\n\\\\n        WEIGHT_SILENT x (1 - silence_rate(finding))\\\\n\\\\n    which is one rule with no free parameters, evaluated on the mined corpus.\\\\n    Silence about ACL (9.2% of studies) stays worth nearly the full 0.25,\\\\n    because a report that discusses ligaments and omits the ACL is saying\\\\n    something. Silence about Synovitis (82.2%) falls to 0.044, because a\\\\n    corpus that mentions it in one study out of six is not saying anything.\\\\n\\\\n    Returns a copy; the target columns are passed through unchanged.\\\\n    """\\\\n    rates = silence_rates(labels)\\\\n    out = labels.copy()\\\\n    for name in config.LABELS:\\\\n        column = f"{name}_weight"\\\\n        silent = out[column] == WEIGHT_SILENT\\\\n        out.loc[silent, column] = WEIGHT_SILENT * (1.0 - rates[name])\\\\n    return out\\\\n\\\\n\\\\ndef _informative(present) -> bool:\\\\n    """False when an extractor said nothing about this finding."""\\\\n    if present is None:\\\\n        return False\\\\n    if isinstance(present, float) and np.isnan(present):\\\\n        return False\\\\n    return True\\\\n\\\\n\\\\ndef _score(present, severity) -> float:\\\\n    """One extractor\\\\\\\'s contribution to the target, in [0, 1]."""\\\\n    if not _informative(present):\\\\n        return 0.0\\\\n    if int(present) == 1:\\\\n        return float(severity) if severity == severity and severity is not None else 1.0\\\\n    return 0.0\\\\n\\\\n\\\\ndef lexicon_frame(reports, ids) -> pd.DataFrame:\\\\n    """Run the rule-based extractor over a set of reports."""\\\\n    records = []\\\\n    for report in reports:\\\\n        extracted = lexicon.extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = extracted[name]["present"]\\\\n            row[f"{name}_severity"] = extracted[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\\n\\\\ndef fuse(lex: pd.DataFrame, llm: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend two extractor frames into soft targets and sample weights."""\\\\n    merged = lex.merge(llm, on=config.ID_COL, suffixes=("_lex", "_llm"))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        llm_present = merged[f"{name}_llm"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity_lex"].to_numpy()\\\\n        llm_sev = merged[f"{name}_severity_llm"].to_numpy()\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            llm_score = _score(llm_present[i], llm_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            llm_ok = _informative(llm_present[i])\\\\n            if lex_ok and llm_ok:\\\\n                agree = (lex_score > 0) == (llm_score > 0)\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\ndef fuse_scores(lex: pd.DataFrame, llm_scores: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend the lexicon frame with an already-combined LLM score frame.\\\\n\\\\n    `llm_scores` is the output of `mode_fusion.combine` (or\\\\n    `mode_fusion.from_categorical`): per finding a score in [0, 1], an\\\\n    `_informative` flag, and a `_call` boolean. The weight semantics mirror\\\\n    `fuse`: agreement compares the lexicon\\\\\\\'s call with the LLM\\\\\\\'s call.\\\\n    """\\\\n    merged = lex.merge(llm_scores, on=config.ID_COL, suffixes=("_lex", ""))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity"].to_numpy()\\\\n        llm_score = merged[name].to_numpy(dtype=float)\\\\n        llm_ok = merged[f"{name}_informative"].to_numpy(dtype=bool)\\\\n        llm_call = merged[f"{name}_call"].to_numpy(dtype=bool)\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score[i]\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            if lex_ok and llm_ok[i]:\\\\n                agree = (lex_score > 0) == llm_call[i]\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok[i]:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\n# Order-keeper inside the reference distribution\\\\\\\'s tie blocks. Small enough\\\\n# that the restored shape dominates the target value; the epsilon term only\\\\n# arranges studies within a block.\\\\nSHARPEN_EPSILON = 0.02\\\\n\\\\n\\\\ndef sharpen_targets(fused: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:\\\\n    """Restore `reference`\\\\\\\'s target distribution, keeping `fused`\\\\\\\'s ordering.\\\\n\\\\n    The dual-mode targets rank studies better than the categorical fusion\\\\n    (0.891 vs 0.854 on gold) but drained the confident mass out of the\\\\n    distribution -- extreme targets fell from 73% to 60% of slots and the\\\\n    model trained on them regressed on the leaderboard (0.845 -> 0.827).\\\\n    AUC is invariant under strictly monotone maps, so the ordering gain and\\\\n    the crisp shape are separable: each finding\\\\\\\'s targets are quantile-mapped\\\\n    onto the reference marginal, plus an epsilon-scaled rank term that keeps\\\\n    the map strictly increasing where the reference has ties. Sample weights\\\\n    are not touched.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    out = fused.copy()\\\\n    for name in config.LABELS:\\\\n        targets = fused[name].to_numpy(dtype=float)\\\\n        ref = np.sort(reference[name].to_numpy(dtype=float))\\\\n        ranks = mode_fusion._tied_ranks_01(targets)\\\\n        mapped = np.quantile(ref, np.clip(ranks, 0.0, 1.0))\\\\n        out[name] = np.clip(\\\\n            (mapped + SHARPEN_EPSILON * ranks) / (1.0 + SHARPEN_EPSILON),\\\\n            0.0, 1.0)\\\\n    return out\\\\n\\\\n\\\\ndef extractor_aucs(gold: pd.DataFrame, lex_gold: pd.DataFrame,\\\\n                   llm_gold: pd.DataFrame) -> pd.DataFrame:\\\\n    """Diagnostic only: how each extractor scores on gold.\\\\n\\\\n    These numbers are reported, never fed back into the targets. Letting them\\\\n    set blend weights is exactly the overfit this module was rewritten to avoid.\\\\n    """\\\\n    rows = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        lex_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(lex_gold[name], lex_gold[f"{name}_severity"])])\\\\n        llm_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(llm_gold[name], llm_gold[f"{name}_severity"])])\\\\n        rows.append({\\\\n            "finding": name,\\\\n            "lexicon_auc": calibrate._auc(truth, lex_scores),\\\\n            "llm_auc": calibrate._auc(truth, llm_scores),\\\\n        })\\\\n    return pd.DataFrame(rows).set_index("finding")\\\\n\\\\n\\\\ndef build(train_csv=None, llm_mined=None, llm_gold=None,\\\\n          probs_mined=None, probs_gold=None,\\\\n          out_path: Optional[str] = None):\\\\n    """Full pipeline: extract, fuse, and score against gold.\\\\n\\\\n    When the probability-mode parquets exist alongside the categorical ones,\\\\n    the two modes are combined by `mode_fusion` before the lexicon blend.\\\\n    Otherwise this is the original categorical-only fusion, so the pipeline\\\\n    still runs on a machine that never downloaded the probability dumps.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    gold = calibrate.load_gold(train_csv)\\\\n    unlabelled = calibrate.load_unlabelled(train_csv)\\\\n\\\\n    mined_frame = pd.read_parquet(\\\\n        llm_mined or (config.META_DIR / "llm_labels.parquet"))\\\\n    gold_frame = pd.read_parquet(\\\\n        llm_gold or (config.META_DIR / "llm_labels_gold.parquet"))\\\\n\\\\n    # Align to CSV order so positional comparisons stay valid.\\\\n    gold_frame = (gold_frame.set_index(config.ID_COL)\\\\n                  .loc[gold[config.ID_COL]].reset_index())\\\\n    mined_frame = (mined_frame.set_index(config.ID_COL)\\\\n                   .loc[unlabelled[config.ID_COL]].reset_index())\\\\n\\\\n    probs_mined_path = probs_mined or (config.META_DIR / "llm_probs.parquet")\\\\n    probs_gold_path = probs_gold or (config.META_DIR / "llm_probs_gold.parquet")\\\\n    use_probs = (Path(probs_mined_path).exists()\\\\n                 and Path(probs_gold_path).exists())\\\\n\\\\n    if use_probs:\\\\n        probs_mined_frame = pd.read_parquet(probs_mined_path)\\\\n        probs_gold_frame = pd.read_parquet(probs_gold_path)\\\\n        scores_mined = mode_fusion.combine(mined_frame, probs_mined_frame)\\\\n        scores_gold = mode_fusion.combine(gold_frame, probs_gold_frame)\\\\n    else:\\\\n        scores_mined = mode_fusion.from_categorical(mined_frame)\\\\n        scores_gold = mode_fusion.from_categorical(gold_frame)\\\\n\\\\n    lex_gold = lexicon_frame(gold["Report"], gold[config.ID_COL])\\\\n    lex_mined = lexicon_frame(unlabelled["Report"], unlabelled[config.ID_COL])\\\\n\\\\n    fused = fuse_scores(lex_mined, scores_mined)\\\\n    fused_gold = fuse_scores(lex_gold, scores_gold)\\\\n\\\\n    if use_probs:\\\\n        # The categorical-only fusion of the same inputs supplies the shape\\\\n        # the dual-mode targets are sharpened back onto.\\\\n        ref_mined = fuse_scores(\\\\n            lex_mined, mode_fusion.from_categorical(mined_frame))\\\\n        ref_gold = fuse_scores(\\\\n            lex_gold, mode_fusion.from_categorical(gold_frame))\\\\n        fused = sharpen_targets(fused, ref_mined)\\\\n        fused_gold = sharpen_targets(fused_gold, ref_gold)\\\\n\\\\n    aucs = extractor_aucs(gold, lex_gold, gold_frame)\\\\n\\\\n    if out_path:\\\\n        fused.to_parquet(out_path, index=False)\\\\n\\\\n    return fused, fused_gold, aucs\\\\n\\\\n\\\\ndef report(gold: pd.DataFrame, fused_gold: pd.DataFrame,\\\\n           aucs: pd.DataFrame, fused: pd.DataFrame) -> str:\\\\n    """Per-finding summary of the fused targets.\\\\n\\\\n    The fused AUC column is an honest estimate: no parameter in this module was\\\\n    estimated from the gold studies, so in-sample and held-out are the same.\\\\n    """\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'lexAUC\\\\\\\':>7s} {\\\\\\\'llmAUC\\\\\\\':>7s} {\\\\\\\'fusedAUC\\\\\\\':>9s} "\\\\n        f"{\\\\\\\'delta\\\\\\\':>7s} {\\\\\\\'mean w\\\\\\\':>7s} {\\\\\\\'silent\\\\\\\':>7s}",\\\\n        "-" * 72,\\\\n    ]\\\\n    fused_aucs = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        auc = calibrate._auc(truth, fused_gold[name].to_numpy())\\\\n        fused_aucs.append(auc)\\\\n        best_single = max(aucs.loc[name, "lexicon_auc"], aucs.loc[name, "llm_auc"])\\\\n        silent = float((fused[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n        lines.append(\\\\n            f"{name:18s} {aucs.loc[name, \\\\\\\'lexicon_auc\\\\\\\']:7.3f} "\\\\n            f"{aucs.loc[name, \\\\\\\'llm_auc\\\\\\\']:7.3f} {auc:9.3f} "\\\\n            f"{auc - best_single:+7.3f} "\\\\n            f"{fused[f\\\\\\\'{name}_weight\\\\\\\'].mean():7.2f} {silent:6.0%}"\\\\n        )\\\\n    lines.append("-" * 72)\\\\n    lines.append(\\\\n        f"macro: fused {np.nanmean(fused_aucs):.3f}   "\\\\n        f"lexicon {aucs[\\\\\\\'lexicon_auc\\\\\\\'].mean():.3f}   "\\\\n        f"LLM {aucs[\\\\\\\'llm_auc\\\\\\\'].mean():.3f}"\\\\n    )\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    out = config.META_DIR / "train_labels.parquet"\\\\n    fused, fused_gold, aucs = build(out_path=out)\\\\n    gold = calibrate.load_gold()\\\\n    print(report(gold, fused_gold, aucs, fused))\\\\n    print(f"\\\\\\\\nwrote {out}  {fused.shape}")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/lightning.py\\\': \\\'"""Lightning wrapper for the end-to-end fine-tuning run.\\\\n\\\\nKaggle offers a T4 x2 accelerator, and the training loop is GPU-bound at\\\\n~220 images/second, so a second device is close to a free halving of\\\\nwall-clock. Lightning is here for the distributed plumbing, not for the\\\\nabstraction: the semantics below are a faithful port of the hand-written loop\\\\nin notebook 08, and any difference in the resulting score is a bug rather\\\\nthan a design choice.\\\\n\\\\n**Everything in this module lives here rather than in a notebook cell.**\\\\nLightning\\\\\\\'s `ddp_notebook` strategy forks, so children inherit memory and the\\\\nmodule is not pickled on that path -- but dataloader workers and the spawn\\\\nstart method both do pickle, and a class defined in a notebook cell belongs to\\\\n`__main__` and does not survive that. Keeping these definitions importable as\\\\n`src.model.lightning` makes the module correct under either start method; the\\\\nbootstrap writes the files to disk before importing them. The same rule put\\\\nthe cache\\\\\\\'s process-pool worker in `src/data/cache.py`.\\\\n\\\\nA related consequence of fork, and the reason the notebook is careful about\\\\nit: **CUDA must not be initialised before `Trainer.fit`**. A fork inheriting a\\\\nlive CUDA context fails in the child. `torch.cuda.get_device_name` and\\\\n`get_device_properties` both call `_lazy_init`; `device_count` and\\\\n`is_available` do not.\\\\n\\\\nThree DDP hazards are handled explicitly, because each one runs clean and\\\\nsilently scores worse:\\\\n\\\\n**Memory-mapped shards are never serialised.** The cache is ~29 GB across\\\\nthree memmaps. Holding them as attributes would carry the array bytes into any\\\\nprocess the dataset is pickled into. They are opened lazily per process and\\\\ndropped from `__getstate__`, which also means each dataloader worker maps the\\\\nfiles itself rather than sharing a handle across a fork.\\\\n\\\\n**AUC is computed over gathered predictions.** It is a ranking metric, so\\\\naveraging two per-rank AUCs is not the AUC of the whole set. Predictions and\\\\ntargets are all-gathered before scoring. The distributed sampler pads the\\\\nfinal batch by repeating samples, so at most one study per rank is counted\\\\ntwice out of ~870 -- below the resolution of the measurement.\\\\n\\\\n**The effective batch is held constant.** Each rank receives `batch_size`\\\\nstudies, so the effective batch is `batch_size x devices x accumulation`.\\\\nHalving accumulation when moving from one device to two keeps the optimiser\\\\nseeing the same eight studies per step, and keeps the OneCycleLR step count\\\\nunchanged.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport torch\\\\nfrom torch.utils.data import Dataset\\\\n\\\\nfrom src import config\\\\nfrom src.model import augment as augment_mod\\\\nfrom src.model import averaging\\\\nfrom src.model import conditional\\\\nfrom src.model import dataset as knee_dataset\\\\nfrom src.model import net\\\\n\\\\ntry:  # Lightning 2.x ships under both names depending on the install.\\\\n    import lightning.pytorch as pl\\\\nexcept ImportError:  # pragma: no cover - exercised only on Kaggle/Colab\\\\n    import pytorch_lightning as pl\\\\n\\\\n\\\\ndef auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via the rank statistic, with ties averaged.\\\\n\\\\n    Duplicated from `labels.calibrate` rather than imported: only the model\\\\n    package is materialised into the training notebook, and a metric that\\\\n    disagrees with the one the labels were scored by would be worse than the\\\\n    duplication.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef macro_auc(truth: np.ndarray, probs: np.ndarray) -> float:\\\\n    """Mean per-finding AUC, ignoring findings with no positives present."""\\\\n    scores = [auc(truth[:, i], probs[:, i]) for i in range(truth.shape[1])]\\\\n    return float(np.nanmean(scores))\\\\n\\\\n\\\\nclass StudyVolumeDataset(Dataset):\\\\n    """Study volumes with an even slice subsample and intensity jitter.\\\\n\\\\n    Slices are subsampled evenly rather than at random so a validation score\\\\n    is reproducible. Horizontal flips are deliberately not used: after\\\\n    canonicalisation a flip re-mirrors the knee and swaps medial with lateral,\\\\n    silently corrupting four of the twelve labels.\\\\n\\\\n    **`resample` is the training-only exception, and it exists because the\\\\n    cache is larger than what training reads.** The cache holds 24 slices of\\\\n    each of 24,371 series -- 584,904 slice-images. A run at 16 slices x 5\\\\n    series reads 341,728 of them, the same ones every epoch: the fixed\\\\n    `keep` discards 8 slices of every series and the deterministic series cap\\\\n    discards 3,013 series outright. The remaining 42% is not new data, but it\\\\n    is unseen pixels of the same 4,349 studies, and reaching it costs no\\\\n    storage, no upload and no extra forward passes. `resample=False`\\\\n    reproduces every measured result exactly; the flag is what makes the two\\\\n    comparable.\\\\n    """\\\\n\\\\n    def __init__(self, study_index, studies: Sequence[str],\\\\n                 cache_paths: Sequence[str], slices_per_series: int,\\\\n                 augment=False, resample: bool = False):\\\\n        self.index = study_index\\\\n        self.studies = list(studies)\\\\n        self.cache_paths = list(cache_paths)\\\\n        self.slices_per_series = slices_per_series\\\\n        # Per-epoch resampling of *which* slices and series are read. Training\\\\n        # only -- a validation score has to be reproducible to be a gate.\\\\n        self.resample = resample\\\\n        # `augment` is either a bool -- the legacy single intensity scale that\\\\n        # every measured result was trained under -- or an AugmentConfig. The\\\\n        # bool path is kept byte-for-byte rather than expressed as a config so\\\\n        # that reproducing an old run does not depend on the two being\\\\n        # equivalent. See src/model/augment.py.\\\\n        self.augment = augment\\\\n        self.keep = np.linspace(0, config.CACHE_SLICES - 1,\\\\n                                slices_per_series).round().astype(int)\\\\n        self._shards: Optional[Dict[str, np.ndarray]] = None\\\\n\\\\n    @property\\\\n    def shards(self) -> Dict[str, np.ndarray]:\\\\n        """Open the memmaps on first use, once per process."""\\\\n        if self._shards is None:\\\\n            self._shards = {path: np.load(path, mmap_mode="r")\\\\n                            for path in self.cache_paths}\\\\n        return self._shards\\\\n\\\\n    def __getstate__(self):\\\\n        # Never serialise the memmaps into a spawned process or a dataloader\\\\n        # worker; each one reopens them from the paths instead.\\\\n        state = self.__dict__.copy()\\\\n        state["_shards"] = None\\\\n        return state\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def _resampled_keep(self, available: int, rng) -> np.ndarray:\\\\n        """A fresh even-ish slice subsample: drop one slice per even group.\\\\n\\\\n        Phrased as a drop rather than a pick so the result is exactly\\\\n        `slices_per_series` distinct indices in ascending order. Order matters:\\\\n        `stack_2p5d` hands the backbone each slice with its two neighbours, and\\\\n        a shuffled stack makes those neighbours meaningless -- the same reason\\\\n        `augment.py` implements `slice_reverse` and not slice permutation.\\\\n        Coverage matches `np.linspace` in expectation, so this is a resample of\\\\n        the reference geometry rather than a different one.\\\\n        """\\\\n        n_drop = available - self.slices_per_series\\\\n        groups = np.array_split(np.arange(available), n_drop)\\\\n        drop = {int(rng.choice(group)) for group in groups}\\\\n        return np.array([i for i in range(available) if i not in drop])\\\\n\\\\n    def __getitem__(self, i: int):\\\\n        rng = np.random.default_rng(np.random.randint(0, 2**31))\\\\n        item = knee_dataset.load_study(\\\\n            self.index, self.studies[i], self.shards,\\\\n            series_rng=rng if self.resample else None)\\\\n\\\\n        available = item["volumes"].shape[1]\\\\n        if available > self.slices_per_series:\\\\n            # A thinned cache is already at the target count; only subsample a\\\\n            # full one, or the second cut would drop slices twice.\\\\n            keep = (self._resampled_keep(available, rng) if self.resample\\\\n                    else np.linspace(0, available - 1,\\\\n                                     self.slices_per_series).round().astype(int))\\\\n            item["volumes"] = item["volumes"][:, keep]\\\\n        elif available < self.slices_per_series:\\\\n            # The attention pool masks over the slice axis, so a short volume\\\\n            # trains happily on fewer slices than configured and says nothing.\\\\n            raise ValueError(\\\\n                f"cache supplies {available} slices, slices_per_series is "\\\\n                f"{self.slices_per_series} -- attach the full 24-slice "\\\\n                "shards, or lower the setting")\\\\n\\\\n        if isinstance(self.augment, augment_mod.AugmentConfig):\\\\n            # The same generator that chose the slices and series, so one item\\\\n            # draws from one stream. Its seed comes from the worker\\\\\\\'s numpy\\\\n            # state, which advances rather than resetting between epochs --\\\\n            # that is what makes the resample per-epoch and not per-study.\\\\n            item = augment_mod.augment_study(item, rng, self.augment)\\\\n        elif self.augment:\\\\n            # Legacy path: one intensity scale for the whole study. Every\\\\n            # result on record was trained through exactly this.\\\\n            scale = np.random.uniform(0.9, 1.1)\\\\n            item["volumes"] = np.clip(\\\\n                item["volumes"].astype(np.float32) * scale,\\\\n                0, 255).astype(np.uint8)\\\\n        return item\\\\n\\\\n\\\\nclass KneeLitModule(pl.LightningModule):\\\\n    """The fine-tuning step, loss, schedule, and epoch-level scoring.\\\\n\\\\n    Two validation dataloaders are expected, and which is which matters:\\\\n    index 0 is the validation fold scored against mined labels binarised at\\\\n    0.5, and it is the only thing allowed to select a checkpoint. Index 1 is\\\\n    the gold 58, logged every epoch because it is nearly free and shows\\\\n    whether the run is converging -- and never monitored. Selecting on gold\\\\n    turns the project\\\\\\\'s one honest measuring stick into a model-selection set;\\\\n    an earlier version of this loop did exactly that and inflated its own\\\\n    reported figure by an estimated 0.01-0.03.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: torch.nn.Module, feature_dim: int,\\\\n                 lr: float = 1e-4, weight_decay: float = 1e-4,\\\\n                 total_steps: int = 1000, max_lr_multiplier: float = 3.0,\\\\n                 report_dim: Optional[int] = None,\\\\n                 lambda_report: float = 0.0, lambda_rank: float = 0.0,\\\\n                 rank_queue_size: int = 256, rank_min_gap: float = 0.25,\\\\n                 aux_mode: str = "fixed", aux_gate: str = "binary",\\\\n                 backbone_lr_scale: float = 1.0,\\\\n                 input_mean=None, input_std=None, gated_pool: bool = False,\\\\n                 schedule: str = "onecycle",\\\\n                 warmup_steps: Optional[int] = None,\\\\n                 label_smoothing: float = 0.0, symmetric_beta: float = 0.0,\\\\n                 pos_weight: Optional[Sequence[float]] = None,\\\\n                 lambda_stratified_rank: float = 0.0,\\\\n                 stratified_findings: Sequence[str] = ("Synovitis",),\\\\n                 stratify_by: str = "Effusion",\\\\n                 stratified_caliper: float = 0.15,\\\\n                 lambda_coupling: float = 0.0,\\\\n                 coupling_target: Optional[float] = None,\\\\n                 residual_beta: Optional[float] = None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        if classwise_pool and (lambda_report > 0 or aux_mode != "fixed"):\\\\n            raise ValueError("classwise pooling requires fixed mode without report loss")\\\\n        if aux_mode not in ("fixed", "cosine"):\\\\n            raise ValueError(f"aux_mode must be fixed|cosine, got {aux_mode!r}")\\\\n        if aux_gate not in ("binary", "weighted"):\\\\n            raise ValueError(f"aux_gate must be binary|weighted, got {aux_gate!r}")\\\\n        if schedule not in ("onecycle", "cosine"):\\\\n            raise ValueError(\\\\n                f"schedule must be onecycle|cosine, got {schedule!r}")\\\\n        if warmup_steps is not None and not 0 <= warmup_steps < total_steps:\\\\n            raise ValueError(\\\\n                f"warmup_steps {warmup_steps} must be under total_steps "\\\\n                f"{total_steps}")\\\\n        if lambda_coupling > 0 and coupling_target is None:\\\\n            raise ValueError(\\\\n                "lambda_coupling needs an explicit coupling_target. The "\\\\n                "findings genuinely co-occur (gold OR 6.09, p = 0.003); a "\\\\n                "penalty toward independence optimises to a value the truth "\\\\n                "contradicts. Pass the observed coupling.")\\\\n        for name in tuple(stratified_findings) + (stratify_by,):\\\\n            if name not in config.LABELS:\\\\n                raise ValueError(f"unknown finding {name!r}")\\\\n\\\\n        self.save_hyperparameters(ignore=["backbone"])\\\\n        # Stage 1\\\\\\\'s residual parameterisation, built into the model so it\\\\n        # travels in the checkpoint rather than having to be reconstructed at\\\\n        # every scoring site. See KneeNet.couple.\\\\n        residual_pairs = None\\\\n        if residual_beta is not None:\\\\n            residual_pairs = [(config.LABELS.index(name),\\\\n                               config.LABELS.index(stratify_by))\\\\n                              for name in stratified_findings]\\\\n        self.model = net.KneeNet(backbone, feature_dim,\\\\n                                 input_mean=input_mean, input_std=input_std,\\\\n                                 gated_pool=gated_pool,\\\\n                                 classwise_pool=classwise_pool,\\\\n                                 slice_transformer=slice_transformer,\\\\n                                 residual_pairs=residual_pairs,\\\\n                                 residual_beta=residual_beta)\\\\n        self.backbone_lr_scale = backbone_lr_scale\\\\n        self.lr = lr\\\\n        self.weight_decay = weight_decay\\\\n        self.total_steps = total_steps\\\\n        self.max_lr_multiplier = max_lr_multiplier\\\\n        self.lambda_report = lambda_report\\\\n        self.lambda_rank = lambda_rank\\\\n        self.rank_queue_size = rank_queue_size\\\\n        self.rank_min_gap = rank_min_gap\\\\n        self.aux_mode = aux_mode\\\\n        self.aux_gate = aux_gate\\\\n        self.schedule = schedule\\\\n        self.warmup_steps = warmup_steps\\\\n        self.label_smoothing = label_smoothing\\\\n        self.symmetric_beta = symmetric_beta\\\\n        self.lambda_stratified_rank = lambda_stratified_rank\\\\n        self.stratified_findings = tuple(stratified_findings)\\\\n        self.stratify_by = stratify_by\\\\n        self.stratified_index = config.LABELS.index(stratify_by)\\\\n        self.stratified_caliper = stratified_caliper\\\\n        self.lambda_coupling = lambda_coupling\\\\n        self.coupling_target = coupling_target\\\\n\\\\n        # A buffer rather than a plain tensor: it has to follow the module to\\\\n        # the GPU, and it is not learned. Registered as None when unused so\\\\n        # the state dict is unchanged for every run that does not set it --\\\\n        # and it lives on the wrapper, not on `KneeNet`, so it never reaches\\\\n        # the exported checkpoint the submission loads.\\\\n        if pos_weight is not None:\\\\n            self.register_buffer(\\\\n                "pos_weight", torch.tensor(list(pos_weight), dtype=torch.float32))\\\\n        else:\\\\n            self.register_buffer("pos_weight", None)\\\\n\\\\n        # The report head is an attribute of the *wrapper*, deliberately: the\\\\n        # exported state dict keeps only `model.`-prefixed keys, so auxiliary\\\\n        # parameters vanish at export and notebook 10 keeps loading a bare\\\\n        # KneeNet. Created only when actually trained against -- under DDP a\\\\n        # parameter that never receives gradient is an error, not a no-op.\\\\n        if report_dim is not None and lambda_report > 0:\\\\n            self.report_head = torch.nn.Sequential(\\\\n                torch.nn.LayerNorm(feature_dim),\\\\n                torch.nn.Linear(feature_dim, report_dim))\\\\n        else:\\\\n            self.report_head = None\\\\n\\\\n        # Cross-batch queue for the ranking loss. At one study per rank there\\\\n        # are no pairs inside a batch, so current logits are ranked against\\\\n        # detached logits from recent steps (XBM-style). Slightly stale scores\\\\n        # are the price; per-rank and reset each epoch.\\\\n        self._rank_queue: List[Dict[str, torch.Tensor]] = []\\\\n\\\\n        # Not `_buffers`: nn.Module reserves that name for registered\\\\n        # buffers, and shadowing it corrupts state_dict and .to(device).\\\\n        self._val_outputs: Dict[int, List[Dict[str, torch.Tensor]]] = {0: [], 1: []}\\\\n\\\\n    def forward(self, volumes, meta, mask):\\\\n        return self.model(volumes, meta, mask)\\\\n\\\\n    def on_train_epoch_start(self):\\\\n        self._rank_queue = []\\\\n\\\\n    def _report_loss(self, pooled: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Cosine distance between the projected study embedding and the\\\\n        report\\\\\\\'s sentence embedding, over studies that have one.\\\\n\\\\n        The head must run even when the whole batch lacks reports: DDP\\\\n        requires every parameter to join every backward pass.\\\\n        """\\\\n        predicted = self.report_head(pooled.float())\\\\n        if "report" not in batch:\\\\n            return predicted.sum() * 0.0\\\\n        predicted = torch.nn.functional.normalize(predicted, dim=-1)\\\\n        target = torch.nn.functional.normalize(batch["report"].float(), dim=-1)\\\\n        distance = 1.0 - (predicted * target).sum(dim=-1)\\\\n        mask = batch["report_mask"].float()\\\\n        return (distance * mask).sum() / mask.sum().clamp(min=1.0)\\\\n\\\\n    def _rank_reference(self):\\\\n        """The queued detached scores both ranking losses rank against."""\\\\n        if not self._rank_queue:\\\\n            return None\\\\n        return {key: torch.cat([q[key] for q in self._rank_queue])\\\\n                for key in ("scores", "targets", "weights", "strata")}\\\\n\\\\n    def _rank_push(self, logits: torch.Tensor, batch) -> None:\\\\n        """Add this step\\\\\\\'s studies to the queue, once per training step.\\\\n\\\\n        Pushed after every ranking term has been computed, so a study never\\\\n        ranks against itself. `strata` is the predicted probability of the\\\\n        stratifying finding -- effusion -- carried because the stratified loss\\\\n        needs the reference studies\\\\\\\' effusion scores to decide which pairs are\\\\n        comparable, and recomputing them would mean a second forward pass over\\\\n        studies that have already left the batch.\\\\n        """\\\\n        self._rank_queue.append({\\\\n            "scores": logits.detach().float(),\\\\n            "targets": batch["targets"].detach().float(),\\\\n            "weights": batch["weights"].detach().float(),\\\\n            "strata": torch.sigmoid(\\\\n                logits[:, self.stratified_index].detach().float()),\\\\n        })\\\\n        while sum(len(q["scores"]) for q in self._rank_queue) > self.rank_queue_size:\\\\n            self._rank_queue.pop(0)\\\\n\\\\n    def _rank_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        return net.pairwise_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _stratified_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Stage 1: rank only within an effusion stratum.\\\\n\\\\n        Conditional AUC is within-stratum pair concordance, so this optimises\\\\n        the metric the finding is judged on rather than a proxy for it. Pairs\\\\n        are matched on the model\\\\\\\'s own predicted effusion score, detached: the\\\\n        mined effusion label has its own error, and a hard split would inherit\\\\n        it at the boundary.\\\\n        """\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        strata = torch.sigmoid(scores[:, self.stratified_index].detach())\\\\n        return net.stratified_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            strata, reference["strata"],\\\\n            self.stratified_findings, self.stratified_caliper,\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _coupling_loss(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Hold the head-to-head coupling at the truth\\\\\\\'s value, not at zero."""\\\\n        reference = self._rank_reference()\\\\n        queued = (reference["scores"] if reference is not None\\\\n                  else logits.new_zeros((0, logits.shape[1])))\\\\n        total = logits.sum() * 0.0\\\\n        for name in self.stratified_findings:\\\\n            total = total + net.coupling_penalty(\\\\n                logits.float(), queued, config.LABELS.index(name),\\\\n                self.stratified_index, self.coupling_target)\\\\n        return total\\\\n\\\\n    def _aux_cosine(self, main_loss: torch.Tensor, aux_loss: torch.Tensor,\\\\n                    pooled: torch.Tensor) -> torch.Tensor:\\\\n        """cos between the two losses\\\\\\\' gradients at the shared bottleneck.\\\\n\\\\n        Du et al., "Adapting Auxiliary Losses Using Gradient Similarity"\\\\n        (arXiv:1812.02224), Algorithm 1: apply the auxiliary gradient to the\\\\n        shared parameters only while it points into the same half-space as\\\\n        the main gradient, which guarantees convergence to the main task\\\\\\\'s\\\\n        critical points and blocks negative transfer.\\\\n\\\\n        The paper measures that cosine over the shared parameters theta.\\\\n        Here every shared parameter is reached *only* through `pooled` --\\\\n        `forward_pooled` returns `head(pooled), pooled`, and both losses hang\\\\n        off that one tensor -- so the shared gradients are J^T a and J^T b for\\\\n        the same Jacobian J = d pooled/d theta. Taking the cosine at the\\\\n        bottleneck asks the same question ("does the auxiliary want to move\\\\n        the shared representation the way the main task does?") for the price\\\\n        of two backward passes through the two small heads instead of two\\\\n        through ConvNeXt. It is a proxy, not an identity: J^T can rotate the\\\\n        pair, so agreement here is not arithmetically agreement at theta.\\\\n        The sign is what Algorithm 1 consumes, and the paper itself keeps\\\\n        only the sign because minibatch cosines are too noisy to trust\\\\n        beyond it.\\\\n\\\\n        retain_graph is required: the real backward still has to run\\\\n        afterwards over the same graph.\\\\n\\\\n        Both losses are scaled by a constant first. Training runs in\\\\n        `16-mixed`, and these probe gradients bypass Lightning\\\\\\\'s GradScaler,\\\\n        so an unscaled fp16 backward can flush small values to zero -- which\\\\n        reads as cosine 0, i.e. "aligned", i.e. silently back to a fixed\\\\n        lambda. Cosine is scale-invariant, so scaling both sides costs\\\\n        nothing and keeps the values off the floor.\\\\n        """\\\\n        scale = 1024.0\\\\n        grad_main, = torch.autograd.grad(main_loss * scale, pooled,\\\\n                                         retain_graph=True)\\\\n        grad_aux, = torch.autograd.grad(aux_loss * scale, pooled,\\\\n                                        retain_graph=True)\\\\n        cosine = torch.nn.functional.cosine_similarity(\\\\n            grad_main.flatten().float(), grad_aux.flatten().float(), dim=0)\\\\n        # Ranks must agree, or the "binary" gate is silently a soft one: DDP\\\\n        # averages the two ranks\\\\\\\' gradients, so a per-rank decision would\\\\n        # apply the auxiliary at half strength whenever they disagreed.\\\\n        # torch.distributed rather than self.all_gather, which requires an\\\\n        # attached Trainer and would make this untestable in isolation.\\\\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\\\\n            cosine = cosine.clone()\\\\n            torch.distributed.all_reduce(cosine, op=torch.distributed.ReduceOp.SUM)\\\\n            cosine = cosine / torch.distributed.get_world_size()\\\\n        return cosine\\\\n\\\\n    def _aux_weight(self, cosine: torch.Tensor) -> torch.Tensor:\\\\n        """Algorithm 1 (binary) or Algorithm 2 (cosine-weighted)."""\\\\n        if self.aux_gate == "weighted":\\\\n            return cosine.clamp(min=0.0)\\\\n        return (cosine >= 0).to(cosine.dtype)\\\\n\\\\n    def training_step(self, batch, batch_idx):\\\\n        logits, pooled = self.model.forward_pooled(\\\\n            batch["volumes"], batch["meta"], batch["mask"])\\\\n        loss = net.weighted_bce(\\\\n            logits, batch["targets"], batch["weights"],\\\\n            label_smoothing=self.label_smoothing,\\\\n            pos_weight=self.pos_weight,\\\\n            symmetric_beta=self.symmetric_beta)\\\\n        self.log("bce_loss", loss, on_step=False, on_epoch=True,\\\\n                 sync_dist=True, batch_size=len(logits))\\\\n        main_loss = loss\\\\n\\\\n        if self.report_head is not None:\\\\n            report_loss = self._report_loss(pooled, batch)\\\\n            weight = self.lambda_report\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, report_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                # The diagnostic worth having even when the gate never fires:\\\\n                # a report loss that is mostly anti-aligned explains a fixed\\\\n                # lambda hurting the leaderboard while helping validation.\\\\n                self.log("report_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n                self.log("report_gate", (cosine >= 0).float(), on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * report_loss\\\\n            self.log("report_loss", report_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0:\\\\n            rank_loss = self._rank_loss(logits, batch)\\\\n            weight = self.lambda_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, rank_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("rank_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * rank_loss\\\\n            self.log("rank_loss", rank_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_stratified_rank > 0:\\\\n            stratified_loss = self._stratified_loss(logits, batch)\\\\n            weight = self.lambda_stratified_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, stratified_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("stratified_cosine", cosine, on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * stratified_loss\\\\n            self.log("stratified_rank_loss", stratified_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_coupling > 0:\\\\n            coupling_loss = self._coupling_loss(logits)\\\\n            loss = loss + self.lambda_coupling * coupling_loss\\\\n            self.log("coupling_loss", coupling_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0 or self.lambda_stratified_rank > 0 \\\\\\\\\\\\n                or self.lambda_coupling > 0:\\\\n            self._rank_push(logits, batch)\\\\n\\\\n        self.log("train_loss", loss, prog_bar=True, on_step=True,\\\\n                 on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n        return loss\\\\n\\\\n    def validation_step(self, batch, batch_idx, dataloader_idx=0):\\\\n        logits = self(batch["volumes"], batch["meta"], batch["mask"])\\\\n        self._val_outputs[dataloader_idx].append({\\\\n            "probs": torch.sigmoid(logits.float()),\\\\n            "targets": batch["targets"].float(),\\\\n        })\\\\n\\\\n    def _gathered(self, dataloader_idx: int):\\\\n        """All-gather one buffer and return (truth, probs) as numpy."""\\\\n        entries = self._val_outputs[dataloader_idx]\\\\n        if not entries:\\\\n            return None, None\\\\n        probs = torch.cat([e["probs"] for e in entries])\\\\n        targets = torch.cat([e["targets"] for e in entries])\\\\n\\\\n        # all_gather returns (world_size, N, L) when every rank holds the same\\\\n        # N, which the distributed sampler guarantees by padding.\\\\n        probs = self.all_gather(probs)\\\\n        targets = self.all_gather(targets)\\\\n        if probs.ndim == 3:\\\\n            probs = probs.reshape(-1, probs.shape[-1])\\\\n            targets = targets.reshape(-1, targets.shape[-1])\\\\n        return (targets.cpu().numpy(), probs.cpu().numpy())\\\\n\\\\n    def _log_conditional(self, prefix: str, truth, probs) -> None:\\\\n        """Log conditional AUC beside the pooled number, per stratified finding.\\\\n\\\\n        The plan\\\\\\\'s §3, applied where it is cheapest to apply: pooled AUC on\\\\n        synovitis is reproducible from the effusion label alone (0.698 against\\\\n        the head\\\\\\\'s 0.701), so an epoch-level pooled number cannot show whether\\\\n        the run is learning the finding or learning effusion. Conditional AUC\\\\n        is the same statistic restricted to pairs effusion does not already\\\\n        separate.\\\\n\\\\n        Stratified on the model\\\\\\\'s own predicted effusion, split at its median\\\\n        -- there is no gold effusion label in the mined validation fold, and\\\\n        on the gold set the prediction gives 0.617 against the label\\\\\\\'s 0.630,\\\\n        so nothing material is lost by using the one that always exists.\\\\n\\\\n        Monitoring only, like `gold_macro`. Selecting a checkpoint on any\\\\n        gold-derived number is what inflated an earlier version of this\\\\n        project\\\\\\\'s reported figure.\\\\n        """\\\\n        for name in self.stratified_findings:\\\\n            index = config.LABELS.index(name)\\\\n            strata = conditional.strata_from_scores(probs[:, self.stratified_index])\\\\n            value = conditional.conditional_auc(\\\\n                truth[:, index], probs[:, index], strata)\\\\n            if np.isfinite(value):\\\\n                self.log(f"{prefix}_{name}_conditional", float(value),\\\\n                         rank_zero_only=False)\\\\n            pooled = auc(truth[:, index], probs[:, index])\\\\n            if np.isfinite(pooled):\\\\n                self.log(f"{prefix}_{name}_pooled", float(pooled),\\\\n                         rank_zero_only=False)\\\\n\\\\n    def on_validation_epoch_end(self):\\\\n        val_truth, val_probs = self._gathered(0)\\\\n        if val_truth is not None:\\\\n            # Mined labels are soft; binarise for a ranking metric.\\\\n            binarised = (val_truth >= 0.5).astype(int)\\\\n            val_macro = macro_auc(binarised, val_probs)\\\\n            self.log("val_macro", val_macro, prog_bar=True, rank_zero_only=False)\\\\n            self._log_conditional("val", binarised, val_probs)\\\\n\\\\n        gold_truth, gold_probs = self._gathered(1)\\\\n        if gold_truth is not None:\\\\n            gold_macro = macro_auc(gold_truth.astype(int), gold_probs)\\\\n            # Monitoring only. Never pass this to a checkpoint callback.\\\\n            self.log("gold_macro", gold_macro, prog_bar=True,\\\\n                     rank_zero_only=False)\\\\n            self._log_conditional("gold", gold_truth.astype(int), gold_probs)\\\\n\\\\n        self._val_outputs = {0: [], 1: []}\\\\n\\\\n    def configure_optimizers(self):\\\\n        # One group unless a trunk asks for a slower one. The aggregation is\\\\n        # trained from scratch and wants the full rate; a pretrained trunk at\\\\n        # the same rate is fine for a supervised convnet and destroys a\\\\n        # self-supervised ViT\\\\\\\'s features during OneCycle\\\\\\\'s warmup. Scale 1.0\\\\n        # takes the original single-group path unchanged, so every ConvNeXt\\\\n        # result stays comparable.\\\\n        if self.backbone_lr_scale == 1.0:\\\\n            optimiser = torch.optim.AdamW(self.parameters(), lr=self.lr,\\\\n                                          weight_decay=self.weight_decay)\\\\n            max_lr = self.lr * self.max_lr_multiplier\\\\n        else:\\\\n            trunk_ids = {id(p) for p in self.model.backbone.parameters()}\\\\n            trunk = [p for p in self.parameters() if id(p) in trunk_ids]\\\\n            rest = [p for p in self.parameters() if id(p) not in trunk_ids]\\\\n            trunk_lr = self.lr * self.backbone_lr_scale\\\\n            optimiser = torch.optim.AdamW(\\\\n                [{"params": trunk, "lr": trunk_lr},\\\\n                 {"params": rest, "lr": self.lr}],\\\\n                lr=self.lr, weight_decay=self.weight_decay)\\\\n            max_lr = [trunk_lr * self.max_lr_multiplier,\\\\n                      self.lr * self.max_lr_multiplier]\\\\n\\\\n        scheduler = self._scheduler(optimiser, max_lr)\\\\n        return {\\\\n            "optimizer": optimiser,\\\\n            # Stepped per optimiser step, so accumulation is already folded in.\\\\n            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},\\\\n        }\\\\n\\\\n    def _scheduler(self, optimiser, max_lr):\\\\n        """OneCycle or cosine, with warmup pinned to an absolute step count.\\\\n\\\\n        **The restretch is the thing being fixed here.** `OneCycleLR`\\\\\\\'s\\\\n        `pct_start` is a *fraction* of the run, and it defaults to 0.3. At\\\\n        EPOCHS=12 that puts the learning-rate peak near epoch 3.6; at\\\\n        EPOCHS=20 it moves to epoch 6. Two runs of different lengths are\\\\n        therefore at different points of a differently-shaped schedule at the\\\\n        same epoch number, and "epoch 9 was best" does not mean the same thing\\\\n        in each. Every cross-length comparison this project has drawn was\\\\n        reading through that.\\\\n\\\\n        Setting `warmup_steps` to an absolute count fixes the peak in place\\\\n        regardless of run length. Leaving it None keeps the 0.3 fraction, so\\\\n        the twelve-epoch reference runs stay reproducible.\\\\n\\\\n        `schedule="cosine"` replaces the cycle with linear warmup then cosine\\\\n        decay -- the schedule the RSNA 2022/2023 winning solutions used for\\\\n        fine-tuning, and one with no restretch artefact to reason about at\\\\n        all. It peaks at the same `lr * max_lr_multiplier` as OneCycle so the\\\\n        two are comparable at matched warmup.\\\\n        """\\\\n        warmup = self.warmup_steps\\\\n        if self.schedule == "onecycle":\\\\n            pct_start = 0.3 if warmup is None else warmup / self.total_steps\\\\n            return torch.optim.lr_scheduler.OneCycleLR(\\\\n                optimiser, max_lr=max_lr, total_steps=self.total_steps,\\\\n                pct_start=pct_start)\\\\n\\\\n        warmup = int(round(0.3 * self.total_steps)) if warmup is None else warmup\\\\n        peak = self.max_lr_multiplier\\\\n        decay_steps = max(1, self.total_steps - warmup)\\\\n\\\\n        def factor(step: int) -> float:\\\\n            # Returned as a multiple of the group\\\\\\\'s base lr, which is already\\\\n            # `lr * backbone_lr_scale` for the trunk group -- so the two\\\\n            # groups keep their ratio through the whole schedule.\\\\n            if step < warmup:\\\\n                return peak * (step + 1) / max(1, warmup)\\\\n            progress = min(1.0, (step - warmup) / decay_steps)\\\\n            return peak * 0.5 * (1.0 + np.cos(np.pi * progress))\\\\n\\\\n        return torch.optim.lr_scheduler.LambdaLR(optimiser, factor)\\\\n\\\\n\\\\ndef optimiser_steps(n_studies: int, batch_size: int, devices: int,\\\\n                    accumulate: int, epochs: int) -> int:\\\\n    """Total OneCycleLR steps, which must match what Lightning will take.\\\\n\\\\n    A schedule sized for the wrong number of steps either finishes its cycle\\\\n    early and trains the tail at a near-zero learning rate, or never reaches\\\\n    its peak. Neither raises.\\\\n    """\\\\n    if min(n_studies, batch_size, devices, accumulate, epochs) < 1:\\\\n        raise ValueError("study, batch, device, accumulation and epoch counts must be positive")\\\\n    per_rank_batches = int(np.ceil(n_studies / (batch_size * devices)))\\\\n    return epochs * int(np.ceil(per_rank_batches / accumulate))\\\\n\\\\n\\\\nclass WeightAverage(pl.Callback):\\\\n    """Maintain EMA and SWA of the `KneeNet` weights and write both out.\\\\n\\\\n    This is the selection-free half of the answer to the biased-selector\\\\n    problem described in `src/model/averaging.py`: rather than trusting mined\\\\n    validation to name an epoch, average the trajectory and ship the average.\\\\n    The best-validation checkpoint is still written by `ModelCheckpoint`\\\\n    alongside these, so a run produces three candidates and the comparison\\\\n    between them is the measurement.\\\\n\\\\n    **Updated on optimiser steps, not on batches.** With gradient\\\\n    accumulation, most batches leave the weights untouched, and folding an\\\\n    unchanged model into the EMA `accumulate` times per step would make the\\\\n    effective decay `decay ** accumulate` -- a shorter memory than asked for,\\\\n    varying with a knob that is supposed to be numerically neutral.\\\\n    `on_train_batch_end` samples only when `global_step` advances, after the\\\\n    optimizer has changed the weights, including the final update.\\\\n\\\\n    **Rank zero only.** DDP keeps every rank\\\\\\\'s weights identical after each\\\\n    step, so every rank would compute the same average; doing it once saves\\\\n    the copies. The written files therefore exist only on rank zero, which is\\\\n    also the only rank whose `/kaggle/working` survives.\\\\n\\\\n    **The averages are written from inside `fit`, in `on_train_end`, and read\\\\n    back off disk afterwards.** Under `ddp_notebook` the ranks are forked\\\\n    children: the callback object the notebook constructed is *not* the object\\\\n    that accumulated anything, and asking it for its EMA afterwards returns an\\\\n    empty average with no error. This is the same trap that leaves\\\\n    `best_model_score` as None on that strategy. So the file paths are fixed\\\\n    at construction and the notebook globs for what appeared.\\\\n    """\\\\n\\\\n    def __init__(self, prefix: str, decay: float = 0.999,\\\\n                 swa_fraction: float = 0.25,\\\\n                 total_steps: Optional[int] = None):\\\\n        if not 0.0 <= swa_fraction <= 1.0:\\\\n            raise ValueError(f"swa_fraction must be in [0, 1], got {swa_fraction}")\\\\n        self.prefix = prefix\\\\n        self.decay = decay\\\\n        self.swa_fraction = swa_fraction\\\\n        self.total_steps = total_steps\\\\n        self.average: Optional[averaging.WeightAverage] = None\\\\n        self.written: Dict[str, str] = {}\\\\n        self._last_step = 0\\\\n\\\\n    @property\\\\n    def ema_path(self) -> str:\\\\n        return f"{self.prefix}_ema.pt"\\\\n\\\\n    @property\\\\n    def swa_path(self) -> str:\\\\n        return f"{self.prefix}_swa.pt"\\\\n\\\\n    def _swa_from(self, trainer) -> Optional[int]:\\\\n        """The step the equal-weight tail average starts at."""\\\\n        total = self.total_steps\\\\n        if total is None:\\\\n            total = getattr(trainer, "estimated_stepping_batches", None)\\\\n        if not total or self.swa_fraction <= 0:\\\\n            return None\\\\n        # +1 because `update` counts from one: at total=4 and fraction=0.5 the\\\\n        # window must open at step 3, so that steps 3 and 4 -- the last half --\\\\n        # are what gets averaged, not steps 2 through 4.\\\\n        return int(round(float(total) * (1.0 - self.swa_fraction))) + 1\\\\n\\\\n    def on_train_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        self.average = averaging.WeightAverage(\\\\n            module.model, decay=self.decay, swa_from=self._swa_from(trainer))\\\\n        self._last_step = trainer.global_step\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        # Sample after the update, including the final trained weights.\\\\n        if self.average is not None and trainer.global_step > self._last_step:\\\\n            self.average.update(module.model)\\\\n            self._last_step = trainer.global_step\\\\n\\\\n    def on_train_end(self, trainer, module):\\\\n        # Wrapped, on the same rule the notebook\\\\\\\'s export cell follows: by the\\\\n        # time this fires, hours of GPU time exist as the monitored checkpoint\\\\n        # `ModelCheckpoint` already wrote, and an exception raised here\\\\n        # propagates out of `trainer.fit` and takes the export and scoring\\\\n        # cells with it. The averages are an experiment; the run is not.\\\\n        try:\\\\n            self.export()\\\\n        except Exception as exc:  # noqa: BLE001 - the checkpoint is safe\\\\n            print(f"WeightAverage could not write its averages ({exc}). "\\\\n                  "The monitored checkpoint is unaffected; only the "\\\\n                  "selection-free comparison is lost.")\\\\n\\\\n    def export(self) -> Dict[str, str]:\\\\n        """Write the averages as bare `KneeNet` state dicts.\\\\n\\\\n        The same format `export_from_checkpoint` produces, so notebook 10\\\\n        loads them with no idea they came from an average. Returns\\\\n        {kind: path} for what was actually written -- SWA is absent when the\\\\n        run was too short for any step to fall inside the tail window.\\\\n        """\\\\n        self.written = {}\\\\n        if self.average is None:\\\\n            return self.written\\\\n\\\\n        torch.save(self.average.ema_state(), self.ema_path)\\\\n        self.written["ema"] = self.ema_path\\\\n\\\\n        swa = self.average.swa_state()\\\\n        if swa is not None:\\\\n            torch.save(swa, self.swa_path)\\\\n            self.written["swa"] = self.swa_path\\\\n        return self.written\\\\n\\\\n\\\\nclass EpochReport(pl.Callback):\\\\n    """One line per epoch, from rank zero only.\\\\n\\\\n    Under DDP every rank runs this code, so an unguarded print produces one\\\\n    copy of the log per GPU and makes a run look twice as long as it is.\\\\n\\\\n    **This reads `callback_metrics` in `on_validation_end`, not in\\\\n    `on_validation_epoch_end`.** Lightning runs callback hooks before the\\\\n    LightningModule\\\\\\\'s hook of the same name, so a callback asking for\\\\n    `val_macro` at `on_validation_epoch_end` arrives before\\\\n    `KneeLitModule.on_validation_epoch_end` has computed it and reads an\\\\n    empty dict -- printing `nan` for every epoch of an otherwise healthy run.\\\\n    `ModelCheckpoint` selects in `on_validation_end` for the same reason, so\\\\n    checkpoint selection was never affected by this; only the log was.\\\\n    """\\\\n\\\\n    def __init__(self):\\\\n        self.rows: List[Dict[str, float]] = []\\\\n\\\\n    def on_validation_end(self, trainer, module):\\\\n        if trainer.sanity_checking or not trainer.is_global_zero:\\\\n            return\\\\n        metrics = trainer.callback_metrics\\\\n        val = float(metrics.get("val_macro", float("nan")))\\\\n        gold = float(metrics.get("gold_macro", float("nan")))\\\\n        self.rows.append({"epoch": trainer.current_epoch + 1,\\\\n                          "val_macro": val, "gold_macro": gold})\\\\n        best = max(row["val_macro"] for row in self.rows)\\\\n        marker = "  <- best" if val >= best else ""\\\\n        print(f"  epoch {trainer.current_epoch + 1}: val macro {val:.3f}  "\\\\n              f"gold {gold:.3f} (monitor only){marker}")\\\\n\\\\n\\\\nclass BatchProgress(pl.Callback):\\\\n    """Plain-text tqdm over training batches, rank zero only.\\\\n\\\\n    Lightning\\\\\\\'s own progress bar resolves to the notebook-widget flavour of\\\\n    tqdm inside Kaggle, and the widget never reaches the log of a\\\\n    Save & Run All -- eleven minutes pass per epoch with no sign of life.\\\\n    Plain tqdm writes carriage-returned text to stdout, which survives both\\\\n    the interactive view and the batch log, so the Trainer disables its own\\\\n    bar and takes this instead. `mininterval` keeps the batch log from\\\\n    bloating: one refresh per interval, not one per batch.\\\\n    """\\\\n\\\\n    def __init__(self, refresh_seconds: float = 15.0):\\\\n        self.refresh_seconds = refresh_seconds\\\\n        self._bar = None\\\\n\\\\n    def on_train_epoch_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        import sys\\\\n        from tqdm import tqdm\\\\n        self._bar = tqdm(\\\\n            total=int(trainer.num_training_batches),\\\\n            desc=f"epoch {trainer.current_epoch + 1}/{trainer.max_epochs}",\\\\n            unit="batch", file=sys.stdout,\\\\n            mininterval=self.refresh_seconds, leave=False)\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        if self._bar is not None:\\\\n            self._bar.update(1)\\\\n\\\\n    def on_train_epoch_end(self, trainer, module):\\\\n        if self._bar is not None:\\\\n            self._bar.close()\\\\n            self._bar = None\\\\n\\\\n\\\\ndef export_from_checkpoint(ckpt_path: str, out_path: str) -> int:\\\\n    """Convert a Lightning checkpoint into the plain state dict for scoring.\\\\n\\\\n    Read from the file rather than from a module in memory: under the spawn\\\\n    strategy the process that called `fit` is not the process that trained,\\\\n    and reaching for in-memory weights there is how a run silently ships its\\\\n    initialisation. Returns the number of tensors written.\\\\n    """\\\\n    checkpoint = torch.load(ckpt_path, map_location="cpu")\\\\n    state_dict = checkpoint.get("state_dict", checkpoint)\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in state_dict.items()\\\\n             if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(\\\\n            f"no \\\\\\\'model.\\\\\\\' keys in {ckpt_path}; found "\\\\n            f"{list(state_dict)[:3]}")\\\\n    torch.save(state, out_path)\\\\n    return len(state)\\\\n\\\\n\\\\ndef export_state_dict(module: KneeLitModule, path: str) -> None:\\\\n    """Write a plain `KneeNet` state dict, the format the submission loads.\\\\n\\\\n    Lightning prefixes every key with `model.` because `KneeNet` is an\\\\n    attribute of the wrapper. Notebook 10 constructs a bare `KneeNet` and\\\\n    calls `load_state_dict`, so the prefix has to come off here rather than\\\\n    become a surprise at submission time.\\\\n    """\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in module.state_dict().items()\\\\n             if key.startswith("model.")}\\\\n    torch.save(state, path)\\\\n\\\'}\\nfor path, source in FILES.items():\\n    target = pathlib.Path(path)\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    target.write_text(source)\\nsys.path.insert(0, \\\'.\\\')\\nfor _name in [m for m in sys.modules if m == \\\'src\\\' or m.startswith(\\\'src.\\\')]:\\n    del sys.modules[_name]\\nimportlib.invalidate_caches()\\nfrom src import config\\nfrom src.data import dicom_io, cache, index as index_mod\\nif pathlib.Path(\\\'src/model/side_classifier.py\\\').exists():\\n    from src.model import side_classifier\\nif pathlib.Path(\\\'src/model/net.py\\\').exists():\\n    from src.model import augment as knee_augment, averaging, conditional, embed, dataset as knee_dataset, net, splits\\n    from src.model import artifacts, experiments\\nif pathlib.Path(\\\'src/model/inspect.py\\\').exists():\\n    from src.model import inspect as knee_inspect\\nif pathlib.Path(\\\'src/labels/fuse.py\\\').exists():\\n    from src.labels import fuse\\nif pathlib.Path(\\\'src/model/lightning.py\\\').exists():\\n    from src.model import lightning as knee_lightning\\nprint(\\\'modules ready:\\\', len(FILES), \\\'files\\\')\\nprint(\\\'labels:\\\', len(config.LABELS))\\nprint(\\\'cache geometry:\\\', config.CACHE_SLICES, \\\'x\\\', config.CACHE_SIZE)\\nprint(\\\'data root:\\\', config.DATA_ROOT)\\n\\nEXPERIMENT = os.environ.get(\\\'KNEE_EXPERIMENT\\\', \\\'resample\\\')\\nRUN = experiments.get_experiment(EXPERIMENT)\\nfrom dataclasses import replace\\nRUN = replace(RUN, series_selection=\\\'protocol\\\')\\nprint(\\\'experiment:\\\', RUN.to_dict())\\n\\nimport glob\\nimport numpy as np\\nimport pandas as pd\\n\\ndef find_inputs(pattern):\\n    found = sorted(glob.glob(f\\\'data/**/{pattern}\\\', recursive=True))\\n    found += sorted(glob.glob(f\\\'/kaggle/working/baseline_fold2/{pattern}\\\'))\\n    for root in sorted(glob.glob(\\\'/kaggle/input/*\\\')):\\n        if os.path.basename(root) == \\\'competitions\\\' or os.path.exists(os.path.join(root, \\\'train.csv\\\')) or os.path.exists(os.path.join(root, \\\'train_series.csv\\\')):\\n            continue\\n        found += sorted(glob.glob(f\\\'{root}/**/{pattern}\\\', recursive=True))\\n    return found\\n\\ndef find_competition_file(name):\\n    for candidate in (f\\\'data/{name}\\\', f\\\'/kaggle/input/competitions/*/{name}\\\', f\\\'/kaggle/input/*/{name}\\\'):\\n        hits = sorted(glob.glob(candidate))\\n        if hits:\\n            return hits[0]\\n    raise SystemExit(f\\\'missing competition file: {name}\\\')\\nlabels = pd.read_parquet(find_inputs(\\\'train_labels.parquet\\\')[0])\\nSILENCE_REWEIGHT = False\\nif SILENCE_REWEIGHT:\\n    _rates = fuse.silence_rates(labels)\\n    labels = fuse.rescale_silent_weights(labels)\\n    print(\\\'silence reweighting ON\\\')\\n    for _n in sorted(config.LABELS, key=lambda x: -_rates[x])[:3]:\\n        print(f\\\'  {_n:18s} silent {_rates[_n]:5.1%} -> weight {fuse.WEIGHT_SILENT * (1 - _rates[_n]):.3f}\\\')\\nelse:\\n    print(\\\'silence reweighting OFF (reference labels)\\\')\\nsides = pd.read_parquet(find_inputs(\\\'side_predictions.parquet\\\')[0])\\nreport_hits = find_inputs(\\\'report_embeddings.parquet\\\')\\nreports = pd.read_parquet(report_hits[0]) if report_hits else None\\nif reports is None:\\n    print(\\\'report_embeddings.parquet not attached -- report loss disabled\\\')\\nseries_meta = pd.read_csv(find_competition_file(\\\'train_series.csv\\\'))\\ntrain_csv = pd.read_csv(find_competition_file(\\\'train.csv\\\'))\\ngold = train_csv[train_csv[\\\'ACL\\\'].notna()].reset_index(drop=True)\\nALL_SHARDS = True\\ncache_paths = find_inputs(\\\'cache_train_series_*.npy\\\')\\nif not ALL_SHARDS:\\n    cache_paths = cache_paths[:1]\\nshards, frames = ({}, [])\\nfor cache_path in cache_paths:\\n    manifest_path = cache_path.replace(\\\'cache_\\\', \\\'manifest_\\\').replace(\\\'.npy\\\', \\\'.parquet\\\')\\n    if not os.path.exists(manifest_path):\\n        stem = os.path.basename(cache_path).replace(\\\'cache_\\\', \\\'\\\').replace(\\\'.npy\\\', \\\'\\\')\\n        hits = [m for m in find_inputs(\\\'manifest_train_series_*.parquet\\\') if os.path.basename(m).startswith(f\\\'manifest_{stem}\\\')]\\n        if not hits:\\n            continue\\n        manifest_path = hits[0]\\n    frame = pd.read_parquet(manifest_path)\\n    frame[\\\'cache_path\\\'] = cache_path\\n    shards[cache_path] = np.load(cache_path, mmap_mode=\\\'r\\\')\\n    frames.append(frame)\\n    print(\\\'shard:\\\', os.path.basename(cache_path), shards[cache_path].shape)\\nmanifest = pd.concat(frames, ignore_index=True)\\nGROUPED_FOLDS = False\\ngroups = None\\nif GROUPED_FOLDS:\\n    _index_files = find_inputs(\\\'index_train.parquet\\\') + find_inputs(\\\'index.parquet\\\')\\n    if _index_files:\\n        groups = splits.scanner_key(pd.read_parquet(_index_files[0]))\\n        print(f\\\'grouped folds: {groups.nunique()} scanner groups over {len(groups)} studies\\\')\\n    else:\\n        raise ValueError("GROUPED_FOLDS requires notebook 01\\\'s index parquet")\\nfolded = splits.make_folds(labels, groups=groups)\\nsplits.assert_no_gold_leakage(folded[\\\'StudyInstanceUID\\\'], set(gold[\\\'StudyInstanceUID\\\']))\\nif groups is not None:\\n    _summary = splits.summarise(folded)\\n    print(_summary[[\\\'studies\\\'] + config.LABELS[:4]].to_string())\\n    print(\\\'  fold sizes are uneven by construction -- a scanner group cannot be split. Check the positive rates before trusting a comparison.\\\')\\nMAX_SERIES = 5\\nMAX_SERIES = RUN.max_series\\nindex = knee_dataset.StudyIndex(manifest, series_meta, folded, sides, max_series=MAX_SERIES, reports=reports, series_selection=RUN.series_selection)\\ngold_labels = gold[[\\\'StudyInstanceUID\\\'] + config.LABELS].copy()\\nfor name in config.LABELS:\\n    gold_labels[f\\\'{name}_weight\\\'] = 1.0\\ngold_index = knee_dataset.StudyIndex(manifest, series_meta, gold_labels, sides, max_series=MAX_SERIES, series_selection=RUN.series_selection)\\nif ALL_SHARDS and (set(index.studies) != set(labels[config.ID_COL]) or set(gold_index.studies) != set(gold[config.ID_COL])):\\n    raise ValueError(\\\'incomplete cache coverage: attach every training and gold study\\\')\\nprint(f\\\'studies in this shard: {len(index)}  gold present: {len(gold_index)}\\\')\\n\\nfrom torch.utils.data import DataLoader\\nSLICES_PER_SERIES = 16\\nSLICES_PER_SERIES = RUN.slices\\nRESAMPLE = True\\nRESAMPLE = RUN.resample\\nBATCH_STUDIES = 1\\nEFFECTIVE_STUDIES = 8\\nACCUM_STEPS = max(1, EFFECTIVE_STUDIES // (BATCH_STUDIES * DEVICES))\\nprint(f\\\'{BATCH_STUDIES} studies x {DEVICES} devices x {ACCUM_STEPS} accum = {BATCH_STUDIES * DEVICES * ACCUM_STEPS} effective\\\')\\nCACHE_PATHS = list(shards)\\n\\ndef make_loader(study_index, studies, shuffle=False, augment=False, resample=False):\\n    return DataLoader(knee_lightning.StudyVolumeDataset(study_index, studies, CACHE_PATHS, SLICES_PER_SERIES, augment, resample=resample), batch_size=BATCH_STUDIES, shuffle=shuffle, num_workers=2, collate_fn=knee_dataset.collate_studies, pin_memory=True, persistent_workers=True)\\n\\nimport timm\\nBACKBONE = \\\'convnext_tiny.fb_in22k_ft_in1k\\\'\\nBACKBONE = RUN.backbone\\n_warm = net.build_backbone(BACKBONE)\\nFEATURE_DIM = _warm.num_features\\nprint(f\\\'{BACKBONE}: {FEATURE_DIM} features, {sum((p.numel() for p in _warm.parameters())) / 1000000.0:.1f}M params\\\')\\ndel _warm\\nBACKBONE_DEFAULTS = net.backbone_defaults(BACKBONE)\\nprint(\\\'backbone defaults:\\\', BACKBONE_DEFAULTS)\\nauc = knee_lightning.auc\\n\\nimport time\\nimport timm\\n_required = {\\\'DEVICES\\\': \\\'1. Setup\\\', \\\'knee_lightning\\\': \\\'2. Repo modules\\\', \\\'index\\\': \\\'3. Assemble\\\', \\\'make_loader\\\': \\\'4. Dataset\\\', \\\'FEATURE_DIM\\\': \\\'5. Model\\\'}\\n_missing = {name: cell for name, cell in _required.items() if name not in globals()}\\nif _missing:\\n    raise SystemExit(\\\'This cell needs the cells above it, and this kernel has not run them:\\\\n\\\' + \\\'\\\\n\\\'.join((f\\\'    {name!r} comes from section {cell}\\\' for name, cell in _missing.items())) + \\\'\\\\n\\\\nRun the notebook from the top (Runtime -> Run all). Changing FOLD and re-running only this cell works within a LIVE kernel; a restarted runtime has to re-run everything.\\\')\\nEPOCHS = 12\\nLR = 0.0001\\nWARMUP_EPOCHS = 3.6\\nSCHEDULE = \\\'onecycle\\\'\\nSEED = 0\\npl.seed_everything(SEED, workers=True)\\nPOS_WEIGHT_CAP = None\\nif POS_WEIGHT_CAP is None:\\n    POS_WEIGHT = None\\nelse:\\n    _positives = (labels[config.LABELS] >= 0.5).sum()\\n    _negatives = len(labels) - _positives\\n    POS_WEIGHT = [float(min((n / max(p, 1)) ** 0.5, POS_WEIGHT_CAP)) for p, n in zip(_positives, _negatives)]\\n    print(f\\\'pos_weight (sqrt of neg/pos, capped at {POS_WEIGHT_CAP}):\\\')\\n    for _name, _w in zip(config.LABELS, POS_WEIGHT):\\n        _p = int((labels[_name] >= 0.5).sum())\\n        print(f\\\'  {_name:18s} {_w:5.2f}   ({_p} positives, {_p / len(labels):.1%})\\\')\\nLAMBDA_REPORT = 0.0\\nLAMBDA_RANK = 0.0\\nLAMBDA_STRATIFIED_RANK = 0.0\\nSTRATIFIED_FINDINGS = (\\\'Synovitis\\\',)\\nSTRATIFY_BY = \\\'Effusion\\\'\\nSTRATIFIED_CALIPER = 0.15\\nRESIDUAL_BETA = None\\nLAMBDA_COUPLING = 0.0\\nCOUPLING_TARGET = 0.403\\nAUX_MODE = \\\'fixed\\\'\\nAUX_GATE = \\\'binary\\\'\\nREPORT_DIM = len([c for c in reports.columns if c != \\\'StudyInstanceUID\\\']) if reports is not None else None\\nFOLD = 2\\nCHECKPOINT = f\\\'/kaggle/working/baseline_fold2/convnext_fold{FOLD}.pt\\\'\\nAVERAGE_PREFIX = f\\\'/kaggle/working/baseline_fold2/avg_fold{FOLD}\\\'\\nkeyed = folded.set_index(\\\'StudyInstanceUID\\\')\\ntrain_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] != FOLD]\\nval_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] == FOLD]\\ntrain_loader = make_loader(index, train_studies, shuffle=True, augment=True, resample=RESAMPLE)\\nprint(f"resampling slices/series per epoch: {(\\\'on\\\' if RESAMPLE else \\\'off\\\')}")\\nval_loader = make_loader(index, val_studies)\\ngold_loader = make_loader(gold_index, gold_index.studies)\\nprint(f\\\'fold {FOLD}: train {len(train_studies)}  val {len(val_studies)}  gold {len(gold_index)}\\\')\\nTOTAL_STEPS = knee_lightning.optimiser_steps(len(train_studies), BATCH_STUDIES, DEVICES, ACCUM_STEPS, EPOCHS)\\nprint(\\\'optimiser steps:\\\', TOTAL_STEPS)\\nWARMUP_STEPS = int(round(TOTAL_STEPS / EPOCHS * WARMUP_EPOCHS))\\nprint(f"warmup: {WARMUP_STEPS} steps ({WARMUP_EPOCHS} epochs\\\' worth), schedule {SCHEDULE}")\\nmodule = knee_lightning.KneeLitModule(backbone=net.build_backbone(BACKBONE), feature_dim=FEATURE_DIM, lr=LR, total_steps=TOTAL_STEPS, report_dim=REPORT_DIM, lambda_report=LAMBDA_REPORT, lambda_rank=LAMBDA_RANK, aux_mode=AUX_MODE, aux_gate=AUX_GATE, schedule=SCHEDULE, warmup_steps=WARMUP_STEPS, lambda_stratified_rank=LAMBDA_STRATIFIED_RANK, stratified_findings=STRATIFIED_FINDINGS, stratify_by=STRATIFY_BY, stratified_caliper=STRATIFIED_CALIPER, lambda_coupling=LAMBDA_COUPLING, coupling_target=COUPLING_TARGET if LAMBDA_COUPLING > 0 else None, residual_beta=RESIDUAL_BETA, pos_weight=POS_WEIGHT, classwise_pool=RUN.classwise_pool, slice_transformer=RUN.slice_transformer, **BACKBONE_DEFAULTS)\\nprint(f"trunk lr: {LR * BACKBONE_DEFAULTS[\\\'backbone_lr_scale\\\']:.1e}  head lr: {LR:.1e}  input norm: {(\\\'on\\\' if BACKBONE_DEFAULTS[\\\'input_mean\\\'] else \\\'off\\\')}")\\nprint(f\\\'aux losses: report {LAMBDA_REPORT} (dim {REPORT_DIM})  rank {LAMBDA_RANK}  mode {AUX_MODE}\\\' + (f\\\'/{AUX_GATE}\\\' if AUX_MODE == \\\'cosine\\\' else \\\'\\\'))\\nprint(f\\\'synovitis: stratified rank {LAMBDA_STRATIFIED_RANK} (caliper {STRATIFIED_CALIPER})  residual beta {RESIDUAL_BETA}  coupling {LAMBDA_COUPLING}\\\' + (f\\\' -> {COUPLING_TARGET}\\\' if LAMBDA_COUPLING > 0 else \\\'\\\'))\\nENSEMBLE_DONE = {0: \\\'2026-08-31, 12 ep, val 0.8814, gold 0.8683\\\', 1: \\\'2026-08-31, 12 ep, val 0.8726, gold 0.8747\\\', 2: \\\'2026-08-31, 12 ep, val 0.8675, gold 0.8659\\\', 3: \\\'2026-08-31, 12 ep, val 0.8736, gold 0.8915\\\', 4: \\\'2026-08-31, 12 ep, val 0.8697, gold 0.8753\\\'}\\nENSEMBLE_TODO = [f for f in range(5) if f not in ENSEMBLE_DONE]\\n_expected = {\\\'SLICES_PER_SERIES\\\': (SLICES_PER_SERIES, 16), \\\'MAX_SERIES\\\': (MAX_SERIES, 5), \\\'EPOCHS\\\': (EPOCHS, 12), \\\'LAMBDA_REPORT\\\': (LAMBDA_REPORT, 0.0), \\\'LAMBDA_RANK\\\': (LAMBDA_RANK, 0.0), \\\'LAMBDA_STRATIFIED_RANK\\\': (LAMBDA_STRATIFIED_RANK, 0.0), \\\'RESIDUAL_BETA\\\': (RESIDUAL_BETA, None), \\\'LAMBDA_COUPLING\\\': (LAMBDA_COUPLING, 0.0), \\\'POS_WEIGHT_CAP\\\': (POS_WEIGHT_CAP, None), \\\'RESAMPLE\\\': (RESAMPLE, False), \\\'SERIES_SELECTION\\\': (RUN.series_selection, \\\'uniform\\\'), \\\'CLASSWISE_POOL\\\': (RUN.classwise_pool, False), \\\'SLICE_TRANSFORMER\\\': (RUN.slice_transformer, False), \\\'SILENCE_REWEIGHT\\\': (SILENCE_REWEIGHT, False), \\\'BACKBONE\\\': (BACKBONE, \\\'convnext_tiny.fb_in22k_ft_in1k\\\')}\\n_wrong = {k: v for k, (v, want) in _expected.items() if v != want}\\nprint(\\\'=\\\' * 68)\\nif _wrong:\\n    print(f\\\'  OFF-REFERENCE FOLD {FOLD}. These differ from the ensemble config:\\\')\\n    for k, v in _wrong.items():\\n        print(f\\\'    {k} = {v!r}, reference is {_expected[k][1]!r}\\\')\\n    print(\\\'  An ensemble member trained off-reference makes the average a\\\')\\n    print(\\\'  blend of two models, and the averaging hides it. Fix these, or\\\')\\n    print(\\\'  accept that this run is an experiment and not a member.\\\')\\nelif FOLD in ENSEMBLE_DONE:\\n    print(f\\\'  FOLD {FOLD} IS ALREADY DONE -- {ENSEMBLE_DONE[FOLD]}\\\')\\n    print(f\\\'  Retraining it is fine, but the folds still missing are {ENSEMBLE_TODO}.\\\')\\nelse:\\n    print(f\\\'  ENSEMBLE FOLD {FOLD}: reference config, no auxiliary loss\\\')\\n    print(f\\\'  done: {sorted(ENSEMBLE_DONE)}   outstanding after this: {[f for f in ENSEMBLE_TODO if f != FOLD]}\\\')\\n    print(f\\\'  export convnext_fold{FOLD}.pt and COPY IT OFF THE VM before the\\\')\\n    print(\\\'  session ends -- /kaggle/working does not survive.\\\')\\nprint(\\\'=\\\' * 68)\\ncheckpointer = pl.callbacks.ModelCheckpoint(dirpath=\\\'/kaggle/working/baseline_fold2\\\', filename=f\\\'lit_fold{FOLD}\\\', monitor=\\\'val_macro\\\', mode=\\\'max\\\', save_top_k=1)\\nreport = knee_lightning.EpochReport()\\naverager = knee_lightning.WeightAverage(prefix=AVERAGE_PREFIX, decay=0.999, swa_fraction=0.25, total_steps=TOTAL_STEPS)\\ncsv_logger = pl.loggers.CSVLogger(\\\'/kaggle/working/baseline_fold2\\\', name=f\\\'lit_fold{FOLD}\\\', version=\\\'metrics\\\')\\nSTRATEGY = \\\'ddp\\\' if DEVICES > 1 else \\\'auto\\\'\\nif STRATEGY == \\\'ddp_notebook\\\' and torch.cuda.is_initialized():\\n    raise SystemExit("CUDA is already initialised in this process, so the forked workers cannot start.\\\\nRun -> Restart Session, then run the cells in order. If it recurs, set STRATEGY = \\\'ddp_spawn\\\' in this cell.")\\ntrainer = pl.Trainer(max_epochs=EPOCHS, accelerator=\\\'gpu\\\', devices=DEVICES, strategy=STRATEGY, precision=\\\'16-mixed\\\', accumulate_grad_batches=ACCUM_STEPS, callbacks=[checkpointer, report, averager, knee_lightning.BatchProgress()], logger=csv_logger, log_every_n_steps=50, enable_model_summary=False, enable_progress_bar=False)\\nstart = time.time()\\nassert (FOLD, EPOCHS, SEED, SLICES_PER_SERIES, MAX_SERIES) == (EXPECTED_FOLD, 12, 0, 16, 5)\\nassert not RESAMPLE and (not GROUPED_FOLDS)\\nassert RUN.classwise_pool == (EXPERIMENT == \\\'classwise\\\')\\nassert not SILENCE_REWEIGHT and POS_WEIGHT is None and (RESIDUAL_BETA is None)\\nassert (LAMBDA_REPORT, LAMBDA_RANK, LAMBDA_STRATIFIED_RANK, LAMBDA_COUPLING) == (0.0, 0.0, 0.0, 0.0)\\nassert len(index) == 4349 and len(gold_index) == 58 and (len(CACHE_PATHS) == 3)\\nassert len(train_studies) == 4349 - EXPECTED_VAL and len(val_studies) == EXPECTED_VAL\\nassert BATCH_STUDIES * DEVICES * ACCUM_STEPS == 8\\nassert BACKBONE == (\\\'convnextv2_tiny.fcmae_ft_in22k_in1k\\\' if EXPERIMENT == \\\'convnextv2\\\' else \\\'convnext_tiny.fb_in22k_ft_in1k\\\')\\nif MATCH_BACKBONE:\\n    assert FEATURE_DIM == 768\\n    assert BACKBONE_DEFAULTS == {\\\'input_mean\\\': None, \\\'input_std\\\': None, \\\'backbone_lr_scale\\\': 1.0}\\nif MATCH_SLICE:\\n    assert RUN.series_selection == \\\'protocol\\\'\\n    assert RUN.slice_transformer == (EXPERIMENT == \\\'slice_transformer\\\')\\nelse:\\n    assert RUN.series_selection == (\\\'protocol\\\' if EXPERIMENT == \\\'protocol\\\' else \\\'uniform\\\')\\nassert all((volume.shape[1:] == (24, 224, 224) for volume in shards.values()))\\nimport hashlib, json, pathlib\\n_initial = hashlib.sha256()\\nfor _key, _tensor in sorted(module.state_dict().items()):\\n    _initial.update(_key.encode())\\n    _initial.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n_contract = {\\\'experiment\\\': RUN.to_dict(), \\\'fold\\\': FOLD, \\\'epochs\\\': EPOCHS, \\\'seed\\\': SEED, \\\'devices\\\': DEVICES, \\\'effective_batch\\\': EFFECTIVE_STUDIES, \\\'steps\\\': TOTAL_STEPS, \\\'lr\\\': LR, \\\'schedule\\\': SCHEDULE, \\\'warmup_steps\\\': WARMUP_STEPS, \\\'initial_weights_sha256\\\': _initial.hexdigest(), \\\'torch\\\': torch.__version__, \\\'timm\\\': timm.__version__, \\\'lightning\\\': pl.__version__}\\nif MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE:\\n    _excluded = (\\\'model.slice_encoder.\\\',) if MATCH_SLICE else (\\\'model.backbone.\\\',) if MATCH_BACKBONE else (\\\'model.study_pool.\\\', \\\'model.head.\\\')\\n    _shared = hashlib.sha256()\\n    for _key, _tensor in sorted(module.state_dict().items()):\\n        if not _key.startswith(_excluded):\\n            _shared.update(_key.encode())\\n            _shared.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n    _contract[\\\'shared_initial_weights_sha256\\\'] = _shared.hexdigest()\\nif MATCH_BACKBONE:\\n    _contract[\\\'aggregation_seed\\\'] = SEED\\n    _contract[\\\'feature_dim\\\'] = FEATURE_DIM\\n    _contract[\\\'input_defaults\\\'] = BACKBONE_DEFAULTS\\nif \\\'REFERENCE_CONTRACT\\\' in globals():\\n    _fold_specific = {\\\'fold\\\', \\\'steps\\\', \\\'warmup_steps\\\'}\\n    assert {k: v for k, v in _contract.items() if k not in _fold_specific} == {k: v for k, v in REFERENCE_CONTRACT.items() if k not in _fold_specific}, \\\'Completion settings or initialization differ from screened protocol\\\'\\n_baseline_contract = pathlib.Path(BASELINE_CONTRACT_PATH)\\nif EXPERIMENT != \\\'baseline\\\' and _baseline_contract.exists():\\n    _control = json.loads(_baseline_contract.read_text())\\n    _ignored = {\\\'experiment\\\'} | ({\\\'initial_weights_sha256\\\'} if MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE else set())\\n    assert {k: v for k, v in _contract.items() if k not in _ignored} == {k: v for k, v in _control.items() if k not in _ignored}, \\\'Candidate settings or shared initialization differ from baseline\\\'\\nif int(os.environ.get(\\\'LOCAL_RANK\\\', \\\'0\\\')) == 0:\\n    pathlib.Path(\\\'run_contract.json\\\').write_text(json.dumps(_contract, indent=2) + \\\'\\\\n\\\')\\nprint(\\\'Matched-run preflight passed\\\', flush=True)\\ntrainer.fit(module, train_dataloaders=train_loader, val_dataloaders=[val_loader, gold_loader])\\nif not trainer.is_global_zero:\\n    raise SystemExit(0)\\nelapsed = time.time() - start\\n\\ndef recover_checkpoint(fold):\\n    """The checkpoint this run wrote, when `best_model_path` came back empty.\\n\\n    NOT `sorted(glob(...))[0]`, which is what this was. That picks the first\\n    name ALPHABETICALLY, and "-" (45) sorts before "." (46), which sorts\\n    before "_" (95): `lit_fold1-v1.ckpt` beats `lit_fold1.ckpt`, which beats\\n    `lit_fold1_old.ckpt`. Lightning writes those `-vN` suffixes on every name\\n    collision, so a session that resumed after a crash silently recovered a\\n    checkpoint from the earlier attempt.\\n\\n    The exact name wins when it exists, and otherwise the newest file does --\\n    which is this run\\\'s, since this cell runs seconds after fit returned.\\n    """\\n    exact = f\\\'/kaggle/working/baseline_fold2/lit_fold{fold}.ckpt\\\'\\n    if os.path.exists(exact):\\n        return exact\\n    found = glob.glob(f\\\'/kaggle/working/baseline_fold2/lit_fold{fold}*.ckpt\\\')\\n    return max(found, key=os.path.getmtime) if found else None\\ncheckpoint_path = checkpointer.best_model_path or \\\'\\\'\\nif not checkpoint_path or not os.path.exists(checkpoint_path):\\n    checkpoint_path = recover_checkpoint(FOLD)\\n    if checkpoint_path is None:\\n        raise SystemExit(\\\'training finished but no checkpoint was written -- check the epoch log for whether val_macro was ever computed\\\')\\n    print(f\\\'best_model_path was empty; recovered {checkpoint_path}\\\')\\nprint(\\\'\\\\nbest checkpoint:\\\', checkpoint_path)\\nn_tensors = knee_lightning.export_from_checkpoint(checkpoint_path, CHECKPOINT)\\nprint(f\\\'wrote {CHECKPOINT} ({n_tensors} tensors)\\\')\\nAVERAGES = {kind: f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\' for kind in (\\\'ema\\\', \\\'swa\\\') if os.path.exists(f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\')}\\nfor kind, path in AVERAGES.items():\\n    print(f\\\'wrote {path} ({kind.upper()} weights, selection-free)\\\')\\n_actual_spec = RUN.to_dict()\\n_actual_spec.update(backbone=BACKBONE, slices=SLICES_PER_SERIES, max_series=MAX_SERIES, resample=RESAMPLE)\\nfor _path in [CHECKPOINT] + list(AVERAGES.values()):\\n    artifacts.write_metadata(_path, _actual_spec, FOLD, train_studies, val_studies)\\nfolded[[config.ID_COL, \\\'fold\\\']].to_parquet(\\\'/kaggle/working/baseline_fold2/folds.parquet\\\', index=False)\\nif not AVERAGES:\\n    print(\\\'NOTE: no averaged weights on disk -- the WeightAverage callback did not run to on_train_end. The monitored checkpoint is fine; only the P2 comparison is unavailable.\\\')\\ntry:\\n    history = pd.read_csv(os.path.join(csv_logger.log_dir, \\\'metrics.csv\\\'))\\n    per_epoch_rows = history.dropna(subset=[\\\'val_macro\\\']).groupby(\\\'epoch\\\')[[\\\'val_macro\\\', \\\'gold_macro\\\']].last()\\n    best_epoch = per_epoch_rows[\\\'val_macro\\\'].idxmax()\\n    print(f"\\\\n{\\\'epoch\\\':>5s} {\\\'val macro\\\':>9s} {\\\'gold\\\':>7s}")\\n    for epoch_no, row in per_epoch_rows.iterrows():\\n        marker = \\\'  <- selected\\\' if epoch_no == best_epoch else \\\'\\\'\\n        print(f"{epoch_no + 1:5d} {row[\\\'val_macro\\\']:9.3f} {row[\\\'gold_macro\\\']:7.3f}{marker}")\\n    best = per_epoch_rows.loc[best_epoch]\\n    print(f"\\\\nbest val macro {best[\\\'val_macro\\\']:.3f} at epoch {best_epoch + 1}; gold there {best[\\\'gold_macro\\\']:.3f} (monitor only, never selected on)")\\n    _val = per_epoch_rows[\\\'val_macro\\\']\\n    _gold = per_epoch_rows[\\\'gold_macro\\\']\\n    _tail = _gold.loc[best_epoch:]\\n    print(\\\'\\\\nP1 -- val/gold divergence\\\')\\n    print(f\\\'  val   peak {_val.max():.3f} at epoch {_val.idxmax() + 1}, final {_val.iloc[-1]:.3f}  ({_val.iloc[-1] - _val.max():+.3f})\\\')\\n    print(f\\\'  gold  peak {_gold.max():.3f} at epoch {_gold.idxmax() + 1}, final {_gold.iloc[-1]:.3f}  ({_gold.iloc[-1] - _gold.max():+.3f})\\\')\\n    print(f\\\'  gold from the selected epoch on: range {_tail.max() - _tail.min():.3f} over {len(_tail)} epochs\\\')\\n    if _val.iloc[-1] < _val.max() - 0.005 and _tail.max() - _tail.min() < 0.02:\\n        print(\\\'  -> DIVERGENCE REPRODUCES. Mined val fell while gold held flat: the selector is biased, and every fold selected on it is paying for it. Ship the averaged weights.\\\')\\n    else:\\n        print(\\\'  -> no divergence in this run. Selecting on mined val is defensible here; re-read before adopting the averages fleet-wide.\\\')\\nexcept Exception as exc:\\n    print(f\\\'\\\\ncould not read the epoch table ({exc}); look under {csv_logger.save_dir} for metrics.csv\\\')\\nper_epoch = elapsed / EPOCHS\\nshare = len(index) / 4349\\nprint(f\\\'\\\\nthis shard is {share:.0%} of the training set\\\')\\nprint(f\\\'per epoch: {per_epoch / 60:.1f} min on {DEVICES} GPU(s)\\\')\\nprint(f\\\'projected 5 folds x {EPOCHS} epochs: {per_epoch / share * EPOCHS * 5 / 3600:.1f} h\\\')\\n\\ndevice = \\\'cuda\\\'\\nscoring_path = CHECKPOINT\\nif not os.path.exists(scoring_path):\\n    scoring_path = recover_checkpoint(FOLD)\\n    if scoring_path is None:\\n        raise SystemExit(f\\\'neither {CHECKPOINT} nor a lit_fold{FOLD} checkpoint is present\\\')\\n    print(f\\\'{CHECKPOINT} missing; scoring from {scoring_path}\\\')\\n\\ndef load_for_scoring(path):\\n    state = net.load_knee_state(path, map_location=device)\\n    model = net.KneeNet(net.build_backbone(BACKBONE, pretrained=False), FEATURE_DIM, input_mean=BACKBONE_DEFAULTS[\\\'input_mean\\\'], input_std=BACKBONE_DEFAULTS[\\\'input_std\\\'], **net.aggregation_defaults(state)).to(device)\\n    model.load_state_dict(state)\\n    return model.eval()\\n\\ndef gold_predictions(model):\\n    ids, rows = ([], [])\\n    with torch.no_grad(), torch.amp.autocast(\\\'cuda\\\'):\\n        for batch in gold_loader:\\n            logits = model(batch[\\\'volumes\\\'].to(device), batch[\\\'meta\\\'].to(device), batch[\\\'mask\\\'].to(device))\\n            rows.append(torch.sigmoid(logits.float()).cpu().numpy())\\n            ids += batch[config.ID_COL]\\n    return (ids, np.concatenate(rows))\\nprint(\\\'scoring with input normalisation:\\\', \\\'on\\\' if BACKBONE_DEFAULTS[\\\'input_mean\\\'] else \\\'off\\\')\\nscoring_model = load_for_scoring(scoring_path)\\ngold_ids, gold_probs = gold_predictions(scoring_model)\\ntruth = gold.set_index(\\\'StudyInstanceUID\\\').loc[gold_ids]\\nFROZEN = dict(zip(config.LABELS, [0.728, 0.671, 0.698, 0.68, 0.895, 0.899, 0.772, 0.897, 0.746, 0.862, 0.711, 0.683]))\\nREFERENCE_GOLD = 0.875\\nBEST_GOLD = 0.875\\nprint(f"{\\\'finding\\\':18s} {\\\'this run\\\':>9s}  {\\\'frozen (retired)\\\':>16s}")\\nprint(\\\'-\\\' * 47)\\nscores = []\\nfor i, name in enumerate(config.LABELS):\\n    value = auc(truth[name].to_numpy().astype(int), gold_probs[:, i])\\n    scores.append(value)\\n    print(f\\\'{name:18s} {value:9.3f}  {FROZEN[name]:16.3f}\\\')\\nprint(\\\'-\\\' * 47)\\nmacro = float(np.nanmean(scores))\\nprint(f"{\\\'macro\\\':18s} {macro:9.3f}  {np.mean(list(FROZEN.values())):16.3f}")\\nprint(f\\\'\\\\nthe comparisons that matter (fold 1, 16 x 5):\\\')\\nprint(f\\\'  vs reference  gold {REFERENCE_GOLD:.3f} (LB 0.889): {macro - REFERENCE_GOLD:+.3f}\\\')\\nprint(f\\\'  vs best       gold {BEST_GOLD:.3f} (LB 0.891): {macro - BEST_GOLD:+.3f}\\\')\\nif abs(macro - REFERENCE_GOLD) < 0.02:\\n    print("  -> inside the gold 58\\\'s ~0.02 noise floor: this is a null against the reference, whatever the frozen column says.")\\nif FOLD != 1:\\n    print(f"  -> NOTE: this is fold {FOLD}; both references are fold 1, so these deltas are not comparable. Judge fold members by the ensemble\\\'s leaderboard score, not by gold here.")\\nif AVERAGES:\\n    print(\\\'\\\\nP2 -- selection-free weights (gold macro)\\\')\\n    print(f"  {\\\'best mined-val epoch\\\':24s} {macro:.3f}")\\n    for kind, path in AVERAGES.items():\\n        _ids, _probs = gold_predictions(load_for_scoring(path))\\n        _truth = gold.set_index(\\\'StudyInstanceUID\\\').loc[_ids]\\n        _macro = float(np.nanmean([auc(_truth[name].to_numpy().astype(int), _probs[:, i]) for i, name in enumerate(config.LABELS)]))\\n        print(f"  {kind.upper() + \\\' (selection-free)\\\':24s} {_macro:.3f}  ({_macro - macro:+.3f} vs selected)")\\n    print("  gold\\\'s macro SE on 58 studies is ~0.02; treat anything under that as a tie and decide on the leaderboard.")\\nGOLD_PREDICTIONS = f\\\'/kaggle/working/baseline_fold2/gold_predictions_fold{FOLD}.parquet\\\'\\npd.DataFrame(gold_probs, columns=config.LABELS).assign(**{config.ID_COL: gold_ids})[config.SUBMISSION_COLUMNS].to_parquet(GOLD_PREDICTIONS, index=False)\\nprint(\\\'\\\\nwrote\\\', GOLD_PREDICTIONS)\\n\\noof_ids, oof_rows = ([], [])\\nwith torch.no_grad(), torch.amp.autocast(\\\'cuda\\\'):\\n    for batch in val_loader:\\n        logits = scoring_model(batch[\\\'volumes\\\'].to(device), batch[\\\'meta\\\'].to(device), batch[\\\'mask\\\'].to(device))\\n        oof_rows.append(torch.sigmoid(logits.float()).cpu().numpy())\\n        oof_ids += batch[config.ID_COL]\\noof = pd.DataFrame(np.concatenate(oof_rows), columns=config.LABELS)\\noof.insert(0, config.ID_COL, oof_ids)\\noof[\\\'fold\\\'] = FOLD\\nOOF_PATH = f\\\'/kaggle/working/baseline_fold2/oof_fold{FOLD}.parquet\\\'\\noof.to_parquet(OOF_PATH, index=False)\\nprint(f\\\'wrote {OOF_PATH}: {len(oof)} studies x {len(config.LABELS)} findings\\\')\\nprint(f\\\'mean predicted rate: {oof[config.LABELS].to_numpy().mean():.3f}\\\')\\n_spread = oof[config.LABELS].to_numpy().std(axis=0)\\nif float(_spread.min()) < 0.01:\\n    _flat = [n for n, s in zip(config.LABELS, _spread) if s < 0.01]\\n    print(f\\\'  WARNING: near-constant predictions for {_flat} -- check before using this dump for label work\\\')\\n\', \'slice_transformer\': \'\\ndef attached_files(pattern):\\n    import pathlib\\n    found = []\\n    for root in sorted(pathlib.Path("/kaggle/input").iterdir()):\\n        if root.name not in ("competitions", "rsna-knee-abnormality-detection"):\\n            found.extend(root.rglob(pattern))\\n    return found\\n\\nimport hashlib, os, pathlib\\nEXPECTED_FOLD = 2\\nEXPECTED_VAL = 870\\nMATCH_CLASSWISE = False\\nMATCH_BACKBONE = False\\nMATCH_SLICE = True\\nBASELINE_CONTRACT_PATH = "/kaggle/working/baseline_fold2/run_contract.json"\\nos.environ["KNEE_EXPERIMENT"] = \\\'slice_transformer\\\'\\nos.environ["PYTHONUNBUFFERED"] = "1"\\npaths = attached_files("train_labels.parquet")\\nassert len(paths) == 1, f"Expected one label file, found {paths}"\\nassert hashlib.sha256(paths[0].read_bytes()).hexdigest() == \\\'e54e46531aec6cd16131d70aa283c990542dfa17daa890ac54da663cf74569b0\\\', "Label file changed"\\nimport faulthandler\\nfaulthandler.enable()\\n# Keep fatal-signal diagnostics, but do not arm a background stack-dump timer.\\n# Fold 2 and fold 4 SIGSEGVs occurred inside scheduled dumps while training\\n# was progressing. Parent time limits are configured by the staging command.\\n\\n\\nimport os\\nimport subprocess\\nos.environ[\\\'PYTORCH_NVML_BASED_CUDA_CHECK\\\'] = \\\'1\\\'\\nimport torch\\nDEVICES = torch.cuda.device_count()\\nif DEVICES == 0:\\n    raise SystemExit(\\\'Settings -> Accelerator -> GPU T4 x2\\\')\\nprint(subprocess.run([\\\'nvidia-smi\\\', \\\'--query-gpu=name,memory.total\\\', \\\'--format=csv,noheader\\\'], capture_output=True, text=True).stdout.strip())\\nprint(\\\'devices:\\\', DEVICES, \\\'| cuda initialised:\\\', torch.cuda.is_initialized())\\nif DEVICES < 2:\\n    print(\\\'NOTE: one GPU visible. Settings -> Accelerator -> GPU T4 x2 roughly halves wall-clock. The run is correct either way -- ACCUM_STEPS below adapts to hold the effective batch constant.\\\')\\ntry:\\n    import lightning.pytorch as pl\\nexcept ImportError:\\n    try:\\n        import pytorch_lightning as pl\\n    except ImportError:\\n        raise SystemExit(\\\'pip install lightning, then re-run this cell\\\')\\nprint(\\\'lightning:\\\', pl.__version__)\\n\\nimport importlib, pathlib, sys\\nFILES = {\\\'src/__init__.py\\\': \\\'\\\', \\\'src/config.py\\\': \\\'"""Central configuration: label names, cache geometry, and environment paths.\\\\n\\\\nEverything here is imported by both the training pipeline and the submission\\\\nnotebook, so it must stay dependency-free apart from the standard library.\\\\n"""\\\\n\\\\nimport os\\\\nfrom pathlib import Path\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Labels\\\\n# ---------------------------------------------------------------------------\\\\n# Exact submission spelling. The apostrophe in "Baker\\\\\\\'s" and the spaces are\\\\n# part of the column names -- do not normalise them anywhere that touches a\\\\n# submission file.\\\\nLABELS = [\\\\n    "ACL",\\\\n    "MCL",\\\\n    "Medial Meniscus",\\\\n    "Lateral Meniscus",\\\\n    "Medial OA",\\\\n    "Lateral OA",\\\\n    "PF OA",\\\\n    "Effusion",\\\\n    "Synovitis",\\\\n    "Baker\\\\\\\'s",\\\\n    "Contusion",\\\\n    "Fracture",\\\\n]\\\\n\\\\nID_COL = "StudyInstanceUID"\\\\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\\\\n\\\\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\\\\n# any augmentation that mirrors the image must also swap these pairs.\\\\nLATERAL_LABEL_PAIRS = [\\\\n    ("Medial Meniscus", "Lateral Meniscus"),\\\\n    ("Medial OA", "Lateral OA"),\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cache geometry\\\\n# ---------------------------------------------------------------------------\\\\n# Kept as constants so a higher-resolution variant is a config change rather\\\\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\\\\n# x 24,371 series ~= 30 GB, which fits Kaggle\\\\\\\'s 200 GB private dataset limit.\\\\nCACHE_SIZE = 224          # in-plane resolution after resampling\\\\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\\\\nCACHE_DTYPE = "uint8"\\\\n\\\\n# Robust intensity normalisation percentiles. MRI has no standardised units,\\\\n# so every volume is scaled against its own distribution.\\\\nCLIP_PCT_LOW = 1.0\\\\nCLIP_PCT_HIGH = 99.0\\\\n\\\\n# Canonical laterality: left knees are mirrored to match right knees so the\\\\n# medial compartment always falls on the same side of the image.\\\\nCANONICAL_SIDE = "R"\\\\n\\\\n# Mirroring is applied at load time, not baked into the cache.\\\\n#\\\\n# The side tag is blank on most series in this dataset and the series\\\\n# descriptions are anonymised, so roughly 42% of series have no recoverable\\\\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\\\\n# would mean rebuilding all of it every time side detection improves -- and it\\\\n# is expected to improve, via study-level propagation and then a classifier\\\\n# trained on the series whose side is known. The cache therefore stores volumes\\\\n# in their acquired orientation with the side recorded alongside, and the data\\\\n# loader flips on read, which costs nothing.\\\\nMIRROR_IN_CACHE = False\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Paths\\\\n# ---------------------------------------------------------------------------\\\\nCOMPETITION = "rsna-knee-abnormality-detection"\\\\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\\\\n\\\\n\\\\n# What a competition data root must contain to be recognised.\\\\n_MARKERS = ("train_series", "test_series")\\\\n\\\\n\\\\ndef _detect_data_root() -> Path:\\\\n    """Locate the competition data across Kaggle, Colab, and local runs.\\\\n\\\\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\\\\\\\'s input directory is\\\\n    searched by content rather than by name: the mount is not reliably at\\\\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\\\\n    through to the local fallback, which then fails much later with a confusing\\\\n    path in the message.\\\\n    """\\\\n    env = os.environ.get("KNEE_DATA_ROOT")\\\\n    if env:\\\\n        return Path(env)\\\\n\\\\n    kaggle_input = Path("/kaggle/input")\\\\n    if kaggle_input.exists():\\\\n        exact = kaggle_input / COMPETITION\\\\n        if exact.exists():\\\\n            return exact\\\\n        # Search two levels: Kaggle mounts competition data under\\\\n        # /kaggle/input/competitions/<slug>/ as well as directly under\\\\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\\\\n        try:\\\\n            for child in sorted(kaggle_input.iterdir()):\\\\n                if not child.is_dir():\\\\n                    continue\\\\n                if any((child / m).exists() for m in _MARKERS):\\\\n                    return child\\\\n                for grandchild in sorted(child.iterdir()):\\\\n                    if grandchild.is_dir() and any(\\\\n                            (grandchild / m).exists() for m in _MARKERS):\\\\n                        return grandchild\\\\n        except OSError:\\\\n            pass\\\\n\\\\n    return PROJECT_ROOT / "data"\\\\n\\\\n\\\\nDATA_ROOT = _detect_data_root()\\\\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\\\\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\\\\n\\\\nTRAIN_CSV = META_DIR / "train.csv"\\\\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\\\\nTEST_CSV = META_DIR / "test.csv"\\\\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\\\\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\\\\n\\\\nINDEX_PARQUET = META_DIR / "index.parquet"\\\\n\\\\nON_KAGGLE = Path("/kaggle").exists()\\\\n\\\', \\\'src/data/__init__.py\\\': \\\'\\\', \\\'src/data/dicom_io.py\\\': \\\'"""DICOM reading, geometric slice ordering, and laterality resolution.\\\\n\\\\nTwo operations here are load-bearing and fail silently if done naively:\\\\n\\\\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\\\\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\\\\n   slice normal derived from ImageOrientationPatient, which is defined by the\\\\n   scanner geometry rather than by convention.\\\\n\\\\n2. **Laterality.** Four of the twelve labels are compartment-specific\\\\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\\\\n   on the opposite side of the frame from a right knee, so left knees must be\\\\n   mirrored into a canonical orientation before a model can learn the\\\\n   distinction.\\\\n"""\\\\n\\\\nimport re\\\\nfrom pathlib import Path\\\\nfrom typing import Iterable, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pydicom\\\\n\\\\n# Header fields pulled during the index scan. Kept explicit so the parquet\\\\n# schema is stable and the scan stays header-only.\\\\nHEADER_FIELDS = [\\\\n    "StudyInstanceUID",\\\\n    "SeriesInstanceUID",\\\\n    "SOPInstanceUID",\\\\n    "Rows",\\\\n    "Columns",\\\\n    "PixelSpacing",\\\\n    "ImagePositionPatient",\\\\n    "ImageOrientationPatient",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "ScanningSequence",\\\\n    "SequenceVariant",\\\\n    # (0018,0010). The one header that states contrast administration\\\\n    # outright, and the index built before 2026-08-31 does not carry it --\\\\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\\\\n    # which are anonymised away. Absence of a string is not absence of\\\\n    # contrast; absence of this tag is at least a direct question asked.\\\\n    "ContrastBolusAgent",\\\\n    "SeriesDescription",\\\\n    "ProtocolName",\\\\n    "BodyPartExamined",\\\\n    "Manufacturer",\\\\n    "ManufacturerModelName",\\\\n    "MagneticFieldStrength",\\\\n    "ImageLaterality",\\\\n    "Laterality",\\\\n    "InstanceNumber",\\\\n]\\\\n\\\\n# DICOM DS/IS values carry their original string representation, which would\\\\n# land in the index as object dtype. Cast these explicitly.\\\\nNUMERIC_FIELDS = {\\\\n    "Rows",\\\\n    "Columns",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "MagneticFieldStrength",\\\\n    "InstanceNumber",\\\\n}\\\\n\\\\n_LEFT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n_RIGHT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n\\\\n\\\\ndef read_header(path) -> dict:\\\\n    """Read one DICOM\\\\\\\'s headers without decoding pixel data."""\\\\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\\\\n    row = {"path": str(path)}\\\\n    for field in HEADER_FIELDS:\\\\n        value = getattr(ds, field, None)\\\\n        if value is None:\\\\n            row[field] = None\\\\n        elif isinstance(value, pydicom.multival.MultiValue):\\\\n            # Geometry fields are numeric; ScanningSequence and friends are\\\\n            # multi-valued strings, so fall back to a joined token.\\\\n            try:\\\\n                row[field] = [float(v) for v in value]\\\\n            except (TypeError, ValueError):\\\\n                row[field] = "\\\\\\\\\\\\\\\\".join(str(v) for v in value)\\\\n        elif field in NUMERIC_FIELDS:\\\\n            try:\\\\n                row[field] = float(value)\\\\n            except (TypeError, ValueError):\\\\n                row[field] = None\\\\n        elif isinstance(value, (int, float, str)):\\\\n            row[field] = str(value) if isinstance(value, str) else value\\\\n        else:\\\\n            row[field] = str(value)\\\\n    return row\\\\n\\\\n\\\\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\\\\n    """Unit normal of the image plane, from the two direction cosines.\\\\n\\\\n    Falls back to the z axis when orientation is missing, which keeps a\\\\n    malformed series ordered by patient-superior position rather than crashing.\\\\n    """\\\\n    if orientation is None or len(orientation) != 6:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n\\\\n    row_cosine = np.asarray(orientation[:3], dtype=float)\\\\n    col_cosine = np.asarray(orientation[3:], dtype=float)\\\\n    normal = np.cross(row_cosine, col_cosine)\\\\n\\\\n    magnitude = np.linalg.norm(normal)\\\\n    if magnitude == 0:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n    return normal / magnitude\\\\n\\\\n\\\\ndef slice_position(header: dict) -> float:\\\\n    """Scalar position of a slice along its own stack axis."""\\\\n    position = header.get("ImagePositionPatient")\\\\n    if position is None or len(position) != 3:\\\\n        # No geometry available; fall back to InstanceNumber so ordering is at\\\\n        # least deterministic. The index scan flags these series separately.\\\\n        number = header.get("InstanceNumber")\\\\n        return float(number) if number is not None else 0.0\\\\n\\\\n    normal = slice_normal(header.get("ImageOrientationPatient"))\\\\n    return float(np.dot(np.asarray(position, dtype=float), normal))\\\\n\\\\n\\\\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\\\\n    """Order a series\\\\\\\' slices along the acquisition axis."""\\\\n    return sorted(headers, key=slice_position)\\\\n\\\\n\\\\ndef has_geometry(headers: Sequence[dict]) -> bool:\\\\n    """True when every slice carries the position and orientation tags."""\\\\n    for header in headers:\\\\n        position = header.get("ImagePositionPatient")\\\\n        orientation = header.get("ImageOrientationPatient")\\\\n        if position is None or len(position) != 3:\\\\n            return False\\\\n        if orientation is None or len(orientation) != 6:\\\\n            return False\\\\n    return True\\\\n\\\\n\\\\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\\\\n    """Determine whether a series images a left or right knee.\\\\n\\\\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\\\\n    description fields, which carry side markers in several languages. Returns\\\\n    "L", "R", or None when the side cannot be established -- callers must treat\\\\n    None as unusable rather than guessing, since a wrong side silently swaps the\\\\n    medial and lateral labels.\\\\n    """\\\\n    for header in headers:\\\\n        for field in ("ImageLaterality", "Laterality"):\\\\n            value = header.get(field)\\\\n            if isinstance(value, str):\\\\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\\\\n                # often present-but-empty than absent. An empty string is not a\\\\n                # side and must not be read as one.\\\\n                token = value.strip().upper()\\\\n                if token in ("L", "LEFT"):\\\\n                    return "L"\\\\n                if token in ("R", "RIGHT"):\\\\n                    return "R"\\\\n\\\\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\\\\n        for header in headers:\\\\n            text = header.get(field)\\\\n            if not isinstance(text, str):\\\\n                continue\\\\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\\\\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\\\\n            left = bool(_LEFT_PATTERN.search(spaced))\\\\n            right = bool(_RIGHT_PATTERN.search(spaced))\\\\n            # Ambiguous strings mentioning both sides tell us nothing.\\\\n            if left and not right:\\\\n                return "L"\\\\n            if right and not left:\\\\n                return "R"\\\\n\\\\n    return None\\\\n\\\\n\\\\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\\\\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\\\\n\\\\n    Applies the rescale slope and intercept when present. Slices whose in-plane\\\\n    shape differs from the first are skipped rather than silently broadcast.\\\\n    """\\\\n    slices = []\\\\n    reference_shape = None\\\\n    skipped = 0\\\\n\\\\n    for path in paths:\\\\n        try:\\\\n            ds = pydicom.dcmread(str(path), force=True)\\\\n            pixels = ds.pixel_array.astype(np.float32)\\\\n        except Exception:  # noqa: BLE001\\\\n            # A truncated or malformed file loses one slice, not the series.\\\\n            # The dataset contains files whose declared pixel length exceeds\\\\n            # what is actually stored, and dropping 23 good slices over one\\\\n            # bad one costs a whole study its imaging.\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\\\\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\\\\n        if slope != 1.0 or intercept != 0.0:\\\\n            pixels = pixels * slope + intercept\\\\n\\\\n        if reference_shape is None:\\\\n            reference_shape = pixels.shape\\\\n        elif pixels.shape != reference_shape:\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slices.append(pixels)\\\\n\\\\n    if not slices:\\\\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\\\\n\\\\n    return np.stack(slices, axis=0)\\\\n\\\\n\\\\ndef series_paths(series_dir) -> List[str]:\\\\n    """All DICOM files under one series directory."""\\\\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\\\\n\\\\n\\\\ndef resolve_study_laterality(series_sides: dict) -> dict:\\\\n    """Fill in unknown series sides from other series in the same study.\\\\n\\\\n    A knee MRI study images one knee, so a side established on any series\\\\n    applies to all of them. This matters because the DICOM side tags are\\\\n    populated inconsistently across the contributing institutions -- a study\\\\n    can carry the tag on its coronal series and omit it on the sagittals.\\\\n\\\\n    Report text is deliberately not used as a source here. Reports exist only\\\\n    for training studies, so recovering a side from them would mirror the\\\\n    training volumes and leave the test volumes untouched -- a train/test skew\\\\n    that is worse than the missing side it fixes.\\\\n\\\\n    Args:\\\\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\\\\n\\\\n    Returns:\\\\n        The same structure with unknown entries filled where the study allows.\\\\n        Studies whose series disagree are left untouched, since a conflict\\\\n        means the assumption of one knee per study does not hold there.\\\\n    """\\\\n    resolved = {}\\\\n    for study_uid, series in series_sides.items():\\\\n        known = {side for side in series.values() if side in ("L", "R")}\\\\n\\\\n        if len(known) == 1:\\\\n            side = known.pop()\\\\n            resolved[study_uid] = {uid: side for uid in series}\\\\n        else:\\\\n            # No evidence, or contradictory evidence. Guessing would silently\\\\n            # swap the medial and lateral labels on four of twelve findings.\\\\n            resolved[study_uid] = dict(series)\\\\n\\\\n    return resolved\\\\n\\\\n\\\\ndef laterality_coverage(series_sides: dict) -> dict:\\\\n    """Counts before and after study-level propagation, for auditing."""\\\\n    before = sum(1 for s in series_sides.values() for v in s.values()\\\\n                 if v in ("L", "R"))\\\\n    total = sum(len(s) for s in series_sides.values())\\\\n\\\\n    filled = resolve_study_laterality(series_sides)\\\\n    after = sum(1 for s in filled.values() for v in s.values()\\\\n                if v in ("L", "R"))\\\\n\\\\n    conflicts = sum(\\\\n        1 for s in series_sides.values()\\\\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\\\\n    )\\\\n\\\\n    return {\\\\n        "total_series": total,\\\\n        "known_before": before,\\\\n        "known_after": after,\\\\n        "recovered": after - before,\\\\n        "conflicting_studies": conflicts,\\\\n    }\\\\n\\\', \\\'src/data/cache.py\\\': \\\'"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\\\\n\\\\nThe same code path runs during cache construction and at submission time. Any\\\\ndivergence between the two produces a train/test preprocessing skew, which is\\\\ninvisible in local validation and only shows up as a collapsed leaderboard\\\\nscore -- so callers must import from here rather than reimplementing.\\\\n\\\\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\\\\norientation it was acquired in. Mirroring to a canonical side happens at load\\\\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport cv2\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\\\\n    """Scale a volume to uint8 using its own robust percentile range.\\\\n\\\\n    MRI carries no standardised units -- intensity depends on scanner, field\\\\n    strength, coil, and whether fat suppression was applied -- so each volume is\\\\n    scaled against its own distribution rather than a global constant.\\\\n    """\\\\n    finite = volume[np.isfinite(volume)]\\\\n    if finite.size == 0:\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\\\\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\\\\n        # Flat or degenerate volume; nothing to stretch.\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\\\\n    return (scaled * 255.0).round().astype(np.uint8)\\\\n\\\\n\\\\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\\\\n    """Resample every slice to size x size with area/linear interpolation."""\\\\n    size = size or config.CACHE_SIZE\\\\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\\\\n\\\\n    for i, plane in enumerate(volume):\\\\n        # Area interpolation avoids the aliasing that bilinear introduces when\\\\n        # downsampling 800px acquisitions; linear is correct when upsampling.\\\\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\\\\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\\\\n\\\\n    return output\\\\n\\\\n\\\\ndef select_slices(count: int, target: int = None) -> List[int]:\\\\n    """Indices of `target` slices spread evenly across a stack of `count`.\\\\n\\\\n    Short series are handled by repeating edge slices rather than zero-padding,\\\\n    so the volume stays anatomically continuous.\\\\n    """\\\\n    target = target or config.CACHE_SLICES\\\\n    if count <= 0:\\\\n        raise ValueError("cannot select slices from an empty series")\\\\n    if count == target:\\\\n        return list(range(count))\\\\n\\\\n    positions = np.linspace(0, count - 1, target)\\\\n    return [int(round(p)) for p in positions]\\\\n\\\\n\\\\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\\\\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\\\\n\\\\n    Mirroring a knee means reflecting through the patient\\\\\\\'s left-right axis,\\\\n    and which array axis that corresponds to depends on the acquisition plane:\\\\n\\\\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\\\\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\\\\n\\\\n    A blanket column flip is wrong for sagittal series. It leaves medial and\\\\n    lateral exactly where they were -- they are separated across slices, not\\\\n    across the image -- and instead reflects anterior-posterior, moving the\\\\n    patella to the back of the knee.\\\\n\\\\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\\\\n    """\\\\n    left_right = np.array([1.0, 0.0, 0.0])\\\\n\\\\n    if orientation is None or len(orientation) != 6:\\\\n        # No geometry to reason from. Columns is the common case (coronal and\\\\n        # axial together outnumber sagittal), but the caller should prefer not\\\\n        # to mirror at all when geometry is missing.\\\\n        return 2\\\\n\\\\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\\\\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\\\\n    normal = dicom_io.slice_normal(orientation)\\\\n\\\\n    alignments = [\\\\n        (abs(float(normal @ left_right)), 0),\\\\n        (abs(float(height @ left_right)), 1),\\\\n        (abs(float(width @ left_right)), 2),\\\\n    ]\\\\n    return max(alignments)[1]\\\\n\\\\n\\\\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\\\\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\\\\n    """Mirror left knees so the medial compartment is always on one side.\\\\n\\\\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\\\\n    Lateral OA). Without this the model would have to infer the side before it\\\\n    could place a finding, from a cue that is not reliably in the image.\\\\n\\\\n    The axis reflected is chosen from the acquisition geometry -- see\\\\n    mirror_axis -- because it differs between sagittal and coronal series.\\\\n    """\\\\n    if side is None:\\\\n        # Unknown side: leave untouched and let the caller decide. Guessing\\\\n        # would silently swap four of the twelve labels.\\\\n        return volume\\\\n    if side.upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = mirror_axis(orientation)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef submission_indices(count: int, slices: int) -> List[int]:\\\\n    """Raw slice indices a `slices`-deep submission actually reads.\\\\n\\\\n    **Not** `select_slices(count, slices)`, and the difference is the whole\\\\n    reason this exists. The submission decodes `CACHE_SLICES` and then\\\\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\\\\n    `slices` evenly across the raw stack in one step lands on *different\\\\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\\\\n    then be scored on anatomy it was not trained to read, and nothing would\\\\n    raise -- the shapes agree either way. That is the same failure the\\\\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\\\\n\\\\n    So the two selections are composed rather than collapsed: the coarse pick\\\\n    the cache would have made, then the fine pick the model applies to it.\\\\n    """\\\\n    coarse = select_slices(count)\\\\n    if slices == config.CACHE_SLICES:\\\\n        return coarse\\\\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\\\\n\\\\n\\\\ndef build_series_volume(headers: Sequence[dict],\\\\n                        slices: Optional[int] = None) -> Dict:\\\\n    """Full pipeline for one series: sort, decode, normalise, resize.\\\\n\\\\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\\\\n    not by default. The recorded `laterality` is what the loader mirrors on.\\\\n    Returns the volume plus the provenance needed to audit it later.\\\\n\\\\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\\\\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\\\\n    decoding done for images the model never sees, and inference time is a\\\\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\\\\n    construction is unchanged and every existing shard stays valid.\\\\n\\\\n    **It is not bit-identical to decoding the full depth.**\\\\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\\\\n    computed over whatever was decoded, so a shallower stack shifts the\\\\n    intensity mapping slightly. The slices themselves are the same files --\\\\n    see `submission_indices` -- but the uint8 values differ a little, and the\\\\n    submission notebook measures that difference rather than assuming it away.\\\\n    """\\\\n    if not headers:\\\\n        raise ValueError("no headers supplied")\\\\n\\\\n    depth = config.CACHE_SLICES if slices is None else slices\\\\n    ordered = dicom_io.sort_slices(headers)\\\\n    first_header = ordered[0]\\\\n    side = dicom_io.resolve_laterality(ordered)\\\\n    geometry_ok = dicom_io.has_geometry(ordered)\\\\n\\\\n    indices = submission_indices(len(ordered), depth)\\\\n    selected = [ordered[i] for i in indices]\\\\n\\\\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\\\\n\\\\n    # Slices skipped as unreadable leave the volume short of the requested\\\\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\\\\n    # decoded so the depth is exact regardless of how many files were dropped.\\\\n    if raw.shape[0] != depth:\\\\n        raw = raw[select_slices(raw.shape[0], depth)]\\\\n\\\\n    resized = resize_volume(raw.astype(np.float32))\\\\n    normalized = normalize_intensity(resized)\\\\n\\\\n    # Left as acquired by default; the loader mirrors using the recorded side.\\\\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\\\\n    volume = (canonicalize_side(normalized, side,\\\\n                                first_header.get("ImageOrientationPatient"))\\\\n              if config.MIRROR_IN_CACHE else normalized)\\\\n\\\\n    first = ordered[0]\\\\n    return {\\\\n        "volume": volume,\\\\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\\\\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\\\\n        "n_slices_raw": len(ordered),\\\\n        "laterality": side,\\\\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\\\\n                         and side.upper() != config.CANONICAL_SIDE),\\\\n        "geometry_ok": geometry_ok,\\\\n        "rows": int(first.get("Rows") or 0),\\\\n        "cols": int(first.get("Columns") or 0),\\\\n    }\\\\n\\\\n\\\\ndef build_series_record(item):\\\\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\\\\n\\\\n    Defined at module level deliberately. A worker defined inside a notebook\\\\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\\\\n    spawn start method and survives under fork only by accident. Keeping it\\\\n    importable makes the cache build work the same way everywhere.\\\\n\\\\n    Failures are returned rather than raised: one unreadable series must not\\\\n    end a ninety-minute run, and the manifest records what to investigate.\\\\n    """\\\\n    series_uid, headers = item\\\\n    try:\\\\n        result = build_series_volume(headers)\\\\n        return series_uid, result["volume"], {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": result["StudyInstanceUID"],\\\\n            "n_slices_raw": result["n_slices_raw"],\\\\n            "laterality": result["laterality"],\\\\n            "mirrored": result["mirrored"],\\\\n            "geometry_ok": result["geometry_ok"],\\\\n            "rows": result["rows"],\\\\n            "cols": result["cols"],\\\\n            "error": None,\\\\n        }\\\\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\\\\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\\\\n        return series_uid, None, {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": study_uid,\\\\n            "n_slices_raw": len(headers) if headers else 0,\\\\n            "laterality": None,\\\\n            "mirrored": False,\\\\n            "geometry_ok": False,\\\\n            "rows": 0,\\\\n            "cols": 0,\\\\n            "error": str(exc)[:200],\\\\n        }\\\\n\\\', \\\'src/data/index.py\\\': \\\'"""Header-only scan of the DICOM tree into a single parquet index.\\\\n\\\\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\\\\nthe cost is dominated by file-system traversal rather than decoding. The\\\\nresulting index drives cache construction and lets series be filtered and\\\\naudited without touching the 570 GB of imaging again.\\\\n\\\\nUsage (in a Kaggle notebook):\\\\n\\\\n    from src.data.index import build_index\\\\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\\\\n"""\\\\n\\\\nimport argparse\\\\nimport os\\\\nfrom concurrent.futures import ThreadPoolExecutor\\\\nfrom pathlib import Path\\\\nfrom typing import Iterator, List, Optional\\\\n\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef iter_dicom_paths(root) -> Iterator[Path]:\\\\n    """Yield every .dcm under root/study/series/ without materialising a list."""\\\\n    root = Path(root)\\\\n    for study_dir in sorted(root.iterdir()):\\\\n        if not study_dir.is_dir():\\\\n            continue\\\\n        for series_dir in sorted(study_dir.iterdir()):\\\\n            if not series_dir.is_dir():\\\\n                continue\\\\n            for path in sorted(series_dir.glob("*.dcm")):\\\\n                yield path\\\\n\\\\n\\\\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\\\\n    """Read headers for a batch of files in parallel.\\\\n\\\\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\\\\n    released during file reads and a process pool would only add overhead.\\\\n    """\\\\n    def safe_read(path):\\\\n        try:\\\\n            return dicom_io.read_header(path)\\\\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\\\\n            return {"path": str(path), "error": str(exc)}\\\\n\\\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\\\n        rows = list(pool.map(safe_read, paths))\\\\n\\\\n    return pd.DataFrame(rows)\\\\n\\\\n\\\\ndef build_index(\\\\n    split: str = "train_series",\\\\n    out_path: Optional[str] = None,\\\\n    data_root=None,\\\\n    workers: int = 8,\\\\n    batch_size: int = 5000,\\\\n    limit: Optional[int] = None,\\\\n) -> pd.DataFrame:\\\\n    """Scan one split\\\\\\\'s DICOM tree and write a parquet index.\\\\n\\\\n    Processes in batches so a long scan reports progress and so peak memory\\\\n    stays flat regardless of how many files the split holds.\\\\n    """\\\\n    root = Path(data_root or config.DATA_ROOT) / split\\\\n    if not root.exists():\\\\n        raise FileNotFoundError(f"{root} does not exist")\\\\n\\\\n    try:\\\\n        from tqdm.auto import tqdm\\\\n    except ImportError:  # progress is a convenience, never a requirement\\\\n        tqdm = None\\\\n\\\\n    frames = []\\\\n    batch: List[Path] = []\\\\n    scanned = 0\\\\n\\\\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\\\\n    # counts files without a total rather than forcing a full listing first,\\\\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\\\\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\\\\n\\\\n    for path in iter_dicom_paths(root):\\\\n        batch.append(path)\\\\n        if limit is not None and len(batch) + scanned >= limit:\\\\n            break\\\\n        if len(batch) >= batch_size:\\\\n            frames.append(scan_paths(batch, workers=workers))\\\\n            scanned += len(batch)\\\\n            if progress is not None:\\\\n                progress.update(len(batch))\\\\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\\\\n            else:\\\\n                print(f"  scanned {scanned} files", flush=True)\\\\n            batch = []\\\\n\\\\n    if batch:\\\\n        frames.append(scan_paths(batch, workers=workers))\\\\n        scanned += len(batch)\\\\n        if progress is not None:\\\\n            progress.update(len(batch))\\\\n\\\\n    if progress is not None:\\\\n        progress.close()\\\\n\\\\n    if not frames:\\\\n        raise RuntimeError(f"no DICOM files found under {root}")\\\\n\\\\n    index = pd.concat(frames, ignore_index=True)\\\\n    print(f"scanned {scanned} files across {index[\\\\\\\'SeriesInstanceUID\\\\\\\'].nunique()} series")\\\\n\\\\n    if "error" in index.columns:\\\\n        failures = int(index["error"].notna().sum())\\\\n        if failures:\\\\n            print(f"WARNING: {failures} files failed to parse")\\\\n\\\\n    if out_path:\\\\n        os.makedirs(Path(out_path).parent, exist_ok=True)\\\\n        index.to_parquet(out_path, index=False)\\\\n        print(f"wrote {out_path}")\\\\n\\\\n    return index\\\\n\\\\n\\\\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\\\\n    """Per-series roll-up: slice count, geometry completeness, laterality.\\\\n\\\\n    Surfaces the series that need attention -- missing geometry, unresolvable\\\\n    side -- before an hour of cache construction is spent on them.\\\\n    """\\\\n    records = []\\\\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\\\\n        headers = group.to_dict("records")\\\\n        records.append({\\\\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "n_slices": len(headers),\\\\n            "geometry_ok": dicom_io.has_geometry(headers),\\\\n            "laterality": dicom_io.resolve_laterality(headers),\\\\n            "rows": headers[0].get("Rows"),\\\\n            "cols": headers[0].get("Columns"),\\\\n        })\\\\n\\\\n    return pd.DataFrame(records)\\\\n\\\\n\\\\ndef main():\\\\n    parser = argparse.ArgumentParser(description=__doc__)\\\\n    parser.add_argument("--split", default="train_series")\\\\n    parser.add_argument("--out", default=None)\\\\n    parser.add_argument("--workers", type=int, default=8)\\\\n    parser.add_argument("--limit", type=int, default=None)\\\\n    args = parser.parse_args()\\\\n\\\\n    build_index(split=args.split, out_path=args.out,\\\\n                workers=args.workers, limit=args.limit)\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/__init__.py\\\': \\\'\\\', \\\'src/model/side_classifier.py\\\': \\\'"""Predict whether a cached series images a left or right knee.\\\\n\\\\nThe DICOM side tag is blank on ~41% of series and the descriptions are\\\\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\\\\ncompartment-specific, and placing them requires knowing the side, so the\\\\nremainder is predicted from the imaging itself.\\\\n\\\\nThis trains on the 14,403 series whose side is known and applies to the rest.\\\\nBoth halves come from the same cache, so it behaves identically at submission\\\\ntime -- unlike report text, which exists only for training studies.\\\\n\\\\n**Why the whole ordered stack rather than one representative slice.** For\\\\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\\\\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\\\\nthe left-right axis IS the slice axis, so individual slices look much the same\\\\neither way -- what differs is the direction of travel through the volume, which\\\\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\\\\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\\\\nchannels, which preserves it.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:  # the module is importable without torch for testing\\\\n    torch = None\\\\n    nn = object\\\\n\\\\nfrom src import config\\\\n\\\\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\\\\n# so half resolution is ample and quadruples throughput.\\\\nINPUT_SIZE = 112\\\\n\\\\nSIDES = ["L", "R"]\\\\n\\\\n\\\\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\\\\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\\\\n\\\\n    Slices become channels, keeping their order, so the network can use the\\\\n    medial-to-lateral direction that distinguishes sides on sagittal series.\\\\n    """\\\\n    import cv2\\\\n\\\\n    if volumes.ndim == 3:\\\\n        volumes = volumes[None]\\\\n\\\\n    count, depth, _, _ = volumes.shape\\\\n    output = np.empty((count, depth, size, size), dtype=np.float32)\\\\n\\\\n    for i in range(count):\\\\n        for j in range(depth):\\\\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\\\\n                                      interpolation=cv2.INTER_AREA)\\\\n\\\\n    return output / 255.0\\\\n\\\\n\\\\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\\\\n    """Mirror a prepared batch along each item\\\\\\\'s medial-lateral axis.\\\\n\\\\n    Used for augmentation: mirroring an input flips its label, which doubles\\\\n    the training set and forces the model to key on anatomy rather than on any\\\\n    incidental left/right imbalance in the source institutions.\\\\n\\\\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\\\\n    in the prepared batch slices are dimension 1 and columns dimension 3.\\\\n    """\\\\n    output = np.empty_like(batch)\\\\n    for i, axis in enumerate(plane_axis):\\\\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\\\\n    return output\\\\n\\\\n\\\\nif torch is None:\\\\n    # Importable without torch so the array handling above can be tested on a\\\\n    # machine that never trains. The network itself needs the real thing.\\\\n    SideNet = None\\\\nelse:\\\\n  class SideNet(nn.Module):\\\\n      """A small convolutional classifier over slice-channel input.\\\\n\\\\n      Trained from scratch deliberately. It is a coarse two-class problem with\\\\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\\\\n      competition\\\\\\\'s stance on pretrained weights is still unconfirmed, which this\\\\n      piece of the pipeline should not depend on.\\\\n      """\\\\n\\\\n      def __init__(self, in_channels: int = None, width: int = 32):\\\\n          super().__init__()\\\\n          in_channels = in_channels or config.CACHE_SLICES\\\\n\\\\n          def block(inp, out, stride=2):\\\\n              return nn.Sequential(\\\\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n              )\\\\n\\\\n          self.features = nn.Sequential(\\\\n              block(in_channels, width),        # 112 -> 56\\\\n              block(width, width * 2),          # 56 -> 28\\\\n              block(width * 2, width * 4),      # 28 -> 14\\\\n              block(width * 4, width * 4),      # 14 -> 7\\\\n          )\\\\n          self.head = nn.Sequential(\\\\n              nn.AdaptiveAvgPool2d(1),\\\\n              nn.Flatten(),\\\\n              nn.Dropout(0.2),\\\\n              nn.Linear(width * 4, 1),\\\\n          )\\\\n\\\\n      def forward(self, x):\\\\n          return self.head(self.features(x)).squeeze(-1)\\\\n\\\\n\\\\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\\\\n    """Collapse per-series predictions to one side per study.\\\\n\\\\n    A study is one knee, so its series must agree. Voting across them is\\\\n    strictly more reliable than any single series -- and the observed\\\\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\\\\n\\\\n    Returns {study_uid: "L" | "R"}.\\\\n    """\\\\n    if len(predictions) != len(study_uids):\\\\n        raise ValueError(\\\\n            f"predictions and study_uids differ in length: "\\\\n            f"{len(predictions)} vs {len(study_uids)}")\\\\n    if confidences is None:\\\\n        confidences = np.ones(len(predictions))\\\\n    elif len(confidences) != len(predictions):\\\\n        # zip would truncate silently, shifting weights onto the wrong series.\\\\n        raise ValueError(\\\\n            f"confidences length {len(confidences)} does not match "\\\\n            f"{len(predictions)} predictions")\\\\n\\\\n    scores = {}\\\\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\\\\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\\\\n\\\\n    result = {}\\\\n    for study, entries in scores.items():\\\\n        total = sum(weight for _, weight in entries)\\\\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\\\\n        result[study] = "R" if mean >= 0.5 else "L"\\\\n    return result\\\\n\\\\n\\\\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\\\\n    """Fraction correct, with predictions as probabilities of the right knee."""\\\\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\\\\n\\\', \\\'src/model/embed.py\\\': \\\'"""Frozen DINOv2 slice embeddings.\\\\n\\\\nRuns the backbone once over every cached slice and stores the result, so the\\\\naggregation stack trains on cached features in minutes rather than hours. The\\\\nsame code runs at submission time, which is the point -- a divergence between\\\\nhow features are computed for training and for inference is invisible locally\\\\nand fatal on the leaderboard.\\\\n\\\\n**Why frozen, and why first.** The competition scores a second prize track on\\\\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\\\\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\\\\nthe aggregation design -- which is where the modelling actually happens -- be\\\\niterated dozens of times a day instead of three.\\\\n\\\\n**What is stored per slice.** The CLS token concatenated with the patch tokens\\\\nreduced by `POOL_GRID`:\\\\n\\\\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\\\\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\\\\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\\\\n\\\\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\\\\n\\\\nThe first model used grid 1, and its weakness appeared exactly where the mean\\\\npredicts one: regional findings (effusion, OA, Baker\\\\\\\'s) averaged 0.865 on gold\\\\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\\\\ntear occupies two or three patches out of 256, so a mean over all of them\\\\nburies it in 250-odd patches of normal tissue.\\\\n\\\\nWide embeddings must be projected before the aggregation stack -- see\\\\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\\\\nin it against 8.2M with a projection.\\\\n"""\\\\n\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:\\\\n    torch = None\\\\n    nn = None\\\\n\\\\nfrom src import config\\\\n\\\\n# The registers variant. Registers exist to absorb the high-norm artefact\\\\n# tokens that otherwise contaminate patch features -- precisely what gets\\\\n# pooled here.\\\\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\\\\n\\\\n# timm\\\\\\\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\\\\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\\\\n# position embeddings are interpolated rather than silently mismatched.\\\\nINPUT_SIZE = config.CACHE_SIZE\\\\n\\\\n# DINOv2-S token width.\\\\nTOKEN_DIM = 384\\\\n\\\\n# How the 16x16 patch grid is reduced before storage.\\\\n#\\\\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\\\\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\\\\n#\\\\n# The mean is what the first model used, and its weakness showed up exactly\\\\n# where predicted: regional findings averaged 0.865 on gold while focal ones\\\\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\\\\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\\\\n# attention pooling to find it.\\\\nPOOL_GRID = 1\\\\n\\\\n\\\\ndef embed_dim(grid: int = None) -> int:\\\\n    """Stored width per slice: CLS plus the pooled patch tokens."""\\\\n    grid = POOL_GRID if grid is None else grid\\\\n    tokens = 1 + (grid * grid if grid > 1 else 1)\\\\n    return TOKEN_DIM * tokens\\\\n\\\\n\\\\nEMBED_DIM = embed_dim()\\\\n\\\\n# ImageNet statistics, which DINOv2 was trained under.\\\\nMEAN = (0.485, 0.456, 0.406)\\\\nSTD = (0.229, 0.224, 0.225)\\\\n\\\\n\\\\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\\\\n                  checkpoint_path: Optional[str] = None):\\\\n    """A frozen DINOv2 feature extractor in eval mode.\\\\n\\\\n    `checkpoint_path` loads weights from disk, which the submission notebook\\\\n    needs: internet is disabled there, so timm cannot fetch anything.\\\\n    """\\\\n    import timm\\\\n\\\\n    model = timm.create_model(\\\\n        model_name,\\\\n        pretrained=pretrained and checkpoint_path is None,\\\\n        img_size=INPUT_SIZE,\\\\n        num_classes=0,\\\\n    )\\\\n\\\\n    if checkpoint_path:\\\\n        state = torch.load(checkpoint_path, map_location="cpu")\\\\n        state = state.get("state_dict", state)\\\\n        model.load_state_dict(state, strict=False)\\\\n\\\\n    model.eval()\\\\n    for parameter in model.parameters():\\\\n        parameter.requires_grad = False\\\\n    return model\\\\n\\\\n\\\\ndef normalize(slices):\\\\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\\\\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\\\\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\\\\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\\\\n    return (x - mean) / std\\\\n\\\\n\\\\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\\\\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\\\\n\\\\n    The patch sequence is reshaped back to its square grid before pooling, so\\\\n    neighbouring patches are averaged with their neighbours rather than with\\\\n    whatever happened to sit beside them in a flattened sequence.\\\\n    """\\\\n    import torch.nn.functional as F\\\\n\\\\n    count, n_patches, channels = patches.shape\\\\n    side = int(round(n_patches ** 0.5))\\\\n    if side * side != n_patches:\\\\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\\\\n        return patches.mean(dim=1)\\\\n\\\\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\\\\n    pooled = F.adaptive_avg_pool2d(square, grid)\\\\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\\\\n\\\\n\\\\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\\\\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\\\\n\\\\n    Uses forward_features so the patch tokens are reachable at all; the model\\\\\\\'s\\\\n    default pooled output discards them.\\\\n    """\\\\n    grid = POOL_GRID if grid is None else grid\\\\n\\\\n    with torch.no_grad():\\\\n        tokens = encoder.forward_features(normalize(slices))\\\\n\\\\n        if tokens.ndim == 2:\\\\n            # A backbone that already pools; repeat so the width is predictable.\\\\n            repeats = embed_dim(grid) // tokens.shape[-1]\\\\n            return tokens.repeat(1, repeats)\\\\n\\\\n        cls = tokens[:, 0]\\\\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\\\\n        patches = tokens[:, prefix:]\\\\n\\\\n        if grid > 1:\\\\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\\\\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\\\\n\\\\n\\\\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\\\\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\\\\n\\\\n    KneeNet expects a module mapping slices to features. When features are\\\\n    already computed, the "backbone" is an identity that reshapes -- which\\\\n    keeps one model definition for both the cached and the end-to-end path.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim: int = EMBED_DIM):\\\\n        super().__init__()\\\\n        self.feature_dim = feature_dim\\\\n\\\\n    def forward(self, x):\\\\n        return x\\\\n\\\', \\\'src/model/dataset.py\\\': \\\'"""Study-level dataset over the cached volumes.\\\\n\\\\nJoins four sources into one training sample per study:\\\\n\\\\n    cache shards      the uint8 volumes, memory-mapped\\\\n    manifest          which shard row holds which series\\\\n    fused labels      12 soft targets and 12 sample weights per study\\\\n    side predictions  L/R per series, from the tag or the classifier\\\\n\\\\nA sample is a whole study, not a series. The twelve findings are study-level\\\\nlabels, and a single series often cannot support them -- effusion reads off a\\\\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\\\\nhas to see the study\\\\\\\'s series together.\\\\n\\\\n**Laterality is handled two ways, depending on the path.**\\\\n\\\\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\\\\nside at load time, along the axis the acquisition plane dictates: slice order\\\\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\\\\nand axial, where it lies in-plane. Doing it at load time rather than in the\\\\ncache means side detection can improve without rebuilding 29 GB.\\\\n\\\\nFor the frozen path this is not available -- the embeddings are computed from\\\\nthe volume, so the image can no longer be flipped. There the side travels to\\\\nthe model as a covariate (`is_right`, `side_known`) instead. That still\\\\naddresses the original concern, which was that the model should not have to\\\\n*infer* the side from cues that are not reliably present; it is handed the\\\\nanswer either way. Which of the two performs better is an experiment, not a\\\\nsettled question.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\\\\nPLANE_MIRROR_AXIS = {\\\\n    "Sagittal": 0,   # medial-lateral is the slice axis\\\\n    "Coronal": 2,    # medial-lateral lies in-plane\\\\n    "Axial": 2,\\\\n}\\\\nDEFAULT_MIRROR_AXIS = 2\\\\n\\\\n# Series metadata fed to the model as conditioning. Findings are\\\\n# sequence-specific, and these three are given in both train and test.\\\\n# `is_right` and `side_known` carry laterality as a covariate rather than\\\\n# normalising it away by mirroring. With precomputed frozen embeddings the\\\\n# image can no longer be flipped -- the features are already computed -- so the\\\\n# side is handed to the model instead, which addresses the original concern\\\\n# (the model should not have to *infer* the side) by a different route.\\\\n# Mirroring remains available for the end-to-end path and is an A/B to run,\\\\n# not a settled question.\\\\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\\\\n                "is_coronal", "is_axial", "is_right", "side_known"]\\\\n\\\\n\\\\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\\\\n                  plane: Optional[str]) -> np.ndarray:\\\\n    """Reflect a left knee onto the canonical side, along the correct axis."""\\\\n    if not isinstance(side, str) or side.strip().upper() not in ("L", "R"):\\\\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\\\\n        # on four of the twelve findings.\\\\n        return volume\\\\n    if side.strip().upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef series_metadata(row: pd.Series) -> np.ndarray:\\\\n    """Conditioning vector for one series.\\\\n\\\\n    Fluid sensitivity and fat suppression are kept separate: the data\\\\n    description notes they are often correlated but not equivalent, so\\\\n    collapsing them would discard the cases where they disagree.\\\\n    """\\\\n    plane = str(row.get("Anatomical_Plane", ""))\\\\n    side = row.get("side_predicted")\\\\n    side = side.strip().upper() if isinstance(side, str) else None\\\\n    known = side in ("L", "R")\\\\n    return np.array([\\\\n        sequence_flag(row.get("Fluid_Sensitive")),\\\\n        sequence_flag(row.get("Fat_Suppression")),\\\\n        float(plane == "Sagittal"),\\\\n        float(plane == "Coronal"),\\\\n        float(plane == "Axial"),\\\\n        float(known and side == "R"),\\\\n        float(known),\\\\n    ], dtype=np.float32)\\\\n\\\\n\\\\ndef sequence_flag(value) -> float:\\\\n    """Parse CSV flags without treating NaN or the string \\\\\\\'False\\\\\\\' as true."""\\\\n    if value is None or pd.isna(value):\\\\n        return 0.0\\\\n    if isinstance(value, str):\\\\n        value = value.strip().lower()\\\\n        if value in ("true", "yes", "1", "1.0"):\\\\n            return 1.0\\\\n        if value in ("false", "no", "0", "0.0", ""):\\\\n            return 0.0\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    if value not in (0, 1):\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    return float(value)\\\\n\\\\n\\\\ndef select_series(rows: pd.DataFrame, maximum=None, strategy="uniform", rng=None):\\\\n    """Positions to retain, preserving order and optionally protocol coverage.\\\\n\\\\n    Protocol selection first covers acquisition planes, then distinct\\\\n    (plane, fluid sensitivity, fat suppression) tuples, then repeats. It\\\\n    uses only metadata available at inference, never labels or reports.\\\\n    """\\\\n    if strategy not in ("uniform", "protocol"):\\\\n        raise ValueError(f"unknown series selection: {strategy}")\\\\n    if maximum is not None and maximum < 1:\\\\n        raise ValueError("max_series must be positive or None")\\\\n    count = len(rows)\\\\n    if maximum is None or count <= maximum:\\\\n        return np.arange(count)\\\\n    if strategy == "uniform":\\\\n        chosen = (np.linspace(0, count - 1, maximum).round().astype(int)\\\\n                  if rng is None else rng.choice(count, maximum, replace=False))\\\\n        return np.sort(chosen)\\\\n    protocols = [(str(row.get("Anatomical_Plane", "")),\\\\n                  sequence_flag(row.get("Fluid_Sensitive")),\\\\n                  sequence_flag(row.get("Fat_Suppression")))\\\\n                 for _, row in rows.iterrows()]\\\\n    planes, covered, chosen = set(), set(), []\\\\n    priority = (np.arange(count) if rng is None else rng.permutation(count))\\\\n    while len(chosen) < maximum:\\\\n        remaining = [int(i) for i in priority if i not in chosen]\\\\n        best = max(remaining, key=lambda i: (\\\\n            protocols[i][0] in PLANE_MIRROR_AXIS and protocols[i][0] not in planes,\\\\n            protocols[i] not in covered,\\\\n            min((abs(i - j) for j in chosen), default=0)))\\\\n        chosen.append(best)\\\\n        planes.add(protocols[best][0])\\\\n        covered.add(protocols[best])\\\\n    return np.array(sorted(chosen))\\\\n\\\\n\\\\nclass StudyIndex:\\\\n    """Everything needed to assemble a study, resolved once up front.\\\\n\\\\n    Built from frames rather than files so it can be constructed in a test\\\\n    without touching disk.\\\\n    """\\\\n\\\\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\\\\n                 labels: Optional[pd.DataFrame] = None,\\\\n                 sides: Optional[pd.DataFrame] = None,\\\\n                 max_series: Optional[int] = None,\\\\n                 reports: Optional[pd.DataFrame] = None,\\\\n                 series_selection: str = "uniform"):\\\\n        frame = manifest.merge(\\\\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\\\\n                         "Fat_Suppression", "Anatomical_Plane"]],\\\\n            on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n\\\\n        if sides is not None:\\\\n            frame = frame.merge(\\\\n                sides[["SeriesInstanceUID", "side_predicted"]],\\\\n                on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n        else:\\\\n            frame["side_predicted"] = frame.get("laterality_resolved")\\\\n\\\\n        # Drop series whose volume failed to build; their row in the cache is\\\\n        # zeros and would train the model on blank images.\\\\n        if "error" in frame.columns:\\\\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\\\\n\\\\n        if series_selection == "protocol":\\\\n            frame = frame.sort_values("SeriesInstanceUID", kind="stable").reset_index(drop=True)\\\\n        self.series = frame\\\\n        self.max_series = max_series\\\\n        self.series_selection = series_selection\\\\n        select_series(frame.iloc[:0], max_series, series_selection)\\\\n        self.labels = labels\\\\n\\\\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\\\\n        self._by_study = {\\\\n            study: group.reset_index(drop=True)\\\\n            for study, group in frame.groupby(config.ID_COL)\\\\n        }\\\\n\\\\n        if labels is not None:\\\\n            if labels[config.ID_COL].duplicated().any():\\\\n                raise ValueError("duplicate study IDs in labels")\\\\n            keyed = labels.set_index(config.ID_COL)\\\\n            self.studies = [s for s in self.studies if s in keyed.index]\\\\n            self._labels = keyed\\\\n        else:\\\\n            self._labels = None\\\\n\\\\n        # Report sentence embeddings, a training-time-only signal: reports\\\\n        # exist for train studies and never for test, so these feed an\\\\n        # auxiliary loss rather than the model input. Stored as one float32\\\\n        # matrix rather than a DataFrame so __getitem__ is an array row.\\\\n        if reports is not None:\\\\n            keyed_reports = reports.set_index(config.ID_COL)\\\\n            self._report_rows = {study: i for i, study\\\\n                                 in enumerate(keyed_reports.index)}\\\\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\\\\n            self.report_dim = self._report_matrix.shape[1]\\\\n        else:\\\\n            self._report_rows = None\\\\n            self._report_matrix = None\\\\n            self.report_dim = None\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def series_for(self, study: str, rng=None) -> pd.DataFrame:\\\\n        """Series of a study, capped deterministically when a cap is set.\\\\n\\\\n        The cap exists for the efficiency track: inference cost scales with\\\\n        series count, and a study with a dozen series contributes far more\\\\n        runtime than one with three. Selection is by position rather than at\\\\n        random so a prediction is reproducible.\\\\n\\\\n        `rng` is the training-only exception. The deterministic cap discards\\\\n        the same 3,013 of 24,371 series (12.4%) on every epoch of every fold --\\\\n        1,432 studies carry more series than `max_series` and always show the\\\\n        model the same subset. Passing a generator resamples the cap per epoch,\\\\n        which reaches those series without adding a byte to the cache. Order is\\\\n        preserved, so the series-axis metadata still lines up. Never pass one\\\\n        for validation or inference: a prediction that changes between two\\\\n        identical runs cannot be compared against anything.\\\\n        """\\\\n        rows = self._by_study[study]\\\\n        keep = select_series(rows, self.max_series, self.series_selection, rng)\\\\n        return rows.iloc[keep].reset_index(drop=True)\\\\n\\\\n    def targets_for(self, study: str):\\\\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\\\\n        if self._labels is None or study not in self._labels.index:\\\\n            return None, None\\\\n        row = self._labels.loc[study]\\\\n        targets = np.array([float(row[name]) for name in config.LABELS],\\\\n                           dtype=np.float32)\\\\n        weights = np.array(\\\\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\\\\n            dtype=np.float32)\\\\n        return targets, weights\\\\n\\\\n    def report_for(self, study: str) -> Optional[np.ndarray]:\\\\n        """The study\\\\\\\'s report embedding, or None when unavailable."""\\\\n        if self._report_rows is None or study not in self._report_rows:\\\\n            return None\\\\n        return self._report_matrix[self._report_rows[study]]\\\\n\\\\n\\\\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray],\\\\n               series_rng=None) -> Dict:\\\\n    """Assemble one study: mirrored volumes, conditioning, targets, weights.\\\\n\\\\n    `series_rng` is forwarded to `series_for` and is training-only; see there.\\\\n    """\\\\n    rows = index.series_for(study, rng=series_rng)\\\\n\\\\n    volumes, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        volume = np.asarray(shard[int(row["position"])])\\\\n        volume = mirror_volume(volume, row.get("side_predicted"),\\\\n                               row.get("Anatomical_Plane"))\\\\n        volumes.append(volume)\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(volumes) if volumes else np.zeros(\\\\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\\\\n            dtype=np.uint8),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef load_study_features(index: StudyIndex, study: str,\\\\n                        shards: Dict[str, np.ndarray]) -> Dict:\\\\n    """Assemble one study from precomputed slice embeddings.\\\\n\\\\n    Mirroring is unavailable here -- the features were computed from the image\\\\n    and cannot be flipped after the fact -- so laterality reaches the model\\\\n    through `series_metadata` instead.\\\\n    """\\\\n    rows = index.series_for(study)\\\\n\\\\n    features, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\\\\n                                                                dtype=np.float32),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\\\\n    """Pad a batch of studies to the largest series count, with a mask.\\\\n\\\\n    Series counts vary from one to a dozen, so the batch is padded and masked\\\\n    rather than truncated. The mask must reach the attention pooling: without\\\\n    it, padding rows are averaged in as if they were real series.\\\\n    """\\\\n    import torch\\\\n\\\\n    longest = max(len(item["volumes"]) for item in batch)\\\\n\\\\n    # Per-series shape is taken from the data rather than assumed, so the same\\\\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\\\\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\\\\n                     None)\\\\n    if reference is None:\\\\n        raise ValueError("every study in the batch is empty")\\\\n    per_series = reference.shape[1:]\\\\n    dtype = reference.dtype\\\\n\\\\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\\\\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\\\\n    mask = np.zeros((len(batch), longest), dtype=bool)\\\\n\\\\n    for i, item in enumerate(batch):\\\\n        count = len(item["volumes"])\\\\n        if count:\\\\n            volumes[i, :count] = item["volumes"]\\\\n            meta[i, :count] = item["meta"]\\\\n            mask[i, :count] = True\\\\n\\\\n    out = {\\\\n        config.ID_COL: [item[config.ID_COL] for item in batch],\\\\n        "volumes": torch.from_numpy(volumes),\\\\n        "meta": torch.from_numpy(meta),\\\\n        "mask": torch.from_numpy(mask),\\\\n    }\\\\n\\\\n    if batch[0]["targets"] is not None:\\\\n        out["targets"] = torch.from_numpy(\\\\n            np.stack([item["targets"] for item in batch]))\\\\n        out["weights"] = torch.from_numpy(\\\\n            np.stack([item["weights"] for item in batch]))\\\\n\\\\n    # Report embeddings, present only where the index carries them. Studies\\\\n    # without one get a zero row and a False mask so the auxiliary loss can\\\\n    # exclude them without a ragged batch.\\\\n    reports = [item.get("report") for item in batch]\\\\n    dims = [r.shape[0] for r in reports if r is not None]\\\\n    if dims:\\\\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\\\\n        report_mask = np.zeros(len(batch), dtype=bool)\\\\n        for i, row in enumerate(reports):\\\\n            if row is not None:\\\\n                report[i] = row\\\\n                report_mask[i] = True\\\\n        out["report"] = torch.from_numpy(report)\\\\n        out["report_mask"] = torch.from_numpy(report_mask)\\\\n    return out\\\\n\\\', \\\'src/model/net.py\\\': \\\'"""Study-level model: slices to series to study to twelve logits.\\\\n\\\\nThe backbone is deliberately the least interesting part. It is swappable, and\\\\nwhich one wins is settled by measurement rather than by argument -- at 4,349\\\\nstudies with mined labels, the aggregation above it matters more.\\\\n\\\\n    (B, S, D, H, W) uint8 volumes\\\\n      -> backbone over slices          (B*S*D, F)\\\\n      -> attention pool over slices    (B*S, F)\\\\n      -> FiLM on sequence metadata     (B*S, F)\\\\n      -> transformer over series       (B, F)      masked\\\\n      -> head                          (B, 12)\\\\n\\\\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\\\\nand Baker\\\\\\\'s cysts read off fluid-sensitive series, cartilage loss off\\\\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\\\\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\\\\nevery series through one undifferentiated path throws away free signal.\\\\n\\\\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\\\\nrows must be excluded from both the attention pool and the transformer, or a\\\\nshort study is diluted by rows of zeros that the model reads as real series.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport torch\\\\nimport torch.nn as nn\\\\nimport torch.nn.functional as F\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import META_COLUMNS\\\\n\\\\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\\\\n# backbone expects. Gives each slice a little through-plane context for free.\\\\nSLICE_CONTEXT = 3\\\\n\\\\n\\\\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\\\\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\\\\n\\\\n    Edge slices repeat, which is preferable to zero padding: a black plane at\\\\n    the edge of a volume is a feature the model would learn to key on.\\\\n    """\\\\n    n, depth, height, width = volumes.shape\\\\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\\\\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\\\\n    stacked = torch.stack([previous, volumes, following], dim=2)\\\\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\\\\n\\\\n\\\\n# The statistics DINOv2 was pretrained under. `embed.py` has always applied\\\\n# these on the frozen path; the end-to-end path fed raw [0,1] and ConvNeXt\\\\n# absorbed the shift into its first layer over twelve epochs. A pre-norm ViT\\\\n# is far less forgiving, so the same trunk was being fed a distribution it\\\\n# never saw during pretraining.\\\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\\\n\\\\n# NORMALISATION_BUFFERS -- why input_mean/input_std are persistent.\\\\n#\\\\n# They were registered with persistent=False on the argument that they are\\\\n# constants of the backbone family rather than learned state. That was true\\\\n# and it was still wrong: absent from the state dict, they have to be supplied\\\\n# again at every construction site, a site that forgets raises nothing, and a\\\\n# strict load restores every weight while leaving normalisation off. Two\\\\n# scoring sites forgot. The 24-slice DINOv2 run logged gold 0.864 from the\\\\n# in-memory model and 0.823 from the same weights reloaded -- an\\\\n# ImageNet-normalised trunk scored on raw [0,1].\\\\n#\\\\n# So they travel now, and `KneeNet._load_from_state_dict` handles both\\\\n# directions: a checkpoint that carries them imposes them on whatever the\\\\n# caller built, and one written before this change loads strictly without\\\\n# them. ConvNeXt is unaffected either way -- its defaults are None, so its\\\\n# state dict gains no keys and every existing reference number stands.\\\\n\\\\n# How far below the head\\\\\\\'s learning rate a pretrained trunk should sit. The\\\\n# optimiser puts every parameter in one group at OneCycle\\\\\\\'s peak of\\\\n# `lr * max_lr_multiplier` = 3e-4, which a supervised ConvNeXt survives and a\\\\n# self-supervised ViT does not: it loses its pretrained features during warmup\\\\n# and then relearns them from 4,349 studies, which is the one regime ViTs are\\\\n# worst in. Measured symptom: gold 0.6 at epoch 4 where ConvNeXt reached\\\\n# 0.729, and *below* the same trunk frozen (0.770).\\\\nVIT_BACKBONE_LR_SCALE = 0.1\\\\n\\\\n\\\\ndef backbone_defaults(name: str) -> dict:\\\\n    """Input normalisation and trunk learning-rate scale for a backbone family.\\\\n\\\\n    Returns ConvNeXt\\\\\\\'s existing behaviour unchanged -- no normalisation, one\\\\n    learning rate -- so the 0.861 fold-1 reference stays comparable. That\\\\n    includes ConvNeXt-V2, whose FCMAE weights are a drop-in for V1 at the same\\\\n    `timm` interface, and DINOv3\\\\\\\'s ConvNeXt variants: keeping the convnet path\\\\n    identical is what makes a backbone swap a one-variable experiment.\\\\n\\\\n    ViT-family trunks are the exception on both axes, and both were measured\\\\n    rather than argued -- see VIT_BACKBONE_LR_SCALE.\\\\n    """\\\\n    lowered = name.lower()\\\\n    is_vit = "vit" in lowered or ("dinov" in lowered and "convnext" not in lowered)\\\\n    if is_vit:\\\\n        return {"input_mean": IMAGENET_MEAN, "input_std": IMAGENET_STD,\\\\n                "backbone_lr_scale": VIT_BACKBONE_LR_SCALE}\\\\n    return {"input_mean": None, "input_std": None, "backbone_lr_scale": 1.0}\\\\n\\\\n\\\\ndef build_backbone(name: str, pretrained: bool = True,\\\\n                   image_size: Optional[int] = None):\\\\n    """Create a timm trunk mapping (N, 3, H, W) -> (N, num_features).\\\\n\\\\n    Every backbone this project runs is constructed through here so the\\\\n    training and submission notebooks cannot build different trunks from the\\\\n    same name -- notebook 10 hard-coded ConvNeXt for a while, which would have\\\\n    met a DINOv2 checkpoint with a shape error at submission time.\\\\n\\\\n    ViT-family weights (DINOv2 among them) are published at a fixed input size\\\\n    and timm\\\\\\\'s patch embedding refuses anything else, so the cache\\\\\\\'s size is\\\\n    passed explicitly and the position embeddings are interpolated -- 224 is\\\\n    16x14, which divides cleanly. ConvNets accept any size and reject the\\\\n    argument, hence the fallback rather than an unconditional pass.\\\\n    """\\\\n    import timm\\\\n\\\\n    image_size = config.CACHE_SIZE if image_size is None else image_size\\\\n    kwargs = dict(pretrained=pretrained, num_classes=0,\\\\n                  in_chans=SLICE_CONTEXT)\\\\n    try:\\\\n        return timm.create_model(name, img_size=image_size, **kwargs)\\\\n    except TypeError:\\\\n        return timm.create_model(name, **kwargs)\\\\n\\\\n\\\\nclass AttentionPool(nn.Module):\\\\n    """Masked attention pooling over a sequence of embeddings.\\\\n\\\\n    Preferred to mean pooling because the informative slices are a minority --\\\\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\\\\n    dilutes it by the rest. This is the attention-based MIL pooling of Ilse et\\\\n    al., "Attention-based Deep Multiple Instance Learning" (ICML 2018,\\\\n    arXiv:1802.04712), equation 8: a trainable weighted average, permutation\\\\n    invariant, with the weights produced by a two-layer scorer.\\\\n\\\\n    `gated=True` selects the same paper\\\\\\\'s equation 9, which replaces the single\\\\n    tanh branch with `tanh(Vh) * sigmoid(Uh)`. The stated motivation is that\\\\n    tanh is close to linear over most of its range, so it cannot express the\\\\n    non-linear "how relevant is this instance" function well; the sigmoid\\\\n    branch supplies a multiplicative gate that can suppress an instance\\\\n    outright. It is a strictly larger family -- a gate saturated at 1\\\\n    reproduces the ungated pool -- and it costs one extra `dim x hidden`\\\\n    matrix per pool.\\\\n\\\\n    The two variants produce different state dict keys on purpose. A gated\\\\n    checkpoint loaded into an ungated model raises on the missing keys rather\\\\n    than scoring quietly through the wrong pooling, which is the failure mode\\\\n    every other geometry mismatch in this project has had.\\\\n    """\\\\n\\\\n    def __init__(self, dim: int, hidden: int = 128, gated: bool = False):\\\\n        super().__init__()\\\\n        self.gated = gated\\\\n        if gated:\\\\n            self.attend = nn.Linear(dim, hidden)\\\\n            self.gate = nn.Linear(dim, hidden)\\\\n            self.weigh = nn.Linear(hidden, 1)\\\\n        else:\\\\n            self.score = nn.Sequential(\\\\n                nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\\\\n\\\\n    def scores(self, x: torch.Tensor) -> torch.Tensor:\\\\n        """(B, N, F) -> (B, N) unnormalised attention logits."""\\\\n        if not self.gated:\\\\n            return self.score(x).squeeze(-1)\\\\n        return self.weigh(torch.tanh(self.attend(x))\\\\n                          * torch.sigmoid(self.gate(x))).squeeze(-1)\\\\n\\\\n    def forward(self, x: torch.Tensor,\\\\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\\\\n        # x: (B, N, F), mask: (B, N) with True for real entries.\\\\n        weights = self.scores(x)\\\\n        if mask is not None:\\\\n            weights = weights.masked_fill(~mask, float("-inf"))\\\\n            weights = torch.where(mask.any(dim=1, keepdim=True), weights,\\\\n                                  torch.zeros_like(weights))\\\\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return (x * weights).sum(dim=1)\\\\n\\\\n\\\\nclass ClassAttentionPool(nn.Module):\\\\n    """Separate gated series attention for each finding, sharing the features."""\\\\n\\\\n    def __init__(self, dim: int, n_labels: int, hidden: int = 128):\\\\n        super().__init__()\\\\n        self.attend = nn.Linear(dim, hidden)\\\\n        self.gate = nn.Linear(dim, hidden)\\\\n        self.weigh = nn.Linear(hidden, n_labels)\\\\n\\\\n    def forward(self, x, mask=None):\\\\n        scores = self.weigh(torch.tanh(self.attend(x)) * torch.sigmoid(self.gate(x)))\\\\n        if mask is not None:\\\\n            scores = scores.masked_fill(~mask.unsqueeze(-1), float("-inf"))\\\\n            scores = torch.where(mask.any(dim=1)[:, None, None], scores,\\\\n                                 torch.zeros_like(scores))\\\\n        weights = scores.softmax(dim=1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return torch.einsum("bsl,bsf->blf", weights, x)\\\\n\\\\n\\\\ndef aggregation_defaults(state: dict) -> dict:\\\\n    """Recover the pooling variant from weights at every scoring site."""\\\\n    return {\\\\n        "slice_transformer": "slice_encoder.project.weight" in state,\\\\n        "gated_pool": "slice_pool.gate.weight" in state,\\\\n        "classwise_pool": ("study_pool.weigh.weight" in state and\\\\n                           state["study_pool.weigh.weight"].shape[0] > 1),\\\\n    }\\\\n\\\\n\\\\nclass FiLM(nn.Module):\\\\n    """Feature-wise modulation of series embeddings by sequence metadata."""\\\\n\\\\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\\\\n        super().__init__()\\\\n        self.to_scale_shift = nn.Sequential(\\\\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\\\\n            nn.Linear(dim, dim * 2))\\\\n\\\\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\\\\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\\\\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\\\\n        return x * (1.0 + scale) + shift\\\\n\\\\n\\\\nclass SliceSequenceEncoder(nn.Module):\\\\n    """Contextualize sampled slices before pooling; positions are relative, not mm.\\\\n\\\\n    A narrow residual branch preserves the existing feature interface. Its\\\\n    output starts at zero, so the new model initially reproduces the control.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim, width=256, layers=2, dropout=0.2):\\\\n        super().__init__()\\\\n        self.norm = nn.LayerNorm(feature_dim)\\\\n        self.project = nn.Linear(feature_dim, width)\\\\n        self.position = nn.Linear(1, width)\\\\n        # Construct layers independently (TransformerEncoder otherwise clones\\\\n        # one layer with identical initial values).\\\\n        self.layers = nn.ModuleList([\\\\n            nn.TransformerEncoderLayer(width, 8, width * 2, dropout,\\\\n                                       batch_first=True, norm_first=True)\\\\n            for _ in range(layers)])\\\\n        self.output = nn.Linear(width, feature_dim)\\\\n        nn.init.zeros_(self.output.weight)\\\\n        nn.init.zeros_(self.output.bias)\\\\n\\\\n    def forward(self, features):\\\\n        positions = torch.linspace(-1, 1, features.shape[1],\\\\n                                   device=features.device, dtype=features.dtype)\\\\n        x = self.project(self.norm(features)) + self.position(positions[:, None])[None]\\\\n        for layer in self.layers:\\\\n            x = layer(x)\\\\n        return features + self.output(x)\\\\n\\\\n\\\\nclass KneeNet(nn.Module):\\\\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\\\\n\\\\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\\\\n    a frozen encoder is supported and is the intended first configuration: with\\\\n    the trunk frozen, slice embeddings are computed once and the stack above\\\\n    trains in minutes rather than hours.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\\\\n                 n_labels: int = None, series_layers: int = 2,\\\\n                 n_heads: int = 8, dropout: float = 0.2,\\\\n                 freeze_backbone: bool = False,\\\\n                 input_dim: Optional[int] = None,\\\\n                 input_mean=None, input_std=None,\\\\n                 gated_pool: bool = False,\\\\n                 residual_pairs=None, residual_beta=None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        # None is valid on the frozen path, where the trunk ran offline.\\\\n        self.backbone = backbone if backbone is not None else nn.Identity()\\\\n        self.feature_dim = feature_dim\\\\n        self.classwise_pool = classwise_pool\\\\n        n_labels = n_labels or len(config.LABELS)\\\\n\\\\n        if freeze_backbone:\\\\n            for parameter in self.backbone.parameters():\\\\n                parameter.requires_grad = False\\\\n\\\\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\\\\n        # running the whole stack at that width would put ~85M parameters in\\\\n        # FiLM alone. A projection keeps the aggregation at a sane size.\\\\n        self.project = (nn.Linear(input_dim, feature_dim)\\\\n                        if input_dim and input_dim != feature_dim\\\\n                        else nn.Identity())\\\\n\\\\n        # These now travel in the state dict, and `_load_from_state_dict`\\\\n        # below reconciles the two directions with older files. See\\\\n        # NORMALISATION_BUFFERS for why the earlier persistent=False cost a\\\\n        # measured run.\\\\n        #\\\\n        # Registered as None rather than assigned as a plain attribute: a\\\\n        # plain attribute of the same name makes a later register_buffer\\\\n        # raise "attribute already exists", and adopting a checkpoint\\\\\\\'s\\\\n        # statistics is exactly that late registration.\\\\n        if input_mean is not None:\\\\n            self.register_buffer(\\\\n                "input_mean", torch.tensor(input_mean).view(1, -1, 1, 1))\\\\n            self.register_buffer(\\\\n                "input_std", torch.tensor(input_std).view(1, -1, 1, 1))\\\\n        else:\\\\n            self.register_buffer("input_mean", None)\\\\n            self.register_buffer("input_std", None)\\\\n\\\\n        self.slice_pool = AttentionPool(feature_dim, gated=gated_pool)\\\\n        self.film = FiLM(feature_dim)\\\\n\\\\n        layer = nn.TransformerEncoderLayer(\\\\n            d_model=feature_dim, nhead=n_heads,\\\\n            dim_feedforward=feature_dim * 2, dropout=dropout,\\\\n            batch_first=True, norm_first=True)\\\\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\\\\n        self.study_pool = (ClassAttentionPool(feature_dim, n_labels)\\\\n                           if classwise_pool else AttentionPool(feature_dim, gated=gated_pool))\\\\n\\\\n        self.head = nn.Sequential(\\\\n            nn.LayerNorm(feature_dim),\\\\n            nn.Dropout(dropout),\\\\n            nn.Linear(feature_dim, n_labels))\\\\n\\\\n        # Residual parameterisation: for each (child, parent) pair,\\\\n        #\\\\n        #     child_logit = beta * stopgrad(parent_logit) + g(features)\\\\n        #\\\\n        # with g the head\\\\\\\'s own child column. The plan\\\\\\\'s Stage 1: the\\\\n        # synovitis head reconstructs from the effusion head at R^2 0.55 with\\\\n        # a coefficient of +0.843, so effusion is most of what it computes.\\\\n        # Handing it that term for free, detached, leaves g nothing to learn\\\\n        # except the part effusion does not explain -- which is the part the\\\\n        # conditional metric measures.\\\\n        #\\\\n        # `beta` is calibrated on out-of-fold predictions, not derived from\\\\n        # r_true = 0.403: a rank correlation between two binary labels does\\\\n        # not map onto a logit coefficient. See\\\\n        # `src.model.conditional.calibrate_coupling`.\\\\n        #\\\\n        # Registered as None when unconfigured, exactly like the\\\\n        # normalisation buffers above and for the same reason -- a run that\\\\n        # does not use this writes an unchanged state dict, and a run that\\\\n        # does carries the coupling into the file so a bare\\\\n        # `KneeNet(backbone, dim)` at a scoring site reproduces it instead of\\\\n        # silently scoring an uncoupled model.\\\\n        if residual_pairs:\\\\n            pairs = torch.tensor(list(residual_pairs), dtype=torch.long)\\\\n            if pairs.ndim != 2 or pairs.shape[1] != 2:\\\\n                raise ValueError(\\\\n                    "residual_pairs must be (child, parent) index pairs, got "\\\\n                    f"shape {tuple(pairs.shape)}")\\\\n            if residual_beta is None:\\\\n                raise ValueError("residual_pairs given without residual_beta")\\\\n            values = (list(residual_beta) if hasattr(residual_beta, "__len__")\\\\n                      else [float(residual_beta)] * len(pairs))\\\\n            beta = torch.tensor(values, dtype=torch.float32)\\\\n            if len(beta) != len(pairs):\\\\n                raise ValueError(\\\\n                    f"residual_beta has {len(beta)} values for "\\\\n                    f"{len(pairs)} pairs")\\\\n            self.register_buffer("residual_pairs", pairs)\\\\n            self.register_buffer("residual_beta", beta)\\\\n        else:\\\\n            self.register_buffer("residual_pairs", None)\\\\n            self.register_buffer("residual_beta", None)\\\\n\\\\n        # Add after all existing modules and restore CPU RNG so unchanged\\\\n        # modules (including the Lightning wrapper) initialize identically.\\\\n        with torch.random.fork_rng(devices=[]):\\\\n            self.slice_encoder = (SliceSequenceEncoder(feature_dim, dropout=dropout)\\\\n                                  if slice_transformer else nn.Identity())\\\\n\\\\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\\\\n                              missing_keys, unexpected_keys, error_msgs):\\\\n        """Reconcile the normalisation buffers with checkpoints either side.\\\\n\\\\n        Three cases, and the middle one is the bug this exists for:\\\\n\\\\n        - **The file carries the statistics and this model has none.** Adopt\\\\n          them. A bare `KneeNet(backbone, dim)` built at a scoring site now\\\\n          normalises the way the run that wrote the file did, instead of\\\\n          feeding an ImageNet-normalised trunk raw [0,1] and reading as a\\\\n          worse checkpoint.\\\\n        - **The file predates this change and this model has statistics.**\\\\n          Keep what the caller passed. Every checkpoint written before\\\\n          2026-08-28 is missing the two keys, and a strict load must not\\\\n          start failing on files this project already shipped.\\\\n        - **The file carries statistics and this model has different ones.**\\\\n          The file wins, for the same reason as the first case: the weights\\\\n          and the statistics they were trained under belong together, and the\\\\n          construction site is the thing that has been wrong before.\\\\n\\\\n        Adopting is a `register_buffer` on a slot registered as None in\\\\n        `__init__`, which is why it is registered there rather than assigned.\\\\n        """\\\\n        for name in ("input_mean", "input_std",\\\\n                     "residual_pairs", "residual_beta"):\\\\n            key = prefix + name\\\\n            if key in state_dict:\\\\n                self.register_buffer(name, state_dict[key].clone())\\\\n            elif self._buffers.get(name) is not None:\\\\n                # Pre-fill so the strict load below sees no missing key.\\\\n                state_dict[key] = self._buffers[name]\\\\n        super()._load_from_state_dict(\\\\n            state_dict, prefix, local_metadata, strict, missing_keys,\\\\n            unexpected_keys, error_msgs)\\\\n\\\\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\\\\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\\\\n        count, depth = volumes.shape[0], volumes.shape[1]\\\\n        if volumes.dtype == torch.uint8:\\\\n            volumes = volumes.float().div_(255.0)\\\\n\\\\n        slices = stack_2p5d(volumes)\\\\n        if self.input_mean is not None:\\\\n            slices = (slices - self.input_mean) / self.input_std\\\\n        features = self.backbone(slices)\\\\n        features = features.reshape(count, depth, self.feature_dim)\\\\n        return self.slice_pool(self.slice_encoder(features))\\\\n\\\\n    def pool_study(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                   mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, F) one embedding per study.\\\\n\\\\n        The study representation the head reads. Exposed separately so\\\\n        auxiliary objectives (report-embedding regression) can supervise the\\\\n        representation without their parameters entering this module -- the\\\\n        exported state dict must stay loadable by a bare `KneeNet`.\\\\n        """\\\\n        series = self.film(series, meta)\\\\n\\\\n        # The transformer takes True for positions to IGNORE, the inverse of\\\\n        # our mask.\\\\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\\\\n        # A fully padded row emits NaN from the attention; zero it before\\\\n        # pooling so one short study cannot poison the batch.\\\\n        encoded = torch.nan_to_num(encoded)\\\\n\\\\n        return self.study_pool(encoded, mask=mask)\\\\n\\\\n    def couple(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Add the detached parent term to each configured child logit.\\\\n\\\\n        A no-op unless the model was built (or loaded) with residual pairs, so\\\\n        every checkpoint written before this existed behaves identically.\\\\n\\\\n        The parent term is detached: the point is to relieve the child of\\\\n        learning effusion, not to let the synovitis objective reach back and\\\\n        reshape the effusion head, which is the one head on this finding that\\\\n        is reliable.\\\\n        """\\\\n        if self.residual_pairs is None:\\\\n            return logits\\\\n        child = self.residual_pairs[:, 0]\\\\n        parent = self.residual_pairs[:, 1]\\\\n        addition = torch.zeros_like(logits)\\\\n        addition[:, child] = (self.residual_beta.to(logits.dtype)\\\\n                              * logits[:, parent].detach())\\\\n        return logits + addition\\\\n\\\\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                  mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, n_labels).\\\\n\\\\n        Everything above the backbone. Shared by the end-to-end path and the\\\\n        frozen path so the two cannot drift into different architectures.\\\\n        """\\\\n        return self.classify(self.pool_study(series, meta, mask))\\\\n\\\\n    def classify(self, pooled):\\\\n        logits = self.head(pooled)\\\\n        if self.classwise_pool:\\\\n            logits = logits.diagonal(dim1=1, dim2=2)\\\\n        return self.couple(logits)\\\\n\\\\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\\\\n                         mask: torch.Tensor) -> torch.Tensor:\\\\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\\\\n\\\\n        The backbone has already run, so this starts at the slice pooling.\\\\n        """\\\\n        batch, series, depth, _ = features.shape\\\\n        projected = self.project(features.reshape(batch * series, depth, -1))\\\\n        pooled = self.slice_pool(projected)\\\\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\\\\n                              meta, mask)\\\\n\\\\n    def forward_pooled(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                       mask: torch.Tensor):\\\\n        """End-to-end path returning (logits, pooled study embedding).\\\\n\\\\n        The training loop wants both: logits for the label losses, the pooled\\\\n        representation for auxiliary supervision. Computing them together\\\\n        avoids a second pass through the backbone.\\\\n        """\\\\n        batch, series = volumes.shape[0], volumes.shape[1]\\\\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\\\\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\\\\n        pooled = self.pool_study(embeddings, meta, mask)\\\\n        return self.classify(pooled), pooled\\\\n\\\\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                mask: torch.Tensor) -> torch.Tensor:\\\\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\\\\n        logits, _ = self.forward_pooled(volumes, meta, mask)\\\\n        return logits\\\\n\\\\n\\\\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\\\\n                 weights: torch.Tensor, label_smoothing: float = 0.0,\\\\n                 pos_weight: Optional[torch.Tensor] = None,\\\\n                 symmetric_beta: float = 0.0) -> torch.Tensor:\\\\n    """BCE against soft targets, weighted per study and finding.\\\\n\\\\n    Targets are probabilities rather than 0/1 -- they come from two extractors\\\\n    that disagree often enough that thresholding would teach the model to\\\\n    reproduce extraction errors. Weights carry how much each target is\\\\n    believed: full for agreement, less for conflict, least for silence.\\\\n\\\\n    Three optional terms, all inert at their defaults so the reference runs\\\\n    stay bit-comparable. Each addresses something the plan names:\\\\n\\\\n    `label_smoothing` pulls every target toward 0.5 by `eps`. The mined labels\\\\n    disagree with image-derived ground truth about 18% of the time, and a\\\\n    confident target that is wrong 18% of the time is what late training\\\\n    memorises. Note that soft targets already carry some of this effect --\\\\n    smoothing on top of them is a floor on the target\\\\\\\'s distance from 0.5, not\\\\n    the usual one-hot correction.\\\\n\\\\n    `pos_weight` scales the positive term per finding. The metric is macro\\\\n    AUC, so a rare finding costs exactly as much as a common one; without this\\\\n    the loss is dominated by whatever is prevalent. Passed straight through to\\\\n    torch, which applies it to soft targets correctly.\\\\n\\\\n    `symmetric_beta` adds the reverse cross-entropy term of Wang et al.,\\\\n    "Symmetric Cross Entropy for Robust Learning with Noisy Labels" (ICCV\\\\n    2019). RCE swaps the roles of prediction and target, which bounds the loss\\\\n    a wrong label can contribute: a confidently mislabelled study cannot drag\\\\n    the gradient the way plain CE lets it. Targets are clamped off 0 and 1\\\\n    because RCE takes their log; the clamp is what the paper\\\\\\\'s `A` constant\\\\n    does.\\\\n    """\\\\n    if label_smoothing > 0:\\\\n        targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing\\\\n\\\\n    loss = F.binary_cross_entropy_with_logits(\\\\n        logits, targets, pos_weight=pos_weight, reduction="none")\\\\n\\\\n    if symmetric_beta > 0:\\\\n        probs = torch.sigmoid(logits)\\\\n        clamped = targets.clamp(1e-4, 1.0 - 1e-4)\\\\n        reverse = -(probs * torch.log(clamped)\\\\n                    + (1.0 - probs) * torch.log1p(-clamped))\\\\n        loss = loss + symmetric_beta * reverse\\\\n\\\\n    total = weights.sum()\\\\n    if total <= 0:\\\\n        return loss.mean() * 0.0\\\\n    return (loss * weights).sum() / total\\\\n\\\\n\\\\ndef pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                       ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                       weights: Optional[torch.Tensor] = None,\\\\n                       ref_weights: Optional[torch.Tensor] = None,\\\\n                       min_gap: float = 0.25,\\\\n                       pair_mask: Optional[torch.Tensor] = None,\\\\n                       max_margin: Optional[float] = None) -> torch.Tensor:\\\\n    """Per-finding RankNet loss between a batch and a reference set.\\\\n\\\\n    The metric is macro AUC -- a ranking statistic -- while BCE optimises\\\\n    calibration; this term optimises the ordering directly. Pairs are formed\\\\n    per finding between the current batch (`scores`/`targets`, (B, L), with\\\\n    gradient) and a reference set ((Q, L), typically detached scores from\\\\n    recent steps), because at one study per rank there are no pairs *within*\\\\n    a batch to rank.\\\\n\\\\n    Only pairs whose soft targets differ by at least `min_gap` count: the\\\\n    targets are extractor probabilities, and ordering two studies the\\\\n    extractor itself barely distinguishes would train the model to reproduce\\\\n    extraction noise. Surviving pairs are weighted by the target gap and by\\\\n    both studies\\\\\\\' label weights, so a confident 0.9-vs-0.1 pair outweighs a\\\\n    marginal one. Returns a zero that still touches `scores` when no pair\\\\n    qualifies, so the graph is identical either way.\\\\n\\\\n    `pair_mask` drops pairs before weighting. Broadcastable to (B, Q, L), so\\\\n    it can restrict which findings are ranked, which pairs are comparable, or\\\\n    both. `stratified_rank_loss` below builds one; at None every pair counts\\\\n    and this is the loss the reference runs used, unchanged.\\\\n\\\\n    `max_margin` clamps the logit difference before the softplus, and exists\\\\n    because leaving it unbounded diverged a real run (2026-08-31). A satisfied\\\\n    pair\\\\\\\'s loss decays to zero as its margin grows, but a **violated** pair\\\\\\\'s\\\\n    grows linearly and forever, so when the pair constraints contradict each\\\\n    other -- which is what within-stratum synovitis pairs do, since 90% of the\\\\n    mined targets sit at zero and the surviving pairs are mostly "mentioned vs\\\\n    silent" -- inflating the logit scale reduces loss on the majority faster\\\\n    than it raises it on the minority. The head then saturates, every study\\\\n    ties, AUC lands on exactly 0.5, and the shared trunk goes with it: that run\\\\n    took BCE from 0.60 to 5.99 and macro from 0.86 to 0.71. Clamping caps what\\\\n    any one pair can demand, so a contradictory set cannot pay for scale.\\\\n    Default None leaves every earlier run bit-identical.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    if ref_scores.numel() == 0:\\\\n        return zero\\\\n\\\\n    gap = targets.unsqueeze(1) - ref_targets.unsqueeze(0)      # (B, Q, L)\\\\n    pair_weight = gap.abs() * (gap.abs() >= min_gap)\\\\n    if weights is not None and ref_weights is not None:\\\\n        pair_weight = (pair_weight * weights.unsqueeze(1)\\\\n                       * ref_weights.unsqueeze(0))\\\\n    if pair_mask is not None:\\\\n        pair_weight = pair_weight * pair_mask.to(pair_weight.dtype)\\\\n\\\\n    total = pair_weight.sum()\\\\n    if total <= 0:\\\\n        return zero\\\\n\\\\n    margin = scores.unsqueeze(1) - ref_scores.unsqueeze(0)     # (B, Q, L)\\\\n    if max_margin is not None:\\\\n        margin = margin.clamp(-max_margin, max_margin)\\\\n    loss = F.softplus(-torch.sign(gap) * margin)\\\\n    return (loss * pair_weight).sum() / total\\\\n\\\\n\\\\ndef finding_mask(findings: Sequence[str], n_labels: int = None,\\\\n                 device=None) -> torch.Tensor:\\\\n    """(1, 1, L) mask selecting the findings a loss is allowed to touch.\\\\n\\\\n    The stratified objective is for synovitis. Applying it to all twelve\\\\n    findings would trade away ranking on eleven that are already fine, on a\\\\n    stratification that means nothing for them.\\\\n    """\\\\n    n_labels = n_labels or len(config.LABELS)\\\\n    mask = torch.zeros(n_labels, dtype=torch.float32, device=device)\\\\n    for name in findings:\\\\n        mask[config.LABELS.index(name)] = 1.0\\\\n    return mask.view(1, 1, -1)\\\\n\\\\n\\\\ndef caliper_pairs(strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                  caliper: float) -> torch.Tensor:\\\\n    """(B, Q, 1) mask of pairs whose stratifier scores are within `caliper`.\\\\n\\\\n    Matching on a continuous score rather than splitting it at a threshold,\\\\n    because the mined effusion label carries its own extraction error and a\\\\n    hard split inherits it at the boundary. The predicted effusion score is\\\\n    continuous, scores AUC 0.888, and is available at every training step.\\\\n    """\\\\n    distance = (strata.unsqueeze(1) - ref_strata.unsqueeze(0)).abs()\\\\n    return (distance <= caliper).unsqueeze(-1)\\\\n\\\\n\\\\ndef stratified_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                         ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                         strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                         findings: Sequence[str], caliper: float = 0.15,\\\\n                         weights: Optional[torch.Tensor] = None,\\\\n                         ref_weights: Optional[torch.Tensor] = None,\\\\n                         min_gap: float = 0.25,\\\\n                         max_margin: Optional[float] = 8.0) -> torch.Tensor:\\\\n    """RankNet restricted to pairs from the same effusion stratum.\\\\n\\\\n    Conditional AUC *is* within-stratum pair concordance, so this is not an\\\\n    analogy to the metric -- it is the metric, differentiably. Pairs the\\\\n    conditional number ignores are the ones effusion already orders correctly,\\\\n    and training on them is what produced a synovitis head that reconstructs\\\\n    from the effusion head at R^2 0.55.\\\\n\\\\n    The known risk, stated in the plan and not solved here: stripping effusion\\\\n    removes the one reliable cue and leaves a harder task supervised by labels\\\\n    that are silent in 82.2% of reports against a true prevalence of 46.6%. If\\\\n    label quality binds, this loss makes the finding worse, and the conditional\\\\n    metric is what will show it.\\\\n\\\\n    `strata` is the detached predicted effusion probability for the current\\\\n    batch and `ref_strata` the same for the queued reference studies.\\\\n\\\\n    `max_margin` defaults to a finite 8.0 here rather than to None, because\\\\n    this is the loss whose pairs contradict each other -- see\\\\n    `pairwise_rank_loss`. A margin of 8 logits is a probability ratio of about\\\\n    3,000:1; a pair that wants more than that is not asking for an ordering.\\\\n    """\\\\n    mask = caliper_pairs(strata, ref_strata, caliper) * finding_mask(\\\\n        findings, scores.shape[-1], scores.device)\\\\n    return pairwise_rank_loss(scores, targets, ref_scores, ref_targets,\\\\n                              weights, ref_weights, min_gap=min_gap,\\\\n                              pair_mask=mask, max_margin=max_margin)\\\\n\\\\n\\\\ndef coupling_penalty(scores: torch.Tensor, ref_scores: torch.Tensor,\\\\n                     child: int, parent: int, target: float) -> torch.Tensor:\\\\n    """Push corr(child logit, parent logit) toward `target`, not toward zero.\\\\n\\\\n    The findings genuinely co-occur: on the gold set the odds ratio is 6.09\\\\n    (p = 0.003) and the label correlation is 0.403, against the model\\\\\\\'s 0.651.\\\\n    Driving the two heads to independence optimises toward a value the truth\\\\n    contradicts, so `target` is required and is meant to be the observed\\\\n    coupling rather than 0.\\\\n\\\\n    Batches here are one study per rank, so the correlation is computed over\\\\n    the current batch concatenated with the detached reference queue. Returns\\\\n    a zero that still touches `scores` when there are too few studies to\\\\n    estimate a correlation, so the graph does not change shape between steps.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    pool = (torch.cat([scores, ref_scores], dim=0)\\\\n            if ref_scores.numel() else scores)\\\\n    if pool.shape[0] < 8:\\\\n        return zero\\\\n\\\\n    a = pool[:, child] - pool[:, child].mean()\\\\n    b = pool[:, parent] - pool[:, parent].mean()\\\\n    denominator = a.norm() * b.norm()\\\\n    if denominator <= 0:\\\\n        return zero\\\\n    return ((a * b).sum() / denominator - target) ** 2\\\\n\\\\n\\\\ndef load_knee_state(path, map_location="cpu") -> dict:\\\\n    """Read a `KneeNet` state dict from either checkpoint format.\\\\n\\\\n    The end-to-end path writes two things over its life: a bare state dict\\\\n    from `torch.save(model.state_dict(), ...)`, and a Lightning checkpoint,\\\\n    which nests the same tensors under `state_dict` and prefixes every key\\\\n    with `model.` because `KneeNet` is an attribute of the wrapper. The\\\\n    submission must not care which it was handed -- requiring a manual\\\\n    conversion step is requiring a step that can be forgotten, and it was.\\\\n\\\\n    Lightning checkpoints also carry optimiser state and callback objects,\\\\n    which `weights_only=True` refuses, so that is attempted first and relaxed\\\\n    only on failure. These are files this project wrote.\\\\n    """\\\\n    try:\\\\n        obj = torch.load(path, map_location=map_location, weights_only=True)\\\\n    except TypeError:       # torch too old to know the argument\\\\n        obj = torch.load(path, map_location=map_location)\\\\n    except Exception:       # noqa: BLE001 - a Lightning checkpoint\\\\n        obj = torch.load(path, map_location=map_location, weights_only=False)\\\\n\\\\n    state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj\\\\n    if any(key.startswith("model.") for key in state):\\\\n        state = {key[len("model."):]: value for key, value in state.items()\\\\n                 if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(f"no weights found in {path}")\\\\n    return state\\\\n\\\', \\\'src/model/experiments.py\\\': \\\'"""Pre-registered, single-change experiments. No preset implies a score gain."""\\\\n\\\\nfrom dataclasses import asdict, dataclass, replace\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass Experiment:\\\\n    name: str = "baseline"\\\\n    backbone: str = "convnext_tiny.fb_in22k_ft_in1k"\\\\n    slices: int = 16\\\\n    max_series: int = 5\\\\n    series_selection: str = "uniform"\\\\n    resample: bool = False\\\\n    classwise_pool: bool = False\\\\n    slice_transformer: bool = False\\\\n\\\\n    def to_dict(self):\\\\n        result = asdict(self)\\\\n        # Preserve existing experiment contracts and checkpoint sidecars.\\\\n        if not self.slice_transformer:\\\\n            result.pop("slice_transformer")\\\\n        return result\\\\n\\\\n\\\\nBASELINE = Experiment()\\\\nPRESETS = {\\\\n    "baseline": BASELINE,\\\\n    "resample": replace(BASELINE, name="resample", resample=True),\\\\n    "protocol": replace(BASELINE, name="protocol", series_selection="protocol"),\\\\n    "classwise": replace(BASELINE, name="classwise", classwise_pool=True),\\\\n    "convnextv2": replace(BASELINE, name="convnextv2",\\\\n                          backbone="convnextv2_tiny.fcmae_ft_in22k_in1k"),\\\\n    "slice_transformer": replace(BASELINE, name="slice_transformer",\\\\n                                 series_selection="protocol", slice_transformer=True),\\\\n}\\\\n\\\\n\\\\ndef get_experiment(name):\\\\n    if name not in PRESETS:\\\\n        raise ValueError(f"unknown experiment {name!r}; choose {sorted(PRESETS)}")\\\\n    return PRESETS[name]\\\\n\\\', \\\'src/model/artifacts.py\\\': \\\'"""Explicit model provenance and inference for heterogeneous ensembles."""\\\\n\\\\nimport hashlib\\\\nimport json\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nimport torch\\\\n\\\\nfrom src import config\\\\nfrom src.model import dataset, net\\\\n\\\\n\\\\ndef sha256(path):\\\\n    digest = hashlib.sha256()\\\\n    with open(path, "rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(8 << 20), b""):\\\\n            digest.update(chunk)\\\\n    return digest.hexdigest()\\\\n\\\\n\\\\ndef write_metadata(checkpoint, experiment, fold, train_uids, val_uids):\\\\n    """Bind input geometry and split membership to the actual exported weights."""\\\\n    train_uids, val_uids = sorted(train_uids), sorted(val_uids)\\\\n    if set(train_uids) & set(val_uids):\\\\n        raise ValueError("training and validation studies overlap")\\\\n    payload = {\\\\n        "schema_version": 1, "labels": config.LABELS,\\\\n        "sha256": sha256(checkpoint), "experiment": experiment,\\\\n        "cache_size": config.CACHE_SIZE, "cache_slices": config.CACHE_SLICES,\\\\n        "fold": int(fold), "train_uids": train_uids, "val_uids": val_uids,\\\\n    }\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    path.write_text(json.dumps(payload, indent=2) + "\\\\\\\\n")\\\\n    return path\\\\n\\\\n\\\\ndef load_metadata(checkpoint):\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    data = json.loads(path.read_text())\\\\n    if data["schema_version"] != 1 or data["labels"] != config.LABELS:\\\\n        raise ValueError(f"unsupported schema or label order: {path}")\\\\n    if data["sha256"] != sha256(checkpoint):\\\\n        raise ValueError(f"checkpoint checksum mismatch: {checkpoint}")\\\\n    if data["cache_size"] != config.CACHE_SIZE or data["cache_slices"] != config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint requires a different source cache geometry")\\\\n    spec = data["experiment"]\\\\n    if not 1 <= spec["slices"] <= config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint slice count is outside the cache")\\\\n    dataset.select_series(pd.DataFrame(), spec["max_series"], spec["series_selection"])\\\\n    return data\\\\n\\\\n\\\\ndef load_ensemble(manifest_path, device="cpu"):\\\\n    """Load exactly the members named in JSON, never a directory-wide glob.\\\\n\\\\n    Format: {"members": [{"checkpoint": "run/fold0.pt", "weight": 1.0}]}.\\\\n    Paths are relative to this manifest. Each checkpoint requires its .pt.json\\\\n    sidecar, written by notebook 08. Weights are global, not per finding.\\\\n    """\\\\n    manifest_path = Path(manifest_path)\\\\n    entries = json.loads(manifest_path.read_text())["members"]\\\\n    if not entries:\\\\n        raise ValueError("ensemble has no members")\\\\n    members, digests, identities = [], set(), set()\\\\n    for entry in entries:\\\\n        path = (manifest_path.parent / entry["checkpoint"]).resolve()\\\\n        weight = float(entry.get("weight", 1.0))\\\\n        if not np.isfinite(weight) or weight <= 0:\\\\n            raise ValueError("ensemble weights must be finite and positive")\\\\n        metadata = load_metadata(path)\\\\n        identity = (metadata["experiment"]["name"], metadata["fold"])\\\\n        if metadata["sha256"] in digests or identity in identities:\\\\n            raise ValueError("duplicate checkpoint or experiment/fold in ensemble")\\\\n        digests.add(metadata["sha256"])\\\\n        identities.add(identity)\\\\n        spec = metadata["experiment"]\\\\n        state = net.load_knee_state(path)\\\\n        trunk = net.build_backbone(spec["backbone"], pretrained=False)\\\\n        defaults = net.backbone_defaults(spec["backbone"])\\\\n        model = net.KneeNet(trunk, trunk.num_features,\\\\n                            input_mean=defaults["input_mean"],\\\\n                            input_std=defaults["input_std"],\\\\n                            **net.aggregation_defaults(state))\\\\n        model.load_state_dict(state)\\\\n        members.append({"model": model.to(device).eval(), "spec": spec,\\\\n                        "weight": weight, "metadata": metadata})\\\\n    return members\\\\n\\\\n\\\\n@torch.inference_mode()\\\\ndef predict_members(members, volumes, rows, device="cpu"):\\\\n    """Predict from full acquired volumes and one aligned metadata row per series."""\\\\n    if len(rows) != len(volumes) or not len(rows):\\\\n        raise ValueError("volumes and metadata must be nonempty and aligned")\\\\n    order = np.argsort(rows["SeriesInstanceUID"].astype(str).to_numpy(), kind="stable")\\\\n    rows = rows.iloc[order].reset_index(drop=True)\\\\n    volumes = [volumes[i] for i in order]\\\\n    outputs = []\\\\n    for member in members:\\\\n        spec = member["spec"]\\\\n        chosen = dataset.select_series(rows, spec["max_series"], spec["series_selection"])\\\\n        picked = rows.iloc[chosen]\\\\n        stack = []\\\\n        for i, (_, row) in zip(chosen, picked.iterrows()):\\\\n            volume = dataset.mirror_volume(volumes[i], row.get("side_predicted"),\\\\n                                            row.get("Anatomical_Plane"))\\\\n            if volume.shape != (config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE):\\\\n                raise ValueError(f"unexpected volume geometry: {volume.shape}")\\\\n            keep = np.linspace(0, len(volume) - 1, spec["slices"]).round().astype(int)\\\\n            stack.append(volume[keep])\\\\n        tensor = torch.from_numpy(np.stack(stack)).unsqueeze(0).to(device)\\\\n        meta = torch.from_numpy(np.stack([\\\\n            dataset.series_metadata(row) for _, row in picked.iterrows()\\\\n        ])).unsqueeze(0).to(device)\\\\n        mask = torch.ones(1, len(stack), dtype=torch.bool, device=device)\\\\n        device_type = torch.device(device).type\\\\n        with torch.autocast(device_type, enabled=device_type == "cuda"):\\\\n            logits = member["model"](tensor, meta, mask)\\\\n        probs = logits.float().sigmoid().cpu().numpy()[0]\\\\n        if not np.isfinite(probs).all():\\\\n            raise ValueError("non-finite model prediction")\\\\n        outputs.append(probs)\\\\n    return np.stack(outputs)\\\\n\\\', \\\'src/model/conditional.py\\\': \\\'"""Effusion-stratified conditional AUC: the metric synovitis is judged on.\\\\n\\\\nPooled AUC cannot measure this finding. On the 58 gold studies the *gold\\\\neffusion label by itself* scores 0.698 as a synovitis classifier, against the\\\\nsynovitis head\\\\\\\'s 0.701 -- so a number in that region is obtainable with no\\\\nsynovitis-specific skill at all, and seven consecutive experiments read as\\\\nnull partly because they were scored on it.\\\\n\\\\nConditional AUC is the concordance of positive/negative pairs drawn from\\\\n**within the same effusion stratum**. Cross-stratum pairs, where effusion\\\\nalone orders the pair correctly, are dropped. What remains is the only\\\\nquantity on this gold set that moves when synovitis-specific skill moves:\\\\n\\\\n    pooled       0.701      837 pairs, ~55% of them cross-stratum\\\\n    conditional  0.630      376 within-stratum pairs\\\\n\\\\nThe two are linked by pair mass, which is why `pooled_from_conditional` can\\\\nstate the exact pooled number a conditional target implies on this set --\\\\nconditional 0.75 is pooled 0.755, and a *perfect* conditional score is pooled\\\\n0.867. Anything above that is unreachable without changing the gating.\\\\n\\\\nDeliberately free of torch and of pandas-at-import: this module is imported by\\\\nthe analysis tools, bootstrapped into the notebooks, and used by the training\\\\nloop\\\\\\\'s validation hook, and `src/model/lightning.py` cannot be imported by any\\\\nof the first two without dragging in the Lightning framework.\\\\n\\\\nTwo rules this module exists to enforce, from the plan\\\\\\\'s standing rules:\\\\n\\\\n1. Never judge synovitis on macro, and never on pooled alone.\\\\n2. Compare paired. Independent Hanley-McNeil standard errors are +/-0.07 here\\\\n   and carry between-study variance that a matched comparison removes; a real\\\\n   movement of 0.05 is invisible against them and visible against a paired\\\\n   bootstrap of the difference.\\\\n"""\\\\n\\\\nfrom typing import Dict, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Pooled AUC, and the noise band around it\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef auc(y_true, scores) -> float:\\\\n    """ROC AUC via the rank statistic, ties averaged. Matches the training loop."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n    return float((ranks[positives].sum() - n_pos * (n_pos + 1) / 2)\\\\n                 / (n_pos * n_neg))\\\\n\\\\n\\\\ndef auc_se(a: float, n_pos: int, n_neg: int) -> float:\\\\n    """Hanley & McNeil (1982) standard error of an AUC.\\\\n\\\\n    Printed beside pooled numbers so a difference inside the band is visibly\\\\n    not evidence. Never used to judge a *difference* between two models on the\\\\n    same studies -- that is what `paired_bootstrap` is for.\\\\n    """\\\\n    if not np.isfinite(a) or n_pos < 1 or n_neg < 1:\\\\n        return float("nan")\\\\n    q1 = a / (2.0 - a)\\\\n    q2 = 2.0 * a * a / (1.0 + a)\\\\n    numerator = (a * (1 - a) + (n_pos - 1) * (q1 - a * a)\\\\n                 + (n_neg - 1) * (q2 - a * a))\\\\n    return float(np.sqrt(max(numerator, 0.0) / (n_pos * n_neg)))\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Strata\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef strata_from_scores(effusion, threshold: Optional[float] = None):\\\\n    """Binary effusion strata from a continuous effusion score.\\\\n\\\\n    The plan stratifies on *predicted* effusion rather than on the mined\\\\n    effusion label, because the prediction scores AUC 0.888 while the mined\\\\n    label carries its own extraction error that would blur the strata. The\\\\n    default split is the median, which is what reproduces the plan\\\\\\\'s 0.617\\\\n    against the gold label\\\\\\\'s 0.630 -- close enough that the metric is\\\\n    computable during training, where no gold effusion label exists.\\\\n\\\\n    A median split rather than a fixed probability because the head is not\\\\n    calibrated across runs: a 0.70 cut that balances one model\\\\\\\'s studies\\\\n    puts 80% of another model\\\\\\\'s on one side, and the number would then move\\\\n    between runs for a reason that has nothing to do with synovitis.\\\\n    """\\\\n    effusion = np.asarray(effusion, dtype=float)\\\\n    if threshold is None:\\\\n        threshold = float(np.median(effusion))\\\\n    return (effusion >= threshold).astype(int)\\\\n\\\\n\\\\ndef comparable_pairs(y_true, strata, caliper: Optional[float] = None):\\\\n    """(n_pos, n_neg) boolean mask of pairs that share an effusion stratum.\\\\n\\\\n    With `caliper=None`, `strata` is treated as categorical and pairs must\\\\n    match exactly -- the gold-label and median-split cases. With a caliper,\\\\n    `strata` is a continuous score and a pair is comparable when the two\\\\n    studies\\\\\\\' effusion scores lie within `caliper` of each other, which is the\\\\n    matching rule the stratified training loss uses.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    left, right = strata[positives], strata[~positives]\\\\n    if caliper is None:\\\\n        return left[:, None] == right[None, :]\\\\n    return np.abs(left[:, None].astype(float)\\\\n                  - right[None, :].astype(float)) <= caliper\\\\n\\\\n\\\\ndef _concordance(y_true, scores):\\\\n    """(n_pos, n_neg) matrix of 1 / 0.5 / 0 per pair. Ties count as half."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    difference = scores[positives][:, None] - scores[~positives][None, :]\\\\n    return (difference > 0).astype(float) + 0.5 * (difference == 0)\\\\n\\\\n\\\\ndef conditional_auc(y_true, scores, strata,\\\\n                    caliper: Optional[float] = None) -> float:\\\\n    """Concordance over within-stratum pairs only. NaN if there are none.\\\\n\\\\n    Pair-weighted, not stratum-averaged: the strata are 286 and 90 pairs on\\\\n    the gold set, and averaging them would give the 5-positive stratum equal\\\\n    say with the 22-positive one. Pair weighting is also what makes the\\\\n    conversion to pooled AUC exact.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    total = mask.sum()\\\\n    if total == 0:\\\\n        return float("nan")\\\\n    return float((_concordance(y_true, scores) * mask).sum() / total)\\\\n\\\\n\\\\ndef pair_counts(y_true, strata, caliper: Optional[float] = None) -> Dict[str, int]:\\\\n    """How many pairs the conditional number rests on, and the total."""\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    return {"within": int(mask.sum()), "cross": int((~mask).sum()),\\\\n            "total": int(mask.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Where the error sits\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef pair_decomposition(y_true, scores, strata) -> list:\\\\n    """Every discordant pair, split by the stratum of each member.\\\\n\\\\n    The table in the plan\\\\\\\'s §1.4. Reading it is how the addressable headroom\\\\n    was located: within-stratum pairs are 44.9% of the total and are the only\\\\n    ones a synovitis-specific improvement can move. The cell where a\\\\n    stratum-0 positive meets a stratum-1 negative is ranked worse than chance\\\\n    -- that is the gating tax, and it is paid on five studies.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    concordance = _concordance(y_true, scores)\\\\n    error = 1.0 - concordance\\\\n    total_error = error.sum()\\\\n\\\\n    rows = []\\\\n    for pos_level in sorted(set(strata[positives].tolist()), reverse=True):\\\\n        for neg_level in sorted(set(strata[~positives].tolist()), reverse=True):\\\\n            mask = ((strata[positives] == pos_level)[:, None]\\\\n                    & (strata[~positives] == neg_level)[None, :])\\\\n            pairs = int(mask.sum())\\\\n            if pairs == 0:\\\\n                continue\\\\n            cell_error = float((error * mask).sum())\\\\n            rows.append({\\\\n                "pos_stratum": int(pos_level), "neg_stratum": int(neg_level),\\\\n                "within": pos_level == neg_level,\\\\n                "pairs": pairs,\\\\n                "misranked": cell_error / pairs,\\\\n                "error_share": cell_error / total_error if total_error else float("nan"),\\\\n            })\\\\n    return rows\\\\n\\\\n\\\\ndef pooled_from_conditional(y_true, scores, strata,\\\\n                            targets: Sequence[float]) -> list:\\\\n    """The pooled AUC each conditional target implies, on these studies.\\\\n\\\\n    Cross-stratum concordance is held at what this model already achieves and\\\\n    only the within-stratum mass is moved, so the answer is arithmetic rather\\\\n    than a fit. It exists to keep two conversations in the same units: the\\\\n    leaderboard and every historical table are pooled, the plan\\\\\\\'s targets are\\\\n    conditional, and quoting one as the other is how a 0.63 -> 0.75 move gets\\\\n    dismissed as "+0.05, inside the noise".\\\\n\\\\n    The value at `target=1.0` is the ceiling under an unchanged gating -- 0.867\\\\n    here. It is an arithmetic bound, not a reachable target: one model produces\\\\n    both halves, and a residualised model can buy within-stratum ordering by\\\\n    selling the gating.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata)\\\\n    concordance = _concordance(y_true, scores)\\\\n    within, cross = mask.sum(), (~mask).sum()\\\\n    cross_concordant = float((concordance * ~mask).sum())\\\\n    total = within + cross\\\\n    return [{"conditional": float(t),\\\\n             "pooled": float((cross_concordant + t * within) / total)}\\\\n            for t in targets]\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Uncertainty: paired, always\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef _resamples(n: int, draws: int, seed: int):\\\\n    rng = np.random.default_rng(seed)\\\\n    return rng.integers(0, n, size=(draws, n))\\\\n\\\\n\\\\ndef bootstrap_conditional(y_true, scores, strata, caliper=None,\\\\n                          draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """Percentile CI for one model\\\\\\\'s conditional AUC, plus P(AUC <= 0.5).\\\\n\\\\n    Studies are resampled, not pairs: a study appears in many pairs and\\\\n    resampling pairs would treat those as independent observations and report\\\\n    an interval far narrower than the evidence supports.\\\\n\\\\n    `p_at_or_below_chance` is the fraction of resamples at or under 0.5. It is\\\\n    the number the gate is read on, because "is there any synovitis-specific\\\\n    signal at all" is a question about which side of chance the estimate sits,\\\\n    not about whether it differs from some other model.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    strata = np.asarray(strata)\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = conditional_auc(y_true[index], scores[index], strata[index],\\\\n                                caliper)\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = conditional_auc(y_true, scores, strata, caliper)\\\\n    if values.size == 0:\\\\n        return {"auc": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "sd": float("nan"), "p_at_or_below_chance": float("nan"),\\\\n                "draws": 0}\\\\n    return {"auc": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "sd": float(values.std(ddof=1)),\\\\n            "p_at_or_below_chance": float((values <= 0.5).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\ndef paired_bootstrap(y_true, scores_a, scores_b, strata=None, caliper=None,\\\\n                     draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """CI for (A - B) on the same studies, conditional or pooled.\\\\n\\\\n    `strata=None` compares pooled AUCs; otherwise both models are scored on\\\\n    the same within-stratum pairs. Either way the *same* resample indexes both\\\\n    models, so between-study variance -- which is most of the +/-0.07 an\\\\n    unpaired standard error reports -- cancels.\\\\n\\\\n    `p_a_not_better` is the fraction of resamples where the difference is at\\\\n    or below zero. Read it as the evidence against A, and pre-register the\\\\n    threshold before running, not after seeing it.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores_a = np.asarray(scores_a, dtype=float)\\\\n    scores_b = np.asarray(scores_b, dtype=float)\\\\n    strata = None if strata is None else np.asarray(strata)\\\\n\\\\n    def statistic(y, a, b, s):\\\\n        if s is None:\\\\n            return auc(y, a) - auc(y, b)\\\\n        return (conditional_auc(y, a, s, caliper)\\\\n                - conditional_auc(y, b, s, caliper))\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = statistic(y_true[index], scores_a[index], scores_b[index],\\\\n                          None if strata is None else strata[index])\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = statistic(y_true, scores_a, scores_b, strata)\\\\n    if values.size == 0:\\\\n        return {"delta": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "p_a_not_better": float("nan"), "draws": 0}\\\\n    return {"delta": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "p_a_not_better": float((values <= 0).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Calibrating the residual coupling\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef logit(probability, floor: float = 1e-6):\\\\n    """Probabilities to logits, clipped off the asymptotes."""\\\\n    clipped = np.clip(np.asarray(probability, dtype=float), floor, 1 - floor)\\\\n    return np.log(clipped / (1 - clipped))\\\\n\\\\n\\\\ndef calibrate_coupling(labels, parent_logits, iterations: int = 50) -> dict:\\\\n    """Fit `P(child=1) = sigmoid(a + b * parent_logit)` and return b.\\\\n\\\\n    This is the beta for the residual parameterisation\\\\n    `child_logit = beta * stopgrad(parent_logit) + g(features)`, and where it\\\\n    comes from matters more than its value:\\\\n\\\\n    - **Not** from the model\\\\\\\'s own +0.843 reconstruction coefficient. That is\\\\n      the coupling the current model already has, and reproducing it by\\\\n      construction would hand the residual head exactly the term it is\\\\n      supposed to replace.\\\\n    - **Not** from r_true = 0.403. A rank correlation between two binary\\\\n      labels does not map onto a logit coefficient, and the plan says so\\\\n      explicitly.\\\\n\\\\n    It is fitted out of fold, on labels, by IRLS -- the coupling the *truth*\\\\n    supports between the two findings, measured on whatever label set is\\\\n    trustworthy enough to fit it. `separated` flags the degenerate case where\\\\n    the fit runs away because the parent perfectly orders the child; the\\\\n    coefficient is then meaningless and should not be used.\\\\n    """\\\\n    y = np.asarray(labels, dtype=float)\\\\n    x = np.asarray(parent_logits, dtype=float)\\\\n    ok = np.isfinite(y) & np.isfinite(x)\\\\n    y, x = y[ok], x[ok]\\\\n    if y.size < 10 or y.std() == 0:\\\\n        return {"beta": float("nan"), "intercept": float("nan"),\\\\n                "n": int(y.size), "separated": True}\\\\n\\\\n    design = np.column_stack([np.ones_like(x), x])\\\\n    coefficients = np.zeros(2)\\\\n    for _ in range(iterations):\\\\n        # Clipped before the exponential: under separation the coefficients\\\\n        # run away and the naive form overflows on the way to being flagged.\\\\n        eta = np.clip(design @ coefficients, -700, 700)\\\\n        mu = 1.0 / (1.0 + np.exp(-eta))\\\\n        weight = np.clip(mu * (1 - mu), 1e-9, None)\\\\n        gradient = design.T @ (y - mu)\\\\n        hessian = design.T @ (design * weight[:, None])\\\\n        try:\\\\n            step = np.linalg.solve(hessian, gradient)\\\\n        except np.linalg.LinAlgError:\\\\n            return {"beta": float("nan"), "intercept": float("nan"),\\\\n                    "n": int(y.size), "separated": True}\\\\n        coefficients = coefficients + step\\\\n        if np.max(np.abs(step)) < 1e-8:\\\\n            break\\\\n    separated = bool(np.max(np.abs(coefficients)) > 20)\\\\n    return {"beta": float(coefficients[1]), "intercept": float(coefficients[0]),\\\\n            "n": int(y.size), "separated": separated}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# The Stage 0 gate\\\\n# ---------------------------------------------------------------------------\\\\n#\\\\n# The plan says "clearly > 0.5", "= 0.5" and "ambiguous" without giving\\\\n# numbers. They are fixed here, once, in code, so they cannot be chosen after\\\\n# seeing a result -- and here rather than in the tool because the training\\\\n# notebooks read the same gate and two copies would drift.\\\\n\\\\nGATE_ACTIONS = {\\\\n    "clear": ("synovitis-specific signal survives conditioning on effusion",\\\\n              "Stage 1 -- train the conditional task directly"),\\\\n    "chance": ("the score is a better effusion proxy, nothing more",\\\\n               "Stage 3 (plan §4) or Stage 4 descope (plan §10) -- the plan "\\\\n               "gives both; decide before running either"),\\\\n    "ambiguous": ("expected at n=58; the CI straddles chance",\\\\n                  "Stage 1 at reduced confidence, and prioritise Stage 2 "\\\\n                  "(expand the gold set)"),\\\\n}\\\\n\\\\n\\\\ndef gate_reading(stats: dict) -> str:\\\\n    """clear | chance | ambiguous, from a `bootstrap_conditional` result.\\\\n\\\\n        clear      2.5th percentile above 0.5, and P(AUC <= 0.5) < 0.05\\\\n        chance     point estimate <= 0.55 and P(AUC <= 0.5) >= 0.25\\\\n        ambiguous  anything else -- the expected outcome at n=58\\\\n    """\\\\n    if not np.isfinite(stats.get("auc", float("nan"))):\\\\n        return "ambiguous"\\\\n    if stats["lo"] > 0.5 and stats["p_at_or_below_chance"] < 0.05:\\\\n        return "clear"\\\\n    if stats["auc"] <= 0.55 and stats["p_at_or_below_chance"] >= 0.25:\\\\n        return "chance"\\\\n    return "ambiguous"\\\\n\\\', \\\'src/model/augment.py\\\': \\\'"""Training augmentation for study volumes.\\\\n\\\\nThe pipeline shipped with one augmentation -- a single intensity scale in\\\\n[0.9, 1.1] applied to a whole study -- while the plan (§3.4) calls for what the\\\\nRSNA winners used: geometric transforms, photometric jitter, noise, region\\\\ndropout, and slice-order permutation, all kept consistent across the slices of\\\\na series.\\\\n\\\\n**Series-consistent by construction.** Every transform draws its parameters\\\\nonce per series and applies them to all of that series\\\\\\\' slices. The alternative\\\\n-- fresh parameters per slice -- turns a coherent volume into a stack of\\\\nunrelated images, which is exactly the through-plane structure `stack_2p5d`\\\\nexists to give the backbone.\\\\n\\\\n**What is deliberately absent, and why:**\\\\n\\\\n- **Rotation, perspective, affine.** They need `scipy.ndimage` or `cv2` and run\\\\n  per-slice interpolation over ~80 slices per study; at 4,349 studies an epoch\\\\n  that is a dataloader bottleneck, not a free regulariser. Worth adding behind\\\\n  a measurement of loader throughput, not before.\\\\n- **Arbitrary slice permutation.** The plan lists it, but it would break the\\\\n  2.5D stacking: `stack_2p5d` gives each slice its two neighbours as channels,\\\\n  and a shuffled stack makes those neighbours meaningless. `slice_reverse`\\\\n  is the order change that keeps neighbour structure intact -- it reverses the\\\\n  through-plane direction rather than destroying it.\\\\n- **MixUp.** Not implementable at `BATCH_STUDIES = 1`: a within-batch\\\\n  permutation is the identity. Recorded in `docs/plan.md`.\\\\n\\\\n**Laterality flip is a study-level transform, not a series-level one.** All of\\\\na study\\\\\\\'s series image the same knee, so they must flip together, and the flip\\\\nswaps medial with lateral -- which means `config.LATERAL_LABEL_PAIRS` has to be\\\\nswapped in the targets at the same time. `config.py` has carried that constant\\\\nand that warning since before there was anything to use it. Getting this half\\\\nright (flipping pixels, not labels) corrupts four of the twelve findings\\\\nsilently, which is why the dataset refused to flip at all until now.\\\\n"""\\\\n\\\\nfrom dataclasses import dataclass\\\\nfrom typing import Optional, Tuple\\\\n\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import (DEFAULT_MIRROR_AXIS, META_COLUMNS,\\\\n                               PLANE_MIRROR_AXIS)\\\\n\\\\n# Index of `is_sagittal` in the per-series metadata row. The flip axis depends\\\\n# on the acquisition plane, and this is where that is already encoded -- no\\\\n# need to reach back into the index for `Anatomical_Plane`.\\\\nSAGITTAL_COLUMN = META_COLUMNS.index("is_sagittal")\\\\n\\\\n\\\\ndef flip_volume(volume: np.ndarray, plane=None,\\\\n                sagittal: Optional[bool] = None) -> np.ndarray:\\\\n    """Mirror one series across the medial-lateral axis.\\\\n\\\\n    The axis comes from `dataset.PLANE_MIRROR_AXIS`, the same table\\\\n    `mirror_volume` uses to canonicalise laterality -- deliberately, because a\\\\n    training flip and a test-time flip that disagreed about which axis is\\\\n    medial-lateral would be a mismatch that runs clean and scores worse. On a\\\\n    sagittal series medial-lateral is the slice axis; on coronal and axial it\\\\n    lies in-plane.\\\\n\\\\n    Give it either `plane` (the string from the series metadata) or `sagittal`\\\\n    (the bit already encoded in a metadata row).\\\\n    """\\\\n    if sagittal is not None:\\\\n        axis = PLANE_MIRROR_AXIS["Sagittal"] if sagittal else DEFAULT_MIRROR_AXIS\\\\n    else:\\\\n        axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef lateral_swap_order():\\\\n    """Index permutation that exchanges the medial/lateral finding pairs.\\\\n\\\\n    A mirrored knee puts the medial compartment where the lateral one was, so\\\\n    any prediction or target read off a flipped image has to be permuted by\\\\n    this before it means anything. Used by the training augmentation and by\\\\n    the submission\\\\\\\'s flip TTA, from one definition, so the two cannot drift.\\\\n    """\\\\n    order = list(range(len(config.LABELS)))\\\\n    index_of = {name: i for i, name in enumerate(config.LABELS)}\\\\n    for medial, lateral in config.LATERAL_LABEL_PAIRS:\\\\n        a, b = index_of[medial], index_of[lateral]\\\\n        order[a], order[b] = order[b], order[a]\\\\n    return order\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass AugmentConfig:\\\\n    """Which augmentations run, and how hard.\\\\n\\\\n    The defaults reproduce the incumbent behaviour exactly -- one intensity\\\\n    scale, nothing else -- so `AugmentConfig()` is the reference configuration\\\\n    and every measured result stays comparable. Turn things on one at a time.\\\\n    """\\\\n\\\\n    intensity_scale: Optional[Tuple[float, float]] = (0.9, 1.1)\\\\n    intensity_shift: float = 0.0        # +/- this many 8-bit levels\\\\n    gamma: Optional[Tuple[float, float]] = None\\\\n    noise_std: float = 0.0              # gaussian sigma, in 8-bit levels\\\\n    cutout_boxes: int = 0\\\\n    cutout_fraction: float = 0.25       # box side, as a fraction of the image\\\\n    slice_reverse: float = 0.0          # probability, per series\\\\n    laterality_flip: float = 0.0        # probability, per study\\\\n\\\\n    def __post_init__(self):\\\\n        if not 0.0 <= self.slice_reverse <= 1.0:\\\\n            raise ValueError("slice_reverse must be a probability")\\\\n        if not 0.0 <= self.laterality_flip <= 1.0:\\\\n            raise ValueError("laterality_flip must be a probability")\\\\n        if not 0.0 < self.cutout_fraction <= 1.0:\\\\n            raise ValueError("cutout_fraction must be in (0, 1]")\\\\n\\\\n\\\\nREFERENCE = AugmentConfig()\\\\n\\\\n\\\\ndef augment_series(volume: np.ndarray, rng: np.random.Generator,\\\\n                   cfg: AugmentConfig = REFERENCE) -> np.ndarray:\\\\n    """(D, H, W) uint8 -> (D, H, W) uint8, one parameter draw for the series.\\\\n\\\\n    Photometric work happens in float32 and is clipped back to uint8 once at\\\\n    the end, so a scale followed by a shift does not round twice.\\\\n    """\\\\n    working = volume.astype(np.float32)\\\\n\\\\n    if cfg.intensity_scale is not None:\\\\n        low, high = cfg.intensity_scale\\\\n        working *= rng.uniform(low, high)\\\\n\\\\n    if cfg.intensity_shift:\\\\n        working += rng.uniform(-cfg.intensity_shift, cfg.intensity_shift)\\\\n\\\\n    if cfg.gamma is not None:\\\\n        low, high = cfg.gamma\\\\n        # Gamma is defined on [0, 1]; the volume is 8-bit, so scale in and out.\\\\n        working = np.clip(working, 0, 255)\\\\n        working = 255.0 * np.power(working / 255.0, rng.uniform(low, high))\\\\n\\\\n    if cfg.noise_std > 0:\\\\n        # One noise field for the whole series rather than per slice: the\\\\n        # sensor noise being imitated is spatially structured, and per-slice\\\\n        # noise is trivially averaged away by the slice pooling.\\\\n        working = working + rng.normal(0.0, cfg.noise_std,\\\\n                                       size=working.shape[1:])[None]\\\\n\\\\n    if cfg.cutout_boxes > 0:\\\\n        depth, height, width = working.shape\\\\n        box_h = max(1, int(round(height * cfg.cutout_fraction)))\\\\n        box_w = max(1, int(round(width * cfg.cutout_fraction)))\\\\n        for _ in range(cfg.cutout_boxes):\\\\n            top = int(rng.integers(0, max(1, height - box_h + 1)))\\\\n            left = int(rng.integers(0, max(1, width - box_w + 1)))\\\\n            # The SAME box on every slice: a box that moves through the stack\\\\n            # is a different occlusion per slice, which the slice pooling can\\\\n            # simply average out instead of learning around.\\\\n            working[:, top:top + box_h, left:left + box_w] = 0.0\\\\n\\\\n    if cfg.slice_reverse and rng.random() < cfg.slice_reverse:\\\\n        working = working[::-1]\\\\n\\\\n    return np.clip(working, 0, 255).astype(np.uint8)\\\\n\\\\n\\\\ndef flip_laterality(item: dict, rng: np.random.Generator) -> dict:\\\\n    """Mirror every series of a study and swap the medial/lateral labels.\\\\n\\\\n    Volumes reach here already canonicalised to the right knee, so this maps\\\\n    the study onto the left and back-labels it accordingly -- doubling the\\\\n    laterality distribution the model sees without inventing anatomy.\\\\n\\\\n    The flip axis depends on the acquisition plane, exactly as\\\\n    `dataset.mirror_volume` decides it: on a sagittal series left-right runs\\\\n    *through* the plane, so mirroring reverses the slice order; on coronal and\\\\n    axial it lies in-plane, so it flips the columns. `is_sagittal` in the\\\\n    metadata row is what distinguishes them.\\\\n\\\\n    Targets and weights are permuted by `config.LATERAL_LABEL_PAIRS`. Both, not\\\\n    just the targets: a weight says how much a finding\\\\\\\'s label is believed, and\\\\n    it belongs to the finding, so it has to travel with it.\\\\n    """\\\\n    volumes, meta = item["volumes"], item["meta"]\\\\n    flipped = np.empty_like(volumes)\\\\n    for i in range(volumes.shape[0]):\\\\n        flipped[i] = flip_volume(\\\\n            volumes[i], sagittal=bool(meta[i, SAGITTAL_COLUMN] > 0.5))\\\\n    item["volumes"] = flipped\\\\n\\\\n    order = lateral_swap_order()\\\\n    for key in ("targets", "weights"):\\\\n        if item.get(key) is not None:\\\\n            item[key] = np.asarray(item[key])[order]\\\\n\\\\n    # `is_right` in the metadata now describes the wrong side. Left as it is on\\\\n    # purpose: it is a *prediction* from the side classifier describing the\\\\n    # image as acquired, and the model reads it as conditioning rather than as\\\\n    # a label. Rewriting it here would tell the model the flip happened, which\\\\n    # is precisely the invariance the augmentation exists to teach.\\\\n    return item\\\\n\\\\n\\\\ndef augment_study(item: dict, rng: np.random.Generator,\\\\n                  cfg: AugmentConfig = REFERENCE) -> dict:\\\\n    """Apply the study-level flip, then per-series photometric work."""\\\\n    if cfg.laterality_flip and rng.random() < cfg.laterality_flip:\\\\n        item = flip_laterality(item, rng)\\\\n\\\\n    volumes = item["volumes"]\\\\n    if volumes.shape[0]:\\\\n        item["volumes"] = np.stack(\\\\n            [augment_series(volumes[i], rng, cfg)\\\\n             for i in range(volumes.shape[0])])\\\\n    return item\\\\n\\\', \\\'src/model/averaging.py\\\': \\\'"""Weight averaging: EMA, SWA, and model soups.\\\\n\\\\n**Why this module exists.** Checkpoint selection here is broken in a specific,\\\\ndiagnosable way. Training labels are mined from radiology reports and agree\\\\nwith image-derived ground truth about 82% of the time, and the validation fold\\\\nis mined from the *same* distribution. So the network first fits the structure\\\\nthe two sources share and then memorises the 18% they do not -- Arpit et al.,\\\\n"A Closer Look at Memorization in Deep Networks" (ICML 2017), and Liu et al.,\\\\n"Early-Learning Regularization Prevents Memorization of Noisy Labels" (NeurIPS\\\\n2020). Mined validation agreement keeps *rising* through that second phase,\\\\nbecause agreeing with the noise is what it measures. It is therefore a biased\\\\nselector, not merely a noisy one, and "take the best epoch on validation"\\\\nselects into the memorisation phase on purpose.\\\\n\\\\nMeasured here: mined val rose to 0.860 at epoch 9 and fell to 0.843 by epoch\\\\n20, while gold held flat at 0.864-0.869 across eleven consecutive epochs.\\\\n\\\\nThe fix is to stop selecting. Averaging weights is selection-free:\\\\n\\\\n- **EMA** (Tarvainen & Valpola, NeurIPS 2017) -- an exponential moving average\\\\n  maintained during training. Cheap, standard, and already the default in most\\\\n  detection and segmentation pipelines.\\\\n- **SWA** (Izmailov et al., UAI 2018, arXiv:1803.05407) -- an equal-weight\\\\n  average over the tail of the trajectory, which lands in a flatter and better\\\\n  generalising region than any single point on it.\\\\n- **Model soups** (Wortsman et al., ICML 2022, arXiv:2203.05482) -- average\\\\n  the weights of several finished fine-tunes instead of picking the best one\\\\n  on a held-out set. The paper\\\\\\\'s argument is directed at exactly the recipe\\\\n  failing here: "pick the individual model which performs best on a held-out\\\\n  validation set, discarding the remainder".\\\\n\\\\n**Averaging weights is only meaningful for models in the same basin.** Two\\\\nruns whose aggregation was randomly initialised from different seeds are not,\\\\nand averaging them produces a model that is worse than either -- silently, as\\\\na plausible-looking set of numbers. `assert_soupable` checks what it can\\\\n(shapes and key sets); it cannot check the basin, so the training notebook\\\\nseeds every fold identically, and the docstrings below say when a soup is\\\\nwell-posed.\\\\n\\\\n**Nothing here may select on the gold 58.** `greedy_soup` takes a caller\\\\\\\'s\\\\nscore function precisely so the choice of scoring set stays visible at the\\\\ncall site: it must be mined validation or an out-of-fold dump. Selecting a\\\\nsoup on gold would turn the project\\\\\\\'s one honest measuring stick into a\\\\nmodel-selection set, which has already cost this project an inflated figure\\\\nonce.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\\\\n\\\\nimport torch\\\\n\\\\nState = Dict[str, torch.Tensor]\\\\n\\\\n\\\\ndef _is_averageable(tensor: torch.Tensor) -> bool:\\\\n    """Floating-point tensors average; integer counters and flags do not.\\\\n\\\\n    `num_batches_tracked` on a BatchNorm is the canonical example: averaging\\\\n    it produces a fractional count, and torch stores it as int64 anyway. The\\\\n    aggregation stack is all LayerNorm so this does not arise today, but a\\\\n    convnet backbone with BatchNorm would hit it immediately.\\\\n    """\\\\n    return tensor.is_floating_point()\\\\n\\\\n\\\\ndef assert_soupable(states: Sequence[State]) -> None:\\\\n    """Refuse states that cannot be averaged elementwise.\\\\n\\\\n    Catches the two mechanical failures -- a different architecture, and a\\\\n    checkpoint saved at a different width -- before they become a tensor of\\\\n    the wrong shape several cells later.\\\\n    """\\\\n    if not states:\\\\n        raise ValueError("no states to average")\\\\n    reference = states[0]\\\\n    for i, state in enumerate(states[1:], start=1):\\\\n        if set(state) != set(reference):\\\\n            missing = sorted(set(reference) - set(state))[:3]\\\\n            extra = sorted(set(state) - set(reference))[:3]\\\\n            raise ValueError(\\\\n                f"state {i} has different keys from state 0 "\\\\n                f"(missing {missing}, extra {extra}) -- these are not the "\\\\n                "same architecture")\\\\n        for key, tensor in state.items():\\\\n            if tensor.shape != reference[key].shape:\\\\n                raise ValueError(\\\\n                    f"state {i} has {key} at {tuple(tensor.shape)} against "\\\\n                    f"{tuple(reference[key].shape)} in state 0")\\\\n\\\\n\\\\ndef average_states(states: Sequence[State],\\\\n                   weights: Optional[Sequence[float]] = None) -> State:\\\\n    """Elementwise weighted average of state dicts.\\\\n\\\\n    `weights` defaults to equal, which is SWA and the uniform soup. Averaging\\\\n    happens in float64 and is cast back to each tensor\\\\\\\'s own dtype, so a run\\\\n    of fp16 checkpoints does not accumulate rounding across a dozen members.\\\\n\\\\n    Non-floating tensors are taken from the first state rather than averaged.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if weights is None:\\\\n        weights = [1.0] * len(states)\\\\n    if len(weights) != len(states):\\\\n        raise ValueError(f"{len(weights)} weights for {len(states)} states")\\\\n    total = float(sum(weights))\\\\n    if total <= 0:\\\\n        raise ValueError("weights must sum to something positive")\\\\n\\\\n    averaged: State = {}\\\\n    for key, reference in states[0].items():\\\\n        if not _is_averageable(reference):\\\\n            averaged[key] = reference.clone()\\\\n            continue\\\\n        accumulator = torch.zeros_like(reference, dtype=torch.float64)\\\\n        for state, weight in zip(states, weights):\\\\n            accumulator += state[key].to(torch.float64) * float(weight)\\\\n        averaged[key] = (accumulator / total).to(reference.dtype)\\\\n    return averaged\\\\n\\\\n\\\\nclass WeightAverage:\\\\n    """EMA and SWA of one model\\\\\\\'s weights, maintained during training.\\\\n\\\\n    Both are accumulated from the same call so a run pays one copy of the\\\\n    weights per update and produces both artefacts. `decay` is the EMA\\\\\\\'s, and\\\\n    0.999 is the usual choice; `swa_from` is the step at which the equal-weight\\\\n    tail average starts, so a caller sets it to 75% of the planned steps to\\\\n    average over the last quarter.\\\\n\\\\n    **The averages live wherever the model does, in float32.** Accumulating on\\\\n    the host instead would copy the whole trunk across the PCIe bus on every\\\\n    optimiser step -- ~110 MB for ConvNeXt-T plus the aggregation, twice per\\\\n    step once the SWA window opens, about 11,000 times over a twenty-epoch\\\\n    run. That is a wall-clock tax on the run for no benefit; two shadows on\\\\n    the device cost ~220 MB of a 16 GB T4, which the batch has to spare.\\\\n    Float32 regardless of training precision: under `16-mixed` the parameters\\\\n    are fp32 anyway (autocast does not change them), but an fp16 accumulation\\\\n    over thousands of steps would round away exactly the small updates an EMA\\\\n    exists to keep. `export` moves the result to the host.\\\\n\\\\n    `device` overrides the placement -- pass "cpu" if a run is ever tight\\\\n    enough on device memory that 220 MB matters.\\\\n    """\\\\n\\\\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999,\\\\n                 swa_from: Optional[int] = None, device=None):\\\\n        if not 0.0 < decay < 1.0:\\\\n            raise ValueError(f"decay must be in (0, 1), got {decay}")\\\\n        self.decay = decay\\\\n        self.swa_from = swa_from\\\\n        self.steps = 0\\\\n        self.swa_count = 0\\\\n        state = model.state_dict()\\\\n        self.ema: State = {\\\\n            k: (v.detach().clone().to(torch.float32) if device is None\\\\n                else v.detach().to(device, torch.float32).clone())\\\\n            for k, v in state.items() if _is_averageable(v)}\\\\n        self.swa: State = {k: torch.zeros_like(v) for k, v in self.ema.items()}\\\\n        # Everything not averaged -- integer counters. The normalisation\\\\n        # buffers are float, so they average harmlessly to themselves.\\\\n        self._fixed: State = {k: v.detach().cpu().clone()\\\\n                              for k, v in state.items()\\\\n                              if not _is_averageable(v)}\\\\n\\\\n    @torch.no_grad()\\\\n    def update(self, model: torch.nn.Module) -> None:\\\\n        """Fold one optimiser step\\\\\\\'s weights into both averages."""\\\\n        self.steps += 1\\\\n        state = model.state_dict()\\\\n        in_window = (self.swa_from is not None and self.steps >= self.swa_from)\\\\n        if in_window:\\\\n            self.swa_count += 1\\\\n\\\\n        for key, shadow in self.ema.items():\\\\n            # Fetched once and reused by both averages: reading it per average\\\\n            # doubled the copy for every step of the SWA window.\\\\n            current = state[key].detach().to(shadow.device, torch.float32,\\\\n                                             non_blocking=True)\\\\n            shadow.mul_(self.decay).add_(current, alpha=1.0 - self.decay)\\\\n            if in_window:\\\\n                accumulator = self.swa[key]\\\\n                # Running mean, so the tail can be any length without a second\\\\n                # pass or a list of checkpoints held in memory.\\\\n                accumulator.add_((current - accumulator) / self.swa_count)\\\\n\\\\n    def ema_state(self) -> State:\\\\n        """The EMA weights, as a state dict a `KneeNet` loads.\\\\n\\\\n        Moved to the host on the way out: a checkpoint holding CUDA tensors\\\\n        reloads onto a device that may not exist in the process reading it.\\\\n        """\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.ema.items()}}\\\\n\\\\n    def swa_state(self) -> Optional[State]:\\\\n        """The tail average, or None if no step fell inside the window."""\\\\n        if self.swa_count == 0:\\\\n            return None\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.swa.items()}}\\\\n\\\\n\\\\ndef greedy_soup(states: Sequence[State], score: Callable[[State], float],\\\\n                names: Optional[Sequence[str]] = None,\\\\n                tolerance: float = 0.0) -> Tuple[State, List[str]]:\\\\n    """Wortsman et al.\\\\\\\'s greedy soup: add a member only if it helps.\\\\n\\\\n    Members are ranked by their individual score, then considered in order;\\\\n    each is kept if the average *including* it scores at least as well as the\\\\n    average without it. The result is never worse than the best single member\\\\n    on the scoring set, which is what makes greedy preferable to a uniform\\\\n    soup when the members are of uneven quality.\\\\n\\\\n    `score` maps a state dict to a number that is better when larger --\\\\n    macro AUC here. **It must not read the gold 58.** Score on the mined\\\\n    validation fold or on an out-of-fold dump; the whole reason this takes a\\\\n    callable rather than a dataset is to keep that choice at the call site\\\\n    where a reviewer can see it.\\\\n\\\\n    `tolerance` lets a member in when it is within that much of the running\\\\n    best, for the case where the scoring set is too small to resolve the\\\\n    difference and averaging more members is the safer default. At 0.0 this is\\\\n    the paper\\\\\\\'s algorithm exactly.\\\\n\\\\n    Returns the souped state and the names of the members that went into it,\\\\n    in the order they were added.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if names is None:\\\\n        names = [f"member{i}" for i in range(len(states))]\\\\n    if len(names) != len(states):\\\\n        raise ValueError(f"{len(names)} names for {len(states)} states")\\\\n\\\\n    ranked = sorted(zip(names, states), key=lambda pair: score(pair[1]),\\\\n                    reverse=True)\\\\n\\\\n    kept_names: List[str] = [ranked[0][0]]\\\\n    kept_states: List[State] = [ranked[0][1]]\\\\n    best = score(kept_states[0])\\\\n\\\\n    for name, state in ranked[1:]:\\\\n        candidate = average_states(kept_states + [state])\\\\n        value = score(candidate)\\\\n        if value >= best - tolerance:\\\\n            kept_names.append(name)\\\\n            kept_states.append(state)\\\\n            best = max(best, value)\\\\n\\\\n    return average_states(kept_states), kept_names\\\\n\\\\n\\\\ndef rank_average(predictions: Sequence["np.ndarray"]) -> "np.ndarray":\\\\n    """Average ensemble members by within-member rank, not by probability.\\\\n\\\\n    The metric is macro ROC-AUC, which reads only the ordering of studies\\\\n    within a finding. Averaging probabilities lets a member with a compressed\\\\n    or shifted output distribution contribute less than one with a confident\\\\n    one, even when the two rank studies identically -- the mean is sensitive to\\\\n    calibration and AUC is not. Converting each member to ranks first removes\\\\n    that: every member gets one vote per finding, whatever its scale.\\\\n\\\\n    Each member is (N studies, L findings); ranking is per finding, per member,\\\\n    with ties averaged, and the result is scaled to [0, 1] so it can be written\\\\n    into a submission unchanged. Because AUC is invariant to any increasing\\\\n    transform, that scaling costs nothing and keeps the output looking like the\\\\n    probabilities the rest of the pipeline passes around.\\\\n\\\\n    A single member is returned rank-transformed rather than untouched, so that\\\\n    a one-model and a five-model submission are produced by the same code path.\\\\n    """\\\\n    import numpy as np\\\\n\\\\n    if not predictions:\\\\n        raise ValueError("no predictions to average")\\\\n    stacked = np.stack([np.asarray(p, dtype=float) for p in predictions])\\\\n    if stacked.ndim != 3:\\\\n        raise ValueError(\\\\n            f"expected (members, studies, findings), got {stacked.shape}")\\\\n\\\\n    n_studies = stacked.shape[1]\\\\n    if n_studies < 2:\\\\n        # Ranks are meaningless for a single study, and the scaling below\\\\n        # would divide by zero.\\\\n        return stacked.mean(axis=0)\\\\n\\\\n    ranked = np.empty_like(stacked)\\\\n    for member in range(stacked.shape[0]):\\\\n        for finding in range(stacked.shape[2]):\\\\n            column = stacked[member, :, finding]\\\\n            order = np.argsort(column, kind="mergesort")\\\\n            positions = np.empty(n_studies, dtype=float)\\\\n            positions[order] = np.arange(n_studies, dtype=float)\\\\n            # Average the positions of tied values, so a member that predicts\\\\n            # a constant contributes no ordering rather than an arbitrary one.\\\\n            unique, inverse, counts = np.unique(column, return_inverse=True,\\\\n                                                return_counts=True)\\\\n            sums = np.zeros(len(unique))\\\\n            np.add.at(sums, inverse, positions)\\\\n            ranked[member, :, finding] = (sums / counts)[inverse]\\\\n\\\\n    return ranked.mean(axis=0) / (n_studies - 1)\\\\n\\\', \\\'src/model/splits.py\\\': \\\'"""Cross-validation splits and the gold holdout.\\\\n\\\\nTwo rules govern every split here, and both exist because the obvious\\\\nalternative produces a number that looks better and means less.\\\\n\\\\n**Split by study, never by series.** A study\\\\\\\'s ~5.5 series image the same knee\\\\nin the same session. A random series split puts near-duplicates on both sides\\\\nand reports an accuracy the model does not have.\\\\n\\\\n**The 58 gold studies never enter training.** They are the only radiologist\\\\nlabels in the dataset and the only honest measurement available. Everything\\\\nelse is trained on mined labels, and a model evaluated against mined labels can\\\\nscore well by reproducing the extractor\\\\\\\'s mistakes. `assert_no_gold_leakage`\\\\nis called by the training entrypoint rather than trusted to discipline.\\\\n"""\\\\n\\\\nfrom typing import Iterator, List, Optional, Sequence, Tuple\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\nN_FOLDS = 5\\\\nSEED = 0\\\\n\\\\n\\\\ndef load_gold_uids(train_csv=None) -> set:\\\\n    """Study UIDs of the 58 radiologist-labelled studies."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()]\\\\n    return set(labelled[config.ID_COL])\\\\n\\\\n\\\\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\\\\n    """Fail loudly if a gold study appears in training data.\\\\n\\\\n    Called from the training entrypoint. The failure this guards against is\\\\n    silent -- a leaked gold study inflates the one number being used to decide\\\\n    whether the model works.\\\\n    """\\\\n    leaked = set(train_uids) & gold_uids\\\\n    if leaked:\\\\n        raise ValueError(\\\\n            f"{len(leaked)} gold studies leaked into training: "\\\\n            f"{sorted(leaked)[:3]}..."\\\\n        )\\\\n\\\\n\\\\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\\\\n    """A coarse key for balancing folds across a 12-label problem.\\\\n\\\\n    Full multi-label stratification over 2^12 combinations would leave most\\\\n    cells empty at 4,349 studies. The count of positive findings captures the\\\\n    axis that matters -- studies with many findings are rare and should not\\\\n    cluster in one fold.\\\\n    """\\\\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\\\\n    return positives.clip(upper=4)\\\\n\\\\n\\\\ndef scanner_key(index: pd.DataFrame) -> pd.Series:\\\\n    """A per-study scanner identity, from the header scan\\\\\\\'s index.\\\\n\\\\n    The dataset spans 16-19 sites on five continents, and a CNN reads scanner\\\\n    signature as readily as it reads pathology. A fold split that puts the same\\\\n    scanner on both sides therefore reports an accuracy that does not survive\\\\n    a new site: the plan\\\\\\\'s figure is ~0.053 of inflation from random K-fold,\\\\n    with one measured model showing a 0.136 grouped-vs-random gap.\\\\n\\\\n    Site is not given, so manufacturer + model + field strength stands in for\\\\n    it. That is coarser than a site id -- two hospitals with the same scanner\\\\n    collapse into one group, which makes the split *more* conservative, not\\\\n    less -- and it is what the DICOM headers actually carry.\\\\n\\\\n    Takes the frame `src/data/index.py` writes (one row per instance or\\\\n    series, carrying `StudyInstanceUID`), and returns one key per study,\\\\n    indexed by study UID. A study whose headers name none of the three fields\\\\n    gets "unknown", and those all land in one group together.\\\\n    """\\\\n    fields = [c for c in ("Manufacturer", "ManufacturerModelName",\\\\n                          "MagneticFieldStrength") if c in index.columns]\\\\n    if not fields:\\\\n        raise ValueError(\\\\n            "the index carries none of Manufacturer / ManufacturerModelName / "\\\\n            f"MagneticFieldStrength; it has {list(index.columns)[:8]}")\\\\n\\\\n    # fillna after astype: pandas keeps a missing value missing through\\\\n    # astype(str) rather than rendering it "nan", and the join below would\\\\n    # then fail on a float instead of producing a key.\\\\n    joined = index[fields].astype(str).fillna("nan").agg("|".join, axis=1)\\\\n    joined = joined.where(~joined.str.fullmatch(r"(nan\\\\\\\\|?)+"), "unknown")\\\\n    # The first series scanned decides the study. A study imaged on two\\\\n    # scanners is rare and either group is defensible; taking the first keeps\\\\n    # the assignment deterministic.\\\\n    return joined.groupby(index[config.ID_COL]).first()\\\\n\\\\n\\\\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\\\\n               seed: int = SEED,\\\\n               groups: Optional[pd.Series] = None) -> pd.DataFrame:\\\\n    """Assign each study to a fold, stratified by positive-finding count.\\\\n\\\\n    Returns the frame with a `fold` column added.\\\\n\\\\n    `groups` maps study UID -> group id (see `scanner_key`). When given, a\\\\n    whole group lands in one fold: the split becomes leave-scanner-out rather\\\\n    than leave-study-out, so validation measures transfer to unseen equipment\\\\n    instead of rewarding memorised scanner signature.\\\\n\\\\n    Grouping costs balance. Groups are sized very unequally -- one scanner may\\\\n    hold a quarter of the corpus -- so groups are assigned largest-first to\\\\n    whichever fold is currently smallest, which is the standard greedy\\\\n    approximation and the best available without splitting a group. Check\\\\n    `summarise()` afterwards: badly imbalanced folds are the price of grouping,\\\\n    and worth knowing before a run rather than after.\\\\n\\\\n    **Changing this re-cuts every fold**, and every reference number on record\\\\n    (fold 1: val 0.872 / gold 0.861 / LB 0.889) belongs to the ungrouped\\\\n    split. Adopt it when the fold ensemble is rebuilt, not in the middle of an\\\\n    experiment that is being compared against those numbers.\\\\n    """\\\\n    frame = labels.copy().reset_index(drop=True)\\\\n    key = stratification_key(frame)\\\\n    rng = np.random.default_rng(seed)\\\\n\\\\n    frame["fold"] = -1\\\\n    if groups is None:\\\\n        for value in key.unique():\\\\n            rows = np.flatnonzero(key.to_numpy() == value)\\\\n            rng.shuffle(rows)\\\\n            frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\\\\n    else:\\\\n        mapped = frame[config.ID_COL].map(groups).fillna("unknown")\\\\n        sizes = mapped.value_counts()\\\\n        # Shuffle before sorting so equal-sized groups do not always fall the\\\\n        # same way, then place the largest first into the emptiest fold.\\\\n        order = sizes.sample(frac=1.0, random_state=seed).sort_values(\\\\n            ascending=False, kind="mergesort").index\\\\n        counts = np.zeros(n_folds, dtype=int)\\\\n        for group in order:\\\\n            fold = int(np.argmin(counts))\\\\n            rows = np.flatnonzero(mapped.to_numpy() == group)\\\\n            frame.loc[rows, "fold"] = fold\\\\n            counts[fold] += len(rows)\\\\n\\\\n    if (frame["fold"] < 0).any():\\\\n        raise RuntimeError("some studies were not assigned a fold")\\\\n    return frame\\\\n\\\\n\\\\ndef iter_folds(labels: pd.DataFrame,\\\\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\\\\n    """Yield (train, validation) frames for each fold."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    for fold in range(n_folds):\\\\n        validation = labels[labels["fold"] == fold]\\\\n        train = labels[labels["fold"] != fold]\\\\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\\\\n\\\\n\\\\ndef series_for_studies(manifest: pd.DataFrame,\\\\n                       study_uids: Sequence[str]) -> pd.DataFrame:\\\\n    """All cached series belonging to a set of studies."""\\\\n    wanted = set(study_uids)\\\\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\\\\n\\\\n\\\\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\\\\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    rows = []\\\\n    for fold in range(n_folds):\\\\n        subset = labels[labels["fold"] == fold]\\\\n        row = {"fold": fold, "studies": len(subset)}\\\\n        for name in config.LABELS:\\\\n            row[name] = float((subset[name] > 0.5).mean())\\\\n        rows.append(row)\\\\n    return pd.DataFrame(rows).set_index("fold")\\\\n\\\', \\\'src/labels/__init__.py\\\': \\\'\\\', \\\'src/labels/calibrate.py\\\': \\\'"""Score a report extractor against the 58 radiologist-labelled studies.\\\\n\\\\nThose 58 labels are the only trustworthy signal in the dataset, and this is\\\\ntheir highest-value use: measuring how much a mined label can be believed,\\\\nper finding. They are never used for training.\\\\n\\\\nThe per-finding AUC produced here drives the kill criterion -- a finding whose\\\\nmined labels score below KILL_THRESHOLD is noise, and the model should predict\\\\nthe training prior for it rather than learn from corrupted targets.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Below this AUC against gold, a finding\\\\\\\'s mined labels are not worth training\\\\n# on. Chosen to sit meaningfully above chance while tolerating the noise of a\\\\n# 58-sample estimate.\\\\nKILL_THRESHOLD = 0.70\\\\n\\\\n\\\\ndef load_gold(train_csv=None) -> pd.DataFrame:\\\\n    """The 58 studies carrying radiologist labels."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()].copy()\\\\n    for name in config.LABELS:\\\\n        labelled[name] = labelled[name].astype(int)\\\\n    return labelled.reset_index(drop=True)\\\\n\\\\n\\\\ndef load_unlabelled(train_csv=None) -> pd.DataFrame:\\\\n    """The 4,349 studies carrying only a report."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    return frame[frame[config.LABELS[0]].isna()].reset_index(drop=True)\\\\n\\\\n\\\\ndef _auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via rank statistic, so sklearn stays an optional dependency."""\\\\n    positives = y_true == 1\\\\n    n_pos = int(positives.sum())\\\\n    n_neg = int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n\\\\n    # Average ranks within ties, otherwise a constant score would not score 0.5.\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)\\\\n    tie_sums = np.zeros(len(unique))\\\\n    np.add.at(tie_sums, inverse, ranks)\\\\n    ranks = (tie_sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef score_extractor(\\\\n    extract_fn: Callable[[str, str], Dict],\\\\n    gold: Optional[pd.DataFrame] = None,\\\\n) -> pd.DataFrame:\\\\n    """Per-finding reliability of an extractor against the gold studies.\\\\n\\\\n    `extract_fn(report, finding)` must return a dict with `present` (1/0/None)\\\\n    and `severity` (float or None), matching the lexicon module\\\\\\\'s contract.\\\\n    """\\\\n    gold = gold if gold is not None else load_gold()\\\\n    reports = gold["Report"].tolist()\\\\n    records = []\\\\n\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy()\\\\n        results = [extract_fn(report, name) for report in reports]\\\\n\\\\n        present = np.array([\\\\n            r["present"] if r["present"] is not None else -1 for r in results\\\\n        ])\\\\n        severity = np.array([\\\\n            r["severity"] if r["severity"] is not None else 0.0 for r in results\\\\n        ])\\\\n\\\\n        predicted = (present == 1).astype(int)\\\\n        true_pos = int(((predicted == 1) & (truth == 1)).sum())\\\\n        false_pos = int(((predicted == 1) & (truth == 0)).sum())\\\\n        false_neg = int(((predicted == 0) & (truth == 1)).sum())\\\\n\\\\n        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else np.nan\\\\n        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else np.nan\\\\n        auc = _auc(truth, severity)\\\\n\\\\n        # How often a finding goes unmentioned, and how often it is truly\\\\n        # positive when unmentioned. The second number is what fusion should\\\\n        # use as the soft target for silence -- treating it as a hard zero is\\\\n        # what injects systematic noise.\\\\n        unmentioned = present == -1\\\\n        silence_rate = float(unmentioned.mean())\\\\n        prior_if_silent = (\\\\n            float(truth[unmentioned].mean()) if unmentioned.any() else np.nan\\\\n        )\\\\n\\\\n        records.append({\\\\n            "finding": name,\\\\n            "n_positive": int(truth.sum()),\\\\n            "mention_rate": 1.0 - silence_rate,\\\\n            "precision": precision,\\\\n            "recall": recall,\\\\n            "auc": auc,\\\\n            "prior_if_silent": prior_if_silent,\\\\n            "usable": bool(auc >= KILL_THRESHOLD) if np.isfinite(auc) else False,\\\\n        })\\\\n\\\\n    return pd.DataFrame(records).set_index("finding")\\\\n\\\\n\\\\ndef format_report(table: pd.DataFrame) -> str:\\\\n    """Human-readable reliability table."""\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'n+\\\\\\\':>3s} {\\\\\\\'ment\\\\\\\':>5s} {\\\\\\\'prec\\\\\\\':>5s} {\\\\\\\'rec\\\\\\\':>5s} "\\\\n        f"{\\\\\\\'AUC\\\\\\\':>5s} {\\\\\\\'P(1|silent)\\\\\\\':>11s} {\\\\\\\'use\\\\\\\':>4s}",\\\\n        "-" * 66,\\\\n    ]\\\\n    for name, row in table.iterrows():\\\\n        lines.append(\\\\n            f"{name:18s} {int(row[\\\\\\\'n_positive\\\\\\\']):3d} "\\\\n            f"{row[\\\\\\\'mention_rate\\\\\\\']:4.0%} {row[\\\\\\\'precision\\\\\\\']:5.2f} "\\\\n            f"{row[\\\\\\\'recall\\\\\\\']:5.2f} {row[\\\\\\\'auc\\\\\\\']:5.3f} "\\\\n            f"{row[\\\\\\\'prior_if_silent\\\\\\\']:11.2f} {\\\\\\\'yes\\\\\\\' if row[\\\\\\\'usable\\\\\\\'] else \\\\\\\'NO\\\\\\\':>4s}"\\\\n        )\\\\n    lines.append("-" * 66)\\\\n    lines.append(f"macro AUC: {table[\\\\\\\'auc\\\\\\\'].mean():.3f}   "\\\\n                 f"usable findings: {int(table[\\\\\\\'usable\\\\\\\'].sum())}/{len(table)}")\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    from src.labels import lexicon\\\\n\\\\n    table = score_extractor(lexicon.extract_finding)\\\\n    print(format_report(table))\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/labels/lexicon.py\\\': \\\'"""Rule-based extraction of the twelve findings from multilingual reports.\\\\n\\\\nThe corpus spans roughly eleven languages -- English (39%), Greek, Croatian/\\\\nSerbian, Spanish, Turkish, German, Bulgarian/Russian, Dutch, French, Italian,\\\\nPortuguese. Rather than a language x finding cross-product, this module keys on\\\\nthe Greek and Latin stems that clinical vocabulary shares across all of them\\\\n(menisc- / menisk- / μηνίσκ- / мениск-), with language-specific additions only\\\\nwhere the local term is genuinely unrelated (Turkish "kıkırdak" for cartilage).\\\\n\\\\nEach finding resolves to one of three states:\\\\n\\\\n    1     an unnegated mention above the severity floor\\\\n    0     mentioned but negated, or present only below the severity floor\\\\n    None  never mentioned -- which is *not* the same as absent, because\\\\n          reports narrate positives and omit much else\\\\n\\\\nCallers must preserve the None state. Collapsing it to 0 injects systematic\\\\nnoise into exactly the findings that are least often narrated.\\\\n"""\\\\n\\\\nimport re\\\\nimport unicodedata\\\\nfrom typing import Dict, List, Optional\\\\n\\\\nfrom src import config\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cues\\\\n# ---------------------------------------------------------------------------\\\\n# Negation, across every language present in the corpus. Matched as whole words\\\\n# within a bounded window before the finding term.\\\\nNEGATION_CUES = [\\\\n    # English / Latin-script general\\\\n    "no", "not", "without", "absent", "negative", "unremarkable", "intact",\\\\n    "normal", "preserved", "free of", "rule out", "ruled out",\\\\n    # Spanish / Portuguese\\\\n    "sin", "no", "ausencia", "ausente", "conservado", "conservada", "integro",\\\\n    "íntegro", "integra", "íntegra", "normales", "sem", "nao", "não",\\\\n    # Dutch\\\\n    "geen", "niet", "zonder", "intacte", "ongestoord",\\\\n    # German\\\\n    "kein", "keine", "keinen", "nicht", "ohne", "unauffällig", "regelrecht",\\\\n    "unauffaellig",\\\\n    # French\\\\n    "pas", "sans", "absence", "aucun", "aucune", "normale",\\\\n    # Italian\\\\n    "non", "senza", "assenza", "nella norma",\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "değil", "degil", "normaldir", "korunmuş", "korunmus",\\\\n    # Croatian / Serbian / Bosnian\\\\n    "bez", "nema", "nije", "uredan", "uredna", "uredno", "očuvan", "ocuvan",\\\\n    # Bulgarian / Russian\\\\n    "без", "нет", "няма", "не", "отсутств", "интактн", "сохранен",\\\\n    # Greek\\\\n    "χωρίς", "δεν", "ουδεμία", "ουδέν", "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Hedged or sub-threshold mentions. Ground truth was adjudicated against\\\\n# explicit severity thresholds, so "trace effusion" and "large effusion" are\\\\n# not the same label.\\\\nMINIMAL_CUES = [\\\\n    "minimal", "mínimo", "minimo", "minima", "mínima", "trace", "tiny",\\\\n    "slight", "small", "mild", "leve", "ligero", "ligera", "discreto",\\\\n    "discreta", "gering", "geringe", "klein", "kleine", "wenig", "licht",\\\\n    "lichte", "petit", "petite", "faible", "lieve", "piccolo", "az", "küçük",\\\\n    "kucuk", "hafif", "minimalan", "mali", "blag", "малък", "минимален",\\\\n    "незначител", "ελάχιστ", "μικρ", "ήπι",\\\\n]\\\\n\\\\nUNCERTAIN_CUES = [\\\\n    "possible", "possibly", "suspicious", "suspected", "probable", "probably",\\\\n    "questionable", "equivocal", "cannot exclude", "cannot be excluded",\\\\n    "may represent", "posible", "sospecha", "sospechoso", "dudoso",\\\\n    "verdacht", "möglich", "moglich", "fraglich", "mogelijk", "verdenking",\\\\n    "suspect", "possible", "suspicion", "şüpheli", "supheli", "olası", "olasi",\\\\n    "sumnja", "moguć", "mogu", "възможн", "подозр", "πιθαν", "ύποπτ",\\\\n]\\\\n\\\\n# Negation that *follows* the finding term. Turkish is verb-final, so\\\\n# "eklem ici sivi yoktur" negates with a trailing word and a backwards-only\\\\n# search misses it entirely -- that is a tenth of this corpus. English and\\\\n# several others do the same with trailing adjectives ("meniscus intact").\\\\n# Kept deliberately narrower than NEGATION_CUES: a forward search over the full\\\\n# cue list would let an unrelated "normal" in the same clause cancel a real\\\\n# finding.\\\\nPOSTFIX_NEGATION_CUES = [\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "saptanmamistir", "değil", "degil", "normaldir", "normaldi", "korunmuş",\\\\n    "korunmus", "izlenmedi",\\\\n    # English / Latin-script trailing adjectives\\\\n    "intact", "unremarkable", "preserved", "normal", "normale", "normales",\\\\n    "within normal limits", "is normal", "are normal",\\\\n    # Dutch / German\\\\n    "intacte", "unauffällig", "unauffaellig", "regelrecht", "ongestoord",\\\\n    # Croatian / Serbian\\\\n    "uredan", "uredna", "uredno", "očuvan", "ocuvan", "očuvana", "ocuvana",\\\\n    # Bulgarian / Russian\\\\n    "сохранен", "интактн", "не се визуализира", "непроменена",\\\\n    # Greek\\\\n    "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Terms that signal a torn or damaged structure, used by the ligament and\\\\n# meniscus findings where the anatomy word alone means nothing.\\\\nINJURY_STEMS = [\\\\n    "tear", "torn", "rupture", "ruptur", "disrupt", "discontinu", "avuls",\\\\n    "sprain", "lesion", "lesión", "lesao", "lesão", "injur", "insufficien",\\\\n    "rotura", "ruptura", "roto", "rota", "desgarro", "riss", "einriss",\\\\n    "scheur", "letsel", "déchirure", "dechirure", "rupture", "lacerazion",\\\\n    "yırtık", "yirtik", "rüptür", "ruptur", "kopma", "puknuć", "puknuc",\\\\n    "ruptura", "razdor", "разрыв", "руптура", "скъсване", "увреж",\\\\n    "ρήξη", "ρηξη", "ρήγμα",\\\\n]\\\\n\\\\n\\\\ndef _norm(text: str) -> str:\\\\n    """Lowercase and strip combining marks, keeping non-Latin scripts intact."""\\\\n    lowered = text.lower()\\\\n    decomposed = unicodedata.normalize("NFD", lowered)\\\\n    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))\\\\n    return unicodedata.normalize("NFC", stripped)\\\\n\\\\n\\\\ndef _stem_pattern(stems: List[str]) -> re.Pattern:\\\\n    """Compile stems into one alternation matching at a word boundary."""\\\\n    escaped = sorted((re.escape(_norm(s)) for s in stems), key=len, reverse=True)\\\\n    return re.compile(r"(?<![^\\\\\\\\W\\\\\\\\d_])(?:" + "|".join(escaped) + r")", re.UNICODE)\\\\n\\\\n\\\\n# Degenerative disease stems. The three OA findings differ only by which\\\\n# compartment the disease is localised to, so they share this anchor and are\\\\n# separated by the localiser lists below. Anchoring on the compartment word\\\\n# instead does not work: "medial" appears in almost every report describing\\\\n# normal anatomy.\\\\nOA_STEMS = [\\\\n    "osteoarthr", "osteoarthrit", "arthrosis", "arthrose", "artrosis",\\\\n    "artroz", "artroza", "gonarthros", "gonartros", "gonartroz",\\\\n    "degenerative change", "degenerative disease", "cambios degenerativos",\\\\n    "degenerativne", "degenerativ", "chondropath", "condropat", "chondromalac",\\\\n    "condromalac", "chondral loss", "chondral thinning", "cartilage loss",\\\\n    "cartilage thinning", "perdida de cartilago", "adelgazamiento del cartilago",\\\\n    "knorpelverlust", "knorpelschaden", "kraakbeenverlies", "kikirdak kayb",\\\\n    "kikirdak incelme", "hrskavice", "denudacij", "osteofit", "osteophyt",\\\\n    "espolon", "spur formation", "остеоартр", "артроз", "остеофит",\\\\n    "хондромалац", "изтънен", "οστεοαρθρ", "αρθριτ", "χονδροπαθ",\\\\n    "χονδρομαλακ", "οστεοφυτ",\\\\n]\\\\n\\\\nMEDIAL_LOCALIZERS = [\\\\n    "medial", "media", "mediale", "medialen", "medialer", "medialne",\\\\n    "medijaln", "interno", "interna", "innen", "innenseit", "binnen",\\\\n    "ic kompartman", "медиальн", "медиалн", "вътреш", "εσω", "εσωτερικ",\\\\n]\\\\n\\\\nLATERAL_LOCALIZERS = [\\\\n    "lateral", "laterale", "lateralen", "lateraler", "lateraln", "externo",\\\\n    "externa", "aussen", "buiten", "dis kompartman", "латеральн", "латералн",\\\\n    "външ", "εξω", "εξωτερικ",\\\\n]\\\\n\\\\nPF_LOCALIZERS = [\\\\n    "patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",\\\\n    "retropatellar", "retropatelar", "patellar", "patelar", "patella",\\\\n    "rotulian", "rotuliana", "rotuliano", "trochlea", "troclea",\\\\n    "femoropatelarn", "patellofemoraln", "пателлофеморальн", "пателофеморалн",\\\\n    "надколенник", "капачка", "επιγονατιδ", "επιγονατιδομηριαι",\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Finding definitions\\\\n# ---------------------------------------------------------------------------\\\\n# `anatomy` locates the structure; `requires_injury` marks findings where the\\\\n# structure being named is not itself abnormal (a meniscus is always present --\\\\n# only a torn one is a positive label).\\\\nFINDINGS = {\\\\n    "ACL": {\\\\n        "anatomy": ["acl", "lca", "anterior cruciate", "ligamento cruzado anterior",\\\\n                    "voorste kruisband", "vorderes kreuzband", "kreuzband",\\\\n                    "ligament croise anterieur", "legamento crociato anteriore",\\\\n                    "on capraz bag", "onc apraz", "capraz bag", "prednji ukrizeni",\\\\n                    "prednja ukrstena", "predna krstna", "передня крестообразн",\\\\n                    "предна кръстна", "крестообразн", "prosthios chiastos",\\\\n                    "προσθιο χιαστο", "χιαστο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "MCL": {\\\\n        "anatomy": ["mcl", "medial collateral", "ligamento colateral medial",\\\\n                    "ligamento lateral interno", "mediale collaterale",\\\\n                    "innenband", "mediales kollateralband", "mediale band",\\\\n                    "ligament collateral medial", "ic kollateral",\\\\n                    "medial kollateral", "medijalni kolateralni",\\\\n                    "медиальн коллатеральн", "вътрешна колатерал",\\\\n                    "εσω πλαγιο", "εσωτερικο πλαγιο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Medial Meniscus": {\\\\n        "anatomy": ["medial menisc", "menisco medial", "menisco interno",\\\\n                    "mediale meniscus", "innenmeniskus", "medialer meniskus",\\\\n                    "menisque interne", "menisque medial", "menisco mediale",\\\\n                    "medial menisk", "ic menisk", "medijalni meniskus",\\\\n                    "medijalnog meniskusa", "медиальн мениск",\\\\n                    "вътрешния менискус", "εσω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Lateral Meniscus": {\\\\n        "anatomy": ["lateral menisc", "menisco lateral", "menisco externo",\\\\n                    "laterale meniscus", "aussenmeniskus", "lateraler meniskus",\\\\n                    "menisque externe", "menisque lateral", "menisco laterale",\\\\n                    "lateral menisk", "dis menisk", "lateralni meniskus",\\\\n                    "латеральн мениск", "външния менискус", "εξω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    # The OA family anchors on the disease stem and requires a compartment\\\\n    # localiser nearby, within a tighter window than other findings use.\\\\n    "Medial OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": MEDIAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Lateral OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": LATERAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "PF OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": PF_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Effusion": {\\\\n        "anatomy": ["effusion", "efusion", "derrame", "hydrops", "hidrartros",\\\\n                    "gelenkerguss", "erguss", "gewrichtsvocht", "vocht",\\\\n                    "epanchement", "versamento", "efuzyon", "eklem ici sivi",\\\\n                    "eklem sivisi", "izliv", "zglobni izliv", "wypot",\\\\n                    "выпот", "излив", "ставен излив", "συλλογη υγρου",\\\\n                    "αρθριτικη συλλογη", "ενδαρθρικη συλλογη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Synovitis": {\\\\n        "anatomy": ["synovit", "sinovit", "synovial thickening",\\\\n                    "engrosamiento sinovial", "synoviale verdickung",\\\\n                    "synoviale proliferation", "synovitis", "sinovitis",\\\\n                    "sinovijalis", "sinovijalna", "синовит", "υμενιτιδα",\\\\n                    "αρθρικου υμενα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Baker\\\\\\\'s": {\\\\n        "anatomy": ["baker", "popliteal cyst", "quiste poplite", "quistes poplite",\\\\n                    "bakerse cyste", "poplitealzyste", "bakerzyste",\\\\n                    "kyste poplite", "cisti poplitea", "poplitealna cista",\\\\n                    "bakerova cista", "poplitealni", "baker kisti",\\\\n                    "киста бейкера", "подколенна киста", "κυστη baker",\\\\n                    "κυστη μπεικερ", "ιγνυακη κυστη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Contusion": {\\\\n        "anatomy": ["contusion", "contusion osea", "bone contusion",\\\\n                    "bone marrow edema", "bone marrow oedema", "marrow edema",\\\\n                    "edema oseo", "edema de medula osea", "edema medular",\\\\n                    "knochenmarksodem", "knochenodem", "knochenmarködem",\\\\n                    "beenmergoedeem", "botoedeem", "oedeme osseux",\\\\n                    "edema midollare", "kemik ilig odem", "kemik odem",\\\\n                    "kontuzyon", "kostana kontuzija", "edem kosti",\\\\n                    "костномозгов отек", "костномозъчен едем", "оστικο οιδημα",\\\\n                    "οιδημα μυελου", "μυελικο οιδημα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Fracture": {\\\\n        "anatomy": ["fracture", "fractura", "fraktur", "fractuur", "frattura",\\\\n                    "kirik", "kırık", "prelom", "fraktura", "perelom",\\\\n                    "перелом", "фрактура", "счупване", "καταγμα", "καταγματ"],\\\\n        "requires_injury": False,\\\\n    },\\\\n}\\\\n\\\\n# Window, in characters, searched backwards from a term for a negation cue.\\\\nNEGATION_WINDOW = 60\\\\n# Window searched around a term for severity and certainty cues.\\\\nSEVERITY_WINDOW = 45\\\\n# Window searched forward from a term for a trailing negator.\\\\nPOSTFIX_NEGATION_WINDOW = 40\\\\n# Window searched around an anatomy match for the injury or context stem.\\\\nCONTEXT_WINDOW = 90\\\\n\\\\n_NEGATION_RE = _stem_pattern(NEGATION_CUES)\\\\n_POSTFIX_NEGATION_RE = _stem_pattern(POSTFIX_NEGATION_CUES)\\\\n_MINIMAL_RE = _stem_pattern(MINIMAL_CUES)\\\\n_UNCERTAIN_RE = _stem_pattern(UNCERTAIN_CUES)\\\\n_INJURY_RE = _stem_pattern(INJURY_STEMS)\\\\n\\\\n_COMPILED = {}\\\\nfor _name, _spec in FINDINGS.items():\\\\n    _COMPILED[_name] = {\\\\n        "anatomy": _stem_pattern(_spec["anatomy"]),\\\\n        "requires_injury": _spec["requires_injury"],\\\\n        "context": _stem_pattern(_spec["context"]) if _spec.get("context") else None,\\\\n        "context_window": _spec.get("context_window", CONTEXT_WINDOW),\\\\n        "context_is_localizer": _spec.get("context_is_localizer", False),\\\\n    }\\\\n\\\\n# Clause boundaries. Negation does not carry across them, so "no tear. Effusion\\\\n# present" must not read as a negated effusion.\\\\n_CLAUSE_SPLIT = re.compile(r"[.;:\\\\\\\\n]|\\\\\\\\bbut\\\\\\\\b|\\\\\\\\bhowever\\\\\\\\b|\\\\\\\\bpero\\\\\\\\b|\\\\\\\\bmaar\\\\\\\\b|\\\\\\\\baber\\\\\\\\b")\\\\n\\\\n\\\\ndef _clause_bounds(text: str, position: int) -> tuple:\\\\n    """Start and end of the clause containing `position`."""\\\\n    start = 0\\\\n    end = len(text)\\\\n    for match in _CLAUSE_SPLIT.finditer(text):\\\\n        if match.end() <= position:\\\\n            start = match.end()\\\\n        elif match.start() > position:\\\\n            end = match.start()\\\\n            break\\\\n    return start, end\\\\n\\\\n\\\\ndef _is_negated(text: str, position: int, length: int = 0) -> bool:\\\\n    """True when a negation cue applies to the term at `position`.\\\\n\\\\n    Searches backwards over the full cue list, and forwards over the narrower\\\\n    postfix list -- verb-final languages put the negator after the finding.\\\\n    Both searches stop at the clause boundary so a negation cannot leak into a\\\\n    neighbouring statement.\\\\n    """\\\\n    clause_start, clause_end = _clause_bounds(text, position)\\\\n\\\\n    window_start = max(clause_start, position - NEGATION_WINDOW)\\\\n    if _NEGATION_RE.search(text, window_start, position):\\\\n        return True\\\\n\\\\n    term_end = position + length\\\\n    window_end = min(clause_end, term_end + POSTFIX_NEGATION_WINDOW)\\\\n    return bool(_POSTFIX_NEGATION_RE.search(text, term_end, window_end))\\\\n\\\\n\\\\ndef _severity(text: str, position: int, length: int) -> float:\\\\n    """Confidence that the mention clears the adjudication threshold.\\\\n\\\\n    1.0 for an unqualified positive, lower when the report hedges the finding\\\\n    or calls it minimal. These scores feed the fusion step as soft targets\\\\n    rather than being thresholded here.\\\\n    """\\\\n    start = max(0, position - SEVERITY_WINDOW)\\\\n    end = min(len(text), position + length + SEVERITY_WINDOW)\\\\n    window = text[start:end]\\\\n\\\\n    score = 1.0\\\\n    if _MINIMAL_RE.search(window):\\\\n        score *= 0.35\\\\n    if _UNCERTAIN_RE.search(window):\\\\n        score *= 0.5\\\\n    return score\\\\n\\\\n\\\\ndef _has_context(text: str, position: int, length: int, pattern,\\\\n                 window: int = CONTEXT_WINDOW, clause_scoped: bool = True) -> bool:\\\\n    """True when a required injury, disease, or localiser stem sits nearby.\\\\n\\\\n    Injury context is clause-scoped: a tear described in one sentence must not\\\\n    license a finding named in the next. Compartment localisers are not, because\\\\n    reports routinely name the disease in one clause and the compartment in the\\\\n    following one ("Gonarthrosis. Most marked laterally."). Enforcing clause\\\\n    scope on localisers cost Lateral OA two thirds of its recall.\\\\n    """\\\\n    if pattern is None:\\\\n        return True\\\\n\\\\n    if clause_scoped:\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n    else:\\\\n        clause_start, clause_end = 0, len(text)\\\\n\\\\n    start = max(clause_start, position - window)\\\\n    end = min(clause_end, position + length + window)\\\\n    return bool(pattern.search(text, start, end))\\\\n\\\\n\\\\ndef extract_finding(report: str, finding: str) -> Dict:\\\\n    """Resolve one finding from one report."""\\\\n    spec = _COMPILED[finding]\\\\n    text = _norm(report)\\\\n\\\\n    context_pattern = spec["context"]\\\\n    context_window = spec["context_window"]\\\\n    localizer_based = spec["context_is_localizer"]\\\\n    if spec["requires_injury"]:\\\\n        context_pattern = _INJURY_RE\\\\n        context_window = CONTEXT_WINDOW\\\\n        localizer_based = False\\\\n\\\\n    mentioned = False\\\\n    best = None\\\\n\\\\n    for match in spec["anatomy"].finditer(text):\\\\n        position, length = match.start(), match.end() - match.start()\\\\n        in_context = _has_context(text, position, length, context_pattern,\\\\n                                  context_window,\\\\n                                  clause_scoped=not localizer_based)\\\\n\\\\n        if localizer_based and not in_context:\\\\n            # Degenerative disease was described, but not in this compartment.\\\\n            # That is silence about this finding, not evidence against it --\\\\n            # counting it as a mention would report a confident zero for a\\\\n            # compartment the radiologist never discussed.\\\\n            continue\\\\n\\\\n        mentioned = True\\\\n\\\\n        if not in_context:\\\\n            # The structure is named but nothing abnormal is asserted about it,\\\\n            # which is a normal-anatomy sentence rather than a finding.\\\\n            continue\\\\n        if _is_negated(text, position, length):\\\\n            continue\\\\n\\\\n        score = _severity(text, position, length)\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n        candidate = {\\\\n            "present": 1,\\\\n            "severity": score,\\\\n            "span": report[clause_start:clause_end].strip()[:200],\\\\n        }\\\\n        # Keep the most confident positive mention.\\\\n        if best is None or score > best["severity"]:\\\\n            best = candidate\\\\n\\\\n    if best is not None:\\\\n        return best\\\\n    if mentioned:\\\\n        return {"present": 0, "severity": 0.0, "span": None}\\\\n    return {"present": None, "severity": None, "span": None}\\\\n\\\\n\\\\ndef extract_report(report: str) -> Dict[str, Dict]:\\\\n    """Resolve all twelve findings from one report."""\\\\n    if not isinstance(report, str) or not report.strip():\\\\n        return {name: {"present": None, "severity": None, "span": None}\\\\n                for name in config.LABELS}\\\\n    return {name: extract_finding(report, name) for name in config.LABELS}\\\\n\\\\n\\\\ndef extract_frame(reports, ids=None):\\\\n    """Vectorise extraction over a series of reports into a DataFrame.\\\\n\\\\n    Returns one row per report with `<finding>` (0/1/NaN) and\\\\n    `<finding>_severity` columns.\\\\n    """\\\\n    import pandas as pd\\\\n\\\\n    records = []\\\\n    for report in reports:\\\\n        result = extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = result[name]["present"]\\\\n            row[f"{name}_severity"] = result[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    if ids is not None:\\\\n        frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\', \\\'src/labels/fuse.py\\\': \\\'"""Combine the rule-based and LLM extractors into soft training targets.\\\\n\\\\nProduces one row per study with, for each finding, a target in [0, 1] and a\\\\nsample weight. Neither is a hard label: the extractors disagree often enough,\\\\nand are wrong often enough, that training on thresholded output would teach the\\\\nmodel to reproduce extraction errors.\\\\n\\\\n**Nothing here is fitted on the 58 gold studies.** An earlier version derived\\\\nper-finding blend weights from gold AUCs and per-finding silence priors from\\\\ngold positive rates -- 24 numbers estimated from 58 samples. It scored macro AUC\\\\n0.860 in-sample and 0.812 under leave-one-out, which is worse than the LLM\\\\nextractor alone at 0.838. The apparent gain was the fit, not the fusion.\\\\n\\\\nWhat replaced it is deliberately parameter-free:\\\\n\\\\n**A fixed blend.** Targets are 0.10 x lexicon + 0.90 x LLM for every finding.\\\\nThe ratio sits on a plateau, not an argmax: with the dual-mode LLM scores,\\\\n0.00 through 0.15 all land within 0.002 on gold (0.889-0.891) and the weight\\\\nfalls away steadily above that -- the 0.30 chosen for the categorical-only\\\\nLLM cost ~0.010 against the stronger extractor. A small nonzero weight keeps\\\\nthe lexicon\\\\\\\'s independent evidence without letting it drag the ranking.\\\\n\\\\n**Silence scores zero, and is handled by the weight instead.** Measuring\\\\nP(positive | both silent) on gold and using it as the target was worse than\\\\nusing zero, on every variant tested. The concern that motivated it -- that a\\\\nsilent finding is not a confident negative -- is real, and is addressed by\\\\ngiving those studies a low sample weight rather than a fabricated target.\\\\n\\\\n**Disagreement lowers the weight rather than picking a winner.** A study where\\\\nthe extractors conflict is genuinely ambiguous and should not pull as hard as\\\\none where they concur.\\\\n"""\\\\n\\\\nfrom pathlib import Path\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.labels import calibrate, lexicon\\\\n\\\\n# Blend ratio. Chosen from a plateau, not an argmax -- see module docstring.\\\\nLEXICON_WEIGHT = 0.10\\\\nLLM_WEIGHT = 1.0 - LEXICON_WEIGHT\\\\n\\\\n# Sample weights by how much the two extractors corroborate each other.\\\\nWEIGHT_AGREE = 1.0        # both informative, same call\\\\nWEIGHT_DISAGREE = 0.4     # both informative, opposite calls\\\\nWEIGHT_SINGLE = 0.7       # one informative, one silent\\\\nWEIGHT_SILENT = 0.25      # neither informative; target is zero but barely counts\\\\n\\\\n\\\\ndef silence_rates(labels: pd.DataFrame) -> pd.Series:\\\\n    """Per finding, the share of studies neither extractor said anything about.\\\\n\\\\n    Measured on the mined corpus -- 4,349 studies -- and never on gold. That\\\\n    distinction is what separates this from the per-finding silence priors this\\\\n    module removed: those were 24 numbers fitted on 58 samples and scored 0.860\\\\n    in-sample against 0.812 leave-one-out. A rate over 4,349 studies is an\\\\n    observation, not a fit.\\\\n    """\\\\n    return pd.Series(\\\\n        {name: float((labels[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n         for name in config.LABELS})\\\\n\\\\n\\\\ndef rescale_silent_weights(labels: pd.DataFrame) -> pd.DataFrame:\\\\n    """Down-weight silence in proportion to how uninformative it is.\\\\n\\\\n    **The problem, measured on Synovitis.** 82.2% of studies are silent about\\\\n    it, the mined target averages 0.087, and the true prevalence on gold is\\\\n    46.6%. So roughly 1,666 of the 3,576 silent studies are true positives\\\\n    being trained toward zero -- and because silence is so common here, those\\\\n    studies carry 57% of the finding\\\\\\\'s gradient. The model is being taught to\\\\n    rank real positives low. Synovitis duly scores 0.682 on gold, the only\\\\n    finding where the fine-tuned model is WORSE than the retired frozen one\\\\n    (0.746), which is what fitting label noise harder looks like.\\\\n\\\\n    **The fix, and why it is not the one that failed.** Targets are untouched:\\\\n    fabricating P(positive | silent) as a target measured worse on every\\\\n    variant tested, and that stands. What changes is the *weight*, which this\\\\n    module\\\\\\\'s own docstring already names as the mechanism for silence being an\\\\n    unconfident negative. The weight becomes\\\\n\\\\n        WEIGHT_SILENT x (1 - silence_rate(finding))\\\\n\\\\n    which is one rule with no free parameters, evaluated on the mined corpus.\\\\n    Silence about ACL (9.2% of studies) stays worth nearly the full 0.25,\\\\n    because a report that discusses ligaments and omits the ACL is saying\\\\n    something. Silence about Synovitis (82.2%) falls to 0.044, because a\\\\n    corpus that mentions it in one study out of six is not saying anything.\\\\n\\\\n    Returns a copy; the target columns are passed through unchanged.\\\\n    """\\\\n    rates = silence_rates(labels)\\\\n    out = labels.copy()\\\\n    for name in config.LABELS:\\\\n        column = f"{name}_weight"\\\\n        silent = out[column] == WEIGHT_SILENT\\\\n        out.loc[silent, column] = WEIGHT_SILENT * (1.0 - rates[name])\\\\n    return out\\\\n\\\\n\\\\ndef _informative(present) -> bool:\\\\n    """False when an extractor said nothing about this finding."""\\\\n    if present is None:\\\\n        return False\\\\n    if isinstance(present, float) and np.isnan(present):\\\\n        return False\\\\n    return True\\\\n\\\\n\\\\ndef _score(present, severity) -> float:\\\\n    """One extractor\\\\\\\'s contribution to the target, in [0, 1]."""\\\\n    if not _informative(present):\\\\n        return 0.0\\\\n    if int(present) == 1:\\\\n        return float(severity) if severity == severity and severity is not None else 1.0\\\\n    return 0.0\\\\n\\\\n\\\\ndef lexicon_frame(reports, ids) -> pd.DataFrame:\\\\n    """Run the rule-based extractor over a set of reports."""\\\\n    records = []\\\\n    for report in reports:\\\\n        extracted = lexicon.extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = extracted[name]["present"]\\\\n            row[f"{name}_severity"] = extracted[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\\n\\\\ndef fuse(lex: pd.DataFrame, llm: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend two extractor frames into soft targets and sample weights."""\\\\n    merged = lex.merge(llm, on=config.ID_COL, suffixes=("_lex", "_llm"))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        llm_present = merged[f"{name}_llm"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity_lex"].to_numpy()\\\\n        llm_sev = merged[f"{name}_severity_llm"].to_numpy()\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            llm_score = _score(llm_present[i], llm_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            llm_ok = _informative(llm_present[i])\\\\n            if lex_ok and llm_ok:\\\\n                agree = (lex_score > 0) == (llm_score > 0)\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\ndef fuse_scores(lex: pd.DataFrame, llm_scores: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend the lexicon frame with an already-combined LLM score frame.\\\\n\\\\n    `llm_scores` is the output of `mode_fusion.combine` (or\\\\n    `mode_fusion.from_categorical`): per finding a score in [0, 1], an\\\\n    `_informative` flag, and a `_call` boolean. The weight semantics mirror\\\\n    `fuse`: agreement compares the lexicon\\\\\\\'s call with the LLM\\\\\\\'s call.\\\\n    """\\\\n    merged = lex.merge(llm_scores, on=config.ID_COL, suffixes=("_lex", ""))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity"].to_numpy()\\\\n        llm_score = merged[name].to_numpy(dtype=float)\\\\n        llm_ok = merged[f"{name}_informative"].to_numpy(dtype=bool)\\\\n        llm_call = merged[f"{name}_call"].to_numpy(dtype=bool)\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score[i]\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            if lex_ok and llm_ok[i]:\\\\n                agree = (lex_score > 0) == llm_call[i]\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok[i]:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\n# Order-keeper inside the reference distribution\\\\\\\'s tie blocks. Small enough\\\\n# that the restored shape dominates the target value; the epsilon term only\\\\n# arranges studies within a block.\\\\nSHARPEN_EPSILON = 0.02\\\\n\\\\n\\\\ndef sharpen_targets(fused: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:\\\\n    """Restore `reference`\\\\\\\'s target distribution, keeping `fused`\\\\\\\'s ordering.\\\\n\\\\n    The dual-mode targets rank studies better than the categorical fusion\\\\n    (0.891 vs 0.854 on gold) but drained the confident mass out of the\\\\n    distribution -- extreme targets fell from 73% to 60% of slots and the\\\\n    model trained on them regressed on the leaderboard (0.845 -> 0.827).\\\\n    AUC is invariant under strictly monotone maps, so the ordering gain and\\\\n    the crisp shape are separable: each finding\\\\\\\'s targets are quantile-mapped\\\\n    onto the reference marginal, plus an epsilon-scaled rank term that keeps\\\\n    the map strictly increasing where the reference has ties. Sample weights\\\\n    are not touched.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    out = fused.copy()\\\\n    for name in config.LABELS:\\\\n        targets = fused[name].to_numpy(dtype=float)\\\\n        ref = np.sort(reference[name].to_numpy(dtype=float))\\\\n        ranks = mode_fusion._tied_ranks_01(targets)\\\\n        mapped = np.quantile(ref, np.clip(ranks, 0.0, 1.0))\\\\n        out[name] = np.clip(\\\\n            (mapped + SHARPEN_EPSILON * ranks) / (1.0 + SHARPEN_EPSILON),\\\\n            0.0, 1.0)\\\\n    return out\\\\n\\\\n\\\\ndef extractor_aucs(gold: pd.DataFrame, lex_gold: pd.DataFrame,\\\\n                   llm_gold: pd.DataFrame) -> pd.DataFrame:\\\\n    """Diagnostic only: how each extractor scores on gold.\\\\n\\\\n    These numbers are reported, never fed back into the targets. Letting them\\\\n    set blend weights is exactly the overfit this module was rewritten to avoid.\\\\n    """\\\\n    rows = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        lex_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(lex_gold[name], lex_gold[f"{name}_severity"])])\\\\n        llm_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(llm_gold[name], llm_gold[f"{name}_severity"])])\\\\n        rows.append({\\\\n            "finding": name,\\\\n            "lexicon_auc": calibrate._auc(truth, lex_scores),\\\\n            "llm_auc": calibrate._auc(truth, llm_scores),\\\\n        })\\\\n    return pd.DataFrame(rows).set_index("finding")\\\\n\\\\n\\\\ndef build(train_csv=None, llm_mined=None, llm_gold=None,\\\\n          probs_mined=None, probs_gold=None,\\\\n          out_path: Optional[str] = None):\\\\n    """Full pipeline: extract, fuse, and score against gold.\\\\n\\\\n    When the probability-mode parquets exist alongside the categorical ones,\\\\n    the two modes are combined by `mode_fusion` before the lexicon blend.\\\\n    Otherwise this is the original categorical-only fusion, so the pipeline\\\\n    still runs on a machine that never downloaded the probability dumps.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    gold = calibrate.load_gold(train_csv)\\\\n    unlabelled = calibrate.load_unlabelled(train_csv)\\\\n\\\\n    mined_frame = pd.read_parquet(\\\\n        llm_mined or (config.META_DIR / "llm_labels.parquet"))\\\\n    gold_frame = pd.read_parquet(\\\\n        llm_gold or (config.META_DIR / "llm_labels_gold.parquet"))\\\\n\\\\n    # Align to CSV order so positional comparisons stay valid.\\\\n    gold_frame = (gold_frame.set_index(config.ID_COL)\\\\n                  .loc[gold[config.ID_COL]].reset_index())\\\\n    mined_frame = (mined_frame.set_index(config.ID_COL)\\\\n                   .loc[unlabelled[config.ID_COL]].reset_index())\\\\n\\\\n    probs_mined_path = probs_mined or (config.META_DIR / "llm_probs.parquet")\\\\n    probs_gold_path = probs_gold or (config.META_DIR / "llm_probs_gold.parquet")\\\\n    use_probs = (Path(probs_mined_path).exists()\\\\n                 and Path(probs_gold_path).exists())\\\\n\\\\n    if use_probs:\\\\n        probs_mined_frame = pd.read_parquet(probs_mined_path)\\\\n        probs_gold_frame = pd.read_parquet(probs_gold_path)\\\\n        scores_mined = mode_fusion.combine(mined_frame, probs_mined_frame)\\\\n        scores_gold = mode_fusion.combine(gold_frame, probs_gold_frame)\\\\n    else:\\\\n        scores_mined = mode_fusion.from_categorical(mined_frame)\\\\n        scores_gold = mode_fusion.from_categorical(gold_frame)\\\\n\\\\n    lex_gold = lexicon_frame(gold["Report"], gold[config.ID_COL])\\\\n    lex_mined = lexicon_frame(unlabelled["Report"], unlabelled[config.ID_COL])\\\\n\\\\n    fused = fuse_scores(lex_mined, scores_mined)\\\\n    fused_gold = fuse_scores(lex_gold, scores_gold)\\\\n\\\\n    if use_probs:\\\\n        # The categorical-only fusion of the same inputs supplies the shape\\\\n        # the dual-mode targets are sharpened back onto.\\\\n        ref_mined = fuse_scores(\\\\n            lex_mined, mode_fusion.from_categorical(mined_frame))\\\\n        ref_gold = fuse_scores(\\\\n            lex_gold, mode_fusion.from_categorical(gold_frame))\\\\n        fused = sharpen_targets(fused, ref_mined)\\\\n        fused_gold = sharpen_targets(fused_gold, ref_gold)\\\\n\\\\n    aucs = extractor_aucs(gold, lex_gold, gold_frame)\\\\n\\\\n    if out_path:\\\\n        fused.to_parquet(out_path, index=False)\\\\n\\\\n    return fused, fused_gold, aucs\\\\n\\\\n\\\\ndef report(gold: pd.DataFrame, fused_gold: pd.DataFrame,\\\\n           aucs: pd.DataFrame, fused: pd.DataFrame) -> str:\\\\n    """Per-finding summary of the fused targets.\\\\n\\\\n    The fused AUC column is an honest estimate: no parameter in this module was\\\\n    estimated from the gold studies, so in-sample and held-out are the same.\\\\n    """\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'lexAUC\\\\\\\':>7s} {\\\\\\\'llmAUC\\\\\\\':>7s} {\\\\\\\'fusedAUC\\\\\\\':>9s} "\\\\n        f"{\\\\\\\'delta\\\\\\\':>7s} {\\\\\\\'mean w\\\\\\\':>7s} {\\\\\\\'silent\\\\\\\':>7s}",\\\\n        "-" * 72,\\\\n    ]\\\\n    fused_aucs = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        auc = calibrate._auc(truth, fused_gold[name].to_numpy())\\\\n        fused_aucs.append(auc)\\\\n        best_single = max(aucs.loc[name, "lexicon_auc"], aucs.loc[name, "llm_auc"])\\\\n        silent = float((fused[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n        lines.append(\\\\n            f"{name:18s} {aucs.loc[name, \\\\\\\'lexicon_auc\\\\\\\']:7.3f} "\\\\n            f"{aucs.loc[name, \\\\\\\'llm_auc\\\\\\\']:7.3f} {auc:9.3f} "\\\\n            f"{auc - best_single:+7.3f} "\\\\n            f"{fused[f\\\\\\\'{name}_weight\\\\\\\'].mean():7.2f} {silent:6.0%}"\\\\n        )\\\\n    lines.append("-" * 72)\\\\n    lines.append(\\\\n        f"macro: fused {np.nanmean(fused_aucs):.3f}   "\\\\n        f"lexicon {aucs[\\\\\\\'lexicon_auc\\\\\\\'].mean():.3f}   "\\\\n        f"LLM {aucs[\\\\\\\'llm_auc\\\\\\\'].mean():.3f}"\\\\n    )\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    out = config.META_DIR / "train_labels.parquet"\\\\n    fused, fused_gold, aucs = build(out_path=out)\\\\n    gold = calibrate.load_gold()\\\\n    print(report(gold, fused_gold, aucs, fused))\\\\n    print(f"\\\\\\\\nwrote {out}  {fused.shape}")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/lightning.py\\\': \\\'"""Lightning wrapper for the end-to-end fine-tuning run.\\\\n\\\\nKaggle offers a T4 x2 accelerator, and the training loop is GPU-bound at\\\\n~220 images/second, so a second device is close to a free halving of\\\\nwall-clock. Lightning is here for the distributed plumbing, not for the\\\\nabstraction: the semantics below are a faithful port of the hand-written loop\\\\nin notebook 08, and any difference in the resulting score is a bug rather\\\\nthan a design choice.\\\\n\\\\n**Everything in this module lives here rather than in a notebook cell.**\\\\nLightning\\\\\\\'s `ddp_notebook` strategy forks, so children inherit memory and the\\\\nmodule is not pickled on that path -- but dataloader workers and the spawn\\\\nstart method both do pickle, and a class defined in a notebook cell belongs to\\\\n`__main__` and does not survive that. Keeping these definitions importable as\\\\n`src.model.lightning` makes the module correct under either start method; the\\\\nbootstrap writes the files to disk before importing them. The same rule put\\\\nthe cache\\\\\\\'s process-pool worker in `src/data/cache.py`.\\\\n\\\\nA related consequence of fork, and the reason the notebook is careful about\\\\nit: **CUDA must not be initialised before `Trainer.fit`**. A fork inheriting a\\\\nlive CUDA context fails in the child. `torch.cuda.get_device_name` and\\\\n`get_device_properties` both call `_lazy_init`; `device_count` and\\\\n`is_available` do not.\\\\n\\\\nThree DDP hazards are handled explicitly, because each one runs clean and\\\\nsilently scores worse:\\\\n\\\\n**Memory-mapped shards are never serialised.** The cache is ~29 GB across\\\\nthree memmaps. Holding them as attributes would carry the array bytes into any\\\\nprocess the dataset is pickled into. They are opened lazily per process and\\\\ndropped from `__getstate__`, which also means each dataloader worker maps the\\\\nfiles itself rather than sharing a handle across a fork.\\\\n\\\\n**AUC is computed over gathered predictions.** It is a ranking metric, so\\\\naveraging two per-rank AUCs is not the AUC of the whole set. Predictions and\\\\ntargets are all-gathered before scoring. The distributed sampler pads the\\\\nfinal batch by repeating samples, so at most one study per rank is counted\\\\ntwice out of ~870 -- below the resolution of the measurement.\\\\n\\\\n**The effective batch is held constant.** Each rank receives `batch_size`\\\\nstudies, so the effective batch is `batch_size x devices x accumulation`.\\\\nHalving accumulation when moving from one device to two keeps the optimiser\\\\nseeing the same eight studies per step, and keeps the OneCycleLR step count\\\\nunchanged.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport torch\\\\nfrom torch.utils.data import Dataset\\\\n\\\\nfrom src import config\\\\nfrom src.model import augment as augment_mod\\\\nfrom src.model import averaging\\\\nfrom src.model import conditional\\\\nfrom src.model import dataset as knee_dataset\\\\nfrom src.model import net\\\\n\\\\ntry:  # Lightning 2.x ships under both names depending on the install.\\\\n    import lightning.pytorch as pl\\\\nexcept ImportError:  # pragma: no cover - exercised only on Kaggle/Colab\\\\n    import pytorch_lightning as pl\\\\n\\\\n\\\\ndef auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via the rank statistic, with ties averaged.\\\\n\\\\n    Duplicated from `labels.calibrate` rather than imported: only the model\\\\n    package is materialised into the training notebook, and a metric that\\\\n    disagrees with the one the labels were scored by would be worse than the\\\\n    duplication.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef macro_auc(truth: np.ndarray, probs: np.ndarray) -> float:\\\\n    """Mean per-finding AUC, ignoring findings with no positives present."""\\\\n    scores = [auc(truth[:, i], probs[:, i]) for i in range(truth.shape[1])]\\\\n    return float(np.nanmean(scores))\\\\n\\\\n\\\\nclass StudyVolumeDataset(Dataset):\\\\n    """Study volumes with an even slice subsample and intensity jitter.\\\\n\\\\n    Slices are subsampled evenly rather than at random so a validation score\\\\n    is reproducible. Horizontal flips are deliberately not used: after\\\\n    canonicalisation a flip re-mirrors the knee and swaps medial with lateral,\\\\n    silently corrupting four of the twelve labels.\\\\n\\\\n    **`resample` is the training-only exception, and it exists because the\\\\n    cache is larger than what training reads.** The cache holds 24 slices of\\\\n    each of 24,371 series -- 584,904 slice-images. A run at 16 slices x 5\\\\n    series reads 341,728 of them, the same ones every epoch: the fixed\\\\n    `keep` discards 8 slices of every series and the deterministic series cap\\\\n    discards 3,013 series outright. The remaining 42% is not new data, but it\\\\n    is unseen pixels of the same 4,349 studies, and reaching it costs no\\\\n    storage, no upload and no extra forward passes. `resample=False`\\\\n    reproduces every measured result exactly; the flag is what makes the two\\\\n    comparable.\\\\n    """\\\\n\\\\n    def __init__(self, study_index, studies: Sequence[str],\\\\n                 cache_paths: Sequence[str], slices_per_series: int,\\\\n                 augment=False, resample: bool = False):\\\\n        self.index = study_index\\\\n        self.studies = list(studies)\\\\n        self.cache_paths = list(cache_paths)\\\\n        self.slices_per_series = slices_per_series\\\\n        # Per-epoch resampling of *which* slices and series are read. Training\\\\n        # only -- a validation score has to be reproducible to be a gate.\\\\n        self.resample = resample\\\\n        # `augment` is either a bool -- the legacy single intensity scale that\\\\n        # every measured result was trained under -- or an AugmentConfig. The\\\\n        # bool path is kept byte-for-byte rather than expressed as a config so\\\\n        # that reproducing an old run does not depend on the two being\\\\n        # equivalent. See src/model/augment.py.\\\\n        self.augment = augment\\\\n        self.keep = np.linspace(0, config.CACHE_SLICES - 1,\\\\n                                slices_per_series).round().astype(int)\\\\n        self._shards: Optional[Dict[str, np.ndarray]] = None\\\\n\\\\n    @property\\\\n    def shards(self) -> Dict[str, np.ndarray]:\\\\n        """Open the memmaps on first use, once per process."""\\\\n        if self._shards is None:\\\\n            self._shards = {path: np.load(path, mmap_mode="r")\\\\n                            for path in self.cache_paths}\\\\n        return self._shards\\\\n\\\\n    def __getstate__(self):\\\\n        # Never serialise the memmaps into a spawned process or a dataloader\\\\n        # worker; each one reopens them from the paths instead.\\\\n        state = self.__dict__.copy()\\\\n        state["_shards"] = None\\\\n        return state\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def _resampled_keep(self, available: int, rng) -> np.ndarray:\\\\n        """A fresh even-ish slice subsample: drop one slice per even group.\\\\n\\\\n        Phrased as a drop rather than a pick so the result is exactly\\\\n        `slices_per_series` distinct indices in ascending order. Order matters:\\\\n        `stack_2p5d` hands the backbone each slice with its two neighbours, and\\\\n        a shuffled stack makes those neighbours meaningless -- the same reason\\\\n        `augment.py` implements `slice_reverse` and not slice permutation.\\\\n        Coverage matches `np.linspace` in expectation, so this is a resample of\\\\n        the reference geometry rather than a different one.\\\\n        """\\\\n        n_drop = available - self.slices_per_series\\\\n        groups = np.array_split(np.arange(available), n_drop)\\\\n        drop = {int(rng.choice(group)) for group in groups}\\\\n        return np.array([i for i in range(available) if i not in drop])\\\\n\\\\n    def __getitem__(self, i: int):\\\\n        rng = np.random.default_rng(np.random.randint(0, 2**31))\\\\n        item = knee_dataset.load_study(\\\\n            self.index, self.studies[i], self.shards,\\\\n            series_rng=rng if self.resample else None)\\\\n\\\\n        available = item["volumes"].shape[1]\\\\n        if available > self.slices_per_series:\\\\n            # A thinned cache is already at the target count; only subsample a\\\\n            # full one, or the second cut would drop slices twice.\\\\n            keep = (self._resampled_keep(available, rng) if self.resample\\\\n                    else np.linspace(0, available - 1,\\\\n                                     self.slices_per_series).round().astype(int))\\\\n            item["volumes"] = item["volumes"][:, keep]\\\\n        elif available < self.slices_per_series:\\\\n            # The attention pool masks over the slice axis, so a short volume\\\\n            # trains happily on fewer slices than configured and says nothing.\\\\n            raise ValueError(\\\\n                f"cache supplies {available} slices, slices_per_series is "\\\\n                f"{self.slices_per_series} -- attach the full 24-slice "\\\\n                "shards, or lower the setting")\\\\n\\\\n        if isinstance(self.augment, augment_mod.AugmentConfig):\\\\n            # The same generator that chose the slices and series, so one item\\\\n            # draws from one stream. Its seed comes from the worker\\\\\\\'s numpy\\\\n            # state, which advances rather than resetting between epochs --\\\\n            # that is what makes the resample per-epoch and not per-study.\\\\n            item = augment_mod.augment_study(item, rng, self.augment)\\\\n        elif self.augment:\\\\n            # Legacy path: one intensity scale for the whole study. Every\\\\n            # result on record was trained through exactly this.\\\\n            scale = np.random.uniform(0.9, 1.1)\\\\n            item["volumes"] = np.clip(\\\\n                item["volumes"].astype(np.float32) * scale,\\\\n                0, 255).astype(np.uint8)\\\\n        return item\\\\n\\\\n\\\\nclass KneeLitModule(pl.LightningModule):\\\\n    """The fine-tuning step, loss, schedule, and epoch-level scoring.\\\\n\\\\n    Two validation dataloaders are expected, and which is which matters:\\\\n    index 0 is the validation fold scored against mined labels binarised at\\\\n    0.5, and it is the only thing allowed to select a checkpoint. Index 1 is\\\\n    the gold 58, logged every epoch because it is nearly free and shows\\\\n    whether the run is converging -- and never monitored. Selecting on gold\\\\n    turns the project\\\\\\\'s one honest measuring stick into a model-selection set;\\\\n    an earlier version of this loop did exactly that and inflated its own\\\\n    reported figure by an estimated 0.01-0.03.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: torch.nn.Module, feature_dim: int,\\\\n                 lr: float = 1e-4, weight_decay: float = 1e-4,\\\\n                 total_steps: int = 1000, max_lr_multiplier: float = 3.0,\\\\n                 report_dim: Optional[int] = None,\\\\n                 lambda_report: float = 0.0, lambda_rank: float = 0.0,\\\\n                 rank_queue_size: int = 256, rank_min_gap: float = 0.25,\\\\n                 aux_mode: str = "fixed", aux_gate: str = "binary",\\\\n                 backbone_lr_scale: float = 1.0,\\\\n                 input_mean=None, input_std=None, gated_pool: bool = False,\\\\n                 schedule: str = "onecycle",\\\\n                 warmup_steps: Optional[int] = None,\\\\n                 label_smoothing: float = 0.0, symmetric_beta: float = 0.0,\\\\n                 pos_weight: Optional[Sequence[float]] = None,\\\\n                 lambda_stratified_rank: float = 0.0,\\\\n                 stratified_findings: Sequence[str] = ("Synovitis",),\\\\n                 stratify_by: str = "Effusion",\\\\n                 stratified_caliper: float = 0.15,\\\\n                 lambda_coupling: float = 0.0,\\\\n                 coupling_target: Optional[float] = None,\\\\n                 residual_beta: Optional[float] = None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        if classwise_pool and (lambda_report > 0 or aux_mode != "fixed"):\\\\n            raise ValueError("classwise pooling requires fixed mode without report loss")\\\\n        if aux_mode not in ("fixed", "cosine"):\\\\n            raise ValueError(f"aux_mode must be fixed|cosine, got {aux_mode!r}")\\\\n        if aux_gate not in ("binary", "weighted"):\\\\n            raise ValueError(f"aux_gate must be binary|weighted, got {aux_gate!r}")\\\\n        if schedule not in ("onecycle", "cosine"):\\\\n            raise ValueError(\\\\n                f"schedule must be onecycle|cosine, got {schedule!r}")\\\\n        if warmup_steps is not None and not 0 <= warmup_steps < total_steps:\\\\n            raise ValueError(\\\\n                f"warmup_steps {warmup_steps} must be under total_steps "\\\\n                f"{total_steps}")\\\\n        if lambda_coupling > 0 and coupling_target is None:\\\\n            raise ValueError(\\\\n                "lambda_coupling needs an explicit coupling_target. The "\\\\n                "findings genuinely co-occur (gold OR 6.09, p = 0.003); a "\\\\n                "penalty toward independence optimises to a value the truth "\\\\n                "contradicts. Pass the observed coupling.")\\\\n        for name in tuple(stratified_findings) + (stratify_by,):\\\\n            if name not in config.LABELS:\\\\n                raise ValueError(f"unknown finding {name!r}")\\\\n\\\\n        self.save_hyperparameters(ignore=["backbone"])\\\\n        # Stage 1\\\\\\\'s residual parameterisation, built into the model so it\\\\n        # travels in the checkpoint rather than having to be reconstructed at\\\\n        # every scoring site. See KneeNet.couple.\\\\n        residual_pairs = None\\\\n        if residual_beta is not None:\\\\n            residual_pairs = [(config.LABELS.index(name),\\\\n                               config.LABELS.index(stratify_by))\\\\n                              for name in stratified_findings]\\\\n        self.model = net.KneeNet(backbone, feature_dim,\\\\n                                 input_mean=input_mean, input_std=input_std,\\\\n                                 gated_pool=gated_pool,\\\\n                                 classwise_pool=classwise_pool,\\\\n                                 slice_transformer=slice_transformer,\\\\n                                 residual_pairs=residual_pairs,\\\\n                                 residual_beta=residual_beta)\\\\n        self.backbone_lr_scale = backbone_lr_scale\\\\n        self.lr = lr\\\\n        self.weight_decay = weight_decay\\\\n        self.total_steps = total_steps\\\\n        self.max_lr_multiplier = max_lr_multiplier\\\\n        self.lambda_report = lambda_report\\\\n        self.lambda_rank = lambda_rank\\\\n        self.rank_queue_size = rank_queue_size\\\\n        self.rank_min_gap = rank_min_gap\\\\n        self.aux_mode = aux_mode\\\\n        self.aux_gate = aux_gate\\\\n        self.schedule = schedule\\\\n        self.warmup_steps = warmup_steps\\\\n        self.label_smoothing = label_smoothing\\\\n        self.symmetric_beta = symmetric_beta\\\\n        self.lambda_stratified_rank = lambda_stratified_rank\\\\n        self.stratified_findings = tuple(stratified_findings)\\\\n        self.stratify_by = stratify_by\\\\n        self.stratified_index = config.LABELS.index(stratify_by)\\\\n        self.stratified_caliper = stratified_caliper\\\\n        self.lambda_coupling = lambda_coupling\\\\n        self.coupling_target = coupling_target\\\\n\\\\n        # A buffer rather than a plain tensor: it has to follow the module to\\\\n        # the GPU, and it is not learned. Registered as None when unused so\\\\n        # the state dict is unchanged for every run that does not set it --\\\\n        # and it lives on the wrapper, not on `KneeNet`, so it never reaches\\\\n        # the exported checkpoint the submission loads.\\\\n        if pos_weight is not None:\\\\n            self.register_buffer(\\\\n                "pos_weight", torch.tensor(list(pos_weight), dtype=torch.float32))\\\\n        else:\\\\n            self.register_buffer("pos_weight", None)\\\\n\\\\n        # The report head is an attribute of the *wrapper*, deliberately: the\\\\n        # exported state dict keeps only `model.`-prefixed keys, so auxiliary\\\\n        # parameters vanish at export and notebook 10 keeps loading a bare\\\\n        # KneeNet. Created only when actually trained against -- under DDP a\\\\n        # parameter that never receives gradient is an error, not a no-op.\\\\n        if report_dim is not None and lambda_report > 0:\\\\n            self.report_head = torch.nn.Sequential(\\\\n                torch.nn.LayerNorm(feature_dim),\\\\n                torch.nn.Linear(feature_dim, report_dim))\\\\n        else:\\\\n            self.report_head = None\\\\n\\\\n        # Cross-batch queue for the ranking loss. At one study per rank there\\\\n        # are no pairs inside a batch, so current logits are ranked against\\\\n        # detached logits from recent steps (XBM-style). Slightly stale scores\\\\n        # are the price; per-rank and reset each epoch.\\\\n        self._rank_queue: List[Dict[str, torch.Tensor]] = []\\\\n\\\\n        # Not `_buffers`: nn.Module reserves that name for registered\\\\n        # buffers, and shadowing it corrupts state_dict and .to(device).\\\\n        self._val_outputs: Dict[int, List[Dict[str, torch.Tensor]]] = {0: [], 1: []}\\\\n\\\\n    def forward(self, volumes, meta, mask):\\\\n        return self.model(volumes, meta, mask)\\\\n\\\\n    def on_train_epoch_start(self):\\\\n        self._rank_queue = []\\\\n\\\\n    def _report_loss(self, pooled: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Cosine distance between the projected study embedding and the\\\\n        report\\\\\\\'s sentence embedding, over studies that have one.\\\\n\\\\n        The head must run even when the whole batch lacks reports: DDP\\\\n        requires every parameter to join every backward pass.\\\\n        """\\\\n        predicted = self.report_head(pooled.float())\\\\n        if "report" not in batch:\\\\n            return predicted.sum() * 0.0\\\\n        predicted = torch.nn.functional.normalize(predicted, dim=-1)\\\\n        target = torch.nn.functional.normalize(batch["report"].float(), dim=-1)\\\\n        distance = 1.0 - (predicted * target).sum(dim=-1)\\\\n        mask = batch["report_mask"].float()\\\\n        return (distance * mask).sum() / mask.sum().clamp(min=1.0)\\\\n\\\\n    def _rank_reference(self):\\\\n        """The queued detached scores both ranking losses rank against."""\\\\n        if not self._rank_queue:\\\\n            return None\\\\n        return {key: torch.cat([q[key] for q in self._rank_queue])\\\\n                for key in ("scores", "targets", "weights", "strata")}\\\\n\\\\n    def _rank_push(self, logits: torch.Tensor, batch) -> None:\\\\n        """Add this step\\\\\\\'s studies to the queue, once per training step.\\\\n\\\\n        Pushed after every ranking term has been computed, so a study never\\\\n        ranks against itself. `strata` is the predicted probability of the\\\\n        stratifying finding -- effusion -- carried because the stratified loss\\\\n        needs the reference studies\\\\\\\' effusion scores to decide which pairs are\\\\n        comparable, and recomputing them would mean a second forward pass over\\\\n        studies that have already left the batch.\\\\n        """\\\\n        self._rank_queue.append({\\\\n            "scores": logits.detach().float(),\\\\n            "targets": batch["targets"].detach().float(),\\\\n            "weights": batch["weights"].detach().float(),\\\\n            "strata": torch.sigmoid(\\\\n                logits[:, self.stratified_index].detach().float()),\\\\n        })\\\\n        while sum(len(q["scores"]) for q in self._rank_queue) > self.rank_queue_size:\\\\n            self._rank_queue.pop(0)\\\\n\\\\n    def _rank_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        return net.pairwise_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _stratified_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Stage 1: rank only within an effusion stratum.\\\\n\\\\n        Conditional AUC is within-stratum pair concordance, so this optimises\\\\n        the metric the finding is judged on rather than a proxy for it. Pairs\\\\n        are matched on the model\\\\\\\'s own predicted effusion score, detached: the\\\\n        mined effusion label has its own error, and a hard split would inherit\\\\n        it at the boundary.\\\\n        """\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        strata = torch.sigmoid(scores[:, self.stratified_index].detach())\\\\n        return net.stratified_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            strata, reference["strata"],\\\\n            self.stratified_findings, self.stratified_caliper,\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _coupling_loss(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Hold the head-to-head coupling at the truth\\\\\\\'s value, not at zero."""\\\\n        reference = self._rank_reference()\\\\n        queued = (reference["scores"] if reference is not None\\\\n                  else logits.new_zeros((0, logits.shape[1])))\\\\n        total = logits.sum() * 0.0\\\\n        for name in self.stratified_findings:\\\\n            total = total + net.coupling_penalty(\\\\n                logits.float(), queued, config.LABELS.index(name),\\\\n                self.stratified_index, self.coupling_target)\\\\n        return total\\\\n\\\\n    def _aux_cosine(self, main_loss: torch.Tensor, aux_loss: torch.Tensor,\\\\n                    pooled: torch.Tensor) -> torch.Tensor:\\\\n        """cos between the two losses\\\\\\\' gradients at the shared bottleneck.\\\\n\\\\n        Du et al., "Adapting Auxiliary Losses Using Gradient Similarity"\\\\n        (arXiv:1812.02224), Algorithm 1: apply the auxiliary gradient to the\\\\n        shared parameters only while it points into the same half-space as\\\\n        the main gradient, which guarantees convergence to the main task\\\\\\\'s\\\\n        critical points and blocks negative transfer.\\\\n\\\\n        The paper measures that cosine over the shared parameters theta.\\\\n        Here every shared parameter is reached *only* through `pooled` --\\\\n        `forward_pooled` returns `head(pooled), pooled`, and both losses hang\\\\n        off that one tensor -- so the shared gradients are J^T a and J^T b for\\\\n        the same Jacobian J = d pooled/d theta. Taking the cosine at the\\\\n        bottleneck asks the same question ("does the auxiliary want to move\\\\n        the shared representation the way the main task does?") for the price\\\\n        of two backward passes through the two small heads instead of two\\\\n        through ConvNeXt. It is a proxy, not an identity: J^T can rotate the\\\\n        pair, so agreement here is not arithmetically agreement at theta.\\\\n        The sign is what Algorithm 1 consumes, and the paper itself keeps\\\\n        only the sign because minibatch cosines are too noisy to trust\\\\n        beyond it.\\\\n\\\\n        retain_graph is required: the real backward still has to run\\\\n        afterwards over the same graph.\\\\n\\\\n        Both losses are scaled by a constant first. Training runs in\\\\n        `16-mixed`, and these probe gradients bypass Lightning\\\\\\\'s GradScaler,\\\\n        so an unscaled fp16 backward can flush small values to zero -- which\\\\n        reads as cosine 0, i.e. "aligned", i.e. silently back to a fixed\\\\n        lambda. Cosine is scale-invariant, so scaling both sides costs\\\\n        nothing and keeps the values off the floor.\\\\n        """\\\\n        scale = 1024.0\\\\n        grad_main, = torch.autograd.grad(main_loss * scale, pooled,\\\\n                                         retain_graph=True)\\\\n        grad_aux, = torch.autograd.grad(aux_loss * scale, pooled,\\\\n                                        retain_graph=True)\\\\n        cosine = torch.nn.functional.cosine_similarity(\\\\n            grad_main.flatten().float(), grad_aux.flatten().float(), dim=0)\\\\n        # Ranks must agree, or the "binary" gate is silently a soft one: DDP\\\\n        # averages the two ranks\\\\\\\' gradients, so a per-rank decision would\\\\n        # apply the auxiliary at half strength whenever they disagreed.\\\\n        # torch.distributed rather than self.all_gather, which requires an\\\\n        # attached Trainer and would make this untestable in isolation.\\\\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\\\\n            cosine = cosine.clone()\\\\n            torch.distributed.all_reduce(cosine, op=torch.distributed.ReduceOp.SUM)\\\\n            cosine = cosine / torch.distributed.get_world_size()\\\\n        return cosine\\\\n\\\\n    def _aux_weight(self, cosine: torch.Tensor) -> torch.Tensor:\\\\n        """Algorithm 1 (binary) or Algorithm 2 (cosine-weighted)."""\\\\n        if self.aux_gate == "weighted":\\\\n            return cosine.clamp(min=0.0)\\\\n        return (cosine >= 0).to(cosine.dtype)\\\\n\\\\n    def training_step(self, batch, batch_idx):\\\\n        logits, pooled = self.model.forward_pooled(\\\\n            batch["volumes"], batch["meta"], batch["mask"])\\\\n        loss = net.weighted_bce(\\\\n            logits, batch["targets"], batch["weights"],\\\\n            label_smoothing=self.label_smoothing,\\\\n            pos_weight=self.pos_weight,\\\\n            symmetric_beta=self.symmetric_beta)\\\\n        self.log("bce_loss", loss, on_step=False, on_epoch=True,\\\\n                 sync_dist=True, batch_size=len(logits))\\\\n        main_loss = loss\\\\n\\\\n        if self.report_head is not None:\\\\n            report_loss = self._report_loss(pooled, batch)\\\\n            weight = self.lambda_report\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, report_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                # The diagnostic worth having even when the gate never fires:\\\\n                # a report loss that is mostly anti-aligned explains a fixed\\\\n                # lambda hurting the leaderboard while helping validation.\\\\n                self.log("report_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n                self.log("report_gate", (cosine >= 0).float(), on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * report_loss\\\\n            self.log("report_loss", report_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0:\\\\n            rank_loss = self._rank_loss(logits, batch)\\\\n            weight = self.lambda_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, rank_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("rank_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * rank_loss\\\\n            self.log("rank_loss", rank_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_stratified_rank > 0:\\\\n            stratified_loss = self._stratified_loss(logits, batch)\\\\n            weight = self.lambda_stratified_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, stratified_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("stratified_cosine", cosine, on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * stratified_loss\\\\n            self.log("stratified_rank_loss", stratified_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_coupling > 0:\\\\n            coupling_loss = self._coupling_loss(logits)\\\\n            loss = loss + self.lambda_coupling * coupling_loss\\\\n            self.log("coupling_loss", coupling_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0 or self.lambda_stratified_rank > 0 \\\\\\\\\\\\n                or self.lambda_coupling > 0:\\\\n            self._rank_push(logits, batch)\\\\n\\\\n        self.log("train_loss", loss, prog_bar=True, on_step=True,\\\\n                 on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n        return loss\\\\n\\\\n    def validation_step(self, batch, batch_idx, dataloader_idx=0):\\\\n        logits = self(batch["volumes"], batch["meta"], batch["mask"])\\\\n        self._val_outputs[dataloader_idx].append({\\\\n            "probs": torch.sigmoid(logits.float()),\\\\n            "targets": batch["targets"].float(),\\\\n        })\\\\n\\\\n    def _gathered(self, dataloader_idx: int):\\\\n        """All-gather one buffer and return (truth, probs) as numpy."""\\\\n        entries = self._val_outputs[dataloader_idx]\\\\n        if not entries:\\\\n            return None, None\\\\n        probs = torch.cat([e["probs"] for e in entries])\\\\n        targets = torch.cat([e["targets"] for e in entries])\\\\n\\\\n        # all_gather returns (world_size, N, L) when every rank holds the same\\\\n        # N, which the distributed sampler guarantees by padding.\\\\n        probs = self.all_gather(probs)\\\\n        targets = self.all_gather(targets)\\\\n        if probs.ndim == 3:\\\\n            probs = probs.reshape(-1, probs.shape[-1])\\\\n            targets = targets.reshape(-1, targets.shape[-1])\\\\n        return (targets.cpu().numpy(), probs.cpu().numpy())\\\\n\\\\n    def _log_conditional(self, prefix: str, truth, probs) -> None:\\\\n        """Log conditional AUC beside the pooled number, per stratified finding.\\\\n\\\\n        The plan\\\\\\\'s §3, applied where it is cheapest to apply: pooled AUC on\\\\n        synovitis is reproducible from the effusion label alone (0.698 against\\\\n        the head\\\\\\\'s 0.701), so an epoch-level pooled number cannot show whether\\\\n        the run is learning the finding or learning effusion. Conditional AUC\\\\n        is the same statistic restricted to pairs effusion does not already\\\\n        separate.\\\\n\\\\n        Stratified on the model\\\\\\\'s own predicted effusion, split at its median\\\\n        -- there is no gold effusion label in the mined validation fold, and\\\\n        on the gold set the prediction gives 0.617 against the label\\\\\\\'s 0.630,\\\\n        so nothing material is lost by using the one that always exists.\\\\n\\\\n        Monitoring only, like `gold_macro`. Selecting a checkpoint on any\\\\n        gold-derived number is what inflated an earlier version of this\\\\n        project\\\\\\\'s reported figure.\\\\n        """\\\\n        for name in self.stratified_findings:\\\\n            index = config.LABELS.index(name)\\\\n            strata = conditional.strata_from_scores(probs[:, self.stratified_index])\\\\n            value = conditional.conditional_auc(\\\\n                truth[:, index], probs[:, index], strata)\\\\n            if np.isfinite(value):\\\\n                self.log(f"{prefix}_{name}_conditional", float(value),\\\\n                         rank_zero_only=False)\\\\n            pooled = auc(truth[:, index], probs[:, index])\\\\n            if np.isfinite(pooled):\\\\n                self.log(f"{prefix}_{name}_pooled", float(pooled),\\\\n                         rank_zero_only=False)\\\\n\\\\n    def on_validation_epoch_end(self):\\\\n        val_truth, val_probs = self._gathered(0)\\\\n        if val_truth is not None:\\\\n            # Mined labels are soft; binarise for a ranking metric.\\\\n            binarised = (val_truth >= 0.5).astype(int)\\\\n            val_macro = macro_auc(binarised, val_probs)\\\\n            self.log("val_macro", val_macro, prog_bar=True, rank_zero_only=False)\\\\n            self._log_conditional("val", binarised, val_probs)\\\\n\\\\n        gold_truth, gold_probs = self._gathered(1)\\\\n        if gold_truth is not None:\\\\n            gold_macro = macro_auc(gold_truth.astype(int), gold_probs)\\\\n            # Monitoring only. Never pass this to a checkpoint callback.\\\\n            self.log("gold_macro", gold_macro, prog_bar=True,\\\\n                     rank_zero_only=False)\\\\n            self._log_conditional("gold", gold_truth.astype(int), gold_probs)\\\\n\\\\n        self._val_outputs = {0: [], 1: []}\\\\n\\\\n    def configure_optimizers(self):\\\\n        # One group unless a trunk asks for a slower one. The aggregation is\\\\n        # trained from scratch and wants the full rate; a pretrained trunk at\\\\n        # the same rate is fine for a supervised convnet and destroys a\\\\n        # self-supervised ViT\\\\\\\'s features during OneCycle\\\\\\\'s warmup. Scale 1.0\\\\n        # takes the original single-group path unchanged, so every ConvNeXt\\\\n        # result stays comparable.\\\\n        if self.backbone_lr_scale == 1.0:\\\\n            optimiser = torch.optim.AdamW(self.parameters(), lr=self.lr,\\\\n                                          weight_decay=self.weight_decay)\\\\n            max_lr = self.lr * self.max_lr_multiplier\\\\n        else:\\\\n            trunk_ids = {id(p) for p in self.model.backbone.parameters()}\\\\n            trunk = [p for p in self.parameters() if id(p) in trunk_ids]\\\\n            rest = [p for p in self.parameters() if id(p) not in trunk_ids]\\\\n            trunk_lr = self.lr * self.backbone_lr_scale\\\\n            optimiser = torch.optim.AdamW(\\\\n                [{"params": trunk, "lr": trunk_lr},\\\\n                 {"params": rest, "lr": self.lr}],\\\\n                lr=self.lr, weight_decay=self.weight_decay)\\\\n            max_lr = [trunk_lr * self.max_lr_multiplier,\\\\n                      self.lr * self.max_lr_multiplier]\\\\n\\\\n        scheduler = self._scheduler(optimiser, max_lr)\\\\n        return {\\\\n            "optimizer": optimiser,\\\\n            # Stepped per optimiser step, so accumulation is already folded in.\\\\n            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},\\\\n        }\\\\n\\\\n    def _scheduler(self, optimiser, max_lr):\\\\n        """OneCycle or cosine, with warmup pinned to an absolute step count.\\\\n\\\\n        **The restretch is the thing being fixed here.** `OneCycleLR`\\\\\\\'s\\\\n        `pct_start` is a *fraction* of the run, and it defaults to 0.3. At\\\\n        EPOCHS=12 that puts the learning-rate peak near epoch 3.6; at\\\\n        EPOCHS=20 it moves to epoch 6. Two runs of different lengths are\\\\n        therefore at different points of a differently-shaped schedule at the\\\\n        same epoch number, and "epoch 9 was best" does not mean the same thing\\\\n        in each. Every cross-length comparison this project has drawn was\\\\n        reading through that.\\\\n\\\\n        Setting `warmup_steps` to an absolute count fixes the peak in place\\\\n        regardless of run length. Leaving it None keeps the 0.3 fraction, so\\\\n        the twelve-epoch reference runs stay reproducible.\\\\n\\\\n        `schedule="cosine"` replaces the cycle with linear warmup then cosine\\\\n        decay -- the schedule the RSNA 2022/2023 winning solutions used for\\\\n        fine-tuning, and one with no restretch artefact to reason about at\\\\n        all. It peaks at the same `lr * max_lr_multiplier` as OneCycle so the\\\\n        two are comparable at matched warmup.\\\\n        """\\\\n        warmup = self.warmup_steps\\\\n        if self.schedule == "onecycle":\\\\n            pct_start = 0.3 if warmup is None else warmup / self.total_steps\\\\n            return torch.optim.lr_scheduler.OneCycleLR(\\\\n                optimiser, max_lr=max_lr, total_steps=self.total_steps,\\\\n                pct_start=pct_start)\\\\n\\\\n        warmup = int(round(0.3 * self.total_steps)) if warmup is None else warmup\\\\n        peak = self.max_lr_multiplier\\\\n        decay_steps = max(1, self.total_steps - warmup)\\\\n\\\\n        def factor(step: int) -> float:\\\\n            # Returned as a multiple of the group\\\\\\\'s base lr, which is already\\\\n            # `lr * backbone_lr_scale` for the trunk group -- so the two\\\\n            # groups keep their ratio through the whole schedule.\\\\n            if step < warmup:\\\\n                return peak * (step + 1) / max(1, warmup)\\\\n            progress = min(1.0, (step - warmup) / decay_steps)\\\\n            return peak * 0.5 * (1.0 + np.cos(np.pi * progress))\\\\n\\\\n        return torch.optim.lr_scheduler.LambdaLR(optimiser, factor)\\\\n\\\\n\\\\ndef optimiser_steps(n_studies: int, batch_size: int, devices: int,\\\\n                    accumulate: int, epochs: int) -> int:\\\\n    """Total OneCycleLR steps, which must match what Lightning will take.\\\\n\\\\n    A schedule sized for the wrong number of steps either finishes its cycle\\\\n    early and trains the tail at a near-zero learning rate, or never reaches\\\\n    its peak. Neither raises.\\\\n    """\\\\n    if min(n_studies, batch_size, devices, accumulate, epochs) < 1:\\\\n        raise ValueError("study, batch, device, accumulation and epoch counts must be positive")\\\\n    per_rank_batches = int(np.ceil(n_studies / (batch_size * devices)))\\\\n    return epochs * int(np.ceil(per_rank_batches / accumulate))\\\\n\\\\n\\\\nclass WeightAverage(pl.Callback):\\\\n    """Maintain EMA and SWA of the `KneeNet` weights and write both out.\\\\n\\\\n    This is the selection-free half of the answer to the biased-selector\\\\n    problem described in `src/model/averaging.py`: rather than trusting mined\\\\n    validation to name an epoch, average the trajectory and ship the average.\\\\n    The best-validation checkpoint is still written by `ModelCheckpoint`\\\\n    alongside these, so a run produces three candidates and the comparison\\\\n    between them is the measurement.\\\\n\\\\n    **Updated on optimiser steps, not on batches.** With gradient\\\\n    accumulation, most batches leave the weights untouched, and folding an\\\\n    unchanged model into the EMA `accumulate` times per step would make the\\\\n    effective decay `decay ** accumulate` -- a shorter memory than asked for,\\\\n    varying with a knob that is supposed to be numerically neutral.\\\\n    `on_train_batch_end` samples only when `global_step` advances, after the\\\\n    optimizer has changed the weights, including the final update.\\\\n\\\\n    **Rank zero only.** DDP keeps every rank\\\\\\\'s weights identical after each\\\\n    step, so every rank would compute the same average; doing it once saves\\\\n    the copies. The written files therefore exist only on rank zero, which is\\\\n    also the only rank whose `/kaggle/working` survives.\\\\n\\\\n    **The averages are written from inside `fit`, in `on_train_end`, and read\\\\n    back off disk afterwards.** Under `ddp_notebook` the ranks are forked\\\\n    children: the callback object the notebook constructed is *not* the object\\\\n    that accumulated anything, and asking it for its EMA afterwards returns an\\\\n    empty average with no error. This is the same trap that leaves\\\\n    `best_model_score` as None on that strategy. So the file paths are fixed\\\\n    at construction and the notebook globs for what appeared.\\\\n    """\\\\n\\\\n    def __init__(self, prefix: str, decay: float = 0.999,\\\\n                 swa_fraction: float = 0.25,\\\\n                 total_steps: Optional[int] = None):\\\\n        if not 0.0 <= swa_fraction <= 1.0:\\\\n            raise ValueError(f"swa_fraction must be in [0, 1], got {swa_fraction}")\\\\n        self.prefix = prefix\\\\n        self.decay = decay\\\\n        self.swa_fraction = swa_fraction\\\\n        self.total_steps = total_steps\\\\n        self.average: Optional[averaging.WeightAverage] = None\\\\n        self.written: Dict[str, str] = {}\\\\n        self._last_step = 0\\\\n\\\\n    @property\\\\n    def ema_path(self) -> str:\\\\n        return f"{self.prefix}_ema.pt"\\\\n\\\\n    @property\\\\n    def swa_path(self) -> str:\\\\n        return f"{self.prefix}_swa.pt"\\\\n\\\\n    def _swa_from(self, trainer) -> Optional[int]:\\\\n        """The step the equal-weight tail average starts at."""\\\\n        total = self.total_steps\\\\n        if total is None:\\\\n            total = getattr(trainer, "estimated_stepping_batches", None)\\\\n        if not total or self.swa_fraction <= 0:\\\\n            return None\\\\n        # +1 because `update` counts from one: at total=4 and fraction=0.5 the\\\\n        # window must open at step 3, so that steps 3 and 4 -- the last half --\\\\n        # are what gets averaged, not steps 2 through 4.\\\\n        return int(round(float(total) * (1.0 - self.swa_fraction))) + 1\\\\n\\\\n    def on_train_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        self.average = averaging.WeightAverage(\\\\n            module.model, decay=self.decay, swa_from=self._swa_from(trainer))\\\\n        self._last_step = trainer.global_step\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        # Sample after the update, including the final trained weights.\\\\n        if self.average is not None and trainer.global_step > self._last_step:\\\\n            self.average.update(module.model)\\\\n            self._last_step = trainer.global_step\\\\n\\\\n    def on_train_end(self, trainer, module):\\\\n        # Wrapped, on the same rule the notebook\\\\\\\'s export cell follows: by the\\\\n        # time this fires, hours of GPU time exist as the monitored checkpoint\\\\n        # `ModelCheckpoint` already wrote, and an exception raised here\\\\n        # propagates out of `trainer.fit` and takes the export and scoring\\\\n        # cells with it. The averages are an experiment; the run is not.\\\\n        try:\\\\n            self.export()\\\\n        except Exception as exc:  # noqa: BLE001 - the checkpoint is safe\\\\n            print(f"WeightAverage could not write its averages ({exc}). "\\\\n                  "The monitored checkpoint is unaffected; only the "\\\\n                  "selection-free comparison is lost.")\\\\n\\\\n    def export(self) -> Dict[str, str]:\\\\n        """Write the averages as bare `KneeNet` state dicts.\\\\n\\\\n        The same format `export_from_checkpoint` produces, so notebook 10\\\\n        loads them with no idea they came from an average. Returns\\\\n        {kind: path} for what was actually written -- SWA is absent when the\\\\n        run was too short for any step to fall inside the tail window.\\\\n        """\\\\n        self.written = {}\\\\n        if self.average is None:\\\\n            return self.written\\\\n\\\\n        torch.save(self.average.ema_state(), self.ema_path)\\\\n        self.written["ema"] = self.ema_path\\\\n\\\\n        swa = self.average.swa_state()\\\\n        if swa is not None:\\\\n            torch.save(swa, self.swa_path)\\\\n            self.written["swa"] = self.swa_path\\\\n        return self.written\\\\n\\\\n\\\\nclass EpochReport(pl.Callback):\\\\n    """One line per epoch, from rank zero only.\\\\n\\\\n    Under DDP every rank runs this code, so an unguarded print produces one\\\\n    copy of the log per GPU and makes a run look twice as long as it is.\\\\n\\\\n    **This reads `callback_metrics` in `on_validation_end`, not in\\\\n    `on_validation_epoch_end`.** Lightning runs callback hooks before the\\\\n    LightningModule\\\\\\\'s hook of the same name, so a callback asking for\\\\n    `val_macro` at `on_validation_epoch_end` arrives before\\\\n    `KneeLitModule.on_validation_epoch_end` has computed it and reads an\\\\n    empty dict -- printing `nan` for every epoch of an otherwise healthy run.\\\\n    `ModelCheckpoint` selects in `on_validation_end` for the same reason, so\\\\n    checkpoint selection was never affected by this; only the log was.\\\\n    """\\\\n\\\\n    def __init__(self):\\\\n        self.rows: List[Dict[str, float]] = []\\\\n\\\\n    def on_validation_end(self, trainer, module):\\\\n        if trainer.sanity_checking or not trainer.is_global_zero:\\\\n            return\\\\n        metrics = trainer.callback_metrics\\\\n        val = float(metrics.get("val_macro", float("nan")))\\\\n        gold = float(metrics.get("gold_macro", float("nan")))\\\\n        self.rows.append({"epoch": trainer.current_epoch + 1,\\\\n                          "val_macro": val, "gold_macro": gold})\\\\n        best = max(row["val_macro"] for row in self.rows)\\\\n        marker = "  <- best" if val >= best else ""\\\\n        print(f"  epoch {trainer.current_epoch + 1}: val macro {val:.3f}  "\\\\n              f"gold {gold:.3f} (monitor only){marker}")\\\\n\\\\n\\\\nclass BatchProgress(pl.Callback):\\\\n    """Plain-text tqdm over training batches, rank zero only.\\\\n\\\\n    Lightning\\\\\\\'s own progress bar resolves to the notebook-widget flavour of\\\\n    tqdm inside Kaggle, and the widget never reaches the log of a\\\\n    Save & Run All -- eleven minutes pass per epoch with no sign of life.\\\\n    Plain tqdm writes carriage-returned text to stdout, which survives both\\\\n    the interactive view and the batch log, so the Trainer disables its own\\\\n    bar and takes this instead. `mininterval` keeps the batch log from\\\\n    bloating: one refresh per interval, not one per batch.\\\\n    """\\\\n\\\\n    def __init__(self, refresh_seconds: float = 15.0):\\\\n        self.refresh_seconds = refresh_seconds\\\\n        self._bar = None\\\\n\\\\n    def on_train_epoch_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        import sys\\\\n        from tqdm import tqdm\\\\n        self._bar = tqdm(\\\\n            total=int(trainer.num_training_batches),\\\\n            desc=f"epoch {trainer.current_epoch + 1}/{trainer.max_epochs}",\\\\n            unit="batch", file=sys.stdout,\\\\n            mininterval=self.refresh_seconds, leave=False)\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        if self._bar is not None:\\\\n            self._bar.update(1)\\\\n\\\\n    def on_train_epoch_end(self, trainer, module):\\\\n        if self._bar is not None:\\\\n            self._bar.close()\\\\n            self._bar = None\\\\n\\\\n\\\\ndef export_from_checkpoint(ckpt_path: str, out_path: str) -> int:\\\\n    """Convert a Lightning checkpoint into the plain state dict for scoring.\\\\n\\\\n    Read from the file rather than from a module in memory: under the spawn\\\\n    strategy the process that called `fit` is not the process that trained,\\\\n    and reaching for in-memory weights there is how a run silently ships its\\\\n    initialisation. Returns the number of tensors written.\\\\n    """\\\\n    checkpoint = torch.load(ckpt_path, map_location="cpu")\\\\n    state_dict = checkpoint.get("state_dict", checkpoint)\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in state_dict.items()\\\\n             if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(\\\\n            f"no \\\\\\\'model.\\\\\\\' keys in {ckpt_path}; found "\\\\n            f"{list(state_dict)[:3]}")\\\\n    torch.save(state, out_path)\\\\n    return len(state)\\\\n\\\\n\\\\ndef export_state_dict(module: KneeLitModule, path: str) -> None:\\\\n    """Write a plain `KneeNet` state dict, the format the submission loads.\\\\n\\\\n    Lightning prefixes every key with `model.` because `KneeNet` is an\\\\n    attribute of the wrapper. Notebook 10 constructs a bare `KneeNet` and\\\\n    calls `load_state_dict`, so the prefix has to come off here rather than\\\\n    become a surprise at submission time.\\\\n    """\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in module.state_dict().items()\\\\n             if key.startswith("model.")}\\\\n    torch.save(state, path)\\\\n\\\'}\\nfor path, source in FILES.items():\\n    target = pathlib.Path(path)\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    target.write_text(source)\\nsys.path.insert(0, \\\'.\\\')\\nfor _name in [m for m in sys.modules if m == \\\'src\\\' or m.startswith(\\\'src.\\\')]:\\n    del sys.modules[_name]\\nimportlib.invalidate_caches()\\nfrom src import config\\nfrom src.data import dicom_io, cache, index as index_mod\\nif pathlib.Path(\\\'src/model/side_classifier.py\\\').exists():\\n    from src.model import side_classifier\\nif pathlib.Path(\\\'src/model/net.py\\\').exists():\\n    from src.model import augment as knee_augment, averaging, conditional, embed, dataset as knee_dataset, net, splits\\n    from src.model import artifacts, experiments\\nif pathlib.Path(\\\'src/model/inspect.py\\\').exists():\\n    from src.model import inspect as knee_inspect\\nif pathlib.Path(\\\'src/labels/fuse.py\\\').exists():\\n    from src.labels import fuse\\nif pathlib.Path(\\\'src/model/lightning.py\\\').exists():\\n    from src.model import lightning as knee_lightning\\nprint(\\\'modules ready:\\\', len(FILES), \\\'files\\\')\\nprint(\\\'labels:\\\', len(config.LABELS))\\nprint(\\\'cache geometry:\\\', config.CACHE_SLICES, \\\'x\\\', config.CACHE_SIZE)\\nprint(\\\'data root:\\\', config.DATA_ROOT)\\n\\nEXPERIMENT = os.environ.get(\\\'KNEE_EXPERIMENT\\\', \\\'resample\\\')\\nRUN = experiments.get_experiment(EXPERIMENT)\\nfrom dataclasses import replace\\nRUN = replace(RUN, series_selection=\\\'protocol\\\')\\nprint(\\\'experiment:\\\', RUN.to_dict())\\n\\nimport glob\\nimport numpy as np\\nimport pandas as pd\\n\\ndef find_inputs(pattern):\\n    found = sorted(glob.glob(f\\\'data/**/{pattern}\\\', recursive=True))\\n    found += sorted(glob.glob(f\\\'/kaggle/working/slice_transformer_fold2/{pattern}\\\'))\\n    for root in sorted(glob.glob(\\\'/kaggle/input/*\\\')):\\n        if os.path.basename(root) == \\\'competitions\\\' or os.path.exists(os.path.join(root, \\\'train.csv\\\')) or os.path.exists(os.path.join(root, \\\'train_series.csv\\\')):\\n            continue\\n        found += sorted(glob.glob(f\\\'{root}/**/{pattern}\\\', recursive=True))\\n    return found\\n\\ndef find_competition_file(name):\\n    for candidate in (f\\\'data/{name}\\\', f\\\'/kaggle/input/competitions/*/{name}\\\', f\\\'/kaggle/input/*/{name}\\\'):\\n        hits = sorted(glob.glob(candidate))\\n        if hits:\\n            return hits[0]\\n    raise SystemExit(f\\\'missing competition file: {name}\\\')\\nlabels = pd.read_parquet(find_inputs(\\\'train_labels.parquet\\\')[0])\\nSILENCE_REWEIGHT = False\\nif SILENCE_REWEIGHT:\\n    _rates = fuse.silence_rates(labels)\\n    labels = fuse.rescale_silent_weights(labels)\\n    print(\\\'silence reweighting ON\\\')\\n    for _n in sorted(config.LABELS, key=lambda x: -_rates[x])[:3]:\\n        print(f\\\'  {_n:18s} silent {_rates[_n]:5.1%} -> weight {fuse.WEIGHT_SILENT * (1 - _rates[_n]):.3f}\\\')\\nelse:\\n    print(\\\'silence reweighting OFF (reference labels)\\\')\\nsides = pd.read_parquet(find_inputs(\\\'side_predictions.parquet\\\')[0])\\nreport_hits = find_inputs(\\\'report_embeddings.parquet\\\')\\nreports = pd.read_parquet(report_hits[0]) if report_hits else None\\nif reports is None:\\n    print(\\\'report_embeddings.parquet not attached -- report loss disabled\\\')\\nseries_meta = pd.read_csv(find_competition_file(\\\'train_series.csv\\\'))\\ntrain_csv = pd.read_csv(find_competition_file(\\\'train.csv\\\'))\\ngold = train_csv[train_csv[\\\'ACL\\\'].notna()].reset_index(drop=True)\\nALL_SHARDS = True\\ncache_paths = find_inputs(\\\'cache_train_series_*.npy\\\')\\nif not ALL_SHARDS:\\n    cache_paths = cache_paths[:1]\\nshards, frames = ({}, [])\\nfor cache_path in cache_paths:\\n    manifest_path = cache_path.replace(\\\'cache_\\\', \\\'manifest_\\\').replace(\\\'.npy\\\', \\\'.parquet\\\')\\n    if not os.path.exists(manifest_path):\\n        stem = os.path.basename(cache_path).replace(\\\'cache_\\\', \\\'\\\').replace(\\\'.npy\\\', \\\'\\\')\\n        hits = [m for m in find_inputs(\\\'manifest_train_series_*.parquet\\\') if os.path.basename(m).startswith(f\\\'manifest_{stem}\\\')]\\n        if not hits:\\n            continue\\n        manifest_path = hits[0]\\n    frame = pd.read_parquet(manifest_path)\\n    frame[\\\'cache_path\\\'] = cache_path\\n    shards[cache_path] = np.load(cache_path, mmap_mode=\\\'r\\\')\\n    frames.append(frame)\\n    print(\\\'shard:\\\', os.path.basename(cache_path), shards[cache_path].shape)\\nmanifest = pd.concat(frames, ignore_index=True)\\nGROUPED_FOLDS = False\\ngroups = None\\nif GROUPED_FOLDS:\\n    _index_files = find_inputs(\\\'index_train.parquet\\\') + find_inputs(\\\'index.parquet\\\')\\n    if _index_files:\\n        groups = splits.scanner_key(pd.read_parquet(_index_files[0]))\\n        print(f\\\'grouped folds: {groups.nunique()} scanner groups over {len(groups)} studies\\\')\\n    else:\\n        raise ValueError("GROUPED_FOLDS requires notebook 01\\\'s index parquet")\\nfolded = splits.make_folds(labels, groups=groups)\\nsplits.assert_no_gold_leakage(folded[\\\'StudyInstanceUID\\\'], set(gold[\\\'StudyInstanceUID\\\']))\\nif groups is not None:\\n    _summary = splits.summarise(folded)\\n    print(_summary[[\\\'studies\\\'] + config.LABELS[:4]].to_string())\\n    print(\\\'  fold sizes are uneven by construction -- a scanner group cannot be split. Check the positive rates before trusting a comparison.\\\')\\nMAX_SERIES = 5\\nMAX_SERIES = RUN.max_series\\nindex = knee_dataset.StudyIndex(manifest, series_meta, folded, sides, max_series=MAX_SERIES, reports=reports, series_selection=RUN.series_selection)\\ngold_labels = gold[[\\\'StudyInstanceUID\\\'] + config.LABELS].copy()\\nfor name in config.LABELS:\\n    gold_labels[f\\\'{name}_weight\\\'] = 1.0\\ngold_index = knee_dataset.StudyIndex(manifest, series_meta, gold_labels, sides, max_series=MAX_SERIES, series_selection=RUN.series_selection)\\nif ALL_SHARDS and (set(index.studies) != set(labels[config.ID_COL]) or set(gold_index.studies) != set(gold[config.ID_COL])):\\n    raise ValueError(\\\'incomplete cache coverage: attach every training and gold study\\\')\\nprint(f\\\'studies in this shard: {len(index)}  gold present: {len(gold_index)}\\\')\\n\\nfrom torch.utils.data import DataLoader\\nSLICES_PER_SERIES = 16\\nSLICES_PER_SERIES = RUN.slices\\nRESAMPLE = True\\nRESAMPLE = RUN.resample\\nBATCH_STUDIES = 1\\nEFFECTIVE_STUDIES = 8\\nACCUM_STEPS = max(1, EFFECTIVE_STUDIES // (BATCH_STUDIES * DEVICES))\\nprint(f\\\'{BATCH_STUDIES} studies x {DEVICES} devices x {ACCUM_STEPS} accum = {BATCH_STUDIES * DEVICES * ACCUM_STEPS} effective\\\')\\nCACHE_PATHS = list(shards)\\n\\ndef make_loader(study_index, studies, shuffle=False, augment=False, resample=False):\\n    return DataLoader(knee_lightning.StudyVolumeDataset(study_index, studies, CACHE_PATHS, SLICES_PER_SERIES, augment, resample=resample), batch_size=BATCH_STUDIES, shuffle=shuffle, num_workers=2, collate_fn=knee_dataset.collate_studies, pin_memory=True, persistent_workers=True)\\n\\nimport timm\\nBACKBONE = \\\'convnext_tiny.fb_in22k_ft_in1k\\\'\\nBACKBONE = RUN.backbone\\n_warm = net.build_backbone(BACKBONE)\\nFEATURE_DIM = _warm.num_features\\nprint(f\\\'{BACKBONE}: {FEATURE_DIM} features, {sum((p.numel() for p in _warm.parameters())) / 1000000.0:.1f}M params\\\')\\ndel _warm\\nBACKBONE_DEFAULTS = net.backbone_defaults(BACKBONE)\\nprint(\\\'backbone defaults:\\\', BACKBONE_DEFAULTS)\\nauc = knee_lightning.auc\\n\\nimport time\\nimport timm\\n_required = {\\\'DEVICES\\\': \\\'1. Setup\\\', \\\'knee_lightning\\\': \\\'2. Repo modules\\\', \\\'index\\\': \\\'3. Assemble\\\', \\\'make_loader\\\': \\\'4. Dataset\\\', \\\'FEATURE_DIM\\\': \\\'5. Model\\\'}\\n_missing = {name: cell for name, cell in _required.items() if name not in globals()}\\nif _missing:\\n    raise SystemExit(\\\'This cell needs the cells above it, and this kernel has not run them:\\\\n\\\' + \\\'\\\\n\\\'.join((f\\\'    {name!r} comes from section {cell}\\\' for name, cell in _missing.items())) + \\\'\\\\n\\\\nRun the notebook from the top (Runtime -> Run all). Changing FOLD and re-running only this cell works within a LIVE kernel; a restarted runtime has to re-run everything.\\\')\\nEPOCHS = 12\\nLR = 0.0001\\nWARMUP_EPOCHS = 3.6\\nSCHEDULE = \\\'onecycle\\\'\\nSEED = 0\\npl.seed_everything(SEED, workers=True)\\nPOS_WEIGHT_CAP = None\\nif POS_WEIGHT_CAP is None:\\n    POS_WEIGHT = None\\nelse:\\n    _positives = (labels[config.LABELS] >= 0.5).sum()\\n    _negatives = len(labels) - _positives\\n    POS_WEIGHT = [float(min((n / max(p, 1)) ** 0.5, POS_WEIGHT_CAP)) for p, n in zip(_positives, _negatives)]\\n    print(f\\\'pos_weight (sqrt of neg/pos, capped at {POS_WEIGHT_CAP}):\\\')\\n    for _name, _w in zip(config.LABELS, POS_WEIGHT):\\n        _p = int((labels[_name] >= 0.5).sum())\\n        print(f\\\'  {_name:18s} {_w:5.2f}   ({_p} positives, {_p / len(labels):.1%})\\\')\\nLAMBDA_REPORT = 0.0\\nLAMBDA_RANK = 0.0\\nLAMBDA_STRATIFIED_RANK = 0.0\\nSTRATIFIED_FINDINGS = (\\\'Synovitis\\\',)\\nSTRATIFY_BY = \\\'Effusion\\\'\\nSTRATIFIED_CALIPER = 0.15\\nRESIDUAL_BETA = None\\nLAMBDA_COUPLING = 0.0\\nCOUPLING_TARGET = 0.403\\nAUX_MODE = \\\'fixed\\\'\\nAUX_GATE = \\\'binary\\\'\\nREPORT_DIM = len([c for c in reports.columns if c != \\\'StudyInstanceUID\\\']) if reports is not None else None\\nFOLD = 2\\nCHECKPOINT = f\\\'/kaggle/working/slice_transformer_fold2/convnext_fold{FOLD}.pt\\\'\\nAVERAGE_PREFIX = f\\\'/kaggle/working/slice_transformer_fold2/avg_fold{FOLD}\\\'\\nkeyed = folded.set_index(\\\'StudyInstanceUID\\\')\\ntrain_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] != FOLD]\\nval_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] == FOLD]\\ntrain_loader = make_loader(index, train_studies, shuffle=True, augment=True, resample=RESAMPLE)\\nprint(f"resampling slices/series per epoch: {(\\\'on\\\' if RESAMPLE else \\\'off\\\')}")\\nval_loader = make_loader(index, val_studies)\\ngold_loader = make_loader(gold_index, gold_index.studies)\\nprint(f\\\'fold {FOLD}: train {len(train_studies)}  val {len(val_studies)}  gold {len(gold_index)}\\\')\\nTOTAL_STEPS = knee_lightning.optimiser_steps(len(train_studies), BATCH_STUDIES, DEVICES, ACCUM_STEPS, EPOCHS)\\nprint(\\\'optimiser steps:\\\', TOTAL_STEPS)\\nWARMUP_STEPS = int(round(TOTAL_STEPS / EPOCHS * WARMUP_EPOCHS))\\nprint(f"warmup: {WARMUP_STEPS} steps ({WARMUP_EPOCHS} epochs\\\' worth), schedule {SCHEDULE}")\\nmodule = knee_lightning.KneeLitModule(backbone=net.build_backbone(BACKBONE), feature_dim=FEATURE_DIM, lr=LR, total_steps=TOTAL_STEPS, report_dim=REPORT_DIM, lambda_report=LAMBDA_REPORT, lambda_rank=LAMBDA_RANK, aux_mode=AUX_MODE, aux_gate=AUX_GATE, schedule=SCHEDULE, warmup_steps=WARMUP_STEPS, lambda_stratified_rank=LAMBDA_STRATIFIED_RANK, stratified_findings=STRATIFIED_FINDINGS, stratify_by=STRATIFY_BY, stratified_caliper=STRATIFIED_CALIPER, lambda_coupling=LAMBDA_COUPLING, coupling_target=COUPLING_TARGET if LAMBDA_COUPLING > 0 else None, residual_beta=RESIDUAL_BETA, pos_weight=POS_WEIGHT, classwise_pool=RUN.classwise_pool, slice_transformer=RUN.slice_transformer, **BACKBONE_DEFAULTS)\\nprint(f"trunk lr: {LR * BACKBONE_DEFAULTS[\\\'backbone_lr_scale\\\']:.1e}  head lr: {LR:.1e}  input norm: {(\\\'on\\\' if BACKBONE_DEFAULTS[\\\'input_mean\\\'] else \\\'off\\\')}")\\nprint(f\\\'aux losses: report {LAMBDA_REPORT} (dim {REPORT_DIM})  rank {LAMBDA_RANK}  mode {AUX_MODE}\\\' + (f\\\'/{AUX_GATE}\\\' if AUX_MODE == \\\'cosine\\\' else \\\'\\\'))\\nprint(f\\\'synovitis: stratified rank {LAMBDA_STRATIFIED_RANK} (caliper {STRATIFIED_CALIPER})  residual beta {RESIDUAL_BETA}  coupling {LAMBDA_COUPLING}\\\' + (f\\\' -> {COUPLING_TARGET}\\\' if LAMBDA_COUPLING > 0 else \\\'\\\'))\\nENSEMBLE_DONE = {0: \\\'2026-08-31, 12 ep, val 0.8814, gold 0.8683\\\', 1: \\\'2026-08-31, 12 ep, val 0.8726, gold 0.8747\\\', 2: \\\'2026-08-31, 12 ep, val 0.8675, gold 0.8659\\\', 3: \\\'2026-08-31, 12 ep, val 0.8736, gold 0.8915\\\', 4: \\\'2026-08-31, 12 ep, val 0.8697, gold 0.8753\\\'}\\nENSEMBLE_TODO = [f for f in range(5) if f not in ENSEMBLE_DONE]\\n_expected = {\\\'SLICES_PER_SERIES\\\': (SLICES_PER_SERIES, 16), \\\'MAX_SERIES\\\': (MAX_SERIES, 5), \\\'EPOCHS\\\': (EPOCHS, 12), \\\'LAMBDA_REPORT\\\': (LAMBDA_REPORT, 0.0), \\\'LAMBDA_RANK\\\': (LAMBDA_RANK, 0.0), \\\'LAMBDA_STRATIFIED_RANK\\\': (LAMBDA_STRATIFIED_RANK, 0.0), \\\'RESIDUAL_BETA\\\': (RESIDUAL_BETA, None), \\\'LAMBDA_COUPLING\\\': (LAMBDA_COUPLING, 0.0), \\\'POS_WEIGHT_CAP\\\': (POS_WEIGHT_CAP, None), \\\'RESAMPLE\\\': (RESAMPLE, False), \\\'SERIES_SELECTION\\\': (RUN.series_selection, \\\'uniform\\\'), \\\'CLASSWISE_POOL\\\': (RUN.classwise_pool, False), \\\'SLICE_TRANSFORMER\\\': (RUN.slice_transformer, False), \\\'SILENCE_REWEIGHT\\\': (SILENCE_REWEIGHT, False), \\\'BACKBONE\\\': (BACKBONE, \\\'convnext_tiny.fb_in22k_ft_in1k\\\')}\\n_wrong = {k: v for k, (v, want) in _expected.items() if v != want}\\nprint(\\\'=\\\' * 68)\\nif _wrong:\\n    print(f\\\'  OFF-REFERENCE FOLD {FOLD}. These differ from the ensemble config:\\\')\\n    for k, v in _wrong.items():\\n        print(f\\\'    {k} = {v!r}, reference is {_expected[k][1]!r}\\\')\\n    print(\\\'  An ensemble member trained off-reference makes the average a\\\')\\n    print(\\\'  blend of two models, and the averaging hides it. Fix these, or\\\')\\n    print(\\\'  accept that this run is an experiment and not a member.\\\')\\nelif FOLD in ENSEMBLE_DONE:\\n    print(f\\\'  FOLD {FOLD} IS ALREADY DONE -- {ENSEMBLE_DONE[FOLD]}\\\')\\n    print(f\\\'  Retraining it is fine, but the folds still missing are {ENSEMBLE_TODO}.\\\')\\nelse:\\n    print(f\\\'  ENSEMBLE FOLD {FOLD}: reference config, no auxiliary loss\\\')\\n    print(f\\\'  done: {sorted(ENSEMBLE_DONE)}   outstanding after this: {[f for f in ENSEMBLE_TODO if f != FOLD]}\\\')\\n    print(f\\\'  export convnext_fold{FOLD}.pt and COPY IT OFF THE VM before the\\\')\\n    print(\\\'  session ends -- /kaggle/working does not survive.\\\')\\nprint(\\\'=\\\' * 68)\\ncheckpointer = pl.callbacks.ModelCheckpoint(dirpath=\\\'/kaggle/working/slice_transformer_fold2\\\', filename=f\\\'lit_fold{FOLD}\\\', monitor=\\\'val_macro\\\', mode=\\\'max\\\', save_top_k=1)\\nreport = knee_lightning.EpochReport()\\naverager = knee_lightning.WeightAverage(prefix=AVERAGE_PREFIX, decay=0.999, swa_fraction=0.25, total_steps=TOTAL_STEPS)\\ncsv_logger = pl.loggers.CSVLogger(\\\'/kaggle/working/slice_transformer_fold2\\\', name=f\\\'lit_fold{FOLD}\\\', version=\\\'metrics\\\')\\nSTRATEGY = \\\'ddp\\\' if DEVICES > 1 else \\\'auto\\\'\\nif STRATEGY == \\\'ddp_notebook\\\' and torch.cuda.is_initialized():\\n    raise SystemExit("CUDA is already initialised in this process, so the forked workers cannot start.\\\\nRun -> Restart Session, then run the cells in order. If it recurs, set STRATEGY = \\\'ddp_spawn\\\' in this cell.")\\ntrainer = pl.Trainer(max_epochs=EPOCHS, accelerator=\\\'gpu\\\', devices=DEVICES, strategy=STRATEGY, precision=\\\'16-mixed\\\', accumulate_grad_batches=ACCUM_STEPS, callbacks=[checkpointer, report, averager, knee_lightning.BatchProgress()], logger=csv_logger, log_every_n_steps=50, enable_model_summary=False, enable_progress_bar=False)\\nstart = time.time()\\nassert (FOLD, EPOCHS, SEED, SLICES_PER_SERIES, MAX_SERIES) == (EXPECTED_FOLD, 12, 0, 16, 5)\\nassert not RESAMPLE and (not GROUPED_FOLDS)\\nassert RUN.classwise_pool == (EXPERIMENT == \\\'classwise\\\')\\nassert not SILENCE_REWEIGHT and POS_WEIGHT is None and (RESIDUAL_BETA is None)\\nassert (LAMBDA_REPORT, LAMBDA_RANK, LAMBDA_STRATIFIED_RANK, LAMBDA_COUPLING) == (0.0, 0.0, 0.0, 0.0)\\nassert len(index) == 4349 and len(gold_index) == 58 and (len(CACHE_PATHS) == 3)\\nassert len(train_studies) == 4349 - EXPECTED_VAL and len(val_studies) == EXPECTED_VAL\\nassert BATCH_STUDIES * DEVICES * ACCUM_STEPS == 8\\nassert BACKBONE == (\\\'convnextv2_tiny.fcmae_ft_in22k_in1k\\\' if EXPERIMENT == \\\'convnextv2\\\' else \\\'convnext_tiny.fb_in22k_ft_in1k\\\')\\nif MATCH_BACKBONE:\\n    assert FEATURE_DIM == 768\\n    assert BACKBONE_DEFAULTS == {\\\'input_mean\\\': None, \\\'input_std\\\': None, \\\'backbone_lr_scale\\\': 1.0}\\nif MATCH_SLICE:\\n    assert RUN.series_selection == \\\'protocol\\\'\\n    assert RUN.slice_transformer == (EXPERIMENT == \\\'slice_transformer\\\')\\nelse:\\n    assert RUN.series_selection == (\\\'protocol\\\' if EXPERIMENT == \\\'protocol\\\' else \\\'uniform\\\')\\nassert all((volume.shape[1:] == (24, 224, 224) for volume in shards.values()))\\nimport hashlib, json, pathlib\\n_initial = hashlib.sha256()\\nfor _key, _tensor in sorted(module.state_dict().items()):\\n    _initial.update(_key.encode())\\n    _initial.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n_contract = {\\\'experiment\\\': RUN.to_dict(), \\\'fold\\\': FOLD, \\\'epochs\\\': EPOCHS, \\\'seed\\\': SEED, \\\'devices\\\': DEVICES, \\\'effective_batch\\\': EFFECTIVE_STUDIES, \\\'steps\\\': TOTAL_STEPS, \\\'lr\\\': LR, \\\'schedule\\\': SCHEDULE, \\\'warmup_steps\\\': WARMUP_STEPS, \\\'initial_weights_sha256\\\': _initial.hexdigest(), \\\'torch\\\': torch.__version__, \\\'timm\\\': timm.__version__, \\\'lightning\\\': pl.__version__}\\nif MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE:\\n    _excluded = (\\\'model.slice_encoder.\\\',) if MATCH_SLICE else (\\\'model.backbone.\\\',) if MATCH_BACKBONE else (\\\'model.study_pool.\\\', \\\'model.head.\\\')\\n    _shared = hashlib.sha256()\\n    for _key, _tensor in sorted(module.state_dict().items()):\\n        if not _key.startswith(_excluded):\\n            _shared.update(_key.encode())\\n            _shared.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n    _contract[\\\'shared_initial_weights_sha256\\\'] = _shared.hexdigest()\\nif MATCH_BACKBONE:\\n    _contract[\\\'aggregation_seed\\\'] = SEED\\n    _contract[\\\'feature_dim\\\'] = FEATURE_DIM\\n    _contract[\\\'input_defaults\\\'] = BACKBONE_DEFAULTS\\nif \\\'REFERENCE_CONTRACT\\\' in globals():\\n    _fold_specific = {\\\'fold\\\', \\\'steps\\\', \\\'warmup_steps\\\'}\\n    assert {k: v for k, v in _contract.items() if k not in _fold_specific} == {k: v for k, v in REFERENCE_CONTRACT.items() if k not in _fold_specific}, \\\'Completion settings or initialization differ from screened protocol\\\'\\n_baseline_contract = pathlib.Path(BASELINE_CONTRACT_PATH)\\nif EXPERIMENT != \\\'baseline\\\' and _baseline_contract.exists():\\n    _control = json.loads(_baseline_contract.read_text())\\n    _ignored = {\\\'experiment\\\'} | ({\\\'initial_weights_sha256\\\'} if MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE else set())\\n    assert {k: v for k, v in _contract.items() if k not in _ignored} == {k: v for k, v in _control.items() if k not in _ignored}, \\\'Candidate settings or shared initialization differ from baseline\\\'\\nif int(os.environ.get(\\\'LOCAL_RANK\\\', \\\'0\\\')) == 0:\\n    pathlib.Path(\\\'run_contract.json\\\').write_text(json.dumps(_contract, indent=2) + \\\'\\\\n\\\')\\nprint(\\\'Matched-run preflight passed\\\', flush=True)\\ntrainer.fit(module, train_dataloaders=train_loader, val_dataloaders=[val_loader, gold_loader])\\nif not trainer.is_global_zero:\\n    raise SystemExit(0)\\nelapsed = time.time() - start\\n\\ndef recover_checkpoint(fold):\\n    """The checkpoint this run wrote, when `best_model_path` came back empty.\\n\\n    NOT `sorted(glob(...))[0]`, which is what this was. That picks the first\\n    name ALPHABETICALLY, and "-" (45) sorts before "." (46), which sorts\\n    before "_" (95): `lit_fold1-v1.ckpt` beats `lit_fold1.ckpt`, which beats\\n    `lit_fold1_old.ckpt`. Lightning writes those `-vN` suffixes on every name\\n    collision, so a session that resumed after a crash silently recovered a\\n    checkpoint from the earlier attempt.\\n\\n    The exact name wins when it exists, and otherwise the newest file does --\\n    which is this run\\\'s, since this cell runs seconds after fit returned.\\n    """\\n    exact = f\\\'/kaggle/working/slice_transformer_fold2/lit_fold{fold}.ckpt\\\'\\n    if os.path.exists(exact):\\n        return exact\\n    found = glob.glob(f\\\'/kaggle/working/slice_transformer_fold2/lit_fold{fold}*.ckpt\\\')\\n    return max(found, key=os.path.getmtime) if found else None\\ncheckpoint_path = checkpointer.best_model_path or \\\'\\\'\\nif not checkpoint_path or not os.path.exists(checkpoint_path):\\n    checkpoint_path = recover_checkpoint(FOLD)\\n    if checkpoint_path is None:\\n        raise SystemExit(\\\'training finished but no checkpoint was written -- check the epoch log for whether val_macro was ever computed\\\')\\n    print(f\\\'best_model_path was empty; recovered {checkpoint_path}\\\')\\nprint(\\\'\\\\nbest checkpoint:\\\', checkpoint_path)\\nn_tensors = knee_lightning.export_from_checkpoint(checkpoint_path, CHECKPOINT)\\nprint(f\\\'wrote {CHECKPOINT} ({n_tensors} tensors)\\\')\\nAVERAGES = {kind: f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\' for kind in (\\\'ema\\\', \\\'swa\\\') if os.path.exists(f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\')}\\nfor kind, path in AVERAGES.items():\\n    print(f\\\'wrote {path} ({kind.upper()} weights, selection-free)\\\')\\n_actual_spec = RUN.to_dict()\\n_actual_spec.update(backbone=BACKBONE, slices=SLICES_PER_SERIES, max_series=MAX_SERIES, resample=RESAMPLE)\\nfor _path in [CHECKPOINT] + list(AVERAGES.values()):\\n    artifacts.write_metadata(_path, _actual_spec, FOLD, train_studies, val_studies)\\nfolded[[config.ID_COL, \\\'fold\\\']].to_parquet(\\\'/kaggle/working/slice_transformer_fold2/folds.parquet\\\', index=False)\\nif not AVERAGES:\\n    print(\\\'NOTE: no averaged weights on disk -- the WeightAverage callback did not run to on_train_end. The monitored checkpoint is fine; only the P2 comparison is unavailable.\\\')\\ntry:\\n    history = pd.read_csv(os.path.join(csv_logger.log_dir, \\\'metrics.csv\\\'))\\n    per_epoch_rows = history.dropna(subset=[\\\'val_macro\\\']).groupby(\\\'epoch\\\')[[\\\'val_macro\\\', \\\'gold_macro\\\']].last()\\n    best_epoch = per_epoch_rows[\\\'val_macro\\\'].idxmax()\\n    print(f"\\\\n{\\\'epoch\\\':>5s} {\\\'val macro\\\':>9s} {\\\'gold\\\':>7s}")\\n    for epoch_no, row in per_epoch_rows.iterrows():\\n        marker = \\\'  <- selected\\\' if epoch_no == best_epoch else \\\'\\\'\\n        print(f"{epoch_no + 1:5d} {row[\\\'val_macro\\\']:9.3f} {row[\\\'gold_macro\\\']:7.3f}{marker}")\\n    best = per_epoch_rows.loc[best_epoch]\\n    print(f"\\\\nbest val macro {best[\\\'val_macro\\\']:.3f} at epoch {best_epoch + 1}; gold there {best[\\\'gold_macro\\\']:.3f} (monitor only, never selected on)")\\n    _val = per_epoch_rows[\\\'val_macro\\\']\\n    _gold = per_epoch_rows[\\\'gold_macro\\\']\\n    _tail = _gold.loc[best_epoch:]\\n    print(\\\'\\\\nP1 -- val/gold divergence\\\')\\n    print(f\\\'  val   peak {_val.max():.3f} at epoch {_val.idxmax() + 1}, final {_val.iloc[-1]:.3f}  ({_val.iloc[-1] - _val.max():+.3f})\\\')\\n    print(f\\\'  gold  peak {_gold.max():.3f} at epoch {_gold.idxmax() + 1}, final {_gold.iloc[-1]:.3f}  ({_gold.iloc[-1] - _gold.max():+.3f})\\\')\\n    print(f\\\'  gold from the selected epoch on: range {_tail.max() - _tail.min():.3f} over {len(_tail)} epochs\\\')\\n    if _val.iloc[-1] < _val.max() - 0.005 and _tail.max() - _tail.min() < 0.02:\\n        print(\\\'  -> DIVERGENCE REPRODUCES. Mined val fell while gold held flat: the selector is biased, and every fold selected on it is paying for it. Ship the averaged weights.\\\')\\n    else:\\n        print(\\\'  -> no divergence in this run. Selecting on mined val is defensible here; re-read before adopting the averages fleet-wide.\\\')\\nexcept Exception as exc:\\n    print(f\\\'\\\\ncould not read the epoch table ({exc}); look under {csv_logger.save_dir} for metrics.csv\\\')\\nper_epoch = elapsed / EPOCHS\\nshare = len(index) / 4349\\nprint(f\\\'\\\\nthis shard is {share:.0%} of the training set\\\')\\nprint(f\\\'per epoch: {per_epoch / 60:.1f} min on {DEVICES} GPU(s)\\\')\\nprint(f\\\'projected 5 folds x {EPOCHS} epochs: {per_epoch / share * EPOCHS * 5 / 3600:.1f} h\\\')\\n\\ndevice = \\\'cuda\\\'\\nscoring_path = CHECKPOINT\\nif not os.path.exists(scoring_path):\\n    scoring_path = recover_checkpoint(FOLD)\\n    if scoring_path is None:\\n        raise SystemExit(f\\\'neither {CHECKPOINT} nor a lit_fold{FOLD} checkpoint is present\\\')\\n    print(f\\\'{CHECKPOINT} missing; scoring from {scoring_path}\\\')\\n\\ndef load_for_scoring(path):\\n    state = net.load_knee_state(path, map_location=device)\\n    model = net.KneeNet(net.build_backbone(BACKBONE, pretrained=False), FEATURE_DIM, input_mean=BACKBONE_DEFAULTS[\\\'input_mean\\\'], input_std=BACKBONE_DEFAULTS[\\\'input_std\\\'], **net.aggregation_defaults(state)).to(device)\\n    model.load_state_dict(state)\\n    return model.eval()\\n\\ndef gold_predictions(model):\\n    ids, rows = ([], [])\\n    with torch.no_grad(), torch.amp.autocast(\\\'cuda\\\'):\\n        for batch in gold_loader:\\n            logits = model(batch[\\\'volumes\\\'].to(device), batch[\\\'meta\\\'].to(device), batch[\\\'mask\\\'].to(device))\\n            rows.append(torch.sigmoid(logits.float()).cpu().numpy())\\n            ids += batch[config.ID_COL]\\n    return (ids, np.concatenate(rows))\\nprint(\\\'scoring with input normalisation:\\\', \\\'on\\\' if BACKBONE_DEFAULTS[\\\'input_mean\\\'] else \\\'off\\\')\\nscoring_model = load_for_scoring(scoring_path)\\ngold_ids, gold_probs = gold_predictions(scoring_model)\\ntruth = gold.set_index(\\\'StudyInstanceUID\\\').loc[gold_ids]\\nFROZEN = dict(zip(config.LABELS, [0.728, 0.671, 0.698, 0.68, 0.895, 0.899, 0.772, 0.897, 0.746, 0.862, 0.711, 0.683]))\\nREFERENCE_GOLD = 0.875\\nBEST_GOLD = 0.875\\nprint(f"{\\\'finding\\\':18s} {\\\'this run\\\':>9s}  {\\\'frozen (retired)\\\':>16s}")\\nprint(\\\'-\\\' * 47)\\nscores = []\\nfor i, name in enumerate(config.LABELS):\\n    value = auc(truth[name].to_numpy().astype(int), gold_probs[:, i])\\n    scores.append(value)\\n    print(f\\\'{name:18s} {value:9.3f}  {FROZEN[name]:16.3f}\\\')\\nprint(\\\'-\\\' * 47)\\nmacro = float(np.nanmean(scores))\\nprint(f"{\\\'macro\\\':18s} {macro:9.3f}  {np.mean(list(FROZEN.values())):16.3f}")\\nprint(f\\\'\\\\nthe comparisons that matter (fold 1, 16 x 5):\\\')\\nprint(f\\\'  vs reference  gold {REFERENCE_GOLD:.3f} (LB 0.889): {macro - REFERENCE_GOLD:+.3f}\\\')\\nprint(f\\\'  vs best       gold {BEST_GOLD:.3f} (LB 0.891): {macro - BEST_GOLD:+.3f}\\\')\\nif abs(macro - REFERENCE_GOLD) < 0.02:\\n    print("  -> inside the gold 58\\\'s ~0.02 noise floor: this is a null against the reference, whatever the frozen column says.")\\nif FOLD != 1:\\n    print(f"  -> NOTE: this is fold {FOLD}; both references are fold 1, so these deltas are not comparable. Judge fold members by the ensemble\\\'s leaderboard score, not by gold here.")\\nif AVERAGES:\\n    print(\\\'\\\\nP2 -- selection-free weights (gold macro)\\\')\\n    print(f"  {\\\'best mined-val epoch\\\':24s} {macro:.3f}")\\n    for kind, path in AVERAGES.items():\\n        _ids, _probs = gold_predictions(load_for_scoring(path))\\n        _truth = gold.set_index(\\\'StudyInstanceUID\\\').loc[_ids]\\n        _macro = float(np.nanmean([auc(_truth[name].to_numpy().astype(int), _probs[:, i]) for i, name in enumerate(config.LABELS)]))\\n        print(f"  {kind.upper() + \\\' (selection-free)\\\':24s} {_macro:.3f}  ({_macro - macro:+.3f} vs selected)")\\n    print("  gold\\\'s macro SE on 58 studies is ~0.02; treat anything under that as a tie and decide on the leaderboard.")\\nGOLD_PREDICTIONS = f\\\'/kaggle/working/slice_transformer_fold2/gold_predictions_fold{FOLD}.parquet\\\'\\npd.DataFrame(gold_probs, columns=config.LABELS).assign(**{config.ID_COL: gold_ids})[config.SUBMISSION_COLUMNS].to_parquet(GOLD_PREDICTIONS, index=False)\\nprint(\\\'\\\\nwrote\\\', GOLD_PREDICTIONS)\\n\\noof_ids, oof_rows = ([], [])\\nwith torch.no_grad(), torch.amp.autocast(\\\'cuda\\\'):\\n    for batch in val_loader:\\n        logits = scoring_model(batch[\\\'volumes\\\'].to(device), batch[\\\'meta\\\'].to(device), batch[\\\'mask\\\'].to(device))\\n        oof_rows.append(torch.sigmoid(logits.float()).cpu().numpy())\\n        oof_ids += batch[config.ID_COL]\\noof = pd.DataFrame(np.concatenate(oof_rows), columns=config.LABELS)\\noof.insert(0, config.ID_COL, oof_ids)\\noof[\\\'fold\\\'] = FOLD\\nOOF_PATH = f\\\'/kaggle/working/slice_transformer_fold2/oof_fold{FOLD}.parquet\\\'\\noof.to_parquet(OOF_PATH, index=False)\\nprint(f\\\'wrote {OOF_PATH}: {len(oof)} studies x {len(config.LABELS)} findings\\\')\\nprint(f\\\'mean predicted rate: {oof[config.LABELS].to_numpy().mean():.3f}\\\')\\n_spread = oof[config.LABELS].to_numpy().std(axis=0)\\nif float(_spread.min()) < 0.01:\\n    _flat = [n for n, s in zip(config.LABELS, _spread) if s < 0.01]\\n    print(f\\\'  WARNING: near-constant predictions for {_flat} -- check before using this dump for label work\\\')\\n\'}\nSMOKE_SOURCE = \'\\ndef attached_files(pattern):\\n    import pathlib\\n    found = []\\n    for root in sorted(pathlib.Path("/kaggle/input").iterdir()):\\n        if root.name not in ("competitions", "rsna-knee-abnormality-detection"):\\n            found.extend(root.rglob(pattern))\\n    return found\\n\\nimport hashlib, os, pathlib\\nEXPECTED_FOLD = 2\\nEXPECTED_VAL = 870\\nMATCH_CLASSWISE = False\\nMATCH_BACKBONE = False\\nMATCH_SLICE = True\\nBASELINE_CONTRACT_PATH = "/kaggle/working/baseline_fold2/run_contract.json"\\nos.environ["KNEE_EXPERIMENT"] = \\\'slice_transformer\\\'\\nos.environ["PYTHONUNBUFFERED"] = "1"\\npaths = attached_files("train_labels.parquet")\\nassert len(paths) == 1, f"Expected one label file, found {paths}"\\nassert hashlib.sha256(paths[0].read_bytes()).hexdigest() == \\\'e54e46531aec6cd16131d70aa283c990542dfa17daa890ac54da663cf74569b0\\\', "Label file changed"\\nimport faulthandler\\nfaulthandler.enable()\\n# Keep fatal-signal diagnostics, but do not arm a background stack-dump timer.\\n# Fold 2 and fold 4 SIGSEGVs occurred inside scheduled dumps while training\\n# was progressing. Parent time limits are configured by the staging command.\\n\\n\\nimport os\\nimport subprocess\\nos.environ[\\\'PYTORCH_NVML_BASED_CUDA_CHECK\\\'] = \\\'1\\\'\\nimport torch\\nDEVICES = torch.cuda.device_count()\\nif DEVICES == 0:\\n    raise SystemExit(\\\'Settings -> Accelerator -> GPU T4 x2\\\')\\nprint(subprocess.run([\\\'nvidia-smi\\\', \\\'--query-gpu=name,memory.total\\\', \\\'--format=csv,noheader\\\'], capture_output=True, text=True).stdout.strip())\\nprint(\\\'devices:\\\', DEVICES, \\\'| cuda initialised:\\\', torch.cuda.is_initialized())\\nif DEVICES < 2:\\n    print(\\\'NOTE: one GPU visible. Settings -> Accelerator -> GPU T4 x2 roughly halves wall-clock. The run is correct either way -- ACCUM_STEPS below adapts to hold the effective batch constant.\\\')\\ntry:\\n    import lightning.pytorch as pl\\nexcept ImportError:\\n    try:\\n        import pytorch_lightning as pl\\n    except ImportError:\\n        raise SystemExit(\\\'pip install lightning, then re-run this cell\\\')\\nprint(\\\'lightning:\\\', pl.__version__)\\n\\nimport importlib, pathlib, sys\\nFILES = {\\\'src/__init__.py\\\': \\\'\\\', \\\'src/config.py\\\': \\\'"""Central configuration: label names, cache geometry, and environment paths.\\\\n\\\\nEverything here is imported by both the training pipeline and the submission\\\\nnotebook, so it must stay dependency-free apart from the standard library.\\\\n"""\\\\n\\\\nimport os\\\\nfrom pathlib import Path\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Labels\\\\n# ---------------------------------------------------------------------------\\\\n# Exact submission spelling. The apostrophe in "Baker\\\\\\\'s" and the spaces are\\\\n# part of the column names -- do not normalise them anywhere that touches a\\\\n# submission file.\\\\nLABELS = [\\\\n    "ACL",\\\\n    "MCL",\\\\n    "Medial Meniscus",\\\\n    "Lateral Meniscus",\\\\n    "Medial OA",\\\\n    "Lateral OA",\\\\n    "PF OA",\\\\n    "Effusion",\\\\n    "Synovitis",\\\\n    "Baker\\\\\\\'s",\\\\n    "Contusion",\\\\n    "Fracture",\\\\n]\\\\n\\\\nID_COL = "StudyInstanceUID"\\\\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\\\\n\\\\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\\\\n# any augmentation that mirrors the image must also swap these pairs.\\\\nLATERAL_LABEL_PAIRS = [\\\\n    ("Medial Meniscus", "Lateral Meniscus"),\\\\n    ("Medial OA", "Lateral OA"),\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cache geometry\\\\n# ---------------------------------------------------------------------------\\\\n# Kept as constants so a higher-resolution variant is a config change rather\\\\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\\\\n# x 24,371 series ~= 30 GB, which fits Kaggle\\\\\\\'s 200 GB private dataset limit.\\\\nCACHE_SIZE = 224          # in-plane resolution after resampling\\\\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\\\\nCACHE_DTYPE = "uint8"\\\\n\\\\n# Robust intensity normalisation percentiles. MRI has no standardised units,\\\\n# so every volume is scaled against its own distribution.\\\\nCLIP_PCT_LOW = 1.0\\\\nCLIP_PCT_HIGH = 99.0\\\\n\\\\n# Canonical laterality: left knees are mirrored to match right knees so the\\\\n# medial compartment always falls on the same side of the image.\\\\nCANONICAL_SIDE = "R"\\\\n\\\\n# Mirroring is applied at load time, not baked into the cache.\\\\n#\\\\n# The side tag is blank on most series in this dataset and the series\\\\n# descriptions are anonymised, so roughly 42% of series have no recoverable\\\\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\\\\n# would mean rebuilding all of it every time side detection improves -- and it\\\\n# is expected to improve, via study-level propagation and then a classifier\\\\n# trained on the series whose side is known. The cache therefore stores volumes\\\\n# in their acquired orientation with the side recorded alongside, and the data\\\\n# loader flips on read, which costs nothing.\\\\nMIRROR_IN_CACHE = False\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Paths\\\\n# ---------------------------------------------------------------------------\\\\nCOMPETITION = "rsna-knee-abnormality-detection"\\\\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\\\\n\\\\n\\\\n# What a competition data root must contain to be recognised.\\\\n_MARKERS = ("train_series", "test_series")\\\\n\\\\n\\\\ndef _detect_data_root() -> Path:\\\\n    """Locate the competition data across Kaggle, Colab, and local runs.\\\\n\\\\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\\\\\\\'s input directory is\\\\n    searched by content rather than by name: the mount is not reliably at\\\\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\\\\n    through to the local fallback, which then fails much later with a confusing\\\\n    path in the message.\\\\n    """\\\\n    env = os.environ.get("KNEE_DATA_ROOT")\\\\n    if env:\\\\n        return Path(env)\\\\n\\\\n    kaggle_input = Path("/kaggle/input")\\\\n    if kaggle_input.exists():\\\\n        exact = kaggle_input / COMPETITION\\\\n        if exact.exists():\\\\n            return exact\\\\n        # Search two levels: Kaggle mounts competition data under\\\\n        # /kaggle/input/competitions/<slug>/ as well as directly under\\\\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\\\\n        try:\\\\n            for child in sorted(kaggle_input.iterdir()):\\\\n                if not child.is_dir():\\\\n                    continue\\\\n                if any((child / m).exists() for m in _MARKERS):\\\\n                    return child\\\\n                for grandchild in sorted(child.iterdir()):\\\\n                    if grandchild.is_dir() and any(\\\\n                            (grandchild / m).exists() for m in _MARKERS):\\\\n                        return grandchild\\\\n        except OSError:\\\\n            pass\\\\n\\\\n    return PROJECT_ROOT / "data"\\\\n\\\\n\\\\nDATA_ROOT = _detect_data_root()\\\\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\\\\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\\\\n\\\\nTRAIN_CSV = META_DIR / "train.csv"\\\\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\\\\nTEST_CSV = META_DIR / "test.csv"\\\\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\\\\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\\\\n\\\\nINDEX_PARQUET = META_DIR / "index.parquet"\\\\n\\\\nON_KAGGLE = Path("/kaggle").exists()\\\\n\\\', \\\'src/data/__init__.py\\\': \\\'\\\', \\\'src/data/dicom_io.py\\\': \\\'"""DICOM reading, geometric slice ordering, and laterality resolution.\\\\n\\\\nTwo operations here are load-bearing and fail silently if done naively:\\\\n\\\\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\\\\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\\\\n   slice normal derived from ImageOrientationPatient, which is defined by the\\\\n   scanner geometry rather than by convention.\\\\n\\\\n2. **Laterality.** Four of the twelve labels are compartment-specific\\\\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\\\\n   on the opposite side of the frame from a right knee, so left knees must be\\\\n   mirrored into a canonical orientation before a model can learn the\\\\n   distinction.\\\\n"""\\\\n\\\\nimport re\\\\nfrom pathlib import Path\\\\nfrom typing import Iterable, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pydicom\\\\n\\\\n# Header fields pulled during the index scan. Kept explicit so the parquet\\\\n# schema is stable and the scan stays header-only.\\\\nHEADER_FIELDS = [\\\\n    "StudyInstanceUID",\\\\n    "SeriesInstanceUID",\\\\n    "SOPInstanceUID",\\\\n    "Rows",\\\\n    "Columns",\\\\n    "PixelSpacing",\\\\n    "ImagePositionPatient",\\\\n    "ImageOrientationPatient",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "ScanningSequence",\\\\n    "SequenceVariant",\\\\n    # (0018,0010). The one header that states contrast administration\\\\n    # outright, and the index built before 2026-08-31 does not carry it --\\\\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\\\\n    # which are anonymised away. Absence of a string is not absence of\\\\n    # contrast; absence of this tag is at least a direct question asked.\\\\n    "ContrastBolusAgent",\\\\n    "SeriesDescription",\\\\n    "ProtocolName",\\\\n    "BodyPartExamined",\\\\n    "Manufacturer",\\\\n    "ManufacturerModelName",\\\\n    "MagneticFieldStrength",\\\\n    "ImageLaterality",\\\\n    "Laterality",\\\\n    "InstanceNumber",\\\\n]\\\\n\\\\n# DICOM DS/IS values carry their original string representation, which would\\\\n# land in the index as object dtype. Cast these explicitly.\\\\nNUMERIC_FIELDS = {\\\\n    "Rows",\\\\n    "Columns",\\\\n    "SliceThickness",\\\\n    "SpacingBetweenSlices",\\\\n    "EchoTime",\\\\n    "RepetitionTime",\\\\n    "InversionTime",\\\\n    "MagneticFieldStrength",\\\\n    "InstanceNumber",\\\\n}\\\\n\\\\n_LEFT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n_RIGHT_PATTERN = re.compile(\\\\n    r"\\\\\\\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\\\\\\\b",\\\\n    re.IGNORECASE,\\\\n)\\\\n\\\\n\\\\ndef read_header(path) -> dict:\\\\n    """Read one DICOM\\\\\\\'s headers without decoding pixel data."""\\\\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\\\\n    row = {"path": str(path)}\\\\n    for field in HEADER_FIELDS:\\\\n        value = getattr(ds, field, None)\\\\n        if value is None:\\\\n            row[field] = None\\\\n        elif isinstance(value, pydicom.multival.MultiValue):\\\\n            # Geometry fields are numeric; ScanningSequence and friends are\\\\n            # multi-valued strings, so fall back to a joined token.\\\\n            try:\\\\n                row[field] = [float(v) for v in value]\\\\n            except (TypeError, ValueError):\\\\n                row[field] = "\\\\\\\\\\\\\\\\".join(str(v) for v in value)\\\\n        elif field in NUMERIC_FIELDS:\\\\n            try:\\\\n                row[field] = float(value)\\\\n            except (TypeError, ValueError):\\\\n                row[field] = None\\\\n        elif isinstance(value, (int, float, str)):\\\\n            row[field] = str(value) if isinstance(value, str) else value\\\\n        else:\\\\n            row[field] = str(value)\\\\n    return row\\\\n\\\\n\\\\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\\\\n    """Unit normal of the image plane, from the two direction cosines.\\\\n\\\\n    Falls back to the z axis when orientation is missing, which keeps a\\\\n    malformed series ordered by patient-superior position rather than crashing.\\\\n    """\\\\n    if orientation is None or len(orientation) != 6:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n\\\\n    row_cosine = np.asarray(orientation[:3], dtype=float)\\\\n    col_cosine = np.asarray(orientation[3:], dtype=float)\\\\n    normal = np.cross(row_cosine, col_cosine)\\\\n\\\\n    magnitude = np.linalg.norm(normal)\\\\n    if magnitude == 0:\\\\n        return np.array([0.0, 0.0, 1.0])\\\\n    return normal / magnitude\\\\n\\\\n\\\\ndef slice_position(header: dict) -> float:\\\\n    """Scalar position of a slice along its own stack axis."""\\\\n    position = header.get("ImagePositionPatient")\\\\n    if position is None or len(position) != 3:\\\\n        # No geometry available; fall back to InstanceNumber so ordering is at\\\\n        # least deterministic. The index scan flags these series separately.\\\\n        number = header.get("InstanceNumber")\\\\n        return float(number) if number is not None else 0.0\\\\n\\\\n    normal = slice_normal(header.get("ImageOrientationPatient"))\\\\n    return float(np.dot(np.asarray(position, dtype=float), normal))\\\\n\\\\n\\\\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\\\\n    """Order a series\\\\\\\' slices along the acquisition axis."""\\\\n    return sorted(headers, key=slice_position)\\\\n\\\\n\\\\ndef has_geometry(headers: Sequence[dict]) -> bool:\\\\n    """True when every slice carries the position and orientation tags."""\\\\n    for header in headers:\\\\n        position = header.get("ImagePositionPatient")\\\\n        orientation = header.get("ImageOrientationPatient")\\\\n        if position is None or len(position) != 3:\\\\n            return False\\\\n        if orientation is None or len(orientation) != 6:\\\\n            return False\\\\n    return True\\\\n\\\\n\\\\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\\\\n    """Determine whether a series images a left or right knee.\\\\n\\\\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\\\\n    description fields, which carry side markers in several languages. Returns\\\\n    "L", "R", or None when the side cannot be established -- callers must treat\\\\n    None as unusable rather than guessing, since a wrong side silently swaps the\\\\n    medial and lateral labels.\\\\n    """\\\\n    for header in headers:\\\\n        for field in ("ImageLaterality", "Laterality"):\\\\n            value = header.get(field)\\\\n            if isinstance(value, str):\\\\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\\\\n                # often present-but-empty than absent. An empty string is not a\\\\n                # side and must not be read as one.\\\\n                token = value.strip().upper()\\\\n                if token in ("L", "LEFT"):\\\\n                    return "L"\\\\n                if token in ("R", "RIGHT"):\\\\n                    return "R"\\\\n\\\\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\\\\n        for header in headers:\\\\n            text = header.get(field)\\\\n            if not isinstance(text, str):\\\\n                continue\\\\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\\\\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\\\\n            left = bool(_LEFT_PATTERN.search(spaced))\\\\n            right = bool(_RIGHT_PATTERN.search(spaced))\\\\n            # Ambiguous strings mentioning both sides tell us nothing.\\\\n            if left and not right:\\\\n                return "L"\\\\n            if right and not left:\\\\n                return "R"\\\\n\\\\n    return None\\\\n\\\\n\\\\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\\\\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\\\\n\\\\n    Applies the rescale slope and intercept when present. Slices whose in-plane\\\\n    shape differs from the first are skipped rather than silently broadcast.\\\\n    """\\\\n    slices = []\\\\n    reference_shape = None\\\\n    skipped = 0\\\\n\\\\n    for path in paths:\\\\n        try:\\\\n            ds = pydicom.dcmread(str(path), force=True)\\\\n            pixels = ds.pixel_array.astype(np.float32)\\\\n        except Exception:  # noqa: BLE001\\\\n            # A truncated or malformed file loses one slice, not the series.\\\\n            # The dataset contains files whose declared pixel length exceeds\\\\n            # what is actually stored, and dropping 23 good slices over one\\\\n            # bad one costs a whole study its imaging.\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\\\\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\\\\n        if slope != 1.0 or intercept != 0.0:\\\\n            pixels = pixels * slope + intercept\\\\n\\\\n        if reference_shape is None:\\\\n            reference_shape = pixels.shape\\\\n        elif pixels.shape != reference_shape:\\\\n            skipped += 1\\\\n            continue\\\\n\\\\n        slices.append(pixels)\\\\n\\\\n    if not slices:\\\\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\\\\n\\\\n    return np.stack(slices, axis=0)\\\\n\\\\n\\\\ndef series_paths(series_dir) -> List[str]:\\\\n    """All DICOM files under one series directory."""\\\\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\\\\n\\\\n\\\\ndef resolve_study_laterality(series_sides: dict) -> dict:\\\\n    """Fill in unknown series sides from other series in the same study.\\\\n\\\\n    A knee MRI study images one knee, so a side established on any series\\\\n    applies to all of them. This matters because the DICOM side tags are\\\\n    populated inconsistently across the contributing institutions -- a study\\\\n    can carry the tag on its coronal series and omit it on the sagittals.\\\\n\\\\n    Report text is deliberately not used as a source here. Reports exist only\\\\n    for training studies, so recovering a side from them would mirror the\\\\n    training volumes and leave the test volumes untouched -- a train/test skew\\\\n    that is worse than the missing side it fixes.\\\\n\\\\n    Args:\\\\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\\\\n\\\\n    Returns:\\\\n        The same structure with unknown entries filled where the study allows.\\\\n        Studies whose series disagree are left untouched, since a conflict\\\\n        means the assumption of one knee per study does not hold there.\\\\n    """\\\\n    resolved = {}\\\\n    for study_uid, series in series_sides.items():\\\\n        known = {side for side in series.values() if side in ("L", "R")}\\\\n\\\\n        if len(known) == 1:\\\\n            side = known.pop()\\\\n            resolved[study_uid] = {uid: side for uid in series}\\\\n        else:\\\\n            # No evidence, or contradictory evidence. Guessing would silently\\\\n            # swap the medial and lateral labels on four of twelve findings.\\\\n            resolved[study_uid] = dict(series)\\\\n\\\\n    return resolved\\\\n\\\\n\\\\ndef laterality_coverage(series_sides: dict) -> dict:\\\\n    """Counts before and after study-level propagation, for auditing."""\\\\n    before = sum(1 for s in series_sides.values() for v in s.values()\\\\n                 if v in ("L", "R"))\\\\n    total = sum(len(s) for s in series_sides.values())\\\\n\\\\n    filled = resolve_study_laterality(series_sides)\\\\n    after = sum(1 for s in filled.values() for v in s.values()\\\\n                if v in ("L", "R"))\\\\n\\\\n    conflicts = sum(\\\\n        1 for s in series_sides.values()\\\\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\\\\n    )\\\\n\\\\n    return {\\\\n        "total_series": total,\\\\n        "known_before": before,\\\\n        "known_after": after,\\\\n        "recovered": after - before,\\\\n        "conflicting_studies": conflicts,\\\\n    }\\\\n\\\', \\\'src/data/cache.py\\\': \\\'"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\\\\n\\\\nThe same code path runs during cache construction and at submission time. Any\\\\ndivergence between the two produces a train/test preprocessing skew, which is\\\\ninvisible in local validation and only shows up as a collapsed leaderboard\\\\nscore -- so callers must import from here rather than reimplementing.\\\\n\\\\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\\\\norientation it was acquired in. Mirroring to a canonical side happens at load\\\\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport cv2\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\\\\n    """Scale a volume to uint8 using its own robust percentile range.\\\\n\\\\n    MRI carries no standardised units -- intensity depends on scanner, field\\\\n    strength, coil, and whether fat suppression was applied -- so each volume is\\\\n    scaled against its own distribution rather than a global constant.\\\\n    """\\\\n    finite = volume[np.isfinite(volume)]\\\\n    if finite.size == 0:\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\\\\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\\\\n        # Flat or degenerate volume; nothing to stretch.\\\\n        return np.zeros(volume.shape, dtype=np.uint8)\\\\n\\\\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\\\\n    return (scaled * 255.0).round().astype(np.uint8)\\\\n\\\\n\\\\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\\\\n    """Resample every slice to size x size with area/linear interpolation."""\\\\n    size = size or config.CACHE_SIZE\\\\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\\\\n\\\\n    for i, plane in enumerate(volume):\\\\n        # Area interpolation avoids the aliasing that bilinear introduces when\\\\n        # downsampling 800px acquisitions; linear is correct when upsampling.\\\\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\\\\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\\\\n\\\\n    return output\\\\n\\\\n\\\\ndef select_slices(count: int, target: int = None) -> List[int]:\\\\n    """Indices of `target` slices spread evenly across a stack of `count`.\\\\n\\\\n    Short series are handled by repeating edge slices rather than zero-padding,\\\\n    so the volume stays anatomically continuous.\\\\n    """\\\\n    target = target or config.CACHE_SLICES\\\\n    if count <= 0:\\\\n        raise ValueError("cannot select slices from an empty series")\\\\n    if count == target:\\\\n        return list(range(count))\\\\n\\\\n    positions = np.linspace(0, count - 1, target)\\\\n    return [int(round(p)) for p in positions]\\\\n\\\\n\\\\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\\\\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\\\\n\\\\n    Mirroring a knee means reflecting through the patient\\\\\\\'s left-right axis,\\\\n    and which array axis that corresponds to depends on the acquisition plane:\\\\n\\\\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\\\\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\\\\n\\\\n    A blanket column flip is wrong for sagittal series. It leaves medial and\\\\n    lateral exactly where they were -- they are separated across slices, not\\\\n    across the image -- and instead reflects anterior-posterior, moving the\\\\n    patella to the back of the knee.\\\\n\\\\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\\\\n    """\\\\n    left_right = np.array([1.0, 0.0, 0.0])\\\\n\\\\n    if orientation is None or len(orientation) != 6:\\\\n        # No geometry to reason from. Columns is the common case (coronal and\\\\n        # axial together outnumber sagittal), but the caller should prefer not\\\\n        # to mirror at all when geometry is missing.\\\\n        return 2\\\\n\\\\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\\\\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\\\\n    normal = dicom_io.slice_normal(orientation)\\\\n\\\\n    alignments = [\\\\n        (abs(float(normal @ left_right)), 0),\\\\n        (abs(float(height @ left_right)), 1),\\\\n        (abs(float(width @ left_right)), 2),\\\\n    ]\\\\n    return max(alignments)[1]\\\\n\\\\n\\\\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\\\\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\\\\n    """Mirror left knees so the medial compartment is always on one side.\\\\n\\\\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\\\\n    Lateral OA). Without this the model would have to infer the side before it\\\\n    could place a finding, from a cue that is not reliably in the image.\\\\n\\\\n    The axis reflected is chosen from the acquisition geometry -- see\\\\n    mirror_axis -- because it differs between sagittal and coronal series.\\\\n    """\\\\n    if side is None:\\\\n        # Unknown side: leave untouched and let the caller decide. Guessing\\\\n        # would silently swap four of the twelve labels.\\\\n        return volume\\\\n    if side.upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = mirror_axis(orientation)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef submission_indices(count: int, slices: int) -> List[int]:\\\\n    """Raw slice indices a `slices`-deep submission actually reads.\\\\n\\\\n    **Not** `select_slices(count, slices)`, and the difference is the whole\\\\n    reason this exists. The submission decodes `CACHE_SLICES` and then\\\\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\\\\n    `slices` evenly across the raw stack in one step lands on *different\\\\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\\\\n    then be scored on anatomy it was not trained to read, and nothing would\\\\n    raise -- the shapes agree either way. That is the same failure the\\\\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\\\\n\\\\n    So the two selections are composed rather than collapsed: the coarse pick\\\\n    the cache would have made, then the fine pick the model applies to it.\\\\n    """\\\\n    coarse = select_slices(count)\\\\n    if slices == config.CACHE_SLICES:\\\\n        return coarse\\\\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\\\\n\\\\n\\\\ndef build_series_volume(headers: Sequence[dict],\\\\n                        slices: Optional[int] = None) -> Dict:\\\\n    """Full pipeline for one series: sort, decode, normalise, resize.\\\\n\\\\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\\\\n    not by default. The recorded `laterality` is what the loader mirrors on.\\\\n    Returns the volume plus the provenance needed to audit it later.\\\\n\\\\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\\\\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\\\\n    decoding done for images the model never sees, and inference time is a\\\\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\\\\n    construction is unchanged and every existing shard stays valid.\\\\n\\\\n    **It is not bit-identical to decoding the full depth.**\\\\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\\\\n    computed over whatever was decoded, so a shallower stack shifts the\\\\n    intensity mapping slightly. The slices themselves are the same files --\\\\n    see `submission_indices` -- but the uint8 values differ a little, and the\\\\n    submission notebook measures that difference rather than assuming it away.\\\\n    """\\\\n    if not headers:\\\\n        raise ValueError("no headers supplied")\\\\n\\\\n    depth = config.CACHE_SLICES if slices is None else slices\\\\n    ordered = dicom_io.sort_slices(headers)\\\\n    first_header = ordered[0]\\\\n    side = dicom_io.resolve_laterality(ordered)\\\\n    geometry_ok = dicom_io.has_geometry(ordered)\\\\n\\\\n    indices = submission_indices(len(ordered), depth)\\\\n    selected = [ordered[i] for i in indices]\\\\n\\\\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\\\\n\\\\n    # Slices skipped as unreadable leave the volume short of the requested\\\\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\\\\n    # decoded so the depth is exact regardless of how many files were dropped.\\\\n    if raw.shape[0] != depth:\\\\n        raw = raw[select_slices(raw.shape[0], depth)]\\\\n\\\\n    resized = resize_volume(raw.astype(np.float32))\\\\n    normalized = normalize_intensity(resized)\\\\n\\\\n    # Left as acquired by default; the loader mirrors using the recorded side.\\\\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\\\\n    volume = (canonicalize_side(normalized, side,\\\\n                                first_header.get("ImageOrientationPatient"))\\\\n              if config.MIRROR_IN_CACHE else normalized)\\\\n\\\\n    first = ordered[0]\\\\n    return {\\\\n        "volume": volume,\\\\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\\\\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\\\\n        "n_slices_raw": len(ordered),\\\\n        "laterality": side,\\\\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\\\\n                         and side.upper() != config.CANONICAL_SIDE),\\\\n        "geometry_ok": geometry_ok,\\\\n        "rows": int(first.get("Rows") or 0),\\\\n        "cols": int(first.get("Columns") or 0),\\\\n    }\\\\n\\\\n\\\\ndef build_series_record(item):\\\\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\\\\n\\\\n    Defined at module level deliberately. A worker defined inside a notebook\\\\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\\\\n    spawn start method and survives under fork only by accident. Keeping it\\\\n    importable makes the cache build work the same way everywhere.\\\\n\\\\n    Failures are returned rather than raised: one unreadable series must not\\\\n    end a ninety-minute run, and the manifest records what to investigate.\\\\n    """\\\\n    series_uid, headers = item\\\\n    try:\\\\n        result = build_series_volume(headers)\\\\n        return series_uid, result["volume"], {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": result["StudyInstanceUID"],\\\\n            "n_slices_raw": result["n_slices_raw"],\\\\n            "laterality": result["laterality"],\\\\n            "mirrored": result["mirrored"],\\\\n            "geometry_ok": result["geometry_ok"],\\\\n            "rows": result["rows"],\\\\n            "cols": result["cols"],\\\\n            "error": None,\\\\n        }\\\\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\\\\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\\\\n        return series_uid, None, {\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "StudyInstanceUID": study_uid,\\\\n            "n_slices_raw": len(headers) if headers else 0,\\\\n            "laterality": None,\\\\n            "mirrored": False,\\\\n            "geometry_ok": False,\\\\n            "rows": 0,\\\\n            "cols": 0,\\\\n            "error": str(exc)[:200],\\\\n        }\\\\n\\\', \\\'src/data/index.py\\\': \\\'"""Header-only scan of the DICOM tree into a single parquet index.\\\\n\\\\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\\\\nthe cost is dominated by file-system traversal rather than decoding. The\\\\nresulting index drives cache construction and lets series be filtered and\\\\naudited without touching the 570 GB of imaging again.\\\\n\\\\nUsage (in a Kaggle notebook):\\\\n\\\\n    from src.data.index import build_index\\\\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\\\\n"""\\\\n\\\\nimport argparse\\\\nimport os\\\\nfrom concurrent.futures import ThreadPoolExecutor\\\\nfrom pathlib import Path\\\\nfrom typing import Iterator, List, Optional\\\\n\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.data import dicom_io\\\\n\\\\n\\\\ndef iter_dicom_paths(root) -> Iterator[Path]:\\\\n    """Yield every .dcm under root/study/series/ without materialising a list."""\\\\n    root = Path(root)\\\\n    for study_dir in sorted(root.iterdir()):\\\\n        if not study_dir.is_dir():\\\\n            continue\\\\n        for series_dir in sorted(study_dir.iterdir()):\\\\n            if not series_dir.is_dir():\\\\n                continue\\\\n            for path in sorted(series_dir.glob("*.dcm")):\\\\n                yield path\\\\n\\\\n\\\\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\\\\n    """Read headers for a batch of files in parallel.\\\\n\\\\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\\\\n    released during file reads and a process pool would only add overhead.\\\\n    """\\\\n    def safe_read(path):\\\\n        try:\\\\n            return dicom_io.read_header(path)\\\\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\\\\n            return {"path": str(path), "error": str(exc)}\\\\n\\\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\\\n        rows = list(pool.map(safe_read, paths))\\\\n\\\\n    return pd.DataFrame(rows)\\\\n\\\\n\\\\ndef build_index(\\\\n    split: str = "train_series",\\\\n    out_path: Optional[str] = None,\\\\n    data_root=None,\\\\n    workers: int = 8,\\\\n    batch_size: int = 5000,\\\\n    limit: Optional[int] = None,\\\\n) -> pd.DataFrame:\\\\n    """Scan one split\\\\\\\'s DICOM tree and write a parquet index.\\\\n\\\\n    Processes in batches so a long scan reports progress and so peak memory\\\\n    stays flat regardless of how many files the split holds.\\\\n    """\\\\n    root = Path(data_root or config.DATA_ROOT) / split\\\\n    if not root.exists():\\\\n        raise FileNotFoundError(f"{root} does not exist")\\\\n\\\\n    try:\\\\n        from tqdm.auto import tqdm\\\\n    except ImportError:  # progress is a convenience, never a requirement\\\\n        tqdm = None\\\\n\\\\n    frames = []\\\\n    batch: List[Path] = []\\\\n    scanned = 0\\\\n\\\\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\\\\n    # counts files without a total rather than forcing a full listing first,\\\\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\\\\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\\\\n\\\\n    for path in iter_dicom_paths(root):\\\\n        batch.append(path)\\\\n        if limit is not None and len(batch) + scanned >= limit:\\\\n            break\\\\n        if len(batch) >= batch_size:\\\\n            frames.append(scan_paths(batch, workers=workers))\\\\n            scanned += len(batch)\\\\n            if progress is not None:\\\\n                progress.update(len(batch))\\\\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\\\\n            else:\\\\n                print(f"  scanned {scanned} files", flush=True)\\\\n            batch = []\\\\n\\\\n    if batch:\\\\n        frames.append(scan_paths(batch, workers=workers))\\\\n        scanned += len(batch)\\\\n        if progress is not None:\\\\n            progress.update(len(batch))\\\\n\\\\n    if progress is not None:\\\\n        progress.close()\\\\n\\\\n    if not frames:\\\\n        raise RuntimeError(f"no DICOM files found under {root}")\\\\n\\\\n    index = pd.concat(frames, ignore_index=True)\\\\n    print(f"scanned {scanned} files across {index[\\\\\\\'SeriesInstanceUID\\\\\\\'].nunique()} series")\\\\n\\\\n    if "error" in index.columns:\\\\n        failures = int(index["error"].notna().sum())\\\\n        if failures:\\\\n            print(f"WARNING: {failures} files failed to parse")\\\\n\\\\n    if out_path:\\\\n        os.makedirs(Path(out_path).parent, exist_ok=True)\\\\n        index.to_parquet(out_path, index=False)\\\\n        print(f"wrote {out_path}")\\\\n\\\\n    return index\\\\n\\\\n\\\\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\\\\n    """Per-series roll-up: slice count, geometry completeness, laterality.\\\\n\\\\n    Surfaces the series that need attention -- missing geometry, unresolvable\\\\n    side -- before an hour of cache construction is spent on them.\\\\n    """\\\\n    records = []\\\\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\\\\n        headers = group.to_dict("records")\\\\n        records.append({\\\\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\\\\n            "SeriesInstanceUID": series_uid,\\\\n            "n_slices": len(headers),\\\\n            "geometry_ok": dicom_io.has_geometry(headers),\\\\n            "laterality": dicom_io.resolve_laterality(headers),\\\\n            "rows": headers[0].get("Rows"),\\\\n            "cols": headers[0].get("Columns"),\\\\n        })\\\\n\\\\n    return pd.DataFrame(records)\\\\n\\\\n\\\\ndef main():\\\\n    parser = argparse.ArgumentParser(description=__doc__)\\\\n    parser.add_argument("--split", default="train_series")\\\\n    parser.add_argument("--out", default=None)\\\\n    parser.add_argument("--workers", type=int, default=8)\\\\n    parser.add_argument("--limit", type=int, default=None)\\\\n    args = parser.parse_args()\\\\n\\\\n    build_index(split=args.split, out_path=args.out,\\\\n                workers=args.workers, limit=args.limit)\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/__init__.py\\\': \\\'\\\', \\\'src/model/side_classifier.py\\\': \\\'"""Predict whether a cached series images a left or right knee.\\\\n\\\\nThe DICOM side tag is blank on ~41% of series and the descriptions are\\\\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\\\\ncompartment-specific, and placing them requires knowing the side, so the\\\\nremainder is predicted from the imaging itself.\\\\n\\\\nThis trains on the 14,403 series whose side is known and applies to the rest.\\\\nBoth halves come from the same cache, so it behaves identically at submission\\\\ntime -- unlike report text, which exists only for training studies.\\\\n\\\\n**Why the whole ordered stack rather than one representative slice.** For\\\\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\\\\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\\\\nthe left-right axis IS the slice axis, so individual slices look much the same\\\\neither way -- what differs is the direction of travel through the volume, which\\\\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\\\\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\\\\nchannels, which preserves it.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:  # the module is importable without torch for testing\\\\n    torch = None\\\\n    nn = object\\\\n\\\\nfrom src import config\\\\n\\\\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\\\\n# so half resolution is ample and quadruples throughput.\\\\nINPUT_SIZE = 112\\\\n\\\\nSIDES = ["L", "R"]\\\\n\\\\n\\\\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\\\\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\\\\n\\\\n    Slices become channels, keeping their order, so the network can use the\\\\n    medial-to-lateral direction that distinguishes sides on sagittal series.\\\\n    """\\\\n    import cv2\\\\n\\\\n    if volumes.ndim == 3:\\\\n        volumes = volumes[None]\\\\n\\\\n    count, depth, _, _ = volumes.shape\\\\n    output = np.empty((count, depth, size, size), dtype=np.float32)\\\\n\\\\n    for i in range(count):\\\\n        for j in range(depth):\\\\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\\\\n                                      interpolation=cv2.INTER_AREA)\\\\n\\\\n    return output / 255.0\\\\n\\\\n\\\\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\\\\n    """Mirror a prepared batch along each item\\\\\\\'s medial-lateral axis.\\\\n\\\\n    Used for augmentation: mirroring an input flips its label, which doubles\\\\n    the training set and forces the model to key on anatomy rather than on any\\\\n    incidental left/right imbalance in the source institutions.\\\\n\\\\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\\\\n    in the prepared batch slices are dimension 1 and columns dimension 3.\\\\n    """\\\\n    output = np.empty_like(batch)\\\\n    for i, axis in enumerate(plane_axis):\\\\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\\\\n    return output\\\\n\\\\n\\\\nif torch is None:\\\\n    # Importable without torch so the array handling above can be tested on a\\\\n    # machine that never trains. The network itself needs the real thing.\\\\n    SideNet = None\\\\nelse:\\\\n  class SideNet(nn.Module):\\\\n      """A small convolutional classifier over slice-channel input.\\\\n\\\\n      Trained from scratch deliberately. It is a coarse two-class problem with\\\\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\\\\n      competition\\\\\\\'s stance on pretrained weights is still unconfirmed, which this\\\\n      piece of the pipeline should not depend on.\\\\n      """\\\\n\\\\n      def __init__(self, in_channels: int = None, width: int = 32):\\\\n          super().__init__()\\\\n          in_channels = in_channels or config.CACHE_SLICES\\\\n\\\\n          def block(inp, out, stride=2):\\\\n              return nn.Sequential(\\\\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\\\\n                  nn.BatchNorm2d(out),\\\\n                  nn.ReLU(inplace=True),\\\\n              )\\\\n\\\\n          self.features = nn.Sequential(\\\\n              block(in_channels, width),        # 112 -> 56\\\\n              block(width, width * 2),          # 56 -> 28\\\\n              block(width * 2, width * 4),      # 28 -> 14\\\\n              block(width * 4, width * 4),      # 14 -> 7\\\\n          )\\\\n          self.head = nn.Sequential(\\\\n              nn.AdaptiveAvgPool2d(1),\\\\n              nn.Flatten(),\\\\n              nn.Dropout(0.2),\\\\n              nn.Linear(width * 4, 1),\\\\n          )\\\\n\\\\n      def forward(self, x):\\\\n          return self.head(self.features(x)).squeeze(-1)\\\\n\\\\n\\\\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\\\\n    """Collapse per-series predictions to one side per study.\\\\n\\\\n    A study is one knee, so its series must agree. Voting across them is\\\\n    strictly more reliable than any single series -- and the observed\\\\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\\\\n\\\\n    Returns {study_uid: "L" | "R"}.\\\\n    """\\\\n    if len(predictions) != len(study_uids):\\\\n        raise ValueError(\\\\n            f"predictions and study_uids differ in length: "\\\\n            f"{len(predictions)} vs {len(study_uids)}")\\\\n    if confidences is None:\\\\n        confidences = np.ones(len(predictions))\\\\n    elif len(confidences) != len(predictions):\\\\n        # zip would truncate silently, shifting weights onto the wrong series.\\\\n        raise ValueError(\\\\n            f"confidences length {len(confidences)} does not match "\\\\n            f"{len(predictions)} predictions")\\\\n\\\\n    scores = {}\\\\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\\\\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\\\\n\\\\n    result = {}\\\\n    for study, entries in scores.items():\\\\n        total = sum(weight for _, weight in entries)\\\\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\\\\n        result[study] = "R" if mean >= 0.5 else "L"\\\\n    return result\\\\n\\\\n\\\\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\\\\n    """Fraction correct, with predictions as probabilities of the right knee."""\\\\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\\\\n\\\', \\\'src/model/embed.py\\\': \\\'"""Frozen DINOv2 slice embeddings.\\\\n\\\\nRuns the backbone once over every cached slice and stores the result, so the\\\\naggregation stack trains on cached features in minutes rather than hours. The\\\\nsame code runs at submission time, which is the point -- a divergence between\\\\nhow features are computed for training and for inference is invisible locally\\\\nand fatal on the leaderboard.\\\\n\\\\n**Why frozen, and why first.** The competition scores a second prize track on\\\\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\\\\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\\\\nthe aggregation design -- which is where the modelling actually happens -- be\\\\niterated dozens of times a day instead of three.\\\\n\\\\n**What is stored per slice.** The CLS token concatenated with the patch tokens\\\\nreduced by `POOL_GRID`:\\\\n\\\\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\\\\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\\\\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\\\\n\\\\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\\\\n\\\\nThe first model used grid 1, and its weakness appeared exactly where the mean\\\\npredicts one: regional findings (effusion, OA, Baker\\\\\\\'s) averaged 0.865 on gold\\\\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\\\\ntear occupies two or three patches out of 256, so a mean over all of them\\\\nburies it in 250-odd patches of normal tissue.\\\\n\\\\nWide embeddings must be projected before the aggregation stack -- see\\\\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\\\\nin it against 8.2M with a projection.\\\\n"""\\\\n\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\n\\\\ntry:\\\\n    import torch\\\\n    import torch.nn as nn\\\\nexcept ImportError:\\\\n    torch = None\\\\n    nn = None\\\\n\\\\nfrom src import config\\\\n\\\\n# The registers variant. Registers exist to absorb the high-norm artefact\\\\n# tokens that otherwise contaminate patch features -- precisely what gets\\\\n# pooled here.\\\\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\\\\n\\\\n# timm\\\\\\\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\\\\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\\\\n# position embeddings are interpolated rather than silently mismatched.\\\\nINPUT_SIZE = config.CACHE_SIZE\\\\n\\\\n# DINOv2-S token width.\\\\nTOKEN_DIM = 384\\\\n\\\\n# How the 16x16 patch grid is reduced before storage.\\\\n#\\\\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\\\\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\\\\n#\\\\n# The mean is what the first model used, and its weakness showed up exactly\\\\n# where predicted: regional findings averaged 0.865 on gold while focal ones\\\\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\\\\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\\\\n# attention pooling to find it.\\\\nPOOL_GRID = 1\\\\n\\\\n\\\\ndef embed_dim(grid: int = None) -> int:\\\\n    """Stored width per slice: CLS plus the pooled patch tokens."""\\\\n    grid = POOL_GRID if grid is None else grid\\\\n    tokens = 1 + (grid * grid if grid > 1 else 1)\\\\n    return TOKEN_DIM * tokens\\\\n\\\\n\\\\nEMBED_DIM = embed_dim()\\\\n\\\\n# ImageNet statistics, which DINOv2 was trained under.\\\\nMEAN = (0.485, 0.456, 0.406)\\\\nSTD = (0.229, 0.224, 0.225)\\\\n\\\\n\\\\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\\\\n                  checkpoint_path: Optional[str] = None):\\\\n    """A frozen DINOv2 feature extractor in eval mode.\\\\n\\\\n    `checkpoint_path` loads weights from disk, which the submission notebook\\\\n    needs: internet is disabled there, so timm cannot fetch anything.\\\\n    """\\\\n    import timm\\\\n\\\\n    model = timm.create_model(\\\\n        model_name,\\\\n        pretrained=pretrained and checkpoint_path is None,\\\\n        img_size=INPUT_SIZE,\\\\n        num_classes=0,\\\\n    )\\\\n\\\\n    if checkpoint_path:\\\\n        state = torch.load(checkpoint_path, map_location="cpu")\\\\n        state = state.get("state_dict", state)\\\\n        model.load_state_dict(state, strict=False)\\\\n\\\\n    model.eval()\\\\n    for parameter in model.parameters():\\\\n        parameter.requires_grad = False\\\\n    return model\\\\n\\\\n\\\\ndef normalize(slices):\\\\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\\\\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\\\\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\\\\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\\\\n    return (x - mean) / std\\\\n\\\\n\\\\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\\\\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\\\\n\\\\n    The patch sequence is reshaped back to its square grid before pooling, so\\\\n    neighbouring patches are averaged with their neighbours rather than with\\\\n    whatever happened to sit beside them in a flattened sequence.\\\\n    """\\\\n    import torch.nn.functional as F\\\\n\\\\n    count, n_patches, channels = patches.shape\\\\n    side = int(round(n_patches ** 0.5))\\\\n    if side * side != n_patches:\\\\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\\\\n        return patches.mean(dim=1)\\\\n\\\\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\\\\n    pooled = F.adaptive_avg_pool2d(square, grid)\\\\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\\\\n\\\\n\\\\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\\\\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\\\\n\\\\n    Uses forward_features so the patch tokens are reachable at all; the model\\\\\\\'s\\\\n    default pooled output discards them.\\\\n    """\\\\n    grid = POOL_GRID if grid is None else grid\\\\n\\\\n    with torch.no_grad():\\\\n        tokens = encoder.forward_features(normalize(slices))\\\\n\\\\n        if tokens.ndim == 2:\\\\n            # A backbone that already pools; repeat so the width is predictable.\\\\n            repeats = embed_dim(grid) // tokens.shape[-1]\\\\n            return tokens.repeat(1, repeats)\\\\n\\\\n        cls = tokens[:, 0]\\\\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\\\\n        patches = tokens[:, prefix:]\\\\n\\\\n        if grid > 1:\\\\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\\\\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\\\\n\\\\n\\\\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\\\\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\\\\n\\\\n    KneeNet expects a module mapping slices to features. When features are\\\\n    already computed, the "backbone" is an identity that reshapes -- which\\\\n    keeps one model definition for both the cached and the end-to-end path.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim: int = EMBED_DIM):\\\\n        super().__init__()\\\\n        self.feature_dim = feature_dim\\\\n\\\\n    def forward(self, x):\\\\n        return x\\\\n\\\', \\\'src/model/dataset.py\\\': \\\'"""Study-level dataset over the cached volumes.\\\\n\\\\nJoins four sources into one training sample per study:\\\\n\\\\n    cache shards      the uint8 volumes, memory-mapped\\\\n    manifest          which shard row holds which series\\\\n    fused labels      12 soft targets and 12 sample weights per study\\\\n    side predictions  L/R per series, from the tag or the classifier\\\\n\\\\nA sample is a whole study, not a series. The twelve findings are study-level\\\\nlabels, and a single series often cannot support them -- effusion reads off a\\\\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\\\\nhas to see the study\\\\\\\'s series together.\\\\n\\\\n**Laterality is handled two ways, depending on the path.**\\\\n\\\\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\\\\nside at load time, along the axis the acquisition plane dictates: slice order\\\\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\\\\nand axial, where it lies in-plane. Doing it at load time rather than in the\\\\ncache means side detection can improve without rebuilding 29 GB.\\\\n\\\\nFor the frozen path this is not available -- the embeddings are computed from\\\\nthe volume, so the image can no longer be flipped. There the side travels to\\\\nthe model as a covariate (`is_right`, `side_known`) instead. That still\\\\naddresses the original concern, which was that the model should not have to\\\\n*infer* the side from cues that are not reliably present; it is handed the\\\\nanswer either way. Which of the two performs better is an experiment, not a\\\\nsettled question.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\\\\nPLANE_MIRROR_AXIS = {\\\\n    "Sagittal": 0,   # medial-lateral is the slice axis\\\\n    "Coronal": 2,    # medial-lateral lies in-plane\\\\n    "Axial": 2,\\\\n}\\\\nDEFAULT_MIRROR_AXIS = 2\\\\n\\\\n# Series metadata fed to the model as conditioning. Findings are\\\\n# sequence-specific, and these three are given in both train and test.\\\\n# `is_right` and `side_known` carry laterality as a covariate rather than\\\\n# normalising it away by mirroring. With precomputed frozen embeddings the\\\\n# image can no longer be flipped -- the features are already computed -- so the\\\\n# side is handed to the model instead, which addresses the original concern\\\\n# (the model should not have to *infer* the side) by a different route.\\\\n# Mirroring remains available for the end-to-end path and is an A/B to run,\\\\n# not a settled question.\\\\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\\\\n                "is_coronal", "is_axial", "is_right", "side_known"]\\\\n\\\\n\\\\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\\\\n                  plane: Optional[str]) -> np.ndarray:\\\\n    """Reflect a left knee onto the canonical side, along the correct axis."""\\\\n    if not isinstance(side, str) or side.strip().upper() not in ("L", "R"):\\\\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\\\\n        # on four of the twelve findings.\\\\n        return volume\\\\n    if side.strip().upper() == config.CANONICAL_SIDE:\\\\n        return volume\\\\n\\\\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef series_metadata(row: pd.Series) -> np.ndarray:\\\\n    """Conditioning vector for one series.\\\\n\\\\n    Fluid sensitivity and fat suppression are kept separate: the data\\\\n    description notes they are often correlated but not equivalent, so\\\\n    collapsing them would discard the cases where they disagree.\\\\n    """\\\\n    plane = str(row.get("Anatomical_Plane", ""))\\\\n    side = row.get("side_predicted")\\\\n    side = side.strip().upper() if isinstance(side, str) else None\\\\n    known = side in ("L", "R")\\\\n    return np.array([\\\\n        sequence_flag(row.get("Fluid_Sensitive")),\\\\n        sequence_flag(row.get("Fat_Suppression")),\\\\n        float(plane == "Sagittal"),\\\\n        float(plane == "Coronal"),\\\\n        float(plane == "Axial"),\\\\n        float(known and side == "R"),\\\\n        float(known),\\\\n    ], dtype=np.float32)\\\\n\\\\n\\\\ndef sequence_flag(value) -> float:\\\\n    """Parse CSV flags without treating NaN or the string \\\\\\\'False\\\\\\\' as true."""\\\\n    if value is None or pd.isna(value):\\\\n        return 0.0\\\\n    if isinstance(value, str):\\\\n        value = value.strip().lower()\\\\n        if value in ("true", "yes", "1", "1.0"):\\\\n            return 1.0\\\\n        if value in ("false", "no", "0", "0.0", ""):\\\\n            return 0.0\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    if value not in (0, 1):\\\\n        raise ValueError(f"invalid sequence flag: {value!r}")\\\\n    return float(value)\\\\n\\\\n\\\\ndef select_series(rows: pd.DataFrame, maximum=None, strategy="uniform", rng=None):\\\\n    """Positions to retain, preserving order and optionally protocol coverage.\\\\n\\\\n    Protocol selection first covers acquisition planes, then distinct\\\\n    (plane, fluid sensitivity, fat suppression) tuples, then repeats. It\\\\n    uses only metadata available at inference, never labels or reports.\\\\n    """\\\\n    if strategy not in ("uniform", "protocol"):\\\\n        raise ValueError(f"unknown series selection: {strategy}")\\\\n    if maximum is not None and maximum < 1:\\\\n        raise ValueError("max_series must be positive or None")\\\\n    count = len(rows)\\\\n    if maximum is None or count <= maximum:\\\\n        return np.arange(count)\\\\n    if strategy == "uniform":\\\\n        chosen = (np.linspace(0, count - 1, maximum).round().astype(int)\\\\n                  if rng is None else rng.choice(count, maximum, replace=False))\\\\n        return np.sort(chosen)\\\\n    protocols = [(str(row.get("Anatomical_Plane", "")),\\\\n                  sequence_flag(row.get("Fluid_Sensitive")),\\\\n                  sequence_flag(row.get("Fat_Suppression")))\\\\n                 for _, row in rows.iterrows()]\\\\n    planes, covered, chosen = set(), set(), []\\\\n    priority = (np.arange(count) if rng is None else rng.permutation(count))\\\\n    while len(chosen) < maximum:\\\\n        remaining = [int(i) for i in priority if i not in chosen]\\\\n        best = max(remaining, key=lambda i: (\\\\n            protocols[i][0] in PLANE_MIRROR_AXIS and protocols[i][0] not in planes,\\\\n            protocols[i] not in covered,\\\\n            min((abs(i - j) for j in chosen), default=0)))\\\\n        chosen.append(best)\\\\n        planes.add(protocols[best][0])\\\\n        covered.add(protocols[best])\\\\n    return np.array(sorted(chosen))\\\\n\\\\n\\\\nclass StudyIndex:\\\\n    """Everything needed to assemble a study, resolved once up front.\\\\n\\\\n    Built from frames rather than files so it can be constructed in a test\\\\n    without touching disk.\\\\n    """\\\\n\\\\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\\\\n                 labels: Optional[pd.DataFrame] = None,\\\\n                 sides: Optional[pd.DataFrame] = None,\\\\n                 max_series: Optional[int] = None,\\\\n                 reports: Optional[pd.DataFrame] = None,\\\\n                 series_selection: str = "uniform"):\\\\n        frame = manifest.merge(\\\\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\\\\n                         "Fat_Suppression", "Anatomical_Plane"]],\\\\n            on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n\\\\n        if sides is not None:\\\\n            frame = frame.merge(\\\\n                sides[["SeriesInstanceUID", "side_predicted"]],\\\\n                on="SeriesInstanceUID", how="left", validate="one_to_one")\\\\n        else:\\\\n            frame["side_predicted"] = frame.get("laterality_resolved")\\\\n\\\\n        # Drop series whose volume failed to build; their row in the cache is\\\\n        # zeros and would train the model on blank images.\\\\n        if "error" in frame.columns:\\\\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\\\\n\\\\n        if series_selection == "protocol":\\\\n            frame = frame.sort_values("SeriesInstanceUID", kind="stable").reset_index(drop=True)\\\\n        self.series = frame\\\\n        self.max_series = max_series\\\\n        self.series_selection = series_selection\\\\n        select_series(frame.iloc[:0], max_series, series_selection)\\\\n        self.labels = labels\\\\n\\\\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\\\\n        self._by_study = {\\\\n            study: group.reset_index(drop=True)\\\\n            for study, group in frame.groupby(config.ID_COL)\\\\n        }\\\\n\\\\n        if labels is not None:\\\\n            if labels[config.ID_COL].duplicated().any():\\\\n                raise ValueError("duplicate study IDs in labels")\\\\n            keyed = labels.set_index(config.ID_COL)\\\\n            self.studies = [s for s in self.studies if s in keyed.index]\\\\n            self._labels = keyed\\\\n        else:\\\\n            self._labels = None\\\\n\\\\n        # Report sentence embeddings, a training-time-only signal: reports\\\\n        # exist for train studies and never for test, so these feed an\\\\n        # auxiliary loss rather than the model input. Stored as one float32\\\\n        # matrix rather than a DataFrame so __getitem__ is an array row.\\\\n        if reports is not None:\\\\n            keyed_reports = reports.set_index(config.ID_COL)\\\\n            self._report_rows = {study: i for i, study\\\\n                                 in enumerate(keyed_reports.index)}\\\\n            self._report_matrix = keyed_reports.to_numpy(dtype=np.float32)\\\\n            self.report_dim = self._report_matrix.shape[1]\\\\n        else:\\\\n            self._report_rows = None\\\\n            self._report_matrix = None\\\\n            self.report_dim = None\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def series_for(self, study: str, rng=None) -> pd.DataFrame:\\\\n        """Series of a study, capped deterministically when a cap is set.\\\\n\\\\n        The cap exists for the efficiency track: inference cost scales with\\\\n        series count, and a study with a dozen series contributes far more\\\\n        runtime than one with three. Selection is by position rather than at\\\\n        random so a prediction is reproducible.\\\\n\\\\n        `rng` is the training-only exception. The deterministic cap discards\\\\n        the same 3,013 of 24,371 series (12.4%) on every epoch of every fold --\\\\n        1,432 studies carry more series than `max_series` and always show the\\\\n        model the same subset. Passing a generator resamples the cap per epoch,\\\\n        which reaches those series without adding a byte to the cache. Order is\\\\n        preserved, so the series-axis metadata still lines up. Never pass one\\\\n        for validation or inference: a prediction that changes between two\\\\n        identical runs cannot be compared against anything.\\\\n        """\\\\n        rows = self._by_study[study]\\\\n        keep = select_series(rows, self.max_series, self.series_selection, rng)\\\\n        return rows.iloc[keep].reset_index(drop=True)\\\\n\\\\n    def targets_for(self, study: str):\\\\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\\\\n        if self._labels is None or study not in self._labels.index:\\\\n            return None, None\\\\n        row = self._labels.loc[study]\\\\n        targets = np.array([float(row[name]) for name in config.LABELS],\\\\n                           dtype=np.float32)\\\\n        weights = np.array(\\\\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\\\\n            dtype=np.float32)\\\\n        return targets, weights\\\\n\\\\n    def report_for(self, study: str) -> Optional[np.ndarray]:\\\\n        """The study\\\\\\\'s report embedding, or None when unavailable."""\\\\n        if self._report_rows is None or study not in self._report_rows:\\\\n            return None\\\\n        return self._report_matrix[self._report_rows[study]]\\\\n\\\\n\\\\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray],\\\\n               series_rng=None) -> Dict:\\\\n    """Assemble one study: mirrored volumes, conditioning, targets, weights.\\\\n\\\\n    `series_rng` is forwarded to `series_for` and is training-only; see there.\\\\n    """\\\\n    rows = index.series_for(study, rng=series_rng)\\\\n\\\\n    volumes, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        volume = np.asarray(shard[int(row["position"])])\\\\n        volume = mirror_volume(volume, row.get("side_predicted"),\\\\n                               row.get("Anatomical_Plane"))\\\\n        volumes.append(volume)\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(volumes) if volumes else np.zeros(\\\\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\\\\n            dtype=np.uint8),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef load_study_features(index: StudyIndex, study: str,\\\\n                        shards: Dict[str, np.ndarray]) -> Dict:\\\\n    """Assemble one study from precomputed slice embeddings.\\\\n\\\\n    Mirroring is unavailable here -- the features were computed from the image\\\\n    and cannot be flipped after the fact -- so laterality reaches the model\\\\n    through `series_metadata` instead.\\\\n    """\\\\n    rows = index.series_for(study)\\\\n\\\\n    features, meta = [], []\\\\n    for _, row in rows.iterrows():\\\\n        shard = shards[row["cache_path"]]\\\\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\\\\n        meta.append(series_metadata(row))\\\\n\\\\n    targets, weights = index.targets_for(study)\\\\n    return {\\\\n        config.ID_COL: study,\\\\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\\\\n                                                                dtype=np.float32),\\\\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\\\\n                                                     dtype=np.float32),\\\\n        "targets": targets,\\\\n        "weights": weights,\\\\n        "report": index.report_for(study),\\\\n    }\\\\n\\\\n\\\\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\\\\n    """Pad a batch of studies to the largest series count, with a mask.\\\\n\\\\n    Series counts vary from one to a dozen, so the batch is padded and masked\\\\n    rather than truncated. The mask must reach the attention pooling: without\\\\n    it, padding rows are averaged in as if they were real series.\\\\n    """\\\\n    import torch\\\\n\\\\n    longest = max(len(item["volumes"]) for item in batch)\\\\n\\\\n    # Per-series shape is taken from the data rather than assumed, so the same\\\\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\\\\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\\\\n                     None)\\\\n    if reference is None:\\\\n        raise ValueError("every study in the batch is empty")\\\\n    per_series = reference.shape[1:]\\\\n    dtype = reference.dtype\\\\n\\\\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\\\\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\\\\n    mask = np.zeros((len(batch), longest), dtype=bool)\\\\n\\\\n    for i, item in enumerate(batch):\\\\n        count = len(item["volumes"])\\\\n        if count:\\\\n            volumes[i, :count] = item["volumes"]\\\\n            meta[i, :count] = item["meta"]\\\\n            mask[i, :count] = True\\\\n\\\\n    out = {\\\\n        config.ID_COL: [item[config.ID_COL] for item in batch],\\\\n        "volumes": torch.from_numpy(volumes),\\\\n        "meta": torch.from_numpy(meta),\\\\n        "mask": torch.from_numpy(mask),\\\\n    }\\\\n\\\\n    if batch[0]["targets"] is not None:\\\\n        out["targets"] = torch.from_numpy(\\\\n            np.stack([item["targets"] for item in batch]))\\\\n        out["weights"] = torch.from_numpy(\\\\n            np.stack([item["weights"] for item in batch]))\\\\n\\\\n    # Report embeddings, present only where the index carries them. Studies\\\\n    # without one get a zero row and a False mask so the auxiliary loss can\\\\n    # exclude them without a ragged batch.\\\\n    reports = [item.get("report") for item in batch]\\\\n    dims = [r.shape[0] for r in reports if r is not None]\\\\n    if dims:\\\\n        report = np.zeros((len(batch), dims[0]), dtype=np.float32)\\\\n        report_mask = np.zeros(len(batch), dtype=bool)\\\\n        for i, row in enumerate(reports):\\\\n            if row is not None:\\\\n                report[i] = row\\\\n                report_mask[i] = True\\\\n        out["report"] = torch.from_numpy(report)\\\\n        out["report_mask"] = torch.from_numpy(report_mask)\\\\n    return out\\\\n\\\', \\\'src/model/net.py\\\': \\\'"""Study-level model: slices to series to study to twelve logits.\\\\n\\\\nThe backbone is deliberately the least interesting part. It is swappable, and\\\\nwhich one wins is settled by measurement rather than by argument -- at 4,349\\\\nstudies with mined labels, the aggregation above it matters more.\\\\n\\\\n    (B, S, D, H, W) uint8 volumes\\\\n      -> backbone over slices          (B*S*D, F)\\\\n      -> attention pool over slices    (B*S, F)\\\\n      -> FiLM on sequence metadata     (B*S, F)\\\\n      -> transformer over series       (B, F)      masked\\\\n      -> head                          (B, 12)\\\\n\\\\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\\\\nand Baker\\\\\\\'s cysts read off fluid-sensitive series, cartilage loss off\\\\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\\\\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\\\\nevery series through one undifferentiated path throws away free signal.\\\\n\\\\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\\\\nrows must be excluded from both the attention pool and the transformer, or a\\\\nshort study is diluted by rows of zeros that the model reads as real series.\\\\n"""\\\\n\\\\nfrom typing import Optional, Sequence\\\\n\\\\nimport torch\\\\nimport torch.nn as nn\\\\nimport torch.nn.functional as F\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import META_COLUMNS\\\\n\\\\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\\\\n# backbone expects. Gives each slice a little through-plane context for free.\\\\nSLICE_CONTEXT = 3\\\\n\\\\n\\\\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\\\\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\\\\n\\\\n    Edge slices repeat, which is preferable to zero padding: a black plane at\\\\n    the edge of a volume is a feature the model would learn to key on.\\\\n    """\\\\n    n, depth, height, width = volumes.shape\\\\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\\\\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\\\\n    stacked = torch.stack([previous, volumes, following], dim=2)\\\\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\\\\n\\\\n\\\\n# The statistics DINOv2 was pretrained under. `embed.py` has always applied\\\\n# these on the frozen path; the end-to-end path fed raw [0,1] and ConvNeXt\\\\n# absorbed the shift into its first layer over twelve epochs. A pre-norm ViT\\\\n# is far less forgiving, so the same trunk was being fed a distribution it\\\\n# never saw during pretraining.\\\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\\\n\\\\n# NORMALISATION_BUFFERS -- why input_mean/input_std are persistent.\\\\n#\\\\n# They were registered with persistent=False on the argument that they are\\\\n# constants of the backbone family rather than learned state. That was true\\\\n# and it was still wrong: absent from the state dict, they have to be supplied\\\\n# again at every construction site, a site that forgets raises nothing, and a\\\\n# strict load restores every weight while leaving normalisation off. Two\\\\n# scoring sites forgot. The 24-slice DINOv2 run logged gold 0.864 from the\\\\n# in-memory model and 0.823 from the same weights reloaded -- an\\\\n# ImageNet-normalised trunk scored on raw [0,1].\\\\n#\\\\n# So they travel now, and `KneeNet._load_from_state_dict` handles both\\\\n# directions: a checkpoint that carries them imposes them on whatever the\\\\n# caller built, and one written before this change loads strictly without\\\\n# them. ConvNeXt is unaffected either way -- its defaults are None, so its\\\\n# state dict gains no keys and every existing reference number stands.\\\\n\\\\n# How far below the head\\\\\\\'s learning rate a pretrained trunk should sit. The\\\\n# optimiser puts every parameter in one group at OneCycle\\\\\\\'s peak of\\\\n# `lr * max_lr_multiplier` = 3e-4, which a supervised ConvNeXt survives and a\\\\n# self-supervised ViT does not: it loses its pretrained features during warmup\\\\n# and then relearns them from 4,349 studies, which is the one regime ViTs are\\\\n# worst in. Measured symptom: gold 0.6 at epoch 4 where ConvNeXt reached\\\\n# 0.729, and *below* the same trunk frozen (0.770).\\\\nVIT_BACKBONE_LR_SCALE = 0.1\\\\n\\\\n\\\\ndef backbone_defaults(name: str) -> dict:\\\\n    """Input normalisation and trunk learning-rate scale for a backbone family.\\\\n\\\\n    Returns ConvNeXt\\\\\\\'s existing behaviour unchanged -- no normalisation, one\\\\n    learning rate -- so the 0.861 fold-1 reference stays comparable. That\\\\n    includes ConvNeXt-V2, whose FCMAE weights are a drop-in for V1 at the same\\\\n    `timm` interface, and DINOv3\\\\\\\'s ConvNeXt variants: keeping the convnet path\\\\n    identical is what makes a backbone swap a one-variable experiment.\\\\n\\\\n    ViT-family trunks are the exception on both axes, and both were measured\\\\n    rather than argued -- see VIT_BACKBONE_LR_SCALE.\\\\n    """\\\\n    lowered = name.lower()\\\\n    is_vit = "vit" in lowered or ("dinov" in lowered and "convnext" not in lowered)\\\\n    if is_vit:\\\\n        return {"input_mean": IMAGENET_MEAN, "input_std": IMAGENET_STD,\\\\n                "backbone_lr_scale": VIT_BACKBONE_LR_SCALE}\\\\n    return {"input_mean": None, "input_std": None, "backbone_lr_scale": 1.0}\\\\n\\\\n\\\\ndef build_backbone(name: str, pretrained: bool = True,\\\\n                   image_size: Optional[int] = None):\\\\n    """Create a timm trunk mapping (N, 3, H, W) -> (N, num_features).\\\\n\\\\n    Every backbone this project runs is constructed through here so the\\\\n    training and submission notebooks cannot build different trunks from the\\\\n    same name -- notebook 10 hard-coded ConvNeXt for a while, which would have\\\\n    met a DINOv2 checkpoint with a shape error at submission time.\\\\n\\\\n    ViT-family weights (DINOv2 among them) are published at a fixed input size\\\\n    and timm\\\\\\\'s patch embedding refuses anything else, so the cache\\\\\\\'s size is\\\\n    passed explicitly and the position embeddings are interpolated -- 224 is\\\\n    16x14, which divides cleanly. ConvNets accept any size and reject the\\\\n    argument, hence the fallback rather than an unconditional pass.\\\\n    """\\\\n    import timm\\\\n\\\\n    image_size = config.CACHE_SIZE if image_size is None else image_size\\\\n    kwargs = dict(pretrained=pretrained, num_classes=0,\\\\n                  in_chans=SLICE_CONTEXT)\\\\n    try:\\\\n        return timm.create_model(name, img_size=image_size, **kwargs)\\\\n    except TypeError:\\\\n        return timm.create_model(name, **kwargs)\\\\n\\\\n\\\\nclass AttentionPool(nn.Module):\\\\n    """Masked attention pooling over a sequence of embeddings.\\\\n\\\\n    Preferred to mean pooling because the informative slices are a minority --\\\\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\\\\n    dilutes it by the rest. This is the attention-based MIL pooling of Ilse et\\\\n    al., "Attention-based Deep Multiple Instance Learning" (ICML 2018,\\\\n    arXiv:1802.04712), equation 8: a trainable weighted average, permutation\\\\n    invariant, with the weights produced by a two-layer scorer.\\\\n\\\\n    `gated=True` selects the same paper\\\\\\\'s equation 9, which replaces the single\\\\n    tanh branch with `tanh(Vh) * sigmoid(Uh)`. The stated motivation is that\\\\n    tanh is close to linear over most of its range, so it cannot express the\\\\n    non-linear "how relevant is this instance" function well; the sigmoid\\\\n    branch supplies a multiplicative gate that can suppress an instance\\\\n    outright. It is a strictly larger family -- a gate saturated at 1\\\\n    reproduces the ungated pool -- and it costs one extra `dim x hidden`\\\\n    matrix per pool.\\\\n\\\\n    The two variants produce different state dict keys on purpose. A gated\\\\n    checkpoint loaded into an ungated model raises on the missing keys rather\\\\n    than scoring quietly through the wrong pooling, which is the failure mode\\\\n    every other geometry mismatch in this project has had.\\\\n    """\\\\n\\\\n    def __init__(self, dim: int, hidden: int = 128, gated: bool = False):\\\\n        super().__init__()\\\\n        self.gated = gated\\\\n        if gated:\\\\n            self.attend = nn.Linear(dim, hidden)\\\\n            self.gate = nn.Linear(dim, hidden)\\\\n            self.weigh = nn.Linear(hidden, 1)\\\\n        else:\\\\n            self.score = nn.Sequential(\\\\n                nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\\\\n\\\\n    def scores(self, x: torch.Tensor) -> torch.Tensor:\\\\n        """(B, N, F) -> (B, N) unnormalised attention logits."""\\\\n        if not self.gated:\\\\n            return self.score(x).squeeze(-1)\\\\n        return self.weigh(torch.tanh(self.attend(x))\\\\n                          * torch.sigmoid(self.gate(x))).squeeze(-1)\\\\n\\\\n    def forward(self, x: torch.Tensor,\\\\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\\\\n        # x: (B, N, F), mask: (B, N) with True for real entries.\\\\n        weights = self.scores(x)\\\\n        if mask is not None:\\\\n            weights = weights.masked_fill(~mask, float("-inf"))\\\\n            weights = torch.where(mask.any(dim=1, keepdim=True), weights,\\\\n                                  torch.zeros_like(weights))\\\\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return (x * weights).sum(dim=1)\\\\n\\\\n\\\\nclass ClassAttentionPool(nn.Module):\\\\n    """Separate gated series attention for each finding, sharing the features."""\\\\n\\\\n    def __init__(self, dim: int, n_labels: int, hidden: int = 128):\\\\n        super().__init__()\\\\n        self.attend = nn.Linear(dim, hidden)\\\\n        self.gate = nn.Linear(dim, hidden)\\\\n        self.weigh = nn.Linear(hidden, n_labels)\\\\n\\\\n    def forward(self, x, mask=None):\\\\n        scores = self.weigh(torch.tanh(self.attend(x)) * torch.sigmoid(self.gate(x)))\\\\n        if mask is not None:\\\\n            scores = scores.masked_fill(~mask.unsqueeze(-1), float("-inf"))\\\\n            scores = torch.where(mask.any(dim=1)[:, None, None], scores,\\\\n                                 torch.zeros_like(scores))\\\\n        weights = scores.softmax(dim=1)\\\\n        if mask is not None:\\\\n            weights = weights * mask.unsqueeze(-1)\\\\n        return torch.einsum("bsl,bsf->blf", weights, x)\\\\n\\\\n\\\\ndef aggregation_defaults(state: dict) -> dict:\\\\n    """Recover the pooling variant from weights at every scoring site."""\\\\n    return {\\\\n        "slice_transformer": "slice_encoder.project.weight" in state,\\\\n        "gated_pool": "slice_pool.gate.weight" in state,\\\\n        "classwise_pool": ("study_pool.weigh.weight" in state and\\\\n                           state["study_pool.weigh.weight"].shape[0] > 1),\\\\n    }\\\\n\\\\n\\\\nclass FiLM(nn.Module):\\\\n    """Feature-wise modulation of series embeddings by sequence metadata."""\\\\n\\\\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\\\\n        super().__init__()\\\\n        self.to_scale_shift = nn.Sequential(\\\\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\\\\n            nn.Linear(dim, dim * 2))\\\\n\\\\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\\\\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\\\\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\\\\n        return x * (1.0 + scale) + shift\\\\n\\\\n\\\\nclass SliceSequenceEncoder(nn.Module):\\\\n    """Contextualize sampled slices before pooling; positions are relative, not mm.\\\\n\\\\n    A narrow residual branch preserves the existing feature interface. Its\\\\n    output starts at zero, so the new model initially reproduces the control.\\\\n    """\\\\n\\\\n    def __init__(self, feature_dim, width=256, layers=2, dropout=0.2):\\\\n        super().__init__()\\\\n        self.norm = nn.LayerNorm(feature_dim)\\\\n        self.project = nn.Linear(feature_dim, width)\\\\n        self.position = nn.Linear(1, width)\\\\n        # Construct layers independently (TransformerEncoder otherwise clones\\\\n        # one layer with identical initial values).\\\\n        self.layers = nn.ModuleList([\\\\n            nn.TransformerEncoderLayer(width, 8, width * 2, dropout,\\\\n                                       batch_first=True, norm_first=True)\\\\n            for _ in range(layers)])\\\\n        self.output = nn.Linear(width, feature_dim)\\\\n        nn.init.zeros_(self.output.weight)\\\\n        nn.init.zeros_(self.output.bias)\\\\n\\\\n    def forward(self, features):\\\\n        positions = torch.linspace(-1, 1, features.shape[1],\\\\n                                   device=features.device, dtype=features.dtype)\\\\n        x = self.project(self.norm(features)) + self.position(positions[:, None])[None]\\\\n        for layer in self.layers:\\\\n            x = layer(x)\\\\n        return features + self.output(x)\\\\n\\\\n\\\\nclass KneeNet(nn.Module):\\\\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\\\\n\\\\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\\\\n    a frozen encoder is supported and is the intended first configuration: with\\\\n    the trunk frozen, slice embeddings are computed once and the stack above\\\\n    trains in minutes rather than hours.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\\\\n                 n_labels: int = None, series_layers: int = 2,\\\\n                 n_heads: int = 8, dropout: float = 0.2,\\\\n                 freeze_backbone: bool = False,\\\\n                 input_dim: Optional[int] = None,\\\\n                 input_mean=None, input_std=None,\\\\n                 gated_pool: bool = False,\\\\n                 residual_pairs=None, residual_beta=None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        # None is valid on the frozen path, where the trunk ran offline.\\\\n        self.backbone = backbone if backbone is not None else nn.Identity()\\\\n        self.feature_dim = feature_dim\\\\n        self.classwise_pool = classwise_pool\\\\n        n_labels = n_labels or len(config.LABELS)\\\\n\\\\n        if freeze_backbone:\\\\n            for parameter in self.backbone.parameters():\\\\n                parameter.requires_grad = False\\\\n\\\\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\\\\n        # running the whole stack at that width would put ~85M parameters in\\\\n        # FiLM alone. A projection keeps the aggregation at a sane size.\\\\n        self.project = (nn.Linear(input_dim, feature_dim)\\\\n                        if input_dim and input_dim != feature_dim\\\\n                        else nn.Identity())\\\\n\\\\n        # These now travel in the state dict, and `_load_from_state_dict`\\\\n        # below reconciles the two directions with older files. See\\\\n        # NORMALISATION_BUFFERS for why the earlier persistent=False cost a\\\\n        # measured run.\\\\n        #\\\\n        # Registered as None rather than assigned as a plain attribute: a\\\\n        # plain attribute of the same name makes a later register_buffer\\\\n        # raise "attribute already exists", and adopting a checkpoint\\\\\\\'s\\\\n        # statistics is exactly that late registration.\\\\n        if input_mean is not None:\\\\n            self.register_buffer(\\\\n                "input_mean", torch.tensor(input_mean).view(1, -1, 1, 1))\\\\n            self.register_buffer(\\\\n                "input_std", torch.tensor(input_std).view(1, -1, 1, 1))\\\\n        else:\\\\n            self.register_buffer("input_mean", None)\\\\n            self.register_buffer("input_std", None)\\\\n\\\\n        self.slice_pool = AttentionPool(feature_dim, gated=gated_pool)\\\\n        self.film = FiLM(feature_dim)\\\\n\\\\n        layer = nn.TransformerEncoderLayer(\\\\n            d_model=feature_dim, nhead=n_heads,\\\\n            dim_feedforward=feature_dim * 2, dropout=dropout,\\\\n            batch_first=True, norm_first=True)\\\\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\\\\n        self.study_pool = (ClassAttentionPool(feature_dim, n_labels)\\\\n                           if classwise_pool else AttentionPool(feature_dim, gated=gated_pool))\\\\n\\\\n        self.head = nn.Sequential(\\\\n            nn.LayerNorm(feature_dim),\\\\n            nn.Dropout(dropout),\\\\n            nn.Linear(feature_dim, n_labels))\\\\n\\\\n        # Residual parameterisation: for each (child, parent) pair,\\\\n        #\\\\n        #     child_logit = beta * stopgrad(parent_logit) + g(features)\\\\n        #\\\\n        # with g the head\\\\\\\'s own child column. The plan\\\\\\\'s Stage 1: the\\\\n        # synovitis head reconstructs from the effusion head at R^2 0.55 with\\\\n        # a coefficient of +0.843, so effusion is most of what it computes.\\\\n        # Handing it that term for free, detached, leaves g nothing to learn\\\\n        # except the part effusion does not explain -- which is the part the\\\\n        # conditional metric measures.\\\\n        #\\\\n        # `beta` is calibrated on out-of-fold predictions, not derived from\\\\n        # r_true = 0.403: a rank correlation between two binary labels does\\\\n        # not map onto a logit coefficient. See\\\\n        # `src.model.conditional.calibrate_coupling`.\\\\n        #\\\\n        # Registered as None when unconfigured, exactly like the\\\\n        # normalisation buffers above and for the same reason -- a run that\\\\n        # does not use this writes an unchanged state dict, and a run that\\\\n        # does carries the coupling into the file so a bare\\\\n        # `KneeNet(backbone, dim)` at a scoring site reproduces it instead of\\\\n        # silently scoring an uncoupled model.\\\\n        if residual_pairs:\\\\n            pairs = torch.tensor(list(residual_pairs), dtype=torch.long)\\\\n            if pairs.ndim != 2 or pairs.shape[1] != 2:\\\\n                raise ValueError(\\\\n                    "residual_pairs must be (child, parent) index pairs, got "\\\\n                    f"shape {tuple(pairs.shape)}")\\\\n            if residual_beta is None:\\\\n                raise ValueError("residual_pairs given without residual_beta")\\\\n            values = (list(residual_beta) if hasattr(residual_beta, "__len__")\\\\n                      else [float(residual_beta)] * len(pairs))\\\\n            beta = torch.tensor(values, dtype=torch.float32)\\\\n            if len(beta) != len(pairs):\\\\n                raise ValueError(\\\\n                    f"residual_beta has {len(beta)} values for "\\\\n                    f"{len(pairs)} pairs")\\\\n            self.register_buffer("residual_pairs", pairs)\\\\n            self.register_buffer("residual_beta", beta)\\\\n        else:\\\\n            self.register_buffer("residual_pairs", None)\\\\n            self.register_buffer("residual_beta", None)\\\\n\\\\n        # Add after all existing modules and restore CPU RNG so unchanged\\\\n        # modules (including the Lightning wrapper) initialize identically.\\\\n        with torch.random.fork_rng(devices=[]):\\\\n            self.slice_encoder = (SliceSequenceEncoder(feature_dim, dropout=dropout)\\\\n                                  if slice_transformer else nn.Identity())\\\\n\\\\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\\\\n                              missing_keys, unexpected_keys, error_msgs):\\\\n        """Reconcile the normalisation buffers with checkpoints either side.\\\\n\\\\n        Three cases, and the middle one is the bug this exists for:\\\\n\\\\n        - **The file carries the statistics and this model has none.** Adopt\\\\n          them. A bare `KneeNet(backbone, dim)` built at a scoring site now\\\\n          normalises the way the run that wrote the file did, instead of\\\\n          feeding an ImageNet-normalised trunk raw [0,1] and reading as a\\\\n          worse checkpoint.\\\\n        - **The file predates this change and this model has statistics.**\\\\n          Keep what the caller passed. Every checkpoint written before\\\\n          2026-08-28 is missing the two keys, and a strict load must not\\\\n          start failing on files this project already shipped.\\\\n        - **The file carries statistics and this model has different ones.**\\\\n          The file wins, for the same reason as the first case: the weights\\\\n          and the statistics they were trained under belong together, and the\\\\n          construction site is the thing that has been wrong before.\\\\n\\\\n        Adopting is a `register_buffer` on a slot registered as None in\\\\n        `__init__`, which is why it is registered there rather than assigned.\\\\n        """\\\\n        for name in ("input_mean", "input_std",\\\\n                     "residual_pairs", "residual_beta"):\\\\n            key = prefix + name\\\\n            if key in state_dict:\\\\n                self.register_buffer(name, state_dict[key].clone())\\\\n            elif self._buffers.get(name) is not None:\\\\n                # Pre-fill so the strict load below sees no missing key.\\\\n                state_dict[key] = self._buffers[name]\\\\n        super()._load_from_state_dict(\\\\n            state_dict, prefix, local_metadata, strict, missing_keys,\\\\n            unexpected_keys, error_msgs)\\\\n\\\\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\\\\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\\\\n        count, depth = volumes.shape[0], volumes.shape[1]\\\\n        if volumes.dtype == torch.uint8:\\\\n            volumes = volumes.float().div_(255.0)\\\\n\\\\n        slices = stack_2p5d(volumes)\\\\n        if self.input_mean is not None:\\\\n            slices = (slices - self.input_mean) / self.input_std\\\\n        features = self.backbone(slices)\\\\n        features = features.reshape(count, depth, self.feature_dim)\\\\n        return self.slice_pool(self.slice_encoder(features))\\\\n\\\\n    def pool_study(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                   mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, F) one embedding per study.\\\\n\\\\n        The study representation the head reads. Exposed separately so\\\\n        auxiliary objectives (report-embedding regression) can supervise the\\\\n        representation without their parameters entering this module -- the\\\\n        exported state dict must stay loadable by a bare `KneeNet`.\\\\n        """\\\\n        series = self.film(series, meta)\\\\n\\\\n        # The transformer takes True for positions to IGNORE, the inverse of\\\\n        # our mask.\\\\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\\\\n        # A fully padded row emits NaN from the attention; zero it before\\\\n        # pooling so one short study cannot poison the batch.\\\\n        encoded = torch.nan_to_num(encoded)\\\\n\\\\n        return self.study_pool(encoded, mask=mask)\\\\n\\\\n    def couple(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Add the detached parent term to each configured child logit.\\\\n\\\\n        A no-op unless the model was built (or loaded) with residual pairs, so\\\\n        every checkpoint written before this existed behaves identically.\\\\n\\\\n        The parent term is detached: the point is to relieve the child of\\\\n        learning effusion, not to let the synovitis objective reach back and\\\\n        reshape the effusion head, which is the one head on this finding that\\\\n        is reliable.\\\\n        """\\\\n        if self.residual_pairs is None:\\\\n            return logits\\\\n        child = self.residual_pairs[:, 0]\\\\n        parent = self.residual_pairs[:, 1]\\\\n        addition = torch.zeros_like(logits)\\\\n        addition[:, child] = (self.residual_beta.to(logits.dtype)\\\\n                              * logits[:, parent].detach())\\\\n        return logits + addition\\\\n\\\\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\\\\n                  mask: torch.Tensor) -> torch.Tensor:\\\\n        """(B, S, F) series embeddings -> (B, n_labels).\\\\n\\\\n        Everything above the backbone. Shared by the end-to-end path and the\\\\n        frozen path so the two cannot drift into different architectures.\\\\n        """\\\\n        return self.classify(self.pool_study(series, meta, mask))\\\\n\\\\n    def classify(self, pooled):\\\\n        logits = self.head(pooled)\\\\n        if self.classwise_pool:\\\\n            logits = logits.diagonal(dim1=1, dim2=2)\\\\n        return self.couple(logits)\\\\n\\\\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\\\\n                         mask: torch.Tensor) -> torch.Tensor:\\\\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\\\\n\\\\n        The backbone has already run, so this starts at the slice pooling.\\\\n        """\\\\n        batch, series, depth, _ = features.shape\\\\n        projected = self.project(features.reshape(batch * series, depth, -1))\\\\n        pooled = self.slice_pool(projected)\\\\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\\\\n                              meta, mask)\\\\n\\\\n    def forward_pooled(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                       mask: torch.Tensor):\\\\n        """End-to-end path returning (logits, pooled study embedding).\\\\n\\\\n        The training loop wants both: logits for the label losses, the pooled\\\\n        representation for auxiliary supervision. Computing them together\\\\n        avoids a second pass through the backbone.\\\\n        """\\\\n        batch, series = volumes.shape[0], volumes.shape[1]\\\\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\\\\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\\\\n        pooled = self.pool_study(embeddings, meta, mask)\\\\n        return self.classify(pooled), pooled\\\\n\\\\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\\\\n                mask: torch.Tensor) -> torch.Tensor:\\\\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\\\\n        logits, _ = self.forward_pooled(volumes, meta, mask)\\\\n        return logits\\\\n\\\\n\\\\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\\\\n                 weights: torch.Tensor, label_smoothing: float = 0.0,\\\\n                 pos_weight: Optional[torch.Tensor] = None,\\\\n                 symmetric_beta: float = 0.0) -> torch.Tensor:\\\\n    """BCE against soft targets, weighted per study and finding.\\\\n\\\\n    Targets are probabilities rather than 0/1 -- they come from two extractors\\\\n    that disagree often enough that thresholding would teach the model to\\\\n    reproduce extraction errors. Weights carry how much each target is\\\\n    believed: full for agreement, less for conflict, least for silence.\\\\n\\\\n    Three optional terms, all inert at their defaults so the reference runs\\\\n    stay bit-comparable. Each addresses something the plan names:\\\\n\\\\n    `label_smoothing` pulls every target toward 0.5 by `eps`. The mined labels\\\\n    disagree with image-derived ground truth about 18% of the time, and a\\\\n    confident target that is wrong 18% of the time is what late training\\\\n    memorises. Note that soft targets already carry some of this effect --\\\\n    smoothing on top of them is a floor on the target\\\\\\\'s distance from 0.5, not\\\\n    the usual one-hot correction.\\\\n\\\\n    `pos_weight` scales the positive term per finding. The metric is macro\\\\n    AUC, so a rare finding costs exactly as much as a common one; without this\\\\n    the loss is dominated by whatever is prevalent. Passed straight through to\\\\n    torch, which applies it to soft targets correctly.\\\\n\\\\n    `symmetric_beta` adds the reverse cross-entropy term of Wang et al.,\\\\n    "Symmetric Cross Entropy for Robust Learning with Noisy Labels" (ICCV\\\\n    2019). RCE swaps the roles of prediction and target, which bounds the loss\\\\n    a wrong label can contribute: a confidently mislabelled study cannot drag\\\\n    the gradient the way plain CE lets it. Targets are clamped off 0 and 1\\\\n    because RCE takes their log; the clamp is what the paper\\\\\\\'s `A` constant\\\\n    does.\\\\n    """\\\\n    if label_smoothing > 0:\\\\n        targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing\\\\n\\\\n    loss = F.binary_cross_entropy_with_logits(\\\\n        logits, targets, pos_weight=pos_weight, reduction="none")\\\\n\\\\n    if symmetric_beta > 0:\\\\n        probs = torch.sigmoid(logits)\\\\n        clamped = targets.clamp(1e-4, 1.0 - 1e-4)\\\\n        reverse = -(probs * torch.log(clamped)\\\\n                    + (1.0 - probs) * torch.log1p(-clamped))\\\\n        loss = loss + symmetric_beta * reverse\\\\n\\\\n    total = weights.sum()\\\\n    if total <= 0:\\\\n        return loss.mean() * 0.0\\\\n    return (loss * weights).sum() / total\\\\n\\\\n\\\\ndef pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                       ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                       weights: Optional[torch.Tensor] = None,\\\\n                       ref_weights: Optional[torch.Tensor] = None,\\\\n                       min_gap: float = 0.25,\\\\n                       pair_mask: Optional[torch.Tensor] = None,\\\\n                       max_margin: Optional[float] = None) -> torch.Tensor:\\\\n    """Per-finding RankNet loss between a batch and a reference set.\\\\n\\\\n    The metric is macro AUC -- a ranking statistic -- while BCE optimises\\\\n    calibration; this term optimises the ordering directly. Pairs are formed\\\\n    per finding between the current batch (`scores`/`targets`, (B, L), with\\\\n    gradient) and a reference set ((Q, L), typically detached scores from\\\\n    recent steps), because at one study per rank there are no pairs *within*\\\\n    a batch to rank.\\\\n\\\\n    Only pairs whose soft targets differ by at least `min_gap` count: the\\\\n    targets are extractor probabilities, and ordering two studies the\\\\n    extractor itself barely distinguishes would train the model to reproduce\\\\n    extraction noise. Surviving pairs are weighted by the target gap and by\\\\n    both studies\\\\\\\' label weights, so a confident 0.9-vs-0.1 pair outweighs a\\\\n    marginal one. Returns a zero that still touches `scores` when no pair\\\\n    qualifies, so the graph is identical either way.\\\\n\\\\n    `pair_mask` drops pairs before weighting. Broadcastable to (B, Q, L), so\\\\n    it can restrict which findings are ranked, which pairs are comparable, or\\\\n    both. `stratified_rank_loss` below builds one; at None every pair counts\\\\n    and this is the loss the reference runs used, unchanged.\\\\n\\\\n    `max_margin` clamps the logit difference before the softplus, and exists\\\\n    because leaving it unbounded diverged a real run (2026-08-31). A satisfied\\\\n    pair\\\\\\\'s loss decays to zero as its margin grows, but a **violated** pair\\\\\\\'s\\\\n    grows linearly and forever, so when the pair constraints contradict each\\\\n    other -- which is what within-stratum synovitis pairs do, since 90% of the\\\\n    mined targets sit at zero and the surviving pairs are mostly "mentioned vs\\\\n    silent" -- inflating the logit scale reduces loss on the majority faster\\\\n    than it raises it on the minority. The head then saturates, every study\\\\n    ties, AUC lands on exactly 0.5, and the shared trunk goes with it: that run\\\\n    took BCE from 0.60 to 5.99 and macro from 0.86 to 0.71. Clamping caps what\\\\n    any one pair can demand, so a contradictory set cannot pay for scale.\\\\n    Default None leaves every earlier run bit-identical.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    if ref_scores.numel() == 0:\\\\n        return zero\\\\n\\\\n    gap = targets.unsqueeze(1) - ref_targets.unsqueeze(0)      # (B, Q, L)\\\\n    pair_weight = gap.abs() * (gap.abs() >= min_gap)\\\\n    if weights is not None and ref_weights is not None:\\\\n        pair_weight = (pair_weight * weights.unsqueeze(1)\\\\n                       * ref_weights.unsqueeze(0))\\\\n    if pair_mask is not None:\\\\n        pair_weight = pair_weight * pair_mask.to(pair_weight.dtype)\\\\n\\\\n    total = pair_weight.sum()\\\\n    if total <= 0:\\\\n        return zero\\\\n\\\\n    margin = scores.unsqueeze(1) - ref_scores.unsqueeze(0)     # (B, Q, L)\\\\n    if max_margin is not None:\\\\n        margin = margin.clamp(-max_margin, max_margin)\\\\n    loss = F.softplus(-torch.sign(gap) * margin)\\\\n    return (loss * pair_weight).sum() / total\\\\n\\\\n\\\\ndef finding_mask(findings: Sequence[str], n_labels: int = None,\\\\n                 device=None) -> torch.Tensor:\\\\n    """(1, 1, L) mask selecting the findings a loss is allowed to touch.\\\\n\\\\n    The stratified objective is for synovitis. Applying it to all twelve\\\\n    findings would trade away ranking on eleven that are already fine, on a\\\\n    stratification that means nothing for them.\\\\n    """\\\\n    n_labels = n_labels or len(config.LABELS)\\\\n    mask = torch.zeros(n_labels, dtype=torch.float32, device=device)\\\\n    for name in findings:\\\\n        mask[config.LABELS.index(name)] = 1.0\\\\n    return mask.view(1, 1, -1)\\\\n\\\\n\\\\ndef caliper_pairs(strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                  caliper: float) -> torch.Tensor:\\\\n    """(B, Q, 1) mask of pairs whose stratifier scores are within `caliper`.\\\\n\\\\n    Matching on a continuous score rather than splitting it at a threshold,\\\\n    because the mined effusion label carries its own extraction error and a\\\\n    hard split inherits it at the boundary. The predicted effusion score is\\\\n    continuous, scores AUC 0.888, and is available at every training step.\\\\n    """\\\\n    distance = (strata.unsqueeze(1) - ref_strata.unsqueeze(0)).abs()\\\\n    return (distance <= caliper).unsqueeze(-1)\\\\n\\\\n\\\\ndef stratified_rank_loss(scores: torch.Tensor, targets: torch.Tensor,\\\\n                         ref_scores: torch.Tensor, ref_targets: torch.Tensor,\\\\n                         strata: torch.Tensor, ref_strata: torch.Tensor,\\\\n                         findings: Sequence[str], caliper: float = 0.15,\\\\n                         weights: Optional[torch.Tensor] = None,\\\\n                         ref_weights: Optional[torch.Tensor] = None,\\\\n                         min_gap: float = 0.25,\\\\n                         max_margin: Optional[float] = 8.0) -> torch.Tensor:\\\\n    """RankNet restricted to pairs from the same effusion stratum.\\\\n\\\\n    Conditional AUC *is* within-stratum pair concordance, so this is not an\\\\n    analogy to the metric -- it is the metric, differentiably. Pairs the\\\\n    conditional number ignores are the ones effusion already orders correctly,\\\\n    and training on them is what produced a synovitis head that reconstructs\\\\n    from the effusion head at R^2 0.55.\\\\n\\\\n    The known risk, stated in the plan and not solved here: stripping effusion\\\\n    removes the one reliable cue and leaves a harder task supervised by labels\\\\n    that are silent in 82.2% of reports against a true prevalence of 46.6%. If\\\\n    label quality binds, this loss makes the finding worse, and the conditional\\\\n    metric is what will show it.\\\\n\\\\n    `strata` is the detached predicted effusion probability for the current\\\\n    batch and `ref_strata` the same for the queued reference studies.\\\\n\\\\n    `max_margin` defaults to a finite 8.0 here rather than to None, because\\\\n    this is the loss whose pairs contradict each other -- see\\\\n    `pairwise_rank_loss`. A margin of 8 logits is a probability ratio of about\\\\n    3,000:1; a pair that wants more than that is not asking for an ordering.\\\\n    """\\\\n    mask = caliper_pairs(strata, ref_strata, caliper) * finding_mask(\\\\n        findings, scores.shape[-1], scores.device)\\\\n    return pairwise_rank_loss(scores, targets, ref_scores, ref_targets,\\\\n                              weights, ref_weights, min_gap=min_gap,\\\\n                              pair_mask=mask, max_margin=max_margin)\\\\n\\\\n\\\\ndef coupling_penalty(scores: torch.Tensor, ref_scores: torch.Tensor,\\\\n                     child: int, parent: int, target: float) -> torch.Tensor:\\\\n    """Push corr(child logit, parent logit) toward `target`, not toward zero.\\\\n\\\\n    The findings genuinely co-occur: on the gold set the odds ratio is 6.09\\\\n    (p = 0.003) and the label correlation is 0.403, against the model\\\\\\\'s 0.651.\\\\n    Driving the two heads to independence optimises toward a value the truth\\\\n    contradicts, so `target` is required and is meant to be the observed\\\\n    coupling rather than 0.\\\\n\\\\n    Batches here are one study per rank, so the correlation is computed over\\\\n    the current batch concatenated with the detached reference queue. Returns\\\\n    a zero that still touches `scores` when there are too few studies to\\\\n    estimate a correlation, so the graph does not change shape between steps.\\\\n    """\\\\n    zero = scores.sum() * 0.0\\\\n    pool = (torch.cat([scores, ref_scores], dim=0)\\\\n            if ref_scores.numel() else scores)\\\\n    if pool.shape[0] < 8:\\\\n        return zero\\\\n\\\\n    a = pool[:, child] - pool[:, child].mean()\\\\n    b = pool[:, parent] - pool[:, parent].mean()\\\\n    denominator = a.norm() * b.norm()\\\\n    if denominator <= 0:\\\\n        return zero\\\\n    return ((a * b).sum() / denominator - target) ** 2\\\\n\\\\n\\\\ndef load_knee_state(path, map_location="cpu") -> dict:\\\\n    """Read a `KneeNet` state dict from either checkpoint format.\\\\n\\\\n    The end-to-end path writes two things over its life: a bare state dict\\\\n    from `torch.save(model.state_dict(), ...)`, and a Lightning checkpoint,\\\\n    which nests the same tensors under `state_dict` and prefixes every key\\\\n    with `model.` because `KneeNet` is an attribute of the wrapper. The\\\\n    submission must not care which it was handed -- requiring a manual\\\\n    conversion step is requiring a step that can be forgotten, and it was.\\\\n\\\\n    Lightning checkpoints also carry optimiser state and callback objects,\\\\n    which `weights_only=True` refuses, so that is attempted first and relaxed\\\\n    only on failure. These are files this project wrote.\\\\n    """\\\\n    try:\\\\n        obj = torch.load(path, map_location=map_location, weights_only=True)\\\\n    except TypeError:       # torch too old to know the argument\\\\n        obj = torch.load(path, map_location=map_location)\\\\n    except Exception:       # noqa: BLE001 - a Lightning checkpoint\\\\n        obj = torch.load(path, map_location=map_location, weights_only=False)\\\\n\\\\n    state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj\\\\n    if any(key.startswith("model.") for key in state):\\\\n        state = {key[len("model."):]: value for key, value in state.items()\\\\n                 if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(f"no weights found in {path}")\\\\n    return state\\\\n\\\', \\\'src/model/experiments.py\\\': \\\'"""Pre-registered, single-change experiments. No preset implies a score gain."""\\\\n\\\\nfrom dataclasses import asdict, dataclass, replace\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass Experiment:\\\\n    name: str = "baseline"\\\\n    backbone: str = "convnext_tiny.fb_in22k_ft_in1k"\\\\n    slices: int = 16\\\\n    max_series: int = 5\\\\n    series_selection: str = "uniform"\\\\n    resample: bool = False\\\\n    classwise_pool: bool = False\\\\n    slice_transformer: bool = False\\\\n\\\\n    def to_dict(self):\\\\n        result = asdict(self)\\\\n        # Preserve existing experiment contracts and checkpoint sidecars.\\\\n        if not self.slice_transformer:\\\\n            result.pop("slice_transformer")\\\\n        return result\\\\n\\\\n\\\\nBASELINE = Experiment()\\\\nPRESETS = {\\\\n    "baseline": BASELINE,\\\\n    "resample": replace(BASELINE, name="resample", resample=True),\\\\n    "protocol": replace(BASELINE, name="protocol", series_selection="protocol"),\\\\n    "classwise": replace(BASELINE, name="classwise", classwise_pool=True),\\\\n    "convnextv2": replace(BASELINE, name="convnextv2",\\\\n                          backbone="convnextv2_tiny.fcmae_ft_in22k_in1k"),\\\\n    "slice_transformer": replace(BASELINE, name="slice_transformer",\\\\n                                 series_selection="protocol", slice_transformer=True),\\\\n}\\\\n\\\\n\\\\ndef get_experiment(name):\\\\n    if name not in PRESETS:\\\\n        raise ValueError(f"unknown experiment {name!r}; choose {sorted(PRESETS)}")\\\\n    return PRESETS[name]\\\\n\\\', \\\'src/model/artifacts.py\\\': \\\'"""Explicit model provenance and inference for heterogeneous ensembles."""\\\\n\\\\nimport hashlib\\\\nimport json\\\\nfrom pathlib import Path\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nimport torch\\\\n\\\\nfrom src import config\\\\nfrom src.model import dataset, net\\\\n\\\\n\\\\ndef sha256(path):\\\\n    digest = hashlib.sha256()\\\\n    with open(path, "rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(8 << 20), b""):\\\\n            digest.update(chunk)\\\\n    return digest.hexdigest()\\\\n\\\\n\\\\ndef write_metadata(checkpoint, experiment, fold, train_uids, val_uids):\\\\n    """Bind input geometry and split membership to the actual exported weights."""\\\\n    train_uids, val_uids = sorted(train_uids), sorted(val_uids)\\\\n    if set(train_uids) & set(val_uids):\\\\n        raise ValueError("training and validation studies overlap")\\\\n    payload = {\\\\n        "schema_version": 1, "labels": config.LABELS,\\\\n        "sha256": sha256(checkpoint), "experiment": experiment,\\\\n        "cache_size": config.CACHE_SIZE, "cache_slices": config.CACHE_SLICES,\\\\n        "fold": int(fold), "train_uids": train_uids, "val_uids": val_uids,\\\\n    }\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    path.write_text(json.dumps(payload, indent=2) + "\\\\\\\\n")\\\\n    return path\\\\n\\\\n\\\\ndef load_metadata(checkpoint):\\\\n    path = Path(str(checkpoint) + ".json")\\\\n    data = json.loads(path.read_text())\\\\n    if data["schema_version"] != 1 or data["labels"] != config.LABELS:\\\\n        raise ValueError(f"unsupported schema or label order: {path}")\\\\n    if data["sha256"] != sha256(checkpoint):\\\\n        raise ValueError(f"checkpoint checksum mismatch: {checkpoint}")\\\\n    if data["cache_size"] != config.CACHE_SIZE or data["cache_slices"] != config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint requires a different source cache geometry")\\\\n    spec = data["experiment"]\\\\n    if not 1 <= spec["slices"] <= config.CACHE_SLICES:\\\\n        raise ValueError("checkpoint slice count is outside the cache")\\\\n    dataset.select_series(pd.DataFrame(), spec["max_series"], spec["series_selection"])\\\\n    return data\\\\n\\\\n\\\\ndef load_ensemble(manifest_path, device="cpu"):\\\\n    """Load exactly the members named in JSON, never a directory-wide glob.\\\\n\\\\n    Format: {"members": [{"checkpoint": "run/fold0.pt", "weight": 1.0}]}.\\\\n    Paths are relative to this manifest. Each checkpoint requires its .pt.json\\\\n    sidecar, written by notebook 08. Weights are global, not per finding.\\\\n    """\\\\n    manifest_path = Path(manifest_path)\\\\n    entries = json.loads(manifest_path.read_text())["members"]\\\\n    if not entries:\\\\n        raise ValueError("ensemble has no members")\\\\n    members, digests, identities = [], set(), set()\\\\n    for entry in entries:\\\\n        path = (manifest_path.parent / entry["checkpoint"]).resolve()\\\\n        weight = float(entry.get("weight", 1.0))\\\\n        if not np.isfinite(weight) or weight <= 0:\\\\n            raise ValueError("ensemble weights must be finite and positive")\\\\n        metadata = load_metadata(path)\\\\n        identity = (metadata["experiment"]["name"], metadata["fold"])\\\\n        if metadata["sha256"] in digests or identity in identities:\\\\n            raise ValueError("duplicate checkpoint or experiment/fold in ensemble")\\\\n        digests.add(metadata["sha256"])\\\\n        identities.add(identity)\\\\n        spec = metadata["experiment"]\\\\n        state = net.load_knee_state(path)\\\\n        trunk = net.build_backbone(spec["backbone"], pretrained=False)\\\\n        defaults = net.backbone_defaults(spec["backbone"])\\\\n        model = net.KneeNet(trunk, trunk.num_features,\\\\n                            input_mean=defaults["input_mean"],\\\\n                            input_std=defaults["input_std"],\\\\n                            **net.aggregation_defaults(state))\\\\n        model.load_state_dict(state)\\\\n        members.append({"model": model.to(device).eval(), "spec": spec,\\\\n                        "weight": weight, "metadata": metadata})\\\\n    return members\\\\n\\\\n\\\\n@torch.inference_mode()\\\\ndef predict_members(members, volumes, rows, device="cpu"):\\\\n    """Predict from full acquired volumes and one aligned metadata row per series."""\\\\n    if len(rows) != len(volumes) or not len(rows):\\\\n        raise ValueError("volumes and metadata must be nonempty and aligned")\\\\n    order = np.argsort(rows["SeriesInstanceUID"].astype(str).to_numpy(), kind="stable")\\\\n    rows = rows.iloc[order].reset_index(drop=True)\\\\n    volumes = [volumes[i] for i in order]\\\\n    outputs = []\\\\n    for member in members:\\\\n        spec = member["spec"]\\\\n        chosen = dataset.select_series(rows, spec["max_series"], spec["series_selection"])\\\\n        picked = rows.iloc[chosen]\\\\n        stack = []\\\\n        for i, (_, row) in zip(chosen, picked.iterrows()):\\\\n            volume = dataset.mirror_volume(volumes[i], row.get("side_predicted"),\\\\n                                            row.get("Anatomical_Plane"))\\\\n            if volume.shape != (config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE):\\\\n                raise ValueError(f"unexpected volume geometry: {volume.shape}")\\\\n            keep = np.linspace(0, len(volume) - 1, spec["slices"]).round().astype(int)\\\\n            stack.append(volume[keep])\\\\n        tensor = torch.from_numpy(np.stack(stack)).unsqueeze(0).to(device)\\\\n        meta = torch.from_numpy(np.stack([\\\\n            dataset.series_metadata(row) for _, row in picked.iterrows()\\\\n        ])).unsqueeze(0).to(device)\\\\n        mask = torch.ones(1, len(stack), dtype=torch.bool, device=device)\\\\n        device_type = torch.device(device).type\\\\n        with torch.autocast(device_type, enabled=device_type == "cuda"):\\\\n            logits = member["model"](tensor, meta, mask)\\\\n        probs = logits.float().sigmoid().cpu().numpy()[0]\\\\n        if not np.isfinite(probs).all():\\\\n            raise ValueError("non-finite model prediction")\\\\n        outputs.append(probs)\\\\n    return np.stack(outputs)\\\\n\\\', \\\'src/model/conditional.py\\\': \\\'"""Effusion-stratified conditional AUC: the metric synovitis is judged on.\\\\n\\\\nPooled AUC cannot measure this finding. On the 58 gold studies the *gold\\\\neffusion label by itself* scores 0.698 as a synovitis classifier, against the\\\\nsynovitis head\\\\\\\'s 0.701 -- so a number in that region is obtainable with no\\\\nsynovitis-specific skill at all, and seven consecutive experiments read as\\\\nnull partly because they were scored on it.\\\\n\\\\nConditional AUC is the concordance of positive/negative pairs drawn from\\\\n**within the same effusion stratum**. Cross-stratum pairs, where effusion\\\\nalone orders the pair correctly, are dropped. What remains is the only\\\\nquantity on this gold set that moves when synovitis-specific skill moves:\\\\n\\\\n    pooled       0.701      837 pairs, ~55% of them cross-stratum\\\\n    conditional  0.630      376 within-stratum pairs\\\\n\\\\nThe two are linked by pair mass, which is why `pooled_from_conditional` can\\\\nstate the exact pooled number a conditional target implies on this set --\\\\nconditional 0.75 is pooled 0.755, and a *perfect* conditional score is pooled\\\\n0.867. Anything above that is unreachable without changing the gating.\\\\n\\\\nDeliberately free of torch and of pandas-at-import: this module is imported by\\\\nthe analysis tools, bootstrapped into the notebooks, and used by the training\\\\nloop\\\\\\\'s validation hook, and `src/model/lightning.py` cannot be imported by any\\\\nof the first two without dragging in the Lightning framework.\\\\n\\\\nTwo rules this module exists to enforce, from the plan\\\\\\\'s standing rules:\\\\n\\\\n1. Never judge synovitis on macro, and never on pooled alone.\\\\n2. Compare paired. Independent Hanley-McNeil standard errors are +/-0.07 here\\\\n   and carry between-study variance that a matched comparison removes; a real\\\\n   movement of 0.05 is invisible against them and visible against a paired\\\\n   bootstrap of the difference.\\\\n"""\\\\n\\\\nfrom typing import Dict, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Pooled AUC, and the noise band around it\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef auc(y_true, scores) -> float:\\\\n    """ROC AUC via the rank statistic, ties averaged. Matches the training loop."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n    return float((ranks[positives].sum() - n_pos * (n_pos + 1) / 2)\\\\n                 / (n_pos * n_neg))\\\\n\\\\n\\\\ndef auc_se(a: float, n_pos: int, n_neg: int) -> float:\\\\n    """Hanley & McNeil (1982) standard error of an AUC.\\\\n\\\\n    Printed beside pooled numbers so a difference inside the band is visibly\\\\n    not evidence. Never used to judge a *difference* between two models on the\\\\n    same studies -- that is what `paired_bootstrap` is for.\\\\n    """\\\\n    if not np.isfinite(a) or n_pos < 1 or n_neg < 1:\\\\n        return float("nan")\\\\n    q1 = a / (2.0 - a)\\\\n    q2 = 2.0 * a * a / (1.0 + a)\\\\n    numerator = (a * (1 - a) + (n_pos - 1) * (q1 - a * a)\\\\n                 + (n_neg - 1) * (q2 - a * a))\\\\n    return float(np.sqrt(max(numerator, 0.0) / (n_pos * n_neg)))\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Strata\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef strata_from_scores(effusion, threshold: Optional[float] = None):\\\\n    """Binary effusion strata from a continuous effusion score.\\\\n\\\\n    The plan stratifies on *predicted* effusion rather than on the mined\\\\n    effusion label, because the prediction scores AUC 0.888 while the mined\\\\n    label carries its own extraction error that would blur the strata. The\\\\n    default split is the median, which is what reproduces the plan\\\\\\\'s 0.617\\\\n    against the gold label\\\\\\\'s 0.630 -- close enough that the metric is\\\\n    computable during training, where no gold effusion label exists.\\\\n\\\\n    A median split rather than a fixed probability because the head is not\\\\n    calibrated across runs: a 0.70 cut that balances one model\\\\\\\'s studies\\\\n    puts 80% of another model\\\\\\\'s on one side, and the number would then move\\\\n    between runs for a reason that has nothing to do with synovitis.\\\\n    """\\\\n    effusion = np.asarray(effusion, dtype=float)\\\\n    if threshold is None:\\\\n        threshold = float(np.median(effusion))\\\\n    return (effusion >= threshold).astype(int)\\\\n\\\\n\\\\ndef comparable_pairs(y_true, strata, caliper: Optional[float] = None):\\\\n    """(n_pos, n_neg) boolean mask of pairs that share an effusion stratum.\\\\n\\\\n    With `caliper=None`, `strata` is treated as categorical and pairs must\\\\n    match exactly -- the gold-label and median-split cases. With a caliper,\\\\n    `strata` is a continuous score and a pair is comparable when the two\\\\n    studies\\\\\\\' effusion scores lie within `caliper` of each other, which is the\\\\n    matching rule the stratified training loss uses.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    left, right = strata[positives], strata[~positives]\\\\n    if caliper is None:\\\\n        return left[:, None] == right[None, :]\\\\n    return np.abs(left[:, None].astype(float)\\\\n                  - right[None, :].astype(float)) <= caliper\\\\n\\\\n\\\\ndef _concordance(y_true, scores):\\\\n    """(n_pos, n_neg) matrix of 1 / 0.5 / 0 per pair. Ties count as half."""\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    difference = scores[positives][:, None] - scores[~positives][None, :]\\\\n    return (difference > 0).astype(float) + 0.5 * (difference == 0)\\\\n\\\\n\\\\ndef conditional_auc(y_true, scores, strata,\\\\n                    caliper: Optional[float] = None) -> float:\\\\n    """Concordance over within-stratum pairs only. NaN if there are none.\\\\n\\\\n    Pair-weighted, not stratum-averaged: the strata are 286 and 90 pairs on\\\\n    the gold set, and averaging them would give the 5-positive stratum equal\\\\n    say with the 22-positive one. Pair weighting is also what makes the\\\\n    conversion to pooled AUC exact.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    total = mask.sum()\\\\n    if total == 0:\\\\n        return float("nan")\\\\n    return float((_concordance(y_true, scores) * mask).sum() / total)\\\\n\\\\n\\\\ndef pair_counts(y_true, strata, caliper: Optional[float] = None) -> Dict[str, int]:\\\\n    """How many pairs the conditional number rests on, and the total."""\\\\n    mask = comparable_pairs(y_true, strata, caliper)\\\\n    return {"within": int(mask.sum()), "cross": int((~mask).sum()),\\\\n            "total": int(mask.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Where the error sits\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef pair_decomposition(y_true, scores, strata) -> list:\\\\n    """Every discordant pair, split by the stratum of each member.\\\\n\\\\n    The table in the plan\\\\\\\'s §1.4. Reading it is how the addressable headroom\\\\n    was located: within-stratum pairs are 44.9% of the total and are the only\\\\n    ones a synovitis-specific improvement can move. The cell where a\\\\n    stratum-0 positive meets a stratum-1 negative is ranked worse than chance\\\\n    -- that is the gating tax, and it is paid on five studies.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    strata = np.asarray(strata)\\\\n    positives = y_true == 1\\\\n    concordance = _concordance(y_true, scores)\\\\n    error = 1.0 - concordance\\\\n    total_error = error.sum()\\\\n\\\\n    rows = []\\\\n    for pos_level in sorted(set(strata[positives].tolist()), reverse=True):\\\\n        for neg_level in sorted(set(strata[~positives].tolist()), reverse=True):\\\\n            mask = ((strata[positives] == pos_level)[:, None]\\\\n                    & (strata[~positives] == neg_level)[None, :])\\\\n            pairs = int(mask.sum())\\\\n            if pairs == 0:\\\\n                continue\\\\n            cell_error = float((error * mask).sum())\\\\n            rows.append({\\\\n                "pos_stratum": int(pos_level), "neg_stratum": int(neg_level),\\\\n                "within": pos_level == neg_level,\\\\n                "pairs": pairs,\\\\n                "misranked": cell_error / pairs,\\\\n                "error_share": cell_error / total_error if total_error else float("nan"),\\\\n            })\\\\n    return rows\\\\n\\\\n\\\\ndef pooled_from_conditional(y_true, scores, strata,\\\\n                            targets: Sequence[float]) -> list:\\\\n    """The pooled AUC each conditional target implies, on these studies.\\\\n\\\\n    Cross-stratum concordance is held at what this model already achieves and\\\\n    only the within-stratum mass is moved, so the answer is arithmetic rather\\\\n    than a fit. It exists to keep two conversations in the same units: the\\\\n    leaderboard and every historical table are pooled, the plan\\\\\\\'s targets are\\\\n    conditional, and quoting one as the other is how a 0.63 -> 0.75 move gets\\\\n    dismissed as "+0.05, inside the noise".\\\\n\\\\n    The value at `target=1.0` is the ceiling under an unchanged gating -- 0.867\\\\n    here. It is an arithmetic bound, not a reachable target: one model produces\\\\n    both halves, and a residualised model can buy within-stratum ordering by\\\\n    selling the gating.\\\\n    """\\\\n    mask = comparable_pairs(y_true, strata)\\\\n    concordance = _concordance(y_true, scores)\\\\n    within, cross = mask.sum(), (~mask).sum()\\\\n    cross_concordant = float((concordance * ~mask).sum())\\\\n    total = within + cross\\\\n    return [{"conditional": float(t),\\\\n             "pooled": float((cross_concordant + t * within) / total)}\\\\n            for t in targets]\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Uncertainty: paired, always\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef _resamples(n: int, draws: int, seed: int):\\\\n    rng = np.random.default_rng(seed)\\\\n    return rng.integers(0, n, size=(draws, n))\\\\n\\\\n\\\\ndef bootstrap_conditional(y_true, scores, strata, caliper=None,\\\\n                          draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """Percentile CI for one model\\\\\\\'s conditional AUC, plus P(AUC <= 0.5).\\\\n\\\\n    Studies are resampled, not pairs: a study appears in many pairs and\\\\n    resampling pairs would treat those as independent observations and report\\\\n    an interval far narrower than the evidence supports.\\\\n\\\\n    `p_at_or_below_chance` is the fraction of resamples at or under 0.5. It is\\\\n    the number the gate is read on, because "is there any synovitis-specific\\\\n    signal at all" is a question about which side of chance the estimate sits,\\\\n    not about whether it differs from some other model.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    strata = np.asarray(strata)\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = conditional_auc(y_true[index], scores[index], strata[index],\\\\n                                caliper)\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = conditional_auc(y_true, scores, strata, caliper)\\\\n    if values.size == 0:\\\\n        return {"auc": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "sd": float("nan"), "p_at_or_below_chance": float("nan"),\\\\n                "draws": 0}\\\\n    return {"auc": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "sd": float(values.std(ddof=1)),\\\\n            "p_at_or_below_chance": float((values <= 0.5).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\ndef paired_bootstrap(y_true, scores_a, scores_b, strata=None, caliper=None,\\\\n                     draws: int = 4000, seed: int = 0) -> Dict[str, float]:\\\\n    """CI for (A - B) on the same studies, conditional or pooled.\\\\n\\\\n    `strata=None` compares pooled AUCs; otherwise both models are scored on\\\\n    the same within-stratum pairs. Either way the *same* resample indexes both\\\\n    models, so between-study variance -- which is most of the +/-0.07 an\\\\n    unpaired standard error reports -- cancels.\\\\n\\\\n    `p_a_not_better` is the fraction of resamples where the difference is at\\\\n    or below zero. Read it as the evidence against A, and pre-register the\\\\n    threshold before running, not after seeing it.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores_a = np.asarray(scores_a, dtype=float)\\\\n    scores_b = np.asarray(scores_b, dtype=float)\\\\n    strata = None if strata is None else np.asarray(strata)\\\\n\\\\n    def statistic(y, a, b, s):\\\\n        if s is None:\\\\n            return auc(y, a) - auc(y, b)\\\\n        return (conditional_auc(y, a, s, caliper)\\\\n                - conditional_auc(y, b, s, caliper))\\\\n\\\\n    values = []\\\\n    for index in _resamples(len(y_true), draws, seed):\\\\n        value = statistic(y_true[index], scores_a[index], scores_b[index],\\\\n                          None if strata is None else strata[index])\\\\n        if np.isfinite(value):\\\\n            values.append(value)\\\\n    values = np.asarray(values)\\\\n    point = statistic(y_true, scores_a, scores_b, strata)\\\\n    if values.size == 0:\\\\n        return {"delta": point, "lo": float("nan"), "hi": float("nan"),\\\\n                "p_a_not_better": float("nan"), "draws": 0}\\\\n    return {"delta": point,\\\\n            "lo": float(np.percentile(values, 2.5)),\\\\n            "hi": float(np.percentile(values, 97.5)),\\\\n            "p_a_not_better": float((values <= 0).mean()),\\\\n            "draws": int(values.size)}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Calibrating the residual coupling\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef logit(probability, floor: float = 1e-6):\\\\n    """Probabilities to logits, clipped off the asymptotes."""\\\\n    clipped = np.clip(np.asarray(probability, dtype=float), floor, 1 - floor)\\\\n    return np.log(clipped / (1 - clipped))\\\\n\\\\n\\\\ndef calibrate_coupling(labels, parent_logits, iterations: int = 50) -> dict:\\\\n    """Fit `P(child=1) = sigmoid(a + b * parent_logit)` and return b.\\\\n\\\\n    This is the beta for the residual parameterisation\\\\n    `child_logit = beta * stopgrad(parent_logit) + g(features)`, and where it\\\\n    comes from matters more than its value:\\\\n\\\\n    - **Not** from the model\\\\\\\'s own +0.843 reconstruction coefficient. That is\\\\n      the coupling the current model already has, and reproducing it by\\\\n      construction would hand the residual head exactly the term it is\\\\n      supposed to replace.\\\\n    - **Not** from r_true = 0.403. A rank correlation between two binary\\\\n      labels does not map onto a logit coefficient, and the plan says so\\\\n      explicitly.\\\\n\\\\n    It is fitted out of fold, on labels, by IRLS -- the coupling the *truth*\\\\n    supports between the two findings, measured on whatever label set is\\\\n    trustworthy enough to fit it. `separated` flags the degenerate case where\\\\n    the fit runs away because the parent perfectly orders the child; the\\\\n    coefficient is then meaningless and should not be used.\\\\n    """\\\\n    y = np.asarray(labels, dtype=float)\\\\n    x = np.asarray(parent_logits, dtype=float)\\\\n    ok = np.isfinite(y) & np.isfinite(x)\\\\n    y, x = y[ok], x[ok]\\\\n    if y.size < 10 or y.std() == 0:\\\\n        return {"beta": float("nan"), "intercept": float("nan"),\\\\n                "n": int(y.size), "separated": True}\\\\n\\\\n    design = np.column_stack([np.ones_like(x), x])\\\\n    coefficients = np.zeros(2)\\\\n    for _ in range(iterations):\\\\n        # Clipped before the exponential: under separation the coefficients\\\\n        # run away and the naive form overflows on the way to being flagged.\\\\n        eta = np.clip(design @ coefficients, -700, 700)\\\\n        mu = 1.0 / (1.0 + np.exp(-eta))\\\\n        weight = np.clip(mu * (1 - mu), 1e-9, None)\\\\n        gradient = design.T @ (y - mu)\\\\n        hessian = design.T @ (design * weight[:, None])\\\\n        try:\\\\n            step = np.linalg.solve(hessian, gradient)\\\\n        except np.linalg.LinAlgError:\\\\n            return {"beta": float("nan"), "intercept": float("nan"),\\\\n                    "n": int(y.size), "separated": True}\\\\n        coefficients = coefficients + step\\\\n        if np.max(np.abs(step)) < 1e-8:\\\\n            break\\\\n    separated = bool(np.max(np.abs(coefficients)) > 20)\\\\n    return {"beta": float(coefficients[1]), "intercept": float(coefficients[0]),\\\\n            "n": int(y.size), "separated": separated}\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# The Stage 0 gate\\\\n# ---------------------------------------------------------------------------\\\\n#\\\\n# The plan says "clearly > 0.5", "= 0.5" and "ambiguous" without giving\\\\n# numbers. They are fixed here, once, in code, so they cannot be chosen after\\\\n# seeing a result -- and here rather than in the tool because the training\\\\n# notebooks read the same gate and two copies would drift.\\\\n\\\\nGATE_ACTIONS = {\\\\n    "clear": ("synovitis-specific signal survives conditioning on effusion",\\\\n              "Stage 1 -- train the conditional task directly"),\\\\n    "chance": ("the score is a better effusion proxy, nothing more",\\\\n               "Stage 3 (plan §4) or Stage 4 descope (plan §10) -- the plan "\\\\n               "gives both; decide before running either"),\\\\n    "ambiguous": ("expected at n=58; the CI straddles chance",\\\\n                  "Stage 1 at reduced confidence, and prioritise Stage 2 "\\\\n                  "(expand the gold set)"),\\\\n}\\\\n\\\\n\\\\ndef gate_reading(stats: dict) -> str:\\\\n    """clear | chance | ambiguous, from a `bootstrap_conditional` result.\\\\n\\\\n        clear      2.5th percentile above 0.5, and P(AUC <= 0.5) < 0.05\\\\n        chance     point estimate <= 0.55 and P(AUC <= 0.5) >= 0.25\\\\n        ambiguous  anything else -- the expected outcome at n=58\\\\n    """\\\\n    if not np.isfinite(stats.get("auc", float("nan"))):\\\\n        return "ambiguous"\\\\n    if stats["lo"] > 0.5 and stats["p_at_or_below_chance"] < 0.05:\\\\n        return "clear"\\\\n    if stats["auc"] <= 0.55 and stats["p_at_or_below_chance"] >= 0.25:\\\\n        return "chance"\\\\n    return "ambiguous"\\\\n\\\', \\\'src/model/augment.py\\\': \\\'"""Training augmentation for study volumes.\\\\n\\\\nThe pipeline shipped with one augmentation -- a single intensity scale in\\\\n[0.9, 1.1] applied to a whole study -- while the plan (§3.4) calls for what the\\\\nRSNA winners used: geometric transforms, photometric jitter, noise, region\\\\ndropout, and slice-order permutation, all kept consistent across the slices of\\\\na series.\\\\n\\\\n**Series-consistent by construction.** Every transform draws its parameters\\\\nonce per series and applies them to all of that series\\\\\\\' slices. The alternative\\\\n-- fresh parameters per slice -- turns a coherent volume into a stack of\\\\nunrelated images, which is exactly the through-plane structure `stack_2p5d`\\\\nexists to give the backbone.\\\\n\\\\n**What is deliberately absent, and why:**\\\\n\\\\n- **Rotation, perspective, affine.** They need `scipy.ndimage` or `cv2` and run\\\\n  per-slice interpolation over ~80 slices per study; at 4,349 studies an epoch\\\\n  that is a dataloader bottleneck, not a free regulariser. Worth adding behind\\\\n  a measurement of loader throughput, not before.\\\\n- **Arbitrary slice permutation.** The plan lists it, but it would break the\\\\n  2.5D stacking: `stack_2p5d` gives each slice its two neighbours as channels,\\\\n  and a shuffled stack makes those neighbours meaningless. `slice_reverse`\\\\n  is the order change that keeps neighbour structure intact -- it reverses the\\\\n  through-plane direction rather than destroying it.\\\\n- **MixUp.** Not implementable at `BATCH_STUDIES = 1`: a within-batch\\\\n  permutation is the identity. Recorded in `docs/plan.md`.\\\\n\\\\n**Laterality flip is a study-level transform, not a series-level one.** All of\\\\na study\\\\\\\'s series image the same knee, so they must flip together, and the flip\\\\nswaps medial with lateral -- which means `config.LATERAL_LABEL_PAIRS` has to be\\\\nswapped in the targets at the same time. `config.py` has carried that constant\\\\nand that warning since before there was anything to use it. Getting this half\\\\nright (flipping pixels, not labels) corrupts four of the twelve findings\\\\nsilently, which is why the dataset refused to flip at all until now.\\\\n"""\\\\n\\\\nfrom dataclasses import dataclass\\\\nfrom typing import Optional, Tuple\\\\n\\\\nimport numpy as np\\\\n\\\\nfrom src import config\\\\nfrom src.model.dataset import (DEFAULT_MIRROR_AXIS, META_COLUMNS,\\\\n                               PLANE_MIRROR_AXIS)\\\\n\\\\n# Index of `is_sagittal` in the per-series metadata row. The flip axis depends\\\\n# on the acquisition plane, and this is where that is already encoded -- no\\\\n# need to reach back into the index for `Anatomical_Plane`.\\\\nSAGITTAL_COLUMN = META_COLUMNS.index("is_sagittal")\\\\n\\\\n\\\\ndef flip_volume(volume: np.ndarray, plane=None,\\\\n                sagittal: Optional[bool] = None) -> np.ndarray:\\\\n    """Mirror one series across the medial-lateral axis.\\\\n\\\\n    The axis comes from `dataset.PLANE_MIRROR_AXIS`, the same table\\\\n    `mirror_volume` uses to canonicalise laterality -- deliberately, because a\\\\n    training flip and a test-time flip that disagreed about which axis is\\\\n    medial-lateral would be a mismatch that runs clean and scores worse. On a\\\\n    sagittal series medial-lateral is the slice axis; on coronal and axial it\\\\n    lies in-plane.\\\\n\\\\n    Give it either `plane` (the string from the series metadata) or `sagittal`\\\\n    (the bit already encoded in a metadata row).\\\\n    """\\\\n    if sagittal is not None:\\\\n        axis = PLANE_MIRROR_AXIS["Sagittal"] if sagittal else DEFAULT_MIRROR_AXIS\\\\n    else:\\\\n        axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\\\\n    return np.flip(volume, axis=axis).copy()\\\\n\\\\n\\\\ndef lateral_swap_order():\\\\n    """Index permutation that exchanges the medial/lateral finding pairs.\\\\n\\\\n    A mirrored knee puts the medial compartment where the lateral one was, so\\\\n    any prediction or target read off a flipped image has to be permuted by\\\\n    this before it means anything. Used by the training augmentation and by\\\\n    the submission\\\\\\\'s flip TTA, from one definition, so the two cannot drift.\\\\n    """\\\\n    order = list(range(len(config.LABELS)))\\\\n    index_of = {name: i for i, name in enumerate(config.LABELS)}\\\\n    for medial, lateral in config.LATERAL_LABEL_PAIRS:\\\\n        a, b = index_of[medial], index_of[lateral]\\\\n        order[a], order[b] = order[b], order[a]\\\\n    return order\\\\n\\\\n\\\\n@dataclass(frozen=True)\\\\nclass AugmentConfig:\\\\n    """Which augmentations run, and how hard.\\\\n\\\\n    The defaults reproduce the incumbent behaviour exactly -- one intensity\\\\n    scale, nothing else -- so `AugmentConfig()` is the reference configuration\\\\n    and every measured result stays comparable. Turn things on one at a time.\\\\n    """\\\\n\\\\n    intensity_scale: Optional[Tuple[float, float]] = (0.9, 1.1)\\\\n    intensity_shift: float = 0.0        # +/- this many 8-bit levels\\\\n    gamma: Optional[Tuple[float, float]] = None\\\\n    noise_std: float = 0.0              # gaussian sigma, in 8-bit levels\\\\n    cutout_boxes: int = 0\\\\n    cutout_fraction: float = 0.25       # box side, as a fraction of the image\\\\n    slice_reverse: float = 0.0          # probability, per series\\\\n    laterality_flip: float = 0.0        # probability, per study\\\\n\\\\n    def __post_init__(self):\\\\n        if not 0.0 <= self.slice_reverse <= 1.0:\\\\n            raise ValueError("slice_reverse must be a probability")\\\\n        if not 0.0 <= self.laterality_flip <= 1.0:\\\\n            raise ValueError("laterality_flip must be a probability")\\\\n        if not 0.0 < self.cutout_fraction <= 1.0:\\\\n            raise ValueError("cutout_fraction must be in (0, 1]")\\\\n\\\\n\\\\nREFERENCE = AugmentConfig()\\\\n\\\\n\\\\ndef augment_series(volume: np.ndarray, rng: np.random.Generator,\\\\n                   cfg: AugmentConfig = REFERENCE) -> np.ndarray:\\\\n    """(D, H, W) uint8 -> (D, H, W) uint8, one parameter draw for the series.\\\\n\\\\n    Photometric work happens in float32 and is clipped back to uint8 once at\\\\n    the end, so a scale followed by a shift does not round twice.\\\\n    """\\\\n    working = volume.astype(np.float32)\\\\n\\\\n    if cfg.intensity_scale is not None:\\\\n        low, high = cfg.intensity_scale\\\\n        working *= rng.uniform(low, high)\\\\n\\\\n    if cfg.intensity_shift:\\\\n        working += rng.uniform(-cfg.intensity_shift, cfg.intensity_shift)\\\\n\\\\n    if cfg.gamma is not None:\\\\n        low, high = cfg.gamma\\\\n        # Gamma is defined on [0, 1]; the volume is 8-bit, so scale in and out.\\\\n        working = np.clip(working, 0, 255)\\\\n        working = 255.0 * np.power(working / 255.0, rng.uniform(low, high))\\\\n\\\\n    if cfg.noise_std > 0:\\\\n        # One noise field for the whole series rather than per slice: the\\\\n        # sensor noise being imitated is spatially structured, and per-slice\\\\n        # noise is trivially averaged away by the slice pooling.\\\\n        working = working + rng.normal(0.0, cfg.noise_std,\\\\n                                       size=working.shape[1:])[None]\\\\n\\\\n    if cfg.cutout_boxes > 0:\\\\n        depth, height, width = working.shape\\\\n        box_h = max(1, int(round(height * cfg.cutout_fraction)))\\\\n        box_w = max(1, int(round(width * cfg.cutout_fraction)))\\\\n        for _ in range(cfg.cutout_boxes):\\\\n            top = int(rng.integers(0, max(1, height - box_h + 1)))\\\\n            left = int(rng.integers(0, max(1, width - box_w + 1)))\\\\n            # The SAME box on every slice: a box that moves through the stack\\\\n            # is a different occlusion per slice, which the slice pooling can\\\\n            # simply average out instead of learning around.\\\\n            working[:, top:top + box_h, left:left + box_w] = 0.0\\\\n\\\\n    if cfg.slice_reverse and rng.random() < cfg.slice_reverse:\\\\n        working = working[::-1]\\\\n\\\\n    return np.clip(working, 0, 255).astype(np.uint8)\\\\n\\\\n\\\\ndef flip_laterality(item: dict, rng: np.random.Generator) -> dict:\\\\n    """Mirror every series of a study and swap the medial/lateral labels.\\\\n\\\\n    Volumes reach here already canonicalised to the right knee, so this maps\\\\n    the study onto the left and back-labels it accordingly -- doubling the\\\\n    laterality distribution the model sees without inventing anatomy.\\\\n\\\\n    The flip axis depends on the acquisition plane, exactly as\\\\n    `dataset.mirror_volume` decides it: on a sagittal series left-right runs\\\\n    *through* the plane, so mirroring reverses the slice order; on coronal and\\\\n    axial it lies in-plane, so it flips the columns. `is_sagittal` in the\\\\n    metadata row is what distinguishes them.\\\\n\\\\n    Targets and weights are permuted by `config.LATERAL_LABEL_PAIRS`. Both, not\\\\n    just the targets: a weight says how much a finding\\\\\\\'s label is believed, and\\\\n    it belongs to the finding, so it has to travel with it.\\\\n    """\\\\n    volumes, meta = item["volumes"], item["meta"]\\\\n    flipped = np.empty_like(volumes)\\\\n    for i in range(volumes.shape[0]):\\\\n        flipped[i] = flip_volume(\\\\n            volumes[i], sagittal=bool(meta[i, SAGITTAL_COLUMN] > 0.5))\\\\n    item["volumes"] = flipped\\\\n\\\\n    order = lateral_swap_order()\\\\n    for key in ("targets", "weights"):\\\\n        if item.get(key) is not None:\\\\n            item[key] = np.asarray(item[key])[order]\\\\n\\\\n    # `is_right` in the metadata now describes the wrong side. Left as it is on\\\\n    # purpose: it is a *prediction* from the side classifier describing the\\\\n    # image as acquired, and the model reads it as conditioning rather than as\\\\n    # a label. Rewriting it here would tell the model the flip happened, which\\\\n    # is precisely the invariance the augmentation exists to teach.\\\\n    return item\\\\n\\\\n\\\\ndef augment_study(item: dict, rng: np.random.Generator,\\\\n                  cfg: AugmentConfig = REFERENCE) -> dict:\\\\n    """Apply the study-level flip, then per-series photometric work."""\\\\n    if cfg.laterality_flip and rng.random() < cfg.laterality_flip:\\\\n        item = flip_laterality(item, rng)\\\\n\\\\n    volumes = item["volumes"]\\\\n    if volumes.shape[0]:\\\\n        item["volumes"] = np.stack(\\\\n            [augment_series(volumes[i], rng, cfg)\\\\n             for i in range(volumes.shape[0])])\\\\n    return item\\\\n\\\', \\\'src/model/averaging.py\\\': \\\'"""Weight averaging: EMA, SWA, and model soups.\\\\n\\\\n**Why this module exists.** Checkpoint selection here is broken in a specific,\\\\ndiagnosable way. Training labels are mined from radiology reports and agree\\\\nwith image-derived ground truth about 82% of the time, and the validation fold\\\\nis mined from the *same* distribution. So the network first fits the structure\\\\nthe two sources share and then memorises the 18% they do not -- Arpit et al.,\\\\n"A Closer Look at Memorization in Deep Networks" (ICML 2017), and Liu et al.,\\\\n"Early-Learning Regularization Prevents Memorization of Noisy Labels" (NeurIPS\\\\n2020). Mined validation agreement keeps *rising* through that second phase,\\\\nbecause agreeing with the noise is what it measures. It is therefore a biased\\\\nselector, not merely a noisy one, and "take the best epoch on validation"\\\\nselects into the memorisation phase on purpose.\\\\n\\\\nMeasured here: mined val rose to 0.860 at epoch 9 and fell to 0.843 by epoch\\\\n20, while gold held flat at 0.864-0.869 across eleven consecutive epochs.\\\\n\\\\nThe fix is to stop selecting. Averaging weights is selection-free:\\\\n\\\\n- **EMA** (Tarvainen & Valpola, NeurIPS 2017) -- an exponential moving average\\\\n  maintained during training. Cheap, standard, and already the default in most\\\\n  detection and segmentation pipelines.\\\\n- **SWA** (Izmailov et al., UAI 2018, arXiv:1803.05407) -- an equal-weight\\\\n  average over the tail of the trajectory, which lands in a flatter and better\\\\n  generalising region than any single point on it.\\\\n- **Model soups** (Wortsman et al., ICML 2022, arXiv:2203.05482) -- average\\\\n  the weights of several finished fine-tunes instead of picking the best one\\\\n  on a held-out set. The paper\\\\\\\'s argument is directed at exactly the recipe\\\\n  failing here: "pick the individual model which performs best on a held-out\\\\n  validation set, discarding the remainder".\\\\n\\\\n**Averaging weights is only meaningful for models in the same basin.** Two\\\\nruns whose aggregation was randomly initialised from different seeds are not,\\\\nand averaging them produces a model that is worse than either -- silently, as\\\\na plausible-looking set of numbers. `assert_soupable` checks what it can\\\\n(shapes and key sets); it cannot check the basin, so the training notebook\\\\nseeds every fold identically, and the docstrings below say when a soup is\\\\nwell-posed.\\\\n\\\\n**Nothing here may select on the gold 58.** `greedy_soup` takes a caller\\\\\\\'s\\\\nscore function precisely so the choice of scoring set stays visible at the\\\\ncall site: it must be mined validation or an out-of-fold dump. Selecting a\\\\nsoup on gold would turn the project\\\\\\\'s one honest measuring stick into a\\\\nmodel-selection set, which has already cost this project an inflated figure\\\\nonce.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\\\\n\\\\nimport torch\\\\n\\\\nState = Dict[str, torch.Tensor]\\\\n\\\\n\\\\ndef _is_averageable(tensor: torch.Tensor) -> bool:\\\\n    """Floating-point tensors average; integer counters and flags do not.\\\\n\\\\n    `num_batches_tracked` on a BatchNorm is the canonical example: averaging\\\\n    it produces a fractional count, and torch stores it as int64 anyway. The\\\\n    aggregation stack is all LayerNorm so this does not arise today, but a\\\\n    convnet backbone with BatchNorm would hit it immediately.\\\\n    """\\\\n    return tensor.is_floating_point()\\\\n\\\\n\\\\ndef assert_soupable(states: Sequence[State]) -> None:\\\\n    """Refuse states that cannot be averaged elementwise.\\\\n\\\\n    Catches the two mechanical failures -- a different architecture, and a\\\\n    checkpoint saved at a different width -- before they become a tensor of\\\\n    the wrong shape several cells later.\\\\n    """\\\\n    if not states:\\\\n        raise ValueError("no states to average")\\\\n    reference = states[0]\\\\n    for i, state in enumerate(states[1:], start=1):\\\\n        if set(state) != set(reference):\\\\n            missing = sorted(set(reference) - set(state))[:3]\\\\n            extra = sorted(set(state) - set(reference))[:3]\\\\n            raise ValueError(\\\\n                f"state {i} has different keys from state 0 "\\\\n                f"(missing {missing}, extra {extra}) -- these are not the "\\\\n                "same architecture")\\\\n        for key, tensor in state.items():\\\\n            if tensor.shape != reference[key].shape:\\\\n                raise ValueError(\\\\n                    f"state {i} has {key} at {tuple(tensor.shape)} against "\\\\n                    f"{tuple(reference[key].shape)} in state 0")\\\\n\\\\n\\\\ndef average_states(states: Sequence[State],\\\\n                   weights: Optional[Sequence[float]] = None) -> State:\\\\n    """Elementwise weighted average of state dicts.\\\\n\\\\n    `weights` defaults to equal, which is SWA and the uniform soup. Averaging\\\\n    happens in float64 and is cast back to each tensor\\\\\\\'s own dtype, so a run\\\\n    of fp16 checkpoints does not accumulate rounding across a dozen members.\\\\n\\\\n    Non-floating tensors are taken from the first state rather than averaged.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if weights is None:\\\\n        weights = [1.0] * len(states)\\\\n    if len(weights) != len(states):\\\\n        raise ValueError(f"{len(weights)} weights for {len(states)} states")\\\\n    total = float(sum(weights))\\\\n    if total <= 0:\\\\n        raise ValueError("weights must sum to something positive")\\\\n\\\\n    averaged: State = {}\\\\n    for key, reference in states[0].items():\\\\n        if not _is_averageable(reference):\\\\n            averaged[key] = reference.clone()\\\\n            continue\\\\n        accumulator = torch.zeros_like(reference, dtype=torch.float64)\\\\n        for state, weight in zip(states, weights):\\\\n            accumulator += state[key].to(torch.float64) * float(weight)\\\\n        averaged[key] = (accumulator / total).to(reference.dtype)\\\\n    return averaged\\\\n\\\\n\\\\nclass WeightAverage:\\\\n    """EMA and SWA of one model\\\\\\\'s weights, maintained during training.\\\\n\\\\n    Both are accumulated from the same call so a run pays one copy of the\\\\n    weights per update and produces both artefacts. `decay` is the EMA\\\\\\\'s, and\\\\n    0.999 is the usual choice; `swa_from` is the step at which the equal-weight\\\\n    tail average starts, so a caller sets it to 75% of the planned steps to\\\\n    average over the last quarter.\\\\n\\\\n    **The averages live wherever the model does, in float32.** Accumulating on\\\\n    the host instead would copy the whole trunk across the PCIe bus on every\\\\n    optimiser step -- ~110 MB for ConvNeXt-T plus the aggregation, twice per\\\\n    step once the SWA window opens, about 11,000 times over a twenty-epoch\\\\n    run. That is a wall-clock tax on the run for no benefit; two shadows on\\\\n    the device cost ~220 MB of a 16 GB T4, which the batch has to spare.\\\\n    Float32 regardless of training precision: under `16-mixed` the parameters\\\\n    are fp32 anyway (autocast does not change them), but an fp16 accumulation\\\\n    over thousands of steps would round away exactly the small updates an EMA\\\\n    exists to keep. `export` moves the result to the host.\\\\n\\\\n    `device` overrides the placement -- pass "cpu" if a run is ever tight\\\\n    enough on device memory that 220 MB matters.\\\\n    """\\\\n\\\\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999,\\\\n                 swa_from: Optional[int] = None, device=None):\\\\n        if not 0.0 < decay < 1.0:\\\\n            raise ValueError(f"decay must be in (0, 1), got {decay}")\\\\n        self.decay = decay\\\\n        self.swa_from = swa_from\\\\n        self.steps = 0\\\\n        self.swa_count = 0\\\\n        state = model.state_dict()\\\\n        self.ema: State = {\\\\n            k: (v.detach().clone().to(torch.float32) if device is None\\\\n                else v.detach().to(device, torch.float32).clone())\\\\n            for k, v in state.items() if _is_averageable(v)}\\\\n        self.swa: State = {k: torch.zeros_like(v) for k, v in self.ema.items()}\\\\n        # Everything not averaged -- integer counters. The normalisation\\\\n        # buffers are float, so they average harmlessly to themselves.\\\\n        self._fixed: State = {k: v.detach().cpu().clone()\\\\n                              for k, v in state.items()\\\\n                              if not _is_averageable(v)}\\\\n\\\\n    @torch.no_grad()\\\\n    def update(self, model: torch.nn.Module) -> None:\\\\n        """Fold one optimiser step\\\\\\\'s weights into both averages."""\\\\n        self.steps += 1\\\\n        state = model.state_dict()\\\\n        in_window = (self.swa_from is not None and self.steps >= self.swa_from)\\\\n        if in_window:\\\\n            self.swa_count += 1\\\\n\\\\n        for key, shadow in self.ema.items():\\\\n            # Fetched once and reused by both averages: reading it per average\\\\n            # doubled the copy for every step of the SWA window.\\\\n            current = state[key].detach().to(shadow.device, torch.float32,\\\\n                                             non_blocking=True)\\\\n            shadow.mul_(self.decay).add_(current, alpha=1.0 - self.decay)\\\\n            if in_window:\\\\n                accumulator = self.swa[key]\\\\n                # Running mean, so the tail can be any length without a second\\\\n                # pass or a list of checkpoints held in memory.\\\\n                accumulator.add_((current - accumulator) / self.swa_count)\\\\n\\\\n    def ema_state(self) -> State:\\\\n        """The EMA weights, as a state dict a `KneeNet` loads.\\\\n\\\\n        Moved to the host on the way out: a checkpoint holding CUDA tensors\\\\n        reloads onto a device that may not exist in the process reading it.\\\\n        """\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.ema.items()}}\\\\n\\\\n    def swa_state(self) -> Optional[State]:\\\\n        """The tail average, or None if no step fell inside the window."""\\\\n        if self.swa_count == 0:\\\\n            return None\\\\n        return {**self._fixed, **{k: v.detach().cpu().clone()\\\\n                                  for k, v in self.swa.items()}}\\\\n\\\\n\\\\ndef greedy_soup(states: Sequence[State], score: Callable[[State], float],\\\\n                names: Optional[Sequence[str]] = None,\\\\n                tolerance: float = 0.0) -> Tuple[State, List[str]]:\\\\n    """Wortsman et al.\\\\\\\'s greedy soup: add a member only if it helps.\\\\n\\\\n    Members are ranked by their individual score, then considered in order;\\\\n    each is kept if the average *including* it scores at least as well as the\\\\n    average without it. The result is never worse than the best single member\\\\n    on the scoring set, which is what makes greedy preferable to a uniform\\\\n    soup when the members are of uneven quality.\\\\n\\\\n    `score` maps a state dict to a number that is better when larger --\\\\n    macro AUC here. **It must not read the gold 58.** Score on the mined\\\\n    validation fold or on an out-of-fold dump; the whole reason this takes a\\\\n    callable rather than a dataset is to keep that choice at the call site\\\\n    where a reviewer can see it.\\\\n\\\\n    `tolerance` lets a member in when it is within that much of the running\\\\n    best, for the case where the scoring set is too small to resolve the\\\\n    difference and averaging more members is the safer default. At 0.0 this is\\\\n    the paper\\\\\\\'s algorithm exactly.\\\\n\\\\n    Returns the souped state and the names of the members that went into it,\\\\n    in the order they were added.\\\\n    """\\\\n    assert_soupable(states)\\\\n    if names is None:\\\\n        names = [f"member{i}" for i in range(len(states))]\\\\n    if len(names) != len(states):\\\\n        raise ValueError(f"{len(names)} names for {len(states)} states")\\\\n\\\\n    ranked = sorted(zip(names, states), key=lambda pair: score(pair[1]),\\\\n                    reverse=True)\\\\n\\\\n    kept_names: List[str] = [ranked[0][0]]\\\\n    kept_states: List[State] = [ranked[0][1]]\\\\n    best = score(kept_states[0])\\\\n\\\\n    for name, state in ranked[1:]:\\\\n        candidate = average_states(kept_states + [state])\\\\n        value = score(candidate)\\\\n        if value >= best - tolerance:\\\\n            kept_names.append(name)\\\\n            kept_states.append(state)\\\\n            best = max(best, value)\\\\n\\\\n    return average_states(kept_states), kept_names\\\\n\\\\n\\\\ndef rank_average(predictions: Sequence["np.ndarray"]) -> "np.ndarray":\\\\n    """Average ensemble members by within-member rank, not by probability.\\\\n\\\\n    The metric is macro ROC-AUC, which reads only the ordering of studies\\\\n    within a finding. Averaging probabilities lets a member with a compressed\\\\n    or shifted output distribution contribute less than one with a confident\\\\n    one, even when the two rank studies identically -- the mean is sensitive to\\\\n    calibration and AUC is not. Converting each member to ranks first removes\\\\n    that: every member gets one vote per finding, whatever its scale.\\\\n\\\\n    Each member is (N studies, L findings); ranking is per finding, per member,\\\\n    with ties averaged, and the result is scaled to [0, 1] so it can be written\\\\n    into a submission unchanged. Because AUC is invariant to any increasing\\\\n    transform, that scaling costs nothing and keeps the output looking like the\\\\n    probabilities the rest of the pipeline passes around.\\\\n\\\\n    A single member is returned rank-transformed rather than untouched, so that\\\\n    a one-model and a five-model submission are produced by the same code path.\\\\n    """\\\\n    import numpy as np\\\\n\\\\n    if not predictions:\\\\n        raise ValueError("no predictions to average")\\\\n    stacked = np.stack([np.asarray(p, dtype=float) for p in predictions])\\\\n    if stacked.ndim != 3:\\\\n        raise ValueError(\\\\n            f"expected (members, studies, findings), got {stacked.shape}")\\\\n\\\\n    n_studies = stacked.shape[1]\\\\n    if n_studies < 2:\\\\n        # Ranks are meaningless for a single study, and the scaling below\\\\n        # would divide by zero.\\\\n        return stacked.mean(axis=0)\\\\n\\\\n    ranked = np.empty_like(stacked)\\\\n    for member in range(stacked.shape[0]):\\\\n        for finding in range(stacked.shape[2]):\\\\n            column = stacked[member, :, finding]\\\\n            order = np.argsort(column, kind="mergesort")\\\\n            positions = np.empty(n_studies, dtype=float)\\\\n            positions[order] = np.arange(n_studies, dtype=float)\\\\n            # Average the positions of tied values, so a member that predicts\\\\n            # a constant contributes no ordering rather than an arbitrary one.\\\\n            unique, inverse, counts = np.unique(column, return_inverse=True,\\\\n                                                return_counts=True)\\\\n            sums = np.zeros(len(unique))\\\\n            np.add.at(sums, inverse, positions)\\\\n            ranked[member, :, finding] = (sums / counts)[inverse]\\\\n\\\\n    return ranked.mean(axis=0) / (n_studies - 1)\\\\n\\\', \\\'src/model/splits.py\\\': \\\'"""Cross-validation splits and the gold holdout.\\\\n\\\\nTwo rules govern every split here, and both exist because the obvious\\\\nalternative produces a number that looks better and means less.\\\\n\\\\n**Split by study, never by series.** A study\\\\\\\'s ~5.5 series image the same knee\\\\nin the same session. A random series split puts near-duplicates on both sides\\\\nand reports an accuracy the model does not have.\\\\n\\\\n**The 58 gold studies never enter training.** They are the only radiologist\\\\nlabels in the dataset and the only honest measurement available. Everything\\\\nelse is trained on mined labels, and a model evaluated against mined labels can\\\\nscore well by reproducing the extractor\\\\\\\'s mistakes. `assert_no_gold_leakage`\\\\nis called by the training entrypoint rather than trusted to discipline.\\\\n"""\\\\n\\\\nfrom typing import Iterator, List, Optional, Sequence, Tuple\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\nN_FOLDS = 5\\\\nSEED = 0\\\\n\\\\n\\\\ndef load_gold_uids(train_csv=None) -> set:\\\\n    """Study UIDs of the 58 radiologist-labelled studies."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()]\\\\n    return set(labelled[config.ID_COL])\\\\n\\\\n\\\\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\\\\n    """Fail loudly if a gold study appears in training data.\\\\n\\\\n    Called from the training entrypoint. The failure this guards against is\\\\n    silent -- a leaked gold study inflates the one number being used to decide\\\\n    whether the model works.\\\\n    """\\\\n    leaked = set(train_uids) & gold_uids\\\\n    if leaked:\\\\n        raise ValueError(\\\\n            f"{len(leaked)} gold studies leaked into training: "\\\\n            f"{sorted(leaked)[:3]}..."\\\\n        )\\\\n\\\\n\\\\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\\\\n    """A coarse key for balancing folds across a 12-label problem.\\\\n\\\\n    Full multi-label stratification over 2^12 combinations would leave most\\\\n    cells empty at 4,349 studies. The count of positive findings captures the\\\\n    axis that matters -- studies with many findings are rare and should not\\\\n    cluster in one fold.\\\\n    """\\\\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\\\\n    return positives.clip(upper=4)\\\\n\\\\n\\\\ndef scanner_key(index: pd.DataFrame) -> pd.Series:\\\\n    """A per-study scanner identity, from the header scan\\\\\\\'s index.\\\\n\\\\n    The dataset spans 16-19 sites on five continents, and a CNN reads scanner\\\\n    signature as readily as it reads pathology. A fold split that puts the same\\\\n    scanner on both sides therefore reports an accuracy that does not survive\\\\n    a new site: the plan\\\\\\\'s figure is ~0.053 of inflation from random K-fold,\\\\n    with one measured model showing a 0.136 grouped-vs-random gap.\\\\n\\\\n    Site is not given, so manufacturer + model + field strength stands in for\\\\n    it. That is coarser than a site id -- two hospitals with the same scanner\\\\n    collapse into one group, which makes the split *more* conservative, not\\\\n    less -- and it is what the DICOM headers actually carry.\\\\n\\\\n    Takes the frame `src/data/index.py` writes (one row per instance or\\\\n    series, carrying `StudyInstanceUID`), and returns one key per study,\\\\n    indexed by study UID. A study whose headers name none of the three fields\\\\n    gets "unknown", and those all land in one group together.\\\\n    """\\\\n    fields = [c for c in ("Manufacturer", "ManufacturerModelName",\\\\n                          "MagneticFieldStrength") if c in index.columns]\\\\n    if not fields:\\\\n        raise ValueError(\\\\n            "the index carries none of Manufacturer / ManufacturerModelName / "\\\\n            f"MagneticFieldStrength; it has {list(index.columns)[:8]}")\\\\n\\\\n    # fillna after astype: pandas keeps a missing value missing through\\\\n    # astype(str) rather than rendering it "nan", and the join below would\\\\n    # then fail on a float instead of producing a key.\\\\n    joined = index[fields].astype(str).fillna("nan").agg("|".join, axis=1)\\\\n    joined = joined.where(~joined.str.fullmatch(r"(nan\\\\\\\\|?)+"), "unknown")\\\\n    # The first series scanned decides the study. A study imaged on two\\\\n    # scanners is rare and either group is defensible; taking the first keeps\\\\n    # the assignment deterministic.\\\\n    return joined.groupby(index[config.ID_COL]).first()\\\\n\\\\n\\\\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\\\\n               seed: int = SEED,\\\\n               groups: Optional[pd.Series] = None) -> pd.DataFrame:\\\\n    """Assign each study to a fold, stratified by positive-finding count.\\\\n\\\\n    Returns the frame with a `fold` column added.\\\\n\\\\n    `groups` maps study UID -> group id (see `scanner_key`). When given, a\\\\n    whole group lands in one fold: the split becomes leave-scanner-out rather\\\\n    than leave-study-out, so validation measures transfer to unseen equipment\\\\n    instead of rewarding memorised scanner signature.\\\\n\\\\n    Grouping costs balance. Groups are sized very unequally -- one scanner may\\\\n    hold a quarter of the corpus -- so groups are assigned largest-first to\\\\n    whichever fold is currently smallest, which is the standard greedy\\\\n    approximation and the best available without splitting a group. Check\\\\n    `summarise()` afterwards: badly imbalanced folds are the price of grouping,\\\\n    and worth knowing before a run rather than after.\\\\n\\\\n    **Changing this re-cuts every fold**, and every reference number on record\\\\n    (fold 1: val 0.872 / gold 0.861 / LB 0.889) belongs to the ungrouped\\\\n    split. Adopt it when the fold ensemble is rebuilt, not in the middle of an\\\\n    experiment that is being compared against those numbers.\\\\n    """\\\\n    frame = labels.copy().reset_index(drop=True)\\\\n    key = stratification_key(frame)\\\\n    rng = np.random.default_rng(seed)\\\\n\\\\n    frame["fold"] = -1\\\\n    if groups is None:\\\\n        for value in key.unique():\\\\n            rows = np.flatnonzero(key.to_numpy() == value)\\\\n            rng.shuffle(rows)\\\\n            frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\\\\n    else:\\\\n        mapped = frame[config.ID_COL].map(groups).fillna("unknown")\\\\n        sizes = mapped.value_counts()\\\\n        # Shuffle before sorting so equal-sized groups do not always fall the\\\\n        # same way, then place the largest first into the emptiest fold.\\\\n        order = sizes.sample(frac=1.0, random_state=seed).sort_values(\\\\n            ascending=False, kind="mergesort").index\\\\n        counts = np.zeros(n_folds, dtype=int)\\\\n        for group in order:\\\\n            fold = int(np.argmin(counts))\\\\n            rows = np.flatnonzero(mapped.to_numpy() == group)\\\\n            frame.loc[rows, "fold"] = fold\\\\n            counts[fold] += len(rows)\\\\n\\\\n    if (frame["fold"] < 0).any():\\\\n        raise RuntimeError("some studies were not assigned a fold")\\\\n    return frame\\\\n\\\\n\\\\ndef iter_folds(labels: pd.DataFrame,\\\\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\\\\n    """Yield (train, validation) frames for each fold."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    for fold in range(n_folds):\\\\n        validation = labels[labels["fold"] == fold]\\\\n        train = labels[labels["fold"] != fold]\\\\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\\\\n\\\\n\\\\ndef series_for_studies(manifest: pd.DataFrame,\\\\n                       study_uids: Sequence[str]) -> pd.DataFrame:\\\\n    """All cached series belonging to a set of studies."""\\\\n    wanted = set(study_uids)\\\\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\\\\n\\\\n\\\\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\\\\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\\\\n    if "fold" not in labels.columns:\\\\n        labels = make_folds(labels, n_folds=n_folds)\\\\n\\\\n    rows = []\\\\n    for fold in range(n_folds):\\\\n        subset = labels[labels["fold"] == fold]\\\\n        row = {"fold": fold, "studies": len(subset)}\\\\n        for name in config.LABELS:\\\\n            row[name] = float((subset[name] > 0.5).mean())\\\\n        rows.append(row)\\\\n    return pd.DataFrame(rows).set_index("fold")\\\\n\\\', \\\'src/labels/__init__.py\\\': \\\'\\\', \\\'src/labels/calibrate.py\\\': \\\'"""Score a report extractor against the 58 radiologist-labelled studies.\\\\n\\\\nThose 58 labels are the only trustworthy signal in the dataset, and this is\\\\ntheir highest-value use: measuring how much a mined label can be believed,\\\\nper finding. They are never used for training.\\\\n\\\\nThe per-finding AUC produced here drives the kill criterion -- a finding whose\\\\nmined labels score below KILL_THRESHOLD is noise, and the model should predict\\\\nthe training prior for it rather than learn from corrupted targets.\\\\n"""\\\\n\\\\nfrom typing import Callable, Dict, Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\n\\\\n# Below this AUC against gold, a finding\\\\\\\'s mined labels are not worth training\\\\n# on. Chosen to sit meaningfully above chance while tolerating the noise of a\\\\n# 58-sample estimate.\\\\nKILL_THRESHOLD = 0.70\\\\n\\\\n\\\\ndef load_gold(train_csv=None) -> pd.DataFrame:\\\\n    """The 58 studies carrying radiologist labels."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    labelled = frame[frame[config.LABELS[0]].notna()].copy()\\\\n    for name in config.LABELS:\\\\n        labelled[name] = labelled[name].astype(int)\\\\n    return labelled.reset_index(drop=True)\\\\n\\\\n\\\\ndef load_unlabelled(train_csv=None) -> pd.DataFrame:\\\\n    """The 4,349 studies carrying only a report."""\\\\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\\\\n    return frame[frame[config.LABELS[0]].isna()].reset_index(drop=True)\\\\n\\\\n\\\\ndef _auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via rank statistic, so sklearn stays an optional dependency."""\\\\n    positives = y_true == 1\\\\n    n_pos = int(positives.sum())\\\\n    n_neg = int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n\\\\n    # Average ranks within ties, otherwise a constant score would not score 0.5.\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)\\\\n    tie_sums = np.zeros(len(unique))\\\\n    np.add.at(tie_sums, inverse, ranks)\\\\n    ranks = (tie_sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef score_extractor(\\\\n    extract_fn: Callable[[str, str], Dict],\\\\n    gold: Optional[pd.DataFrame] = None,\\\\n) -> pd.DataFrame:\\\\n    """Per-finding reliability of an extractor against the gold studies.\\\\n\\\\n    `extract_fn(report, finding)` must return a dict with `present` (1/0/None)\\\\n    and `severity` (float or None), matching the lexicon module\\\\\\\'s contract.\\\\n    """\\\\n    gold = gold if gold is not None else load_gold()\\\\n    reports = gold["Report"].tolist()\\\\n    records = []\\\\n\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy()\\\\n        results = [extract_fn(report, name) for report in reports]\\\\n\\\\n        present = np.array([\\\\n            r["present"] if r["present"] is not None else -1 for r in results\\\\n        ])\\\\n        severity = np.array([\\\\n            r["severity"] if r["severity"] is not None else 0.0 for r in results\\\\n        ])\\\\n\\\\n        predicted = (present == 1).astype(int)\\\\n        true_pos = int(((predicted == 1) & (truth == 1)).sum())\\\\n        false_pos = int(((predicted == 1) & (truth == 0)).sum())\\\\n        false_neg = int(((predicted == 0) & (truth == 1)).sum())\\\\n\\\\n        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else np.nan\\\\n        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else np.nan\\\\n        auc = _auc(truth, severity)\\\\n\\\\n        # How often a finding goes unmentioned, and how often it is truly\\\\n        # positive when unmentioned. The second number is what fusion should\\\\n        # use as the soft target for silence -- treating it as a hard zero is\\\\n        # what injects systematic noise.\\\\n        unmentioned = present == -1\\\\n        silence_rate = float(unmentioned.mean())\\\\n        prior_if_silent = (\\\\n            float(truth[unmentioned].mean()) if unmentioned.any() else np.nan\\\\n        )\\\\n\\\\n        records.append({\\\\n            "finding": name,\\\\n            "n_positive": int(truth.sum()),\\\\n            "mention_rate": 1.0 - silence_rate,\\\\n            "precision": precision,\\\\n            "recall": recall,\\\\n            "auc": auc,\\\\n            "prior_if_silent": prior_if_silent,\\\\n            "usable": bool(auc >= KILL_THRESHOLD) if np.isfinite(auc) else False,\\\\n        })\\\\n\\\\n    return pd.DataFrame(records).set_index("finding")\\\\n\\\\n\\\\ndef format_report(table: pd.DataFrame) -> str:\\\\n    """Human-readable reliability table."""\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'n+\\\\\\\':>3s} {\\\\\\\'ment\\\\\\\':>5s} {\\\\\\\'prec\\\\\\\':>5s} {\\\\\\\'rec\\\\\\\':>5s} "\\\\n        f"{\\\\\\\'AUC\\\\\\\':>5s} {\\\\\\\'P(1|silent)\\\\\\\':>11s} {\\\\\\\'use\\\\\\\':>4s}",\\\\n        "-" * 66,\\\\n    ]\\\\n    for name, row in table.iterrows():\\\\n        lines.append(\\\\n            f"{name:18s} {int(row[\\\\\\\'n_positive\\\\\\\']):3d} "\\\\n            f"{row[\\\\\\\'mention_rate\\\\\\\']:4.0%} {row[\\\\\\\'precision\\\\\\\']:5.2f} "\\\\n            f"{row[\\\\\\\'recall\\\\\\\']:5.2f} {row[\\\\\\\'auc\\\\\\\']:5.3f} "\\\\n            f"{row[\\\\\\\'prior_if_silent\\\\\\\']:11.2f} {\\\\\\\'yes\\\\\\\' if row[\\\\\\\'usable\\\\\\\'] else \\\\\\\'NO\\\\\\\':>4s}"\\\\n        )\\\\n    lines.append("-" * 66)\\\\n    lines.append(f"macro AUC: {table[\\\\\\\'auc\\\\\\\'].mean():.3f}   "\\\\n                 f"usable findings: {int(table[\\\\\\\'usable\\\\\\\'].sum())}/{len(table)}")\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    from src.labels import lexicon\\\\n\\\\n    table = score_extractor(lexicon.extract_finding)\\\\n    print(format_report(table))\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/labels/lexicon.py\\\': \\\'"""Rule-based extraction of the twelve findings from multilingual reports.\\\\n\\\\nThe corpus spans roughly eleven languages -- English (39%), Greek, Croatian/\\\\nSerbian, Spanish, Turkish, German, Bulgarian/Russian, Dutch, French, Italian,\\\\nPortuguese. Rather than a language x finding cross-product, this module keys on\\\\nthe Greek and Latin stems that clinical vocabulary shares across all of them\\\\n(menisc- / menisk- / μηνίσκ- / мениск-), with language-specific additions only\\\\nwhere the local term is genuinely unrelated (Turkish "kıkırdak" for cartilage).\\\\n\\\\nEach finding resolves to one of three states:\\\\n\\\\n    1     an unnegated mention above the severity floor\\\\n    0     mentioned but negated, or present only below the severity floor\\\\n    None  never mentioned -- which is *not* the same as absent, because\\\\n          reports narrate positives and omit much else\\\\n\\\\nCallers must preserve the None state. Collapsing it to 0 injects systematic\\\\nnoise into exactly the findings that are least often narrated.\\\\n"""\\\\n\\\\nimport re\\\\nimport unicodedata\\\\nfrom typing import Dict, List, Optional\\\\n\\\\nfrom src import config\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Cues\\\\n# ---------------------------------------------------------------------------\\\\n# Negation, across every language present in the corpus. Matched as whole words\\\\n# within a bounded window before the finding term.\\\\nNEGATION_CUES = [\\\\n    # English / Latin-script general\\\\n    "no", "not", "without", "absent", "negative", "unremarkable", "intact",\\\\n    "normal", "preserved", "free of", "rule out", "ruled out",\\\\n    # Spanish / Portuguese\\\\n    "sin", "no", "ausencia", "ausente", "conservado", "conservada", "integro",\\\\n    "íntegro", "integra", "íntegra", "normales", "sem", "nao", "não",\\\\n    # Dutch\\\\n    "geen", "niet", "zonder", "intacte", "ongestoord",\\\\n    # German\\\\n    "kein", "keine", "keinen", "nicht", "ohne", "unauffällig", "regelrecht",\\\\n    "unauffaellig",\\\\n    # French\\\\n    "pas", "sans", "absence", "aucun", "aucune", "normale",\\\\n    # Italian\\\\n    "non", "senza", "assenza", "nella norma",\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "değil", "degil", "normaldir", "korunmuş", "korunmus",\\\\n    # Croatian / Serbian / Bosnian\\\\n    "bez", "nema", "nije", "uredan", "uredna", "uredno", "očuvan", "ocuvan",\\\\n    # Bulgarian / Russian\\\\n    "без", "нет", "няма", "не", "отсутств", "интактн", "сохранен",\\\\n    # Greek\\\\n    "χωρίς", "δεν", "ουδεμία", "ουδέν", "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Hedged or sub-threshold mentions. Ground truth was adjudicated against\\\\n# explicit severity thresholds, so "trace effusion" and "large effusion" are\\\\n# not the same label.\\\\nMINIMAL_CUES = [\\\\n    "minimal", "mínimo", "minimo", "minima", "mínima", "trace", "tiny",\\\\n    "slight", "small", "mild", "leve", "ligero", "ligera", "discreto",\\\\n    "discreta", "gering", "geringe", "klein", "kleine", "wenig", "licht",\\\\n    "lichte", "petit", "petite", "faible", "lieve", "piccolo", "az", "küçük",\\\\n    "kucuk", "hafif", "minimalan", "mali", "blag", "малък", "минимален",\\\\n    "незначител", "ελάχιστ", "μικρ", "ήπι",\\\\n]\\\\n\\\\nUNCERTAIN_CUES = [\\\\n    "possible", "possibly", "suspicious", "suspected", "probable", "probably",\\\\n    "questionable", "equivocal", "cannot exclude", "cannot be excluded",\\\\n    "may represent", "posible", "sospecha", "sospechoso", "dudoso",\\\\n    "verdacht", "möglich", "moglich", "fraglich", "mogelijk", "verdenking",\\\\n    "suspect", "possible", "suspicion", "şüpheli", "supheli", "olası", "olasi",\\\\n    "sumnja", "moguć", "mogu", "възможн", "подозр", "πιθαν", "ύποπτ",\\\\n]\\\\n\\\\n# Negation that *follows* the finding term. Turkish is verb-final, so\\\\n# "eklem ici sivi yoktur" negates with a trailing word and a backwards-only\\\\n# search misses it entirely -- that is a tenth of this corpus. English and\\\\n# several others do the same with trailing adjectives ("meniscus intact").\\\\n# Kept deliberately narrower than NEGATION_CUES: a forward search over the full\\\\n# cue list would let an unrelated "normal" in the same clause cancel a real\\\\n# finding.\\\\nPOSTFIX_NEGATION_CUES = [\\\\n    # Turkish\\\\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\\\\n    "saptanmamistir", "değil", "degil", "normaldir", "normaldi", "korunmuş",\\\\n    "korunmus", "izlenmedi",\\\\n    # English / Latin-script trailing adjectives\\\\n    "intact", "unremarkable", "preserved", "normal", "normale", "normales",\\\\n    "within normal limits", "is normal", "are normal",\\\\n    # Dutch / German\\\\n    "intacte", "unauffällig", "unauffaellig", "regelrecht", "ongestoord",\\\\n    # Croatian / Serbian\\\\n    "uredan", "uredna", "uredno", "očuvan", "ocuvan", "očuvana", "ocuvana",\\\\n    # Bulgarian / Russian\\\\n    "сохранен", "интактн", "не се визуализира", "непроменена",\\\\n    # Greek\\\\n    "φυσιολογικ", "ακέραι",\\\\n]\\\\n\\\\n# Terms that signal a torn or damaged structure, used by the ligament and\\\\n# meniscus findings where the anatomy word alone means nothing.\\\\nINJURY_STEMS = [\\\\n    "tear", "torn", "rupture", "ruptur", "disrupt", "discontinu", "avuls",\\\\n    "sprain", "lesion", "lesión", "lesao", "lesão", "injur", "insufficien",\\\\n    "rotura", "ruptura", "roto", "rota", "desgarro", "riss", "einriss",\\\\n    "scheur", "letsel", "déchirure", "dechirure", "rupture", "lacerazion",\\\\n    "yırtık", "yirtik", "rüptür", "ruptur", "kopma", "puknuć", "puknuc",\\\\n    "ruptura", "razdor", "разрыв", "руптура", "скъсване", "увреж",\\\\n    "ρήξη", "ρηξη", "ρήγμα",\\\\n]\\\\n\\\\n\\\\ndef _norm(text: str) -> str:\\\\n    """Lowercase and strip combining marks, keeping non-Latin scripts intact."""\\\\n    lowered = text.lower()\\\\n    decomposed = unicodedata.normalize("NFD", lowered)\\\\n    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))\\\\n    return unicodedata.normalize("NFC", stripped)\\\\n\\\\n\\\\ndef _stem_pattern(stems: List[str]) -> re.Pattern:\\\\n    """Compile stems into one alternation matching at a word boundary."""\\\\n    escaped = sorted((re.escape(_norm(s)) for s in stems), key=len, reverse=True)\\\\n    return re.compile(r"(?<![^\\\\\\\\W\\\\\\\\d_])(?:" + "|".join(escaped) + r")", re.UNICODE)\\\\n\\\\n\\\\n# Degenerative disease stems. The three OA findings differ only by which\\\\n# compartment the disease is localised to, so they share this anchor and are\\\\n# separated by the localiser lists below. Anchoring on the compartment word\\\\n# instead does not work: "medial" appears in almost every report describing\\\\n# normal anatomy.\\\\nOA_STEMS = [\\\\n    "osteoarthr", "osteoarthrit", "arthrosis", "arthrose", "artrosis",\\\\n    "artroz", "artroza", "gonarthros", "gonartros", "gonartroz",\\\\n    "degenerative change", "degenerative disease", "cambios degenerativos",\\\\n    "degenerativne", "degenerativ", "chondropath", "condropat", "chondromalac",\\\\n    "condromalac", "chondral loss", "chondral thinning", "cartilage loss",\\\\n    "cartilage thinning", "perdida de cartilago", "adelgazamiento del cartilago",\\\\n    "knorpelverlust", "knorpelschaden", "kraakbeenverlies", "kikirdak kayb",\\\\n    "kikirdak incelme", "hrskavice", "denudacij", "osteofit", "osteophyt",\\\\n    "espolon", "spur formation", "остеоартр", "артроз", "остеофит",\\\\n    "хондромалац", "изтънен", "οστεοαρθρ", "αρθριτ", "χονδροπαθ",\\\\n    "χονδρομαλακ", "οστεοφυτ",\\\\n]\\\\n\\\\nMEDIAL_LOCALIZERS = [\\\\n    "medial", "media", "mediale", "medialen", "medialer", "medialne",\\\\n    "medijaln", "interno", "interna", "innen", "innenseit", "binnen",\\\\n    "ic kompartman", "медиальн", "медиалн", "вътреш", "εσω", "εσωτερικ",\\\\n]\\\\n\\\\nLATERAL_LOCALIZERS = [\\\\n    "lateral", "laterale", "lateralen", "lateraler", "lateraln", "externo",\\\\n    "externa", "aussen", "buiten", "dis kompartman", "латеральн", "латералн",\\\\n    "външ", "εξω", "εξωτερικ",\\\\n]\\\\n\\\\nPF_LOCALIZERS = [\\\\n    "patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",\\\\n    "retropatellar", "retropatelar", "patellar", "patelar", "patella",\\\\n    "rotulian", "rotuliana", "rotuliano", "trochlea", "troclea",\\\\n    "femoropatelarn", "patellofemoraln", "пателлофеморальн", "пателофеморалн",\\\\n    "надколенник", "капачка", "επιγονατιδ", "επιγονατιδομηριαι",\\\\n]\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Finding definitions\\\\n# ---------------------------------------------------------------------------\\\\n# `anatomy` locates the structure; `requires_injury` marks findings where the\\\\n# structure being named is not itself abnormal (a meniscus is always present --\\\\n# only a torn one is a positive label).\\\\nFINDINGS = {\\\\n    "ACL": {\\\\n        "anatomy": ["acl", "lca", "anterior cruciate", "ligamento cruzado anterior",\\\\n                    "voorste kruisband", "vorderes kreuzband", "kreuzband",\\\\n                    "ligament croise anterieur", "legamento crociato anteriore",\\\\n                    "on capraz bag", "onc apraz", "capraz bag", "prednji ukrizeni",\\\\n                    "prednja ukrstena", "predna krstna", "передня крестообразн",\\\\n                    "предна кръстна", "крестообразн", "prosthios chiastos",\\\\n                    "προσθιο χιαστο", "χιαστο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "MCL": {\\\\n        "anatomy": ["mcl", "medial collateral", "ligamento colateral medial",\\\\n                    "ligamento lateral interno", "mediale collaterale",\\\\n                    "innenband", "mediales kollateralband", "mediale band",\\\\n                    "ligament collateral medial", "ic kollateral",\\\\n                    "medial kollateral", "medijalni kolateralni",\\\\n                    "медиальн коллатеральн", "вътрешна колатерал",\\\\n                    "εσω πλαγιο", "εσωτερικο πλαγιο"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Medial Meniscus": {\\\\n        "anatomy": ["medial menisc", "menisco medial", "menisco interno",\\\\n                    "mediale meniscus", "innenmeniskus", "medialer meniskus",\\\\n                    "menisque interne", "menisque medial", "menisco mediale",\\\\n                    "medial menisk", "ic menisk", "medijalni meniskus",\\\\n                    "medijalnog meniskusa", "медиальн мениск",\\\\n                    "вътрешния менискус", "εσω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    "Lateral Meniscus": {\\\\n        "anatomy": ["lateral menisc", "menisco lateral", "menisco externo",\\\\n                    "laterale meniscus", "aussenmeniskus", "lateraler meniskus",\\\\n                    "menisque externe", "menisque lateral", "menisco laterale",\\\\n                    "lateral menisk", "dis menisk", "lateralni meniskus",\\\\n                    "латеральн мениск", "външния менискус", "εξω μηνισκ"],\\\\n        "requires_injury": True,\\\\n    },\\\\n    # The OA family anchors on the disease stem and requires a compartment\\\\n    # localiser nearby, within a tighter window than other findings use.\\\\n    "Medial OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": MEDIAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Lateral OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": LATERAL_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "PF OA": {\\\\n        "anatomy": OA_STEMS,\\\\n        "requires_injury": False,\\\\n        "context": PF_LOCALIZERS,\\\\n        "context_window": 55,\\\\n        "context_is_localizer": True,\\\\n    },\\\\n    "Effusion": {\\\\n        "anatomy": ["effusion", "efusion", "derrame", "hydrops", "hidrartros",\\\\n                    "gelenkerguss", "erguss", "gewrichtsvocht", "vocht",\\\\n                    "epanchement", "versamento", "efuzyon", "eklem ici sivi",\\\\n                    "eklem sivisi", "izliv", "zglobni izliv", "wypot",\\\\n                    "выпот", "излив", "ставен излив", "συλλογη υγρου",\\\\n                    "αρθριτικη συλλογη", "ενδαρθρικη συλλογη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Synovitis": {\\\\n        "anatomy": ["synovit", "sinovit", "synovial thickening",\\\\n                    "engrosamiento sinovial", "synoviale verdickung",\\\\n                    "synoviale proliferation", "synovitis", "sinovitis",\\\\n                    "sinovijalis", "sinovijalna", "синовит", "υμενιτιδα",\\\\n                    "αρθρικου υμενα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Baker\\\\\\\'s": {\\\\n        "anatomy": ["baker", "popliteal cyst", "quiste poplite", "quistes poplite",\\\\n                    "bakerse cyste", "poplitealzyste", "bakerzyste",\\\\n                    "kyste poplite", "cisti poplitea", "poplitealna cista",\\\\n                    "bakerova cista", "poplitealni", "baker kisti",\\\\n                    "киста бейкера", "подколенна киста", "κυστη baker",\\\\n                    "κυστη μπεικερ", "ιγνυακη κυστη"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Contusion": {\\\\n        "anatomy": ["contusion", "contusion osea", "bone contusion",\\\\n                    "bone marrow edema", "bone marrow oedema", "marrow edema",\\\\n                    "edema oseo", "edema de medula osea", "edema medular",\\\\n                    "knochenmarksodem", "knochenodem", "knochenmarködem",\\\\n                    "beenmergoedeem", "botoedeem", "oedeme osseux",\\\\n                    "edema midollare", "kemik ilig odem", "kemik odem",\\\\n                    "kontuzyon", "kostana kontuzija", "edem kosti",\\\\n                    "костномозгов отек", "костномозъчен едем", "оστικο οιδημα",\\\\n                    "οιδημα μυελου", "μυελικο οιδημα"],\\\\n        "requires_injury": False,\\\\n    },\\\\n    "Fracture": {\\\\n        "anatomy": ["fracture", "fractura", "fraktur", "fractuur", "frattura",\\\\n                    "kirik", "kırık", "prelom", "fraktura", "perelom",\\\\n                    "перелом", "фрактура", "счупване", "καταγμα", "καταγματ"],\\\\n        "requires_injury": False,\\\\n    },\\\\n}\\\\n\\\\n# Window, in characters, searched backwards from a term for a negation cue.\\\\nNEGATION_WINDOW = 60\\\\n# Window searched around a term for severity and certainty cues.\\\\nSEVERITY_WINDOW = 45\\\\n# Window searched forward from a term for a trailing negator.\\\\nPOSTFIX_NEGATION_WINDOW = 40\\\\n# Window searched around an anatomy match for the injury or context stem.\\\\nCONTEXT_WINDOW = 90\\\\n\\\\n_NEGATION_RE = _stem_pattern(NEGATION_CUES)\\\\n_POSTFIX_NEGATION_RE = _stem_pattern(POSTFIX_NEGATION_CUES)\\\\n_MINIMAL_RE = _stem_pattern(MINIMAL_CUES)\\\\n_UNCERTAIN_RE = _stem_pattern(UNCERTAIN_CUES)\\\\n_INJURY_RE = _stem_pattern(INJURY_STEMS)\\\\n\\\\n_COMPILED = {}\\\\nfor _name, _spec in FINDINGS.items():\\\\n    _COMPILED[_name] = {\\\\n        "anatomy": _stem_pattern(_spec["anatomy"]),\\\\n        "requires_injury": _spec["requires_injury"],\\\\n        "context": _stem_pattern(_spec["context"]) if _spec.get("context") else None,\\\\n        "context_window": _spec.get("context_window", CONTEXT_WINDOW),\\\\n        "context_is_localizer": _spec.get("context_is_localizer", False),\\\\n    }\\\\n\\\\n# Clause boundaries. Negation does not carry across them, so "no tear. Effusion\\\\n# present" must not read as a negated effusion.\\\\n_CLAUSE_SPLIT = re.compile(r"[.;:\\\\\\\\n]|\\\\\\\\bbut\\\\\\\\b|\\\\\\\\bhowever\\\\\\\\b|\\\\\\\\bpero\\\\\\\\b|\\\\\\\\bmaar\\\\\\\\b|\\\\\\\\baber\\\\\\\\b")\\\\n\\\\n\\\\ndef _clause_bounds(text: str, position: int) -> tuple:\\\\n    """Start and end of the clause containing `position`."""\\\\n    start = 0\\\\n    end = len(text)\\\\n    for match in _CLAUSE_SPLIT.finditer(text):\\\\n        if match.end() <= position:\\\\n            start = match.end()\\\\n        elif match.start() > position:\\\\n            end = match.start()\\\\n            break\\\\n    return start, end\\\\n\\\\n\\\\ndef _is_negated(text: str, position: int, length: int = 0) -> bool:\\\\n    """True when a negation cue applies to the term at `position`.\\\\n\\\\n    Searches backwards over the full cue list, and forwards over the narrower\\\\n    postfix list -- verb-final languages put the negator after the finding.\\\\n    Both searches stop at the clause boundary so a negation cannot leak into a\\\\n    neighbouring statement.\\\\n    """\\\\n    clause_start, clause_end = _clause_bounds(text, position)\\\\n\\\\n    window_start = max(clause_start, position - NEGATION_WINDOW)\\\\n    if _NEGATION_RE.search(text, window_start, position):\\\\n        return True\\\\n\\\\n    term_end = position + length\\\\n    window_end = min(clause_end, term_end + POSTFIX_NEGATION_WINDOW)\\\\n    return bool(_POSTFIX_NEGATION_RE.search(text, term_end, window_end))\\\\n\\\\n\\\\ndef _severity(text: str, position: int, length: int) -> float:\\\\n    """Confidence that the mention clears the adjudication threshold.\\\\n\\\\n    1.0 for an unqualified positive, lower when the report hedges the finding\\\\n    or calls it minimal. These scores feed the fusion step as soft targets\\\\n    rather than being thresholded here.\\\\n    """\\\\n    start = max(0, position - SEVERITY_WINDOW)\\\\n    end = min(len(text), position + length + SEVERITY_WINDOW)\\\\n    window = text[start:end]\\\\n\\\\n    score = 1.0\\\\n    if _MINIMAL_RE.search(window):\\\\n        score *= 0.35\\\\n    if _UNCERTAIN_RE.search(window):\\\\n        score *= 0.5\\\\n    return score\\\\n\\\\n\\\\ndef _has_context(text: str, position: int, length: int, pattern,\\\\n                 window: int = CONTEXT_WINDOW, clause_scoped: bool = True) -> bool:\\\\n    """True when a required injury, disease, or localiser stem sits nearby.\\\\n\\\\n    Injury context is clause-scoped: a tear described in one sentence must not\\\\n    license a finding named in the next. Compartment localisers are not, because\\\\n    reports routinely name the disease in one clause and the compartment in the\\\\n    following one ("Gonarthrosis. Most marked laterally."). Enforcing clause\\\\n    scope on localisers cost Lateral OA two thirds of its recall.\\\\n    """\\\\n    if pattern is None:\\\\n        return True\\\\n\\\\n    if clause_scoped:\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n    else:\\\\n        clause_start, clause_end = 0, len(text)\\\\n\\\\n    start = max(clause_start, position - window)\\\\n    end = min(clause_end, position + length + window)\\\\n    return bool(pattern.search(text, start, end))\\\\n\\\\n\\\\ndef extract_finding(report: str, finding: str) -> Dict:\\\\n    """Resolve one finding from one report."""\\\\n    spec = _COMPILED[finding]\\\\n    text = _norm(report)\\\\n\\\\n    context_pattern = spec["context"]\\\\n    context_window = spec["context_window"]\\\\n    localizer_based = spec["context_is_localizer"]\\\\n    if spec["requires_injury"]:\\\\n        context_pattern = _INJURY_RE\\\\n        context_window = CONTEXT_WINDOW\\\\n        localizer_based = False\\\\n\\\\n    mentioned = False\\\\n    best = None\\\\n\\\\n    for match in spec["anatomy"].finditer(text):\\\\n        position, length = match.start(), match.end() - match.start()\\\\n        in_context = _has_context(text, position, length, context_pattern,\\\\n                                  context_window,\\\\n                                  clause_scoped=not localizer_based)\\\\n\\\\n        if localizer_based and not in_context:\\\\n            # Degenerative disease was described, but not in this compartment.\\\\n            # That is silence about this finding, not evidence against it --\\\\n            # counting it as a mention would report a confident zero for a\\\\n            # compartment the radiologist never discussed.\\\\n            continue\\\\n\\\\n        mentioned = True\\\\n\\\\n        if not in_context:\\\\n            # The structure is named but nothing abnormal is asserted about it,\\\\n            # which is a normal-anatomy sentence rather than a finding.\\\\n            continue\\\\n        if _is_negated(text, position, length):\\\\n            continue\\\\n\\\\n        score = _severity(text, position, length)\\\\n        clause_start, clause_end = _clause_bounds(text, position)\\\\n        candidate = {\\\\n            "present": 1,\\\\n            "severity": score,\\\\n            "span": report[clause_start:clause_end].strip()[:200],\\\\n        }\\\\n        # Keep the most confident positive mention.\\\\n        if best is None or score > best["severity"]:\\\\n            best = candidate\\\\n\\\\n    if best is not None:\\\\n        return best\\\\n    if mentioned:\\\\n        return {"present": 0, "severity": 0.0, "span": None}\\\\n    return {"present": None, "severity": None, "span": None}\\\\n\\\\n\\\\ndef extract_report(report: str) -> Dict[str, Dict]:\\\\n    """Resolve all twelve findings from one report."""\\\\n    if not isinstance(report, str) or not report.strip():\\\\n        return {name: {"present": None, "severity": None, "span": None}\\\\n                for name in config.LABELS}\\\\n    return {name: extract_finding(report, name) for name in config.LABELS}\\\\n\\\\n\\\\ndef extract_frame(reports, ids=None):\\\\n    """Vectorise extraction over a series of reports into a DataFrame.\\\\n\\\\n    Returns one row per report with `<finding>` (0/1/NaN) and\\\\n    `<finding>_severity` columns.\\\\n    """\\\\n    import pandas as pd\\\\n\\\\n    records = []\\\\n    for report in reports:\\\\n        result = extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = result[name]["present"]\\\\n            row[f"{name}_severity"] = result[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    if ids is not None:\\\\n        frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\', \\\'src/labels/fuse.py\\\': \\\'"""Combine the rule-based and LLM extractors into soft training targets.\\\\n\\\\nProduces one row per study with, for each finding, a target in [0, 1] and a\\\\nsample weight. Neither is a hard label: the extractors disagree often enough,\\\\nand are wrong often enough, that training on thresholded output would teach the\\\\nmodel to reproduce extraction errors.\\\\n\\\\n**Nothing here is fitted on the 58 gold studies.** An earlier version derived\\\\nper-finding blend weights from gold AUCs and per-finding silence priors from\\\\ngold positive rates -- 24 numbers estimated from 58 samples. It scored macro AUC\\\\n0.860 in-sample and 0.812 under leave-one-out, which is worse than the LLM\\\\nextractor alone at 0.838. The apparent gain was the fit, not the fusion.\\\\n\\\\nWhat replaced it is deliberately parameter-free:\\\\n\\\\n**A fixed blend.** Targets are 0.10 x lexicon + 0.90 x LLM for every finding.\\\\nThe ratio sits on a plateau, not an argmax: with the dual-mode LLM scores,\\\\n0.00 through 0.15 all land within 0.002 on gold (0.889-0.891) and the weight\\\\nfalls away steadily above that -- the 0.30 chosen for the categorical-only\\\\nLLM cost ~0.010 against the stronger extractor. A small nonzero weight keeps\\\\nthe lexicon\\\\\\\'s independent evidence without letting it drag the ranking.\\\\n\\\\n**Silence scores zero, and is handled by the weight instead.** Measuring\\\\nP(positive | both silent) on gold and using it as the target was worse than\\\\nusing zero, on every variant tested. The concern that motivated it -- that a\\\\nsilent finding is not a confident negative -- is real, and is addressed by\\\\ngiving those studies a low sample weight rather than a fabricated target.\\\\n\\\\n**Disagreement lowers the weight rather than picking a winner.** A study where\\\\nthe extractors conflict is genuinely ambiguous and should not pull as hard as\\\\none where they concur.\\\\n"""\\\\n\\\\nfrom pathlib import Path\\\\nfrom typing import Optional\\\\n\\\\nimport numpy as np\\\\nimport pandas as pd\\\\n\\\\nfrom src import config\\\\nfrom src.labels import calibrate, lexicon\\\\n\\\\n# Blend ratio. Chosen from a plateau, not an argmax -- see module docstring.\\\\nLEXICON_WEIGHT = 0.10\\\\nLLM_WEIGHT = 1.0 - LEXICON_WEIGHT\\\\n\\\\n# Sample weights by how much the two extractors corroborate each other.\\\\nWEIGHT_AGREE = 1.0        # both informative, same call\\\\nWEIGHT_DISAGREE = 0.4     # both informative, opposite calls\\\\nWEIGHT_SINGLE = 0.7       # one informative, one silent\\\\nWEIGHT_SILENT = 0.25      # neither informative; target is zero but barely counts\\\\n\\\\n\\\\ndef silence_rates(labels: pd.DataFrame) -> pd.Series:\\\\n    """Per finding, the share of studies neither extractor said anything about.\\\\n\\\\n    Measured on the mined corpus -- 4,349 studies -- and never on gold. That\\\\n    distinction is what separates this from the per-finding silence priors this\\\\n    module removed: those were 24 numbers fitted on 58 samples and scored 0.860\\\\n    in-sample against 0.812 leave-one-out. A rate over 4,349 studies is an\\\\n    observation, not a fit.\\\\n    """\\\\n    return pd.Series(\\\\n        {name: float((labels[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n         for name in config.LABELS})\\\\n\\\\n\\\\ndef rescale_silent_weights(labels: pd.DataFrame) -> pd.DataFrame:\\\\n    """Down-weight silence in proportion to how uninformative it is.\\\\n\\\\n    **The problem, measured on Synovitis.** 82.2% of studies are silent about\\\\n    it, the mined target averages 0.087, and the true prevalence on gold is\\\\n    46.6%. So roughly 1,666 of the 3,576 silent studies are true positives\\\\n    being trained toward zero -- and because silence is so common here, those\\\\n    studies carry 57% of the finding\\\\\\\'s gradient. The model is being taught to\\\\n    rank real positives low. Synovitis duly scores 0.682 on gold, the only\\\\n    finding where the fine-tuned model is WORSE than the retired frozen one\\\\n    (0.746), which is what fitting label noise harder looks like.\\\\n\\\\n    **The fix, and why it is not the one that failed.** Targets are untouched:\\\\n    fabricating P(positive | silent) as a target measured worse on every\\\\n    variant tested, and that stands. What changes is the *weight*, which this\\\\n    module\\\\\\\'s own docstring already names as the mechanism for silence being an\\\\n    unconfident negative. The weight becomes\\\\n\\\\n        WEIGHT_SILENT x (1 - silence_rate(finding))\\\\n\\\\n    which is one rule with no free parameters, evaluated on the mined corpus.\\\\n    Silence about ACL (9.2% of studies) stays worth nearly the full 0.25,\\\\n    because a report that discusses ligaments and omits the ACL is saying\\\\n    something. Silence about Synovitis (82.2%) falls to 0.044, because a\\\\n    corpus that mentions it in one study out of six is not saying anything.\\\\n\\\\n    Returns a copy; the target columns are passed through unchanged.\\\\n    """\\\\n    rates = silence_rates(labels)\\\\n    out = labels.copy()\\\\n    for name in config.LABELS:\\\\n        column = f"{name}_weight"\\\\n        silent = out[column] == WEIGHT_SILENT\\\\n        out.loc[silent, column] = WEIGHT_SILENT * (1.0 - rates[name])\\\\n    return out\\\\n\\\\n\\\\ndef _informative(present) -> bool:\\\\n    """False when an extractor said nothing about this finding."""\\\\n    if present is None:\\\\n        return False\\\\n    if isinstance(present, float) and np.isnan(present):\\\\n        return False\\\\n    return True\\\\n\\\\n\\\\ndef _score(present, severity) -> float:\\\\n    """One extractor\\\\\\\'s contribution to the target, in [0, 1]."""\\\\n    if not _informative(present):\\\\n        return 0.0\\\\n    if int(present) == 1:\\\\n        return float(severity) if severity == severity and severity is not None else 1.0\\\\n    return 0.0\\\\n\\\\n\\\\ndef lexicon_frame(reports, ids) -> pd.DataFrame:\\\\n    """Run the rule-based extractor over a set of reports."""\\\\n    records = []\\\\n    for report in reports:\\\\n        extracted = lexicon.extract_report(report)\\\\n        row = {}\\\\n        for name in config.LABELS:\\\\n            row[name] = extracted[name]["present"]\\\\n            row[f"{name}_severity"] = extracted[name]["severity"]\\\\n        records.append(row)\\\\n\\\\n    frame = pd.DataFrame(records)\\\\n    frame.insert(0, config.ID_COL, list(ids))\\\\n    return frame\\\\n\\\\n\\\\ndef fuse(lex: pd.DataFrame, llm: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend two extractor frames into soft targets and sample weights."""\\\\n    merged = lex.merge(llm, on=config.ID_COL, suffixes=("_lex", "_llm"))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        llm_present = merged[f"{name}_llm"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity_lex"].to_numpy()\\\\n        llm_sev = merged[f"{name}_severity_llm"].to_numpy()\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            llm_score = _score(llm_present[i], llm_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            llm_ok = _informative(llm_present[i])\\\\n            if lex_ok and llm_ok:\\\\n                agree = (lex_score > 0) == (llm_score > 0)\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\ndef fuse_scores(lex: pd.DataFrame, llm_scores: pd.DataFrame) -> pd.DataFrame:\\\\n    """Blend the lexicon frame with an already-combined LLM score frame.\\\\n\\\\n    `llm_scores` is the output of `mode_fusion.combine` (or\\\\n    `mode_fusion.from_categorical`): per finding a score in [0, 1], an\\\\n    `_informative` flag, and a `_call` boolean. The weight semantics mirror\\\\n    `fuse`: agreement compares the lexicon\\\\\\\'s call with the LLM\\\\\\\'s call.\\\\n    """\\\\n    merged = lex.merge(llm_scores, on=config.ID_COL, suffixes=("_lex", ""))\\\\n    if len(merged) != len(lex):\\\\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\\\\n\\\\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\\\\n\\\\n    for name in config.LABELS:\\\\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\\\\n        lex_sev = merged[f"{name}_severity"].to_numpy()\\\\n        llm_score = merged[name].to_numpy(dtype=float)\\\\n        llm_ok = merged[f"{name}_informative"].to_numpy(dtype=bool)\\\\n        llm_call = merged[f"{name}_call"].to_numpy(dtype=bool)\\\\n\\\\n        targets = np.empty(len(merged))\\\\n        sample_weights = np.empty(len(merged))\\\\n\\\\n        for i in range(len(merged)):\\\\n            lex_score = _score(lex_present[i], lex_sev[i])\\\\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score[i]\\\\n\\\\n            lex_ok = _informative(lex_present[i])\\\\n            if lex_ok and llm_ok[i]:\\\\n                agree = (lex_score > 0) == llm_call[i]\\\\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\\\\n            elif lex_ok or llm_ok[i]:\\\\n                sample_weights[i] = WEIGHT_SINGLE\\\\n            else:\\\\n                sample_weights[i] = WEIGHT_SILENT\\\\n\\\\n        output[name] = np.clip(targets, 0.0, 1.0)\\\\n        output[f"{name}_weight"] = sample_weights\\\\n\\\\n    return pd.DataFrame(output)\\\\n\\\\n\\\\n# Order-keeper inside the reference distribution\\\\\\\'s tie blocks. Small enough\\\\n# that the restored shape dominates the target value; the epsilon term only\\\\n# arranges studies within a block.\\\\nSHARPEN_EPSILON = 0.02\\\\n\\\\n\\\\ndef sharpen_targets(fused: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:\\\\n    """Restore `reference`\\\\\\\'s target distribution, keeping `fused`\\\\\\\'s ordering.\\\\n\\\\n    The dual-mode targets rank studies better than the categorical fusion\\\\n    (0.891 vs 0.854 on gold) but drained the confident mass out of the\\\\n    distribution -- extreme targets fell from 73% to 60% of slots and the\\\\n    model trained on them regressed on the leaderboard (0.845 -> 0.827).\\\\n    AUC is invariant under strictly monotone maps, so the ordering gain and\\\\n    the crisp shape are separable: each finding\\\\\\\'s targets are quantile-mapped\\\\n    onto the reference marginal, plus an epsilon-scaled rank term that keeps\\\\n    the map strictly increasing where the reference has ties. Sample weights\\\\n    are not touched.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    out = fused.copy()\\\\n    for name in config.LABELS:\\\\n        targets = fused[name].to_numpy(dtype=float)\\\\n        ref = np.sort(reference[name].to_numpy(dtype=float))\\\\n        ranks = mode_fusion._tied_ranks_01(targets)\\\\n        mapped = np.quantile(ref, np.clip(ranks, 0.0, 1.0))\\\\n        out[name] = np.clip(\\\\n            (mapped + SHARPEN_EPSILON * ranks) / (1.0 + SHARPEN_EPSILON),\\\\n            0.0, 1.0)\\\\n    return out\\\\n\\\\n\\\\ndef extractor_aucs(gold: pd.DataFrame, lex_gold: pd.DataFrame,\\\\n                   llm_gold: pd.DataFrame) -> pd.DataFrame:\\\\n    """Diagnostic only: how each extractor scores on gold.\\\\n\\\\n    These numbers are reported, never fed back into the targets. Letting them\\\\n    set blend weights is exactly the overfit this module was rewritten to avoid.\\\\n    """\\\\n    rows = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        lex_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(lex_gold[name], lex_gold[f"{name}_severity"])])\\\\n        llm_scores = np.array([\\\\n            _score(p, s) for p, s in\\\\n            zip(llm_gold[name], llm_gold[f"{name}_severity"])])\\\\n        rows.append({\\\\n            "finding": name,\\\\n            "lexicon_auc": calibrate._auc(truth, lex_scores),\\\\n            "llm_auc": calibrate._auc(truth, llm_scores),\\\\n        })\\\\n    return pd.DataFrame(rows).set_index("finding")\\\\n\\\\n\\\\ndef build(train_csv=None, llm_mined=None, llm_gold=None,\\\\n          probs_mined=None, probs_gold=None,\\\\n          out_path: Optional[str] = None):\\\\n    """Full pipeline: extract, fuse, and score against gold.\\\\n\\\\n    When the probability-mode parquets exist alongside the categorical ones,\\\\n    the two modes are combined by `mode_fusion` before the lexicon blend.\\\\n    Otherwise this is the original categorical-only fusion, so the pipeline\\\\n    still runs on a machine that never downloaded the probability dumps.\\\\n    """\\\\n    from src.labels import mode_fusion\\\\n\\\\n    gold = calibrate.load_gold(train_csv)\\\\n    unlabelled = calibrate.load_unlabelled(train_csv)\\\\n\\\\n    mined_frame = pd.read_parquet(\\\\n        llm_mined or (config.META_DIR / "llm_labels.parquet"))\\\\n    gold_frame = pd.read_parquet(\\\\n        llm_gold or (config.META_DIR / "llm_labels_gold.parquet"))\\\\n\\\\n    # Align to CSV order so positional comparisons stay valid.\\\\n    gold_frame = (gold_frame.set_index(config.ID_COL)\\\\n                  .loc[gold[config.ID_COL]].reset_index())\\\\n    mined_frame = (mined_frame.set_index(config.ID_COL)\\\\n                   .loc[unlabelled[config.ID_COL]].reset_index())\\\\n\\\\n    probs_mined_path = probs_mined or (config.META_DIR / "llm_probs.parquet")\\\\n    probs_gold_path = probs_gold or (config.META_DIR / "llm_probs_gold.parquet")\\\\n    use_probs = (Path(probs_mined_path).exists()\\\\n                 and Path(probs_gold_path).exists())\\\\n\\\\n    if use_probs:\\\\n        probs_mined_frame = pd.read_parquet(probs_mined_path)\\\\n        probs_gold_frame = pd.read_parquet(probs_gold_path)\\\\n        scores_mined = mode_fusion.combine(mined_frame, probs_mined_frame)\\\\n        scores_gold = mode_fusion.combine(gold_frame, probs_gold_frame)\\\\n    else:\\\\n        scores_mined = mode_fusion.from_categorical(mined_frame)\\\\n        scores_gold = mode_fusion.from_categorical(gold_frame)\\\\n\\\\n    lex_gold = lexicon_frame(gold["Report"], gold[config.ID_COL])\\\\n    lex_mined = lexicon_frame(unlabelled["Report"], unlabelled[config.ID_COL])\\\\n\\\\n    fused = fuse_scores(lex_mined, scores_mined)\\\\n    fused_gold = fuse_scores(lex_gold, scores_gold)\\\\n\\\\n    if use_probs:\\\\n        # The categorical-only fusion of the same inputs supplies the shape\\\\n        # the dual-mode targets are sharpened back onto.\\\\n        ref_mined = fuse_scores(\\\\n            lex_mined, mode_fusion.from_categorical(mined_frame))\\\\n        ref_gold = fuse_scores(\\\\n            lex_gold, mode_fusion.from_categorical(gold_frame))\\\\n        fused = sharpen_targets(fused, ref_mined)\\\\n        fused_gold = sharpen_targets(fused_gold, ref_gold)\\\\n\\\\n    aucs = extractor_aucs(gold, lex_gold, gold_frame)\\\\n\\\\n    if out_path:\\\\n        fused.to_parquet(out_path, index=False)\\\\n\\\\n    return fused, fused_gold, aucs\\\\n\\\\n\\\\ndef report(gold: pd.DataFrame, fused_gold: pd.DataFrame,\\\\n           aucs: pd.DataFrame, fused: pd.DataFrame) -> str:\\\\n    """Per-finding summary of the fused targets.\\\\n\\\\n    The fused AUC column is an honest estimate: no parameter in this module was\\\\n    estimated from the gold studies, so in-sample and held-out are the same.\\\\n    """\\\\n    lines = [\\\\n        f"{\\\\\\\'finding\\\\\\\':18s} {\\\\\\\'lexAUC\\\\\\\':>7s} {\\\\\\\'llmAUC\\\\\\\':>7s} {\\\\\\\'fusedAUC\\\\\\\':>9s} "\\\\n        f"{\\\\\\\'delta\\\\\\\':>7s} {\\\\\\\'mean w\\\\\\\':>7s} {\\\\\\\'silent\\\\\\\':>7s}",\\\\n        "-" * 72,\\\\n    ]\\\\n    fused_aucs = []\\\\n    for name in config.LABELS:\\\\n        truth = gold[name].to_numpy().astype(int)\\\\n        auc = calibrate._auc(truth, fused_gold[name].to_numpy())\\\\n        fused_aucs.append(auc)\\\\n        best_single = max(aucs.loc[name, "lexicon_auc"], aucs.loc[name, "llm_auc"])\\\\n        silent = float((fused[f"{name}_weight"] == WEIGHT_SILENT).mean())\\\\n        lines.append(\\\\n            f"{name:18s} {aucs.loc[name, \\\\\\\'lexicon_auc\\\\\\\']:7.3f} "\\\\n            f"{aucs.loc[name, \\\\\\\'llm_auc\\\\\\\']:7.3f} {auc:9.3f} "\\\\n            f"{auc - best_single:+7.3f} "\\\\n            f"{fused[f\\\\\\\'{name}_weight\\\\\\\'].mean():7.2f} {silent:6.0%}"\\\\n        )\\\\n    lines.append("-" * 72)\\\\n    lines.append(\\\\n        f"macro: fused {np.nanmean(fused_aucs):.3f}   "\\\\n        f"lexicon {aucs[\\\\\\\'lexicon_auc\\\\\\\'].mean():.3f}   "\\\\n        f"LLM {aucs[\\\\\\\'llm_auc\\\\\\\'].mean():.3f}"\\\\n    )\\\\n    return "\\\\\\\\n".join(lines)\\\\n\\\\n\\\\ndef main():\\\\n    out = config.META_DIR / "train_labels.parquet"\\\\n    fused, fused_gold, aucs = build(out_path=out)\\\\n    gold = calibrate.load_gold()\\\\n    print(report(gold, fused_gold, aucs, fused))\\\\n    print(f"\\\\\\\\nwrote {out}  {fused.shape}")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    main()\\\\n\\\', \\\'src/model/lightning.py\\\': \\\'"""Lightning wrapper for the end-to-end fine-tuning run.\\\\n\\\\nKaggle offers a T4 x2 accelerator, and the training loop is GPU-bound at\\\\n~220 images/second, so a second device is close to a free halving of\\\\nwall-clock. Lightning is here for the distributed plumbing, not for the\\\\nabstraction: the semantics below are a faithful port of the hand-written loop\\\\nin notebook 08, and any difference in the resulting score is a bug rather\\\\nthan a design choice.\\\\n\\\\n**Everything in this module lives here rather than in a notebook cell.**\\\\nLightning\\\\\\\'s `ddp_notebook` strategy forks, so children inherit memory and the\\\\nmodule is not pickled on that path -- but dataloader workers and the spawn\\\\nstart method both do pickle, and a class defined in a notebook cell belongs to\\\\n`__main__` and does not survive that. Keeping these definitions importable as\\\\n`src.model.lightning` makes the module correct under either start method; the\\\\nbootstrap writes the files to disk before importing them. The same rule put\\\\nthe cache\\\\\\\'s process-pool worker in `src/data/cache.py`.\\\\n\\\\nA related consequence of fork, and the reason the notebook is careful about\\\\nit: **CUDA must not be initialised before `Trainer.fit`**. A fork inheriting a\\\\nlive CUDA context fails in the child. `torch.cuda.get_device_name` and\\\\n`get_device_properties` both call `_lazy_init`; `device_count` and\\\\n`is_available` do not.\\\\n\\\\nThree DDP hazards are handled explicitly, because each one runs clean and\\\\nsilently scores worse:\\\\n\\\\n**Memory-mapped shards are never serialised.** The cache is ~29 GB across\\\\nthree memmaps. Holding them as attributes would carry the array bytes into any\\\\nprocess the dataset is pickled into. They are opened lazily per process and\\\\ndropped from `__getstate__`, which also means each dataloader worker maps the\\\\nfiles itself rather than sharing a handle across a fork.\\\\n\\\\n**AUC is computed over gathered predictions.** It is a ranking metric, so\\\\naveraging two per-rank AUCs is not the AUC of the whole set. Predictions and\\\\ntargets are all-gathered before scoring. The distributed sampler pads the\\\\nfinal batch by repeating samples, so at most one study per rank is counted\\\\ntwice out of ~870 -- below the resolution of the measurement.\\\\n\\\\n**The effective batch is held constant.** Each rank receives `batch_size`\\\\nstudies, so the effective batch is `batch_size x devices x accumulation`.\\\\nHalving accumulation when moving from one device to two keeps the optimiser\\\\nseeing the same eight studies per step, and keeps the OneCycleLR step count\\\\nunchanged.\\\\n"""\\\\n\\\\nfrom typing import Dict, List, Optional, Sequence\\\\n\\\\nimport numpy as np\\\\nimport torch\\\\nfrom torch.utils.data import Dataset\\\\n\\\\nfrom src import config\\\\nfrom src.model import augment as augment_mod\\\\nfrom src.model import averaging\\\\nfrom src.model import conditional\\\\nfrom src.model import dataset as knee_dataset\\\\nfrom src.model import net\\\\n\\\\ntry:  # Lightning 2.x ships under both names depending on the install.\\\\n    import lightning.pytorch as pl\\\\nexcept ImportError:  # pragma: no cover - exercised only on Kaggle/Colab\\\\n    import pytorch_lightning as pl\\\\n\\\\n\\\\ndef auc(y_true: np.ndarray, scores: np.ndarray) -> float:\\\\n    """ROC AUC via the rank statistic, with ties averaged.\\\\n\\\\n    Duplicated from `labels.calibrate` rather than imported: only the model\\\\n    package is materialised into the training notebook, and a metric that\\\\n    disagrees with the one the labels were scored by would be worse than the\\\\n    duplication.\\\\n    """\\\\n    y_true = np.asarray(y_true)\\\\n    scores = np.asarray(scores, dtype=float)\\\\n    positives = y_true == 1\\\\n    n_pos, n_neg = int(positives.sum()), int((~positives).sum())\\\\n    if n_pos == 0 or n_neg == 0:\\\\n        return float("nan")\\\\n\\\\n    order = np.argsort(scores, kind="mergesort")\\\\n    ranks = np.empty(len(scores), dtype=float)\\\\n    ranks[order] = np.arange(1, len(scores) + 1)\\\\n    unique, inverse, counts = np.unique(scores, return_inverse=True,\\\\n                                        return_counts=True)\\\\n    sums = np.zeros(len(unique))\\\\n    np.add.at(sums, inverse, ranks)\\\\n    ranks = (sums / counts)[inverse]\\\\n\\\\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\\\\n\\\\n\\\\ndef macro_auc(truth: np.ndarray, probs: np.ndarray) -> float:\\\\n    """Mean per-finding AUC, ignoring findings with no positives present."""\\\\n    scores = [auc(truth[:, i], probs[:, i]) for i in range(truth.shape[1])]\\\\n    return float(np.nanmean(scores))\\\\n\\\\n\\\\nclass StudyVolumeDataset(Dataset):\\\\n    """Study volumes with an even slice subsample and intensity jitter.\\\\n\\\\n    Slices are subsampled evenly rather than at random so a validation score\\\\n    is reproducible. Horizontal flips are deliberately not used: after\\\\n    canonicalisation a flip re-mirrors the knee and swaps medial with lateral,\\\\n    silently corrupting four of the twelve labels.\\\\n\\\\n    **`resample` is the training-only exception, and it exists because the\\\\n    cache is larger than what training reads.** The cache holds 24 slices of\\\\n    each of 24,371 series -- 584,904 slice-images. A run at 16 slices x 5\\\\n    series reads 341,728 of them, the same ones every epoch: the fixed\\\\n    `keep` discards 8 slices of every series and the deterministic series cap\\\\n    discards 3,013 series outright. The remaining 42% is not new data, but it\\\\n    is unseen pixels of the same 4,349 studies, and reaching it costs no\\\\n    storage, no upload and no extra forward passes. `resample=False`\\\\n    reproduces every measured result exactly; the flag is what makes the two\\\\n    comparable.\\\\n    """\\\\n\\\\n    def __init__(self, study_index, studies: Sequence[str],\\\\n                 cache_paths: Sequence[str], slices_per_series: int,\\\\n                 augment=False, resample: bool = False):\\\\n        self.index = study_index\\\\n        self.studies = list(studies)\\\\n        self.cache_paths = list(cache_paths)\\\\n        self.slices_per_series = slices_per_series\\\\n        # Per-epoch resampling of *which* slices and series are read. Training\\\\n        # only -- a validation score has to be reproducible to be a gate.\\\\n        self.resample = resample\\\\n        # `augment` is either a bool -- the legacy single intensity scale that\\\\n        # every measured result was trained under -- or an AugmentConfig. The\\\\n        # bool path is kept byte-for-byte rather than expressed as a config so\\\\n        # that reproducing an old run does not depend on the two being\\\\n        # equivalent. See src/model/augment.py.\\\\n        self.augment = augment\\\\n        self.keep = np.linspace(0, config.CACHE_SLICES - 1,\\\\n                                slices_per_series).round().astype(int)\\\\n        self._shards: Optional[Dict[str, np.ndarray]] = None\\\\n\\\\n    @property\\\\n    def shards(self) -> Dict[str, np.ndarray]:\\\\n        """Open the memmaps on first use, once per process."""\\\\n        if self._shards is None:\\\\n            self._shards = {path: np.load(path, mmap_mode="r")\\\\n                            for path in self.cache_paths}\\\\n        return self._shards\\\\n\\\\n    def __getstate__(self):\\\\n        # Never serialise the memmaps into a spawned process or a dataloader\\\\n        # worker; each one reopens them from the paths instead.\\\\n        state = self.__dict__.copy()\\\\n        state["_shards"] = None\\\\n        return state\\\\n\\\\n    def __len__(self) -> int:\\\\n        return len(self.studies)\\\\n\\\\n    def _resampled_keep(self, available: int, rng) -> np.ndarray:\\\\n        """A fresh even-ish slice subsample: drop one slice per even group.\\\\n\\\\n        Phrased as a drop rather than a pick so the result is exactly\\\\n        `slices_per_series` distinct indices in ascending order. Order matters:\\\\n        `stack_2p5d` hands the backbone each slice with its two neighbours, and\\\\n        a shuffled stack makes those neighbours meaningless -- the same reason\\\\n        `augment.py` implements `slice_reverse` and not slice permutation.\\\\n        Coverage matches `np.linspace` in expectation, so this is a resample of\\\\n        the reference geometry rather than a different one.\\\\n        """\\\\n        n_drop = available - self.slices_per_series\\\\n        groups = np.array_split(np.arange(available), n_drop)\\\\n        drop = {int(rng.choice(group)) for group in groups}\\\\n        return np.array([i for i in range(available) if i not in drop])\\\\n\\\\n    def __getitem__(self, i: int):\\\\n        rng = np.random.default_rng(np.random.randint(0, 2**31))\\\\n        item = knee_dataset.load_study(\\\\n            self.index, self.studies[i], self.shards,\\\\n            series_rng=rng if self.resample else None)\\\\n\\\\n        available = item["volumes"].shape[1]\\\\n        if available > self.slices_per_series:\\\\n            # A thinned cache is already at the target count; only subsample a\\\\n            # full one, or the second cut would drop slices twice.\\\\n            keep = (self._resampled_keep(available, rng) if self.resample\\\\n                    else np.linspace(0, available - 1,\\\\n                                     self.slices_per_series).round().astype(int))\\\\n            item["volumes"] = item["volumes"][:, keep]\\\\n        elif available < self.slices_per_series:\\\\n            # The attention pool masks over the slice axis, so a short volume\\\\n            # trains happily on fewer slices than configured and says nothing.\\\\n            raise ValueError(\\\\n                f"cache supplies {available} slices, slices_per_series is "\\\\n                f"{self.slices_per_series} -- attach the full 24-slice "\\\\n                "shards, or lower the setting")\\\\n\\\\n        if isinstance(self.augment, augment_mod.AugmentConfig):\\\\n            # The same generator that chose the slices and series, so one item\\\\n            # draws from one stream. Its seed comes from the worker\\\\\\\'s numpy\\\\n            # state, which advances rather than resetting between epochs --\\\\n            # that is what makes the resample per-epoch and not per-study.\\\\n            item = augment_mod.augment_study(item, rng, self.augment)\\\\n        elif self.augment:\\\\n            # Legacy path: one intensity scale for the whole study. Every\\\\n            # result on record was trained through exactly this.\\\\n            scale = np.random.uniform(0.9, 1.1)\\\\n            item["volumes"] = np.clip(\\\\n                item["volumes"].astype(np.float32) * scale,\\\\n                0, 255).astype(np.uint8)\\\\n        return item\\\\n\\\\n\\\\nclass KneeLitModule(pl.LightningModule):\\\\n    """The fine-tuning step, loss, schedule, and epoch-level scoring.\\\\n\\\\n    Two validation dataloaders are expected, and which is which matters:\\\\n    index 0 is the validation fold scored against mined labels binarised at\\\\n    0.5, and it is the only thing allowed to select a checkpoint. Index 1 is\\\\n    the gold 58, logged every epoch because it is nearly free and shows\\\\n    whether the run is converging -- and never monitored. Selecting on gold\\\\n    turns the project\\\\\\\'s one honest measuring stick into a model-selection set;\\\\n    an earlier version of this loop did exactly that and inflated its own\\\\n    reported figure by an estimated 0.01-0.03.\\\\n    """\\\\n\\\\n    def __init__(self, backbone: torch.nn.Module, feature_dim: int,\\\\n                 lr: float = 1e-4, weight_decay: float = 1e-4,\\\\n                 total_steps: int = 1000, max_lr_multiplier: float = 3.0,\\\\n                 report_dim: Optional[int] = None,\\\\n                 lambda_report: float = 0.0, lambda_rank: float = 0.0,\\\\n                 rank_queue_size: int = 256, rank_min_gap: float = 0.25,\\\\n                 aux_mode: str = "fixed", aux_gate: str = "binary",\\\\n                 backbone_lr_scale: float = 1.0,\\\\n                 input_mean=None, input_std=None, gated_pool: bool = False,\\\\n                 schedule: str = "onecycle",\\\\n                 warmup_steps: Optional[int] = None,\\\\n                 label_smoothing: float = 0.0, symmetric_beta: float = 0.0,\\\\n                 pos_weight: Optional[Sequence[float]] = None,\\\\n                 lambda_stratified_rank: float = 0.0,\\\\n                 stratified_findings: Sequence[str] = ("Synovitis",),\\\\n                 stratify_by: str = "Effusion",\\\\n                 stratified_caliper: float = 0.15,\\\\n                 lambda_coupling: float = 0.0,\\\\n                 coupling_target: Optional[float] = None,\\\\n                 residual_beta: Optional[float] = None,\\\\n                 classwise_pool: bool = False,\\\\n                 slice_transformer: bool = False):\\\\n        super().__init__()\\\\n        if classwise_pool and (lambda_report > 0 or aux_mode != "fixed"):\\\\n            raise ValueError("classwise pooling requires fixed mode without report loss")\\\\n        if aux_mode not in ("fixed", "cosine"):\\\\n            raise ValueError(f"aux_mode must be fixed|cosine, got {aux_mode!r}")\\\\n        if aux_gate not in ("binary", "weighted"):\\\\n            raise ValueError(f"aux_gate must be binary|weighted, got {aux_gate!r}")\\\\n        if schedule not in ("onecycle", "cosine"):\\\\n            raise ValueError(\\\\n                f"schedule must be onecycle|cosine, got {schedule!r}")\\\\n        if warmup_steps is not None and not 0 <= warmup_steps < total_steps:\\\\n            raise ValueError(\\\\n                f"warmup_steps {warmup_steps} must be under total_steps "\\\\n                f"{total_steps}")\\\\n        if lambda_coupling > 0 and coupling_target is None:\\\\n            raise ValueError(\\\\n                "lambda_coupling needs an explicit coupling_target. The "\\\\n                "findings genuinely co-occur (gold OR 6.09, p = 0.003); a "\\\\n                "penalty toward independence optimises to a value the truth "\\\\n                "contradicts. Pass the observed coupling.")\\\\n        for name in tuple(stratified_findings) + (stratify_by,):\\\\n            if name not in config.LABELS:\\\\n                raise ValueError(f"unknown finding {name!r}")\\\\n\\\\n        self.save_hyperparameters(ignore=["backbone"])\\\\n        # Stage 1\\\\\\\'s residual parameterisation, built into the model so it\\\\n        # travels in the checkpoint rather than having to be reconstructed at\\\\n        # every scoring site. See KneeNet.couple.\\\\n        residual_pairs = None\\\\n        if residual_beta is not None:\\\\n            residual_pairs = [(config.LABELS.index(name),\\\\n                               config.LABELS.index(stratify_by))\\\\n                              for name in stratified_findings]\\\\n        self.model = net.KneeNet(backbone, feature_dim,\\\\n                                 input_mean=input_mean, input_std=input_std,\\\\n                                 gated_pool=gated_pool,\\\\n                                 classwise_pool=classwise_pool,\\\\n                                 slice_transformer=slice_transformer,\\\\n                                 residual_pairs=residual_pairs,\\\\n                                 residual_beta=residual_beta)\\\\n        self.backbone_lr_scale = backbone_lr_scale\\\\n        self.lr = lr\\\\n        self.weight_decay = weight_decay\\\\n        self.total_steps = total_steps\\\\n        self.max_lr_multiplier = max_lr_multiplier\\\\n        self.lambda_report = lambda_report\\\\n        self.lambda_rank = lambda_rank\\\\n        self.rank_queue_size = rank_queue_size\\\\n        self.rank_min_gap = rank_min_gap\\\\n        self.aux_mode = aux_mode\\\\n        self.aux_gate = aux_gate\\\\n        self.schedule = schedule\\\\n        self.warmup_steps = warmup_steps\\\\n        self.label_smoothing = label_smoothing\\\\n        self.symmetric_beta = symmetric_beta\\\\n        self.lambda_stratified_rank = lambda_stratified_rank\\\\n        self.stratified_findings = tuple(stratified_findings)\\\\n        self.stratify_by = stratify_by\\\\n        self.stratified_index = config.LABELS.index(stratify_by)\\\\n        self.stratified_caliper = stratified_caliper\\\\n        self.lambda_coupling = lambda_coupling\\\\n        self.coupling_target = coupling_target\\\\n\\\\n        # A buffer rather than a plain tensor: it has to follow the module to\\\\n        # the GPU, and it is not learned. Registered as None when unused so\\\\n        # the state dict is unchanged for every run that does not set it --\\\\n        # and it lives on the wrapper, not on `KneeNet`, so it never reaches\\\\n        # the exported checkpoint the submission loads.\\\\n        if pos_weight is not None:\\\\n            self.register_buffer(\\\\n                "pos_weight", torch.tensor(list(pos_weight), dtype=torch.float32))\\\\n        else:\\\\n            self.register_buffer("pos_weight", None)\\\\n\\\\n        # The report head is an attribute of the *wrapper*, deliberately: the\\\\n        # exported state dict keeps only `model.`-prefixed keys, so auxiliary\\\\n        # parameters vanish at export and notebook 10 keeps loading a bare\\\\n        # KneeNet. Created only when actually trained against -- under DDP a\\\\n        # parameter that never receives gradient is an error, not a no-op.\\\\n        if report_dim is not None and lambda_report > 0:\\\\n            self.report_head = torch.nn.Sequential(\\\\n                torch.nn.LayerNorm(feature_dim),\\\\n                torch.nn.Linear(feature_dim, report_dim))\\\\n        else:\\\\n            self.report_head = None\\\\n\\\\n        # Cross-batch queue for the ranking loss. At one study per rank there\\\\n        # are no pairs inside a batch, so current logits are ranked against\\\\n        # detached logits from recent steps (XBM-style). Slightly stale scores\\\\n        # are the price; per-rank and reset each epoch.\\\\n        self._rank_queue: List[Dict[str, torch.Tensor]] = []\\\\n\\\\n        # Not `_buffers`: nn.Module reserves that name for registered\\\\n        # buffers, and shadowing it corrupts state_dict and .to(device).\\\\n        self._val_outputs: Dict[int, List[Dict[str, torch.Tensor]]] = {0: [], 1: []}\\\\n\\\\n    def forward(self, volumes, meta, mask):\\\\n        return self.model(volumes, meta, mask)\\\\n\\\\n    def on_train_epoch_start(self):\\\\n        self._rank_queue = []\\\\n\\\\n    def _report_loss(self, pooled: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Cosine distance between the projected study embedding and the\\\\n        report\\\\\\\'s sentence embedding, over studies that have one.\\\\n\\\\n        The head must run even when the whole batch lacks reports: DDP\\\\n        requires every parameter to join every backward pass.\\\\n        """\\\\n        predicted = self.report_head(pooled.float())\\\\n        if "report" not in batch:\\\\n            return predicted.sum() * 0.0\\\\n        predicted = torch.nn.functional.normalize(predicted, dim=-1)\\\\n        target = torch.nn.functional.normalize(batch["report"].float(), dim=-1)\\\\n        distance = 1.0 - (predicted * target).sum(dim=-1)\\\\n        mask = batch["report_mask"].float()\\\\n        return (distance * mask).sum() / mask.sum().clamp(min=1.0)\\\\n\\\\n    def _rank_reference(self):\\\\n        """The queued detached scores both ranking losses rank against."""\\\\n        if not self._rank_queue:\\\\n            return None\\\\n        return {key: torch.cat([q[key] for q in self._rank_queue])\\\\n                for key in ("scores", "targets", "weights", "strata")}\\\\n\\\\n    def _rank_push(self, logits: torch.Tensor, batch) -> None:\\\\n        """Add this step\\\\\\\'s studies to the queue, once per training step.\\\\n\\\\n        Pushed after every ranking term has been computed, so a study never\\\\n        ranks against itself. `strata` is the predicted probability of the\\\\n        stratifying finding -- effusion -- carried because the stratified loss\\\\n        needs the reference studies\\\\\\\' effusion scores to decide which pairs are\\\\n        comparable, and recomputing them would mean a second forward pass over\\\\n        studies that have already left the batch.\\\\n        """\\\\n        self._rank_queue.append({\\\\n            "scores": logits.detach().float(),\\\\n            "targets": batch["targets"].detach().float(),\\\\n            "weights": batch["weights"].detach().float(),\\\\n            "strata": torch.sigmoid(\\\\n                logits[:, self.stratified_index].detach().float()),\\\\n        })\\\\n        while sum(len(q["scores"]) for q in self._rank_queue) > self.rank_queue_size:\\\\n            self._rank_queue.pop(0)\\\\n\\\\n    def _rank_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        return net.pairwise_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _stratified_loss(self, logits: torch.Tensor, batch) -> torch.Tensor:\\\\n        """Stage 1: rank only within an effusion stratum.\\\\n\\\\n        Conditional AUC is within-stratum pair concordance, so this optimises\\\\n        the metric the finding is judged on rather than a proxy for it. Pairs\\\\n        are matched on the model\\\\\\\'s own predicted effusion score, detached: the\\\\n        mined effusion label has its own error, and a hard split would inherit\\\\n        it at the boundary.\\\\n        """\\\\n        scores = logits.float()\\\\n        reference = self._rank_reference()\\\\n        if reference is None:\\\\n            return scores.sum() * 0.0\\\\n        strata = torch.sigmoid(scores[:, self.stratified_index].detach())\\\\n        return net.stratified_rank_loss(\\\\n            scores, batch["targets"].float(),\\\\n            reference["scores"], reference["targets"],\\\\n            strata, reference["strata"],\\\\n            self.stratified_findings, self.stratified_caliper,\\\\n            batch["weights"].float(), reference["weights"],\\\\n            min_gap=self.rank_min_gap)\\\\n\\\\n    def _coupling_loss(self, logits: torch.Tensor) -> torch.Tensor:\\\\n        """Hold the head-to-head coupling at the truth\\\\\\\'s value, not at zero."""\\\\n        reference = self._rank_reference()\\\\n        queued = (reference["scores"] if reference is not None\\\\n                  else logits.new_zeros((0, logits.shape[1])))\\\\n        total = logits.sum() * 0.0\\\\n        for name in self.stratified_findings:\\\\n            total = total + net.coupling_penalty(\\\\n                logits.float(), queued, config.LABELS.index(name),\\\\n                self.stratified_index, self.coupling_target)\\\\n        return total\\\\n\\\\n    def _aux_cosine(self, main_loss: torch.Tensor, aux_loss: torch.Tensor,\\\\n                    pooled: torch.Tensor) -> torch.Tensor:\\\\n        """cos between the two losses\\\\\\\' gradients at the shared bottleneck.\\\\n\\\\n        Du et al., "Adapting Auxiliary Losses Using Gradient Similarity"\\\\n        (arXiv:1812.02224), Algorithm 1: apply the auxiliary gradient to the\\\\n        shared parameters only while it points into the same half-space as\\\\n        the main gradient, which guarantees convergence to the main task\\\\\\\'s\\\\n        critical points and blocks negative transfer.\\\\n\\\\n        The paper measures that cosine over the shared parameters theta.\\\\n        Here every shared parameter is reached *only* through `pooled` --\\\\n        `forward_pooled` returns `head(pooled), pooled`, and both losses hang\\\\n        off that one tensor -- so the shared gradients are J^T a and J^T b for\\\\n        the same Jacobian J = d pooled/d theta. Taking the cosine at the\\\\n        bottleneck asks the same question ("does the auxiliary want to move\\\\n        the shared representation the way the main task does?") for the price\\\\n        of two backward passes through the two small heads instead of two\\\\n        through ConvNeXt. It is a proxy, not an identity: J^T can rotate the\\\\n        pair, so agreement here is not arithmetically agreement at theta.\\\\n        The sign is what Algorithm 1 consumes, and the paper itself keeps\\\\n        only the sign because minibatch cosines are too noisy to trust\\\\n        beyond it.\\\\n\\\\n        retain_graph is required: the real backward still has to run\\\\n        afterwards over the same graph.\\\\n\\\\n        Both losses are scaled by a constant first. Training runs in\\\\n        `16-mixed`, and these probe gradients bypass Lightning\\\\\\\'s GradScaler,\\\\n        so an unscaled fp16 backward can flush small values to zero -- which\\\\n        reads as cosine 0, i.e. "aligned", i.e. silently back to a fixed\\\\n        lambda. Cosine is scale-invariant, so scaling both sides costs\\\\n        nothing and keeps the values off the floor.\\\\n        """\\\\n        scale = 1024.0\\\\n        grad_main, = torch.autograd.grad(main_loss * scale, pooled,\\\\n                                         retain_graph=True)\\\\n        grad_aux, = torch.autograd.grad(aux_loss * scale, pooled,\\\\n                                        retain_graph=True)\\\\n        cosine = torch.nn.functional.cosine_similarity(\\\\n            grad_main.flatten().float(), grad_aux.flatten().float(), dim=0)\\\\n        # Ranks must agree, or the "binary" gate is silently a soft one: DDP\\\\n        # averages the two ranks\\\\\\\' gradients, so a per-rank decision would\\\\n        # apply the auxiliary at half strength whenever they disagreed.\\\\n        # torch.distributed rather than self.all_gather, which requires an\\\\n        # attached Trainer and would make this untestable in isolation.\\\\n        if torch.distributed.is_available() and torch.distributed.is_initialized():\\\\n            cosine = cosine.clone()\\\\n            torch.distributed.all_reduce(cosine, op=torch.distributed.ReduceOp.SUM)\\\\n            cosine = cosine / torch.distributed.get_world_size()\\\\n        return cosine\\\\n\\\\n    def _aux_weight(self, cosine: torch.Tensor) -> torch.Tensor:\\\\n        """Algorithm 1 (binary) or Algorithm 2 (cosine-weighted)."""\\\\n        if self.aux_gate == "weighted":\\\\n            return cosine.clamp(min=0.0)\\\\n        return (cosine >= 0).to(cosine.dtype)\\\\n\\\\n    def training_step(self, batch, batch_idx):\\\\n        logits, pooled = self.model.forward_pooled(\\\\n            batch["volumes"], batch["meta"], batch["mask"])\\\\n        loss = net.weighted_bce(\\\\n            logits, batch["targets"], batch["weights"],\\\\n            label_smoothing=self.label_smoothing,\\\\n            pos_weight=self.pos_weight,\\\\n            symmetric_beta=self.symmetric_beta)\\\\n        self.log("bce_loss", loss, on_step=False, on_epoch=True,\\\\n                 sync_dist=True, batch_size=len(logits))\\\\n        main_loss = loss\\\\n\\\\n        if self.report_head is not None:\\\\n            report_loss = self._report_loss(pooled, batch)\\\\n            weight = self.lambda_report\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, report_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                # The diagnostic worth having even when the gate never fires:\\\\n                # a report loss that is mostly anti-aligned explains a fixed\\\\n                # lambda hurting the leaderboard while helping validation.\\\\n                self.log("report_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n                self.log("report_gate", (cosine >= 0).float(), on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * report_loss\\\\n            self.log("report_loss", report_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0:\\\\n            rank_loss = self._rank_loss(logits, batch)\\\\n            weight = self.lambda_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, rank_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("rank_cosine", cosine, on_step=False, on_epoch=True,\\\\n                         sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * rank_loss\\\\n            self.log("rank_loss", rank_loss, on_step=False, on_epoch=True,\\\\n                     sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_stratified_rank > 0:\\\\n            stratified_loss = self._stratified_loss(logits, batch)\\\\n            weight = self.lambda_stratified_rank\\\\n            if self.aux_mode == "cosine":\\\\n                cosine = self._aux_cosine(main_loss, stratified_loss, pooled)\\\\n                weight = weight * self._aux_weight(cosine)\\\\n                self.log("stratified_cosine", cosine, on_step=False,\\\\n                         on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n            loss = loss + weight * stratified_loss\\\\n            self.log("stratified_rank_loss", stratified_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_coupling > 0:\\\\n            coupling_loss = self._coupling_loss(logits)\\\\n            loss = loss + self.lambda_coupling * coupling_loss\\\\n            self.log("coupling_loss", coupling_loss, on_step=False,\\\\n                     on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n\\\\n        if self.lambda_rank > 0 or self.lambda_stratified_rank > 0 \\\\\\\\\\\\n                or self.lambda_coupling > 0:\\\\n            self._rank_push(logits, batch)\\\\n\\\\n        self.log("train_loss", loss, prog_bar=True, on_step=True,\\\\n                 on_epoch=True, sync_dist=True, batch_size=len(logits))\\\\n        return loss\\\\n\\\\n    def validation_step(self, batch, batch_idx, dataloader_idx=0):\\\\n        logits = self(batch["volumes"], batch["meta"], batch["mask"])\\\\n        self._val_outputs[dataloader_idx].append({\\\\n            "probs": torch.sigmoid(logits.float()),\\\\n            "targets": batch["targets"].float(),\\\\n        })\\\\n\\\\n    def _gathered(self, dataloader_idx: int):\\\\n        """All-gather one buffer and return (truth, probs) as numpy."""\\\\n        entries = self._val_outputs[dataloader_idx]\\\\n        if not entries:\\\\n            return None, None\\\\n        probs = torch.cat([e["probs"] for e in entries])\\\\n        targets = torch.cat([e["targets"] for e in entries])\\\\n\\\\n        # all_gather returns (world_size, N, L) when every rank holds the same\\\\n        # N, which the distributed sampler guarantees by padding.\\\\n        probs = self.all_gather(probs)\\\\n        targets = self.all_gather(targets)\\\\n        if probs.ndim == 3:\\\\n            probs = probs.reshape(-1, probs.shape[-1])\\\\n            targets = targets.reshape(-1, targets.shape[-1])\\\\n        return (targets.cpu().numpy(), probs.cpu().numpy())\\\\n\\\\n    def _log_conditional(self, prefix: str, truth, probs) -> None:\\\\n        """Log conditional AUC beside the pooled number, per stratified finding.\\\\n\\\\n        The plan\\\\\\\'s §3, applied where it is cheapest to apply: pooled AUC on\\\\n        synovitis is reproducible from the effusion label alone (0.698 against\\\\n        the head\\\\\\\'s 0.701), so an epoch-level pooled number cannot show whether\\\\n        the run is learning the finding or learning effusion. Conditional AUC\\\\n        is the same statistic restricted to pairs effusion does not already\\\\n        separate.\\\\n\\\\n        Stratified on the model\\\\\\\'s own predicted effusion, split at its median\\\\n        -- there is no gold effusion label in the mined validation fold, and\\\\n        on the gold set the prediction gives 0.617 against the label\\\\\\\'s 0.630,\\\\n        so nothing material is lost by using the one that always exists.\\\\n\\\\n        Monitoring only, like `gold_macro`. Selecting a checkpoint on any\\\\n        gold-derived number is what inflated an earlier version of this\\\\n        project\\\\\\\'s reported figure.\\\\n        """\\\\n        for name in self.stratified_findings:\\\\n            index = config.LABELS.index(name)\\\\n            strata = conditional.strata_from_scores(probs[:, self.stratified_index])\\\\n            value = conditional.conditional_auc(\\\\n                truth[:, index], probs[:, index], strata)\\\\n            if np.isfinite(value):\\\\n                self.log(f"{prefix}_{name}_conditional", float(value),\\\\n                         rank_zero_only=False)\\\\n            pooled = auc(truth[:, index], probs[:, index])\\\\n            if np.isfinite(pooled):\\\\n                self.log(f"{prefix}_{name}_pooled", float(pooled),\\\\n                         rank_zero_only=False)\\\\n\\\\n    def on_validation_epoch_end(self):\\\\n        val_truth, val_probs = self._gathered(0)\\\\n        if val_truth is not None:\\\\n            # Mined labels are soft; binarise for a ranking metric.\\\\n            binarised = (val_truth >= 0.5).astype(int)\\\\n            val_macro = macro_auc(binarised, val_probs)\\\\n            self.log("val_macro", val_macro, prog_bar=True, rank_zero_only=False)\\\\n            self._log_conditional("val", binarised, val_probs)\\\\n\\\\n        gold_truth, gold_probs = self._gathered(1)\\\\n        if gold_truth is not None:\\\\n            gold_macro = macro_auc(gold_truth.astype(int), gold_probs)\\\\n            # Monitoring only. Never pass this to a checkpoint callback.\\\\n            self.log("gold_macro", gold_macro, prog_bar=True,\\\\n                     rank_zero_only=False)\\\\n            self._log_conditional("gold", gold_truth.astype(int), gold_probs)\\\\n\\\\n        self._val_outputs = {0: [], 1: []}\\\\n\\\\n    def configure_optimizers(self):\\\\n        # One group unless a trunk asks for a slower one. The aggregation is\\\\n        # trained from scratch and wants the full rate; a pretrained trunk at\\\\n        # the same rate is fine for a supervised convnet and destroys a\\\\n        # self-supervised ViT\\\\\\\'s features during OneCycle\\\\\\\'s warmup. Scale 1.0\\\\n        # takes the original single-group path unchanged, so every ConvNeXt\\\\n        # result stays comparable.\\\\n        if self.backbone_lr_scale == 1.0:\\\\n            optimiser = torch.optim.AdamW(self.parameters(), lr=self.lr,\\\\n                                          weight_decay=self.weight_decay)\\\\n            max_lr = self.lr * self.max_lr_multiplier\\\\n        else:\\\\n            trunk_ids = {id(p) for p in self.model.backbone.parameters()}\\\\n            trunk = [p for p in self.parameters() if id(p) in trunk_ids]\\\\n            rest = [p for p in self.parameters() if id(p) not in trunk_ids]\\\\n            trunk_lr = self.lr * self.backbone_lr_scale\\\\n            optimiser = torch.optim.AdamW(\\\\n                [{"params": trunk, "lr": trunk_lr},\\\\n                 {"params": rest, "lr": self.lr}],\\\\n                lr=self.lr, weight_decay=self.weight_decay)\\\\n            max_lr = [trunk_lr * self.max_lr_multiplier,\\\\n                      self.lr * self.max_lr_multiplier]\\\\n\\\\n        scheduler = self._scheduler(optimiser, max_lr)\\\\n        return {\\\\n            "optimizer": optimiser,\\\\n            # Stepped per optimiser step, so accumulation is already folded in.\\\\n            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},\\\\n        }\\\\n\\\\n    def _scheduler(self, optimiser, max_lr):\\\\n        """OneCycle or cosine, with warmup pinned to an absolute step count.\\\\n\\\\n        **The restretch is the thing being fixed here.** `OneCycleLR`\\\\\\\'s\\\\n        `pct_start` is a *fraction* of the run, and it defaults to 0.3. At\\\\n        EPOCHS=12 that puts the learning-rate peak near epoch 3.6; at\\\\n        EPOCHS=20 it moves to epoch 6. Two runs of different lengths are\\\\n        therefore at different points of a differently-shaped schedule at the\\\\n        same epoch number, and "epoch 9 was best" does not mean the same thing\\\\n        in each. Every cross-length comparison this project has drawn was\\\\n        reading through that.\\\\n\\\\n        Setting `warmup_steps` to an absolute count fixes the peak in place\\\\n        regardless of run length. Leaving it None keeps the 0.3 fraction, so\\\\n        the twelve-epoch reference runs stay reproducible.\\\\n\\\\n        `schedule="cosine"` replaces the cycle with linear warmup then cosine\\\\n        decay -- the schedule the RSNA 2022/2023 winning solutions used for\\\\n        fine-tuning, and one with no restretch artefact to reason about at\\\\n        all. It peaks at the same `lr * max_lr_multiplier` as OneCycle so the\\\\n        two are comparable at matched warmup.\\\\n        """\\\\n        warmup = self.warmup_steps\\\\n        if self.schedule == "onecycle":\\\\n            pct_start = 0.3 if warmup is None else warmup / self.total_steps\\\\n            return torch.optim.lr_scheduler.OneCycleLR(\\\\n                optimiser, max_lr=max_lr, total_steps=self.total_steps,\\\\n                pct_start=pct_start)\\\\n\\\\n        warmup = int(round(0.3 * self.total_steps)) if warmup is None else warmup\\\\n        peak = self.max_lr_multiplier\\\\n        decay_steps = max(1, self.total_steps - warmup)\\\\n\\\\n        def factor(step: int) -> float:\\\\n            # Returned as a multiple of the group\\\\\\\'s base lr, which is already\\\\n            # `lr * backbone_lr_scale` for the trunk group -- so the two\\\\n            # groups keep their ratio through the whole schedule.\\\\n            if step < warmup:\\\\n                return peak * (step + 1) / max(1, warmup)\\\\n            progress = min(1.0, (step - warmup) / decay_steps)\\\\n            return peak * 0.5 * (1.0 + np.cos(np.pi * progress))\\\\n\\\\n        return torch.optim.lr_scheduler.LambdaLR(optimiser, factor)\\\\n\\\\n\\\\ndef optimiser_steps(n_studies: int, batch_size: int, devices: int,\\\\n                    accumulate: int, epochs: int) -> int:\\\\n    """Total OneCycleLR steps, which must match what Lightning will take.\\\\n\\\\n    A schedule sized for the wrong number of steps either finishes its cycle\\\\n    early and trains the tail at a near-zero learning rate, or never reaches\\\\n    its peak. Neither raises.\\\\n    """\\\\n    if min(n_studies, batch_size, devices, accumulate, epochs) < 1:\\\\n        raise ValueError("study, batch, device, accumulation and epoch counts must be positive")\\\\n    per_rank_batches = int(np.ceil(n_studies / (batch_size * devices)))\\\\n    return epochs * int(np.ceil(per_rank_batches / accumulate))\\\\n\\\\n\\\\nclass WeightAverage(pl.Callback):\\\\n    """Maintain EMA and SWA of the `KneeNet` weights and write both out.\\\\n\\\\n    This is the selection-free half of the answer to the biased-selector\\\\n    problem described in `src/model/averaging.py`: rather than trusting mined\\\\n    validation to name an epoch, average the trajectory and ship the average.\\\\n    The best-validation checkpoint is still written by `ModelCheckpoint`\\\\n    alongside these, so a run produces three candidates and the comparison\\\\n    between them is the measurement.\\\\n\\\\n    **Updated on optimiser steps, not on batches.** With gradient\\\\n    accumulation, most batches leave the weights untouched, and folding an\\\\n    unchanged model into the EMA `accumulate` times per step would make the\\\\n    effective decay `decay ** accumulate` -- a shorter memory than asked for,\\\\n    varying with a knob that is supposed to be numerically neutral.\\\\n    `on_train_batch_end` samples only when `global_step` advances, after the\\\\n    optimizer has changed the weights, including the final update.\\\\n\\\\n    **Rank zero only.** DDP keeps every rank\\\\\\\'s weights identical after each\\\\n    step, so every rank would compute the same average; doing it once saves\\\\n    the copies. The written files therefore exist only on rank zero, which is\\\\n    also the only rank whose `/kaggle/working` survives.\\\\n\\\\n    **The averages are written from inside `fit`, in `on_train_end`, and read\\\\n    back off disk afterwards.** Under `ddp_notebook` the ranks are forked\\\\n    children: the callback object the notebook constructed is *not* the object\\\\n    that accumulated anything, and asking it for its EMA afterwards returns an\\\\n    empty average with no error. This is the same trap that leaves\\\\n    `best_model_score` as None on that strategy. So the file paths are fixed\\\\n    at construction and the notebook globs for what appeared.\\\\n    """\\\\n\\\\n    def __init__(self, prefix: str, decay: float = 0.999,\\\\n                 swa_fraction: float = 0.25,\\\\n                 total_steps: Optional[int] = None):\\\\n        if not 0.0 <= swa_fraction <= 1.0:\\\\n            raise ValueError(f"swa_fraction must be in [0, 1], got {swa_fraction}")\\\\n        self.prefix = prefix\\\\n        self.decay = decay\\\\n        self.swa_fraction = swa_fraction\\\\n        self.total_steps = total_steps\\\\n        self.average: Optional[averaging.WeightAverage] = None\\\\n        self.written: Dict[str, str] = {}\\\\n        self._last_step = 0\\\\n\\\\n    @property\\\\n    def ema_path(self) -> str:\\\\n        return f"{self.prefix}_ema.pt"\\\\n\\\\n    @property\\\\n    def swa_path(self) -> str:\\\\n        return f"{self.prefix}_swa.pt"\\\\n\\\\n    def _swa_from(self, trainer) -> Optional[int]:\\\\n        """The step the equal-weight tail average starts at."""\\\\n        total = self.total_steps\\\\n        if total is None:\\\\n            total = getattr(trainer, "estimated_stepping_batches", None)\\\\n        if not total or self.swa_fraction <= 0:\\\\n            return None\\\\n        # +1 because `update` counts from one: at total=4 and fraction=0.5 the\\\\n        # window must open at step 3, so that steps 3 and 4 -- the last half --\\\\n        # are what gets averaged, not steps 2 through 4.\\\\n        return int(round(float(total) * (1.0 - self.swa_fraction))) + 1\\\\n\\\\n    def on_train_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        self.average = averaging.WeightAverage(\\\\n            module.model, decay=self.decay, swa_from=self._swa_from(trainer))\\\\n        self._last_step = trainer.global_step\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        # Sample after the update, including the final trained weights.\\\\n        if self.average is not None and trainer.global_step > self._last_step:\\\\n            self.average.update(module.model)\\\\n            self._last_step = trainer.global_step\\\\n\\\\n    def on_train_end(self, trainer, module):\\\\n        # Wrapped, on the same rule the notebook\\\\\\\'s export cell follows: by the\\\\n        # time this fires, hours of GPU time exist as the monitored checkpoint\\\\n        # `ModelCheckpoint` already wrote, and an exception raised here\\\\n        # propagates out of `trainer.fit` and takes the export and scoring\\\\n        # cells with it. The averages are an experiment; the run is not.\\\\n        try:\\\\n            self.export()\\\\n        except Exception as exc:  # noqa: BLE001 - the checkpoint is safe\\\\n            print(f"WeightAverage could not write its averages ({exc}). "\\\\n                  "The monitored checkpoint is unaffected; only the "\\\\n                  "selection-free comparison is lost.")\\\\n\\\\n    def export(self) -> Dict[str, str]:\\\\n        """Write the averages as bare `KneeNet` state dicts.\\\\n\\\\n        The same format `export_from_checkpoint` produces, so notebook 10\\\\n        loads them with no idea they came from an average. Returns\\\\n        {kind: path} for what was actually written -- SWA is absent when the\\\\n        run was too short for any step to fall inside the tail window.\\\\n        """\\\\n        self.written = {}\\\\n        if self.average is None:\\\\n            return self.written\\\\n\\\\n        torch.save(self.average.ema_state(), self.ema_path)\\\\n        self.written["ema"] = self.ema_path\\\\n\\\\n        swa = self.average.swa_state()\\\\n        if swa is not None:\\\\n            torch.save(swa, self.swa_path)\\\\n            self.written["swa"] = self.swa_path\\\\n        return self.written\\\\n\\\\n\\\\nclass EpochReport(pl.Callback):\\\\n    """One line per epoch, from rank zero only.\\\\n\\\\n    Under DDP every rank runs this code, so an unguarded print produces one\\\\n    copy of the log per GPU and makes a run look twice as long as it is.\\\\n\\\\n    **This reads `callback_metrics` in `on_validation_end`, not in\\\\n    `on_validation_epoch_end`.** Lightning runs callback hooks before the\\\\n    LightningModule\\\\\\\'s hook of the same name, so a callback asking for\\\\n    `val_macro` at `on_validation_epoch_end` arrives before\\\\n    `KneeLitModule.on_validation_epoch_end` has computed it and reads an\\\\n    empty dict -- printing `nan` for every epoch of an otherwise healthy run.\\\\n    `ModelCheckpoint` selects in `on_validation_end` for the same reason, so\\\\n    checkpoint selection was never affected by this; only the log was.\\\\n    """\\\\n\\\\n    def __init__(self):\\\\n        self.rows: List[Dict[str, float]] = []\\\\n\\\\n    def on_validation_end(self, trainer, module):\\\\n        if trainer.sanity_checking or not trainer.is_global_zero:\\\\n            return\\\\n        metrics = trainer.callback_metrics\\\\n        val = float(metrics.get("val_macro", float("nan")))\\\\n        gold = float(metrics.get("gold_macro", float("nan")))\\\\n        self.rows.append({"epoch": trainer.current_epoch + 1,\\\\n                          "val_macro": val, "gold_macro": gold})\\\\n        best = max(row["val_macro"] for row in self.rows)\\\\n        marker = "  <- best" if val >= best else ""\\\\n        print(f"  epoch {trainer.current_epoch + 1}: val macro {val:.3f}  "\\\\n              f"gold {gold:.3f} (monitor only){marker}")\\\\n\\\\n\\\\nclass BatchProgress(pl.Callback):\\\\n    """Plain-text tqdm over training batches, rank zero only.\\\\n\\\\n    Lightning\\\\\\\'s own progress bar resolves to the notebook-widget flavour of\\\\n    tqdm inside Kaggle, and the widget never reaches the log of a\\\\n    Save & Run All -- eleven minutes pass per epoch with no sign of life.\\\\n    Plain tqdm writes carriage-returned text to stdout, which survives both\\\\n    the interactive view and the batch log, so the Trainer disables its own\\\\n    bar and takes this instead. `mininterval` keeps the batch log from\\\\n    bloating: one refresh per interval, not one per batch.\\\\n    """\\\\n\\\\n    def __init__(self, refresh_seconds: float = 15.0):\\\\n        self.refresh_seconds = refresh_seconds\\\\n        self._bar = None\\\\n\\\\n    def on_train_epoch_start(self, trainer, module):\\\\n        if not trainer.is_global_zero:\\\\n            return\\\\n        import sys\\\\n        from tqdm import tqdm\\\\n        self._bar = tqdm(\\\\n            total=int(trainer.num_training_batches),\\\\n            desc=f"epoch {trainer.current_epoch + 1}/{trainer.max_epochs}",\\\\n            unit="batch", file=sys.stdout,\\\\n            mininterval=self.refresh_seconds, leave=False)\\\\n\\\\n    def on_train_batch_end(self, trainer, module, outputs, batch, batch_idx):\\\\n        if self._bar is not None:\\\\n            self._bar.update(1)\\\\n\\\\n    def on_train_epoch_end(self, trainer, module):\\\\n        if self._bar is not None:\\\\n            self._bar.close()\\\\n            self._bar = None\\\\n\\\\n\\\\ndef export_from_checkpoint(ckpt_path: str, out_path: str) -> int:\\\\n    """Convert a Lightning checkpoint into the plain state dict for scoring.\\\\n\\\\n    Read from the file rather than from a module in memory: under the spawn\\\\n    strategy the process that called `fit` is not the process that trained,\\\\n    and reaching for in-memory weights there is how a run silently ships its\\\\n    initialisation. Returns the number of tensors written.\\\\n    """\\\\n    checkpoint = torch.load(ckpt_path, map_location="cpu")\\\\n    state_dict = checkpoint.get("state_dict", checkpoint)\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in state_dict.items()\\\\n             if key.startswith("model.")}\\\\n    if not state:\\\\n        raise ValueError(\\\\n            f"no \\\\\\\'model.\\\\\\\' keys in {ckpt_path}; found "\\\\n            f"{list(state_dict)[:3]}")\\\\n    torch.save(state, out_path)\\\\n    return len(state)\\\\n\\\\n\\\\ndef export_state_dict(module: KneeLitModule, path: str) -> None:\\\\n    """Write a plain `KneeNet` state dict, the format the submission loads.\\\\n\\\\n    Lightning prefixes every key with `model.` because `KneeNet` is an\\\\n    attribute of the wrapper. Notebook 10 constructs a bare `KneeNet` and\\\\n    calls `load_state_dict`, so the prefix has to come off here rather than\\\\n    become a surprise at submission time.\\\\n    """\\\\n    state = {key[len("model."):]: value\\\\n             for key, value in module.state_dict().items()\\\\n             if key.startswith("model.")}\\\\n    torch.save(state, path)\\\\n\\\'}\\nfor path, source in FILES.items():\\n    target = pathlib.Path(path)\\n    target.parent.mkdir(parents=True, exist_ok=True)\\n    target.write_text(source)\\nsys.path.insert(0, \\\'.\\\')\\nfor _name in [m for m in sys.modules if m == \\\'src\\\' or m.startswith(\\\'src.\\\')]:\\n    del sys.modules[_name]\\nimportlib.invalidate_caches()\\nfrom src import config\\nfrom src.data import dicom_io, cache, index as index_mod\\nif pathlib.Path(\\\'src/model/side_classifier.py\\\').exists():\\n    from src.model import side_classifier\\nif pathlib.Path(\\\'src/model/net.py\\\').exists():\\n    from src.model import augment as knee_augment, averaging, conditional, embed, dataset as knee_dataset, net, splits\\n    from src.model import artifacts, experiments\\nif pathlib.Path(\\\'src/model/inspect.py\\\').exists():\\n    from src.model import inspect as knee_inspect\\nif pathlib.Path(\\\'src/labels/fuse.py\\\').exists():\\n    from src.labels import fuse\\nif pathlib.Path(\\\'src/model/lightning.py\\\').exists():\\n    from src.model import lightning as knee_lightning\\nprint(\\\'modules ready:\\\', len(FILES), \\\'files\\\')\\nprint(\\\'labels:\\\', len(config.LABELS))\\nprint(\\\'cache geometry:\\\', config.CACHE_SLICES, \\\'x\\\', config.CACHE_SIZE)\\nprint(\\\'data root:\\\', config.DATA_ROOT)\\n\\nEXPERIMENT = os.environ.get(\\\'KNEE_EXPERIMENT\\\', \\\'resample\\\')\\nRUN = experiments.get_experiment(EXPERIMENT)\\nfrom dataclasses import replace\\nRUN = replace(RUN, series_selection=\\\'protocol\\\')\\nprint(\\\'experiment:\\\', RUN.to_dict())\\n\\nimport glob\\nimport numpy as np\\nimport pandas as pd\\n\\ndef find_inputs(pattern):\\n    found = sorted(glob.glob(f\\\'data/**/{pattern}\\\', recursive=True))\\n    found += sorted(glob.glob(f\\\'/kaggle/working/ddp_smoke/{pattern}\\\'))\\n    for root in sorted(glob.glob(\\\'/kaggle/input/*\\\')):\\n        if os.path.basename(root) == \\\'competitions\\\' or os.path.exists(os.path.join(root, \\\'train.csv\\\')) or os.path.exists(os.path.join(root, \\\'train_series.csv\\\')):\\n            continue\\n        found += sorted(glob.glob(f\\\'{root}/**/{pattern}\\\', recursive=True))\\n    return found\\n\\ndef find_competition_file(name):\\n    for candidate in (f\\\'data/{name}\\\', f\\\'/kaggle/input/competitions/*/{name}\\\', f\\\'/kaggle/input/*/{name}\\\'):\\n        hits = sorted(glob.glob(candidate))\\n        if hits:\\n            return hits[0]\\n    raise SystemExit(f\\\'missing competition file: {name}\\\')\\nlabels = pd.read_parquet(find_inputs(\\\'train_labels.parquet\\\')[0])\\nSILENCE_REWEIGHT = False\\nif SILENCE_REWEIGHT:\\n    _rates = fuse.silence_rates(labels)\\n    labels = fuse.rescale_silent_weights(labels)\\n    print(\\\'silence reweighting ON\\\')\\n    for _n in sorted(config.LABELS, key=lambda x: -_rates[x])[:3]:\\n        print(f\\\'  {_n:18s} silent {_rates[_n]:5.1%} -> weight {fuse.WEIGHT_SILENT * (1 - _rates[_n]):.3f}\\\')\\nelse:\\n    print(\\\'silence reweighting OFF (reference labels)\\\')\\nsides = pd.read_parquet(find_inputs(\\\'side_predictions.parquet\\\')[0])\\nreport_hits = find_inputs(\\\'report_embeddings.parquet\\\')\\nreports = pd.read_parquet(report_hits[0]) if report_hits else None\\nif reports is None:\\n    print(\\\'report_embeddings.parquet not attached -- report loss disabled\\\')\\nseries_meta = pd.read_csv(find_competition_file(\\\'train_series.csv\\\'))\\ntrain_csv = pd.read_csv(find_competition_file(\\\'train.csv\\\'))\\ngold = train_csv[train_csv[\\\'ACL\\\'].notna()].reset_index(drop=True)\\nALL_SHARDS = True\\ncache_paths = find_inputs(\\\'cache_train_series_*.npy\\\')\\nif not ALL_SHARDS:\\n    cache_paths = cache_paths[:1]\\nshards, frames = ({}, [])\\nfor cache_path in cache_paths:\\n    manifest_path = cache_path.replace(\\\'cache_\\\', \\\'manifest_\\\').replace(\\\'.npy\\\', \\\'.parquet\\\')\\n    if not os.path.exists(manifest_path):\\n        stem = os.path.basename(cache_path).replace(\\\'cache_\\\', \\\'\\\').replace(\\\'.npy\\\', \\\'\\\')\\n        hits = [m for m in find_inputs(\\\'manifest_train_series_*.parquet\\\') if os.path.basename(m).startswith(f\\\'manifest_{stem}\\\')]\\n        if not hits:\\n            continue\\n        manifest_path = hits[0]\\n    frame = pd.read_parquet(manifest_path)\\n    frame[\\\'cache_path\\\'] = cache_path\\n    shards[cache_path] = np.load(cache_path, mmap_mode=\\\'r\\\')\\n    frames.append(frame)\\n    print(\\\'shard:\\\', os.path.basename(cache_path), shards[cache_path].shape)\\nmanifest = pd.concat(frames, ignore_index=True)\\nGROUPED_FOLDS = False\\ngroups = None\\nif GROUPED_FOLDS:\\n    _index_files = find_inputs(\\\'index_train.parquet\\\') + find_inputs(\\\'index.parquet\\\')\\n    if _index_files:\\n        groups = splits.scanner_key(pd.read_parquet(_index_files[0]))\\n        print(f\\\'grouped folds: {groups.nunique()} scanner groups over {len(groups)} studies\\\')\\n    else:\\n        raise ValueError("GROUPED_FOLDS requires notebook 01\\\'s index parquet")\\nfolded = splits.make_folds(labels, groups=groups)\\nsplits.assert_no_gold_leakage(folded[\\\'StudyInstanceUID\\\'], set(gold[\\\'StudyInstanceUID\\\']))\\nif groups is not None:\\n    _summary = splits.summarise(folded)\\n    print(_summary[[\\\'studies\\\'] + config.LABELS[:4]].to_string())\\n    print(\\\'  fold sizes are uneven by construction -- a scanner group cannot be split. Check the positive rates before trusting a comparison.\\\')\\nMAX_SERIES = 5\\nMAX_SERIES = RUN.max_series\\nindex = knee_dataset.StudyIndex(manifest, series_meta, folded, sides, max_series=MAX_SERIES, reports=reports, series_selection=RUN.series_selection)\\ngold_labels = gold[[\\\'StudyInstanceUID\\\'] + config.LABELS].copy()\\nfor name in config.LABELS:\\n    gold_labels[f\\\'{name}_weight\\\'] = 1.0\\ngold_index = knee_dataset.StudyIndex(manifest, series_meta, gold_labels, sides, max_series=MAX_SERIES, series_selection=RUN.series_selection)\\nif ALL_SHARDS and (set(index.studies) != set(labels[config.ID_COL]) or set(gold_index.studies) != set(gold[config.ID_COL])):\\n    raise ValueError(\\\'incomplete cache coverage: attach every training and gold study\\\')\\nprint(f\\\'studies in this shard: {len(index)}  gold present: {len(gold_index)}\\\')\\n\\nfrom torch.utils.data import DataLoader\\nSLICES_PER_SERIES = 16\\nSLICES_PER_SERIES = RUN.slices\\nRESAMPLE = True\\nRESAMPLE = RUN.resample\\nBATCH_STUDIES = 1\\nEFFECTIVE_STUDIES = 8\\nACCUM_STEPS = max(1, EFFECTIVE_STUDIES // (BATCH_STUDIES * DEVICES))\\nprint(f\\\'{BATCH_STUDIES} studies x {DEVICES} devices x {ACCUM_STEPS} accum = {BATCH_STUDIES * DEVICES * ACCUM_STEPS} effective\\\')\\nCACHE_PATHS = list(shards)\\n\\ndef make_loader(study_index, studies, shuffle=False, augment=False, resample=False):\\n    return DataLoader(knee_lightning.StudyVolumeDataset(study_index, studies, CACHE_PATHS, SLICES_PER_SERIES, augment, resample=resample), batch_size=BATCH_STUDIES, shuffle=shuffle, num_workers=2, collate_fn=knee_dataset.collate_studies, pin_memory=True, persistent_workers=True)\\n\\nimport timm\\nBACKBONE = \\\'convnext_tiny.fb_in22k_ft_in1k\\\'\\nBACKBONE = RUN.backbone\\n_warm = net.build_backbone(BACKBONE)\\nFEATURE_DIM = _warm.num_features\\nprint(f\\\'{BACKBONE}: {FEATURE_DIM} features, {sum((p.numel() for p in _warm.parameters())) / 1000000.0:.1f}M params\\\')\\ndel _warm\\nBACKBONE_DEFAULTS = net.backbone_defaults(BACKBONE)\\nprint(\\\'backbone defaults:\\\', BACKBONE_DEFAULTS)\\nauc = knee_lightning.auc\\n\\nimport time\\nimport timm\\n_required = {\\\'DEVICES\\\': \\\'1. Setup\\\', \\\'knee_lightning\\\': \\\'2. Repo modules\\\', \\\'index\\\': \\\'3. Assemble\\\', \\\'make_loader\\\': \\\'4. Dataset\\\', \\\'FEATURE_DIM\\\': \\\'5. Model\\\'}\\n_missing = {name: cell for name, cell in _required.items() if name not in globals()}\\nif _missing:\\n    raise SystemExit(\\\'This cell needs the cells above it, and this kernel has not run them:\\\\n\\\' + \\\'\\\\n\\\'.join((f\\\'    {name!r} comes from section {cell}\\\' for name, cell in _missing.items())) + \\\'\\\\n\\\\nRun the notebook from the top (Runtime -> Run all). Changing FOLD and re-running only this cell works within a LIVE kernel; a restarted runtime has to re-run everything.\\\')\\nEPOCHS = 12\\nLR = 0.0001\\nWARMUP_EPOCHS = 3.6\\nSCHEDULE = \\\'onecycle\\\'\\nSEED = 0\\npl.seed_everything(SEED, workers=True)\\nPOS_WEIGHT_CAP = None\\nif POS_WEIGHT_CAP is None:\\n    POS_WEIGHT = None\\nelse:\\n    _positives = (labels[config.LABELS] >= 0.5).sum()\\n    _negatives = len(labels) - _positives\\n    POS_WEIGHT = [float(min((n / max(p, 1)) ** 0.5, POS_WEIGHT_CAP)) for p, n in zip(_positives, _negatives)]\\n    print(f\\\'pos_weight (sqrt of neg/pos, capped at {POS_WEIGHT_CAP}):\\\')\\n    for _name, _w in zip(config.LABELS, POS_WEIGHT):\\n        _p = int((labels[_name] >= 0.5).sum())\\n        print(f\\\'  {_name:18s} {_w:5.2f}   ({_p} positives, {_p / len(labels):.1%})\\\')\\nLAMBDA_REPORT = 0.0\\nLAMBDA_RANK = 0.0\\nLAMBDA_STRATIFIED_RANK = 0.0\\nSTRATIFIED_FINDINGS = (\\\'Synovitis\\\',)\\nSTRATIFY_BY = \\\'Effusion\\\'\\nSTRATIFIED_CALIPER = 0.15\\nRESIDUAL_BETA = None\\nLAMBDA_COUPLING = 0.0\\nCOUPLING_TARGET = 0.403\\nAUX_MODE = \\\'fixed\\\'\\nAUX_GATE = \\\'binary\\\'\\nREPORT_DIM = len([c for c in reports.columns if c != \\\'StudyInstanceUID\\\']) if reports is not None else None\\nFOLD = 2\\nCHECKPOINT = f\\\'/kaggle/working/ddp_smoke/convnext_fold{FOLD}.pt\\\'\\nAVERAGE_PREFIX = f\\\'/kaggle/working/ddp_smoke/avg_fold{FOLD}\\\'\\nkeyed = folded.set_index(\\\'StudyInstanceUID\\\')\\ntrain_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] != FOLD]\\nval_studies = [s for s in index.studies if keyed.loc[s, \\\'fold\\\'] == FOLD]\\ntrain_loader = make_loader(index, train_studies, shuffle=True, augment=True, resample=RESAMPLE)\\nprint(f"resampling slices/series per epoch: {(\\\'on\\\' if RESAMPLE else \\\'off\\\')}")\\nval_loader = make_loader(index, val_studies)\\ngold_loader = make_loader(gold_index, gold_index.studies)\\nprint(f\\\'fold {FOLD}: train {len(train_studies)}  val {len(val_studies)}  gold {len(gold_index)}\\\')\\nTOTAL_STEPS = knee_lightning.optimiser_steps(len(train_studies), BATCH_STUDIES, DEVICES, ACCUM_STEPS, EPOCHS)\\nprint(\\\'optimiser steps:\\\', TOTAL_STEPS)\\nWARMUP_STEPS = int(round(TOTAL_STEPS / EPOCHS * WARMUP_EPOCHS))\\nprint(f"warmup: {WARMUP_STEPS} steps ({WARMUP_EPOCHS} epochs\\\' worth), schedule {SCHEDULE}")\\nmodule = knee_lightning.KneeLitModule(backbone=net.build_backbone(BACKBONE), feature_dim=FEATURE_DIM, lr=LR, total_steps=TOTAL_STEPS, report_dim=REPORT_DIM, lambda_report=LAMBDA_REPORT, lambda_rank=LAMBDA_RANK, aux_mode=AUX_MODE, aux_gate=AUX_GATE, schedule=SCHEDULE, warmup_steps=WARMUP_STEPS, lambda_stratified_rank=LAMBDA_STRATIFIED_RANK, stratified_findings=STRATIFIED_FINDINGS, stratify_by=STRATIFY_BY, stratified_caliper=STRATIFIED_CALIPER, lambda_coupling=LAMBDA_COUPLING, coupling_target=COUPLING_TARGET if LAMBDA_COUPLING > 0 else None, residual_beta=RESIDUAL_BETA, pos_weight=POS_WEIGHT, classwise_pool=RUN.classwise_pool, slice_transformer=RUN.slice_transformer, **BACKBONE_DEFAULTS)\\nprint(f"trunk lr: {LR * BACKBONE_DEFAULTS[\\\'backbone_lr_scale\\\']:.1e}  head lr: {LR:.1e}  input norm: {(\\\'on\\\' if BACKBONE_DEFAULTS[\\\'input_mean\\\'] else \\\'off\\\')}")\\nprint(f\\\'aux losses: report {LAMBDA_REPORT} (dim {REPORT_DIM})  rank {LAMBDA_RANK}  mode {AUX_MODE}\\\' + (f\\\'/{AUX_GATE}\\\' if AUX_MODE == \\\'cosine\\\' else \\\'\\\'))\\nprint(f\\\'synovitis: stratified rank {LAMBDA_STRATIFIED_RANK} (caliper {STRATIFIED_CALIPER})  residual beta {RESIDUAL_BETA}  coupling {LAMBDA_COUPLING}\\\' + (f\\\' -> {COUPLING_TARGET}\\\' if LAMBDA_COUPLING > 0 else \\\'\\\'))\\nENSEMBLE_DONE = {0: \\\'2026-08-31, 12 ep, val 0.8814, gold 0.8683\\\', 1: \\\'2026-08-31, 12 ep, val 0.8726, gold 0.8747\\\', 2: \\\'2026-08-31, 12 ep, val 0.8675, gold 0.8659\\\', 3: \\\'2026-08-31, 12 ep, val 0.8736, gold 0.8915\\\', 4: \\\'2026-08-31, 12 ep, val 0.8697, gold 0.8753\\\'}\\nENSEMBLE_TODO = [f for f in range(5) if f not in ENSEMBLE_DONE]\\n_expected = {\\\'SLICES_PER_SERIES\\\': (SLICES_PER_SERIES, 16), \\\'MAX_SERIES\\\': (MAX_SERIES, 5), \\\'EPOCHS\\\': (EPOCHS, 12), \\\'LAMBDA_REPORT\\\': (LAMBDA_REPORT, 0.0), \\\'LAMBDA_RANK\\\': (LAMBDA_RANK, 0.0), \\\'LAMBDA_STRATIFIED_RANK\\\': (LAMBDA_STRATIFIED_RANK, 0.0), \\\'RESIDUAL_BETA\\\': (RESIDUAL_BETA, None), \\\'LAMBDA_COUPLING\\\': (LAMBDA_COUPLING, 0.0), \\\'POS_WEIGHT_CAP\\\': (POS_WEIGHT_CAP, None), \\\'RESAMPLE\\\': (RESAMPLE, False), \\\'SERIES_SELECTION\\\': (RUN.series_selection, \\\'uniform\\\'), \\\'CLASSWISE_POOL\\\': (RUN.classwise_pool, False), \\\'SLICE_TRANSFORMER\\\': (RUN.slice_transformer, False), \\\'SILENCE_REWEIGHT\\\': (SILENCE_REWEIGHT, False), \\\'BACKBONE\\\': (BACKBONE, \\\'convnext_tiny.fb_in22k_ft_in1k\\\')}\\n_wrong = {k: v for k, (v, want) in _expected.items() if v != want}\\nprint(\\\'=\\\' * 68)\\nif _wrong:\\n    print(f\\\'  OFF-REFERENCE FOLD {FOLD}. These differ from the ensemble config:\\\')\\n    for k, v in _wrong.items():\\n        print(f\\\'    {k} = {v!r}, reference is {_expected[k][1]!r}\\\')\\n    print(\\\'  An ensemble member trained off-reference makes the average a\\\')\\n    print(\\\'  blend of two models, and the averaging hides it. Fix these, or\\\')\\n    print(\\\'  accept that this run is an experiment and not a member.\\\')\\nelif FOLD in ENSEMBLE_DONE:\\n    print(f\\\'  FOLD {FOLD} IS ALREADY DONE -- {ENSEMBLE_DONE[FOLD]}\\\')\\n    print(f\\\'  Retraining it is fine, but the folds still missing are {ENSEMBLE_TODO}.\\\')\\nelse:\\n    print(f\\\'  ENSEMBLE FOLD {FOLD}: reference config, no auxiliary loss\\\')\\n    print(f\\\'  done: {sorted(ENSEMBLE_DONE)}   outstanding after this: {[f for f in ENSEMBLE_TODO if f != FOLD]}\\\')\\n    print(f\\\'  export convnext_fold{FOLD}.pt and COPY IT OFF THE VM before the\\\')\\n    print(\\\'  session ends -- /kaggle/working does not survive.\\\')\\nprint(\\\'=\\\' * 68)\\ncheckpointer = pl.callbacks.ModelCheckpoint(dirpath=\\\'/kaggle/working/ddp_smoke\\\', filename=f\\\'lit_fold{FOLD}\\\', monitor=\\\'val_macro\\\', mode=\\\'max\\\', save_top_k=1)\\nreport = knee_lightning.EpochReport()\\naverager = knee_lightning.WeightAverage(prefix=AVERAGE_PREFIX, decay=0.999, swa_fraction=0.25, total_steps=TOTAL_STEPS)\\ncsv_logger = pl.loggers.CSVLogger(\\\'/kaggle/working/ddp_smoke\\\', name=f\\\'lit_fold{FOLD}\\\', version=\\\'metrics\\\')\\nSTRATEGY = \\\'ddp\\\' if DEVICES > 1 else \\\'auto\\\'\\nif STRATEGY == \\\'ddp_notebook\\\' and torch.cuda.is_initialized():\\n    raise SystemExit("CUDA is already initialised in this process, so the forked workers cannot start.\\\\nRun -> Restart Session, then run the cells in order. If it recurs, set STRATEGY = \\\'ddp_spawn\\\' in this cell.")\\ntrainer = pl.Trainer(accelerator=\\\'gpu\\\', devices=DEVICES, strategy=STRATEGY, precision=\\\'16-mixed\\\', accumulate_grad_batches=ACCUM_STEPS, callbacks=[checkpointer, report, averager, knee_lightning.BatchProgress()], logger=csv_logger, log_every_n_steps=50, enable_model_summary=False, enable_progress_bar=False, max_epochs=1, limit_train_batches=8, limit_val_batches=8, num_sanity_val_steps=0)\\nstart = time.time()\\nassert (FOLD, EPOCHS, SEED, SLICES_PER_SERIES, MAX_SERIES) == (EXPECTED_FOLD, 12, 0, 16, 5)\\nassert not RESAMPLE and (not GROUPED_FOLDS)\\nassert RUN.classwise_pool == (EXPERIMENT == \\\'classwise\\\')\\nassert not SILENCE_REWEIGHT and POS_WEIGHT is None and (RESIDUAL_BETA is None)\\nassert (LAMBDA_REPORT, LAMBDA_RANK, LAMBDA_STRATIFIED_RANK, LAMBDA_COUPLING) == (0.0, 0.0, 0.0, 0.0)\\nassert len(index) == 4349 and len(gold_index) == 58 and (len(CACHE_PATHS) == 3)\\nassert len(train_studies) == 4349 - EXPECTED_VAL and len(val_studies) == EXPECTED_VAL\\nassert BATCH_STUDIES * DEVICES * ACCUM_STEPS == 8\\nassert BACKBONE == (\\\'convnextv2_tiny.fcmae_ft_in22k_in1k\\\' if EXPERIMENT == \\\'convnextv2\\\' else \\\'convnext_tiny.fb_in22k_ft_in1k\\\')\\nif MATCH_BACKBONE:\\n    assert FEATURE_DIM == 768\\n    assert BACKBONE_DEFAULTS == {\\\'input_mean\\\': None, \\\'input_std\\\': None, \\\'backbone_lr_scale\\\': 1.0}\\nif MATCH_SLICE:\\n    assert RUN.series_selection == \\\'protocol\\\'\\n    assert RUN.slice_transformer == (EXPERIMENT == \\\'slice_transformer\\\')\\nelse:\\n    assert RUN.series_selection == (\\\'protocol\\\' if EXPERIMENT == \\\'protocol\\\' else \\\'uniform\\\')\\nassert all((volume.shape[1:] == (24, 224, 224) for volume in shards.values()))\\nimport hashlib, json, pathlib\\n_initial = hashlib.sha256()\\nfor _key, _tensor in sorted(module.state_dict().items()):\\n    _initial.update(_key.encode())\\n    _initial.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n_contract = {\\\'experiment\\\': RUN.to_dict(), \\\'fold\\\': FOLD, \\\'epochs\\\': EPOCHS, \\\'seed\\\': SEED, \\\'devices\\\': DEVICES, \\\'effective_batch\\\': EFFECTIVE_STUDIES, \\\'steps\\\': TOTAL_STEPS, \\\'lr\\\': LR, \\\'schedule\\\': SCHEDULE, \\\'warmup_steps\\\': WARMUP_STEPS, \\\'initial_weights_sha256\\\': _initial.hexdigest(), \\\'torch\\\': torch.__version__, \\\'timm\\\': timm.__version__, \\\'lightning\\\': pl.__version__}\\nif MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE:\\n    _excluded = (\\\'model.slice_encoder.\\\',) if MATCH_SLICE else (\\\'model.backbone.\\\',) if MATCH_BACKBONE else (\\\'model.study_pool.\\\', \\\'model.head.\\\')\\n    _shared = hashlib.sha256()\\n    for _key, _tensor in sorted(module.state_dict().items()):\\n        if not _key.startswith(_excluded):\\n            _shared.update(_key.encode())\\n            _shared.update(_tensor.detach().cpu().contiguous().numpy().tobytes())\\n    _contract[\\\'shared_initial_weights_sha256\\\'] = _shared.hexdigest()\\nif MATCH_BACKBONE:\\n    _contract[\\\'aggregation_seed\\\'] = SEED\\n    _contract[\\\'feature_dim\\\'] = FEATURE_DIM\\n    _contract[\\\'input_defaults\\\'] = BACKBONE_DEFAULTS\\nif \\\'REFERENCE_CONTRACT\\\' in globals():\\n    _fold_specific = {\\\'fold\\\', \\\'steps\\\', \\\'warmup_steps\\\'}\\n    assert {k: v for k, v in _contract.items() if k not in _fold_specific} == {k: v for k, v in REFERENCE_CONTRACT.items() if k not in _fold_specific}, \\\'Completion settings or initialization differ from screened protocol\\\'\\n_baseline_contract = pathlib.Path(BASELINE_CONTRACT_PATH)\\nif EXPERIMENT != \\\'baseline\\\' and _baseline_contract.exists():\\n    _control = json.loads(_baseline_contract.read_text())\\n    _ignored = {\\\'experiment\\\'} | ({\\\'initial_weights_sha256\\\'} if MATCH_CLASSWISE or MATCH_BACKBONE or MATCH_SLICE else set())\\n    assert {k: v for k, v in _contract.items() if k not in _ignored} == {k: v for k, v in _control.items() if k not in _ignored}, \\\'Candidate settings or shared initialization differ from baseline\\\'\\nif int(os.environ.get(\\\'LOCAL_RANK\\\', \\\'0\\\')) == 0:\\n    pathlib.Path(\\\'run_contract.json\\\').write_text(json.dumps(_contract, indent=2) + \\\'\\\\n\\\')\\nprint(\\\'Matched-run preflight passed\\\', flush=True)\\naverager.total_steps = 2\\naverager.swa_fraction = 1.0\\ntrainer.fit(module, train_dataloaders=train_loader, val_dataloaders=[val_loader, gold_loader])\\nif not trainer.is_global_zero:\\n    raise SystemExit(0)\\nelapsed = time.time() - start\\n\\ndef recover_checkpoint(fold):\\n    """The checkpoint this run wrote, when `best_model_path` came back empty.\\n\\n    NOT `sorted(glob(...))[0]`, which is what this was. That picks the first\\n    name ALPHABETICALLY, and "-" (45) sorts before "." (46), which sorts\\n    before "_" (95): `lit_fold1-v1.ckpt` beats `lit_fold1.ckpt`, which beats\\n    `lit_fold1_old.ckpt`. Lightning writes those `-vN` suffixes on every name\\n    collision, so a session that resumed after a crash silently recovered a\\n    checkpoint from the earlier attempt.\\n\\n    The exact name wins when it exists, and otherwise the newest file does --\\n    which is this run\\\'s, since this cell runs seconds after fit returned.\\n    """\\n    exact = f\\\'/kaggle/working/ddp_smoke/lit_fold{fold}.ckpt\\\'\\n    if os.path.exists(exact):\\n        return exact\\n    found = glob.glob(f\\\'/kaggle/working/ddp_smoke/lit_fold{fold}*.ckpt\\\')\\n    return max(found, key=os.path.getmtime) if found else None\\ncheckpoint_path = checkpointer.best_model_path or \\\'\\\'\\nif not checkpoint_path or not os.path.exists(checkpoint_path):\\n    checkpoint_path = recover_checkpoint(FOLD)\\n    if checkpoint_path is None:\\n        raise SystemExit(\\\'training finished but no checkpoint was written -- check the epoch log for whether val_macro was ever computed\\\')\\n    print(f\\\'best_model_path was empty; recovered {checkpoint_path}\\\')\\nprint(\\\'\\\\nbest checkpoint:\\\', checkpoint_path)\\nn_tensors = knee_lightning.export_from_checkpoint(checkpoint_path, CHECKPOINT)\\nprint(f\\\'wrote {CHECKPOINT} ({n_tensors} tensors)\\\')\\nAVERAGES = {kind: f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\' for kind in (\\\'ema\\\', \\\'swa\\\') if os.path.exists(f\\\'{AVERAGE_PREFIX}_{kind}.pt\\\')}\\nfor kind, path in AVERAGES.items():\\n    print(f\\\'wrote {path} ({kind.upper()} weights, selection-free)\\\')\\n_actual_spec = RUN.to_dict()\\n_actual_spec.update(backbone=BACKBONE, slices=SLICES_PER_SERIES, max_series=MAX_SERIES, resample=RESAMPLE)\\nfor _path in [CHECKPOINT] + list(AVERAGES.values()):\\n    artifacts.write_metadata(_path, _actual_spec, FOLD, train_studies, val_studies)\\nfolded[[config.ID_COL, \\\'fold\\\']].to_parquet(\\\'/kaggle/working/ddp_smoke/folds.parquet\\\', index=False)\\nif not AVERAGES:\\n    print(\\\'NOTE: no averaged weights on disk -- the WeightAverage callback did not run to on_train_end. The monitored checkpoint is fine; only the P2 comparison is unavailable.\\\')\\ntry:\\n    history = pd.read_csv(os.path.join(csv_logger.log_dir, \\\'metrics.csv\\\'))\\n    per_epoch_rows = history.dropna(subset=[\\\'val_macro\\\']).groupby(\\\'epoch\\\')[[\\\'val_macro\\\', \\\'gold_macro\\\']].last()\\n    best_epoch = per_epoch_rows[\\\'val_macro\\\'].idxmax()\\n    print(f"\\\\n{\\\'epoch\\\':>5s} {\\\'val macro\\\':>9s} {\\\'gold\\\':>7s}")\\n    for epoch_no, row in per_epoch_rows.iterrows():\\n        marker = \\\'  <- selected\\\' if epoch_no == best_epoch else \\\'\\\'\\n        print(f"{epoch_no + 1:5d} {row[\\\'val_macro\\\']:9.3f} {row[\\\'gold_macro\\\']:7.3f}{marker}")\\n    best = per_epoch_rows.loc[best_epoch]\\n    print(f"\\\\nbest val macro {best[\\\'val_macro\\\']:.3f} at epoch {best_epoch + 1}; gold there {best[\\\'gold_macro\\\']:.3f} (monitor only, never selected on)")\\n    _val = per_epoch_rows[\\\'val_macro\\\']\\n    _gold = per_epoch_rows[\\\'gold_macro\\\']\\n    _tail = _gold.loc[best_epoch:]\\n    print(\\\'\\\\nP1 -- val/gold divergence\\\')\\n    print(f\\\'  val   peak {_val.max():.3f} at epoch {_val.idxmax() + 1}, final {_val.iloc[-1]:.3f}  ({_val.iloc[-1] - _val.max():+.3f})\\\')\\n    print(f\\\'  gold  peak {_gold.max():.3f} at epoch {_gold.idxmax() + 1}, final {_gold.iloc[-1]:.3f}  ({_gold.iloc[-1] - _gold.max():+.3f})\\\')\\n    print(f\\\'  gold from the selected epoch on: range {_tail.max() - _tail.min():.3f} over {len(_tail)} epochs\\\')\\n    if _val.iloc[-1] < _val.max() - 0.005 and _tail.max() - _tail.min() < 0.02:\\n        print(\\\'  -> DIVERGENCE REPRODUCES. Mined val fell while gold held flat: the selector is biased, and every fold selected on it is paying for it. Ship the averaged weights.\\\')\\n    else:\\n        print(\\\'  -> no divergence in this run. Selecting on mined val is defensible here; re-read before adopting the averages fleet-wide.\\\')\\nexcept Exception as exc:\\n    print(f\\\'\\\\ncould not read the epoch table ({exc}); look under {csv_logger.save_dir} for metrics.csv\\\')\\nper_epoch = elapsed / EPOCHS\\nshare = len(index) / 4349\\nprint(f\\\'\\\\nthis shard is {share:.0%} of the training set\\\')\\nprint(f\\\'per epoch: {per_epoch / 60:.1f} min on {DEVICES} GPU(s)\\\')\\nprint(f\\\'projected 5 folds x {EPOCHS} epochs: {per_epoch / share * EPOCHS * 5 / 3600:.1f} h\\\')\\n\\nassert all(pathlib.Path(p).exists() for p in [CHECKPOINT, averager.ema_path, averager.swa_path])\\nprint("DDP_SMOKE_PASSED: fit returned and selected/EMA/SWA exports exist", flush=True)\\n\'\nCOMPARE_SOURCE = \'"""Compare aligned OOF predictions and a fixed 50/50 probability ensemble."""\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\\nfrom src import config\\nfrom src.labels.calibrate import _auc\\nfrom src.model import splits\\n\\n\\ndef read_predictions(paths, folded):\\n    frame = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)\\n    if frame[config.ID_COL].duplicated().any():\\n        raise ValueError("duplicate studies in OOF predictions")\\n    if "fold" not in frame:\\n        raise ValueError("OOF predictions must carry fold assignments")\\n    indexed = frame.set_index(config.ID_COL)\\n    expected = folded.set_index(config.ID_COL)["fold"]\\n    if not indexed.index.isin(expected.index).all():\\n        raise ValueError("OOF includes gold, unknown, or unlabelled studies")\\n    if not indexed["fold"].eq(expected.loc[indexed.index]).all():\\n        raise ValueError("OOF folds disagree with the declared split")\\n    probs = indexed[config.LABELS].to_numpy(float)\\n    if not np.isfinite(probs).all() or ((probs < 0) | (probs > 1)).any():\\n        raise ValueError("OOF probabilities must be finite and in [0, 1]")\\n    return indexed\\n\\n\\ndef aucs(truth, probs):\\n    return np.array([_auc(truth[:, i], probs[:, i]) for i in range(truth.shape[1])])\\n\\n\\ndef compare(labels, baseline, candidate, draws=1000, seed=0):\\n    shared = baseline.index.intersection(candidate.index).sort_values()\\n    if not len(shared):\\n        raise ValueError("no shared OOF studies")\\n    truth = (labels.set_index(config.ID_COL).loc[shared, config.LABELS].to_numpy() >= .5).astype(int)\\n    a, b = [frame.loc[shared, config.LABELS].to_numpy(float) for frame in (baseline, candidate)]\\n    predictions = {"baseline": a, "candidate": b, "blend_50_50": (a + b) / 2}\\n    results = {name: aucs(truth, pred) for name, pred in predictions.items()}\\n    table = pd.DataFrame(results, index=config.LABELS)\\n    table["candidate_delta"] = table.candidate - table.baseline\\n    table["blend_delta"] = table.blend_50_50 - table.baseline\\n    rng = np.random.default_rng(seed)\\n    fold = baseline.loc[shared, "fold"].to_numpy()\\n    blocks = [np.flatnonzero(fold == f) for f in np.unique(fold)]\\n    per_fold = {str(int(f)): {name: float(np.nanmean(aucs(truth[fold == f], pred[fold == f])))\\n                             for name, pred in predictions.items()} for f in np.unique(fold)}\\n    deltas = {"candidate": [], "blend_50_50": []}\\n    for _ in range(draws):\\n        chosen = np.concatenate([rng.choice(block, len(block), replace=True) for block in blocks])\\n        base = aucs(truth[chosen], a[chosen])\\n        for name in deltas:\\n            values = aucs(truth[chosen], predictions[name][chosen])\\n            valid = np.isfinite(base) & np.isfinite(values)\\n            if valid.any():\\n                deltas[name].append(float((values[valid] - base[valid]).mean()))\\n    summary = {"shared_studies": len(shared), "folds": sorted(set(map(int, fold))),\\n               "baseline_only": len(baseline.index.difference(shared)),\\n               "candidate_only": len(candidate.index.difference(shared)),\\n               "macro": {k: float(np.nanmean(v)) for k, v in results.items()},\\n               "per_fold_macro": per_fold,\\n               "paired_macro_delta_95ci": {\\n                   k: np.quantile(v, [.025, .975]).tolist() for k, v in deltas.items()},\\n               "limitation": "Mined-label OOF proxy; fold provenance asserted by files, not proof of training exclusion. Bootstrap holds fitted models fixed; it excludes retraining variability."}\\n    return table, summary\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--baseline", nargs="+", required=True)\\n    parser.add_argument("--candidate", nargs="+", required=True)\\n    parser.add_argument("--labels", default="data/meta/train_labels.parquet")\\n    parser.add_argument("--folds", help="Saved folds.parquet; default is the legacy study split")\\n    parser.add_argument("--draws", type=int, default=1000)\\n    parser.add_argument("--out", required=True)\\n    args = parser.parse_args()\\n    if args.draws < 1:\\n        parser.error("--draws must be positive")\\n    labels = pd.read_parquet(args.labels)\\n    folded = pd.read_parquet(args.folds) if args.folds else splits.make_folds(labels)\\n    a, b = [read_predictions(paths, folded) for paths in (args.baseline, args.candidate)]\\n    table, summary = compare(labels, a, b, args.draws)\\n    path = Path(args.out)\\n    path.mkdir(parents=True, exist_ok=True)\\n    table.to_csv(path / "per_finding.csv", index_label="finding")\\n    (path / "summary.json").write_text(json.dumps(summary, indent=2) + "\\\\n")\\n    print(table.round(4).to_string())\\n    print(json.dumps(summary, indent=2))\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\'\nLABELS_SHA256 = \'e54e46531aec6cd16131d70aa283c990542dfa17daa890ac54da663cf74569b0\'\nRECOVERY = None\nFOLD = 2\nCANDIDATE = \'slice_transformer\'\nSMOKE_ONLY = False\nVALIDATION_STUDIES = 870\nSMOKE_TIMEOUT = None\nRUN_TIMEOUT = None\n"""Remote body embedded by prepare_kaggle_training_pair.py."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef validate_contracts(contracts, candidate):\n    if candidate not in ("protocol", "classwise", "convnextv2", "slice_transformer") or len(contracts) != 2:\n        raise ValueError("Expected a registered baseline/candidate pair")\n    contracts = [dict(contract) for contract in contracts]\n    specs = [dict(contract.pop("experiment")) for contract in contracts]\n    if [spec.pop("name") for spec in specs] != ["baseline", candidate]:\n        raise ValueError("Unexpected experiment ordering")\n    if candidate in ("classwise", "convnextv2", "slice_transformer"):\n        for contract in contracts:\n            if not contract.get("shared_initial_weights_sha256"):\n                raise ValueError("Missing shared initialization hash")\n            contract.pop("initial_weights_sha256")\n    if candidate == "convnextv2":\n        for contract in contracts:\n            if (contract.get("aggregation_seed") != contract.get("seed")\n                    or contract.get("aggregation_seed") is None\n                    or contract.get("feature_dim") != 768\n                    or contract.get("input_defaults") != {\n                        "input_mean": None, "input_std": None, "backbone_lr_scale": 1.0}):\n                raise ValueError("Missing or invalid backbone comparison controls")\n    if contracts[0] != contracts[1]:\n        raise ValueError("Training settings or initial weights differ between runs")\n    if candidate == "slice_transformer":\n        if any(spec["series_selection"] != "protocol" for spec in specs):\n            raise ValueError("Slice comparison requires protocol selection in both arms")\n        for spec in specs:\n            spec.setdefault("slice_transformer", False)\n    variable, expected = {\n        "protocol": ("series_selection", ["uniform", "protocol"]),\n        "classwise": ("classwise_pool", [False, True]),\n        "convnextv2": ("backbone", ["convnext_tiny.fb_in22k_ft_in1k",\n                                    "convnextv2_tiny.fcmae_ft_in22k_in1k"]),\n        "slice_transformer": ("slice_transformer", [False, True]),\n    }[candidate]\n    if [spec.pop(variable) for spec in specs] != expected:\n        raise ValueError("Unexpected experiment variable values")\n    if specs[0] != specs[1]:\n        raise ValueError("More than the registered experiment variable changed")\n\n\ndef main():\n    output = Path("/kaggle/working")\n    started = time.monotonic()\n    progress = {"state": "running", "completed": [], "labels_sha256": LABELS_SHA256, "fold": FOLD}\n\n    def save_progress():\n        progress["elapsed_seconds"] = time.monotonic() - started\n        (output / "pair_progress.json").write_text(json.dumps(progress, indent=2) + "\\n")\n\n    save_progress()\n    try:\n        if RECOVERY:\n            checkpoints = attached_files("baseline_lit_fold1.ckpt")\n            assert len(checkpoints) == 1, "Recovery dataset is missing or ambiguous"\n            with checkpoints[0].open("rb") as handle:\n                assert hashlib.file_digest(handle, "sha256").hexdigest() == RECOVERY["checkpoint_sha256"]\n        smoke = output / "ddp_smoke"\n        smoke.mkdir(exist_ok=False)\n        (smoke / "train.py").write_text(SMOKE_SOURCE)\n        progress["active"] = "ddp_smoke"\n        save_progress()\n        subprocess.run([sys.executable, "-u", str(smoke / "train.py")],\n                       cwd=smoke, check=True, timeout=SMOKE_TIMEOUT)\n        progress["smoke_passed"] = True\n        save_progress()\n        if SMOKE_ONLY:\n            progress.update(state="complete", active=None, smoke_only=True)\n            return\n        for name, source in TRAINING_SCRIPTS.items():\n            directory = output / f"{name}_fold{FOLD}"\n            directory.mkdir(exist_ok=False)\n            script = directory / "train.py"\n            script.write_text(source)\n            progress["active"] = name\n            save_progress()\n            print(f"Starting {name}, fold {FOLD}, 12 epochs", flush=True)\n            subprocess.run([sys.executable, "-u", str(script)], cwd=directory,\n                           env={**os.environ, "PYTHONUNBUFFERED": "1"},\n                           check=True, timeout=RUN_TIMEOUT)\n            progress["completed"].append(name)\n            save_progress()\n\n        sys.path.insert(0, str(output / f"baseline_fold{FOLD}"))\n        import numpy as np\n        import pandas as pd\n        from src import config\n        from src.model import artifacts\n\n        paths = [output / f"{name}_fold{FOLD}" for name in TRAINING_SCRIPTS]\n        contracts = [json.loads((p / "run_contract.json").read_text()) for p in paths]\n        validate_contracts(contracts, CANDIDATE)\n        metadata = [artifacts.load_metadata(p / f"convnext_fold{FOLD}.pt") for p in paths]\n        for key in ("train_uids", "val_uids", "fold", "labels"):\n            if metadata[0][key] != metadata[1][key]:\n                raise ValueError(f"Different checkpoint provenance: {key}")\n        folded = [pd.read_parquet(p / "folds.parquet") for p in paths]\n        pd.testing.assert_frame_equal(folded[0], folded[1])\n        label_paths = attached_files("train_labels.parquet")\n        assert len(label_paths) == 1\n        assert hashlib.sha256(label_paths[0].read_bytes()).hexdigest() == LABELS_SHA256\n        labels = pd.read_parquet(label_paths[0])\n        comparison_path = output / "compare_oof.py"\n        comparison_path.write_text(COMPARE_SOURCE)\n        namespace = {"__name__": "pair_comparison", "__file__": str(comparison_path)}\n        exec(compile(COMPARE_SOURCE, str(comparison_path), "exec"), namespace)\n        frames = [namespace["read_predictions"]([p / f"oof_fold{FOLD}.parquet"], folded[0]) for p in paths]\n        expected = set(metadata[0]["val_uids"])\n        assert len(expected) == VALIDATION_STUDIES\n        assert expected == set(folded[0].loc[folded[0]["fold"] == FOLD, config.ID_COL])\n        assert all(set(frame.index) == expected for frame in frames)\n        table, summary = namespace["compare"](labels, *frames, draws=1000, seed=0)\n        assert np.isfinite(table.to_numpy()).all(), "Undefined validation AUC"\n        delta = summary["macro"]["candidate"] - summary["macro"]["baseline"]\n        # Apply the regression guard to all twelve findings, conservatively.\n        passed = delta >= .003 and bool((table.candidate_delta >= -.02).all())\n        summary["screening_gate"] = {\n            "passed": passed, "macro_delta": delta, "required_macro_delta": .003,\n            "maximum_finding_drop": .02, "guard_applies_to": "all twelve findings",\n            "next_step": (("repeat matched pair on fold 3" if FOLD == 1 else\n                           "review both screening folds before the remaining folds")\n                          if passed else "review before more training"),\n            "note": "Compute allocation screen, not statistical significance or leaderboard gain.",\n        }\n        summary["checkpoint_sha256"] = [m["sha256"] for m in metadata]\n        table.to_csv(output / "per_finding.csv", index_label="finding")\n        (output / "comparison.json").write_text(json.dumps(summary, indent=2) + "\\n")\n        print(table.round(5).to_string(), flush=True)\n        print(json.dumps(summary, indent=2), flush=True)\n        progress.update(state="complete", active=None, screening_passed=passed)\n    except BaseException as exc:\n        progress.update(state="failed", error=f"{type(exc).__name__}: {exc}")\n        raise\n    finally:\n        save_progress()\n\n\nif __name__ == "__main__":\n    main()\n'
LABELS_SHA256 = 'e54e46531aec6cd16131d70aa283c990542dfa17daa890ac54da663cf74569b0'


In [ ]:
stage_drive_inputs(DRIVE_CACHE, LOCAL_INPUTS, LABELS_SHA256)


In [ ]:
ATTEMPT = run_pair_on_drive(PAIR_SOURCE, DRIVE_OUT, LOCAL_INPUTS)
